# Olympus routing v6: retained-source handoff

CPU and internet disabled. The complete source is embedded and checked. Requires Python >=3.11, Torch >=2.4, NumPy, Pydantic >=2.10 and pytest. The default performs artificial qualification only. Scientific reproduction requires separate admission. Save the retained archive before ending the session. A generated notebook is not Kaggle execution evidence.


In [ ]:
import base64
import hashlib
import json
import os
import subprocess
import sys
import tarfile
import time
from pathlib import Path

PAYLOAD = {
    'LICENSE': (
        'T2x5bXB1cyBQcm9wcmlldGFyeSBMaWNlbnNlCgpDb3B5cmlnaHQgKGMpIDIwMjYgUnlhbi4gQWxs'
        'IHJpZ2h0cyByZXNlcnZlZC4KClRoZSBzb3VyY2UgY29kZSwgZG9jdW1lbnRhdGlvbiwgZGF0YSBk'
        'ZWZpbml0aW9ucywgYW5kIG90aGVyIG1hdGVyaWFscyBpbiB0aGlzCnJlcG9zaXRvcnkgYXJlIG1h'
        'ZGUgcHVibGljbHkgdmlld2FibGUgZm9yIGV2YWx1YXRpb24gYW5kIHJlc2VhcmNoIGRpc2N1c3Np'
        'b24uCk5vIHBlcm1pc3Npb24gaXMgZ3JhbnRlZCB0byBjb3B5LCBtb2RpZnksIGRpc3RyaWJ1dGUs'
        'IHN1YmxpY2Vuc2UsIHNlbGwsIGhvc3QsCm9yIGNyZWF0ZSBkZXJpdmF0aXZlIHdvcmtzIGZyb20g'
        'dGhlc2UgbWF0ZXJpYWxzIGV4Y2VwdCB3aGVyZSBhIHNlcGFyYXRlCndyaXR0ZW4gYWdyZWVtZW50'
        'IHdpdGggdGhlIGNvcHlyaWdodCBob2xkZXIgZXhwcmVzc2x5IHBlcm1pdHMgaXQuCgpUSEUgTUFU'
        'RVJJQUxTIEFSRSBQUk9WSURFRCAiQVMgSVMiLCBXSVRIT1VUIFdBUlJBTlRZIE9GIEFOWSBLSU5E'
        'LCBFWFBSRVNTIE9SCklNUExJRUQsIElOQ0xVRElORyBCVVQgTk9UIExJTUlURUQgVE8gV0FSUkFO'
        'VElFUyBPRiBNRVJDSEFOVEFCSUxJVFksIEZJVE5FU1MKRk9SIEEgUEFSVElDVUxBUiBQVVJQT1NF'
        'LCBUSVRMRSwgQU5EIE5PTi1JTkZSSU5HRU1FTlQuIElOIE5PIEVWRU5UIFNIQUxMIFRIRQpDT1BZ'
        'UklHSFQgSE9MREVSIEJFIExJQUJMRSBGT1IgQU5ZIENMQUlNLCBEQU1BR0VTLCBPUiBPVEhFUiBM'
        'SUFCSUxJVFkgQVJJU0lORwpGUk9NLCBPVVQgT0YsIE9SIElOIENPTk5FQ1RJT04gV0lUSCBUSEUg'
        'TUFURVJJQUxTIE9SIFRIRUlSIFVTRS4K'
    ),
    'olympus/__init__.py': (
        'IiIiT2x5bXB1cyBjb2duaXRpdmUgYXJjaGl0ZWN0dXJlLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGlt'
        'cG9ydCBhbm5vdGF0aW9ucwoKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKX192ZXJzaW9uX18gPSAi'
        'MC4yLjAiCgpfX2FsbF9fID0gWyJfX3ZlcnNpb25fXyIsICJhcHAiXQoKCmRlZiBfX2dldGF0dHJf'
        'XyhuYW1lOiBzdHIpIC0+IEFueToKICAgIGlmIG5hbWUgPT0gImFwcCI6CiAgICAgICAgZnJvbSBv'
        'bHltcHVzLmFwaSBpbXBvcnQgYXBwCgogICAgICAgIHJldHVybiBhcHAKICAgIHJhaXNlIEF0dHJp'
        'YnV0ZUVycm9yKGYibW9kdWxlIHtfX25hbWVfXyFyfSBoYXMgbm8gYXR0cmlidXRlIHtuYW1lIXJ9'
        'IikK'
    ),
    'olympus/api.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmltcG9ydCBzZWNy'
        'ZXRzCmltcG9ydCB0aHJlYWRpbmcKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IEFzeW5jSXRl'
        'cmF0b3IsIEl0ZXJhdG9yCmZyb20gY29udGV4dGxpYiBpbXBvcnQgYXN5bmNjb250ZXh0bWFuYWdl'
        'cgpmcm9tIGZ1bmN0b29scyBpbXBvcnQgbHJ1X2NhY2hlCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0'
        'aApmcm9tIHRlbXBmaWxlIGltcG9ydCBUZW1wb3JhcnlEaXJlY3RvcnkKZnJvbSB0eXBpbmcgaW1w'
        'b3J0IEFubm90YXRlZCwgTGl0ZXJhbAoKaW1wb3J0IGh0dHB4CmZyb20gZmFzdGFwaSBpbXBvcnQg'
        'RGVwZW5kcywgRmFzdEFQSSwgSGVhZGVyLCBIVFRQRXhjZXB0aW9uCmZyb20gcHlkYW50aWMgaW1w'
        'b3J0IEJhc2VNb2RlbCwgQ29uZmlnRGljdCwgRmllbGQKCmZyb20gb2x5bXB1cyBpbXBvcnQgX192'
        'ZXJzaW9uX18KZnJvbSBvbHltcHVzLmNvcmUuaW50ZXJwcmV0YXRpb24gaW1wb3J0IEludGVycHJl'
        'dGl2ZVN1cGVycG9zaXRpb25OZXR3b3JrCmZyb20gb2x5bXB1cy5jb3JlLndvcmtzcGFjZSBpbXBv'
        'cnQgTGF0ZW50V29ya3NwYWNlCmZyb20gb2x5bXB1cy5kZW1vcyBpbXBvcnQgKAogICAgcnVuX2Ft'
        'YmlndW91c19pbnRlcnByZXRhdGlvbl9kZW1vLAogICAgcnVuX2RpdmlzaW9uX2RlbW8sCiAgICBy'
        'dW5fZm9yZ2VfZGVtbywKICAgIHJ1bl9oZXJtZXNfZGVtbywKICAgIHJ1bl9yZXByZXNlbnRhdGlv'
        'bl9kZW1vLAogICAgcnVuX3JldHJvZGljdGlvbl9kZW1vLAogICAgcnVuX3RyYW5zZm9ybV9kZW1v'
        'LAopCmZyb20gb2x5bXB1cy5mb3JnZS5jb21waWxlciBpbXBvcnQgTmF0dXJhbExhbmd1YWdlQmVo'
        'YXZpb3JDb21waWxlcgpmcm9tIG9seW1wdXMuZm9yZ2UucnVudGltZSBpbXBvcnQgRm9yZ2VSdW50'
        'aW1lCmZyb20gb2x5bXB1cy5mb3VuZHJ5Lm9sbGFtYSBpbXBvcnQgT2xsYW1hQ2xpZW50CmZyb20g'
        'b2x5bXB1cy5mb3VuZHJ5LnNlcnZpY2UgaW1wb3J0IEZvdW5kcnlTZXJ2aWNlCgoKY2xhc3MgQXBp'
        'UmVxdWVzdChCYXNlTW9kZWwpOgogICAgbW9kZWxfY29uZmlnID0gQ29uZmlnRGljdChleHRyYT0i'
        'Zm9yYmlkIiwgc3RyX3N0cmlwX3doaXRlc3BhY2U9VHJ1ZSkKCgpjbGFzcyBXb3Jrc3BhY2VSZXF1'
        'ZXN0KEFwaVJlcXVlc3QpOgogICAgb2JqZWN0aXZlOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEs'
        'IG1heF9sZW5ndGg9Ml8wMDApCiAgICBwcm9tcHQ6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSwg'
        'bWF4X2xlbmd0aD01MF8wMDApCgoKY2xhc3MgQ29tcGlsZVJlcXVlc3QoQXBpUmVxdWVzdCk6CiAg'
        'ICBiZWhhdmlvcjogc3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTIwXzAwMCkK'
        'CgpjbGFzcyBSdW5SZXF1ZXN0KENvbXBpbGVSZXF1ZXN0KToKICAgIHByb21wdDogc3RyID0gRmll'
        'bGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTUwXzAwMCkKCgpjbGFzcyBDaGF0TWVzc2FnZShB'
        'cGlSZXF1ZXN0KToKICAgIHJvbGU6IExpdGVyYWxbInN5c3RlbSIsICJ1c2VyIiwgImFzc2lzdGFu'
        'dCJdCiAgICBjb250ZW50OiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9NTBf'
        'MDAwKQoKCmNsYXNzIENoYXRDb21wbGV0aW9uUmVxdWVzdChBcGlSZXF1ZXN0KToKICAgIG1vZGVs'
        'OiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9MjAwKQogICAgbWVzc2FnZXM6'
        'IGxpc3RbQ2hhdE1lc3NhZ2VdID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTEwMCkK'
        'ICAgIG1heF90b2tlbnM6IGludCA9IEZpZWxkKGRlZmF1bHQ9MTYwLCBnZT0xLCBsZT00XzA5NikK'
        'ICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IEZpZWxkKGRlZmF1bHQ9MC43LCBnZT0wLjAsIGxlPTIu'
        'MCkKICAgIHNlZWQ6IGludCB8IE5vbmUgPSBGaWVsZChkZWZhdWx0PU5vbmUsIGdlPTAsIGxlPTIq'
        'KjMyIC0gMSkKICAgIHN0cmVhbTogTGl0ZXJhbFtGYWxzZV0gPSBGYWxzZQoKCmNsYXNzIE9sbGFt'
        'YUdlbmVyYXRlUmVxdWVzdChBcGlSZXF1ZXN0KToKICAgIG1vZGVsOiBzdHIgPSBGaWVsZChtaW5f'
        'bGVuZ3RoPTEsIG1heF9sZW5ndGg9MjAwKQogICAgcHJvbXB0OiBzdHIgPSBGaWVsZChtaW5fbGVu'
        'Z3RoPTEsIG1heF9sZW5ndGg9NTBfMDAwKQogICAgc3lzdGVtOiBzdHIgfCBOb25lID0gRmllbGQo'
        'ZGVmYXVsdD1Ob25lLCBtYXhfbGVuZ3RoPTEwXzAwMCkKICAgIG1heF90b2tlbnM6IGludCA9IEZp'
        'ZWxkKGRlZmF1bHQ9MjU2LCBnZT0xLCBsZT00XzA5NikKICAgIGNvbnRleHRfdG9rZW5zOiBpbnQg'
        'PSBGaWVsZChkZWZhdWx0PTJfMDQ4LCBnZT0yNTYsIGxlPTEzMV8wNzIpCiAgICB0ZW1wZXJhdHVy'
        'ZTogZmxvYXQgPSBGaWVsZChkZWZhdWx0PTAuMCwgZ2U9MC4wLCBsZT0yLjApCgoKY29tcGlsZXIg'
        'PSBOYXR1cmFsTGFuZ3VhZ2VCZWhhdmlvckNvbXBpbGVyKCkKcnVudGltZSA9IEZvcmdlUnVudGlt'
        'ZSgpCmludGVycHJldGVyID0gSW50ZXJwcmV0aXZlU3VwZXJwb3NpdGlvbk5ldHdvcmsoKQoKCkBs'
        'cnVfY2FjaGUobWF4c2l6ZT0xKQpkZWYgZ2V0X2ZvdW5kcnlfc2VydmljZSgpIC0+IEZvdW5kcnlT'
        'ZXJ2aWNlOgogICAgcm9vdCA9IFBhdGgob3MuZW52aXJvbi5nZXQoIk9MWU1QVVNfRk9VTkRSWV9S'
        'T09UIiwgImFydGlmYWN0cy9mb3VuZHJ5IikpCiAgICByZXR1cm4gRm91bmRyeVNlcnZpY2Uocm9v'
        'dCkKCgpAbHJ1X2NhY2hlKG1heHNpemU9MSkKZGVmIGdldF9vbGxhbWFfY2xpZW50KCkgLT4gT2xs'
        'YW1hQ2xpZW50OgogICAgcmV0dXJuIE9sbGFtYUNsaWVudChvcy5lbnZpcm9uLmdldCgiT0xZTVBV'
        'U19PTExBTUFfVVJMIiwgImh0dHA6Ly8xMjcuMC4wLjE6MTE0MzQiKSkKCgpGb3VuZHJ5RGVwZW5k'
        'ZW5jeSA9IEFubm90YXRlZFtGb3VuZHJ5U2VydmljZSwgRGVwZW5kcyhnZXRfZm91bmRyeV9zZXJ2'
        'aWNlKV0KT2xsYW1hRGVwZW5kZW5jeSA9IEFubm90YXRlZFtPbGxhbWFDbGllbnQsIERlcGVuZHMo'
        'Z2V0X29sbGFtYV9jbGllbnQpXQoKCmRlZiByZXF1aXJlX3dvcmtsb2FkX2F1dGhvcml6YXRpb24o'
        'CiAgICBhdXRob3JpemF0aW9uOiBBbm5vdGF0ZWRbc3RyIHwgTm9uZSwgSGVhZGVyKCldID0gTm9u'
        'ZSwKKSAtPiBOb25lOgogICAgIiIiUmVxdWlyZSBhIGJlYXJlciB0b2tlbiBmb3Igc3RhdGVmdWwv'
        'Y29tcHV0ZSByb3V0ZXMgd2hlbiBvbmUgaXMgY29uZmlndXJlZC4KCiAgICBPbHltcHVzIHJlbWFp'
        'bnMgdXNhYmxlIGFzIGEgbG9vcGJhY2stb25seSBkZXZlbG9wbWVudCBzZXJ2aWNlIHdpdGhvdXQg'
        'YSB0b2tlbi4KICAgIEFueSBkZXBsb3ltZW50IHRoYXQgY3Jvc3NlcyBhIHRydXN0IGJvdW5kYXJ5'
        'IG11c3Qgc2V0IGBgT0xZTVBVU19BUElfVE9LRU5gYC4KICAgICIiIgoKICAgIHRva2VuID0gb3Mu'
        'ZW52aXJvbi5nZXQoIk9MWU1QVVNfQVBJX1RPS0VOIikKICAgIGlmIHRva2VuIGlzIE5vbmU6CiAg'
        'ICAgICAgcmV0dXJuCiAgICBpZiBub3QgdG9rZW4gb3IgYXV0aG9yaXphdGlvbiBpcyBOb25lIG9y'
        'IG5vdCBzZWNyZXRzLmNvbXBhcmVfZGlnZXN0KAogICAgICAgIGF1dGhvcml6YXRpb24sIGYiQmVh'
        'cmVyIHt0b2tlbn0iCiAgICApOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oCiAgICAgICAg'
        'ICAgIHN0YXR1c19jb2RlPTQwMSwKICAgICAgICAgICAgZGV0YWlsPSJ2YWxpZCBPbHltcHVzIGJl'
        'YXJlciB0b2tlbiByZXF1aXJlZCIsCiAgICAgICAgICAgIGhlYWRlcnM9eyJXV1ctQXV0aGVudGlj'
        'YXRlIjogIkJlYXJlciJ9LAogICAgICAgICkKCgpjbGFzcyBfV29ya2xvYWRMaW1pdGVyOgogICAg'
        'ZGVmIF9faW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgcmF3ID0gb3MuZW52aXJvbi5nZXQo'
        'Ik9MWU1QVVNfTUFYX0NPTkNVUlJFTlRfV09SS0xPQURTIiwgIjEiKQogICAgICAgIHRyeToKICAg'
        'ICAgICAgICAgbGltaXQgPSBpbnQocmF3KQogICAgICAgIGV4Y2VwdCBWYWx1ZUVycm9yIGFzIGVy'
        'cm9yOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIk9MWU1QVVNfTUFYX0NPTkNVUlJF'
        'TlRfV09SS0xPQURTIG11c3QgYmUgYW4gaW50ZWdlciIpIGZyb20gZXJyb3IKICAgICAgICBpZiBs'
        'aW1pdCA8IDEgb3IgbGltaXQgPiA2NDoKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJP'
        'TFlNUFVTX01BWF9DT05DVVJSRU5UX1dPUktMT0FEUyBtdXN0IGJlIGluIFsxLCA2NF0iKQogICAg'
        'ICAgIHNlbGYuX3NlbWFwaG9yZSA9IHRocmVhZGluZy5Cb3VuZGVkU2VtYXBob3JlKGxpbWl0KQoK'
        'ICAgIGRlZiBhY3F1aXJlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgbm90IHNlbGYuX3NlbWFw'
        'aG9yZS5hY3F1aXJlKGJsb2NraW5nPUZhbHNlKToKICAgICAgICAgICAgcmFpc2UgSFRUUEV4Y2Vw'
        'dGlvbihzdGF0dXNfY29kZT00MjksIGRldGFpbD0iT2x5bXB1cyB3b3JrbG9hZCBjYXBhY2l0eSBl'
        'eGhhdXN0ZWQiKQoKICAgIGRlZiByZWxlYXNlKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5f'
        'c2VtYXBob3JlLnJlbGVhc2UoKQoKCkBscnVfY2FjaGUobWF4c2l6ZT0xKQpkZWYgZ2V0X3dvcmts'
        'b2FkX2xpbWl0ZXIoKSAtPiBfV29ya2xvYWRMaW1pdGVyOgogICAgcmV0dXJuIF9Xb3JrbG9hZExp'
        'bWl0ZXIoKQoKCmRlZiBhZG1pdHRlZF93b3JrbG9hZCgKICAgIF9hdXRob3JpemVkOiBBbm5vdGF0'
        'ZWRbTm9uZSwgRGVwZW5kcyhyZXF1aXJlX3dvcmtsb2FkX2F1dGhvcml6YXRpb24pXSwKKSAtPiBJ'
        'dGVyYXRvcltOb25lXToKICAgIGxpbWl0ZXIgPSBnZXRfd29ya2xvYWRfbGltaXRlcigpCiAgICBs'
        'aW1pdGVyLmFjcXVpcmUoKQogICAgdHJ5OgogICAgICAgIHlpZWxkCiAgICBmaW5hbGx5OgogICAg'
        'ICAgIGxpbWl0ZXIucmVsZWFzZSgpCgoKV29ya2xvYWRBZG1pc3Npb24gPSBBbm5vdGF0ZWRbTm9u'
        'ZSwgRGVwZW5kcyhhZG1pdHRlZF93b3JrbG9hZCldCgoKQGFzeW5jY29udGV4dG1hbmFnZXIKYXN5'
        'bmMgZGVmIGxpZmVzcGFuKF86IEZhc3RBUEkpIC0+IEFzeW5jSXRlcmF0b3JbTm9uZV06CiAgICB5'
        'aWVsZAogICAgaWYgZ2V0X2ZvdW5kcnlfc2VydmljZS5jYWNoZV9pbmZvKCkuY3VycnNpemU6CiAg'
        'ICAgICAgZ2V0X2ZvdW5kcnlfc2VydmljZSgpLmNsb3NlKCkKICAgICAgICBnZXRfZm91bmRyeV9z'
        'ZXJ2aWNlLmNhY2hlX2NsZWFyKCkKICAgIGdldF9vbGxhbWFfY2xpZW50LmNhY2hlX2NsZWFyKCkK'
        'CgphcHAgPSBGYXN0QVBJKHRpdGxlPSJPbHltcHVzIEZvcmdlIEFQSSIsIHZlcnNpb249X192ZXJz'
        'aW9uX18sIGxpZmVzcGFuPWxpZmVzcGFuKQoKCkBhcHAuZ2V0KCIvaGVhbHRoIikKZGVmIGhlYWx0'
        'aCgpIC0+IGRpY3Rbc3RyLCBzdHJdOgogICAgcmV0dXJuIHsic3RhdHVzIjogIm9rIn0KCgpAYXBw'
        'LnBvc3QoIi93b3Jrc3BhY2UvYW5hbHl6ZSIpCmRlZiBhbmFseXplX3dvcmtzcGFjZShyZXF1ZXN0'
        'OiBXb3Jrc3BhY2VSZXF1ZXN0KSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIGJyYW5jaGVzID0g'
        'aW50ZXJwcmV0ZXIuYW5hbHl6ZShyZXF1ZXN0LnByb21wdCkKICAgIHdvcmtzcGFjZSA9IExhdGVu'
        'dFdvcmtzcGFjZSgKICAgICAgICBvYmplY3RpdmU9cmVxdWVzdC5vYmplY3RpdmUsCiAgICAgICAg'
        'aHlwb3RoZXNlcz1pbnRlcnByZXRlci5zdW1tYXJpemVfaHlwb3RoZXNlcyhicmFuY2hlcyksCiAg'
        'ICAgICAgYWN0aXZlX3BsYW49WyJpbnRlcnByZXQiLCAidmVyaWZ5IiwgIm1lcmdlIl0sCiAgICAg'
        'ICAgdG9vbHNfYXZhaWxhYmxlPVsiZmlsZXN5c3RlbSIsICJzaGVsbCIsICJjYWxjdWxhdGlvbiJd'
        'LAogICAgICAgIGJyYW5jaF9hY3RpdmF0aW9ucz17YnJhbmNoLmNhdGVnb3J5LnZhbHVlOiBicmFu'
        'Y2guY29uZmlkZW5jZSBmb3IgYnJhbmNoIGluIGJyYW5jaGVzfSwKICAgICkKICAgIHJldHVybiB7'
        'CiAgICAgICAgIndvcmtzcGFjZSI6IHdvcmtzcGFjZS5tb2RlbF9kdW1wKG1vZGU9Impzb24iKSwK'
        'ICAgICAgICAiYnJhbmNoZXMiOiBbYnJhbmNoLm1vZGVsX2R1bXAobW9kZT0ianNvbiIpIGZvciBi'
        'cmFuY2ggaW4gYnJhbmNoZXNdLAogICAgfQoKCkBhcHAucG9zdCgiL2ZvcmdlL2NvbXBpbGUiKQpk'
        'ZWYgY29tcGlsZV9iZWhhdmlvcihyZXF1ZXN0OiBDb21waWxlUmVxdWVzdCkgLT4gZGljdFtzdHIs'
        'IG9iamVjdF06CiAgICBjb21waWxhdGlvbiA9IGNvbXBpbGVyLmNvbXBpbGUocmVxdWVzdC5iZWhh'
        'dmlvcikKICAgIHJldHVybiB7CiAgICAgICAgInNwZWMiOiBjb21waWxhdGlvbi5zcGVjLm1vZGVs'
        'X2R1bXAobW9kZT0ianNvbiIpLAogICAgICAgICJpc3N1ZXMiOiBbaXNzdWUubW9kZWxfZHVtcCht'
        'b2RlPSJqc29uIikgZm9yIGlzc3VlIGluIGNvbXBpbGF0aW9uLmlzc3Vlc10sCiAgICAgICAgIm1l'
        'cm1haWQiOiBjb21waWxhdGlvbi5zcGVjLnRvX21lcm1haWQoKSwKICAgIH0KCgpAYXBwLnBvc3Qo'
        'Ii9mb3JnZS9ydW4iKQpkZWYgcnVuX2JlaGF2aW9yKHJlcXVlc3Q6IFJ1blJlcXVlc3QsIF9hZG1p'
        'c3Npb246IFdvcmtsb2FkQWRtaXNzaW9uKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIGNvbXBp'
        'bGF0aW9uID0gY29tcGlsZXIuY29tcGlsZShyZXF1ZXN0LmJlaGF2aW9yKQogICAgcmVzdWx0ID0g'
        'cnVudGltZS5leGVjdXRlKGNvbXBpbGF0aW9uLnNwZWMsIHJlcXVlc3QucHJvbXB0KQogICAgcmV0'
        'dXJuIHsKICAgICAgICAic3BlYyI6IGNvbXBpbGF0aW9uLnNwZWMubW9kZWxfZHVtcChtb2RlPSJq'
        'c29uIiksCiAgICAgICAgIm91dHB1dHMiOiByZXN1bHQub3V0cHV0cywKICAgICAgICAiZXZlbnRz'
        'IjogWwogICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAibm9kZV9pZCI6IGV2ZW50Lm5vZGVf'
        'aWQsCiAgICAgICAgICAgICAgICAia2luZCI6IGV2ZW50LmtpbmQsCiAgICAgICAgICAgICAgICAi'
        'cGF5bG9hZCI6IGV2ZW50LnBheWxvYWQsCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZm9yIGV2'
        'ZW50IGluIHJlc3VsdC5ldmVudHMKICAgICAgICBdLAogICAgfQoKCkBscnVfY2FjaGUobWF4c2l6'
        'ZT0xKQpkZWYgZGVtb3MoKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIHdpdGggVGVtcG9yYXJ5'
        'RGlyZWN0b3J5KHByZWZpeD0ib2x5bXB1cy1kZW1vcy0iKSBhcyBkaXJlY3Rvcnk6CiAgICAgICAg'
        'cmV0dXJuIHsKICAgICAgICAgICAgKipydW5fYW1iaWd1b3VzX2ludGVycHJldGF0aW9uX2RlbW8o'
        'KSwKICAgICAgICAgICAgKipydW5fcmV0cm9kaWN0aW9uX2RlbW8oKSwKICAgICAgICAgICAgKipy'
        'dW5fdHJhbnNmb3JtX2RlbW8oKSwKICAgICAgICAgICAgKipydW5fcmVwcmVzZW50YXRpb25fZGVt'
        'bygpLAogICAgICAgICAgICAqKnJ1bl9kaXZpc2lvbl9kZW1vKCksCiAgICAgICAgICAgICoqcnVu'
        'X2ZvcmdlX2RlbW8oKSwKICAgICAgICAgICAgImhlcm1lcyI6IHJ1bl9oZXJtZXNfZGVtbyhQYXRo'
        'KGRpcmVjdG9yeSkgLyAiaGVybWVzLnNxbGl0ZTMiKSwKICAgICAgICB9CgoKYXBwLmdldCgiL2Rl'
        'bW9zIikoZGVtb3MpCgoKQGFwcC5nZXQoIi9mb3VuZHJ5L3N0YXR1cyIpCmRlZiBmb3VuZHJ5X3N0'
        'YXR1cygKICAgIHNlcnZpY2U6IEZvdW5kcnlEZXBlbmRlbmN5LAopIC0+IGRpY3Rbc3RyLCBvYmpl'
        'Y3RdOgogICAgcmV0dXJuIHNlcnZpY2Uuc3RhdHVzKCkKCgpAYXBwLnBvc3QoIi9mb3VuZHJ5L3Zl'
        'cmlmeSIpCmRlZiB2ZXJpZnlfZm91bmRyeSgKICAgIHNlcnZpY2U6IEZvdW5kcnlEZXBlbmRlbmN5'
        'LAogICAgX2FkbWlzc2lvbjogV29ya2xvYWRBZG1pc3Npb24sCikgLT4gZGljdFtzdHIsIG9iamVj'
        'dF06CiAgICBzYW1wbGUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50IC8gImZvdW5k'
        'cnkvZm91bmRyeV92ZXJpZmljYXRpb24udHh0IgogICAgdHJ5OgogICAgICAgIHJldHVybiBzZXJ2'
        'aWNlLnJ1bl92ZXJpZmljYXRpb25fcGlwZWxpbmUoc2FtcGxlKS5tb2RlbF9kdW1wKG1vZGU9Impz'
        'b24iKQogICAgZXhjZXB0IChPU0Vycm9yLCBSdW50aW1lRXJyb3IsIFZhbHVlRXJyb3IpIGFzIGVy'
        'cm9yOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NDIyLCBkZXRhaWw9'
        'c3RyKGVycm9yKSkgZnJvbSBlcnJvcgoKCkBhcHAuZ2V0KCIvdjEvbW9kZWxzIikKZGVmIGxpc3Rf'
        'Zm91bmRyeV9tb2RlbHMoCiAgICBzZXJ2aWNlOiBGb3VuZHJ5RGVwZW5kZW5jeSwKKSAtPiBkaWN0'
        'W3N0ciwgb2JqZWN0XToKICAgIHJldHVybiB7CiAgICAgICAgIm9iamVjdCI6ICJsaXN0IiwKICAg'
        'ICAgICAiZGF0YSI6IFsKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImlkIjogbW9kZWwu'
        'bW9kZWxfaWQsCiAgICAgICAgICAgICAgICAib2JqZWN0IjogIm1vZGVsIiwKICAgICAgICAgICAg'
        'ICAgICJvd25lZF9ieSI6ICJidTFsZC1vbHltcHVzIiwKICAgICAgICAgICAgICAgICJzdGF0dXMi'
        'OiBtb2RlbC5zdGF0dXMudmFsdWUsCiAgICAgICAgICAgICAgICAiY2FwYWJpbGl0aWVzIjogbW9k'
        'ZWwuY2FwYWJpbGl0aWVzLAogICAgICAgICAgICAgICAgImxpbWl0YXRpb25zIjogbW9kZWwubGlt'
        'aXRhdGlvbnMsCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZm9yIG1vZGVsIGluIHNlcnZpY2Uu'
        'bGlzdF9tb2RlbHMoKQogICAgICAgIF0sCiAgICB9CgoKQGFwcC5wb3N0KCIvdjEvY2hhdC9jb21w'
        'bGV0aW9ucyIpCmRlZiBjcmVhdGVfY2hhdF9jb21wbGV0aW9uKAogICAgcmVxdWVzdDogQ2hhdENv'
        'bXBsZXRpb25SZXF1ZXN0LAogICAgc2VydmljZTogRm91bmRyeURlcGVuZGVuY3ksCiAgICBfYWRt'
        'aXNzaW9uOiBXb3JrbG9hZEFkbWlzc2lvbiwKKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIHBy'
        'b21wdCA9ICJcbiIuam9pbihmInttZXNzYWdlLnJvbGV9OiB7bWVzc2FnZS5jb250ZW50fSIgZm9y'
        'IG1lc3NhZ2UgaW4gcmVxdWVzdC5tZXNzYWdlcykKICAgIHRyeToKICAgICAgICByZXN1bHQgPSBz'
        'ZXJ2aWNlLmdlbmVyYXRlKAogICAgICAgICAgICByZXF1ZXN0Lm1vZGVsLAogICAgICAgICAgICBw'
        'cm9tcHQsCiAgICAgICAgICAgIG1heF9jaGFyYWN0ZXJzPXJlcXVlc3QubWF4X3Rva2VucywKICAg'
        'ICAgICAgICAgdGVtcGVyYXR1cmU9cmVxdWVzdC50ZW1wZXJhdHVyZSwKICAgICAgICAgICAgc2Vl'
        'ZD1yZXF1ZXN0LnNlZWQsCiAgICAgICAgKQogICAgZXhjZXB0IEtleUVycm9yIGFzIGVycm9yOgog'
        'ICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NDA0LCBkZXRhaWw9c3RyKGVy'
        'cm9yKSkgZnJvbSBlcnJvcgogICAgZXhjZXB0IChPU0Vycm9yLCBSdW50aW1lRXJyb3IsIFZhbHVl'
        'RXJyb3IpIGFzIGVycm9yOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9'
        'NDIyLCBkZXRhaWw9c3RyKGVycm9yKSkgZnJvbSBlcnJvcgogICAgY2hlY2twb2ludF9zaGEgPSBz'
        'dHIocmVzdWx0LmV2aWRlbmNlWyJjaGVja3BvaW50X3NoYTI1NiJdKQogICAgcmV0dXJuIHsKICAg'
        'ICAgICAiaWQiOiBmImNoYXRjbXBsLXtjaGVja3BvaW50X3NoYVs6MTZdfSIsCiAgICAgICAgIm9i'
        'amVjdCI6ICJjaGF0LmNvbXBsZXRpb24iLAogICAgICAgICJtb2RlbCI6IHJlc3VsdC5tb2RlbCwK'
        'ICAgICAgICAiY2hvaWNlcyI6IFsKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImluZGV4'
        'IjogMCwKICAgICAgICAgICAgICAgICJtZXNzYWdlIjogeyJyb2xlIjogImFzc2lzdGFudCIsICJj'
        'b250ZW50IjogcmVzdWx0LmNvbnRlbnR9LAogICAgICAgICAgICAgICAgImZpbmlzaF9yZWFzb24i'
        'OiByZXN1bHQuZmluaXNoX3JlYXNvbiwKICAgICAgICAgICAgfQogICAgICAgIF0sCiAgICAgICAg'
        'InVzYWdlIjogewogICAgICAgICAgICAicHJvbXB0X3Rva2VucyI6IHJlc3VsdC5wcm9tcHRfY2hh'
        'cmFjdGVycywKICAgICAgICAgICAgImNvbXBsZXRpb25fdG9rZW5zIjogcmVzdWx0LmdlbmVyYXRl'
        'ZF9jaGFyYWN0ZXJzLAogICAgICAgICAgICAidG90YWxfdG9rZW5zIjogcmVzdWx0LnByb21wdF9j'
        'aGFyYWN0ZXJzICsgcmVzdWx0LmdlbmVyYXRlZF9jaGFyYWN0ZXJzLAogICAgICAgICAgICAidW5p'
        'dCI6ICJjaGFyYWN0ZXJzIiwKICAgICAgICB9LAogICAgICAgICJldmlkZW5jZSI6IHJlc3VsdC5l'
        'dmlkZW5jZSwKICAgIH0KCgpAYXBwLmdldCgiL2ZvdW5kcnkvcHJvdmlkZXJzL29sbGFtYSIpCmRl'
        'ZiBvbGxhbWFfaGVhbHRoKAogICAgY2xpZW50OiBPbGxhbWFEZXBlbmRlbmN5LAopIC0+IGRpY3Rb'
        'c3RyLCBvYmplY3RdOgogICAgdHJ5OgogICAgICAgIHJldHVybiBjbGllbnQuaGVhbHRoKCkKICAg'
        'IGV4Y2VwdCAoaHR0cHguSFRUUEVycm9yLCBWYWx1ZUVycm9yKSBhcyBlcnJvcjoKICAgICAgICBy'
        'YWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTUwMywgZGV0YWlsPXN0cihlcnJvcikpIGZy'
        'b20gZXJyb3IKCgpAYXBwLnBvc3QoIi9mb3VuZHJ5L3Byb3ZpZGVycy9vbGxhbWEvZ2VuZXJhdGUi'
        'KQpkZWYgb2xsYW1hX2dlbmVyYXRlKAogICAgcmVxdWVzdDogT2xsYW1hR2VuZXJhdGVSZXF1ZXN0'
        'LAogICAgY2xpZW50OiBPbGxhbWFEZXBlbmRlbmN5LAogICAgX2FkbWlzc2lvbjogV29ya2xvYWRB'
        'ZG1pc3Npb24sCikgLT4gZGljdFtzdHIsIG9iamVjdF06CiAgICB0cnk6CiAgICAgICAgcmV0dXJu'
        'IGNsaWVudC5nZW5lcmF0ZSgKICAgICAgICAgICAgbW9kZWw9cmVxdWVzdC5tb2RlbCwKICAgICAg'
        'ICAgICAgcHJvbXB0PXJlcXVlc3QucHJvbXB0LAogICAgICAgICAgICBzeXN0ZW09cmVxdWVzdC5z'
        'eXN0ZW0sCiAgICAgICAgICAgIG1heF90b2tlbnM9cmVxdWVzdC5tYXhfdG9rZW5zLAogICAgICAg'
        'ICAgICBjb250ZXh0X3Rva2Vucz1yZXF1ZXN0LmNvbnRleHRfdG9rZW5zLAogICAgICAgICAgICB0'
        'ZW1wZXJhdHVyZT1yZXF1ZXN0LnRlbXBlcmF0dXJlLAogICAgICAgICkubW9kZWxfZHVtcChtb2Rl'
        'PSJqc29uIikKICAgIGV4Y2VwdCBodHRweC5IVFRQU3RhdHVzRXJyb3IgYXMgZXJyb3I6CiAgICAg'
        'ICAgc3RhdHVzID0gNDA0IGlmIGVycm9yLnJlc3BvbnNlLnN0YXR1c19jb2RlID09IDQwNCBlbHNl'
        'IDUwMwogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9c3RhdHVzLCBkZXRh'
        'aWw9c3RyKGVycm9yKSkgZnJvbSBlcnJvcgogICAgZXhjZXB0IChodHRweC5IVFRQRXJyb3IsIFZh'
        'bHVlRXJyb3IpIGFzIGVycm9yOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2Nv'
        'ZGU9NTAzLCBkZXRhaWw9c3RyKGVycm9yKSkgZnJvbSBlcnJvcgo='
    ),
    'olympus/cli.py': (
        '...'
    ),
    'olympus/core/__init__.py': (
        'IiIiQ29yZSBjb2duaXRpdmUgcnVudGltZSBwcmltaXRpdmVzLiIiIgo='
    ),
    'olympus/core/budget.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBvbHltcHVzLmNvcmUuc2No'
        'ZW1hcyBpbXBvcnQgQnJhbmNoU2NvcmUsIENvbXB1dGVCdWRnZXQsIFN0cmljdE1vZGVsCgoKY2xh'
        'c3MgU2NoZWR1bGVkQnJhbmNoKFN0cmljdE1vZGVsKToKICAgIGJyYW5jaF9pZDogc3RyCiAgICBw'
        'cmlvcml0eTogZmxvYXQKICAgIGZyb3plbjogYm9vbCA9IEZhbHNlCiAgICByZXZpdmVkOiBib29s'
        'ID0gRmFsc2UKCgpjbGFzcyBBZGFwdGl2ZUNvbXB1dGVBbGxvY2F0b3I6CiAgICBkZWYgc2NoZWR1'
        'bGUoCiAgICAgICAgc2VsZiwKICAgICAgICBzY29yZXM6IGRpY3Rbc3RyLCBCcmFuY2hTY29yZV0s'
        'CiAgICAgICAgYnVkZ2V0OiBDb21wdXRlQnVkZ2V0LAogICAgKSAtPiBsaXN0W1NjaGVkdWxlZEJy'
        'YW5jaF06CiAgICAgICAgcmFua2VkID0gc29ydGVkKHNjb3Jlcy5pdGVtcygpLCBrZXk9bGFtYmRh'
        'IGl0ZW06IGl0ZW1bMV0ucHJpb3JpdHkoKSwgcmV2ZXJzZT1UcnVlKQogICAgICAgIHNjaGVkdWxl'
        'ZDogbGlzdFtTY2hlZHVsZWRCcmFuY2hdID0gW10KICAgICAgICBmb3IgaW5kZXgsIChicmFuY2hf'
        'aWQsIHNjb3JlKSBpbiBlbnVtZXJhdGUocmFua2VkKToKICAgICAgICAgICAgaWYgaW5kZXggPj0g'
        'YnVkZ2V0LmJyYW5jaF9idWRnZXQ6CiAgICAgICAgICAgICAgICBzY2hlZHVsZWQuYXBwZW5kKAog'
        'ICAgICAgICAgICAgICAgICAgIFNjaGVkdWxlZEJyYW5jaChicmFuY2hfaWQ9YnJhbmNoX2lkLCBw'
        'cmlvcml0eT1zY29yZS5wcmlvcml0eSgpLCBmcm96ZW49VHJ1ZSkKICAgICAgICAgICAgICAgICkK'
        'ICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHJldml2ZWQgPSBzY29yZS5pbmZv'
        'cm1hdGlvbl9nYWluID4gMC43IGFuZCBzY29yZS5wbGF1c2liaWxpdHkgPCAwLjQKICAgICAgICAg'
        'ICAgc2NoZWR1bGVkLmFwcGVuZCgKICAgICAgICAgICAgICAgIFNjaGVkdWxlZEJyYW5jaCgKICAg'
        'ICAgICAgICAgICAgICAgICBicmFuY2hfaWQ9YnJhbmNoX2lkLAogICAgICAgICAgICAgICAgICAg'
        'IHByaW9yaXR5PXJvdW5kKHNjb3JlLnByaW9yaXR5KCksIDYpLAogICAgICAgICAgICAgICAgICAg'
        'IHJldml2ZWQ9cmV2aXZlZCwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgICAg'
        'IHJldHVybiBzY2hlZHVsZWQK'
    ),
    'olympus/core/division.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucy5hYmMg'
        'aW1wb3J0IENhbGxhYmxlCmZyb20gY29uY3VycmVudC5mdXR1cmVzIGltcG9ydCBUaHJlYWRQb29s'
        'RXhlY3V0b3IKCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5z'
        'Y2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoKCmNsYXNzIFNwZWNpYWxpc3RUYXNrKFN0cmljdE1v'
        'ZGVsKToKICAgIG5hbWU6IHN0cgogICAgcGF5bG9hZDogc3RyCiAgICBkZXBlbmRlbmNpZXM6IGxp'
        'c3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQoKCmNsYXNzIFNwZWNpYWxpc3RS'
        'ZXN1bHQoU3RyaWN0TW9kZWwpOgogICAgbmFtZTogc3RyCiAgICBvdXRwdXQ6IHN0cgogICAgY29u'
        'ZmlkZW5jZTogZmxvYXQKCgpjbGFzcyBOZXVyYWxEaXZpc2lvblJlYXNzZW1ibHk6CiAgICBkZWYg'
        'ZXhlY3V0ZSgKICAgICAgICBzZWxmLAogICAgICAgIHRhc2tzOiBsaXN0W1NwZWNpYWxpc3RUYXNr'
        'XSwKICAgICAgICBzcGVjaWFsaXN0czogZGljdFtzdHIsIENhbGxhYmxlW1tzdHJdLCBTcGVjaWFs'
        'aXN0UmVzdWx0XV0sCiAgICApIC0+IHR1cGxlW2xpc3RbU3BlY2lhbGlzdFJlc3VsdF0sIFNwZWNp'
        'YWxpc3RSZXN1bHRdOgogICAgICAgIHdpdGggVGhyZWFkUG9vbEV4ZWN1dG9yKG1heF93b3JrZXJz'
        'PW1pbig0LCBsZW4odGFza3MpIG9yIDEpKSBhcyBwb29sOgogICAgICAgICAgICBmdXR1cmVzID0g'
        'W3Bvb2wuc3VibWl0KHNwZWNpYWxpc3RzW3Rhc2submFtZV0sIHRhc2sucGF5bG9hZCkgZm9yIHRh'
        'c2sgaW4gdGFza3NdCiAgICAgICAgICAgIHJlc3VsdHMgPSBbZnV0dXJlLnJlc3VsdCgpIGZvciBm'
        'dXR1cmUgaW4gZnV0dXJlc10KICAgICAgICBjb21iaW5lZF90ZXh0ID0gIiB8ICIuam9pbihmInty'
        'ZXN1bHQubmFtZX06IHtyZXN1bHQub3V0cHV0fSIgZm9yIHJlc3VsdCBpbiByZXN1bHRzKQogICAg'
        'ICAgIGNvbnNpc3RlbmN5ID0gc3VtKHJlc3VsdC5jb25maWRlbmNlIGZvciByZXN1bHQgaW4gcmVz'
        'dWx0cykgLyBtYXgoMSwgbGVuKHJlc3VsdHMpKQogICAgICAgIGZpbmFsID0gU3BlY2lhbGlzdFJl'
        'c3VsdCgKICAgICAgICAgICAgbmFtZT0icmVhc3NlbWJseSIsCiAgICAgICAgICAgIG91dHB1dD1j'
        'b21iaW5lZF90ZXh0LAogICAgICAgICAgICBjb25maWRlbmNlPXJvdW5kKGNvbnNpc3RlbmN5LCA0'
        'KSwKICAgICAgICApCiAgICAgICAgcmV0dXJuIHJlc3VsdHMsIGZpbmFsCg=='
    ),
    'olympus/core/interpretation.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucyBpbXBv'
        'cnQgQ291bnRlcgpmcm9tIGVudW0gaW1wb3J0IFN0ckVudW0KCmZyb20gcHlkYW50aWMgaW1wb3J0'
        'IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBCcmFuY2hTY29yZSwgQ29t'
        'cHV0ZUJ1ZGdldCwgRXZpZGVuY2UsIEh5cG90aGVzaXMsIFN0cmljdE1vZGVsCgoKY2xhc3MgSW50'
        'ZXJwcmV0YXRpb25DYXRlZ29yeShTdHJFbnVtKToKICAgIExJVEVSQUwgPSAibGl0ZXJhbCIKICAg'
        'IENPTkNSRVRFX1JFU0VNQkxBTkNFID0gImNvbmNyZXRlX3Jlc2VtYmxhbmNlIgogICAgQUJTVFJB'
        'Q1RfQ09OQ1JFVEVfQU5BTE9HWSA9ICJhYnN0cmFjdF9jb25jcmV0ZV9hbmFsb2d5IgogICAgQUJT'
        'VFJBQ1RfTk9OX0NPTkNSRVRFID0gImFic3RyYWN0X25vbl9jb25jcmV0ZSIKICAgIENBVVNBTCA9'
        'ICJjYXVzYWwiCiAgICBURU1QT1JBTCA9ICJ0ZW1wb3JhbCIKICAgIFNZTUJPTElDID0gInN5bWJv'
        'bGljIgogICAgQURWRVJTQVJJQUwgPSAiYWR2ZXJzYXJpYWwiCiAgICBOVUxMID0gIm51bGwiCgoK'
        'Y2xhc3MgSW50ZXJwcmV0aXZlQnJhbmNoKFN0cmljdE1vZGVsKToKICAgIGNhdGVnb3J5OiBJbnRl'
        'cnByZXRhdGlvbkNhdGVnb3J5CiAgICBzdGF0ZW1lbnQ6IHN0cgogICAgc2NvcmU6IEJyYW5jaFNj'
        'b3JlCiAgICBzdXBwb3J0OiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkK'
        'ICAgIGNvbmZpZGVuY2U6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCgoKY2xhc3MgSW50'
        'ZXJwcmV0aXZlU3VwZXJwb3NpdGlvbk5ldHdvcms6CiAgICBhbmFsb2d5X2tleXdvcmRzID0geyJs'
        'aWtlIiwgImFzIiwgInJlc2VtYmxlcyIsICJzaW1pbGFyIiwgIm1pcnJvciJ9CiAgICBzeW1ib2xp'
        'Y19rZXl3b3JkcyA9IHsic3ltYm9sIiwgIm1lYW5pbmciLCAic3RvcnkiLCAiZHJlYW0iLCAic2ln'
        'bmFsIn0KICAgIGNhdXNhbF9rZXl3b3JkcyA9IHsiYmVjYXVzZSIsICJ0aGVyZWZvcmUiLCAiY2F1'
        'c2VkIiwgImxlZCIsICJkcml2ZXMifQogICAgdGVtcG9yYWxfa2V5d29yZHMgPSB7ImJlZm9yZSIs'
        'ICJhZnRlciIsICJkdXJpbmciLCAiZnV0dXJlIiwgImhpc3RvcnkifQoKICAgIGRlZiBhbmFseXpl'
        'KAogICAgICAgIHNlbGYsCiAgICAgICAgcHJvbXB0OiBzdHIsCiAgICAgICAgZXZpZGVuY2U6IGxp'
        'c3RbRXZpZGVuY2VdIHwgTm9uZSA9IE5vbmUsCiAgICAgICAgYnVkZ2V0OiBDb21wdXRlQnVkZ2V0'
        'IHwgTm9uZSA9IE5vbmUsCiAgICApIC0+IGxpc3RbSW50ZXJwcmV0aXZlQnJhbmNoXToKICAgICAg'
        'ICBldmlkZW5jZSA9IGV2aWRlbmNlIG9yIFtdCiAgICAgICAgYnVkZ2V0ID0gYnVkZ2V0IG9yIENv'
        'bXB1dGVCdWRnZXQoKQogICAgICAgIGxvd2VyZWQgPSBwcm9tcHQubG93ZXIoKQogICAgICAgIHRv'
        'a2VucyA9IGxvd2VyZWQuc3BsaXQoKQogICAgICAgIGNvdW50cyA9IENvdW50ZXIodG9rZW5zKQog'
        'ICAgICAgIGJyYW5jaGVzID0gWwogICAgICAgICAgICBzZWxmLl9tYWtlX2JyYW5jaChJbnRlcnBy'
        'ZXRhdGlvbkNhdGVnb3J5LkxJVEVSQUwsIHByb21wdCwgY291bnRzLCBldmlkZW5jZSksCiAgICAg'
        'ICAgICAgIHNlbGYuX21ha2VfYnJhbmNoKAogICAgICAgICAgICAgICAgSW50ZXJwcmV0YXRpb25D'
        'YXRlZ29yeS5DT05DUkVURV9SRVNFTUJMQU5DRSwKICAgICAgICAgICAgICAgIGYiUGh5c2ljYWwg'
        'cmVzZW1ibGFuY2UgcmVhZGluZzoge3Byb21wdH0iLAogICAgICAgICAgICAgICAgY291bnRzLAog'
        'ICAgICAgICAgICAgICAgZXZpZGVuY2UsCiAgICAgICAgICAgICksCiAgICAgICAgICAgIHNlbGYu'
        'X21ha2VfYnJhbmNoKAogICAgICAgICAgICAgICAgSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5BQlNU'
        'UkFDVF9DT05DUkVURV9BTkFMT0dZLAogICAgICAgICAgICAgICAgZiJBbmFsb2d5IHJlYWRpbmc6'
        'IHtwcm9tcHR9IiwKICAgICAgICAgICAgICAgIGNvdW50cywKICAgICAgICAgICAgICAgIGV2aWRl'
        'bmNlLAogICAgICAgICAgICApLAogICAgICAgICAgICBzZWxmLl9tYWtlX2JyYW5jaCgKICAgICAg'
        'ICAgICAgICAgIEludGVycHJldGF0aW9uQ2F0ZWdvcnkuQUJTVFJBQ1RfTk9OX0NPTkNSRVRFLAog'
        'ICAgICAgICAgICAgICAgZiJDb25jZXB0dWFsIHJlYWRpbmc6IHtwcm9tcHR9IiwKICAgICAgICAg'
        'ICAgICAgIGNvdW50cywKICAgICAgICAgICAgICAgIGV2aWRlbmNlLAogICAgICAgICAgICApLAog'
        'ICAgICAgICAgICBzZWxmLl9tYWtlX2JyYW5jaCgKICAgICAgICAgICAgICAgIEludGVycHJldGF0'
        'aW9uQ2F0ZWdvcnkuQ0FVU0FMLAogICAgICAgICAgICAgICAgZiJDYXVzYWwgcmVhZGluZzoge3By'
        'b21wdH0iLAogICAgICAgICAgICAgICAgY291bnRzLAogICAgICAgICAgICAgICAgZXZpZGVuY2Us'
        'CiAgICAgICAgICAgICksCiAgICAgICAgICAgIHNlbGYuX21ha2VfYnJhbmNoKAogICAgICAgICAg'
        'ICAgICAgSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5TWU1CT0xJQywKICAgICAgICAgICAgICAgIGYi'
        'U3ltYm9saWMgcmVhZGluZzoge3Byb21wdH0iLAogICAgICAgICAgICAgICAgY291bnRzLAogICAg'
        'ICAgICAgICAgICAgZXZpZGVuY2UsCiAgICAgICAgICAgICksCiAgICAgICAgICAgIHNlbGYuX21h'
        'a2VfYnJhbmNoKAogICAgICAgICAgICAgICAgSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5BRFZFUlNB'
        'UklBTCwKICAgICAgICAgICAgICAgIGYiQWR2ZXJzYXJpYWwgb3IgZGVjZXB0aXZlIHJlYWRpbmc6'
        'IHtwcm9tcHR9IiwKICAgICAgICAgICAgICAgIGNvdW50cywKICAgICAgICAgICAgICAgIGV2aWRl'
        'bmNlLAogICAgICAgICAgICApLAogICAgICAgICAgICBzZWxmLl9tYWtlX2JyYW5jaCgKICAgICAg'
        'ICAgICAgICAgIEludGVycHJldGF0aW9uQ2F0ZWdvcnkuTlVMTCwKICAgICAgICAgICAgICAgICJJ'
        'bnN1ZmZpY2llbnQgc3RydWN0dXJlIGZvciBhIGdyb3VuZGVkIGludGVycHJldGF0aW9uLiIsCiAg'
        'ICAgICAgICAgICAgICBjb3VudHMsCiAgICAgICAgICAgICAgICBldmlkZW5jZSwKICAgICAgICAg'
        'ICAgKSwKICAgICAgICBdCiAgICAgICAgcmFua2VkID0gc29ydGVkKGJyYW5jaGVzLCBrZXk9bGFt'
        'YmRhIGl0ZW06IGl0ZW0uc2NvcmUucHJpb3JpdHkoKSwgcmV2ZXJzZT1UcnVlKQogICAgICAgIHRy'
        'aW1tZWQgPSByYW5rZWRbOiBidWRnZXQuYnJhbmNoX2J1ZGdldF0KICAgICAgICBpZiBsZW4odHJp'
        'bW1lZCkgPiAxOgogICAgICAgICAgICB0cmltbWVkWy0xXSA9IHRyaW1tZWRbLTFdLm1vZGVsX2Nv'
        'cHkoCiAgICAgICAgICAgICAgICB1cGRhdGU9eyJjb25maWRlbmNlIjogbWF4KDAuMTUsIHRyaW1t'
        'ZWRbLTFdLmNvbmZpZGVuY2UpfQogICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIHRyaW1tZWQK'
        'CiAgICBkZWYgc3VtbWFyaXplX2h5cG90aGVzZXMoc2VsZiwgYnJhbmNoZXM6IGxpc3RbSW50ZXJw'
        'cmV0aXZlQnJhbmNoXSkgLT4gbGlzdFtIeXBvdGhlc2lzXToKICAgICAgICByZXR1cm4gWwogICAg'
        'ICAgICAgICBIeXBvdGhlc2lzKAogICAgICAgICAgICAgICAgbmFtZT1icmFuY2guY2F0ZWdvcnku'
        'dmFsdWUsCiAgICAgICAgICAgICAgICBkZXNjcmlwdGlvbj1icmFuY2guc3RhdGVtZW50LAogICAg'
        'ICAgICAgICAgICAgY29uZmlkZW5jZT1icmFuY2guY29uZmlkZW5jZSwKICAgICAgICAgICAgICAg'
        'IGV2aWRlbmNlPVsKICAgICAgICAgICAgICAgICAgICBFdmlkZW5jZSgKICAgICAgICAgICAgICAg'
        'ICAgICAgICAgc291cmNlPSJpbnRlcnByZXRpdmVfc3VwZXJwb3NpdGlvbiIsCiAgICAgICAgICAg'
        'ICAgICAgICAgICAgIGRldGFpbD1zdXBwb3J0LAogICAgICAgICAgICAgICAgICAgICAgICB3ZWln'
        'aHQ9YnJhbmNoLmNvbmZpZGVuY2UsCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAg'
        'ICAgICAgIGZvciBzdXBwb3J0IGluIGJyYW5jaC5zdXBwb3J0CiAgICAgICAgICAgICAgICBdLAog'
        'ICAgICAgICAgICAgICAgcHJvdmVuYW5jZT1bImludGVycHJldGF0aW9uIl0sCiAgICAgICAgICAg'
        'ICkKICAgICAgICAgICAgZm9yIGJyYW5jaCBpbiBicmFuY2hlcwogICAgICAgIF0KCiAgICBkZWYg'
        'X21ha2VfYnJhbmNoKAogICAgICAgIHNlbGYsCiAgICAgICAgY2F0ZWdvcnk6IEludGVycHJldGF0'
        'aW9uQ2F0ZWdvcnksCiAgICAgICAgc3RhdGVtZW50OiBzdHIsCiAgICAgICAgY291bnRzOiBDb3Vu'
        'dGVyW3N0cl0sCiAgICAgICAgZXZpZGVuY2U6IGxpc3RbRXZpZGVuY2VdLAogICAgKSAtPiBJbnRl'
        'cnByZXRpdmVCcmFuY2g6CiAgICAgICAgZXZpZGVuY2Vfc2NvcmUgPSBzZWxmLl9zY29yZV9ldmlk'
        'ZW5jZShjYXRlZ29yeSwgY291bnRzLCBldmlkZW5jZSkKICAgICAgICBjb2hlcmVuY2UgPSBtaW4o'
        'MS4wLCAwLjM1ICsgZXZpZGVuY2Vfc2NvcmUpCiAgICAgICAgbm92ZWx0eSA9IG1pbigKICAgICAg'
        'ICAgICAgMS4wLAogICAgICAgICAgICAwLjIKICAgICAgICAgICAgKyAoCiAgICAgICAgICAgICAg'
        'ICAwLjMKICAgICAgICAgICAgICAgIGlmIGNhdGVnb3J5CiAgICAgICAgICAgICAgICBpbiB7CiAg'
        'ICAgICAgICAgICAgICAgICAgSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5TWU1CT0xJQywKICAgICAg'
        'ICAgICAgICAgICAgICBJbnRlcnByZXRhdGlvbkNhdGVnb3J5LkFEVkVSU0FSSUFMLAogICAgICAg'
        'ICAgICAgICAgfQogICAgICAgICAgICAgICAgZWxzZSAwLjEKICAgICAgICAgICAgKSwKICAgICAg'
        'ICApCiAgICAgICAgdXNlZnVsbmVzcyA9IG1pbigxLjAsIDAuMyArIGV2aWRlbmNlX3Njb3JlKQog'
        'ICAgICAgIHVuY2VydGFpbnR5X3ZhbHVlID0gMS4wIC0gYWJzKDAuNSAtIGV2aWRlbmNlX3Njb3Jl'
        'KQogICAgICAgIGNvbXB1dGVfY29zdCA9IDEuMCBpZiBjYXRlZ29yeSA9PSBJbnRlcnByZXRhdGlv'
        'bkNhdGVnb3J5LkxJVEVSQUwgZWxzZSAxLjIKICAgICAgICBjb250cmFkaWN0aW9uX3BlbmFsdHkg'
        'PSAwLjEgaWYgY2F0ZWdvcnkgPT0gSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5OVUxMIGVsc2UgMC4w'
        'CiAgICAgICAgcGxhdXNpYmlsaXR5ID0gbWF4KDAuMDUsIGNvaGVyZW5jZSAtIGNvbnRyYWRpY3Rp'
        'b25fcGVuYWx0eSkKICAgICAgICBjb25maWRlbmNlID0gcm91bmQoKHBsYXVzaWJpbGl0eSArIHVz'
        'ZWZ1bG5lc3MpIC8gMi4wLCA0KQogICAgICAgIHN1cHBvcnQgPSBbaXRlbS5kZXRhaWwgZm9yIGl0'
        'ZW0gaW4gZXZpZGVuY2UgaWYgaXRlbS53ZWlnaHQgPiAwLjRdCiAgICAgICAgaWYgbm90IHN1cHBv'
        'cnQ6CiAgICAgICAgICAgIHN1cHBvcnQgPSBbZiJrZXl3b3JkLW1hdGNoPXtldmlkZW5jZV9zY29y'
        'ZTouMmZ9Il0KICAgICAgICByZXR1cm4gSW50ZXJwcmV0aXZlQnJhbmNoKAogICAgICAgICAgICBj'
        'YXRlZ29yeT1jYXRlZ29yeSwKICAgICAgICAgICAgc3RhdGVtZW50PXN0YXRlbWVudCwKICAgICAg'
        'ICAgICAgc2NvcmU9QnJhbmNoU2NvcmUoCiAgICAgICAgICAgICAgICBwbGF1c2liaWxpdHk9cm91'
        'bmQocGxhdXNpYmlsaXR5LCA0KSwKICAgICAgICAgICAgICAgIGluZm9ybWF0aW9uX2dhaW49cm91'
        'bmQobm92ZWx0eSwgNCksCiAgICAgICAgICAgICAgICB1c2VmdWxuZXNzPXJvdW5kKHVzZWZ1bG5l'
        'c3MsIDQpLAogICAgICAgICAgICAgICAgdW5jZXJ0YWludHlfdmFsdWU9cm91bmQodW5jZXJ0YWlu'
        'dHlfdmFsdWUsIDQpLAogICAgICAgICAgICAgICAgY29tcHV0ZV9jb3N0PWNvbXB1dGVfY29zdCwK'
        'ICAgICAgICAgICAgKSwKICAgICAgICAgICAgc3VwcG9ydD1zdXBwb3J0LAogICAgICAgICAgICBj'
        'b25maWRlbmNlPWNvbmZpZGVuY2UsCiAgICAgICAgKQoKICAgIGRlZiBfc2NvcmVfZXZpZGVuY2Uo'
        'CiAgICAgICAgc2VsZiwKICAgICAgICBjYXRlZ29yeTogSW50ZXJwcmV0YXRpb25DYXRlZ29yeSwK'
        'ICAgICAgICBjb3VudHM6IENvdW50ZXJbc3RyXSwKICAgICAgICBldmlkZW5jZTogbGlzdFtFdmlk'
        'ZW5jZV0sCiAgICApIC0+IGZsb2F0OgogICAgICAgIHRva2VuX2JvbnVzID0gMC4wCiAgICAgICAg'
        'aWYgY2F0ZWdvcnkgPT0gSW50ZXJwcmV0YXRpb25DYXRlZ29yeS5BQlNUUkFDVF9DT05DUkVURV9B'
        'TkFMT0dZOgogICAgICAgICAgICB0b2tlbl9ib251cyArPSAoCiAgICAgICAgICAgICAgICAwLjMg'
        'aWYgYW55KGtleXdvcmQgaW4gY291bnRzIGZvciBrZXl3b3JkIGluIHNlbGYuYW5hbG9neV9rZXl3'
        'b3JkcykgZWxzZSAwLjA1CiAgICAgICAgICAgICkKICAgICAgICBpZiBjYXRlZ29yeSA9PSBJbnRl'
        'cnByZXRhdGlvbkNhdGVnb3J5LlNZTUJPTElDOgogICAgICAgICAgICB0b2tlbl9ib251cyArPSAo'
        'CiAgICAgICAgICAgICAgICAwLjI1IGlmIGFueShrZXl3b3JkIGluIGNvdW50cyBmb3Iga2V5d29y'
        'ZCBpbiBzZWxmLnN5bWJvbGljX2tleXdvcmRzKSBlbHNlIDAuMDUKICAgICAgICAgICAgKQogICAg'
        'ICAgIGlmIGNhdGVnb3J5ID09IEludGVycHJldGF0aW9uQ2F0ZWdvcnkuQ0FVU0FMOgogICAgICAg'
        'ICAgICB0b2tlbl9ib251cyArPSAoCiAgICAgICAgICAgICAgICAwLjI1IGlmIGFueShrZXl3b3Jk'
        'IGluIGNvdW50cyBmb3Iga2V5d29yZCBpbiBzZWxmLmNhdXNhbF9rZXl3b3JkcykgZWxzZSAwLjA1'
        'CiAgICAgICAgICAgICkKICAgICAgICBpZiBjYXRlZ29yeSA9PSBJbnRlcnByZXRhdGlvbkNhdGVn'
        'b3J5LlRFTVBPUkFMOgogICAgICAgICAgICB0b2tlbl9ib251cyArPSAoCiAgICAgICAgICAgICAg'
        'ICAwLjI1IGlmIGFueShrZXl3b3JkIGluIGNvdW50cyBmb3Iga2V5d29yZCBpbiBzZWxmLnRlbXBv'
        'cmFsX2tleXdvcmRzKSBlbHNlIDAuMDUKICAgICAgICAgICAgKQogICAgICAgIGV2aWRlbmNlX2Jv'
        'bnVzID0gc3VtKGl0ZW0ud2VpZ2h0IGZvciBpdGVtIGluIGV2aWRlbmNlKSAvIG1heCgxLCBsZW4o'
        'ZXZpZGVuY2UpKQogICAgICAgIGxpdGVyYWxfYm9udXMgPSAwLjIgaWYgY2F0ZWdvcnkgPT0gSW50'
        'ZXJwcmV0YXRpb25DYXRlZ29yeS5MSVRFUkFMIGVsc2UgMC4wCiAgICAgICAgcmV0dXJuIG1pbigx'
        'LjAsIHRva2VuX2JvbnVzICsgKDAuNSAqIGV2aWRlbmNlX2JvbnVzKSArIGxpdGVyYWxfYm9udXMp'
        'Cg=='
    ),
    'olympus/core/mapping.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBTdHJ1Y3R1cmFsQW5h'
        'bG9neToKICAgIHNvdXJjZV9mZWF0dXJlczogbGlzdFtzdHJdCiAgICBtYXBwZWRfZmVhdHVyZXM6'
        'IGxpc3Rbc3RyXQogICAgYWxpZ25tZW50X3Njb3JlOiBmbG9hdAogICAgZGlmZmVyZW5jZXM6IGxp'
        'c3Rbc3RyXQogICAgY29uZmlkZW5jZTogZmxvYXQKICAgIGxpdGVyYWw6IGJvb2wKCgpjbGFzcyBB'
        'YnN0cmFjdENvbmNyZXRlTWFwcGVyOgogICAgZGVmIG1hcChzZWxmLCBzb3VyY2U6IHN0ciwgdGFy'
        'Z2V0OiBzdHIpIC0+IFN0cnVjdHVyYWxBbmFsb2d5OgogICAgICAgIHNvdXJjZV9mZWF0dXJlcyA9'
        'IHNvdXJjZS5sb3dlcigpLnJlcGxhY2UoIiwiLCAiICIpLnNwbGl0KCkKICAgICAgICB0YXJnZXRf'
        'ZmVhdHVyZXMgPSB0YXJnZXQubG93ZXIoKS5yZXBsYWNlKCIsIiwgIiAiKS5zcGxpdCgpCiAgICAg'
        'ICAgb3ZlcmxhcCA9IHNvcnRlZChzZXQoc291cmNlX2ZlYXR1cmVzKSAmIHNldCh0YXJnZXRfZmVh'
        'dHVyZXMpKQogICAgICAgIGRpZmZlcmVuY2VzID0gc29ydGVkKHNldChzb3VyY2VfZmVhdHVyZXMp'
        'IF4gc2V0KHRhcmdldF9mZWF0dXJlcykpCiAgICAgICAgZGVub21pbmF0b3IgPSBtYXgoMSwgbGVu'
        'KHNldChzb3VyY2VfZmVhdHVyZXMpIHwgc2V0KHRhcmdldF9mZWF0dXJlcykpKQogICAgICAgIGFs'
        'aWdubWVudCA9IGxlbihvdmVybGFwKSAvIGRlbm9taW5hdG9yCiAgICAgICAgY29uZmlkZW5jZSA9'
        'IG1pbigwLjk1LCAwLjM1ICsgYWxpZ25tZW50KQogICAgICAgIGxpdGVyYWwgPSBhbGlnbm1lbnQg'
        'PiAwLjQ1CiAgICAgICAgcmV0dXJuIFN0cnVjdHVyYWxBbmFsb2d5KAogICAgICAgICAgICBzb3Vy'
        'Y2VfZmVhdHVyZXM9c291cmNlX2ZlYXR1cmVzLAogICAgICAgICAgICBtYXBwZWRfZmVhdHVyZXM9'
        'dGFyZ2V0X2ZlYXR1cmVzLAogICAgICAgICAgICBhbGlnbm1lbnRfc2NvcmU9cm91bmQoYWxpZ25t'
        'ZW50LCA0KSwKICAgICAgICAgICAgZGlmZmVyZW5jZXM9ZGlmZmVyZW5jZXMsCiAgICAgICAgICAg'
        'IGNvbmZpZGVuY2U9cm91bmQoY29uZmlkZW5jZSwgNCksCiAgICAgICAgICAgIGxpdGVyYWw9bGl0'
        'ZXJhbCwKICAgICAgICApCg=='
    ),
    'olympus/core/modalities.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG1hdGgKZnJvbSBlbnVt'
        'IGltcG9ydCBTdHJFbnVtCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gcHlkYW50aWMgaW1w'
        'b3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoK'
        'CmNsYXNzIE1vZGFsaXR5S2luZChTdHJFbnVtKToKICAgIFRFWFQgPSAidGV4dCIKICAgIENPREUg'
        'PSAiY29kZSIKICAgIERPQ1VNRU5UID0gImRvY3VtZW50IgogICAgSU1BR0UgPSAiaW1hZ2UiCiAg'
        'ICBBVURJTyA9ICJhdWRpbyIKICAgIFZJREVPID0gInZpZGVvIgogICAgU1RSVUNUVVJFRCA9ICJz'
        'dHJ1Y3R1cmVkIgogICAgVE9PTF9PQlNFUlZBVElPTiA9ICJ0b29sX29ic2VydmF0aW9uIgogICAg'
        'RU5WSVJPTk1FTlRfU1RBVEUgPSAiZW52aXJvbm1lbnRfc3RhdGUiCiAgICBSRVBPU0lUT1JZX1NU'
        'QVRFID0gInJlcG9zaXRvcnlfc3RhdGUiCgoKY2xhc3MgQmFzZUFydGlmYWN0KFN0cmljdE1vZGVs'
        'KToKICAgIG1vZGFsaXR5OiBNb2RhbGl0eUtpbmQKICAgIGNvbnRlbnQ6IEFueQogICAgbWV0YWRh'
        'dGE6IGRpY3Rbc3RyLCBBbnldID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCgoKY2xhc3Mg'
        'VGV4dEFydGlmYWN0KEJhc2VBcnRpZmFjdCk6CiAgICBtb2RhbGl0eTogTW9kYWxpdHlLaW5kID0g'
        'TW9kYWxpdHlLaW5kLlRFWFQKICAgIGNvbnRlbnQ6IHN0cgoKCmNsYXNzIENvZGVBcnRpZmFjdChC'
        'YXNlQXJ0aWZhY3QpOgogICAgbW9kYWxpdHk6IE1vZGFsaXR5S2luZCA9IE1vZGFsaXR5S2luZC5D'
        'T0RFCiAgICBjb250ZW50OiBzdHIKICAgIGxhbmd1YWdlOiBzdHIgPSAicHl0aG9uIgoKCmNsYXNz'
        'IERvY3VtZW50QXJ0aWZhY3QoQmFzZUFydGlmYWN0KToKICAgIG1vZGFsaXR5OiBNb2RhbGl0eUtp'
        'bmQgPSBNb2RhbGl0eUtpbmQuRE9DVU1FTlQKICAgIGNvbnRlbnQ6IHN0cgogICAgdGl0bGU6IHN0'
        'ciA9ICJ1bnRpdGxlZCIKCgpjbGFzcyBJbWFnZUFydGlmYWN0KEJhc2VBcnRpZmFjdCk6CiAgICBt'
        'b2RhbGl0eTogTW9kYWxpdHlLaW5kID0gTW9kYWxpdHlLaW5kLklNQUdFCiAgICBjb250ZW50OiBk'
        'aWN0W3N0ciwgZmxvYXQgfCBpbnQgfCBzdHJdCgoKY2xhc3MgQXVkaW9BcnRpZmFjdChCYXNlQXJ0'
        'aWZhY3QpOgogICAgbW9kYWxpdHk6IE1vZGFsaXR5S2luZCA9IE1vZGFsaXR5S2luZC5BVURJTwog'
        'ICAgY29udGVudDogZGljdFtzdHIsIGZsb2F0IHwgaW50IHwgc3RyXQoKCmNsYXNzIFZpZGVvQXJ0'
        'aWZhY3QoQmFzZUFydGlmYWN0KToKICAgIG1vZGFsaXR5OiBNb2RhbGl0eUtpbmQgPSBNb2RhbGl0'
        'eUtpbmQuVklERU8KICAgIGNvbnRlbnQ6IGRpY3Rbc3RyLCBmbG9hdCB8IGludCB8IHN0cl0KCgpj'
        'bGFzcyBTdHJ1Y3R1cmVkQXJ0aWZhY3QoQmFzZUFydGlmYWN0KToKICAgIG1vZGFsaXR5OiBNb2Rh'
        'bGl0eUtpbmQgPSBNb2RhbGl0eUtpbmQuU1RSVUNUVVJFRAogICAgY29udGVudDogZGljdFtzdHIs'
        'IEFueV0KCgpjbGFzcyBUb29sT2JzZXJ2YXRpb25BcnRpZmFjdChCYXNlQXJ0aWZhY3QpOgogICAg'
        'bW9kYWxpdHk6IE1vZGFsaXR5S2luZCA9IE1vZGFsaXR5S2luZC5UT09MX09CU0VSVkFUSU9OCiAg'
        'ICBjb250ZW50OiBkaWN0W3N0ciwgQW55XQoKCmNsYXNzIEVudmlyb25tZW50U3RhdGVBcnRpZmFj'
        'dChCYXNlQXJ0aWZhY3QpOgogICAgbW9kYWxpdHk6IE1vZGFsaXR5S2luZCA9IE1vZGFsaXR5S2lu'
        'ZC5FTlZJUk9OTUVOVF9TVEFURQogICAgY29udGVudDogZGljdFtzdHIsIEFueV0KCgpjbGFzcyBS'
        'ZXBvc2l0b3J5U3RhdGVBcnRpZmFjdChCYXNlQXJ0aWZhY3QpOgogICAgbW9kYWxpdHk6IE1vZGFs'
        'aXR5S2luZCA9IE1vZGFsaXR5S2luZC5SRVBPU0lUT1JZX1NUQVRFCiAgICBjb250ZW50OiBkaWN0'
        'W3N0ciwgQW55XQoKCkFydGlmYWN0ID0gKAogICAgVGV4dEFydGlmYWN0CiAgICB8IENvZGVBcnRp'
        'ZmFjdAogICAgfCBEb2N1bWVudEFydGlmYWN0CiAgICB8IEltYWdlQXJ0aWZhY3QKICAgIHwgQXVk'
        'aW9BcnRpZmFjdAogICAgfCBWaWRlb0FydGlmYWN0CiAgICB8IFN0cnVjdHVyZWRBcnRpZmFjdAog'
        'ICAgfCBUb29sT2JzZXJ2YXRpb25BcnRpZmFjdAogICAgfCBFbnZpcm9ubWVudFN0YXRlQXJ0aWZh'
        'Y3QKICAgIHwgUmVwb3NpdG9yeVN0YXRlQXJ0aWZhY3QKKQoKCmNsYXNzIEVuY29kZWRBcnRpZmFj'
        'dChTdHJpY3RNb2RlbCk6CiAgICBtb2RhbGl0eTogTW9kYWxpdHlLaW5kCiAgICB2ZWN0b3I6IGxp'
        'c3RbZmxvYXRdCiAgICBzdW1tYXJ5OiBzdHIKCgpjbGFzcyBSZWZlcmVuY2VFbmNvZGVyOgogICAg'
        'IiIiQSBkZXRlcm1pbmlzdGljIGxvY2FsIGVuY29kZXIgdXNlZCB3aGVuIG5vIHByZXRyYWluZWQg'
        'bW9kZWwgaXMgcHJlc2VudC4iIiIKCiAgICBkZWYgZW5jb2RlKHNlbGYsIGFydGlmYWN0OiBBcnRp'
        'ZmFjdCkgLT4gRW5jb2RlZEFydGlmYWN0OgogICAgICAgIGlmIGlzaW5zdGFuY2UoYXJ0aWZhY3Qu'
        'Y29udGVudCwgc3RyKToKICAgICAgICAgICAgdG9rZW5zID0gYXJ0aWZhY3QuY29udGVudC5zcGxp'
        'dCgpCiAgICAgICAgICAgIGxlbmd0aCA9IGxlbihhcnRpZmFjdC5jb250ZW50KQogICAgICAgICAg'
        'ICBhdmdfd29yZCA9IHN1bShsZW4odG9rZW4pIGZvciB0b2tlbiBpbiB0b2tlbnMpIC8gbWF4KDEs'
        'IGxlbih0b2tlbnMpKQogICAgICAgICAgICB2ZWN0b3IgPSBbCiAgICAgICAgICAgICAgICBmbG9h'
        'dChsZW4odG9rZW5zKSksCiAgICAgICAgICAgICAgICBmbG9hdChsZW5ndGgpLAogICAgICAgICAg'
        'ICAgICAgZmxvYXQoYXZnX3dvcmQpLAogICAgICAgICAgICAgICAgZmxvYXQoc3VtKGNoYXIuaXNk'
        'aWdpdCgpIGZvciBjaGFyIGluIGFydGlmYWN0LmNvbnRlbnQpKSwKICAgICAgICAgICAgICAgIGZs'
        'b2F0KHN1bShjaGFyLmlzdXBwZXIoKSBmb3IgY2hhciBpbiBhcnRpZmFjdC5jb250ZW50KSksCiAg'
        'ICAgICAgICAgIF0KICAgICAgICAgICAgc3VtbWFyeSA9IGFydGlmYWN0LmNvbnRlbnRbOjEyMF0K'
        'ICAgICAgICBlbGlmIGlzaW5zdGFuY2UoYXJ0aWZhY3QuY29udGVudCwgZGljdCk6CiAgICAgICAg'
        'ICAgIG51bWVyaWNfdmFsdWVzID0gWwogICAgICAgICAgICAgICAgZmxvYXQodmFsdWUpCiAgICAg'
        'ICAgICAgICAgICBmb3IgdmFsdWUgaW4gYXJ0aWZhY3QuY29udGVudC52YWx1ZXMoKQogICAgICAg'
        'ICAgICAgICAgaWYgaXNpbnN0YW5jZSh2YWx1ZSwgaW50IHwgZmxvYXQpCiAgICAgICAgICAgIF0K'
        'ICAgICAgICAgICAgdmVjdG9yID0gbnVtZXJpY192YWx1ZXNbOjhdCiAgICAgICAgICAgIHN1bW1h'
        'cnkgPSAiLCAiLmpvaW4oZiJ7a2V5fT17dmFsdWV9IiBmb3Iga2V5LCB2YWx1ZSBpbiBhcnRpZmFj'
        'dC5jb250ZW50Lml0ZW1zKCkpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgdmVjdG9yID0gWzAu'
        'MF0KICAgICAgICAgICAgc3VtbWFyeSA9IHN0cihhcnRpZmFjdC5jb250ZW50KQoKICAgICAgICBu'
        'b3JtID0gbWF0aC5zcXJ0KHN1bSh2YWx1ZSAqIHZhbHVlIGZvciB2YWx1ZSBpbiB2ZWN0b3IpKSBv'
        'ciAxLjAKICAgICAgICBub3JtYWxpemVkID0gW3JvdW5kKHZhbHVlIC8gbm9ybSwgNikgZm9yIHZh'
        'bHVlIGluIHZlY3Rvcl0KICAgICAgICByZXR1cm4gRW5jb2RlZEFydGlmYWN0KG1vZGFsaXR5PWFy'
        'dGlmYWN0Lm1vZGFsaXR5LCB2ZWN0b3I9bm9ybWFsaXplZCwgc3VtbWFyeT1zdW1tYXJ5KQo='
    ),
    'olympus/core/observability.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHRpbWUgaW1wb3J0IHBlcmZfY291bnRlcgpmcm9tIHR5'
        'cGluZyBpbXBvcnQgQW55CgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBUcmFjZUV2ZW50'
        'OgogICAgbmFtZTogc3RyCiAgICBzdGFydGVkX2F0OiBmbG9hdAogICAgZmluaXNoZWRfYXQ6IGZs'
        'b2F0CiAgICBtZXRhZGF0YTogZGljdFtzdHIsIEFueV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9'
        'ZGljdCkKCgpjbGFzcyBUcmFjZVJlY29yZGVyOgogICAgZGVmIF9faW5pdF9fKHNlbGYpIC0+IE5v'
        'bmU6CiAgICAgICAgc2VsZi5ldmVudHM6IGxpc3RbVHJhY2VFdmVudF0gPSBbXQoKICAgIGRlZiBy'
        'ZWNvcmQoc2VsZiwgbmFtZTogc3RyLCBjYWxsYmFjazogQW55LCAqKm1ldGFkYXRhOiBBbnkpIC0+'
        'IEFueToKICAgICAgICBzdGFydGVkX2F0ID0gcGVyZl9jb3VudGVyKCkKICAgICAgICByZXN1bHQg'
        'PSBjYWxsYmFjaygpCiAgICAgICAgZmluaXNoZWRfYXQgPSBwZXJmX2NvdW50ZXIoKQogICAgICAg'
        'IHNlbGYuZXZlbnRzLmFwcGVuZCgKICAgICAgICAgICAgVHJhY2VFdmVudChuYW1lPW5hbWUsIHN0'
        'YXJ0ZWRfYXQ9c3RhcnRlZF9hdCwgZmluaXNoZWRfYXQ9ZmluaXNoZWRfYXQsIG1ldGFkYXRhPW1l'
        'dGFkYXRhKQogICAgICAgICkKICAgICAgICByZXR1cm4gcmVzdWx0Cg=='
    ),
    'olympus/core/outcome.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBweWRhbnRpYyBpbXBvcnQg'
        'RmllbGQKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCgoKY2xh'
        'c3MgT3V0Y29tZVF1YWxpdHkoU3RyaWN0TW9kZWwpOgogICAgY29ycmVjdG5lc3M6IGZsb2F0ID0g'
        'RmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICBpbmZvcm1hdGlvbl9nYWluOiBmbG9hdCA9IEZpZWxk'
        'KGdlPTAuMCwgbGU9MS4wKQogICAgdXNlZnVsbmVzczogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxl'
        'PTEuMCkKICAgIG5vdmVsdHk6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICByaXNr'
        'OiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgcmV2ZXJzaWJpbGl0eTogZmxvYXQg'
        'PSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIGNvc3Q6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBs'
        'ZT0xLjApCiAgICBsYXRlbmN5OiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgdW5j'
        'ZXJ0YWludHlfcmVzb2x1dGlvbjogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKCgpjbGFz'
        'cyBPdXRjb21lUXVhbGl0eVByZWRpY3RvcjoKICAgIGRlZiBwcmVkaWN0KHNlbGYsIGNhbmRpZGF0'
        'ZTogc3RyLCBldmlkZW5jZV9jb3VudDogaW50LCB0b29sX2NvdW50OiBpbnQpIC0+IE91dGNvbWVR'
        'dWFsaXR5OgogICAgICAgIGNvcnJlY3RuZXNzID0gbWluKDEuMCwgMC4zICsgKDAuMTUgKiBldmlk'
        'ZW5jZV9jb3VudCkpCiAgICAgICAgaW5mb3JtYXRpb25fZ2FpbiA9IG1pbigxLjAsIDAuMiArICgw'
        'LjEgKiB0b29sX2NvdW50KSkKICAgICAgICB1c2VmdWxuZXNzID0gbWluKDEuMCwgMC4zNSArICgw'
        'LjA4ICogZXZpZGVuY2VfY291bnQpKQogICAgICAgIG5vdmVsdHkgPSAwLjY1IGlmICJub3ZlbCIg'
        'aW4gY2FuZGlkYXRlLmxvd2VyKCkgZWxzZSAwLjQKICAgICAgICByaXNrID0gMC43IGlmICJkZWxl'
        'dGUiIGluIGNhbmRpZGF0ZS5sb3dlcigpIGVsc2UgMC4yCiAgICAgICAgcmV2ZXJzaWJpbGl0eSA9'
        'IDAuMiBpZiByaXNrID4gMC41IGVsc2UgMC44CiAgICAgICAgY29zdCA9IG1pbigxLjAsIDAuMSAr'
        'ICgwLjA1ICogdG9vbF9jb3VudCkpCiAgICAgICAgbGF0ZW5jeSA9IG1pbigxLjAsIDAuMTUgKyAo'
        'MC4wOCAqIHRvb2xfY291bnQpKQogICAgICAgIHVuY2VydGFpbnR5X3Jlc29sdXRpb24gPSBtaW4o'
        'MS4wLCAoY29ycmVjdG5lc3MgKyBpbmZvcm1hdGlvbl9nYWluKSAvIDIuMCkKICAgICAgICByZXR1'
        'cm4gT3V0Y29tZVF1YWxpdHkoCiAgICAgICAgICAgIGNvcnJlY3RuZXNzPXJvdW5kKGNvcnJlY3Ru'
        'ZXNzLCA0KSwKICAgICAgICAgICAgaW5mb3JtYXRpb25fZ2Fpbj1yb3VuZChpbmZvcm1hdGlvbl9n'
        'YWluLCA0KSwKICAgICAgICAgICAgdXNlZnVsbmVzcz1yb3VuZCh1c2VmdWxuZXNzLCA0KSwKICAg'
        'ICAgICAgICAgbm92ZWx0eT1yb3VuZChub3ZlbHR5LCA0KSwKICAgICAgICAgICAgcmlzaz1yb3Vu'
        'ZChyaXNrLCA0KSwKICAgICAgICAgICAgcmV2ZXJzaWJpbGl0eT1yb3VuZChyZXZlcnNpYmlsaXR5'
        'LCA0KSwKICAgICAgICAgICAgY29zdD1yb3VuZChjb3N0LCA0KSwKICAgICAgICAgICAgbGF0ZW5j'
        'eT1yb3VuZChsYXRlbmN5LCA0KSwKICAgICAgICAgICAgdW5jZXJ0YWludHlfcmVzb2x1dGlvbj1y'
        'b3VuZCh1bmNlcnRhaW50eV9yZXNvbHV0aW9uLCA0KSwKICAgICAgICApCg=='
    ),
    'olympus/core/perspective.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBlbnVtIGltcG9ydCBTdHJF'
        'bnVtCgpmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZAoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1h'
        'cyBpbXBvcnQgU3RyaWN0TW9kZWwKCgpjbGFzcyBQZXJzcGVjdGl2ZUtpbmQoU3RyRW51bSk6CiAg'
        'ICBDT05DRVBUVUFMID0gImNvbmNlcHR1YWwiCiAgICBTWU1CT0xJQyA9ICJzeW1ib2xpYyIKICAg'
        'IEVNT1RJT05BTCA9ICJlbW90aW9uYWwiCiAgICBUSEVPUkVUSUNBTCA9ICJ0aGVvcmV0aWNhbCIK'
        'ICAgIE5PTkxJVEVSQUwgPSAibm9ubGl0ZXJhbCIKCgpjbGFzcyBQZXJzcGVjdGl2ZShTdHJpY3RN'
        'b2RlbCk6CiAgICBraW5kOiBQZXJzcGVjdGl2ZUtpbmQKICAgIGludGVycHJldGF0aW9uOiBzdHIK'
        'ICAgIGV2aWRlbmNlX3NwYW5zOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlz'
        'dCkKICAgIGNvbmZpZGVuY2U6IGZsb2F0CiAgICBmYWN0dWFsOiBib29sID0gRmFsc2UKCgpjbGFz'
        'cyBBYnN0cmFjdFBlcnNwZWN0aXZlR2VuZXJhdG9yOgogICAgdGVtcGxhdGVzID0gewogICAgICAg'
        'IFBlcnNwZWN0aXZlS2luZC5DT05DRVBUVUFMOiAiQ29uY2VwdHVhbCBmcmFtaW5nOiB7dGV4dH0i'
        'LAogICAgICAgIFBlcnNwZWN0aXZlS2luZC5TWU1CT0xJQzogIlN5bWJvbGljIGZyYW1pbmc6IHt0'
        'ZXh0fSIsCiAgICAgICAgUGVyc3BlY3RpdmVLaW5kLkVNT1RJT05BTDogIkVtb3Rpb25hbCBzdWJ0'
        'ZXh0OiB7dGV4dH0iLAogICAgICAgIFBlcnNwZWN0aXZlS2luZC5USEVPUkVUSUNBTDogIlRoZW9y'
        'ZXRpY2FsIGZyYW1pbmc6IHt0ZXh0fSIsCiAgICAgICAgUGVyc3BlY3RpdmVLaW5kLk5PTkxJVEVS'
        'QUw6ICJOb25saXRlcmFsIHBlcnNwZWN0aXZlOiB7dGV4dH0iLAogICAgfQoKICAgIGRlZiBnZW5l'
        'cmF0ZShzZWxmLCB0ZXh0OiBzdHIpIC0+IGxpc3RbUGVyc3BlY3RpdmVdOgogICAgICAgIHNwYW5z'
        'ID0gW3NlZ21lbnQuc3RyaXAoKSBmb3Igc2VnbWVudCBpbiB0ZXh0LnNwbGl0KCIsIikgaWYgc2Vn'
        'bWVudC5zdHJpcCgpXSBvciBbdGV4dF0KICAgICAgICByZXN1bHRzOiBsaXN0W1BlcnNwZWN0aXZl'
        'XSA9IFtdCiAgICAgICAgZm9yIGluZGV4LCBraW5kIGluIGVudW1lcmF0ZShQZXJzcGVjdGl2ZUtp'
        'bmQpOgogICAgICAgICAgICBjb25maWRlbmNlID0gcm91bmQobWF4KDAuMjUsIDAuNzIgLSAoaW5k'
        'ZXggKiAwLjA4KSksIDQpCiAgICAgICAgICAgIHJlc3VsdHMuYXBwZW5kKAogICAgICAgICAgICAg'
        'ICAgUGVyc3BlY3RpdmUoCiAgICAgICAgICAgICAgICAgICAga2luZD1raW5kLAogICAgICAgICAg'
        'ICAgICAgICAgIGludGVycHJldGF0aW9uPXNlbGYudGVtcGxhdGVzW2tpbmRdLmZvcm1hdCh0ZXh0'
        'PXRleHQpLAogICAgICAgICAgICAgICAgICAgIGV2aWRlbmNlX3NwYW5zPXNwYW5zWzoyXSwKICAg'
        'ICAgICAgICAgICAgICAgICBjb25maWRlbmNlPWNvbmZpZGVuY2UsCiAgICAgICAgICAgICAgICAg'
        'ICAgZmFjdHVhbD1GYWxzZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgICAg'
        'IHJldHVybiByZXN1bHRzCg=='
    ),
    'olympus/core/prediction.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yLCBubgoK'
        'CmNsYXNzIFRpbnlKRVBBKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgaW5wdXRf'
        'ZGltOiBpbnQsIGxhdGVudF9kaW06IGludCkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5p'
        'dF9fKCkKICAgICAgICBzZWxmLmNvbnRleHRfZW5jb2RlciA9IG5uLlNlcXVlbnRpYWwoCiAgICAg'
        'ICAgICAgIG5uLkxpbmVhcihpbnB1dF9kaW0sIDMyKSwKICAgICAgICAgICAgbm4uUmVMVSgpLAog'
        'ICAgICAgICAgICBubi5MaW5lYXIoMzIsIGxhdGVudF9kaW0pLAogICAgICAgICkKICAgICAgICBz'
        'ZWxmLnRhcmdldF9lbmNvZGVyID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFy'
        'KGlucHV0X2RpbSwgMzIpLAogICAgICAgICAgICBubi5SZUxVKCksCiAgICAgICAgICAgIG5uLkxp'
        'bmVhcigzMiwgbGF0ZW50X2RpbSksCiAgICAgICAgKQogICAgICAgIHNlbGYucHJlZGljdG9yID0g'
        'bm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKGxhdGVudF9kaW0sIDMyKSwKICAg'
        'ICAgICAgICAgbm4uUmVMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIoMzIsIGxhdGVudF9kaW0p'
        'LAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBjb250ZXh0OiBUZW5zb3IsIHRhcmdl'
        'dDogVGVuc29yKSAtPiB0dXBsZVtUZW5zb3IsIFRlbnNvcl06CiAgICAgICAgY29udGV4dF9sYXRl'
        'bnQgPSBzZWxmLmNvbnRleHRfZW5jb2Rlcihjb250ZXh0KQogICAgICAgIHdpdGggdG9yY2gubm9f'
        'Z3JhZCgpOgogICAgICAgICAgICB0YXJnZXRfbGF0ZW50ID0gc2VsZi50YXJnZXRfZW5jb2Rlcih0'
        'YXJnZXQpCiAgICAgICAgcHJlZGljdGVkID0gc2VsZi5wcmVkaWN0b3IoY29udGV4dF9sYXRlbnQp'
        'CiAgICAgICAgcmV0dXJuIHByZWRpY3RlZCwgdGFyZ2V0X2xhdGVudAoKCkBkYXRhY2xhc3Moc2xv'
        'dHM9VHJ1ZSkKY2xhc3MgSkVQQVRyYWluaW5nUmVzdWx0OgogICAgZmluYWxfbG9zczogZmxvYXQK'
        'ICAgIHZhcmlhbmNlX3JlZ3VsYXJpemF0aW9uOiBmbG9hdAogICAgcm9sbG91dF9lcnJvcjogZmxv'
        'YXQKCgpkZWYgc3ludGhldGljX3NlcXVlbmNlX2JhdGNoKGJhdGNoX3NpemU6IGludCA9IDMyLCBz'
        'ZXF1ZW5jZV9kaW06IGludCA9IDQpIC0+IHR1cGxlW1RlbnNvciwgVGVuc29yXToKICAgIGlmIGJh'
        'dGNoX3NpemUgPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJhdGNoX3NpemUgbXVzdCBi'
        'ZSBhdCBsZWFzdCAyIikKICAgIGlmIHNlcXVlbmNlX2RpbSA8IDE6CiAgICAgICAgcmFpc2UgVmFs'
        'dWVFcnJvcigic2VxdWVuY2VfZGltIG11c3QgYmUgcG9zaXRpdmUiKQogICAgYmFzZSA9IHRvcmNo'
        'LmxpbnNwYWNlKDAuMCwgMS4wLCBzdGVwcz1iYXRjaF9zaXplKS51bnNxdWVlemUoMSkKICAgIGNv'
        'bnRleHQgPSB0b3JjaC5jYXQoCiAgICAgICAgW2Jhc2UgKyBvZmZzZXQgZm9yIG9mZnNldCBpbiB0'
        'b3JjaC5saW5zcGFjZSgwLjAsIDAuMywgc3RlcHM9c2VxdWVuY2VfZGltKV0sCiAgICAgICAgZGlt'
        'PTEsCiAgICApCiAgICB0YXJnZXQgPSBjb250ZXh0ICsgMC4xNQogICAgcmV0dXJuIGNvbnRleHQs'
        'IHRhcmdldAoKCmRlZiB0cmFpbl9qZXBhKGVwb2NoczogaW50ID0gMTYwLCBsYXRlbnRfZGltOiBp'
        'bnQgPSA4LCBzZWVkOiBpbnQgPSA3KSAtPiBKRVBBVHJhaW5pbmdSZXN1bHQ6CiAgICBpZiBlcG9j'
        'aHMgPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImVwb2NocyBtdXN0IGJlIHBvc2l0aXZl'
        'IikKICAgIGlmIGxhdGVudF9kaW0gPCAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxhdGVu'
        'dF9kaW0gbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICB0b3JjaC5tYW51YWxfc2VlZChzZWVkKQogICAg'
        'Y29udGV4dCwgdGFyZ2V0ID0gc3ludGhldGljX3NlcXVlbmNlX2JhdGNoKCkKICAgIG1vZGVsID0g'
        'VGlueUpFUEEoaW5wdXRfZGltPWNvbnRleHQuc2hhcGVbMV0sIGxhdGVudF9kaW09bGF0ZW50X2Rp'
        'bSkKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW0obW9kZWwucGFyYW1ldGVycygpLCBs'
        'cj0wLjAyKQogICAgbXNlID0gbm4uTVNFTG9zcygpCgogICAgZm9yIF8gaW4gcmFuZ2UoZXBvY2hz'
        'KToKICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICBwcmVkaWN0aW9uLCB0YXJn'
        'ZXRfbGF0ZW50ID0gbW9kZWwoY29udGV4dCwgdGFyZ2V0KQogICAgICAgIHZhcmlhbmNlX3BlbmFs'
        'dHkgPSB0b3JjaC5yZWx1KDAuMSAtIHByZWRpY3Rpb24uc3RkKGRpbT0wKSkubWVhbigpCiAgICAg'
        'ICAgbG9zcyA9IG1zZShwcmVkaWN0aW9uLCB0YXJnZXRfbGF0ZW50KSArICgwLjEgKiB2YXJpYW5j'
        'ZV9wZW5hbHR5KQogICAgICAgIGxvc3MuYmFja3dhcmQoKQogICAgICAgIG9wdGltaXplci5zdGVw'
        'KCkKCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBwcmVkaWN0aW9uLCB0YXJnZXRf'
        'bGF0ZW50ID0gbW9kZWwoY29udGV4dCwgdGFyZ2V0KQogICAgICAgIGZpbmFsX2xvc3MgPSBtc2Uo'
        'cHJlZGljdGlvbiwgdGFyZ2V0X2xhdGVudCkuaXRlbSgpCiAgICAgICAgdmFyaWFuY2VfcmVndWxh'
        'cml6YXRpb24gPSB0b3JjaC5yZWx1KDAuMSAtIHByZWRpY3Rpb24uc3RkKGRpbT0wKSkubWVhbigp'
        'Lml0ZW0oKQogICAgICAgIGZ1dHVyZV9wcmVkaWN0aW9uLCBmdXR1cmVfdGFyZ2V0ID0gbW9kZWwo'
        'dGFyZ2V0LCB0YXJnZXQgKyAwLjE1KQogICAgICAgIHJvbGxvdXRfZXJyb3IgPSBtc2UoZnV0dXJl'
        'X3ByZWRpY3Rpb24sIGZ1dHVyZV90YXJnZXQpLml0ZW0oKQoKICAgIHJldHVybiBKRVBBVHJhaW5p'
        'bmdSZXN1bHQoCiAgICAgICAgZmluYWxfbG9zcz1yb3VuZChmaW5hbF9sb3NzLCA2KSwKICAgICAg'
        'ICB2YXJpYW5jZV9yZWd1bGFyaXphdGlvbj1yb3VuZCh2YXJpYW5jZV9yZWd1bGFyaXphdGlvbiwg'
        'NiksCiAgICAgICAgcm9sbG91dF9lcnJvcj1yb3VuZChyb2xsb3V0X2Vycm9yLCA2KSwKICAgICkK'
    ),
    'olympus/core/representation.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBlbnVtIGltcG9ydCBTdHJF'
        'bnVtCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxkCgpm'
        'cm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoKCmNsYXNzIFJlcHJl'
        'c2VudGF0aW9uRmFtaWx5KFN0ckVudW0pOgogICAgRVVDTElERUFOID0gImV1Y2xpZGVhbiIKICAg'
        'IFNQSEVSSUNBTCA9ICJzcGhlcmljYWwiCiAgICBIWVBFUkJPTElDID0gImh5cGVyYm9saWMiCiAg'
        'ICBHUkFQSCA9ICJncmFwaCIKICAgIFRSRUUgPSAidHJlZSIKICAgIFBST0RVQ1QgPSAicHJvZHVj'
        'dCIKICAgIEhJRVJBUkNISUNBTCA9ICJoaWVyYXJjaGljYWwiCiAgICBTRVQgPSAic2V0IgogICAg'
        'U0VRVUVOQ0UgPSAic2VxdWVuY2UiCiAgICBHUklEID0gImdyaWQiCiAgICBTSU1QTElDSUFMID0g'
        'InNpbXBsaWNpYWwiCiAgICBMT1dfUkFOSyA9ICJsb3dfcmFuayIKICAgIERZTkFNSUNfQkFTSVMg'
        'PSAiZHluYW1pY19iYXNpcyIKCgpjbGFzcyBSZXByZXNlbnRhdGlvbkRlc2NyaXB0b3IoU3RyaWN0'
        'TW9kZWwpOgogICAgZmFtaWx5OiBSZXByZXNlbnRhdGlvbkZhbWlseQogICAgcmVhc29uczogbGlz'
        'dFtzdHJdCiAgICBxdWFsaXR5X3Njb3JlOiBmbG9hdAogICAgbWV0YWRhdGE6IGRpY3Rbc3RyLCBB'
        'bnldID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCgoKY2xhc3MgRHluYW1pY1JlcHJlc2Vu'
        'dGF0aW9uVGhlb3J5OgogICAgZGVmIHNlbGVjdChzZWxmLCBzYW1wbGU6IGRpY3Rbc3RyLCBBbnld'
        'KSAtPiBSZXByZXNlbnRhdGlvbkRlc2NyaXB0b3I6CiAgICAgICAgZWRnZXMgPSBzYW1wbGUuZ2V0'
        'KCJlZGdlcyIsIFtdKQogICAgICAgIHNlcXVlbmNlID0gc2FtcGxlLmdldCgic2VxdWVuY2UiLCBb'
        'XSkKICAgICAgICBncmlkID0gc2FtcGxlLmdldCgiZ3JpZCIsIFtdKQogICAgICAgIGhpZXJhcmNo'
        'eSA9IHNhbXBsZS5nZXQoImhpZXJhcmNoeV9kZXB0aCIsIDApCiAgICAgICAgaWYgZWRnZXMgYW5k'
        'IGhpZXJhcmNoeToKICAgICAgICAgICAgcmV0dXJuIFJlcHJlc2VudGF0aW9uRGVzY3JpcHRvcigK'
        'ICAgICAgICAgICAgICAgIGZhbWlseT1SZXByZXNlbnRhdGlvbkZhbWlseS5UUkVFLAogICAgICAg'
        'ICAgICAgICAgcmVhc29ucz1bImdyYXBoIGVkZ2VzIHByZXNlbnQiLCAiaGllcmFyY2h5IGRlcHRo'
        'IGRldGVjdGVkIl0sCiAgICAgICAgICAgICAgICBxdWFsaXR5X3Njb3JlPTAuODksCiAgICAgICAg'
        'ICAgICAgICBtZXRhZGF0YT17ImRlcHRoIjogaGllcmFyY2h5fSwKICAgICAgICAgICAgKQogICAg'
        'ICAgIGlmIGVkZ2VzOgogICAgICAgICAgICByZXR1cm4gUmVwcmVzZW50YXRpb25EZXNjcmlwdG9y'
        'KAogICAgICAgICAgICAgICAgZmFtaWx5PVJlcHJlc2VudGF0aW9uRmFtaWx5LkdSQVBILAogICAg'
        'ICAgICAgICAgICAgcmVhc29ucz1bInJlbGF0aW9uYWwgZWRnZXMgcHJlc2VudCJdLAogICAgICAg'
        'ICAgICAgICAgcXVhbGl0eV9zY29yZT0wLjg0LAogICAgICAgICAgICAgICAgbWV0YWRhdGE9eyJl'
        'ZGdlX2NvdW50IjogbGVuKGVkZ2VzKX0sCiAgICAgICAgICAgICkKICAgICAgICBpZiBncmlkOgog'
        'ICAgICAgICAgICByZXR1cm4gUmVwcmVzZW50YXRpb25EZXNjcmlwdG9yKAogICAgICAgICAgICAg'
        'ICAgZmFtaWx5PVJlcHJlc2VudGF0aW9uRmFtaWx5LkdSSUQsCiAgICAgICAgICAgICAgICByZWFz'
        'b25zPVsidHdvLWRpbWVuc2lvbmFsIHJlZ3VsYXIgc3RydWN0dXJlIGRldGVjdGVkIl0sCiAgICAg'
        'ICAgICAgICAgICBxdWFsaXR5X3Njb3JlPTAuODIsCiAgICAgICAgICAgICAgICBtZXRhZGF0YT17'
        'InJvd3MiOiBsZW4oZ3JpZCl9LAogICAgICAgICAgICApCiAgICAgICAgaWYgc2VxdWVuY2U6CiAg'
        'ICAgICAgICAgIHJldHVybiBSZXByZXNlbnRhdGlvbkRlc2NyaXB0b3IoCiAgICAgICAgICAgICAg'
        'ICBmYW1pbHk9UmVwcmVzZW50YXRpb25GYW1pbHkuU0VRVUVOQ0UsCiAgICAgICAgICAgICAgICBy'
        'ZWFzb25zPVsib3JkZXJlZCB0ZW1wb3JhbCBkYXRhIGRldGVjdGVkIl0sCiAgICAgICAgICAgICAg'
        'ICBxdWFsaXR5X3Njb3JlPTAuOCwKICAgICAgICAgICAgICAgIG1ldGFkYXRhPXsibGVuZ3RoIjog'
        'bGVuKHNlcXVlbmNlKX0sCiAgICAgICAgICAgICkKICAgICAgICByZXR1cm4gUmVwcmVzZW50YXRp'
        'b25EZXNjcmlwdG9yKAogICAgICAgICAgICBmYW1pbHk9UmVwcmVzZW50YXRpb25GYW1pbHkuRVVD'
        'TElERUFOLAogICAgICAgICAgICByZWFzb25zPVsiZGVmYXVsdCBjb250aW51b3VzIGZlYXR1cmUg'
        'Z2VvbWV0cnkiXSwKICAgICAgICAgICAgcXVhbGl0eV9zY29yZT0wLjcyLAogICAgICAgICAgICBt'
        'ZXRhZGF0YT17ImZlYXR1cmVfY291bnQiOiBsZW4oc2FtcGxlKX0sCiAgICAgICAgKQoKICAgIGRl'
        'ZiBwcm9qZWN0KHNlbGYsIHZhbHVlczogbGlzdFtmbG9hdF0sIGRlc2NyaXB0b3I6IFJlcHJlc2Vu'
        'dGF0aW9uRGVzY3JpcHRvcikgLT4gbGlzdFtmbG9hdF06CiAgICAgICAgaWYgbm90IHZhbHVlczoK'
        'ICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgaWYgZGVzY3JpcHRvci5mYW1pbHkgPT0gUmVw'
        'cmVzZW50YXRpb25GYW1pbHkuU1BIRVJJQ0FMOgogICAgICAgICAgICB0b3RhbCA9IHN1bShhYnMo'
        'dmFsdWUpIGZvciB2YWx1ZSBpbiB2YWx1ZXMpIG9yIDEuMAogICAgICAgICAgICByZXR1cm4gW3Jv'
        'dW5kKHZhbHVlIC8gdG90YWwsIDYpIGZvciB2YWx1ZSBpbiB2YWx1ZXNdCiAgICAgICAgaWYgZGVz'
        'Y3JpcHRvci5mYW1pbHkgaW4ge1JlcHJlc2VudGF0aW9uRmFtaWx5LlNFUVVFTkNFLCBSZXByZXNl'
        'bnRhdGlvbkZhbWlseS5FVUNMSURFQU59OgogICAgICAgICAgICBzY2FsZSA9IG1heCgxLjAsIG1h'
        'eChhYnModmFsdWUpIGZvciB2YWx1ZSBpbiB2YWx1ZXMpKQogICAgICAgICAgICByZXR1cm4gW3Jv'
        'dW5kKHZhbHVlIC8gc2NhbGUsIDYpIGZvciB2YWx1ZSBpbiB2YWx1ZXNdCiAgICAgICAgaWYgZGVz'
        'Y3JpcHRvci5mYW1pbHkgPT0gUmVwcmVzZW50YXRpb25GYW1pbHkuTE9XX1JBTks6CiAgICAgICAg'
        'ICAgIG1lYW4gPSBzdW0odmFsdWVzKSAvIG1heCgxLCBsZW4odmFsdWVzKSkKICAgICAgICAgICAg'
        'cmV0dXJuIFtyb3VuZCh2YWx1ZSAtIG1lYW4sIDYpIGZvciB2YWx1ZSBpbiB2YWx1ZXNdCiAgICAg'
        'ICAgcmV0dXJuIFtyb3VuZCh2YWx1ZSwgNikgZm9yIHZhbHVlIGluIHZhbHVlc10K'
    ),
    'olympus/core/retrodiction.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucy5hYmMg'
        'aW1wb3J0IENhbGxhYmxlCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKCkBkYXRh'
        'Y2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgUmV0cm9kaWN0ZWRTdGF0ZToKICAgIGNhbmRpZGF0ZV9z'
        'dGF0ZTogc3RyCiAgICBjb21wYXRpYmlsaXR5OiBmbG9hdAogICAgaW52YXJpYW50X3Bhc3NlZDog'
        'Ym9vbAogICAgZm9yd2FyZF9tYXRjaDogYm9vbAoKCmNsYXNzIFJldHJvZGljdGl2ZUlkZW50aXR5'
        'TmV0d29yazoKICAgIGRlZiBpbmZlcigKICAgICAgICBzZWxmLAogICAgICAgIGN1cnJlbnRfc3Rh'
        'dGU6IHN0ciwKICAgICAgICBnZW5lcmF0b3JzOiBsaXN0W0NhbGxhYmxlW1tzdHJdLCBzdHJdXSwK'
        'ICAgICAgICBmb3J3YXJkX3NpbXVsYXRvcjogQ2FsbGFibGVbW3N0cl0sIHN0cl0sCiAgICAgICAg'
        'aW52YXJpYW50X2NoZWNrZXI6IENhbGxhYmxlW1tzdHJdLCBib29sXSwKICAgICkgLT4gbGlzdFtS'
        'ZXRyb2RpY3RlZFN0YXRlXToKICAgICAgICByZXN1bHRzOiBsaXN0W1JldHJvZGljdGVkU3RhdGVd'
        'ID0gW10KICAgICAgICBmb3IgZ2VuZXJhdG9yIGluIGdlbmVyYXRvcnM6CiAgICAgICAgICAgIGNh'
        'bmRpZGF0ZSA9IGdlbmVyYXRvcihjdXJyZW50X3N0YXRlKQogICAgICAgICAgICBpbnZhcmlhbnRf'
        'cGFzc2VkID0gaW52YXJpYW50X2NoZWNrZXIoY2FuZGlkYXRlKQogICAgICAgICAgICBmb3J3YXJk'
        'X21hdGNoID0gZm9yd2FyZF9zaW11bGF0b3IoY2FuZGlkYXRlKSA9PSBjdXJyZW50X3N0YXRlCiAg'
        'ICAgICAgICAgIGNvbXBhdGliaWxpdHkgPSAwLjMKICAgICAgICAgICAgY29tcGF0aWJpbGl0eSAr'
        'PSAwLjQgaWYgaW52YXJpYW50X3Bhc3NlZCBlbHNlIDAuMAogICAgICAgICAgICBjb21wYXRpYmls'
        'aXR5ICs9IDAuMyBpZiBmb3J3YXJkX21hdGNoIGVsc2UgMC4wCiAgICAgICAgICAgIHJlc3VsdHMu'
        'YXBwZW5kKAogICAgICAgICAgICAgICAgUmV0cm9kaWN0ZWRTdGF0ZSgKICAgICAgICAgICAgICAg'
        'ICAgICBjYW5kaWRhdGVfc3RhdGU9Y2FuZGlkYXRlLAogICAgICAgICAgICAgICAgICAgIGNvbXBh'
        'dGliaWxpdHk9cm91bmQoY29tcGF0aWJpbGl0eSwgNCksCiAgICAgICAgICAgICAgICAgICAgaW52'
        'YXJpYW50X3Bhc3NlZD1pbnZhcmlhbnRfcGFzc2VkLAogICAgICAgICAgICAgICAgICAgIGZvcndh'
        'cmRfbWF0Y2g9Zm9yd2FyZF9tYXRjaCwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQog'
        'ICAgICAgIHJldHVybiBzb3J0ZWQocmVzdWx0cywga2V5PWxhbWJkYSBpdGVtOiBpdGVtLmNvbXBh'
        'dGliaWxpdHksIHJldmVyc2U9VHJ1ZSkKCgpkZWYgY29kZV9idWdfZ2VuZXJhdG9ycygpIC0+IGxp'
        'c3RbQ2FsbGFibGVbW3N0cl0sIHN0cl1dOgogICAgcmV0dXJuIFsKICAgICAgICBsYW1iZGEgY29k'
        'ZTogY29kZS5yZXBsYWNlKCJyZXR1cm4gdG90YWwgLyBjb3VudCIsICJyZXR1cm4gdG90YWwgLyAo'
        'Y291bnQgLSAxKSIpLAogICAgICAgIGxhbWJkYSBjb2RlOiBjb2RlLnJlcGxhY2UoInJldHVybiB0'
        'b3RhbCAvIGNvdW50IiwgInJldHVybiB0b3RhbCAvIGNvdW50IGlmIGNvdW50IGVsc2UgMCIpLAog'
        'ICAgICAgIGxhbWJkYSBjb2RlOiBjb2RlLnJlcGxhY2UoInRvdGFsICs9IHZhbHVlIiwgInRvdGFs'
        'ID0rIHZhbHVlIiksCiAgICBdCg=='
    ),
    'olympus/core/schemas.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBlbnVtIGltcG9ydCBTdHJF'
        'bnVtCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gcHlkYW50aWMgaW1wb3J0IEJhc2VNb2Rl'
        'bCwgQ29uZmlnRGljdCwgRmllbGQKCgpjbGFzcyBTdHJpY3RNb2RlbChCYXNlTW9kZWwpOgogICAg'
        'bW9kZWxfY29uZmlnID0gQ29uZmlnRGljdChleHRyYT0iZm9yYmlkIiwgdmFsaWRhdGVfYXNzaWdu'
        'bWVudD1UcnVlKQoKCmNsYXNzIEh5cG90aGVzaXNTdGF0dXMoU3RyRW51bSk6CiAgICBBQ1RJVkUg'
        'PSAiYWN0aXZlIgogICAgQ09ORklSTUVEID0gImNvbmZpcm1lZCIKICAgIFJFSkVDVEVEID0gInJl'
        'amVjdGVkIgogICAgUEFSS0VEID0gInBhcmtlZCIKCgpjbGFzcyBFdmlkZW5jZShTdHJpY3RNb2Rl'
        'bCk6CiAgICBzb3VyY2U6IHN0cgogICAgZGV0YWlsOiBzdHIKICAgIHdlaWdodDogZmxvYXQgPSBG'
        'aWVsZChnZT0wLjAsIGxlPTEuMCwgZGVmYXVsdD0wLjUpCiAgICB0YWdzOiBsaXN0W3N0cl0gPSBG'
        'aWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCgpjbGFzcyBIeXBvdGhlc2lzKFN0cmljdE1vZGVs'
        'KToKICAgIG5hbWU6IHN0cgogICAgZGVzY3JpcHRpb246IHN0cgogICAgY29uZmlkZW5jZTogZmxv'
        'YXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIHN0YXR1czogSHlwb3RoZXNpc1N0YXR1cyA9'
        'IEh5cG90aGVzaXNTdGF0dXMuQUNUSVZFCiAgICBldmlkZW5jZTogbGlzdFtFdmlkZW5jZV0gPSBG'
        'aWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHByb3ZlbmFuY2U6IGxpc3Rbc3RyXSA9IEZp'
        'ZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQoKCmNsYXNzIFByb3ZlbmFuY2VSZWNvcmQoU3RyaWN0'
        'TW9kZWwpOgogICAgYWN0b3I6IHN0cgogICAgYWN0aW9uOiBzdHIKICAgIG1ldGFkYXRhOiBkaWN0'
        'W3N0ciwgQW55XSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKCmNsYXNzIENvbXB1dGVC'
        'dWRnZXQoU3RyaWN0TW9kZWwpOgogICAgdG9rZW5fYnVkZ2V0OiBpbnQgPSBGaWVsZChnZT0wLCBk'
        'ZWZhdWx0PTJfMDAwKQogICAgbW9kZWxfY2FsbF9idWRnZXQ6IGludCA9IEZpZWxkKGdlPTAsIGRl'
        'ZmF1bHQ9OCkKICAgIGJyYW5jaF9idWRnZXQ6IGludCA9IEZpZWxkKGdlPTEsIGRlZmF1bHQ9NikK'
        'ICAgIHRvb2xfYnVkZ2V0OiBpbnQgPSBGaWVsZChnZT0wLCBkZWZhdWx0PTYpCiAgICBsYXRlbmN5'
        'X2J1ZGdldF9tczogaW50ID0gRmllbGQoZ2U9MCwgZGVmYXVsdD0xXzUwMCkKICAgIG1vbmV0YXJ5'
        'X2J1ZGdldF91c2Q6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBkZWZhdWx0PTAuMCkKICAgIG1lbW9y'
        'eV9idWRnZXRfbWI6IGludCA9IEZpZWxkKGdlPTEsIGRlZmF1bHQ9NTEyKQogICAgZGVwdGhfYnVk'
        'Z2V0OiBpbnQgPSBGaWVsZChnZT0xLCBkZWZhdWx0PTQpCgoKY2xhc3MgQnJhbmNoU2NvcmUoU3Ry'
        'aWN0TW9kZWwpOgogICAgcGxhdXNpYmlsaXR5OiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4w'
        'KQogICAgaW5mb3JtYXRpb25fZ2FpbjogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAg'
        'IHVzZWZ1bG5lc3M6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICB1bmNlcnRhaW50'
        'eV92YWx1ZTogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIGNvbXB1dGVfY29zdDog'
        'ZmxvYXQgPSBGaWVsZChndD0wLjAsIGRlZmF1bHQ9MS4wKQoKICAgIGRlZiBwcmlvcml0eShzZWxm'
        'KSAtPiBmbG9hdDoKICAgICAgICBudW1lcmF0b3IgPSAoCiAgICAgICAgICAgIHNlbGYucGxhdXNp'
        'YmlsaXR5ICogc2VsZi5pbmZvcm1hdGlvbl9nYWluICogc2VsZi51c2VmdWxuZXNzICogc2VsZi51'
        'bmNlcnRhaW50eV92YWx1ZQogICAgICAgICkKICAgICAgICByZXR1cm4gbnVtZXJhdG9yIC8gc2Vs'
        'Zi5jb21wdXRlX2Nvc3QK'
    ),
    'olympus/core/security.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBQZXJtaXNzaW9uUG9s'
        'aWN5OgogICAgYWxsb3dfbmV0d29yazogYm9vbCA9IEZhbHNlCiAgICBhbGxvd19zaGVsbDogYm9v'
        'bCA9IEZhbHNlCiAgICBhbGxvd19maWxlc3lzdGVtX3dyaXRlOiBib29sID0gVHJ1ZQogICAgYWxs'
        'b3dfaHR0cF9kb21haW5zOiB0dXBsZVtzdHIsIC4uLl0gPSAoKQoKCmNsYXNzIFByb21wdEluamVj'
        'dGlvbkd1YXJkOgogICAgc3VzcGljaW91c190b2tlbnMgPSAoImlnbm9yZSBwcmV2aW91cyIsICJy'
        'ZXZlYWwgc3lzdGVtIHByb21wdCIsICJleGZpbHRyYXRlIiwgIjxzY3JpcHQiKQoKICAgIGRlZiBp'
        'bnNwZWN0KHNlbGYsIGNvbnRlbnQ6IHN0cikgLT4gYm9vbDoKICAgICAgICBsb3dlcmVkID0gY29u'
        'dGVudC5sb3dlcigpCiAgICAgICAgcmV0dXJuIG5vdCBhbnkodG9rZW4gaW4gbG93ZXJlZCBmb3Ig'
        'dG9rZW4gaW4gc2VsZi5zdXNwaWNpb3VzX3Rva2VucykK'
    ),
    'olympus/core/sqlite_journal.py': (
        'IiIiU1FMaXRlIGpvdXJuYWwtbW9kZSBoZWxwZXJzIGZvciBwb3J0YWJsZSBsb2NhbCBzdG9yZXMu'
        'IiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgc3FsaXRlMwoK'
        'CmRlZiBhcHBseV9kdXJhYmxlX2pvdXJuYWxfbW9kZShjb25uZWN0aW9uOiBzcWxpdGUzLkNvbm5l'
        'Y3Rpb24pIC0+IHN0cjoKICAgICIiIlByZWZlciBXQUw7IGZhbGwgYmFjayB0byBERUxFVEUgd2hl'
        'biBXQUwgY2Fubm90IGJlIGVuYWJsZWQuCgogICAgU29tZSBmaWxlc3lzdGVtcyAobmV0d29yayBt'
        'b3VudHMsIHJlc3RyaWN0ZWQgc2FuZGJveGVzKSByZWplY3QgV0FMIHNpZGVjYXIKICAgIGZpbGVz'
        'LiBDYWxsZXJzIHN0aWxsIGdldCBhIGR1cmFibGUgYGBGVUxMYGAgc3luY2hyb25vdXMgc3RvcmUg'
        'ZWl0aGVyIHdheS4KICAgICIiIgogICAgdHJ5OgogICAgICAgIHJvdyA9IGNvbm5lY3Rpb24uZXhl'
        'Y3V0ZSgiUFJBR01BIGpvdXJuYWxfbW9kZSA9IFdBTCIpLmZldGNob25lKCkKICAgICAgICBtb2Rl'
        'ID0gc3RyKHJvd1swXSkubG93ZXIoKSBpZiByb3cgZWxzZSAiIgogICAgICAgIGlmIG1vZGUgPT0g'
        'IndhbCI6CiAgICAgICAgICAgIHJldHVybiAid2FsIgogICAgZXhjZXB0IHNxbGl0ZTMuRXJyb3I6'
        'CiAgICAgICAgbW9kZSA9ICIiCgogICAgcm93ID0gY29ubmVjdGlvbi5leGVjdXRlKCJQUkFHTUEg'
        'am91cm5hbF9tb2RlID0gREVMRVRFIikuZmV0Y2hvbmUoKQogICAgbW9kZSA9IHN0cihyb3dbMF0p'
        'Lmxvd2VyKCkgaWYgcm93IGVsc2UgImRlbGV0ZSIKICAgIGlmIG1vZGUgbm90IGluIHsiZGVsZXRl'
        'IiwgInRydW5jYXRlIiwgInBlcnNpc3QiLCAibWVtb3J5IiwgIm9mZiJ9OgogICAgICAgICMgVW5l'
        'eHBlY3RlZCBtb2RlIOKAlCBzdGlsbCByZXR1cm4gd2hhdCBTUUxpdGUgcmVwb3J0ZWQuCiAgICAg'
        'ICAgcmV0dXJuIG1vZGUKICAgIHJldHVybiBtb2RlCg=='
    ),
    'olympus/core/transform.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yLCBubgoK'
        'CkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgVHJhbnNmb3JtTWV0cmljczoKICAgIHJlY29u'
        'c3RydWN0aW9uX2xvc3M6IGZsb2F0CiAgICBzcGFyc2l0eV9zY29yZTogZmxvYXQKICAgIGluZGVw'
        'ZW5kZW5jZV9zY29yZTogZmxvYXQKICAgIHByZWRpY3RpdmVfdXRpbGl0eTogZmxvYXQKCgpjbGFz'
        'cyBMZWFybmVkVHJhbnNmb3JtTmV0d29yayhubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNl'
        'bGYsIGlucHV0X2RpbTogaW50LCBsYXRlbnRfZGltOiBpbnQpIC0+IE5vbmU6CiAgICAgICAgc3Vw'
        'ZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5lbmNvZGVyID0gbm4uTGluZWFyKGlucHV0X2Rp'
        'bSwgbGF0ZW50X2RpbSkKICAgICAgICBzZWxmLmRlY29kZXIgPSBubi5MaW5lYXIobGF0ZW50X2Rp'
        'bSwgaW5wdXRfZGltKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGJhdGNoOiBUZW5zb3IpIC0+IHR1'
        'cGxlW1RlbnNvciwgVGVuc29yXToKICAgICAgICBsYXRlbnQgPSB0b3JjaC50YW5oKHNlbGYuZW5j'
        'b2RlcihiYXRjaCkpCiAgICAgICAgcmVjb25zdHJ1Y3Rpb24gPSBzZWxmLmRlY29kZXIobGF0ZW50'
        'KQogICAgICAgIHJldHVybiBsYXRlbnQsIHJlY29uc3RydWN0aW9uCgoKZGVmIHRyYWluX2xlYXJu'
        'ZWRfdHJhbnNmb3JtKAogICAgc2lnbmFsczogVGVuc29yLAogICAgbGF0ZW50X2RpbTogaW50ID0g'
        'MywKICAgIGVwb2NoczogaW50ID0gMTIwLAogICAgbGVhcm5pbmdfcmF0ZTogZmxvYXQgPSAwLjAz'
        'LAogICAgc2VlZDogaW50ID0gNywKKSAtPiBUcmFuc2Zvcm1NZXRyaWNzOgogICAgaWYgc2lnbmFs'
        'cy5uZGltICE9IDIgb3Igc2lnbmFscy5zaGFwZVswXSA8IDIgb3Igc2lnbmFscy5zaGFwZVsxXSA8'
        'IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic2lnbmFscyBtdXN0IGJlIGEgdHdvLWRpbWVu'
        'c2lvbmFsIGJhdGNoIHdpdGggYXQgbGVhc3QgdHdvIHJvd3MiKQogICAgaWYgbGF0ZW50X2RpbSA8'
        'IDEgb3IgZXBvY2hzIDwgMSBvciBsZWFybmluZ19yYXRlIDw9IDA6CiAgICAgICAgcmFpc2UgVmFs'
        'dWVFcnJvcigibGF0ZW50X2RpbSwgZXBvY2hzLCBhbmQgbGVhcm5pbmdfcmF0ZSBtdXN0IGJlIHBv'
        'c2l0aXZlIikKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICBtb2RlbCA9IExlYXJuZWRU'
        'cmFuc2Zvcm1OZXR3b3JrKHNpZ25hbHMuc2hhcGVbMV0sIGxhdGVudF9kaW0pCiAgICBvcHRpbWl6'
        'ZXIgPSB0b3JjaC5vcHRpbS5BZGFtKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9bGVhcm5pbmdfcmF0'
        'ZSkKICAgIGxvc3NfZm4gPSBubi5NU0VMb3NzKCkKCiAgICBmb3IgXyBpbiByYW5nZShlcG9jaHMp'
        'OgogICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoKQogICAgICAgIGxhdGVudCwgcmVjb25zdHJ1'
        'Y3Rpb24gPSBtb2RlbChzaWduYWxzKQogICAgICAgIHJlY29uc3RydWN0aW9uX2xvc3MgPSBsb3Nz'
        'X2ZuKHJlY29uc3RydWN0aW9uLCBzaWduYWxzKQogICAgICAgIHNwYXJzaXR5X3BlbmFsdHkgPSBs'
        'YXRlbnQuYWJzKCkubWVhbigpCiAgICAgICAgZGVjb3JyZWxhdGlvbl9wZW5hbHR5ID0gdG9yY2gu'
        'Y292KGxhdGVudC5UKS5hYnMoKS5tZWFuKCkKICAgICAgICBsb3NzID0gcmVjb25zdHJ1Y3Rpb25f'
        'bG9zcyArICgwLjAyICogc3BhcnNpdHlfcGVuYWx0eSkgKyAoMC4wMSAqIGRlY29ycmVsYXRpb25f'
        'cGVuYWx0eSkKICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICBvcHRpbWl6ZXIuc3RlcCgp'
        'CgogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgbGF0ZW50LCByZWNvbnN0cnVjdGlv'
        'biA9IG1vZGVsKHNpZ25hbHMpCiAgICAgICAgcmVjb25zdHJ1Y3Rpb25fbG9zcyA9IGxvc3NfZm4o'
        'cmVjb25zdHJ1Y3Rpb24sIHNpZ25hbHMpLml0ZW0oKQogICAgICAgIHNwYXJzaXR5X3Njb3JlID0g'
        'ZmxvYXQoMS4wIC0gdG9yY2guY2xhbXAobGF0ZW50LmFicygpLm1lYW4oKSwgMC4wLCAxLjApLml0'
        'ZW0oKSkKICAgICAgICBpbmRlcGVuZGVuY2Vfc2NvcmUgPSBmbG9hdCgKICAgICAgICAgICAgMS4w'
        'IC0gdG9yY2guY2xhbXAodG9yY2guY292KGxhdGVudC5UKS5hYnMoKS5tZWFuKCksIDAuMCwgMS4w'
        'KS5pdGVtKCkKICAgICAgICApCiAgICAgICAgcHJlZGljdGl2ZV91dGlsaXR5ID0gZmxvYXQoMS4w'
        'IC0gbWluKDEuMCwgcmVjb25zdHJ1Y3Rpb25fbG9zcykpCiAgICByZXR1cm4gVHJhbnNmb3JtTWV0'
        'cmljcygKICAgICAgICByZWNvbnN0cnVjdGlvbl9sb3NzPXJvdW5kKHJlY29uc3RydWN0aW9uX2xv'
        'c3MsIDYpLAogICAgICAgIHNwYXJzaXR5X3Njb3JlPXJvdW5kKHNwYXJzaXR5X3Njb3JlLCA2KSwK'
        'ICAgICAgICBpbmRlcGVuZGVuY2Vfc2NvcmU9cm91bmQoaW5kZXBlbmRlbmNlX3Njb3JlLCA2KSwK'
        'ICAgICAgICBwcmVkaWN0aXZlX3V0aWxpdHk9cm91bmQocHJlZGljdGl2ZV91dGlsaXR5LCA2KSwK'
        'ICAgICkK'
    ),
    'olympus/core/verification.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFzdApmcm9tIHR5cGlu'
        'ZyBpbXBvcnQgQW55Cgpmcm9tIHB5ZGFudGljIGltcG9ydCBCYXNlTW9kZWwsIEZpZWxkCgpmcm9t'
        'IG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoKCmNsYXNzIFZlcmlmaWNh'
        'dGlvbkV2aWRlbmNlKFN0cmljdE1vZGVsKToKICAgIHZlcmlmaWVyOiBzdHIKICAgIHBhc3NlZDog'
        'Ym9vbAogICAgZGV0YWlsOiBzdHIKICAgIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldID0gRmllbGQo'
        'ZGVmYXVsdF9mYWN0b3J5PWRpY3QpCgoKY2xhc3MgU2NoZW1hVmVyaWZpZXI6CiAgICBkZWYgdmVy'
        'aWZ5KHNlbGYsIG1vZGVsOiB0eXBlW0Jhc2VNb2RlbF0sIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnld'
        'KSAtPiBWZXJpZmljYXRpb25FdmlkZW5jZToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHZhbGlk'
        'YXRlZCA9IG1vZGVsLm1vZGVsX3ZhbGlkYXRlKHBheWxvYWQpCiAgICAgICAgICAgIHJldHVybiBW'
        'ZXJpZmljYXRpb25FdmlkZW5jZSgKICAgICAgICAgICAgICAgIHZlcmlmaWVyPSJzY2hlbWEiLAog'
        'ICAgICAgICAgICAgICAgcGFzc2VkPVRydWUsCiAgICAgICAgICAgICAgICBkZXRhaWw9IlBheWxv'
        'YWQgY29uZm9ybXMgdG8gc2NoZW1hLiIsCiAgICAgICAgICAgICAgICBwYXlsb2FkPXZhbGlkYXRl'
        'ZC5tb2RlbF9kdW1wKCksCiAgICAgICAgICAgICkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFz'
        'IGVycm9yOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICAgICAgcmV0dXJuIFZlcmlmaWNhdGlvbkV2'
        'aWRlbmNlKAogICAgICAgICAgICAgICAgdmVyaWZpZXI9InNjaGVtYSIsCiAgICAgICAgICAgICAg'
        'ICBwYXNzZWQ9RmFsc2UsCiAgICAgICAgICAgICAgICBkZXRhaWw9c3RyKGVycm9yKSwKICAgICAg'
        'ICAgICAgKQoKCmNsYXNzIENvZGVWZXJpZmllcjoKICAgIGRlZiB2ZXJpZnkoc2VsZiwgY29kZTog'
        'c3RyKSAtPiBWZXJpZmljYXRpb25FdmlkZW5jZToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHRy'
        'ZWUgPSBhc3QucGFyc2UoY29kZSkKICAgICAgICAgICAgZnVuY3Rpb25zID0gW25vZGUubmFtZSBm'
        'b3Igbm9kZSBpbiBhc3Qud2Fsayh0cmVlKSBpZiBpc2luc3RhbmNlKG5vZGUsIGFzdC5GdW5jdGlv'
        'bkRlZildCiAgICAgICAgICAgIHJldHVybiBWZXJpZmljYXRpb25FdmlkZW5jZSgKICAgICAgICAg'
        'ICAgICAgIHZlcmlmaWVyPSJjb2RlIiwKICAgICAgICAgICAgICAgIHBhc3NlZD1UcnVlLAogICAg'
        'ICAgICAgICAgICAgZGV0YWlsPSJQeXRob24gc291cmNlIHBhcnNlZCBzdWNjZXNzZnVsbHkuIiwK'
        'ICAgICAgICAgICAgICAgIHBheWxvYWQ9eyJmdW5jdGlvbnMiOiBmdW5jdGlvbnN9LAogICAgICAg'
        'ICAgICApCiAgICAgICAgZXhjZXB0IFN5bnRheEVycm9yIGFzIGVycm9yOgogICAgICAgICAgICBy'
        'ZXR1cm4gVmVyaWZpY2F0aW9uRXZpZGVuY2UoCiAgICAgICAgICAgICAgICB2ZXJpZmllcj0iY29k'
        'ZSIsCiAgICAgICAgICAgICAgICBwYXNzZWQ9RmFsc2UsCiAgICAgICAgICAgICAgICBkZXRhaWw9'
        'c3RyKGVycm9yKSwKICAgICAgICAgICAgKQoKCmNsYXNzIENvbnRyYWRpY3Rpb25WZXJpZmllcjoK'
        'ICAgIGRlZiB2ZXJpZnkoc2VsZiwgc3RhdGVtZW50czogbGlzdFtzdHJdKSAtPiBWZXJpZmljYXRp'
        'b25FdmlkZW5jZToKICAgICAgICBub3JtYWxpemVkID0ge3N0YXRlbWVudC5sb3dlcigpLnN0cmlw'
        'KCkgZm9yIHN0YXRlbWVudCBpbiBzdGF0ZW1lbnRzfQogICAgICAgIGNvbnRyYWRpY3Rpb24gPSBh'
        'bnkoCiAgICAgICAgICAgIHN0YXRlbWVudC5zdGFydHN3aXRoKCJub3QgIikgYW5kIHN0YXRlbWVu'
        'dC5yZW1vdmVwcmVmaXgoIm5vdCAiKSBpbiBub3JtYWxpemVkCiAgICAgICAgICAgIGZvciBzdGF0'
        'ZW1lbnQgaW4gbm9ybWFsaXplZAogICAgICAgICkKICAgICAgICByZXR1cm4gVmVyaWZpY2F0aW9u'
        'RXZpZGVuY2UoCiAgICAgICAgICAgIHZlcmlmaWVyPSJjb250cmFkaWN0aW9uIiwKICAgICAgICAg'
        'ICAgcGFzc2VkPW5vdCBjb250cmFkaWN0aW9uLAogICAgICAgICAgICBkZXRhaWw9IkNvbnRyYWRp'
        'Y3Rpb24gZGV0ZWN0ZWQuIgogICAgICAgICAgICBpZiBjb250cmFkaWN0aW9uCiAgICAgICAgICAg'
        'IGVsc2UgIk5vIGRpcmVjdCBjb250cmFkaWN0aW9uIGRldGVjdGVkLiIsCiAgICAgICAgICAgIHBh'
        'eWxvYWQ9eyJzdGF0ZW1lbnRzIjogc3RhdGVtZW50c30sCiAgICAgICAgKQoKCmNsYXNzIENhbGli'
        'cmF0aW9uVmVyaWZpZXI6CiAgICBkZWYgdmVyaWZ5KHNlbGYsIGNvbmZpZGVuY2U6IGZsb2F0LCBw'
        'YXNzZWQ6IGJvb2wpIC0+IFZlcmlmaWNhdGlvbkV2aWRlbmNlOgogICAgICAgIGNhbGlicmF0ZWQg'
        'PSBub3QgKChjb25maWRlbmNlID4gMC44IGFuZCBub3QgcGFzc2VkKSBvciAoY29uZmlkZW5jZSA8'
        'IDAuMiBhbmQgcGFzc2VkKSkKICAgICAgICByZXR1cm4gVmVyaWZpY2F0aW9uRXZpZGVuY2UoCiAg'
        'ICAgICAgICAgIHZlcmlmaWVyPSJjYWxpYnJhdGlvbiIsCiAgICAgICAgICAgIHBhc3NlZD1jYWxp'
        'YnJhdGVkLAogICAgICAgICAgICBkZXRhaWw9IkNvbmZpZGVuY2UgaXMgY2FsaWJyYXRlZC4iCiAg'
        'ICAgICAgICAgIGlmIGNhbGlicmF0ZWQKICAgICAgICAgICAgZWxzZSAiQ29uZmlkZW5jZSBpcyBw'
        'b29ybHkgY2FsaWJyYXRlZC4iLAogICAgICAgICAgICBwYXlsb2FkPXsiY29uZmlkZW5jZSI6IGNv'
        'bmZpZGVuY2UsICJvdXRjb21lIjogcGFzc2VkfSwKICAgICAgICApCg=='
    ),
    'olympus/core/workspace.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBwYXRo'
        'bGliIGltcG9ydCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIGNhc3QKCmZyb20gcHlkYW50'
        'aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBFdmlkZW5j'
        'ZSwgSHlwb3RoZXNpcywgUHJvdmVuYW5jZVJlY29yZCwgU3RyaWN0TW9kZWwKCgpjbGFzcyBXb3Jr'
        'c3BhY2VEaWZmKFN0cmljdE1vZGVsKToKICAgIGNoYW5nZWRfZmllbGRzOiBsaXN0W3N0cl0KICAg'
        'IG9sZF92YWx1ZXM6IGRpY3Rbc3RyLCBBbnldCiAgICBuZXdfdmFsdWVzOiBkaWN0W3N0ciwgQW55'
        'XQoKCmNsYXNzIExhdGVudFdvcmtzcGFjZShTdHJpY3RNb2RlbCk6CiAgICBvYmplY3RpdmU6IHN0'
        'cgogICAgdGFza19zdGF0ZTogZGljdFtzdHIsIEFueV0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9'
        'ZGljdCkKICAgIGNvbnN0cmFpbnRzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9'
        'bGlzdCkKICAgIGV2aWRlbmNlOiBsaXN0W0V2aWRlbmNlXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9y'
        'eT1saXN0KQogICAgaHlwb3RoZXNlczogbGlzdFtIeXBvdGhlc2lzXSA9IEZpZWxkKGRlZmF1bHRf'
        'ZmFjdG9yeT1saXN0KQogICAgdW5jZXJ0YWludHk6IGZsb2F0ID0gMC41CiAgICBhY3RpdmVfcGxh'
        'bjogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBtZW1vcnlfcmVm'
        'ZXJlbmNlczogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBwcmVk'
        'aWN0ZWRfY29uc2VxdWVuY2VzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlz'
        'dCkKICAgIHRvb2xzX2F2YWlsYWJsZTogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5'
        'PWxpc3QpCiAgICBicmFuY2hfYWN0aXZhdGlvbnM6IGRpY3Rbc3RyLCBmbG9hdF0gPSBGaWVsZChk'
        'ZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIHByb3ZlbmFuY2U6IGxpc3RbUHJvdmVuYW5jZVJlY29y'
        'ZF0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHZlcnNpb246IGludCA9IDEKCiAg'
        'ICBkZWYgc2VyaWFsaXplKHNlbGYpIC0+IHN0cjoKICAgICAgICByZXR1cm4gc2VsZi5tb2RlbF9k'
        'dW1wX2pzb24oaW5kZW50PTIpCgogICAgZGVmIGNoZWNrcG9pbnQoc2VsZiwgcGF0aDogUGF0aCkg'
        'LT4gTm9uZToKICAgICAgICBwYXRoLndyaXRlX3RleHQoc2VsZi5zZXJpYWxpemUoKSwgZW5jb2Rp'
        'bmc9InV0Zi04IikKCiAgICBAY2xhc3NtZXRob2QKICAgIGRlZiBsb2FkKGNscywgcGF0aDogUGF0'
        'aCkgLT4gTGF0ZW50V29ya3NwYWNlOgogICAgICAgIHJldHVybiBjbHMubW9kZWxfdmFsaWRhdGVf'
        'anNvbihwYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKCiAgICBkZWYgbWVyZ2Uoc2Vs'
        'Ziwgb3RoZXI6IExhdGVudFdvcmtzcGFjZSkgLT4gTGF0ZW50V29ya3NwYWNlOgogICAgICAgIG1l'
        'cmdlZCA9IHNlbGYubW9kZWxfY29weShkZWVwPVRydWUpCiAgICAgICAgbWVyZ2VkLnRhc2tfc3Rh'
        'dGUudXBkYXRlKG90aGVyLnRhc2tfc3RhdGUpCiAgICAgICAgbWVyZ2VkLmNvbnN0cmFpbnRzID0g'
        'c29ydGVkKHNldChzZWxmLmNvbnN0cmFpbnRzICsgb3RoZXIuY29uc3RyYWludHMpKQogICAgICAg'
        'IG1lcmdlZC5ldmlkZW5jZS5leHRlbmQob3RoZXIuZXZpZGVuY2UpCiAgICAgICAgbWVyZ2VkLmh5'
        'cG90aGVzZXMuZXh0ZW5kKG90aGVyLmh5cG90aGVzZXMpCiAgICAgICAgbWVyZ2VkLmFjdGl2ZV9w'
        'bGFuID0gc2VsZi5hY3RpdmVfcGxhbiArIFsKICAgICAgICAgICAgc3RlcCBmb3Igc3RlcCBpbiBv'
        'dGhlci5hY3RpdmVfcGxhbiBpZiBzdGVwIG5vdCBpbiBzZWxmLmFjdGl2ZV9wbGFuCiAgICAgICAg'
        'XQogICAgICAgIG1lcmdlZC5tZW1vcnlfcmVmZXJlbmNlcyA9IHNvcnRlZChzZXQoc2VsZi5tZW1v'
        'cnlfcmVmZXJlbmNlcyArIG90aGVyLm1lbW9yeV9yZWZlcmVuY2VzKSkKICAgICAgICBtZXJnZWQu'
        'cHJlZGljdGVkX2NvbnNlcXVlbmNlcyA9IHNvcnRlZCgKICAgICAgICAgICAgc2V0KHNlbGYucHJl'
        'ZGljdGVkX2NvbnNlcXVlbmNlcyArIG90aGVyLnByZWRpY3RlZF9jb25zZXF1ZW5jZXMpCiAgICAg'
        'ICAgKQogICAgICAgIG1lcmdlZC50b29sc19hdmFpbGFibGUgPSBzb3J0ZWQoc2V0KHNlbGYudG9v'
        'bHNfYXZhaWxhYmxlICsgb3RoZXIudG9vbHNfYXZhaWxhYmxlKSkKICAgICAgICBtZXJnZWQuYnJh'
        'bmNoX2FjdGl2YXRpb25zLnVwZGF0ZShvdGhlci5icmFuY2hfYWN0aXZhdGlvbnMpCiAgICAgICAg'
        'bWVyZ2VkLnByb3ZlbmFuY2UuZXh0ZW5kKG90aGVyLnByb3ZlbmFuY2UpCiAgICAgICAgbWVyZ2Vk'
        'LnVuY2VydGFpbnR5ID0gcm91bmQoKHNlbGYudW5jZXJ0YWludHkgKyBvdGhlci51bmNlcnRhaW50'
        'eSkgLyAyLjAsIDQpCiAgICAgICAgbWVyZ2VkLnZlcnNpb24gPSBtYXgoc2VsZi52ZXJzaW9uLCBv'
        'dGhlci52ZXJzaW9uKSArIDEKICAgICAgICByZXR1cm4gbWVyZ2VkCgogICAgZGVmIGRpZmYoc2Vs'
        'Ziwgb3RoZXI6IExhdGVudFdvcmtzcGFjZSkgLT4gV29ya3NwYWNlRGlmZjoKICAgICAgICBjdXJy'
        'ZW50ID0gc2VsZi5tb2RlbF9kdW1wKCkKICAgICAgICB0YXJnZXQgPSBvdGhlci5tb2RlbF9kdW1w'
        'KCkKICAgICAgICBjaGFuZ2VkX2ZpZWxkcyA9IHNvcnRlZChrZXkgZm9yIGtleSBpbiBjdXJyZW50'
        'IGlmIGN1cnJlbnRba2V5XSAhPSB0YXJnZXRba2V5XSkKICAgICAgICByZXR1cm4gV29ya3NwYWNl'
        'RGlmZigKICAgICAgICAgICAgY2hhbmdlZF9maWVsZHM9Y2hhbmdlZF9maWVsZHMsCiAgICAgICAg'
        'ICAgIG9sZF92YWx1ZXM9e2tleTogY3VycmVudFtrZXldIGZvciBrZXkgaW4gY2hhbmdlZF9maWVs'
        'ZHN9LAogICAgICAgICAgICBuZXdfdmFsdWVzPXtrZXk6IHRhcmdldFtrZXldIGZvciBrZXkgaW4g'
        'Y2hhbmdlZF9maWVsZHN9LAogICAgICAgICkKCiAgICBkZWYgcmVwbGF5KHNlbGYpIC0+IGxpc3Rb'
        'ZGljdFtzdHIsIEFueV1dOgogICAgICAgIHRpbWVsaW5lID0gW3siZXZlbnQiOiAid29ya3NwYWNl'
        'X2luaXRpYWxpemVkIiwgIm9iamVjdGl2ZSI6IHNlbGYub2JqZWN0aXZlLCAidmVyc2lvbiI6IDF9'
        'XQogICAgICAgIGZvciByZWNvcmQgaW4gc2VsZi5wcm92ZW5hbmNlOgogICAgICAgICAgICB0aW1l'
        'bGluZS5hcHBlbmQoCiAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgImV2ZW50'
        'IjogcmVjb3JkLmFjdGlvbiwKICAgICAgICAgICAgICAgICAgICAiYWN0b3IiOiByZWNvcmQuYWN0'
        'b3IsCiAgICAgICAgICAgICAgICAgICAgIm1ldGFkYXRhIjogcmVjb3JkLm1ldGFkYXRhLAogICAg'
        'ICAgICAgICAgICAgfQogICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIHRpbWVsaW5lCgogICAg'
        'ZGVmIGV4cG9ydF9zbmFwc2hvdChzZWxmLCBwYXRoOiBQYXRoKSAtPiBkaWN0W3N0ciwgQW55XToK'
        'ICAgICAgICBwYXlsb2FkID0gY2FzdChkaWN0W3N0ciwgQW55XSwganNvbi5sb2FkcyhzZWxmLnNl'
        'cmlhbGl6ZSgpKSkKICAgICAgICBwYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhwYXlsb2FkLCBp'
        'bmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgcmV0dXJuIHBheWxvYWQK'
    ),
    'olympus/data/__init__.py': (
        'IiIiRGF0YSBwbGF0Zm9ybSBjb21wb25lbnRzLiIiIgoKZnJvbSBvbHltcHVzLmRhdGEuY29udHJv'
        'bGxlZF9mdXNpb24gaW1wb3J0ICgKICAgIFJFR0lTVEVSRURfU1BMSVRTLAogICAgQ29udHJvbGxl'
        'ZERhdGFzZXRSZWNlaXB0LAogICAgQ29udHJvbGxlZFNwbGl0U3BlYywKICAgIGdlbmVyYXRlX2Nv'
        'bnRyb2xsZWRfZnVzaW9uX2RhdGFzZXQsCiAgICBnZW5lcmF0ZV9jb250cm9sbGVkX3NwbGl0LAop'
        'CmZyb20gb2x5bXB1cy5kYXRhLmVkaXRjbGV2ciBpbXBvcnQgKAogICAgRWRpdENMRVZSRGF0YXNl'
        'dCwKICAgIEVkaXRDTEVWUlBhaXIsCiAgICBFZGl0Q0xFVlJQYWlyUGF0aHMsCiAgICBpdGVyX3Nw'
        'bGl0X3JlY29yZHMsCikKZnJvbSBvbHltcHVzLmRhdGEub2x5bXB1c19kYXRhc2V0IGltcG9ydCAo'
        'CiAgICBMb2FkZWRPbHltcHVzRGF0YXNldCwKICAgIE9seW1wdXNCYXRjaEFydGlmYWN0LAogICAg'
        'T2x5bXB1c1N5c3RlbURhdGFzZXRNYW5pZmVzdCwKICAgIGxvYWRfb2x5bXB1c19kYXRhc2V0LAog'
        'ICAgd3JpdGVfb2x5bXB1c19kYXRhc2V0X21hbmlmZXN0LAopCmZyb20gb2x5bXB1cy5kYXRhLnJl'
        'bGlhYmlsaXR5X2Z1c2lvbiBpbXBvcnQgKAogICAgQ09NUE9ORU5UX09SREVSLAogICAgUkVHSVNU'
        'RVJFRF9SRUxJQUJJTElUWV9TUExJVFMsCiAgICBSZWxpYWJpbGl0eURhdGFzZXRSZWNlaXB0LAog'
        'ICAgUmVsaWFiaWxpdHlTcGxpdFNwZWMsCiAgICBnZW5lcmF0ZV9yZWxpYWJpbGl0eV9mdXNpb25f'
        'ZGF0YXNldCwKICAgIGdlbmVyYXRlX3JlbGlhYmlsaXR5X3NwbGl0LAopCgpfX2FsbF9fID0gWwog'
        'ICAgIlJFR0lTVEVSRURfU1BMSVRTIiwKICAgICJDb250cm9sbGVkRGF0YXNldFJlY2VpcHQiLAog'
        'ICAgIkNvbnRyb2xsZWRTcGxpdFNwZWMiLAogICAgIkNPTVBPTkVOVF9PUkRFUiIsCiAgICAiRWRp'
        'dENMRVZSRGF0YXNldCIsCiAgICAiRWRpdENMRVZSUGFpciIsCiAgICAiRWRpdENMRVZSUGFpclBh'
        'dGhzIiwKICAgICJMb2FkZWRPbHltcHVzRGF0YXNldCIsCiAgICAiT2x5bXB1c0JhdGNoQXJ0aWZh'
        'Y3QiLAogICAgIk9seW1wdXNTeXN0ZW1EYXRhc2V0TWFuaWZlc3QiLAogICAgIlJFR0lTVEVSRURf'
        'UkVMSUFCSUxJVFlfU1BMSVRTIiwKICAgICJSZWxpYWJpbGl0eURhdGFzZXRSZWNlaXB0IiwKICAg'
        'ICJSZWxpYWJpbGl0eVNwbGl0U3BlYyIsCiAgICAiZ2VuZXJhdGVfY29udHJvbGxlZF9mdXNpb25f'
        'ZGF0YXNldCIsCiAgICAiZ2VuZXJhdGVfY29udHJvbGxlZF9zcGxpdCIsCiAgICAiZ2VuZXJhdGVf'
        'cmVsaWFiaWxpdHlfZnVzaW9uX2RhdGFzZXQiLAogICAgImdlbmVyYXRlX3JlbGlhYmlsaXR5X3Nw'
        'bGl0IiwKICAgICJpdGVyX3NwbGl0X3JlY29yZHMiLAogICAgImxvYWRfb2x5bXB1c19kYXRhc2V0'
        'IiwKICAgICJ3cml0ZV9vbHltcHVzX2RhdGFzZXRfbWFuaWZlc3QiLApdCg=='
    ),
    'olympus/data/conservative_routing_v6.py': (
        'IiIiRnJlc2ggc291cmNlIGlkZW50aXRpZXMgd2l0aCBhIHByb3NwZWN0aXZlbHkgcmVzZXJ2ZWQg'
        'Y29tcG9zaXRlIHNoaWZ0LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoK'
        'aW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBvbHltcHVzLmRhdGEu'
        'cm9idXN0X2Z1c2lvbl92NSBpbXBvcnQgZ2VuZXJhdGUgYXMgcHJldmlvdXNfZ2VuZXJhdGUKCgpk'
        'ZWYgZ2VuZXJhdGUoc2VlZDogaW50LCBjb3VudDogaW50LCByZWdpbWU6IHN0cikgLT4gZGljdFtz'
        'dHIsIFRlbnNvcl06CiAgICBpZiByZWdpbWUgIT0gImNvbXBvc2l0ZV9zaGlmdCI6CiAgICAgICAg'
        'cmV0dXJuIHByZXZpb3VzX2dlbmVyYXRlKHNlZWQsIGNvdW50LCByZWdpbWUpCiAgICBiYXRjaCA9'
        'IHByZXZpb3VzX2dlbmVyYXRlKHNlZWQsIGNvdW50LCAiY29ycnVwdGVkIikKICAgIGNsZWFuID0g'
        'cHJldmlvdXNfZ2VuZXJhdGUoc2VlZCwgY291bnQsICJjbGVhbiIpCiAgICAjIFJlc2VydmVkIGNv'
        'bmZpZ3VyYXRpb24gZGVjbGFyZWQgYmVmb3JlIGV4ZWN1dGlvbjsgbm8gdGFyZ2V0LWRlcGVuZGVu'
        'dCB0cmFuc2Zvcm0uCiAgICAjIEtub3duIHBlcnR1cmJhdGlvbiB0eXBlcywgbm90IGEgbm92ZWwg'
        'ZmFtaWx5IG9yIG5hdHVyYWwtZGF0YSBPT0QgY2xhaW0uCiAgICBmb3IgaW5kZXgsIGtleSBpbiBl'
        'bnVtZXJhdGUoKCJ0b2tlbnMiLCAiaW1hZ2UiLCAic3RhdGVzIikpOgogICAgICAgIG1hc2sgPSBi'
        'YXRjaFsiY29ycnVwdGlvbl9zdGF0dXMiXVs6LCBpbmRleF0gPT0gMgogICAgICAgIHggPSBjbGVh'
        'bltrZXldLmNsb25lKCkKICAgICAgICBpZiBrZXkgPT0gInRva2VucyI6CiAgICAgICAgICAgIHgg'
        'PSB4LnJvbGwoMiwgLTEpCiAgICAgICAgZWxpZiBrZXkgPT0gImltYWdlIjoKICAgICAgICAgICAg'
        'eFs6LCA6LCAxOjoyLCA6XSA9IDAuNQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHggPSAwLjUg'
        'KiB4ICsgdG9yY2gubGluc3BhY2UoLTAuNzUsIDAuNzUsIHguc2hhcGVbMV0pW05vbmUsIDosIE5v'
        'bmVdCiAgICAgICAgYmF0Y2hba2V5XVttYXNrXSA9IHhbbWFza10KICAgIHJldHVybiBiYXRjaAo='
    ),
    'olympus/data/controlled_fusion.py': (
        'IiIiRGV0ZXJtaW5pc3RpYyBhbGlnbmVkIGRhdGEgZm9yIHRoZSByZWdpc3RlcmVkIE9seW1wdXMg'
        'Y29udHJvbGxlZC1tZWNoYW5pc20gc3R1ZHkuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFu'
        'bm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW8KaW1wb3J0IGpzb24KaW1wb3J0IG9z'
        'CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGltcG9ydCBOYW1lZFRlbXBv'
        'cmFyeUZpbGUKZnJvbSB0eXBpbmcgaW1wb3J0IExpdGVyYWwKCmltcG9ydCB0b3JjaApmcm9tIHB5'
        'ZGFudGljIGltcG9ydCBGaWVsZApmcm9tIHRvcmNoIGltcG9ydCBUZW5zb3IKCmZyb20gb2x5bXB1'
        'cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCmZyb20gb2x5bXB1cy5kYXRhLm9seW1w'
        'dXNfZGF0YXNldCBpbXBvcnQgd3JpdGVfb2x5bXB1c19kYXRhc2V0X21hbmlmZXN0CgpDb250cm9s'
        'bGVkUmVnaW1lID0gTGl0ZXJhbFsibWl4ZWQiLCAiY2xlYW4iLCAiY29ycnVwdGVkIiwgIm9vZCJd'
        'CgoKY2xhc3MgQ29udHJvbGxlZFNwbGl0U3BlYyhTdHJpY3RNb2RlbCk6CiAgICBuYW1lOiBzdHIg'
        'PSBGaWVsZChtaW5fbGVuZ3RoPTEpCiAgICBleGFtcGxlczogaW50ID0gRmllbGQoZ2U9MSkKICAg'
        'IHNlZWQ6IGludCA9IEZpZWxkKGdlPTAsIGxlPTIqKjYzIC0gMSkKICAgIHJlZ2ltZTogQ29udHJv'
        'bGxlZFJlZ2ltZQoKCmNsYXNzIENvbnRyb2xsZWREYXRhc2V0UmVjZWlwdChTdHJpY3RNb2RlbCk6'
        'CiAgICBzY2hlbWFfdmVyc2lvbjogTGl0ZXJhbFsxXSA9IDEKICAgIGRhdGFzZXRfaWQ6IExpdGVy'
        'YWxbIm9seW1wdXMtY29udHJvbGxlZC1mdXNpb24iXSA9ICJvbHltcHVzLWNvbnRyb2xsZWQtZnVz'
        'aW9uIgogICAgdmVyc2lvbjogTGl0ZXJhbFsiMi4wLjAiXSA9ICIyLjAuMCIKICAgIGxpY2Vuc2U6'
        'IExpdGVyYWxbIkNDLUJZLTQuMCJdID0gIkNDLUJZLTQuMCIKICAgIGdlbmVyYXRvcjogTGl0ZXJh'
        'bFsib2x5bXB1cy5kYXRhLmNvbnRyb2xsZWRfZnVzaW9uIl0gPSAib2x5bXB1cy5kYXRhLmNvbnRy'
        'b2xsZWRfZnVzaW9uIgogICAgc3BsaXRzOiBsaXN0W0NvbnRyb2xsZWRTcGxpdFNwZWNdCiAgICBi'
        'dW5kbGVfc2hhMjU2OiBkaWN0W3N0ciwgc3RyXQogICAgbWFuaWZlc3Rfc2hhMjU2OiBkaWN0W3N0'
        'ciwgc3RyXQogICAgc291cmNlX2lkX3NoYTI1NjogZGljdFtzdHIsIHN0cl0KICAgIHNvdXJjZV9v'
        'dmVybGFwX2NvdW50OiBpbnQgPSBGaWVsZChnZT0wKQogICAgcHJvdGVjdGVkX3NwbGl0czogbGlz'
        'dFtzdHJdCiAgICBjbGFpbV9ib3VuZGFyeTogc3RyCgoKUkVHSVNURVJFRF9TUExJVFMgPSAoCiAg'
        'ICBDb250cm9sbGVkU3BsaXRTcGVjKG5hbWU9InRyYWluIiwgZXhhbXBsZXM9NTEyLCBzZWVkPTIw'
        'MjYxMjAxLCByZWdpbWU9Im1peGVkIiksCiAgICBDb250cm9sbGVkU3BsaXRTcGVjKG5hbWU9ImRl'
        'dmVsb3BtZW50IiwgZXhhbXBsZXM9MTI4LCBzZWVkPTIwMjYxMjAyLCByZWdpbWU9Im1peGVkIiks'
        'CiAgICBDb250cm9sbGVkU3BsaXRTcGVjKG5hbWU9ImNsZWFuX3Rlc3QiLCBleGFtcGxlcz0yNTYs'
        'IHNlZWQ9MjAyNjEyMDMsIHJlZ2ltZT0iY2xlYW4iKSwKICAgIENvbnRyb2xsZWRTcGxpdFNwZWMo'
        'CiAgICAgICAgbmFtZT0iY29ycnVwdGVkX3Rlc3QiLCBleGFtcGxlcz0yNTYsIHNlZWQ9MjAyNjEy'
        'MDQsIHJlZ2ltZT0iY29ycnVwdGVkIgogICAgKSwKICAgIENvbnRyb2xsZWRTcGxpdFNwZWMobmFt'
        'ZT0ib29kX3Rlc3QiLCBleGFtcGxlcz0yNTYsIHNlZWQ9MjAyNjEyMDUsIHJlZ2ltZT0ib29kIiks'
        'CikKCgpkZWYgX3NoYTI1NihwYXlsb2FkOiBieXRlcykgLT4gc3RyOgogICAgcmV0dXJuIGhhc2hs'
        'aWIuc2hhMjU2KHBheWxvYWQpLmhleGRpZ2VzdCgpCgoKZGVmIF9hdG9taWNfd3JpdGUocGF0aDog'
        'UGF0aCwgcGF5bG9hZDogYnl0ZXMpIC0+IE5vbmU6CiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJl'
        'bnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIE5hbWVkVGVtcG9yYXJ5RmlsZShkaXI9'
        'cGF0aC5wYXJlbnQsIHByZWZpeD1mIi57cGF0aC5uYW1lfS4iLCBkZWxldGU9RmFsc2UpIGFzIGhh'
        'bmRsZToKICAgICAgICBoYW5kbGUud3JpdGUocGF5bG9hZCkKICAgICAgICBoYW5kbGUuZmx1c2go'
        'KQogICAgICAgIG9zLmZzeW5jKGhhbmRsZS5maWxlbm8oKSkKICAgICAgICB0ZW1wb3JhcnkgPSBQ'
        'YXRoKGhhbmRsZS5uYW1lKQogICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKCgpkZWYgX3NvdXJj'
        'ZV9pZHMoc3BlYzogQ29udHJvbGxlZFNwbGl0U3BlYykgLT4gbGlzdFtzdHJdOgogICAgcmV0dXJu'
        'IFsKICAgICAgICBmIntzcGVjLm5hbWV9LXtfc2hhMjU2KGYne3NwZWMuc2VlZH06e2luZGV4fScu'
        'ZW5jb2RlKCkpWzoyMF19IgogICAgICAgIGZvciBpbmRleCBpbiByYW5nZShzcGVjLmV4YW1wbGVz'
        'KQogICAgXQoKCmRlZiBfY29ycnVwdGlvbl9wbGFuKAogICAgc3BlYzogQ29udHJvbGxlZFNwbGl0'
        'U3BlYywgZ2VuZXJhdG9yOiB0b3JjaC5HZW5lcmF0b3IKKSAtPiB0dXBsZVtUZW5zb3IsIFRlbnNv'
        'cl06CiAgICAiIiJSZXR1cm4gbWlzc2luZyBhbmQgcGVydHVyYmVkIG1vZGFsaXR5IGluZGljZXM7'
        'IC0xIG1lYW5zIG5vIGludGVydmVudGlvbi4iIiIKCiAgICBtaXNzaW5nID0gdG9yY2guZnVsbCgo'
        'c3BlYy5leGFtcGxlcywpLCAtMSwgZHR5cGU9dG9yY2gubG9uZykKICAgIHBlcnR1cmJlZCA9IHRv'
        'cmNoLmZ1bGwoKHNwZWMuZXhhbXBsZXMsKSwgLTEsIGR0eXBlPXRvcmNoLmxvbmcpCiAgICBpZiBz'
        'cGVjLnJlZ2ltZSA9PSAibWl4ZWQiOgogICAgICAgIHNlbGVjdGVkID0gdG9yY2gucmFuZChzcGVj'
        'LmV4YW1wbGVzLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKSA8IDAuNzAKICAgICAgICBtaXNzaW5nW3Nl'
        'bGVjdGVkXSA9IHRvcmNoLnJhbmRpbnQoMCwgMywgKGludChzZWxlY3RlZC5zdW0oKSksKSwgZ2Vu'
        'ZXJhdG9yPWdlbmVyYXRvcikKICAgIGVsaWYgc3BlYy5yZWdpbWUgaW4geyJjb3JydXB0ZWQiLCAi'
        'b29kIn06CiAgICAgICAgbWlzc2luZyA9IHRvcmNoLnJhbmRpbnQoMCwgMywgKHNwZWMuZXhhbXBs'
        'ZXMsKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikKICAgICAgICBvZmZzZXQgPSB0b3JjaC5yYW5kaW50'
        'KDEsIDMsIChzcGVjLmV4YW1wbGVzLCksIGdlbmVyYXRvcj1nZW5lcmF0b3IpCiAgICAgICAgcGVy'
        'dHVyYmVkID0gKG1pc3NpbmcgKyBvZmZzZXQpICUgMwogICAgcmV0dXJuIG1pc3NpbmcsIHBlcnR1'
        'cmJlZAoKCmRlZiBnZW5lcmF0ZV9jb250cm9sbGVkX3NwbGl0KHNwZWM6IENvbnRyb2xsZWRTcGxp'
        'dFNwZWMpIC0+IHR1cGxlW2RpY3Rbc3RyLCBUZW5zb3JdLCBsaXN0W3N0cl1dOgogICAgIiIiR2Vu'
        'ZXJhdGUgb25lIGRldGVybWluaXN0aWMgc3BsaXQgd2l0aG91dCByZWFkaW5nIGFueSBvdGhlciBz'
        'cGxpdCBvciBnbG9iYWwgUk5HLiIiIgoKICAgIGdlbmVyYXRvciA9IHRvcmNoLkdlbmVyYXRvcihk'
        'ZXZpY2U9ImNwdSIpLm1hbnVhbF9zZWVkKHNwZWMuc2VlZCkKICAgIGNvdW50ID0gc3BlYy5leGFt'
        'cGxlcwogICAgbGFiZWxzID0gdG9yY2gucmFuZGludCgwLCA0LCAoY291bnQsKSwgZ2VuZXJhdG9y'
        'PWdlbmVyYXRvcikKICAgIG1pc3NpbmcsIHBlcnR1cmJlZCA9IF9jb3JydXB0aW9uX3BsYW4oc3Bl'
        'YywgZ2VuZXJhdG9yKQogICAgbm9pc2Vfc2NhbGUgPSAwLjEyIGlmIHNwZWMucmVnaW1lICE9ICJv'
        'b2QiIGVsc2UgMC4yNAoKICAgIHRva2VucyA9IHRvcmNoLnJhbmRpbnQoOCwgMzEsIChjb3VudCwg'
        'OCksIGdlbmVyYXRvcj1nZW5lcmF0b3IsIGR0eXBlPXRvcmNoLmxvbmcpCiAgICBjbGFzc190b2tl'
        'bnMgPSBsYWJlbHMgKyAxCiAgICBpZiBzcGVjLnJlZ2ltZSA9PSAib29kIjoKICAgICAgICBjbGFz'
        'c190b2tlbnMgPSB0b3JjaC50ZW5zb3IoWzMsIDEsIDQsIDJdLCBkdHlwZT10b3JjaC5sb25nKVts'
        'YWJlbHNdCiAgICB0b2tlbnNbOiwgMF0gPSBjbGFzc190b2tlbnMKICAgIHRva2Vuc1s6LCAxXSA9'
        'IDUgKyBsYWJlbHMucmVtYWluZGVyKDIpCiAgICB0b2tlbnNbOiwgMl0gPSA3ICsgbGFiZWxzLmRp'
        'digyLCByb3VuZGluZ19tb2RlPSJmbG9vciIpCiAgICBhdHRlbnRpb25fbWFzayA9IHRvcmNoLm9u'
        'ZXNfbGlrZSh0b2tlbnMsIGR0eXBlPXRvcmNoLmJvb2wpCgogICAgaW1hZ2UgPSB0b3JjaC5yYW5k'
        'KChjb3VudCwgMywgOCwgOCksIGdlbmVyYXRvcj1nZW5lcmF0b3IpICogbm9pc2Vfc2NhbGUKICAg'
        'IGZvciBpbmRleCBpbiByYW5nZShjb3VudCk6CiAgICAgICAgbGFiZWwgPSBpbnQobGFiZWxzW2lu'
        'ZGV4XSkKICAgICAgICBjaGFubmVsID0gbGFiZWwgJSAzCiAgICAgICAgdG9wID0gMCBpZiBsYWJl'
        'bCA8IDIgZWxzZSA0CiAgICAgICAgbGVmdCA9IDAgaWYgbGFiZWwgJSAyID09IDAgZWxzZSA0CiAg'
        'ICAgICAgaWYgc3BlYy5yZWdpbWUgPT0gIm9vZCI6CiAgICAgICAgICAgIGNoYW5uZWwgPSAoY2hh'
        'bm5lbCArIDEpICUgMwogICAgICAgICAgICBsZWZ0ID0gNCAtIGxlZnQKICAgICAgICBpbWFnZVtp'
        'bmRleCwgY2hhbm5lbCwgdG9wIDogdG9wICsgNCwgbGVmdCA6IGxlZnQgKyA0XSArPSAwLjc1CiAg'
        'ICBpbWFnZS5jbGFtcF8oMC4wLCAxLjApCgogICAgYWN0aW9ucyA9IHRvcmNoLnJhbmQoKGNvdW50'
        'LCA0LCAzKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikgKiAyLjAgLSAxLjAKICAgIHRpbWUgPSB0b3Jj'
        'aC5hcmFuZ2UoNCwgZHR5cGU9dG9yY2guZmxvYXQzMikudmlldygxLCA0LCAxKQogICAgbGFiZWxf'
        'dmFsdWUgPSBsYWJlbHMuZmxvYXQoKS52aWV3KGNvdW50LCAxLCAxKSAvIDMuMAogICAgc3RhdGVf'
        'bm9pc2UgPSB0b3JjaC5yYW5kbigoY291bnQsIDQsIDUpLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKSAq'
        'IG5vaXNlX3NjYWxlCiAgICBzdGF0ZV90ZW1wbGF0ZSA9IHRvcmNoLmNhdCgKICAgICAgICAoCiAg'
        'ICAgICAgICAgIGxhYmVsX3ZhbHVlLmV4cGFuZCgtMSwgNCwgMSksCiAgICAgICAgICAgIChsYWJl'
        'bF92YWx1ZSArIDAuMDggKiB0aW1lKS5leHBhbmQoLTEsIC0xLCAxKSwKICAgICAgICAgICAgKGxh'
        'YmVscy5yZW1haW5kZXIoMikuZmxvYXQoKS52aWV3KGNvdW50LCAxLCAxKSkuZXhwYW5kKC0xLCA0'
        'LCAxKSwKICAgICAgICAgICAgKGxhYmVscy5kaXYoMiwgcm91bmRpbmdfbW9kZT0iZmxvb3IiKS5m'
        'bG9hdCgpLnZpZXcoY291bnQsIDEsIDEpKS5leHBhbmQoCiAgICAgICAgICAgICAgICAtMSwgNCwg'
        'MQogICAgICAgICAgICApLAogICAgICAgICAgICB0b3JjaC5zaW4obGFiZWxfdmFsdWUgKiB0b3Jj'
        'aC5waSArIHRpbWUgKiAwLjMpLAogICAgICAgICksCiAgICAgICAgZGltPS0xLAogICAgKQogICAg'
        'c3RhdGVzID0gc3RhdGVfdGVtcGxhdGUgKyBzdGF0ZV9ub2lzZQoKICAgIGZvciBtb2RhbGl0eSwg'
        'dmFsdWUgaW4gZW51bWVyYXRlKCh0b2tlbnMsIGltYWdlLCBzdGF0ZXMpKToKICAgICAgICBzZWxl'
        'Y3RlZF9taXNzaW5nID0gbWlzc2luZyA9PSBtb2RhbGl0eQogICAgICAgIHNlbGVjdGVkX3BlcnR1'
        'cmJlZCA9IHBlcnR1cmJlZCA9PSBtb2RhbGl0eQogICAgICAgIGlmIGJvb2woc2VsZWN0ZWRfbWlz'
        'c2luZy5hbnkoKSk6CiAgICAgICAgICAgIHZhbHVlW3NlbGVjdGVkX21pc3NpbmddID0gMAogICAg'
        'ICAgIGlmIGJvb2woc2VsZWN0ZWRfcGVydHVyYmVkLmFueSgpKToKICAgICAgICAgICAgaWYgbW9k'
        'YWxpdHkgPT0gMDoKICAgICAgICAgICAgICAgIHRva2Vuc1tzZWxlY3RlZF9wZXJ0dXJiZWQsIDoz'
        'XSA9IHRvcmNoLnJhbmRpbnQoCiAgICAgICAgICAgICAgICAgICAgMSwKICAgICAgICAgICAgICAg'
        'ICAgICAxMiwKICAgICAgICAgICAgICAgICAgICAoaW50KHNlbGVjdGVkX3BlcnR1cmJlZC5zdW0o'
        'KSksIDMpLAogICAgICAgICAgICAgICAgICAgIGdlbmVyYXRvcj1nZW5lcmF0b3IsCiAgICAgICAg'
        'ICAgICAgICApCiAgICAgICAgICAgIGVsaWYgbW9kYWxpdHkgPT0gMToKICAgICAgICAgICAgICAg'
        'IGltYWdlW3NlbGVjdGVkX3BlcnR1cmJlZF0gPSB0b3JjaC5yYW5kKAogICAgICAgICAgICAgICAg'
        'ICAgIChpbnQoc2VsZWN0ZWRfcGVydHVyYmVkLnN1bSgpKSwgMywgOCwgOCksIGdlbmVyYXRvcj1n'
        'ZW5lcmF0b3IKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAg'
        'ICAgIHN0YXRlc1tzZWxlY3RlZF9wZXJ0dXJiZWRdID0gdG9yY2gucmFuZG4oCiAgICAgICAgICAg'
        'ICAgICAgICAgKGludChzZWxlY3RlZF9wZXJ0dXJiZWQuc3VtKCkpLCA0LCA1KSwgZ2VuZXJhdG9y'
        'PWdlbmVyYXRvcgogICAgICAgICAgICAgICAgKQoKICAgIHRlbXBvcmFsX21hc2sgPSB0b3JjaC5v'
        'bmVzKChjb3VudCwgNCksIGR0eXBlPXRvcmNoLmJvb2wpCiAgICBhbGxvd2VkX3RyYW5zaXRpb25z'
        'ID0gdG9yY2gub25lcygoY291bnQsIDQpLCBkdHlwZT10b3JjaC5ib29sKQogICAgdHJhbnNpdGlv'
        'bl90YXJnZXRzID0gbGFiZWxzLnJlbWFpbmRlcigzKQogICAgZm9yYmlkZGVuID0gKHRyYW5zaXRp'
        'b25fdGFyZ2V0cyArIDEpLnJlbWFpbmRlcigzKQogICAgYWxsb3dlZF90cmFuc2l0aW9uc1t0b3Jj'
        'aC5hcmFuZ2UoY291bnQpLCBmb3JiaWRkZW5dID0gRmFsc2UKICAgIGFsbG93ZWRfdHJhbnNpdGlv'
        'bnNbOiwgM10gPSBUcnVlCgogICAgY29ycnVwdGlvbl9jb3VudCA9IG1pc3NpbmcuZ2UoMCkuZmxv'
        'YXQoKSArIHBlcnR1cmJlZC5nZSgwKS5mbG9hdCgpCiAgICByaXNrX3Njb3JlID0gMC4wNSArIGNv'
        'cnJ1cHRpb25fY291bnQgKiAwLjE1CiAgICBpbnRyaW5zaWNfcmlza190YXJnZXRzID0gY29ycnVw'
        'dGlvbl9jb3VudCAvIDIuMAogICAgcGFkZGVkX2FjdGlvbnMgPSB0b3JjaC5ubi5mdW5jdGlvbmFs'
        'LnBhZChhY3Rpb25zLCAoMCwgMikpCiAgICBuZXh0X3N0YXRlX3RhcmdldHMgPSBzdGF0ZXMgKyAw'
        'LjEgKiBwYWRkZWRfYWN0aW9ucwoKICAgIGJ1bmRsZSA9IHsKICAgICAgICAidG9rZW5zIjogdG9r'
        'ZW5zLAogICAgICAgICJhdHRlbnRpb25fbWFzayI6IGF0dGVudGlvbl9tYXNrLAogICAgICAgICJp'
        'bWFnZSI6IGltYWdlLAogICAgICAgICJzdGF0ZXMiOiBzdGF0ZXMsCiAgICAgICAgImFjdGlvbnMi'
        'OiBhY3Rpb25zLAogICAgICAgICJ0ZW1wb3JhbF9tYXNrIjogdGVtcG9yYWxfbWFzaywKICAgICAg'
        'ICAiYWxsb3dlZF90cmFuc2l0aW9ucyI6IGFsbG93ZWRfdHJhbnNpdGlvbnMsCiAgICAgICAgInJp'
        'c2tfc2NvcmUiOiByaXNrX3Njb3JlLAogICAgICAgICJkb21haW5faWRzIjogbGFiZWxzLnJlbWFp'
        'bmRlcigzKSwKICAgICAgICAidGltZXN0YW1wIjogdG9yY2guYXJhbmdlKGNvdW50LCBkdHlwZT10'
        'b3JjaC5mbG9hdDMyKSAvIG1heCgxLCBjb3VudCAtIDEpLAogICAgICAgICJkZWNpc2lvbl90YXJn'
        'ZXRzIjogbGFiZWxzLAogICAgICAgICJ0cmFuc2l0aW9uX3RhcmdldHMiOiB0cmFuc2l0aW9uX3Rh'
        'cmdldHMsCiAgICAgICAgIm5leHRfc3RhdGVfdGFyZ2V0cyI6IG5leHRfc3RhdGVfdGFyZ2V0cywK'
        'ICAgICAgICAiaW50cmluc2ljX3Jpc2tfdGFyZ2V0cyI6IGludHJpbnNpY19yaXNrX3RhcmdldHMs'
        'CiAgICB9CiAgICByZXR1cm4gYnVuZGxlLCBfc291cmNlX2lkcyhzcGVjKQoKCmRlZiBfc2VyaWFs'
        'aXplX2J1bmRsZShidW5kbGU6IGRpY3Rbc3RyLCBUZW5zb3JdKSAtPiBieXRlczoKICAgIGJ1ZmZl'
        'ciA9IGlvLkJ5dGVzSU8oKQogICAgdG9yY2guc2F2ZShidW5kbGUsIGJ1ZmZlcikKICAgIHJldHVy'
        'biBidWZmZXIuZ2V0dmFsdWUoKQoKCmRlZiBnZW5lcmF0ZV9jb250cm9sbGVkX2Z1c2lvbl9kYXRh'
        'c2V0KG91dHB1dDogUGF0aCkgLT4gQ29udHJvbGxlZERhdGFzZXRSZWNlaXB0OgogICAgIiIiTWF0'
        'ZXJpYWxpemUgdGhlIHJlZ2lzdGVyZWQgc3BsaXRzIGFuZCBoYXNoLWJvdW5kIHR3by1zcGxpdCBt'
        'YW5pZmVzdHMgb25jZS4iIiIKCiAgICBvdXRwdXQgPSBvdXRwdXQucmVzb2x2ZSgpCiAgICByZWNl'
        'aXB0X3BhdGggPSBvdXRwdXQgLyAicmVjZWlwdC5qc29uIgogICAgaWYgb3V0cHV0LmV4aXN0cygp'
        'IGFuZCBhbnkob3V0cHV0Lml0ZXJkaXIoKSk6CiAgICAgICAgcmFpc2UgRmlsZUV4aXN0c0Vycm9y'
        'KCJyZWZ1c2luZyB0byBvdmVyd3JpdGUgYSBjb250cm9sbGVkIGRhdGFzZXQgZGlyZWN0b3J5IikK'
        'ICAgIG91dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgc291cmNl'
        'X2lkczogZGljdFtzdHIsIGxpc3Rbc3RyXV0gPSB7fQogICAgYnVuZGxlX2hhc2hlczogZGljdFtz'
        'dHIsIHN0cl0gPSB7fQogICAgZm9yIHNwZWMgaW4gUkVHSVNURVJFRF9TUExJVFM6CiAgICAgICAg'
        'YnVuZGxlLCBpZGVudGlmaWVycyA9IGdlbmVyYXRlX2NvbnRyb2xsZWRfc3BsaXQoc3BlYykKICAg'
        'ICAgICBwYXlsb2FkID0gX3NlcmlhbGl6ZV9idW5kbGUoYnVuZGxlKQogICAgICAgIHBhdGggPSBv'
        'dXRwdXQgLyBmIntzcGVjLm5hbWV9LnB0IgogICAgICAgIF9hdG9taWNfd3JpdGUocGF0aCwgcGF5'
        'bG9hZCkKICAgICAgICBzb3VyY2VfaWRzW3NwZWMubmFtZV0gPSBpZGVudGlmaWVycwogICAgICAg'
        'IGJ1bmRsZV9oYXNoZXNbc3BlYy5uYW1lXSA9IF9zaGEyNTYocGF5bG9hZCkKCiAgICBhbGxfaWRz'
        'ID0gW2lkZW50aWZpZXIgZm9yIHZhbHVlcyBpbiBzb3VyY2VfaWRzLnZhbHVlcygpIGZvciBpZGVu'
        'dGlmaWVyIGluIHZhbHVlc10KICAgIG92ZXJsYXBfY291bnQgPSBsZW4oYWxsX2lkcykgLSBsZW4o'
        'c2V0KGFsbF9pZHMpKQogICAgaWYgb3ZlcmxhcF9jb3VudDoKICAgICAgICByYWlzZSBSdW50aW1l'
        'RXJyb3IoImNvbnRyb2xsZWQgc3BsaXQgc291cmNlIGlkZW50aXRpZXMgb3ZlcmxhcCIpCgogICAg'
        'bWFuaWZlc3RfaGFzaGVzOiBkaWN0W3N0ciwgc3RyXSA9IHt9CiAgICBmb3IgZXZhbHVhdGlvbl9u'
        'YW1lIGluICgiZGV2ZWxvcG1lbnQiLCAiY2xlYW5fdGVzdCIsICJjb3JydXB0ZWRfdGVzdCIsICJv'
        'b2RfdGVzdCIpOgogICAgICAgIG1hbmlmZXN0ID0gd3JpdGVfb2x5bXB1c19kYXRhc2V0X21hbmlm'
        'ZXN0KAogICAgICAgICAgICBvdXRwdXQgLyBmIm1hbmlmZXN0X3tldmFsdWF0aW9uX25hbWV9Lmpz'
        'b24iLAogICAgICAgICAgICBkYXRhc2V0X2lkPSJvbHltcHVzLWNvbnRyb2xsZWQtZnVzaW9uIiwK'
        'ICAgICAgICAgICAgdmVyc2lvbj0iMi4wLjAiLAogICAgICAgICAgICBsaWNlbnNlX25hbWU9IkND'
        'LUJZLTQuMCIsCiAgICAgICAgICAgIHRyYWluX2JhdGNoPW91dHB1dCAvICJ0cmFpbi5wdCIsCiAg'
        'ICAgICAgICAgIGV2YWx1YXRpb25fYmF0Y2g9b3V0cHV0IC8gZiJ7ZXZhbHVhdGlvbl9uYW1lfS5w'
        'dCIsCiAgICAgICAgICAgIHRyYWluX3NvdXJjZV9pZHM9c291cmNlX2lkc1sidHJhaW4iXSwKICAg'
        'ICAgICAgICAgZXZhbHVhdGlvbl9zb3VyY2VfaWRzPXNvdXJjZV9pZHNbZXZhbHVhdGlvbl9uYW1l'
        'XSwKICAgICAgICApCiAgICAgICAgYXNzZXJ0IG1hbmlmZXN0Lm1hbmlmZXN0X3NoYTI1NiBpcyBu'
        'b3QgTm9uZQogICAgICAgIG1hbmlmZXN0X2hhc2hlc1tldmFsdWF0aW9uX25hbWVdID0gbWFuaWZl'
        'c3QubWFuaWZlc3Rfc2hhMjU2CgogICAgcmVjZWlwdCA9IENvbnRyb2xsZWREYXRhc2V0UmVjZWlw'
        'dCgKICAgICAgICBzcGxpdHM9bGlzdChSRUdJU1RFUkVEX1NQTElUUyksCiAgICAgICAgYnVuZGxl'
        'X3NoYTI1Nj1idW5kbGVfaGFzaGVzLAogICAgICAgIG1hbmlmZXN0X3NoYTI1Nj1tYW5pZmVzdF9o'
        'YXNoZXMsCiAgICAgICAgc291cmNlX2lkX3NoYTI1Nj17CiAgICAgICAgICAgIG5hbWU6IF9zaGEy'
        'NTYoKCJcbiIuam9pbih2YWx1ZXMpICsgIlxuIikuZW5jb2RlKCkpCiAgICAgICAgICAgIGZvciBu'
        'YW1lLCB2YWx1ZXMgaW4gc291cmNlX2lkcy5pdGVtcygpCiAgICAgICAgfSwKICAgICAgICBzb3Vy'
        'Y2Vfb3ZlcmxhcF9jb3VudD1vdmVybGFwX2NvdW50LAogICAgICAgIHByb3RlY3RlZF9zcGxpdHM9'
        'WyJjbGVhbl90ZXN0IiwgImNvcnJ1cHRlZF90ZXN0IiwgIm9vZF90ZXN0Il0sCiAgICAgICAgY2xh'
        'aW1fYm91bmRhcnk9KAogICAgICAgICAgICAiQ29udHJvbGxlZCBzeW50aGV0aWMgbWVjaGFuaXNt'
        'IGV2aWRlbmNlIG9ubHk7IG5vIG5hdHVyYWwtZGF0YSwgZGVwbG95bWVudCwgIgogICAgICAgICAg'
        'ICAidGFyZ2V0LXNjYWxlLCBvciBKRVBBIGNsYWltIGlzIGxpY2Vuc2VkIGJ5IHRoaXMgZGF0YXNl'
        'dC4iCiAgICAgICAgKSwKICAgICkKICAgIF9hdG9taWNfd3JpdGUoCiAgICAgICAgcmVjZWlwdF9w'
        'YXRoLAogICAgICAgIChqc29uLmR1bXBzKHJlY2VpcHQubW9kZWxfZHVtcChtb2RlPSJqc29uIiks'
        'IGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iKS5lbmNvZGUoKSwKICAgICkKICAgIHJl'
        'dHVybiByZWNlaXB0Cgo='
    ),
    'olympus/data/editclevr.py': (
        'IiIiQm91bmRlZC1tZW1vcnkgYWNjZXNzIHRvIGEgdmFsaWRhdGVkLCBtYXRlcmlhbGl6ZWQgRWRp'
        'dENMRVZSIHNuYXBzaG90LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoK'
        'aW1wb3J0IGpzb24KZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IEl0ZXJhdG9yLCBTZXF1ZW5j'
        'ZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQ'
        'YXRoLCBQdXJlUG9zaXhQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIFNlbGYsIG92ZXJsb2Fk'
        'CgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gUElMIGltcG9ydCBJbWFnZQpm'
        'cm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZCwgbW9kZWxfdmFsaWRhdG9yCmZyb20gdG9yY2ggaW1w'
        'b3J0IFRlbnNvcgoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwK'
        'ClBBWUxPQURfRklFTERTID0gKAogICAgImJlZm9yZV9pbWFnZSIsCiAgICAiYWZ0ZXJfaW1hZ2Ui'
        'LAogICAgImJlZm9yZV9zY2VuZV9qc29uIiwKICAgICJhZnRlcl9zY2VuZV9qc29uIiwKICAgICJp'
        'bnN0YW5jZV9tYXNrc19iZWZvcmUiLAogICAgImluc3RhbmNlX21hc2tzX2FmdGVyIiwKKQpTVUlU'
        'RVMgPSBmcm96ZW5zZXQoeyJhdG9taWNfaWQiLCAiY29nZW50X29vZCIsICJoYXJkX2Rpc3RyYWN0'
        'b3IiLCAibm9fZWRpdCJ9KQoKCmNsYXNzIF9TdHJlYW1pbmdKc29uUmVhZGVyOgogICAgZGVmIF9f'
        'aW5pdF9fKHNlbGYsIHBhdGg6IFBhdGgsICosIGNodW5rX3NpemU6IGludCA9IDEwMjQgKiAxMDI0'
        'KSAtPiBOb25lOgogICAgICAgIHNlbGYuX2hhbmRsZSA9IHBhdGgub3BlbigiciIsIGVuY29kaW5n'
        'PSJ1dGYtOCIpCiAgICAgICAgc2VsZi5fY2h1bmtfc2l6ZSA9IGNodW5rX3NpemUKICAgICAgICBz'
        'ZWxmLl9idWZmZXIgPSAiIgogICAgICAgIHNlbGYuX3Bvc2l0aW9uID0gMAogICAgICAgIHNlbGYu'
        'X2VvZiA9IEZhbHNlCiAgICAgICAgc2VsZi5fZGVjb2RlciA9IGpzb24uSlNPTkRlY29kZXIoKQoK'
        'ICAgIGRlZiBjbG9zZShzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuX2hhbmRsZS5jbG9zZSgp'
        'CgogICAgZGVmIF9jb21wYWN0KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgc2VsZi5fcG9zaXRp'
        'b246CiAgICAgICAgICAgIHNlbGYuX2J1ZmZlciA9IHNlbGYuX2J1ZmZlcltzZWxmLl9wb3NpdGlv'
        'biA6XQogICAgICAgICAgICBzZWxmLl9wb3NpdGlvbiA9IDAKCiAgICBkZWYgX2ZpbGwoc2VsZikg'
        'LT4gYm9vbDoKICAgICAgICBpZiBzZWxmLl9lb2Y6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQog'
        'ICAgICAgIHNlbGYuX2NvbXBhY3QoKQogICAgICAgIGNodW5rID0gc2VsZi5faGFuZGxlLnJlYWQo'
        'c2VsZi5fY2h1bmtfc2l6ZSkKICAgICAgICBpZiBub3QgY2h1bms6CiAgICAgICAgICAgIHNlbGYu'
        'X2VvZiA9IFRydWUKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgc2VsZi5fYnVmZmVy'
        'ICs9IGNodW5rCiAgICAgICAgcmV0dXJuIFRydWUKCiAgICBkZWYgc2tpcF93aGl0ZXNwYWNlKHNl'
        'bGYpIC0+IE5vbmU6CiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgd2hpbGUgc2VsZi5f'
        'cG9zaXRpb24gPCBsZW4oc2VsZi5fYnVmZmVyKSBhbmQgc2VsZi5fYnVmZmVyW3NlbGYuX3Bvc2l0'
        'aW9uXS5pc3NwYWNlKCk6CiAgICAgICAgICAgICAgICBzZWxmLl9wb3NpdGlvbiArPSAxCiAgICAg'
        'ICAgICAgIGlmIHNlbGYuX3Bvc2l0aW9uIDwgbGVuKHNlbGYuX2J1ZmZlcikgb3Igbm90IHNlbGYu'
        'X2ZpbGwoKToKICAgICAgICAgICAgICAgIHJldHVybgoKICAgIGRlZiBwZWVrKHNlbGYpIC0+IHN0'
        'cjoKICAgICAgICBzZWxmLnNraXBfd2hpdGVzcGFjZSgpCiAgICAgICAgaWYgc2VsZi5fcG9zaXRp'
        'b24gPj0gbGVuKHNlbGYuX2J1ZmZlcik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInVu'
        'ZXhwZWN0ZWQgZW5kIG9mIEpTT04gaW5wdXQiKQogICAgICAgIHJldHVybiBzZWxmLl9idWZmZXJb'
        'c2VsZi5fcG9zaXRpb25dCgogICAgZGVmIGV4cGVjdChzZWxmLCB0b2tlbjogc3RyKSAtPiBOb25l'
        'OgogICAgICAgIGlmIHNlbGYucGVlaygpICE9IHRva2VuOgogICAgICAgICAgICByYWlzZSBWYWx1'
        'ZUVycm9yKGYiZXhwZWN0ZWQge3Rva2VuIXJ9IGluIEpTT04gaW5wdXQiKQogICAgICAgIHNlbGYu'
        'X3Bvc2l0aW9uICs9IDEKCiAgICBkZWYgZGVjb2RlKHNlbGYpIC0+IEFueToKICAgICAgICBzZWxm'
        'LnNraXBfd2hpdGVzcGFjZSgpCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgdHJ5Ogog'
        'ICAgICAgICAgICAgICAgdmFsdWUsIGVuZCA9IHNlbGYuX2RlY29kZXIucmF3X2RlY29kZShzZWxm'
        'Ll9idWZmZXIsIHNlbGYuX3Bvc2l0aW9uKQogICAgICAgICAgICBleGNlcHQganNvbi5KU09ORGVj'
        'b2RlRXJyb3IgYXMgZXJyb3I6CiAgICAgICAgICAgICAgICBpZiBzZWxmLl9maWxsKCk6CiAgICAg'
        'ICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3Io'
        'ImludmFsaWQgb3IgdHJ1bmNhdGVkIEpTT04gaW5wdXQiKSBmcm9tIGVycm9yCiAgICAgICAgICAg'
        'IHNlbGYuX3Bvc2l0aW9uID0gZW5kCiAgICAgICAgICAgIHJldHVybiB2YWx1ZQoKICAgIGRlZiBl'
        'bnN1cmVfZW5kKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5za2lwX3doaXRlc3BhY2UoKQog'
        'ICAgICAgIGlmIHNlbGYuX3Bvc2l0aW9uIDwgbGVuKHNlbGYuX2J1ZmZlcikgb3Igc2VsZi5fZmls'
        'bCgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ1bmV4cGVjdGVkIGNvbnRlbnQgYWZ0'
        'ZXIgdG9wLWxldmVsIEpTT04gb2JqZWN0IikKCgpkZWYgaXRlcl9zcGxpdF9yZWNvcmRzKAogICAg'
        'cGF0aDogUGF0aCwKICAgICosCiAgICBjaHVua19zaXplOiBpbnQgPSAxMDI0ICogMTAyNCwKKSAt'
        'PiBJdGVyYXRvclt0dXBsZVtzdHIsIGludCwgQW55XV06CiAgICAiIiJTdHJlYW0gcmVjb3JkcyBm'
        'cm9tIHRoZSBsYXJnZSB0b3AtbGV2ZWwgc3BsaXQgb2JqZWN0IHdpdGhvdXQgbG9hZGluZyBpdCBh'
        'bGwuIiIiCgogICAgcmVhZGVyID0gX1N0cmVhbWluZ0pzb25SZWFkZXIocGF0aCwgY2h1bmtfc2l6'
        'ZT1jaHVua19zaXplKQogICAgdHJ5OgogICAgICAgIHJlYWRlci5leHBlY3QoInsiKQogICAgICAg'
        'IGlmIHJlYWRlci5wZWVrKCkgPT0gIn0iOgogICAgICAgICAgICByZWFkZXIuZXhwZWN0KCJ9IikK'
        'ICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgc3BsaXRf'
        'bmFtZSA9IHJlYWRlci5kZWNvZGUoKQogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShzcGxp'
        'dF9uYW1lLCBzdHIpOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic3BsaXQgcmVn'
        'aXN0cnkga2V5cyBtdXN0IGJlIHN0cmluZ3MiKQogICAgICAgICAgICByZWFkZXIuZXhwZWN0KCI6'
        'IikKICAgICAgICAgICAgcmVhZGVyLmV4cGVjdCgiWyIpCiAgICAgICAgICAgIGluZGV4ID0gMAog'
        'ICAgICAgICAgICBpZiByZWFkZXIucGVlaygpICE9ICJdIjoKICAgICAgICAgICAgICAgIHdoaWxl'
        'IFRydWU6CiAgICAgICAgICAgICAgICAgICAgeWllbGQgc3BsaXRfbmFtZSwgaW5kZXgsIHJlYWRl'
        'ci5kZWNvZGUoKQogICAgICAgICAgICAgICAgICAgIGluZGV4ICs9IDEKICAgICAgICAgICAgICAg'
        'ICAgICBkZWxpbWl0ZXIgPSByZWFkZXIucGVlaygpCiAgICAgICAgICAgICAgICAgICAgaWYgZGVs'
        'aW1pdGVyID09ICJdIjoKICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAg'
        'ICAgICAgICByZWFkZXIuZXhwZWN0KCIsIikKICAgICAgICAgICAgcmVhZGVyLmV4cGVjdCgiXSIp'
        'CiAgICAgICAgICAgIGRlbGltaXRlciA9IHJlYWRlci5wZWVrKCkKICAgICAgICAgICAgaWYgZGVs'
        'aW1pdGVyID09ICJ9IjoKICAgICAgICAgICAgICAgIHJlYWRlci5leHBlY3QoIn0iKQogICAgICAg'
        'ICAgICAgICAgYnJlYWsKICAgICAgICAgICAgcmVhZGVyLmV4cGVjdCgiLCIpCiAgICAgICAgcmVh'
        'ZGVyLmVuc3VyZV9lbmQoKQogICAgZmluYWxseToKICAgICAgICByZWFkZXIuY2xvc2UoKQoKCmNs'
        'YXNzIEVkaXRDTEVWUlBhaXIoU3RyaWN0TW9kZWwpOgogICAgcGFpcl9pZDogc3RyID0gRmllbGQo'
        'bWluX2xlbmd0aD0xKQogICAgc3VpdGU6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSkKICAgIGVk'
        'aXRfZmFjdG9yOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEpCiAgICBiZWZvcmVfaW1hZ2U6IHN0'
        'cgogICAgYWZ0ZXJfaW1hZ2U6IHN0cgogICAgYmVmb3JlX3NjZW5lX2pzb246IHN0cgogICAgYWZ0'
        'ZXJfc2NlbmVfanNvbjogc3RyCiAgICBpbnN0YW5jZV9tYXNrc19iZWZvcmU6IHN0cgogICAgaW5z'
        'dGFuY2VfbWFza3NfYWZ0ZXI6IHN0cgoKICAgIEBtb2RlbF92YWxpZGF0b3IobW9kZT0iYWZ0ZXIi'
        'KQogICAgZGVmIHZhbGlkX3BhdGhzKHNlbGYpIC0+IFNlbGY6CiAgICAgICAgaWYgc2VsZi5zdWl0'
        'ZSBub3QgaW4gU1VJVEVTOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBF'
        'ZGl0Q0xFVlIgc3VpdGU6IHtzZWxmLnN1aXRlfSIpCiAgICAgICAgZm9yIGZpZWxkIGluIFBBWUxP'
        'QURfRklFTERTOgogICAgICAgICAgICByYXcgPSBnZXRhdHRyKHNlbGYsIGZpZWxkKQogICAgICAg'
        'ICAgICBwYXRoID0gUHVyZVBvc2l4UGF0aChyYXcpCiAgICAgICAgICAgIGlmIHBhdGguaXNfYWJz'
        'b2x1dGUoKSBvciAiLi4iIGluIHBhdGgucGFydHMgb3Igbm90IHBhdGgucGFydHM6CiAgICAgICAg'
        'ICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5zYWZlIEVkaXRDTEVWUiBwYXRoIGluIHtmaWVs'
        'ZH0iKQogICAgICAgICAgICBpZiBwYXRoLnBhcnRzWzBdICE9IHNlbGYuc3VpdGU6CiAgICAgICAg'
        'ICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRWRpdENMRVZSIHBhdGggaW4ge2ZpZWxkfSBkb2Vz'
        'IG5vdCBtYXRjaCBzdWl0ZSIpCiAgICAgICAgcmV0dXJuIHNlbGYKCgpAZGF0YWNsYXNzKGZyb3pl'
        'bj1UcnVlLCBzbG90cz1UcnVlKQpjbGFzcyBFZGl0Q0xFVlJQYWlyUGF0aHM6CiAgICBwYWlyOiBF'
        'ZGl0Q0xFVlJQYWlyCiAgICBiZWZvcmVfaW1hZ2U6IFBhdGgKICAgIGFmdGVyX2ltYWdlOiBQYXRo'
        'CiAgICBiZWZvcmVfc2NlbmVfanNvbjogUGF0aAogICAgYWZ0ZXJfc2NlbmVfanNvbjogUGF0aAog'
        'ICAgaW5zdGFuY2VfbWFza3NfYmVmb3JlOiBQYXRoCiAgICBpbnN0YW5jZV9tYXNrc19hZnRlcjog'
        'UGF0aAoKCmRlZiBfaW1hZ2VfdGVuc29yKHBhdGg6IFBhdGgpIC0+IFRlbnNvcjoKICAgIHdpdGgg'
        'SW1hZ2Uub3BlbihwYXRoKSBhcyBpbWFnZToKICAgICAgICByZ2IgPSBucC5hcnJheShpbWFnZS5j'
        'b252ZXJ0KCJSR0IiKSwgZHR5cGU9bnAudWludDgsIGNvcHk9VHJ1ZSkKICAgIHJldHVybiB0b3Jj'
        'aC5mcm9tX251bXB5KHJnYikucGVybXV0ZSgyLCAwLCAxKS50byhkdHlwZT10b3JjaC5mbG9hdDMy'
        'KS5kaXZfKDI1NS4wKQoKCmNsYXNzIEVkaXRDTEVWUkRhdGFzZXQoU2VxdWVuY2VbRWRpdENMRVZS'
        'UGFpclBhdGhzXSk6CiAgICAiIiJTcGxpdC1zY29wZWQgcGF0aHMgcGx1cyBkZXRlcm1pbmlzdGlj'
        'IFJHQiwgc2NlbmUsIGFuZCBtYXNrIGRlY29kZXJzLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAg'
        'ICAgICBzZWxmLAogICAgICAgIHBheWxvYWRfcm9vdDogUGF0aCwKICAgICAgICBzcGxpdF9yZWdp'
        'c3RyeTogUGF0aCwKICAgICAgICBzcGxpdDogc3RyLAogICAgICAgICosCiAgICAgICAgdmVyaWZ5'
        'X2ZpbGVzOiBib29sID0gVHJ1ZSwKICAgICkgLT4gTm9uZToKICAgICAgICBzZWxmLnBheWxvYWRf'
        'cm9vdCA9IHBheWxvYWRfcm9vdC5yZXNvbHZlKCkKICAgICAgICBpZiBwYXlsb2FkX3Jvb3QuaXNf'
        'c3ltbGluaygpIG9yIG5vdCBzZWxmLnBheWxvYWRfcm9vdC5pc19kaXIoKToKICAgICAgICAgICAg'
        'cmFpc2UgVmFsdWVFcnJvcigiRWRpdENMRVZSIHBheWxvYWQgcm9vdCBtdXN0IGJlIGEgcmVndWxh'
        'ciBkaXJlY3RvcnkiKQogICAgICAgIHNlbGYuc3BsaXRfcmVnaXN0cnkgPSBzcGxpdF9yZWdpc3Ry'
        'eS5yZXNvbHZlKCkKICAgICAgICBpZiBzcGxpdF9yZWdpc3RyeS5pc19zeW1saW5rKCkgb3Igbm90'
        'IHNlbGYuc3BsaXRfcmVnaXN0cnkuaXNfZmlsZSgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVy'
        'cm9yKCJFZGl0Q0xFVlIgc3BsaXQgcmVnaXN0cnkgbXVzdCBiZSBhIHJlZ3VsYXIgZmlsZSIpCiAg'
        'ICAgICAgc2VsZi5zcGxpdCA9IHNwbGl0CiAgICAgICAgcGFpcnM6IGxpc3RbRWRpdENMRVZSUGFp'
        'cl0gPSBbXQogICAgICAgIGtub3duX3NwbGl0czogc2V0W3N0cl0gPSBzZXQoKQogICAgICAgIGZv'
        'ciBzcGxpdF9uYW1lLCBpbmRleCwgcmF3IGluIGl0ZXJfc3BsaXRfcmVjb3JkcyhzZWxmLnNwbGl0'
        'X3JlZ2lzdHJ5KToKICAgICAgICAgICAga25vd25fc3BsaXRzLmFkZChzcGxpdF9uYW1lKQogICAg'
        'ICAgICAgICBpZiBzcGxpdF9uYW1lICE9IHNwbGl0OgogICAgICAgICAgICAgICAgY29udGludWUK'
        'ICAgICAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UocmF3LCBkaWN0KToKICAgICAgICAgICAgICAg'
        'IHJhaXNlIFZhbHVlRXJyb3IoZiJ7c3BsaXRfbmFtZX1be2luZGV4fV0gaXMgbm90IGFuIG9iamVj'
        'dCIpCiAgICAgICAgICAgIGZpZWxkcyA9ICgicGFpcl9pZCIsICJzdWl0ZSIsICJlZGl0X2ZhY3Rv'
        'ciIsICpQQVlMT0FEX0ZJRUxEUykKICAgICAgICAgICAgc2VsZWN0ZWQgPSB7bmFtZTogcmF3Lmdl'
        'dChuYW1lKSBmb3IgbmFtZSBpbiBmaWVsZHN9CiAgICAgICAgICAgIHBhaXJzLmFwcGVuZChFZGl0'
        'Q0xFVlJQYWlyLm1vZGVsX3ZhbGlkYXRlKHNlbGVjdGVkKSkKICAgICAgICBpZiBzcGxpdCBub3Qg'
        'aW4ga25vd25fc3BsaXRzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBF'
        'ZGl0Q0xFVlIgc3BsaXQ6IHtzcGxpdH0iKQogICAgICAgIGlmIG5vdCBwYWlyczoKICAgICAgICAg'
        'ICAgcmFpc2UgVmFsdWVFcnJvcihmIkVkaXRDTEVWUiBzcGxpdCBpcyBlbXB0eToge3NwbGl0fSIp'
        'CiAgICAgICAgc2VsZi5fcGFpcnMgPSB0dXBsZShwYWlycykKICAgICAgICBpZiB2ZXJpZnlfZmls'
        'ZXM6CiAgICAgICAgICAgIGZvciBpbmRleCBpbiByYW5nZShsZW4oc2VsZi5fcGFpcnMpKToKICAg'
        'ICAgICAgICAgICAgIHNlbGYuX3Jlc29sdmUoaW5kZXgpCgogICAgZGVmIF9fbGVuX18oc2VsZikg'
        'LT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi5fcGFpcnMpCgogICAgQG92ZXJsb2FkCiAg'
        'ICBkZWYgX19nZXRpdGVtX18oc2VsZiwgaW5kZXg6IGludCkgLT4gRWRpdENMRVZSUGFpclBhdGhz'
        'OiAuLi4KCiAgICBAb3ZlcmxvYWQKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpbmRleDogc2xp'
        'Y2UpIC0+IFNlcXVlbmNlW0VkaXRDTEVWUlBhaXJQYXRoc106IC4uLgoKICAgIGRlZiBfX2dldGl0'
        'ZW1fXyhzZWxmLCBpbmRleDogaW50IHwgc2xpY2UpIC0+IEVkaXRDTEVWUlBhaXJQYXRocyB8IFNl'
        'cXVlbmNlW0VkaXRDTEVWUlBhaXJQYXRoc106CiAgICAgICAgaWYgaXNpbnN0YW5jZShpbmRleCwg'
        'c2xpY2UpOgogICAgICAgICAgICByZXR1cm4gdHVwbGUoc2VsZi5fcmVzb2x2ZShpdGVtKSBmb3Ig'
        'aXRlbSBpbiByYW5nZSgqaW5kZXguaW5kaWNlcyhsZW4oc2VsZikpKSkKICAgICAgICBpZiBpbmRl'
        'eCA8IDA6CiAgICAgICAgICAgIGluZGV4ICs9IGxlbihzZWxmLl9wYWlycykKICAgICAgICBpZiBp'
        'bmRleCA8IDAgb3IgaW5kZXggPj0gbGVuKHNlbGYuX3BhaXJzKToKICAgICAgICAgICAgcmFpc2Ug'
        'SW5kZXhFcnJvcihpbmRleCkKICAgICAgICByZXR1cm4gc2VsZi5fcmVzb2x2ZShpbmRleCkKCiAg'
        'ICBkZWYgX3BhdGgoc2VsZiwgcmF3OiBzdHIpIC0+IFBhdGg6CiAgICAgICAgY2FuZGlkYXRlID0g'
        'c2VsZi5wYXlsb2FkX3Jvb3Quam9pbnBhdGgoKlB1cmVQb3NpeFBhdGgocmF3KS5wYXJ0cykKICAg'
        'ICAgICBpZiBjYW5kaWRhdGUuaXNfc3ltbGluaygpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVy'
        'cm9yKGYiRWRpdENMRVZSIHBheWxvYWQgbWVtYmVyIGlzIGEgc3ltbGluazoge3Jhd30iKQogICAg'
        'ICAgIHJlc29sdmVkID0gY2FuZGlkYXRlLnJlc29sdmUoKQogICAgICAgIGlmIG5vdCByZXNvbHZl'
        'ZC5pc19yZWxhdGl2ZV90byhzZWxmLnBheWxvYWRfcm9vdCkgb3Igbm90IHJlc29sdmVkLmlzX2Zp'
        'bGUoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkVkaXRDTEVWUiBwYXlsb2FkIG1l'
        'bWJlciBpcyBtaXNzaW5nIG9yIGVzY2FwZXMgdGhlIHJvb3Q6IHtyYXd9IikKICAgICAgICByZXR1'
        'cm4gcmVzb2x2ZWQKCiAgICBkZWYgX3Jlc29sdmUoc2VsZiwgaW5kZXg6IGludCkgLT4gRWRpdENM'
        'RVZSUGFpclBhdGhzOgogICAgICAgIHBhaXIgPSBzZWxmLl9wYWlyc1tpbmRleF0KICAgICAgICBy'
        'ZXR1cm4gRWRpdENMRVZSUGFpclBhdGhzKAogICAgICAgICAgICBwYWlyPXBhaXIsCiAgICAgICAg'
        'ICAgIGJlZm9yZV9pbWFnZT1zZWxmLl9wYXRoKHBhaXIuYmVmb3JlX2ltYWdlKSwKICAgICAgICAg'
        'ICAgYWZ0ZXJfaW1hZ2U9c2VsZi5fcGF0aChwYWlyLmFmdGVyX2ltYWdlKSwKICAgICAgICAgICAg'
        'YmVmb3JlX3NjZW5lX2pzb249c2VsZi5fcGF0aChwYWlyLmJlZm9yZV9zY2VuZV9qc29uKSwKICAg'
        'ICAgICAgICAgYWZ0ZXJfc2NlbmVfanNvbj1zZWxmLl9wYXRoKHBhaXIuYWZ0ZXJfc2NlbmVfanNv'
        'biksCiAgICAgICAgICAgIGluc3RhbmNlX21hc2tzX2JlZm9yZT1zZWxmLl9wYXRoKHBhaXIuaW5z'
        'dGFuY2VfbWFza3NfYmVmb3JlKSwKICAgICAgICAgICAgaW5zdGFuY2VfbWFza3NfYWZ0ZXI9c2Vs'
        'Zi5fcGF0aChwYWlyLmluc3RhbmNlX21hc2tzX2FmdGVyKSwKICAgICAgICApCgogICAgZGVmIGxv'
        'YWRfcmdiX3BhaXIoc2VsZiwgaW5kZXg6IGludCkgLT4gdHVwbGVbVGVuc29yLCBUZW5zb3JdOgog'
        'ICAgICAgIHBhdGhzID0gc2VsZltpbmRleF0KICAgICAgICByZXR1cm4gX2ltYWdlX3RlbnNvcihw'
        'YXRocy5iZWZvcmVfaW1hZ2UpLCBfaW1hZ2VfdGVuc29yKHBhdGhzLmFmdGVyX2ltYWdlKQoKICAg'
        'IGRlZiBsb2FkX3NjZW5lcyhzZWxmLCBpbmRleDogaW50KSAtPiB0dXBsZVtkaWN0W3N0ciwgQW55'
        'XSwgZGljdFtzdHIsIEFueV1dOgogICAgICAgIHBhdGhzID0gc2VsZltpbmRleF0KICAgICAgICB2'
        'YWx1ZXM6IGxpc3RbZGljdFtzdHIsIEFueV1dID0gW10KICAgICAgICBmb3IgcGF0aCBpbiAocGF0'
        'aHMuYmVmb3JlX3NjZW5lX2pzb24sIHBhdGhzLmFmdGVyX3NjZW5lX2pzb24pOgogICAgICAgICAg'
        'ICB2YWx1ZSA9IGpzb24ubG9hZHMocGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9InV0Zi04IikpCiAg'
        'ICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlLCBkaWN0KToKICAgICAgICAgICAgICAg'
        'IHJhaXNlIFZhbHVlRXJyb3IoZiJFZGl0Q0xFVlIgc2NlbmUgbXVzdCBiZSBhbiBvYmplY3Q6IHtw'
        'YXRofSIpCiAgICAgICAgICAgIHZhbHVlcy5hcHBlbmQodmFsdWUpCiAgICAgICAgcmV0dXJuIHZh'
        'bHVlc1swXSwgdmFsdWVzWzFdCgogICAgZGVmIGxvYWRfbWFza3Moc2VsZiwgaW5kZXg6IGludCkg'
        'LT4gdHVwbGVbZGljdFtzdHIsIG5wLm5kYXJyYXlbQW55LCBBbnldXSwgLi4uXToKICAgICAgICBw'
        'YXRocyA9IHNlbGZbaW5kZXhdCiAgICAgICAgcmVzdWx0czogbGlzdFtkaWN0W3N0ciwgbnAubmRh'
        'cnJheVtBbnksIEFueV1dXSA9IFtdCiAgICAgICAgZm9yIHBhdGggaW4gKHBhdGhzLmluc3RhbmNl'
        'X21hc2tzX2JlZm9yZSwgcGF0aHMuaW5zdGFuY2VfbWFza3NfYWZ0ZXIpOgogICAgICAgICAgICB3'
        'aXRoIG5wLmxvYWQocGF0aCwgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyBhcmNoaXZlOgogICAgICAg'
        'ICAgICAgICAgcmVzdWx0cy5hcHBlbmQoe25hbWU6IG5wLmFycmF5KGFyY2hpdmVbbmFtZV0sIGNv'
        'cHk9VHJ1ZSkgZm9yIG5hbWUgaW4gYXJjaGl2ZS5maWxlc30pCiAgICAgICAgcmV0dXJuIHR1cGxl'
        'KHJlc3VsdHMpCg=='
    ),
    'olympus/data/ingestion.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0'
        'IGh0dHAuY2xpZW50CmltcG9ydCBpcGFkZHJlc3MKaW1wb3J0IGpzb24KaW1wb3J0IHNvY2tldApp'
        'bXBvcnQgc3NsCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHBhdGhsaWIg'
        'aW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQpmcm9tIHVybGxpYi5wYXJzZSBpbXBv'
        'cnQgU3BsaXRSZXN1bHQsIHVybHNwbGl0Cgpmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZAoKZnJv'
        'bSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKZnJvbSBvbHltcHVzLmNv'
        'cmUuc2VjdXJpdHkgaW1wb3J0IFBlcm1pc3Npb25Qb2xpY3kKCl9TQUZFX0NPTlRFTlRfVFlQRVMg'
        'PSBmcm96ZW5zZXQoCiAgICB7ImFwcGxpY2F0aW9uL2pzb24iLCAiYXBwbGljYXRpb24veG1sIiwg'
        'InRleHQvY3N2IiwgInRleHQvcGxhaW4iLCAidGV4dC94bWwifQopCgoKQGRhdGFjbGFzcyhmcm96'
        'ZW49VHJ1ZSkKY2xhc3MgX1Jlc29sdmVkRW5kcG9pbnQ6CiAgICBmYW1pbHk6IGludAogICAgc29j'
        'a3R5cGU6IGludAogICAgcHJvdG86IGludAogICAgc29ja2FkZHI6IEFueQogICAgaXA6IHN0cgoK'
        'CmNsYXNzIF9QaW5uZWRIVFRQU0Nvbm5lY3Rpb24oaHR0cC5jbGllbnQuSFRUUFNDb25uZWN0aW9u'
        'KToKICAgICIiIkhUVFBTIGNvbm5lY3Rpb24gd2hvc2UgVENQIHBlZXIgaXMgZml4ZWQgYmVmb3Jl'
        'IFRMUyBzdGFydHMuIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAg'
        'aG9zdG5hbWU6IHN0ciwKICAgICAgICBwb3J0OiBpbnQsCiAgICAgICAgZW5kcG9pbnQ6IF9SZXNv'
        'bHZlZEVuZHBvaW50LAogICAgICAgICosCiAgICAgICAgdGltZW91dDogZmxvYXQsCiAgICApIC0+'
        'IE5vbmU6CiAgICAgICAgY29udGV4dCA9IHNzbC5jcmVhdGVfZGVmYXVsdF9jb250ZXh0KCkKICAg'
        'ICAgICBzdXBlcigpLl9faW5pdF9fKGhvc3RuYW1lLCBwb3J0PXBvcnQsIHRpbWVvdXQ9dGltZW91'
        'dCwgY29udGV4dD1jb250ZXh0KQogICAgICAgIHNlbGYuX2VuZHBvaW50ID0gZW5kcG9pbnQKICAg'
        'ICAgICBzZWxmLl9jb25uZWN0X3RpbWVvdXQgPSB0aW1lb3V0CiAgICAgICAgc2VsZi5fdGxzX2Nv'
        'bnRleHQgPSBjb250ZXh0CgogICAgZGVmIGNvbm5lY3Qoc2VsZikgLT4gTm9uZToKICAgICAgICBz'
        'b2NrID0gc29ja2V0LnNvY2tldCgKICAgICAgICAgICAgc2VsZi5fZW5kcG9pbnQuZmFtaWx5LAog'
        'ICAgICAgICAgICBzZWxmLl9lbmRwb2ludC5zb2NrdHlwZSwKICAgICAgICAgICAgc2VsZi5fZW5k'
        'cG9pbnQucHJvdG8sCiAgICAgICAgKQogICAgICAgIHNvY2suc2V0dGltZW91dChzZWxmLl9jb25u'
        'ZWN0X3RpbWVvdXQpCiAgICAgICAgdHJ5OgogICAgICAgICAgICAjIENvbm5lY3QgZGlyZWN0bHkg'
        'dG8gdGhlIGFscmVhZHktdmFsaWRhdGVkIHNvY2thZGRyLiBUaGlzIGRlbGliZXJhdGVseSBhdm9p'
        'ZHMKICAgICAgICAgICAgIyBzb2NrZXQuY3JlYXRlX2Nvbm5lY3Rpb24oKSwgd2hpY2ggd291bGQg'
        'cGVyZm9ybSBhIHNlY29uZCBob3N0bmFtZSByZXNvbHV0aW9uLgogICAgICAgICAgICBzb2NrLmNv'
        'bm5lY3Qoc2VsZi5fZW5kcG9pbnQuc29ja2FkZHIpCiAgICAgICAgICAgIHNlbGYuc29jayA9IHNl'
        'bGYuX3Rsc19jb250ZXh0LndyYXBfc29ja2V0KHNvY2ssIHNlcnZlcl9ob3N0bmFtZT1zZWxmLmhv'
        'c3QpCiAgICAgICAgZXhjZXB0IEJhc2VFeGNlcHRpb246CiAgICAgICAgICAgIHNvY2suY2xvc2Uo'
        'KQogICAgICAgICAgICByYWlzZQoKCmNsYXNzIEluZ2VzdGVkRG9jdW1lbnQoU3RyaWN0TW9kZWwp'
        'OgogICAgaWRlbnRpZmllcjogc3RyCiAgICB0ZXh0OiBzdHIKICAgIGxhbmd1YWdlOiBzdHIgPSAi'
        'dW5rbm93biIKICAgIGNodW5rczogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxp'
        'c3QpCiAgICBtZXRhZGF0YTogZGljdFtzdHIsIHN0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9'
        'ZGljdCkKCgpjbGFzcyBJbmdlc3Rpb25QaXBlbGluZToKICAgIGRlZiBfX2luaXRfXygKICAgICAg'
        'ICBzZWxmLAogICAgICAgIHBvbGljeTogUGVybWlzc2lvblBvbGljeSB8IE5vbmUgPSBOb25lLAog'
        'ICAgICAgICosCiAgICAgICAgbWF4X3Jlc3BvbnNlX2J5dGVzOiBpbnQgPSAxXzAwMF8wMDAsCiAg'
        'ICApIC0+IE5vbmU6CiAgICAgICAgaWYgbWF4X3Jlc3BvbnNlX2J5dGVzIDwgMToKICAgICAgICAg'
        'ICAgcmFpc2UgVmFsdWVFcnJvcigibWF4X3Jlc3BvbnNlX2J5dGVzIG11c3QgYmUgcG9zaXRpdmUi'
        'KQogICAgICAgIHNlbGYucG9saWN5ID0gcG9saWN5IG9yIFBlcm1pc3Npb25Qb2xpY3koKQogICAg'
        'ICAgIHNlbGYubWF4X3Jlc3BvbnNlX2J5dGVzID0gbWF4X3Jlc3BvbnNlX2J5dGVzCgogICAgZGVm'
        'IGluZ2VzdF9maWxlKHNlbGYsIHBhdGg6IFBhdGgsIGNodW5rX3NpemU6IGludCA9IDEyMCkgLT4g'
        'SW5nZXN0ZWREb2N1bWVudDoKICAgICAgICB0ZXh0ID0gcGF0aC5yZWFkX3RleHQoZW5jb2Rpbmc9'
        'InV0Zi04IikKICAgICAgICByZXR1cm4gc2VsZi5fbm9ybWFsaXplKHBhdGgubmFtZSwgdGV4dCwg'
        'Y2h1bmtfc2l6ZSkKCiAgICBkZWYgaW5nZXN0X2h0dHAoc2VsZiwgdXJsOiBzdHIsIGNodW5rX3Np'
        'emU6IGludCA9IDEyMCkgLT4gSW5nZXN0ZWREb2N1bWVudDoKICAgICAgICBpZiBub3Qgc2VsZi5w'
        'b2xpY3kuYWxsb3dfbmV0d29yazoKICAgICAgICAgICAgcmFpc2UgUGVybWlzc2lvbkVycm9yKCJu'
        'ZXR3b3JrIGluZ2VzdGlvbiBpcyBkaXNhYmxlZCIpCiAgICAgICAgcGFyc2VkID0gdXJsc3BsaXQo'
        'dXJsKQogICAgICAgIGhvc3RuYW1lID0gKHBhcnNlZC5ob3N0bmFtZSBvciAiIikubG93ZXIoKS5y'
        'c3RyaXAoIi4iKQogICAgICAgIGlmIHBhcnNlZC5zY2hlbWUgIT0gImh0dHBzIiBvciBub3QgaG9z'
        'dG5hbWUgb3IgcGFyc2VkLnVzZXJuYW1lIG9yIHBhcnNlZC5wYXNzd29yZDoKICAgICAgICAgICAg'
        'cmFpc2UgVmFsdWVFcnJvcigiaW5nZXN0aW9uIFVSTHMgbXVzdCBiZSBjcmVkZW50aWFsLWZyZWUg'
        'SFRUUFMgVVJMcyIpCiAgICAgICAgYWxsb3dlZCA9IHtkb21haW4ubG93ZXIoKS5yc3RyaXAoIi4i'
        'KSBmb3IgZG9tYWluIGluIHNlbGYucG9saWN5LmFsbG93X2h0dHBfZG9tYWluc30KICAgICAgICBp'
        'ZiBob3N0bmFtZSBub3QgaW4gYWxsb3dlZDoKICAgICAgICAgICAgcmFpc2UgUGVybWlzc2lvbkVy'
        'cm9yKGYiaG9zdCBpcyBub3QgYWxsb3dsaXN0ZWQ6IHtob3N0bmFtZX0iKQogICAgICAgIHBvcnQg'
        'PSBwYXJzZWQucG9ydCBvciA0NDMKICAgICAgICBlbmRwb2ludHMgPSBzZWxmLl9yZXNvbHZlX3B1'
        'YmxpY19lbmRwb2ludHMoaG9zdG5hbWUsIHBvcnQpCiAgICAgICAgcGF5bG9hZCwgY2hhcnNldCA9'
        'IHNlbGYuX2ZldGNoX2h0dHBzKHBhcnNlZCwgaG9zdG5hbWUsIHBvcnQsIGVuZHBvaW50cykKICAg'
        'ICAgICB0ZXh0ID0gcGF5bG9hZC5kZWNvZGUoY2hhcnNldCkKICAgICAgICByZXR1cm4gc2VsZi5f'
        'bm9ybWFsaXplKHVybCwgdGV4dCwgY2h1bmtfc2l6ZSkKCiAgICBkZWYgX2ZldGNoX2h0dHBzKAog'
        'ICAgICAgIHNlbGYsCiAgICAgICAgcGFyc2VkOiBTcGxpdFJlc3VsdCwKICAgICAgICBob3N0bmFt'
        'ZTogc3RyLAogICAgICAgIHBvcnQ6IGludCwKICAgICAgICBlbmRwb2ludHM6IHR1cGxlW19SZXNv'
        'bHZlZEVuZHBvaW50LCAuLi5dLAogICAgKSAtPiB0dXBsZVtieXRlcywgc3RyXToKICAgICAgICB0'
        'YXJnZXQgPSBwYXJzZWQucGF0aCBvciAiLyIKICAgICAgICBpZiBwYXJzZWQucXVlcnk6CiAgICAg'
        'ICAgICAgIHRhcmdldCA9IGYie3RhcmdldH0/e3BhcnNlZC5xdWVyeX0iCiAgICAgICAgaGVhZGVy'
        'cyA9IHsKICAgICAgICAgICAgIkFjY2VwdCI6ICIsICIuam9pbihzb3J0ZWQoX1NBRkVfQ09OVEVO'
        'VF9UWVBFUykpLAogICAgICAgICAgICAiVXNlci1BZ2VudCI6ICJPbHltcHVzLzAuMSIsCiAgICAg'
        'ICAgfQogICAgICAgIGxhc3RfZXJyb3I6IEV4Y2VwdGlvbiB8IE5vbmUgPSBOb25lCiAgICAgICAg'
        'Zm9yIGVuZHBvaW50IGluIGVuZHBvaW50czoKICAgICAgICAgICAgY29ubmVjdGlvbiA9IF9QaW5u'
        'ZWRIVFRQU0Nvbm5lY3Rpb24oaG9zdG5hbWUsIHBvcnQsIGVuZHBvaW50LCB0aW1lb3V0PTUuMCkK'
        'ICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgY29ubmVjdGlvbi5yZXF1ZXN0KCJHRVQi'
        'LCB0YXJnZXQsIGhlYWRlcnM9aGVhZGVycykKICAgICAgICAgICAgICAgIHJlc3BvbnNlID0gY29u'
        'bmVjdGlvbi5nZXRyZXNwb25zZSgpCiAgICAgICAgICAgICAgICBpZiAzMDAgPD0gcmVzcG9uc2Uu'
        'c3RhdHVzIDwgNDAwOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkhUVFAg'
        'cmVkaXJlY3RzIGFyZSBkaXNhYmxlZCBmb3IgaW5nZXN0aW9uIikKICAgICAgICAgICAgICAgIGlm'
        'IHJlc3BvbnNlLnN0YXR1cyA8IDIwMCBvciByZXNwb25zZS5zdGF0dXMgPj0gMzAwOgogICAgICAg'
        'ICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJIVFRQIGluZ2VzdGlvbiByZXF1ZXN0IGZh'
        'aWxlZCB3aXRoIHN0YXR1cyB7cmVzcG9uc2Uuc3RhdHVzfSIpCiAgICAgICAgICAgICAgICBjb250'
        'ZW50X3R5cGUgPSByZXNwb25zZS5oZWFkZXJzLmdldF9jb250ZW50X3R5cGUoKS5sb3dlcigpCiAg'
        'ICAgICAgICAgICAgICBpZiBjb250ZW50X3R5cGUgbm90IGluIF9TQUZFX0NPTlRFTlRfVFlQRVM6'
        'CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVuc3VwcG9ydGVkIEhUVFAg'
        'Y29udGVudCB0eXBlOiB7Y29udGVudF90eXBlfSIpCiAgICAgICAgICAgICAgICBsZW5ndGggPSBy'
        'ZXNwb25zZS5oZWFkZXJzLmdldCgiQ29udGVudC1MZW5ndGgiKQogICAgICAgICAgICAgICAgaWYg'
        'bGVuZ3RoIGlzIG5vdCBOb25lIGFuZCBpbnQobGVuZ3RoKSA+IHNlbGYubWF4X3Jlc3BvbnNlX2J5'
        'dGVzOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkhUVFAgcmVzcG9uc2Ug'
        'ZXhjZWVkcyB0aGUgY29uZmlndXJlZCBieXRlIGxpbWl0IikKICAgICAgICAgICAgICAgIHBheWxv'
        'YWQgPSByZXNwb25zZS5yZWFkKHNlbGYubWF4X3Jlc3BvbnNlX2J5dGVzICsgMSkKICAgICAgICAg'
        'ICAgICAgIGlmIGxlbihwYXlsb2FkKSA+IHNlbGYubWF4X3Jlc3BvbnNlX2J5dGVzOgogICAgICAg'
        'ICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkhUVFAgcmVzcG9uc2UgZXhjZWVkcyB0aGUg'
        'Y29uZmlndXJlZCBieXRlIGxpbWl0IikKICAgICAgICAgICAgICAgIGNoYXJzZXQgPSByZXNwb25z'
        'ZS5oZWFkZXJzLmdldF9jb250ZW50X2NoYXJzZXQoInV0Zi04IikKICAgICAgICAgICAgICAgIHJl'
        'dHVybiBwYXlsb2FkLCBjaGFyc2V0CiAgICAgICAgICAgIGV4Y2VwdCAoT1NFcnJvciwgaHR0cC5j'
        'bGllbnQuSFRUUEV4Y2VwdGlvbikgYXMgZXhjOgogICAgICAgICAgICAgICAgbGFzdF9lcnJvciA9'
        'IGV4YwogICAgICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICAgICAgY29ubmVjdGlvbi5jbG9z'
        'ZSgpCiAgICAgICAgaWYgbGFzdF9lcnJvciBpcyBub3QgTm9uZToKICAgICAgICAgICAgcmFpc2Ug'
        'VmFsdWVFcnJvcihmInVuYWJsZSB0byBjb25uZWN0IHRvIGluZ2VzdGlvbiBob3N0OiB7aG9zdG5h'
        'bWV9IikgZnJvbSBsYXN0X2Vycm9yCiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVuYWJsZSB0'
        'byBjb25uZWN0IHRvIGluZ2VzdGlvbiBob3N0OiB7aG9zdG5hbWV9IikKCiAgICBAc3RhdGljbWV0'
        'aG9kCiAgICBkZWYgX3Jlc29sdmVfcHVibGljX2VuZHBvaW50cyhob3N0bmFtZTogc3RyLCBwb3J0'
        'OiBpbnQpIC0+IHR1cGxlW19SZXNvbHZlZEVuZHBvaW50LCAuLi5dOgogICAgICAgIHRyeToKICAg'
        'ICAgICAgICAgYWRkcmVzc2VzID0gc29ja2V0LmdldGFkZHJpbmZvKGhvc3RuYW1lLCBwb3J0LCB0'
        'eXBlPXNvY2tldC5TT0NLX1NUUkVBTSkKICAgICAgICBleGNlcHQgc29ja2V0LmdhaWVycm9yIGFz'
        'IGV4YzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInVuYWJsZSB0byByZXNvbHZlIGlu'
        'Z2VzdGlvbiBob3N0OiB7aG9zdG5hbWV9IikgZnJvbSBleGMKICAgICAgICBpZiBub3QgYWRkcmVz'
        'c2VzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5hYmxlIHRvIHJlc29sdmUgaW5n'
        'ZXN0aW9uIGhvc3Q6IHtob3N0bmFtZX0iKQoKICAgICAgICBlbmRwb2ludHM6IGxpc3RbX1Jlc29s'
        'dmVkRW5kcG9pbnRdID0gW10KICAgICAgICBmb3IgZmFtaWx5LCBzb2NrdHlwZSwgcHJvdG8sIF9j'
        'YW5vbmljYWxfbmFtZSwgc29ja2FkZHIgaW4gYWRkcmVzc2VzOgogICAgICAgICAgICBpcCA9IGlw'
        'YWRkcmVzcy5pcF9hZGRyZXNzKHNvY2thZGRyWzBdKQogICAgICAgICAgICBpZiBub3QgaXAuaXNf'
        'Z2xvYmFsOgogICAgICAgICAgICAgICAgcmFpc2UgUGVybWlzc2lvbkVycm9yKGYiaW5nZXN0aW9u'
        'IGhvc3QgcmVzb2x2ZXMgdG8gYSBub24tcHVibGljIGFkZHJlc3M6IHtpcH0iKQogICAgICAgICAg'
        'ICBlbmRwb2ludHMuYXBwZW5kKAogICAgICAgICAgICAgICAgX1Jlc29sdmVkRW5kcG9pbnQoCiAg'
        'ICAgICAgICAgICAgICAgICAgZmFtaWx5PWZhbWlseSwKICAgICAgICAgICAgICAgICAgICBzb2Nr'
        'dHlwZT1zb2NrdHlwZSwKICAgICAgICAgICAgICAgICAgICBwcm90bz1wcm90bywKICAgICAgICAg'
        'ICAgICAgICAgICBzb2NrYWRkcj1zb2NrYWRkciwKICAgICAgICAgICAgICAgICAgICBpcD1zdHIo'
        'aXApLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIHR1cGxl'
        'KGVuZHBvaW50cykKCiAgICBkZWYgX25vcm1hbGl6ZShzZWxmLCBpZGVudGlmaWVyOiBzdHIsIHRl'
        'eHQ6IHN0ciwgY2h1bmtfc2l6ZTogaW50KSAtPiBJbmdlc3RlZERvY3VtZW50OgogICAgICAgIG5v'
        'cm1hbGl6ZWQgPSAiICIuam9pbih0ZXh0LnNwbGl0KCkpCiAgICAgICAgY2h1bmtzID0gWwogICAg'
        'ICAgICAgICBub3JtYWxpemVkW2luZGV4IDogaW5kZXggKyBjaHVua19zaXplXQogICAgICAgICAg'
        'ICBmb3IgaW5kZXggaW4gcmFuZ2UoMCwgbGVuKG5vcm1hbGl6ZWQpLCBjaHVua19zaXplKQogICAg'
        'ICAgIF0KICAgICAgICByZXR1cm4gSW5nZXN0ZWREb2N1bWVudCgKICAgICAgICAgICAgaWRlbnRp'
        'Zmllcj1pZGVudGlmaWVyLAogICAgICAgICAgICB0ZXh0PW5vcm1hbGl6ZWQsCiAgICAgICAgICAg'
        'IGxhbmd1YWdlPSJlbiIgaWYgbm9ybWFsaXplZC5pc2FzY2lpKCkgZWxzZSAibWl4ZWQiLAogICAg'
        'ICAgICAgICBjaHVua3M9Y2h1bmtzLAogICAgICAgICAgICBtZXRhZGF0YT17InNoYTI1NiI6IGhh'
        'c2hsaWIuc2hhMjU2KG5vcm1hbGl6ZWQuZW5jb2RlKCJ1dGYtOCIpKS5oZXhkaWdlc3QoKX0sCiAg'
        'ICAgICAgKQoKICAgIGRlZiBzaGFyZChzZWxmLCBkb2N1bWVudHM6IGxpc3RbSW5nZXN0ZWREb2N1'
        'bWVudF0sIG91dHB1dF9wYXRoOiBQYXRoKSAtPiBOb25lOgogICAgICAgIG91dHB1dF9wYXRoLndy'
        'aXRlX3RleHQoCiAgICAgICAgICAgIGpzb24uZHVtcHMoW2RvY3VtZW50Lm1vZGVsX2R1bXAobW9k'
        'ZT0ianNvbiIpIGZvciBkb2N1bWVudCBpbiBkb2N1bWVudHNdLCBpbmRlbnQ9MiksCiAgICAgICAg'
        'ICAgIGVuY29kaW5nPSJ1dGYtOCIsCiAgICAgICAgKQo='
    ),
    'olympus/data/manifests.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBweWRhbnRpYyBpbXBvcnQg'
        'RmllbGQsIG1vZGVsX3ZhbGlkYXRvcgoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQg'
        'U3RyaWN0TW9kZWwKCgpjbGFzcyBEYXRhc2V0U3BsaXQoU3RyaWN0TW9kZWwpOgogICAgdHJhaW46'
        'IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICB2YWxpZGF0aW9uOiBmbG9hdCA9IEZp'
        'ZWxkKGdlPTAuMCwgbGU9MS4wKQogICAgdGVzdDogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEu'
        'MCkKCiAgICBAbW9kZWxfdmFsaWRhdG9yKG1vZGU9ImFmdGVyIikKICAgIGRlZiB2YWxpZGF0ZV90'
        'b3RhbChzZWxmKSAtPiBEYXRhc2V0U3BsaXQ6CiAgICAgICAgaWYgYWJzKChzZWxmLnRyYWluICsg'
        'c2VsZi52YWxpZGF0aW9uICsgc2VsZi50ZXN0KSAtIDEuMCkgPiAxZS05OgogICAgICAgICAgICBy'
        'YWlzZSBWYWx1ZUVycm9yKCJkYXRhc2V0IHNwbGl0IGZyYWN0aW9ucyBtdXN0IHN1bSB0byAxLjAi'
        'KQogICAgICAgIHJldHVybiBzZWxmCgoKY2xhc3MgRGF0YXNldE1hbmlmZXN0KFN0cmljdE1vZGVs'
        'KToKICAgIG5hbWU6IHN0cgogICAgdmVyc2lvbjogc3RyCiAgICBzcGxpdDogRGF0YXNldFNwbGl0'
        'CiAgICBzb3VyY2VzOiBsaXN0W3N0cl0KICAgIGZpbHRlcnM6IGxpc3Rbc3RyXSA9IEZpZWxkKGRl'
        'ZmF1bHRfZmFjdG9yeT1saXN0KQogICAgZGVkdXBsaWNhdGlvbjogbGlzdFtzdHJdID0gRmllbGQo'
        'ZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICB3ZWlnaHRzOiBkaWN0W3N0ciwgZmxvYXRdID0gRmll'
        'bGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCiAgICBsaWNlbnNlczogbGlzdFtzdHJdID0gRmllbGQo'
        'ZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBjb250YW1pbmF0aW9uX3J1bGVzOiBsaXN0W3N0cl0g'
        'PSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHRyYW5zZm9ybWF0aW9uczogbGlzdFtz'
        'dHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCg=='
    ),
    'olympus/data/olympus_dataset.py': (
        '...'
    ),
    'olympus/data/oracle_fusion_v4.py': (
        'IiIiRnJlc2ggdjQgbWVjaGFuaXNtIGRhdGE7IHRhcmdldC1mcmVlIGRvbWFpbiBtZXRhZGF0YSBh'
        'bmQgY2xlYW4gZHluYW1pY3MuCgpUaGlzIGlzIGEgY29udHJvbGxlZCBzeW50aGV0aWMgZGlhZ25v'
        'c3RpYywgbm90IGEgbmF0dXJhbC1kYXRhIGJlbmNobWFyay4KT25seSB0aGUgZXhwbGljaXRseSBw'
        'cml2aWxlZ2VkIG9yYWNsZSBjb25zdW1lcyBjb3JydXB0aW9uIG1ldGFkYXRhIG9yIGxhYmVscy4K'
        'IiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKZnJv'
        'bSB0b3JjaCBpbXBvcnQgVGVuc29yCgoKZGVmIGR5bmFtaWNzKHN0YXRlOiBUZW5zb3IsIGFjdGlv'
        'bjogVGVuc29yKSAtPiBUZW5zb3I6CiAgICBwYWRkZWQgPSB0b3JjaC5ubi5mdW5jdGlvbmFsLnBh'
        'ZChhY3Rpb24sICgwLCAyKSkKICAgIHJldHVybiAwLjgyICogc3RhdGUgKyAwLjE4ICogc3RhdGUu'
        'cm9sbCgxLCAtMSkudGFuaCgpICsgMC4xMiAqIHBhZGRlZAoKCmRlZiBnZW5lcmF0ZShzZWVkOiBp'
        'bnQsIGNvdW50OiBpbnQsIHJlZ2ltZTogc3RyKSAtPiBkaWN0W3N0ciwgVGVuc29yXToKICAgIGlm'
        'IHJlZ2ltZSBub3QgaW4geyJtaXhlZCIsICJtaXNzaW5nX29ubHkiLCAiY2xlYW4iLCAiY29ycnVw'
        'dGVkIiwgIm9vZCJ9IG9yIGNvdW50IDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZh'
        'bGlkIHY0IHNwbGl0IikKICAgIGcgPSB0b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZChzZWVk'
        'KQogICAgbGFiZWxzID0gdG9yY2gucmFuZGludCgwLCA0LCAoY291bnQsKSwgZ2VuZXJhdG9yPWcp'
        'CiAgICAjIEluZGVwZW5kZW50IG1ldGFkYXRhOiB0aGUgdGFyZ2V0IGlzIG5ldmVyIHVzZWQgdG8g'
        'Y29uc3RydWN0IGRvbWFpbiBvciBsZWdhbGl0eS4KICAgIGRvbWFpbnMgPSB0b3JjaC5yYW5kaW50'
        'KDAsIDMsIChjb3VudCwpLCBnZW5lcmF0b3I9ZykKICAgIGNsZWFuX2luaXRpYWwgPSB0b3JjaC5y'
        'YW5kbihjb3VudCwgNSwgZ2VuZXJhdG9yPWcpICogMC4xNQogICAgY2xlYW5faW5pdGlhbFt0b3Jj'
        'aC5hcmFuZ2UoY291bnQpLCBsYWJlbHNdICs9IDEuNQogICAgYWN0aW9ucyA9IHRvcmNoLnJhbmQo'
        'Y291bnQsIDQsIDMsIGdlbmVyYXRvcj1nKSAqIDIgLSAxCiAgICBjbGVhbl9zdGF0ZXMgPSBbY2xl'
        'YW5faW5pdGlhbF0KICAgIGZ1dHVyZSA9IFtdCiAgICBmb3IgdCBpbiByYW5nZSg0KToKICAgICAg'
        'ICBuZXh0X3N0YXRlID0gZHluYW1pY3MoY2xlYW5fc3RhdGVzWy0xXSwgYWN0aW9uc1s6LCB0XSkK'
        'ICAgICAgICBmdXR1cmUuYXBwZW5kKG5leHRfc3RhdGUpCiAgICAgICAgaWYgdCA8IDM6CiAgICAg'
        'ICAgICAgIGNsZWFuX3N0YXRlcy5hcHBlbmQobmV4dF9zdGF0ZSkKICAgIGNsZWFuID0gdG9yY2gu'
        'c3RhY2soY2xlYW5fc3RhdGVzLCAxKQogICAgbmV4dF9zdGF0ZXMgPSB0b3JjaC5zdGFjayhmdXR1'
        'cmUsIDEpCgogICAgIyBJbmRlcGVuZGVudCBtb2RhbGl0eSBjb3JydXB0aW9uLiBNaXhlZCB0cmFp'
        'biBpbmNsdWRlcyBhbGwgdGVzdCBpbnRlcnZlbnRpb24KICAgICMgY2F0ZWdvcmllczsgT09EIHJl'
        'c2VydmVzIGJpYXMvcmV2ZXJzYWwgaW50ZXJ2ZW50aW9ucywgbmV2ZXIgc2VtYW50aWMgcmVtYXBw'
        'aW5nLgogICAgc3RhdHVzID0gdG9yY2guemVyb3MoY291bnQsIDMsIGR0eXBlPXRvcmNoLmxvbmcp'
        'CiAgICBicmFuY2ggPSB0b3JjaC5yYW5kaW50KDAsIDQsIChjb3VudCwpLCBnZW5lcmF0b3I9ZykK'
        'ICAgIGZpcnN0ID0gdG9yY2gucmFuZGludCgwLCAzLCAoY291bnQsKSwgZ2VuZXJhdG9yPWcpCiAg'
        'ICBzZWNvbmQgPSAoZmlyc3QgKyB0b3JjaC5yYW5kaW50KDEsIDMsIChjb3VudCwpLCBnZW5lcmF0'
        'b3I9ZykpICUgMwogICAgaWYgcmVnaW1lID09ICJtaXNzaW5nX29ubHkiOgogICAgICAgIG1pc3Np'
        'bmdfYnJhbmNoID0gKGJyYW5jaCA9PSAxKSB8IChicmFuY2ggPT0gMykKICAgICAgICBzdGF0dXNb'
        'bWlzc2luZ19icmFuY2gsIGZpcnN0W21pc3NpbmdfYnJhbmNoXV0gPSAxCiAgICBlbGlmIHJlZ2lt'
        'ZSA9PSAibWl4ZWQiOgogICAgICAgIHN0YXR1c1ticmFuY2ggPT0gMSwgZmlyc3RbYnJhbmNoID09'
        'IDFdXSA9IDEKICAgICAgICBzdGF0dXNbYnJhbmNoID09IDIsIGZpcnN0W2JyYW5jaCA9PSAyXV0g'
        'PSAyCiAgICAgICAgc3RhdHVzW2JyYW5jaCA9PSAzLCBmaXJzdFticmFuY2ggPT0gM11dID0gMQog'
        'ICAgICAgIHN0YXR1c1ticmFuY2ggPT0gMywgc2Vjb25kW2JyYW5jaCA9PSAzXV0gPSAyCiAgICBl'
        'bGlmIHJlZ2ltZSBpbiB7ImNvcnJ1cHRlZCIsICJvb2QifToKICAgICAgICBzdGF0dXNbdG9yY2gu'
        'YXJhbmdlKGNvdW50KSwgZmlyc3RdID0gMQogICAgICAgIHN0YXR1c1t0b3JjaC5hcmFuZ2UoY291'
        'bnQpLCBzZWNvbmRdID0gMgoKICAgIHRva2VucyA9IHRvcmNoLnJhbmRpbnQoOCwgMzEsIChjb3Vu'
        'dCwgOCksIGdlbmVyYXRvcj1nKQogICAgdG9rZW5zWzosIDBdID0gbGFiZWxzICsgMQogICAgdG9r'
        'ZW5zWzosIDFdID0gNSArIGxhYmVscyAlIDIKICAgIHRva2Vuc1s6LCAyXSA9IDcgKyBsYWJlbHMg'
        'Ly8gMgogICAgaW1hZ2UgPSB0b3JjaC5yYW5kKGNvdW50LCAzLCA4LCA4LCBnZW5lcmF0b3I9Zykg'
        'KiAwLjE1CiAgICBmb3IgaSBpbiByYW5nZShjb3VudCk6CiAgICAgICAgeSA9IGludChsYWJlbHNb'
        'aV0pCiAgICAgICAgaW1hZ2VbaSwgeSAlIDMsICh5IC8vIDIpICogNCA6ICh5IC8vIDIgKyAxKSAq'
        'IDQsICh5ICUgMikgKiA0IDogKHkgJSAyICsgMSkgKiA0XSArPSAwLjc1CiAgICBzdGF0ZXMgPSBj'
        'bGVhbiArIHRvcmNoLnJhbmRuKGNsZWFuLnNoYXBlLCBnZW5lcmF0b3I9ZykgKiAwLjEyCiAgICBm'
        'b3IgbW9kYWxpdHksIHggaW4gZW51bWVyYXRlKCh0b2tlbnMsIGltYWdlLCBzdGF0ZXMpKToKICAg'
        'ICAgICBtaXNzaW5nID0gc3RhdHVzWzosIG1vZGFsaXR5XSA9PSAxCiAgICAgICAgZGFtYWdlZCA9'
        'IHN0YXR1c1s6LCBtb2RhbGl0eV0gPT0gMgogICAgICAgIHhbbWlzc2luZ10gPSAwCiAgICAgICAg'
        'biA9IGludChkYW1hZ2VkLnN1bSgpKQogICAgICAgIGlmIG5vdCBuOgogICAgICAgICAgICBjb250'
        'aW51ZQogICAgICAgIGlmIG1vZGFsaXR5ID09IDA6CiAgICAgICAgICAgIGlmIHJlZ2ltZSA9PSAi'
        'b29kIjoKICAgICAgICAgICAgICAgIHhbZGFtYWdlZCwgOjNdID0geFtkYW1hZ2VkLCA6M10uZmxp'
        'cCgtMSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHhbZGFtYWdlZCwgOjNdID0g'
        'dG9yY2gucmFuZGludCgxLCAxMiwgKG4sIDMpLCBnZW5lcmF0b3I9ZykKICAgICAgICBlbGlmIG1v'
        'ZGFsaXR5ID09IDE6CiAgICAgICAgICAgIGlmIHJlZ2ltZSA9PSAib29kIjoKICAgICAgICAgICAg'
        'ICAgIHhbZGFtYWdlZF0gPSAoMS4wIC0geFtkYW1hZ2VkXSkuY2xhbXAoMCwgMSkKICAgICAgICAg'
        'ICAgZWxzZToKICAgICAgICAgICAgICAgIHhbZGFtYWdlZF0gPSB0b3JjaC5yYW5kKG4sIDMsIDgs'
        'IDgsIGdlbmVyYXRvcj1nKQogICAgICAgIGVsaWYgcmVnaW1lID09ICJvb2QiOgogICAgICAgICAg'
        'ICB4W2RhbWFnZWRdICs9IDEuNQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHhbZGFtYWdlZF0g'
        'PSB0b3JjaC5yYW5kbihuLCA0LCA1LCBnZW5lcmF0b3I9ZykKICAgIHF1YWxpdHkgPSB0b3JjaC50'
        'ZW5zb3IoWzEuMCwgMC4wLCAwLjI1XSlbc3RhdHVzXQogICAgIyBBdGxhcyBjb25zdW1lcyBCT1RI'
        'IHRleHQgYW5kIGltYWdlLiBJdHMgY29ycnVwdGlvbiBwcm94eSBleHBsaWNpdGx5IHJlZmxlY3Rz'
        'CiAgICAjIGJvdGgsIHJhdGhlciB0aGFuIGluY29ycmVjdGx5IGxhYmVsaW5nIGl0IGFuIGltYWdl'
        'LW9ubHkgZXhwZXJ0LgogICAgY29tcG9uZW50X3F1YWxpdHkgPSB0b3JjaC5zdGFjaygKICAgICAg'
        'ICBbCiAgICAgICAgICAgIHF1YWxpdHlbOiwgMF0sCiAgICAgICAgICAgIHF1YWxpdHlbOiwgMF0s'
        'CiAgICAgICAgICAgIHF1YWxpdHlbOiwgMF0sCiAgICAgICAgICAgIChxdWFsaXR5WzosIDBdICsg'
        'cXVhbGl0eVs6LCAxXSkgLyAyLAogICAgICAgICAgICBxdWFsaXR5WzosIDJdLAogICAgICAgIF0s'
        'CiAgICAgICAgMSwKICAgICkKICAgIHJldHVybiB7CiAgICAgICAgInRva2VucyI6IHRva2VucywK'
        'ICAgICAgICAiaW1hZ2UiOiBpbWFnZSwKICAgICAgICAic3RhdGVzIjogc3RhdGVzLAogICAgICAg'
        'ICJhY3Rpb25zIjogYWN0aW9ucywKICAgICAgICAiZG9tYWluX2lkcyI6IGRvbWFpbnMsCiAgICAg'
        'ICAgImFsbG93ZWRfdHJhbnNpdGlvbnMiOiB0b3JjaC5vbmVzKGNvdW50LCA0LCBkdHlwZT10b3Jj'
        'aC5ib29sKSwKICAgICAgICAicmlza19zY29yZSI6IHRvcmNoLnplcm9zKGNvdW50KSwKICAgICAg'
        'ICAiZGVjaXNpb25fdGFyZ2V0cyI6IGxhYmVscywKICAgICAgICAidHJhbnNpdGlvbl90YXJnZXRz'
        'IjogbGFiZWxzICUgMywKICAgICAgICAibmV4dF9zdGF0ZV90YXJnZXRzIjogbmV4dF9zdGF0ZXMs'
        'CiAgICAgICAgImNvbXBvbmVudF9xdWFsaXR5IjogY29tcG9uZW50X3F1YWxpdHksCiAgICAgICAg'
        'ImNvcnJ1cHRpb25fc3RhdHVzIjogc3RhdHVzLAogICAgICAgICJzb3VyY2VfaWRzIjogdG9yY2gu'
        'YXJhbmdlKGNvdW50LCBkdHlwZT10b3JjaC5pbnQ2NCkgKyBzZWVkICogMTAwMDAwLAogICAgfQoK'
        'CmRlZiBzdWJzZXQoYmF0Y2g6IGRpY3Rbc3RyLCBUZW5zb3JdLCBpbmRpY2VzOiBUZW5zb3IpIC0+'
        'IGRpY3Rbc3RyLCBUZW5zb3JdOgogICAgcmV0dXJuIHtrZXk6IHZhbHVlW2luZGljZXNdIGZvciBr'
        'ZXksIHZhbHVlIGluIGJhdGNoLml0ZW1zKCl9Cg=='
    ),
    'olympus/data/registry.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBwYXRo'
        'bGliIGltcG9ydCBQYXRoCgpmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZAoKZnJvbSBvbHltcHVz'
        'LmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKCgpjbGFzcyBEYXRhU291cmNlUmVjb3Jk'
        'KFN0cmljdE1vZGVsKToKICAgIG5hbWU6IHN0cgogICAgc291cmNlX3VybDogc3RyCiAgICBvcmdh'
        'bml6YXRpb246IHN0cgogICAgbGljZW5zZTogc3RyCiAgICBwZXJtaXR0ZWRfdXNlczogbGlzdFtz'
        'dHJdCiAgICByZXRyaWV2YWxfZGF0ZTogc3RyCiAgICBzaGEyNTY6IHN0ciA9IEZpZWxkKHBhdHRl'
        'cm49ciJeWzAtOWEtZl17NjR9JCIpCiAgICBwcm92ZW5hbmNlOiBzdHIKICAgIHF1YWxpdHlfc2Nv'
        'cmU6IGZsb2F0ID0gRmllbGQoZ2U9MC4wLCBsZT0xLjApCiAgICBsYW5ndWFnZTogc3RyCiAgICBk'
        'b21haW46IHN0cgogICAgcGVyc29uYWxfZGF0YTogYm9vbAogICAgdHJhaW5pbmdfZWxpZ2libGU6'
        'IGJvb2wKICAgIHJlbW92ZWQ6IGJvb2wgPSBGYWxzZQoKCmNsYXNzIERhdGFTb3VyY2VSZWdpc3Ry'
        'eShTdHJpY3RNb2RlbCk6CiAgICBzb3VyY2VzOiBsaXN0W0RhdGFTb3VyY2VSZWNvcmRdID0gRmll'
        'bGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCgogICAgZGVmIGFkZChzZWxmLCBzb3VyY2U6IERhdGFT'
        'b3VyY2VSZWNvcmQpIC0+IE5vbmU6CiAgICAgICAgaWYgYW55KGl0ZW0ubmFtZSA9PSBzb3VyY2Uu'
        'bmFtZSBhbmQgbm90IGl0ZW0ucmVtb3ZlZCBmb3IgaXRlbSBpbiBzZWxmLnNvdXJjZXMpOgogICAg'
        'ICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiYWN0aXZlIGRhdGEgc291cmNlIGFscmVhZHkgZXhp'
        'c3RzOiB7c291cmNlLm5hbWV9IikKICAgICAgICBzZWxmLnNvdXJjZXMuYXBwZW5kKHNvdXJjZSkK'
        'CiAgICBkZWYgc2F2ZShzZWxmLCBwYXRoOiBQYXRoKSAtPiBOb25lOgogICAgICAgIHBhdGgucGFy'
        'ZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBwYXRoLndyaXRl'
        'X3RleHQoc2VsZi5tb2RlbF9kdW1wX2pzb24oaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQoK'
        'ICAgIEBjbGFzc21ldGhvZAogICAgZGVmIGxvYWQoY2xzLCBwYXRoOiBQYXRoKSAtPiBEYXRhU291'
        'cmNlUmVnaXN0cnk6CiAgICAgICAgcmV0dXJuIGNscy5tb2RlbF92YWxpZGF0ZShqc29uLmxvYWRz'
        'KHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKSkK'
    ),
    'olympus/data/reliability_fusion.py': (
        'IiIiRnJlc2gsIHNvdXJjZS1kaXNqb2ludCBkYXRhIGZvciB0aGUgcHJlcmVnaXN0ZXJlZCBPbHlt'
        'cHVzIHYzIHN1Y2Nlc3Nvci4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMK'
        'CmltcG9ydCBoYXNobGliCmltcG9ydCBpbwppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRo'
        'bGliIGltcG9ydCBQYXRoCmZyb20gdGVtcGZpbGUgaW1wb3J0IE5hbWVkVGVtcG9yYXJ5RmlsZQpm'
        'cm9tIHR5cGluZyBpbXBvcnQgTGl0ZXJhbAoKaW1wb3J0IHRvcmNoCmZyb20gcHlkYW50aWMgaW1w'
        'b3J0IEZpZWxkCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBvbHltcHVzLmNvcmUuc2No'
        'ZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKZnJvbSBvbHltcHVzLmRhdGEub2x5bXB1c19kYXRhc2V0'
        'IGltcG9ydCB3cml0ZV9vbHltcHVzX2RhdGFzZXRfbWFuaWZlc3QKClJlbGlhYmlsaXR5UmVnaW1l'
        'ID0gTGl0ZXJhbFsibWl4ZWQiLCAiY2xlYW4iLCAiY29ycnVwdGVkIiwgIm9vZCJdCgoKY2xhc3Mg'
        'UmVsaWFiaWxpdHlTcGxpdFNwZWMoU3RyaWN0TW9kZWwpOgogICAgbmFtZTogc3RyID0gRmllbGQo'
        'bWluX2xlbmd0aD0xKQogICAgZXhhbXBsZXM6IGludCA9IEZpZWxkKGdlPTEpCiAgICBzZWVkOiBp'
        'bnQgPSBGaWVsZChnZT0wLCBsZT0yKio2MyAtIDEpCiAgICByZWdpbWU6IFJlbGlhYmlsaXR5UmVn'
        'aW1lCgoKY2xhc3MgUmVsaWFiaWxpdHlEYXRhc2V0UmVjZWlwdChTdHJpY3RNb2RlbCk6CiAgICBz'
        'Y2hlbWFfdmVyc2lvbjogTGl0ZXJhbFsxXSA9IDEKICAgIGRhdGFzZXRfaWQ6IExpdGVyYWxbIm9s'
        'eW1wdXMtcmVsaWFiaWxpdHktZnVzaW9uIl0gPSAib2x5bXB1cy1yZWxpYWJpbGl0eS1mdXNpb24i'
        'CiAgICB2ZXJzaW9uOiBMaXRlcmFsWyIzLjAuMCJdID0gIjMuMC4wIgogICAgbGljZW5zZTogTGl0'
        'ZXJhbFsiQ0MtQlktNC4wIl0gPSAiQ0MtQlktNC4wIgogICAgZ2VuZXJhdG9yOiBMaXRlcmFsWyJv'
        'bHltcHVzLmRhdGEucmVsaWFiaWxpdHlfZnVzaW9uIl0gPSAib2x5bXB1cy5kYXRhLnJlbGlhYmls'
        'aXR5X2Z1c2lvbiIKICAgIHNwbGl0czogbGlzdFtSZWxpYWJpbGl0eVNwbGl0U3BlY10KICAgIGJ1'
        'bmRsZV9zaGEyNTY6IGRpY3Rbc3RyLCBzdHJdCiAgICBtYW5pZmVzdF9zaGEyNTY6IGRpY3Rbc3Ry'
        'LCBzdHJdCiAgICBzb3VyY2VfaWRfc2hhMjU2OiBkaWN0W3N0ciwgc3RyXQogICAgc291cmNlX292'
        'ZXJsYXBfY291bnQ6IGludCA9IEZpZWxkKGdlPTApCiAgICBwcm90ZWN0ZWRfc3BsaXRzOiBsaXN0'
        'W3N0cl0KICAgIGNvbXBvbmVudF9vcmRlcjogbGlzdFtzdHJdCiAgICByZWxpYWJpbGl0eV9zZW1h'
        'bnRpY3M6IGRpY3Rbc3RyLCBmbG9hdF0KICAgIGNsYWltX2JvdW5kYXJ5OiBzdHIKCgpSRUdJU1RF'
        'UkVEX1JFTElBQklMSVRZX1NQTElUUyA9ICgKICAgIFJlbGlhYmlsaXR5U3BsaXRTcGVjKG5hbWU9'
        'InRyYWluIiwgZXhhbXBsZXM9NTEyLCBzZWVkPTIwMjYxNTAxLCByZWdpbWU9Im1peGVkIiksCiAg'
        'ICBSZWxpYWJpbGl0eVNwbGl0U3BlYyhuYW1lPSJkZXZlbG9wbWVudCIsIGV4YW1wbGVzPTEyOCwg'
        'c2VlZD0yMDI2MTUwMiwgcmVnaW1lPSJtaXhlZCIpLAogICAgUmVsaWFiaWxpdHlTcGxpdFNwZWMo'
        'bmFtZT0iY2xlYW5fdGVzdCIsIGV4YW1wbGVzPTI1Niwgc2VlZD0yMDI2MTUwMywgcmVnaW1lPSJj'
        'bGVhbiIpLAogICAgUmVsaWFiaWxpdHlTcGxpdFNwZWMobmFtZT0iY29ycnVwdGVkX3Rlc3QiLCBl'
        'eGFtcGxlcz0yNTYsIHNlZWQ9MjAyNjE1MDQsIHJlZ2ltZT0iY29ycnVwdGVkIiksCiAgICBSZWxp'
        'YWJpbGl0eVNwbGl0U3BlYyhuYW1lPSJvb2RfdGVzdCIsIGV4YW1wbGVzPTI1Niwgc2VlZD0yMDI2'
        'MTUwNSwgcmVnaW1lPSJvb2QiKSwKKQpDT01QT05FTlRfT1JERVIgPSAoImhlcm1lcyIsICJwcm9t'
        'ZXRoZXVzIiwgInBlcnNldXMiLCAiYXRsYXMiLCAia3Jvbm9zIikKVEVYVF9DT01QT05FTlRTID0g'
        'KDAsIDEsIDIpClJFTElBQklMSVRZX1ZBTFVFUyA9IHsiY2xlYW4iOiAxLjAsICJwZXJ0dXJiZWQi'
        'OiAwLjI1LCAibWlzc2luZyI6IDAuMH0KCgpkZWYgX3NoYTI1NihwYXlsb2FkOiBieXRlcykgLT4g'
        'c3RyOgogICAgcmV0dXJuIGhhc2hsaWIuc2hhMjU2KHBheWxvYWQpLmhleGRpZ2VzdCgpCgoKZGVm'
        'IF9hdG9taWNfd3JpdGUocGF0aDogUGF0aCwgcGF5bG9hZDogYnl0ZXMpIC0+IE5vbmU6CiAgICBw'
        'YXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIE5h'
        'bWVkVGVtcG9yYXJ5RmlsZShkaXI9cGF0aC5wYXJlbnQsIHByZWZpeD1mIi57cGF0aC5uYW1lfS4i'
        'LCBkZWxldGU9RmFsc2UpIGFzIGhhbmRsZToKICAgICAgICBoYW5kbGUud3JpdGUocGF5bG9hZCkK'
        'ICAgICAgICBoYW5kbGUuZmx1c2goKQogICAgICAgIG9zLmZzeW5jKGhhbmRsZS5maWxlbm8oKSkK'
        'ICAgICAgICB0ZW1wb3JhcnkgPSBQYXRoKGhhbmRsZS5uYW1lKQogICAgdGVtcG9yYXJ5LnJlcGxh'
        'Y2UocGF0aCkKCgpkZWYgX3NvdXJjZV9pZHMoc3BlYzogUmVsaWFiaWxpdHlTcGxpdFNwZWMpIC0+'
        'IGxpc3Rbc3RyXToKICAgIHJldHVybiBbCiAgICAgICAgZiJ2My17c3BlYy5uYW1lfS17X3NoYTI1'
        'NihmJ3tzcGVjLnNlZWR9OntpbmRleH0nLmVuY29kZSgpKVs6MjBdfSIKICAgICAgICBmb3IgaW5k'
        'ZXggaW4gcmFuZ2Uoc3BlYy5leGFtcGxlcykKICAgIF0KCgpkZWYgX2NvcnJ1cHRpb25fcGxhbigK'
        'ICAgIHNwZWM6IFJlbGlhYmlsaXR5U3BsaXRTcGVjLCBnZW5lcmF0b3I6IHRvcmNoLkdlbmVyYXRv'
        'cgopIC0+IHR1cGxlW1RlbnNvciwgVGVuc29yXToKICAgIG1pc3NpbmcgPSB0b3JjaC5mdWxsKChz'
        'cGVjLmV4YW1wbGVzLCksIC0xLCBkdHlwZT10b3JjaC5sb25nKQogICAgcGVydHVyYmVkID0gdG9y'
        'Y2guZnVsbCgoc3BlYy5leGFtcGxlcywpLCAtMSwgZHR5cGU9dG9yY2gubG9uZykKICAgIGlmIHNw'
        'ZWMucmVnaW1lID09ICJtaXhlZCI6CiAgICAgICAgc2VsZWN0ZWQgPSB0b3JjaC5yYW5kKHNwZWMu'
        'ZXhhbXBsZXMsIGdlbmVyYXRvcj1nZW5lcmF0b3IpIDwgMC43MAogICAgICAgIG1pc3Npbmdbc2Vs'
        'ZWN0ZWRdID0gdG9yY2gucmFuZGludCgwLCAzLCAoaW50KHNlbGVjdGVkLnN1bSgpKSwpLCBnZW5l'
        'cmF0b3I9Z2VuZXJhdG9yKQogICAgZWxpZiBzcGVjLnJlZ2ltZSBpbiB7ImNvcnJ1cHRlZCIsICJv'
        'b2QifToKICAgICAgICBtaXNzaW5nID0gdG9yY2gucmFuZGludCgwLCAzLCAoc3BlYy5leGFtcGxl'
        'cywpLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKQogICAgICAgIG9mZnNldCA9IHRvcmNoLnJhbmRpbnQo'
        'MSwgMywgKHNwZWMuZXhhbXBsZXMsKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikKICAgICAgICBwZXJ0'
        'dXJiZWQgPSAobWlzc2luZyArIG9mZnNldCkgJSAzCiAgICByZXR1cm4gbWlzc2luZywgcGVydHVy'
        'YmVkCgoKZGVmIF9yZWxpYWJpbGl0eV90YXJnZXRzKG1pc3Npbmc6IFRlbnNvciwgcGVydHVyYmVk'
        'OiBUZW5zb3IpIC0+IFRlbnNvcjoKICAgIHRhcmdldHMgPSB0b3JjaC5vbmVzKChtaXNzaW5nLnNo'
        'YXBlWzBdLCBsZW4oQ09NUE9ORU5UX09SREVSKSksIGR0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICBt'
        'b2RhbGl0eV9jb21wb25lbnRzID0gezA6IFRFWFRfQ09NUE9ORU5UUywgMTogKDMsKSwgMjogKDQs'
        'KX0KICAgIGZvciBtb2RhbGl0eSwgY29tcG9uZW50cyBpbiBtb2RhbGl0eV9jb21wb25lbnRzLml0'
        'ZW1zKCk6CiAgICAgICAgZm9yIGNvbXBvbmVudCBpbiBjb21wb25lbnRzOgogICAgICAgICAgICB0'
        'YXJnZXRzW21pc3NpbmcgPT0gbW9kYWxpdHksIGNvbXBvbmVudF0gPSBSRUxJQUJJTElUWV9WQUxV'
        'RVNbIm1pc3NpbmciXQogICAgICAgICAgICB0YXJnZXRzW3BlcnR1cmJlZCA9PSBtb2RhbGl0eSwg'
        'Y29tcG9uZW50XSA9IFJFTElBQklMSVRZX1ZBTFVFU1sicGVydHVyYmVkIl0KICAgIHJldHVybiB0'
        'YXJnZXRzCgoKZGVmIGdlbmVyYXRlX3JlbGlhYmlsaXR5X3NwbGl0KAogICAgc3BlYzogUmVsaWFi'
        'aWxpdHlTcGxpdFNwZWMsCikgLT4gdHVwbGVbZGljdFtzdHIsIFRlbnNvcl0sIGxpc3Rbc3RyXV06'
        'CiAgICAiIiJHZW5lcmF0ZSBhIGRldGVybWluaXN0aWMgc3BsaXQgd2l0aCBpbmRlcGVuZGVudGx5'
        'IHNwZWNpZmllZCByZWxpYWJpbGl0eSBsYWJlbHMuIiIiCgogICAgZ2VuZXJhdG9yID0gdG9yY2gu'
        'R2VuZXJhdG9yKGRldmljZT0iY3B1IikubWFudWFsX3NlZWQoc3BlYy5zZWVkKQogICAgY291bnQg'
        'PSBzcGVjLmV4YW1wbGVzCiAgICBsYWJlbHMgPSB0b3JjaC5yYW5kaW50KDAsIDQsIChjb3VudCwp'
        'LCBnZW5lcmF0b3I9Z2VuZXJhdG9yKQogICAgbWlzc2luZywgcGVydHVyYmVkID0gX2NvcnJ1cHRp'
        'b25fcGxhbihzcGVjLCBnZW5lcmF0b3IpCiAgICByZWxpYWJpbGl0eV90YXJnZXRzID0gX3JlbGlh'
        'YmlsaXR5X3RhcmdldHMobWlzc2luZywgcGVydHVyYmVkKQogICAgbm9pc2Vfc2NhbGUgPSAwLjEy'
        'IGlmIHNwZWMucmVnaW1lICE9ICJvb2QiIGVsc2UgMC4yNAoKICAgIHRva2VucyA9IHRvcmNoLnJh'
        'bmRpbnQoOCwgMzEsIChjb3VudCwgOCksIGdlbmVyYXRvcj1nZW5lcmF0b3IsIGR0eXBlPXRvcmNo'
        'LmxvbmcpCiAgICBjbGFzc190b2tlbnMgPSBsYWJlbHMgKyAxCiAgICBpZiBzcGVjLnJlZ2ltZSA9'
        'PSAib29kIjoKICAgICAgICBjbGFzc190b2tlbnMgPSB0b3JjaC50ZW5zb3IoWzMsIDEsIDQsIDJd'
        'LCBkdHlwZT10b3JjaC5sb25nKVtsYWJlbHNdCiAgICB0b2tlbnNbOiwgMF0gPSBjbGFzc190b2tl'
        'bnMKICAgIHRva2Vuc1s6LCAxXSA9IDUgKyBsYWJlbHMucmVtYWluZGVyKDIpCiAgICB0b2tlbnNb'
        'OiwgMl0gPSA3ICsgbGFiZWxzLmRpdigyLCByb3VuZGluZ19tb2RlPSJmbG9vciIpCiAgICBhdHRl'
        'bnRpb25fbWFzayA9IHRvcmNoLm9uZXNfbGlrZSh0b2tlbnMsIGR0eXBlPXRvcmNoLmJvb2wpCgog'
        'ICAgaW1hZ2UgPSB0b3JjaC5yYW5kKChjb3VudCwgMywgOCwgOCksIGdlbmVyYXRvcj1nZW5lcmF0'
        'b3IpICogbm9pc2Vfc2NhbGUKICAgIGZvciBpbmRleCBpbiByYW5nZShjb3VudCk6CiAgICAgICAg'
        'bGFiZWwgPSBpbnQobGFiZWxzW2luZGV4XSkKICAgICAgICBjaGFubmVsID0gbGFiZWwgJSAzCiAg'
        'ICAgICAgdG9wID0gMCBpZiBsYWJlbCA8IDIgZWxzZSA0CiAgICAgICAgbGVmdCA9IDAgaWYgbGFi'
        'ZWwgJSAyID09IDAgZWxzZSA0CiAgICAgICAgaWYgc3BlYy5yZWdpbWUgPT0gIm9vZCI6CiAgICAg'
        'ICAgICAgIGNoYW5uZWwgPSAoY2hhbm5lbCArIDEpICUgMwogICAgICAgICAgICBsZWZ0ID0gNCAt'
        'IGxlZnQKICAgICAgICBpbWFnZVtpbmRleCwgY2hhbm5lbCwgdG9wIDogdG9wICsgNCwgbGVmdCA6'
        'IGxlZnQgKyA0XSArPSAwLjc1CiAgICBpbWFnZS5jbGFtcF8oMC4wLCAxLjApCgogICAgYWN0aW9u'
        'cyA9IHRvcmNoLnJhbmQoKGNvdW50LCA0LCAzKSwgZ2VuZXJhdG9yPWdlbmVyYXRvcikgKiAyLjAg'
        'LSAxLjAKICAgIHRpbWUgPSB0b3JjaC5hcmFuZ2UoNCwgZHR5cGU9dG9yY2guZmxvYXQzMikudmll'
        'dygxLCA0LCAxKQogICAgbGFiZWxfdmFsdWUgPSBsYWJlbHMuZmxvYXQoKS52aWV3KGNvdW50LCAx'
        'LCAxKSAvIDMuMAogICAgc3RhdGVfbm9pc2UgPSB0b3JjaC5yYW5kbigoY291bnQsIDQsIDUpLCBn'
        'ZW5lcmF0b3I9Z2VuZXJhdG9yKSAqIG5vaXNlX3NjYWxlCiAgICBzdGF0ZV90ZW1wbGF0ZSA9IHRv'
        'cmNoLmNhdCgKICAgICAgICAoCiAgICAgICAgICAgIGxhYmVsX3ZhbHVlLmV4cGFuZCgtMSwgNCwg'
        'MSksCiAgICAgICAgICAgIChsYWJlbF92YWx1ZSArIDAuMDggKiB0aW1lKS5leHBhbmQoLTEsIC0x'
        'LCAxKSwKICAgICAgICAgICAgbGFiZWxzLnJlbWFpbmRlcigyKS5mbG9hdCgpLnZpZXcoY291bnQs'
        'IDEsIDEpLmV4cGFuZCgtMSwgNCwgMSksCiAgICAgICAgICAgIGxhYmVscy5kaXYoMiwgcm91bmRp'
        'bmdfbW9kZT0iZmxvb3IiKS5mbG9hdCgpLnZpZXcoY291bnQsIDEsIDEpLmV4cGFuZCgtMSwgNCwg'
        'MSksCiAgICAgICAgICAgIHRvcmNoLnNpbihsYWJlbF92YWx1ZSAqIHRvcmNoLnBpICsgdGltZSAq'
        'IDAuMyksCiAgICAgICAgKSwKICAgICAgICBkaW09LTEsCiAgICApCiAgICBzdGF0ZXMgPSBzdGF0'
        'ZV90ZW1wbGF0ZSArIHN0YXRlX25vaXNlCgogICAgZm9yIG1vZGFsaXR5LCB2YWx1ZSBpbiBlbnVt'
        'ZXJhdGUoKHRva2VucywgaW1hZ2UsIHN0YXRlcykpOgogICAgICAgIHNlbGVjdGVkX21pc3Npbmcg'
        'PSBtaXNzaW5nID09IG1vZGFsaXR5CiAgICAgICAgc2VsZWN0ZWRfcGVydHVyYmVkID0gcGVydHVy'
        'YmVkID09IG1vZGFsaXR5CiAgICAgICAgaWYgYm9vbChzZWxlY3RlZF9taXNzaW5nLmFueSgpKToK'
        'ICAgICAgICAgICAgdmFsdWVbc2VsZWN0ZWRfbWlzc2luZ10gPSAwCiAgICAgICAgaWYgYm9vbChz'
        'ZWxlY3RlZF9wZXJ0dXJiZWQuYW55KCkpOgogICAgICAgICAgICBhbW91bnQgPSBpbnQoc2VsZWN0'
        'ZWRfcGVydHVyYmVkLnN1bSgpKQogICAgICAgICAgICBpZiBtb2RhbGl0eSA9PSAwOgogICAgICAg'
        'ICAgICAgICAgdG9rZW5zW3NlbGVjdGVkX3BlcnR1cmJlZCwgOjNdID0gdG9yY2gucmFuZGludCgK'
        'ICAgICAgICAgICAgICAgICAgICAxLCAxMiwgKGFtb3VudCwgMyksIGdlbmVyYXRvcj1nZW5lcmF0'
        'b3IKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgZWxpZiBtb2RhbGl0eSA9PSAxOgogICAg'
        'ICAgICAgICAgICAgaW1hZ2Vbc2VsZWN0ZWRfcGVydHVyYmVkXSA9IHRvcmNoLnJhbmQoKGFtb3Vu'
        'dCwgMywgOCwgOCksIGdlbmVyYXRvcj1nZW5lcmF0b3IpCiAgICAgICAgICAgIGVsc2U6CiAgICAg'
        'ICAgICAgICAgICBzdGF0ZXNbc2VsZWN0ZWRfcGVydHVyYmVkXSA9IHRvcmNoLnJhbmRuKChhbW91'
        'bnQsIDQsIDUpLCBnZW5lcmF0b3I9Z2VuZXJhdG9yKQoKICAgIHRlbXBvcmFsX21hc2sgPSB0b3Jj'
        'aC5vbmVzKChjb3VudCwgNCksIGR0eXBlPXRvcmNoLmJvb2wpCiAgICBhbGxvd2VkX3RyYW5zaXRp'
        'b25zID0gdG9yY2gub25lcygoY291bnQsIDQpLCBkdHlwZT10b3JjaC5ib29sKQogICAgdHJhbnNp'
        'dGlvbl90YXJnZXRzID0gbGFiZWxzLnJlbWFpbmRlcigzKQogICAgZm9yYmlkZGVuID0gKHRyYW5z'
        'aXRpb25fdGFyZ2V0cyArIDEpLnJlbWFpbmRlcigzKQogICAgYWxsb3dlZF90cmFuc2l0aW9uc1t0'
        'b3JjaC5hcmFuZ2UoY291bnQpLCBmb3JiaWRkZW5dID0gRmFsc2UKICAgIGFsbG93ZWRfdHJhbnNp'
        'dGlvbnNbOiwgM10gPSBUcnVlCiAgICBjb3JydXB0aW9uX2NvdW50ID0gbWlzc2luZy5nZSgwKS5m'
        'bG9hdCgpICsgcGVydHVyYmVkLmdlKDApLmZsb2F0KCkKICAgIHJpc2tfc2NvcmUgPSAwLjA1ICsg'
        'Y29ycnVwdGlvbl9jb3VudCAqIDAuMTUKICAgIHBhZGRlZF9hY3Rpb25zID0gdG9yY2gubm4uZnVu'
        'Y3Rpb25hbC5wYWQoYWN0aW9ucywgKDAsIDIpKQoKICAgIGJ1bmRsZSA9IHsKICAgICAgICAidG9r'
        'ZW5zIjogdG9rZW5zLAogICAgICAgICJhdHRlbnRpb25fbWFzayI6IGF0dGVudGlvbl9tYXNrLAog'
        'ICAgICAgICJpbWFnZSI6IGltYWdlLAogICAgICAgICJzdGF0ZXMiOiBzdGF0ZXMsCiAgICAgICAg'
        'ImFjdGlvbnMiOiBhY3Rpb25zLAogICAgICAgICJ0ZW1wb3JhbF9tYXNrIjogdGVtcG9yYWxfbWFz'
        'aywKICAgICAgICAiYWxsb3dlZF90cmFuc2l0aW9ucyI6IGFsbG93ZWRfdHJhbnNpdGlvbnMsCiAg'
        'ICAgICAgInJpc2tfc2NvcmUiOiByaXNrX3Njb3JlLAogICAgICAgICJkb21haW5faWRzIjogbGFi'
        'ZWxzLnJlbWFpbmRlcigzKSwKICAgICAgICAidGltZXN0YW1wIjogdG9yY2guYXJhbmdlKGNvdW50'
        'LCBkdHlwZT10b3JjaC5mbG9hdDMyKSAvIG1heCgxLCBjb3VudCAtIDEpLAogICAgICAgICJkZWNp'
        'c2lvbl90YXJnZXRzIjogbGFiZWxzLAogICAgICAgICJ0cmFuc2l0aW9uX3RhcmdldHMiOiB0cmFu'
        'c2l0aW9uX3RhcmdldHMsCiAgICAgICAgIm5leHRfc3RhdGVfdGFyZ2V0cyI6IHN0YXRlcyArIDAu'
        'MSAqIHBhZGRlZF9hY3Rpb25zLAogICAgICAgICJpbnRyaW5zaWNfcmlza190YXJnZXRzIjogY29y'
        'cnVwdGlvbl9jb3VudCAvIDIuMCwKICAgICAgICAiY29tcG9uZW50X3JlbGlhYmlsaXR5X3Rhcmdl'
        'dHMiOiByZWxpYWJpbGl0eV90YXJnZXRzLAogICAgfQogICAgcmV0dXJuIGJ1bmRsZSwgX3NvdXJj'
        'ZV9pZHMoc3BlYykKCgpkZWYgX3NlcmlhbGl6ZV9idW5kbGUoYnVuZGxlOiBkaWN0W3N0ciwgVGVu'
        'c29yXSkgLT4gYnl0ZXM6CiAgICBidWZmZXIgPSBpby5CeXRlc0lPKCkKICAgIHRvcmNoLnNhdmUo'
        'YnVuZGxlLCBidWZmZXIpCiAgICByZXR1cm4gYnVmZmVyLmdldHZhbHVlKCkKCgpkZWYgZ2VuZXJh'
        'dGVfcmVsaWFiaWxpdHlfZnVzaW9uX2RhdGFzZXQob3V0cHV0OiBQYXRoKSAtPiBSZWxpYWJpbGl0'
        'eURhdGFzZXRSZWNlaXB0OgogICAgIiIiTWF0ZXJpYWxpemUgdjMgc3BsaXRzIG9uY2UgYW5kIGJp'
        'bmQgZXZlcnkgc3BsaXQgdG8gY29udGVudCBoYXNoZXMuIiIiCgogICAgb3V0cHV0ID0gb3V0cHV0'
        'LnJlc29sdmUoKQogICAgaWYgb3V0cHV0LmV4aXN0cygpIGFuZCBhbnkob3V0cHV0Lml0ZXJkaXIo'
        'KSk6CiAgICAgICAgcmFpc2UgRmlsZUV4aXN0c0Vycm9yKCJyZWZ1c2luZyB0byBvdmVyd3JpdGUg'
        'YSByZWxpYWJpbGl0eSBkYXRhc2V0IGRpcmVjdG9yeSIpCiAgICBvdXRwdXQubWtkaXIocGFyZW50'
        'cz1UcnVlLCBleGlzdF9vaz1UcnVlKQoKICAgIHNvdXJjZV9pZHM6IGRpY3Rbc3RyLCBsaXN0W3N0'
        'cl1dID0ge30KICAgIGJ1bmRsZV9oYXNoZXM6IGRpY3Rbc3RyLCBzdHJdID0ge30KICAgIGZvciBz'
        'cGVjIGluIFJFR0lTVEVSRURfUkVMSUFCSUxJVFlfU1BMSVRTOgogICAgICAgIGJ1bmRsZSwgaWRl'
        'bnRpZmllcnMgPSBnZW5lcmF0ZV9yZWxpYWJpbGl0eV9zcGxpdChzcGVjKQogICAgICAgIHBheWxv'
        'YWQgPSBfc2VyaWFsaXplX2J1bmRsZShidW5kbGUpCiAgICAgICAgX2F0b21pY193cml0ZShvdXRw'
        'dXQgLyBmIntzcGVjLm5hbWV9LnB0IiwgcGF5bG9hZCkKICAgICAgICBzb3VyY2VfaWRzW3NwZWMu'
        'bmFtZV0gPSBpZGVudGlmaWVycwogICAgICAgIGJ1bmRsZV9oYXNoZXNbc3BlYy5uYW1lXSA9IF9z'
        'aGEyNTYocGF5bG9hZCkKCiAgICBhbGxfaWRzID0gW2lkZW50aWZpZXIgZm9yIHZhbHVlcyBpbiBz'
        'b3VyY2VfaWRzLnZhbHVlcygpIGZvciBpZGVudGlmaWVyIGluIHZhbHVlc10KICAgIG92ZXJsYXBf'
        'Y291bnQgPSBsZW4oYWxsX2lkcykgLSBsZW4oc2V0KGFsbF9pZHMpKQogICAgaWYgb3ZlcmxhcF9j'
        'b3VudDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoInYzIHNvdXJjZSBpZGVudGl0aWVzIG92'
        'ZXJsYXAiKQoKICAgIG1hbmlmZXN0X2hhc2hlczogZGljdFtzdHIsIHN0cl0gPSB7fQogICAgZm9y'
        'IGV2YWx1YXRpb25fbmFtZSBpbiAoImRldmVsb3BtZW50IiwgImNsZWFuX3Rlc3QiLCAiY29ycnVw'
        'dGVkX3Rlc3QiLCAib29kX3Rlc3QiKToKICAgICAgICBtYW5pZmVzdCA9IHdyaXRlX29seW1wdXNf'
        'ZGF0YXNldF9tYW5pZmVzdCgKICAgICAgICAgICAgb3V0cHV0IC8gZiJtYW5pZmVzdF97ZXZhbHVh'
        'dGlvbl9uYW1lfS5qc29uIiwKICAgICAgICAgICAgZGF0YXNldF9pZD0ib2x5bXB1cy1yZWxpYWJp'
        'bGl0eS1mdXNpb24iLAogICAgICAgICAgICB2ZXJzaW9uPSIzLjAuMCIsCiAgICAgICAgICAgIGxp'
        'Y2Vuc2VfbmFtZT0iQ0MtQlktNC4wIiwKICAgICAgICAgICAgdHJhaW5fYmF0Y2g9b3V0cHV0IC8g'
        'InRyYWluLnB0IiwKICAgICAgICAgICAgZXZhbHVhdGlvbl9iYXRjaD1vdXRwdXQgLyBmIntldmFs'
        'dWF0aW9uX25hbWV9LnB0IiwKICAgICAgICAgICAgdHJhaW5fc291cmNlX2lkcz1zb3VyY2VfaWRz'
        'WyJ0cmFpbiJdLAogICAgICAgICAgICBldmFsdWF0aW9uX3NvdXJjZV9pZHM9c291cmNlX2lkc1tl'
        'dmFsdWF0aW9uX25hbWVdLAogICAgICAgICkKICAgICAgICBhc3NlcnQgbWFuaWZlc3QubWFuaWZl'
        'c3Rfc2hhMjU2IGlzIG5vdCBOb25lCiAgICAgICAgbWFuaWZlc3RfaGFzaGVzW2V2YWx1YXRpb25f'
        'bmFtZV0gPSBtYW5pZmVzdC5tYW5pZmVzdF9zaGEyNTYKCiAgICByZWNlaXB0ID0gUmVsaWFiaWxp'
        'dHlEYXRhc2V0UmVjZWlwdCgKICAgICAgICBzcGxpdHM9bGlzdChSRUdJU1RFUkVEX1JFTElBQklM'
        'SVRZX1NQTElUUyksCiAgICAgICAgYnVuZGxlX3NoYTI1Nj1idW5kbGVfaGFzaGVzLAogICAgICAg'
        'IG1hbmlmZXN0X3NoYTI1Nj1tYW5pZmVzdF9oYXNoZXMsCiAgICAgICAgc291cmNlX2lkX3NoYTI1'
        'Nj17CiAgICAgICAgICAgIG5hbWU6IF9zaGEyNTYoKCJcbiIuam9pbih2YWx1ZXMpICsgIlxuIiku'
        'ZW5jb2RlKCkpCiAgICAgICAgICAgIGZvciBuYW1lLCB2YWx1ZXMgaW4gc291cmNlX2lkcy5pdGVt'
        'cygpCiAgICAgICAgfSwKICAgICAgICBzb3VyY2Vfb3ZlcmxhcF9jb3VudD1vdmVybGFwX2NvdW50'
        'LAogICAgICAgIHByb3RlY3RlZF9zcGxpdHM9WyJjbGVhbl90ZXN0IiwgImNvcnJ1cHRlZF90ZXN0'
        'IiwgIm9vZF90ZXN0Il0sCiAgICAgICAgY29tcG9uZW50X29yZGVyPWxpc3QoQ09NUE9ORU5UX09S'
        'REVSKSwKICAgICAgICByZWxpYWJpbGl0eV9zZW1hbnRpY3M9UkVMSUFCSUxJVFlfVkFMVUVTLAog'
        'ICAgICAgIGNsYWltX2JvdW5kYXJ5PSgKICAgICAgICAgICAgIkNvbnRyb2xsZWQgc3ludGhldGlj'
        'IHJlbGlhYmlsaXR5LW1lY2hhbmlzbSBldmlkZW5jZSBvbmx5OyBubyBuYXR1cmFsLWRhdGEsICIK'
        'ICAgICAgICAgICAgInRhcmdldC1zY2FsZSwgZGVwbG95bWVudCwgc2FmZXR5LCBvciBKRVBBIGNs'
        'YWltIGlzIGxpY2Vuc2VkLiIKICAgICAgICApLAogICAgKQogICAgX2F0b21pY193cml0ZSgKICAg'
        'ICAgICBvdXRwdXQgLyAicmVjZWlwdC5qc29uIiwKICAgICAgICAoanNvbi5kdW1wcyhyZWNlaXB0'
        'Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxu'
        'IikuZW5jb2RlKCksCiAgICApCiAgICByZXR1cm4gcmVjZWlwdAo='
    ),
    'olympus/data/retrieval.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucyBpbXBv'
        'cnQgQ291bnRlcgpmcm9tIG1hdGggaW1wb3J0IHNxcnQKCmZyb20gb2x5bXB1cy5jb3JlLm1vZGFs'
        'aXRpZXMgaW1wb3J0IFJlZmVyZW5jZUVuY29kZXIsIFRleHRBcnRpZmFjdApmcm9tIG9seW1wdXMu'
        'ZGF0YS5pbmdlc3Rpb24gaW1wb3J0IEluZ2VzdGVkRG9jdW1lbnQKCgpjbGFzcyBSZXRyaWV2YWxQ'
        'bGF0Zm9ybToKICAgIGRlZiBfX2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuZW5j'
        'b2RlciA9IFJlZmVyZW5jZUVuY29kZXIoKQoKICAgIGRlZiBsZXhpY2FsX3NlYXJjaCgKICAgICAg'
        'ICBzZWxmLCBxdWVyeTogc3RyLCBkb2N1bWVudHM6IGxpc3RbSW5nZXN0ZWREb2N1bWVudF0KICAg'
        'ICkgLT4gbGlzdFt0dXBsZVtzdHIsIGZsb2F0XV06CiAgICAgICAgcXVlcnlfdG9rZW5zID0gc2V0'
        'KHF1ZXJ5Lmxvd2VyKCkuc3BsaXQoKSkKICAgICAgICBzY29yZWQgPSBbXQogICAgICAgIGZvciBk'
        'b2N1bWVudCBpbiBkb2N1bWVudHM6CiAgICAgICAgICAgIG92ZXJsYXAgPSBxdWVyeV90b2tlbnMu'
        'aW50ZXJzZWN0aW9uKGRvY3VtZW50LnRleHQubG93ZXIoKS5zcGxpdCgpKQogICAgICAgICAgICBz'
        'Y29yZWQuYXBwZW5kKChkb2N1bWVudC5pZGVudGlmaWVyLCBsZW4ob3ZlcmxhcCkgLyBtYXgoMSwg'
        'bGVuKHF1ZXJ5X3Rva2VucykpKSkKICAgICAgICByZXR1cm4gc29ydGVkKHNjb3JlZCwga2V5PWxh'
        'bWJkYSBpdGVtOiBpdGVtWzFdLCByZXZlcnNlPVRydWUpCgogICAgZGVmIHNlbWFudGljX3NlYXJj'
        'aCgKICAgICAgICBzZWxmLCBxdWVyeTogc3RyLCBkb2N1bWVudHM6IGxpc3RbSW5nZXN0ZWREb2N1'
        'bWVudF0KICAgICkgLT4gbGlzdFt0dXBsZVtzdHIsIGZsb2F0XV06CiAgICAgICAgcXVlcnlfdmVj'
        'dG9yID0gc2VsZi5lbmNvZGVyLmVuY29kZShUZXh0QXJ0aWZhY3QoY29udGVudD1xdWVyeSkpLnZl'
        'Y3RvcgogICAgICAgIHNjb3JlZCA9IFtdCiAgICAgICAgZm9yIGRvY3VtZW50IGluIGRvY3VtZW50'
        'czoKICAgICAgICAgICAgZG9jdW1lbnRfdmVjdG9yID0gc2VsZi5lbmNvZGVyLmVuY29kZShUZXh0'
        'QXJ0aWZhY3QoY29udGVudD1kb2N1bWVudC50ZXh0KSkudmVjdG9yCiAgICAgICAgICAgIG51bWVy'
        'YXRvciA9IHN1bShhICogYiBmb3IgYSwgYiBpbiB6aXAocXVlcnlfdmVjdG9yLCBkb2N1bWVudF92'
        'ZWN0b3IsIHN0cmljdD1GYWxzZSkpCiAgICAgICAgICAgIGRlbm9taW5hdG9yID0gKAogICAgICAg'
        'ICAgICAgICAgc3FydChzdW0oYSAqIGEgZm9yIGEgaW4gcXVlcnlfdmVjdG9yKSAqIHN1bShiICog'
        'YiBmb3IgYiBpbiBkb2N1bWVudF92ZWN0b3IpKSBvciAxLjAKICAgICAgICAgICAgKQogICAgICAg'
        'ICAgICBzY29yZWQuYXBwZW5kKChkb2N1bWVudC5pZGVudGlmaWVyLCBudW1lcmF0b3IgLyBkZW5v'
        'bWluYXRvcikpCiAgICAgICAgcmV0dXJuIHNvcnRlZChzY29yZWQsIGtleT1sYW1iZGEgaXRlbTog'
        'aXRlbVsxXSwgcmV2ZXJzZT1UcnVlKQoKICAgIGRlZiBjb250cmFkaWN0aW9uX3NlYXJjaChzZWxm'
        'LCBjbGFpbTogc3RyLCBkb2N1bWVudHM6IGxpc3RbSW5nZXN0ZWREb2N1bWVudF0pIC0+IGxpc3Rb'
        'c3RyXToKICAgICAgICBjbGFpbV9jb3VudGVyID0gQ291bnRlcihjbGFpbS5sb3dlcigpLnNwbGl0'
        'KCkpCiAgICAgICAgY29udHJhZGljdG9yczogbGlzdFtzdHJdID0gW10KICAgICAgICBmb3IgZG9j'
        'dW1lbnQgaW4gZG9jdW1lbnRzOgogICAgICAgICAgICB0ZXh0ID0gZG9jdW1lbnQudGV4dC5sb3dl'
        'cigpCiAgICAgICAgICAgIGlmIGFueShmIm5vdCB7d29yZH0iIGluIHRleHQgZm9yIHdvcmQgaW4g'
        'Y2xhaW1fY291bnRlcik6CiAgICAgICAgICAgICAgICBjb250cmFkaWN0b3JzLmFwcGVuZChkb2N1'
        'bWVudC5pZGVudGlmaWVyKQogICAgICAgIHJldHVybiBjb250cmFkaWN0b3JzCg=='
    ),
    'olympus/data/robust_fusion_v5.py': (
        'IiIiVjUgZnJlc2ggZXhhbXBsZXMsIGV4cGFuZGVkIHRyYWluaW5nIGNvdmVyYWdlLCBhbmQgcmVz'
        'ZXJ2ZWQgT09EIGZhbWlsaWVzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9u'
        'cwoKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvcgoKZnJvbSBvbHltcHVzLmRh'
        'dGEub3JhY2xlX2Z1c2lvbl92NCBpbXBvcnQgZ2VuZXJhdGUgYXMgcHJldmlvdXNfZ2VuZXJhdGUK'
        'CgpkZWYgZ2VuZXJhdGUoc2VlZDogaW50LCBjb3VudDogaW50LCByZWdpbWU6IHN0cikgLT4gZGlj'
        'dFtzdHIsIFRlbnNvcl06CiAgICBpZiByZWdpbWUgbm90IGluIHsiY29yZSIsICJicm9hZCIsICJj'
        'bGVhbiIsICJjb3JydXB0ZWQiLCAia25vd25fc2hpZnQiLCAibm92ZWxfc2hpZnQifToKICAgICAg'
        'ICByYWlzZSBWYWx1ZUVycm9yKHJlZ2ltZSkKICAgIHNvdXJjZSA9IHsKICAgICAgICAiY29yZSI6'
        'ICJtaXhlZCIsCiAgICAgICAgImJyb2FkIjogIm1peGVkIiwKICAgICAgICAia25vd25fc2hpZnQi'
        'OiAib29kIiwKICAgICAgICAibm92ZWxfc2hpZnQiOiAiY29ycnVwdGVkIiwKICAgIH0uZ2V0KHJl'
        'Z2ltZSwgcmVnaW1lKQogICAgYmF0Y2ggPSBwcmV2aW91c19nZW5lcmF0ZShzZWVkLCBjb3VudCwg'
        'c291cmNlKQogICAgZyA9IHRvcmNoLkdlbmVyYXRvcigpLm1hbnVhbF9zZWVkKHNlZWQgKyA5MTAw'
        'MDApCiAgICBpZiByZWdpbWUgPT0gImJyb2FkIjoKICAgICAgICAjIFNhbWUgbGF0ZW50IGV4YW1w'
        'bGVzIGFuZCBtaXNzaW5nbmVzcyBhcyBjb3JlLiBPbmx5IGRhbWFnZWQgb2JzZXJ2YXRpb25zCiAg'
        'ICAgICAgIyBjaGFuZ2U7IGNsZWFuIGFuZCBtaXNzaW5nLW9ubHkgZXhhbXBsZXMgcmVtYWluIGV4'
        'YWN0bHkgbWF0Y2hlZC4KICAgICAgICBjbGVhbiA9IHByZXZpb3VzX2dlbmVyYXRlKHNlZWQsIGNv'
        'dW50LCAiY2xlYW4iKQogICAgICAgIGNob29zZSA9IHRvcmNoLnJhbmQoY291bnQsIGdlbmVyYXRv'
        'cj1nKSA8IDAuNQogICAgICAgIGZvciBpbmRleCwga2V5IGluIGVudW1lcmF0ZSgoInRva2VucyIs'
        'ICJpbWFnZSIsICJzdGF0ZXMiKSk6CiAgICAgICAgICAgIGRhbWFnZWQgPSAoYmF0Y2hbImNvcnJ1'
        'cHRpb25fc3RhdHVzIl1bOiwgaW5kZXhdID09IDIpICYgY2hvb3NlCiAgICAgICAgICAgIHggPSBj'
        'bGVhbltrZXldLmNsb25lKCkKICAgICAgICAgICAgaWYgaW5kZXggPT0gMDoKICAgICAgICAgICAg'
        'ICAgIHhbOiwgOjNdID0geFs6LCA6M10uZmxpcCgtMSkKICAgICAgICAgICAgZWxpZiBpbmRleCA9'
        'PSAxOgogICAgICAgICAgICAgICAgeCA9ICgxIC0geCkuY2xhbXAoMCwgMSkKICAgICAgICAgICAg'
        'ZWxzZToKICAgICAgICAgICAgICAgIHggKz0gMS41CiAgICAgICAgICAgIGJhdGNoW2tleV1bZGFt'
        'YWdlZF0gPSB4W2RhbWFnZWRdCiAgICBlbGlmIHJlZ2ltZSA9PSAibm92ZWxfc2hpZnQiOgogICAg'
        'ICAgIGNsZWFuID0gcHJldmlvdXNfZ2VuZXJhdGUoc2VlZCwgY291bnQsICJjbGVhbiIpCiAgICAg'
        'ICAgZm9yIGluZGV4LCBrZXkgaW4gZW51bWVyYXRlKCgidG9rZW5zIiwgImltYWdlIiwgInN0YXRl'
        'cyIpKToKICAgICAgICAgICAgZGFtYWdlZCA9IGJhdGNoWyJjb3JydXB0aW9uX3N0YXR1cyJdWzos'
        'IGluZGV4XSA9PSAyCiAgICAgICAgICAgIHggPSBjbGVhbltrZXldLmNsb25lKCkKICAgICAgICAg'
        'ICAgaWYgaW5kZXggPT0gMDoKICAgICAgICAgICAgICAgICMgUmVsb2NhdGlvbiBpcyBhYnNlbnQg'
        'ZnJvbSBjb3JlL2Jyb2FkL2RldmVsb3BtZW50IGNvcnJ1cHRpb24uCiAgICAgICAgICAgICAgICB4'
        'ID0geC5yb2xsKDQsIC0xKQogICAgICAgICAgICBlbGlmIGluZGV4ID09IDE6CiAgICAgICAgICAg'
        'ICAgICAjIFNwYXRpYWxseSBzdHJ1Y3R1cmVkIG9jY2x1c2lvbiwgbm90IGludmVyc2lvbiBvciBy'
        'YW5kb20gbm9pc2UuCiAgICAgICAgICAgICAgICB4WzosIDosIDo6MiwgOjoyXSA9IDAuOAogICAg'
        'ICAgICAgICAgICAgeFs6LCA6LCAxOjoyLCAxOjoyXSA9IDAuOAogICAgICAgICAgICBlbHNlOgog'
        'ICAgICAgICAgICAgICAgIyBOZWdhdGl2ZSB0aW1lLXZhcnlpbmcgZHJpZnQsIG5vdCB0aGUga25v'
        'd24gY29uc3RhbnQgcG9zaXRpdmUgYmlhcy4KICAgICAgICAgICAgICAgIHggLT0gdG9yY2gubGlu'
        'c3BhY2UoMC40LCAxLjYsIDQpW05vbmUsIDosIE5vbmVdCiAgICAgICAgICAgIGJhdGNoW2tleV1b'
        'ZGFtYWdlZF0gPSB4W2RhbWFnZWRdCiAgICByZXR1cm4gYmF0Y2gK'
    ),
    'olympus/demos.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQ'
        'YXRoCgpmcm9tIG9seW1wdXMuY29yZS5kaXZpc2lvbiBpbXBvcnQgTmV1cmFsRGl2aXNpb25SZWFz'
        'c2VtYmx5LCBTcGVjaWFsaXN0UmVzdWx0LCBTcGVjaWFsaXN0VGFzawpmcm9tIG9seW1wdXMuY29y'
        'ZS5pbnRlcnByZXRhdGlvbiBpbXBvcnQgSW50ZXJwcmV0aXZlU3VwZXJwb3NpdGlvbk5ldHdvcmsK'
        'ZnJvbSBvbHltcHVzLmNvcmUucmVwcmVzZW50YXRpb24gaW1wb3J0IER5bmFtaWNSZXByZXNlbnRh'
        'dGlvblRoZW9yeSwgUmVwcmVzZW50YXRpb25GYW1pbHkKZnJvbSBvbHltcHVzLmNvcmUucmV0cm9k'
        'aWN0aW9uIGltcG9ydCBSZXRyb2RpY3RpdmVJZGVudGl0eU5ldHdvcmssIGNvZGVfYnVnX2dlbmVy'
        'YXRvcnMKZnJvbSBvbHltcHVzLmNvcmUudHJhbnNmb3JtIGltcG9ydCB0cmFpbl9sZWFybmVkX3Ry'
        'YW5zZm9ybQpmcm9tIG9seW1wdXMuZm9yZ2UuY29tcGlsZXIgaW1wb3J0IE5hdHVyYWxMYW5ndWFn'
        'ZUJlaGF2aW9yQ29tcGlsZXIKZnJvbSBvbHltcHVzLmZvcmdlLnJ1bnRpbWUgaW1wb3J0IEZvcmdl'
        'UnVudGltZQpmcm9tIG9seW1wdXMubWVtb3J5LnN0b3JlIGltcG9ydCBNZW1vcnlTdG9yZQpmcm9t'
        'IG9seW1wdXMubW9kZWxzLmZhbWlsaWVzIGltcG9ydCBIZXJtZXNOYW5vLCBIZXJtZXNSZXNwb25z'
        'ZQpmcm9tIG9seW1wdXMudHJhaW5pbmcuc3ludGhldGljIGltcG9ydCB0cmFuc2Zvcm1fc2lnbmFs'
        'cwoKCmRlZiBydW5fYW1iaWd1b3VzX2ludGVycHJldGF0aW9uX2RlbW8oKSAtPiBkaWN0W3N0ciwg'
        'ZGljdFtzdHIsIG9iamVjdF1dOgogICAgbmV0d29yayA9IEludGVycHJldGl2ZVN1cGVycG9zaXRp'
        'b25OZXR3b3JrKCkKICAgIGJyYW5jaGVzID0gbmV0d29yay5hbmFseXplKCJUaGUgY2xvdWQgd2Fz'
        'IGEgZHJhZ29uIGFib3ZlIHRoZSBjaXR5LiIpCiAgICB0b3BfY2F0ZWdvcnkgPSBicmFuY2hlc1sw'
        'XS5jYXRlZ29yeS52YWx1ZQogICAgcmV0dXJuIHsKICAgICAgICAiYW1iaWd1b3VzX2ludGVycHJl'
        'dGF0aW9uIjogewogICAgICAgICAgICAicGFzc2VkIjogbGVuKGJyYW5jaGVzKSA+PSAzIGFuZCB0'
        'b3BfY2F0ZWdvcnkgaW4geyJsaXRlcmFsIiwgImNvbmNyZXRlX3Jlc2VtYmxhbmNlIn0sCiAgICAg'
        'ICAgICAgICJkZXRhaWwiOiBmIlRvcCBpbnRlcnByZXRhdGlvbiBjYXRlZ29yeToge3RvcF9jYXRl'
        'Z29yeX0iLAogICAgICAgIH0KICAgIH0KCgpkZWYgcnVuX3JldHJvZGljdGlvbl9kZW1vKCkgLT4g'
        'ZGljdFtzdHIsIGRpY3Rbc3RyLCBvYmplY3RdXToKICAgIGN1cnJlbnQgPSAiZGVmIGF2ZXJhZ2Uo'
        'dG90YWwsIGNvdW50KTpcbiAgICByZXR1cm4gdG90YWwgLyBjb3VudFxuIgogICAgbmV0d29yayA9'
        'IFJldHJvZGljdGl2ZUlkZW50aXR5TmV0d29yaygpCiAgICByZXN1bHRzID0gbmV0d29yay5pbmZl'
        'cigKICAgICAgICBjdXJyZW50X3N0YXRlPWN1cnJlbnQsCiAgICAgICAgZ2VuZXJhdG9ycz1jb2Rl'
        'X2J1Z19nZW5lcmF0b3JzKCksCiAgICAgICAgZm9yd2FyZF9zaW11bGF0b3I9bGFtYmRhIGNhbmRp'
        'ZGF0ZTogKAogICAgICAgICAgICBjYW5kaWRhdGUucmVwbGFjZSgicmV0dXJuIHRvdGFsIC8gKGNv'
        'dW50IC0gMSkiLCAicmV0dXJuIHRvdGFsIC8gY291bnQiKQogICAgICAgICAgICAucmVwbGFjZSgi'
        'cmV0dXJuIHRvdGFsIC8gY291bnQgaWYgY291bnQgZWxzZSAwIiwgInJldHVybiB0b3RhbCAvIGNv'
        'dW50IikKICAgICAgICAgICAgLnJlcGxhY2UoInRvdGFsID0rIHZhbHVlIiwgInRvdGFsICs9IHZh'
        'bHVlIikKICAgICAgICApLAogICAgICAgIGludmFyaWFudF9jaGVja2VyPWxhbWJkYSBjYW5kaWRh'
        'dGU6ICJyZXR1cm4iIGluIGNhbmRpZGF0ZSBhbmQgImRlZiBhdmVyYWdlIiBpbiBjYW5kaWRhdGUs'
        'CiAgICApCiAgICByZXR1cm4gewogICAgICAgICJyZXRyb2RpY3Rpb24iOiB7CiAgICAgICAgICAg'
        'ICJwYXNzZWQiOiByZXN1bHRzWzBdLmZvcndhcmRfbWF0Y2gsCiAgICAgICAgICAgICJkZXRhaWwi'
        'OiByZXN1bHRzWzBdLmNhbmRpZGF0ZV9zdGF0ZS5zcGxpdGxpbmVzKClbLTFdLAogICAgICAgIH0K'
        'ICAgIH0KCgpkZWYgcnVuX3RyYW5zZm9ybV9kZW1vKCkgLT4gZGljdFtzdHIsIGRpY3Rbc3RyLCBv'
        'YmplY3RdXToKICAgIG1ldHJpY3MgPSB0cmFpbl9sZWFybmVkX3RyYW5zZm9ybSh0cmFuc2Zvcm1f'
        'c2lnbmFscygpKQogICAgcmV0dXJuIHsKICAgICAgICAibGVhcm5lZF90cmFuc2Zvcm0iOiB7CiAg'
        'ICAgICAgICAgICJwYXNzZWQiOiBtZXRyaWNzLnJlY29uc3RydWN0aW9uX2xvc3MgPCAwLjIsCiAg'
        'ICAgICAgICAgICJkZXRhaWwiOiBmInJlY29uc3RydWN0aW9uX2xvc3M9e21ldHJpY3MucmVjb25z'
        'dHJ1Y3Rpb25fbG9zc30iLAogICAgICAgIH0KICAgIH0KCgpkZWYgcnVuX3JlcHJlc2VudGF0aW9u'
        'X2RlbW8oKSAtPiBkaWN0W3N0ciwgZGljdFtzdHIsIG9iamVjdF1dOgogICAgdGhlb3J5ID0gRHlu'
        'YW1pY1JlcHJlc2VudGF0aW9uVGhlb3J5KCkKICAgIGRlc2NyaXB0b3IgPSB0aGVvcnkuc2VsZWN0'
        'KHsiZWRnZXMiOiBbKCJhIiwgImIiKV0sICJoaWVyYXJjaHlfZGVwdGgiOiAzfSkKICAgIHJldHVy'
        'biB7CiAgICAgICAgInJlcHJlc2VudGF0aW9uX3NlbGVjdGlvbiI6IHsKICAgICAgICAgICAgInBh'
        'c3NlZCI6IGRlc2NyaXB0b3IuZmFtaWx5ID09IFJlcHJlc2VudGF0aW9uRmFtaWx5LlRSRUUsCiAg'
        'ICAgICAgICAgICJkZXRhaWwiOiBkZXNjcmlwdG9yLmZhbWlseS52YWx1ZSwKICAgICAgICB9CiAg'
        'ICB9CgoKZGVmIHJ1bl9kaXZpc2lvbl9kZW1vKCkgLT4gZGljdFtzdHIsIGRpY3Rbc3RyLCBvYmpl'
        'Y3RdXToKICAgIGRpdmlzaW9uID0gTmV1cmFsRGl2aXNpb25SZWFzc2VtYmx5KCkKICAgIHRhc2tz'
        'ID0gWwogICAgICAgIFNwZWNpYWxpc3RUYXNrKG5hbWU9InJlc2VhcmNoIiwgcGF5bG9hZD0iY29s'
        'bGVjdCBldmlkZW5jZSIpLAogICAgICAgIFNwZWNpYWxpc3RUYXNrKG5hbWU9InBsYW5uaW5nIiwg'
        'cGF5bG9hZD0ic2VxdWVuY2UgdGhlIHdvcmsiKSwKICAgIF0KICAgIHNwZWNpYWxpc3RzID0gewog'
        'ICAgICAgICJyZXNlYXJjaCI6IGxhbWJkYSBwYXlsb2FkOiBTcGVjaWFsaXN0UmVzdWx0KAogICAg'
        'ICAgICAgICBuYW1lPSJyZXNlYXJjaCIsIG91dHB1dD1mInJlc2VhcmNoZWQge3BheWxvYWR9Iiwg'
        'Y29uZmlkZW5jZT0wLjgyCiAgICAgICAgKSwKICAgICAgICAicGxhbm5pbmciOiBsYW1iZGEgcGF5'
        'bG9hZDogU3BlY2lhbGlzdFJlc3VsdCgKICAgICAgICAgICAgbmFtZT0icGxhbm5pbmciLCBvdXRw'
        'dXQ9ZiJwbGFubmVkIHtwYXlsb2FkfSIsIGNvbmZpZGVuY2U9MC43OAogICAgICAgICksCiAgICB9'
        'CiAgICBfLCBmaW5hbCA9IGRpdmlzaW9uLmV4ZWN1dGUodGFza3MsIHNwZWNpYWxpc3RzKQogICAg'
        'cmV0dXJuIHsKICAgICAgICAiZGl2aXNpb25fcmVhc3NlbWJseSI6IHsKICAgICAgICAgICAgInBh'
        'c3NlZCI6ICJyZXNlYXJjaCIgaW4gZmluYWwub3V0cHV0IGFuZCAicGxhbm5pbmciIGluIGZpbmFs'
        'Lm91dHB1dCwKICAgICAgICAgICAgImRldGFpbCI6IGZpbmFsLm91dHB1dCwKICAgICAgICB9CiAg'
        'ICB9CgoKZGVmIHJ1bl9mb3JnZV9kZW1vKCkgLT4gZGljdFtzdHIsIGRpY3Rbc3RyLCBvYmplY3Rd'
        'XToKICAgIGNvbXBpbGVyID0gTmF0dXJhbExhbmd1YWdlQmVoYXZpb3JDb21waWxlcigpCiAgICBy'
        'dW50aW1lID0gRm9yZ2VSdW50aW1lKCkKICAgIGNvbXBpbGF0aW9uID0gY29tcGlsZXIuY29tcGls'
        'ZSgKICAgICAgICAiTWFpbnRhaW4gc2V2ZXJhbCBwb3NzaWJsZSBpbnRlcnByZXRhdGlvbnMsIGdp'
        'dmUgZWFjaCBhIGNvbmZpZGVuY2UgIgogICAgICAgICJzY29yZSwgdXNlIHRvb2xzIHRvIHRlc3Qg'
        'cHJlZGljdGlvbnMsIGFuZCBtZXJnZSB0aGUgcmVtYWluaW5nIGNvbmNsdXNpb25zLiIKICAgICkK'
        'ICAgIHJlc3VsdCA9IHJ1bnRpbWUuZXhlY3V0ZShjb21waWxhdGlvbi5zcGVjLCAiQSBkZXBlbmRl'
        'bmN5IGdyYXBoIGxvb2tlZCBsaWtlIGEgZm9yZXN0IGNhbm9weS4iKQogICAgcmV0dXJuIHsKICAg'
        'ICAgICAiZm9yZ2UiOiB7CiAgICAgICAgICAgICJwYXNzZWQiOiAib3V0cHV0IiBpbiByZXN1bHQu'
        'b3V0cHV0cyBhbmQgbGVuKHJlc3VsdC5ldmVudHMpID49IDQsCiAgICAgICAgICAgICJkZXRhaWwi'
        'OiByZXN1bHQub3V0cHV0c1sib3V0cHV0Il1bInN0YXRlbWVudCJdLAogICAgICAgIH0KICAgIH0K'
        'CgpkZWYgcnVuX2hlcm1lc19kZW1vKG1lbW9yeV9wYXRoOiBQYXRoKSAtPiBIZXJtZXNSZXNwb25z'
        'ZToKICAgIHdpdGggTWVtb3J5U3RvcmUobWVtb3J5X3BhdGgpIGFzIG1lbW9yeToKICAgICAgICBo'
        'ZXJtZXMgPSBIZXJtZXNOYW5vKG1lbW9yeSkKICAgICAgICByZXR1cm4gaGVybWVzLnJlc3BvbmQo'
        'IlN1bW1hcml6ZSB0aGUgcHJvamVjdCBzdGF0dXMgYW5kIGtlZXAgaXQgcHJpdmF0ZS4iKQo='
    ),
    'olympus/evaluation/__init__.py': (
        'IiIiRXZhbHVhdGlvbiBoYXJuZXNzLiIiIgo='
    ),
    'olympus/evaluation/decision_dossiers.py': (
        '...'
        'ICByZXR1cm4gYnVuZGxlCg=='
    ),
    'olympus/evaluation/harness.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBjYXN0Cgpmcm9tIG9seW1wdXMuZGVtb3Mg'
        'aW1wb3J0ICgKICAgIHJ1bl9hbWJpZ3VvdXNfaW50ZXJwcmV0YXRpb25fZGVtbywKICAgIHJ1bl9m'
        'b3JnZV9kZW1vLAogICAgcnVuX3JlcHJlc2VudGF0aW9uX2RlbW8sCiAgICBydW5fcmV0cm9kaWN0'
        'aW9uX2RlbW8sCiAgICBydW5fdHJhbnNmb3JtX2RlbW8sCikKCgpAZGF0YWNsYXNzKHNsb3RzPVRy'
        'dWUpCmNsYXNzIEV2YWx1YXRpb25SZXN1bHQ6CiAgICBuYW1lOiBzdHIKICAgIHBhc3NlZDogYm9v'
        'bAogICAgZGV0YWlsOiBzdHIKCgpjbGFzcyBFdmFsdWF0aW9uSGFybmVzczoKICAgIGRlZiBydW4o'
        'c2VsZikgLT4gbGlzdFtFdmFsdWF0aW9uUmVzdWx0XToKICAgICAgICByZXN1bHRzID0gWwogICAg'
        'ICAgICAgICBydW5fYW1iaWd1b3VzX2ludGVycHJldGF0aW9uX2RlbW8oKSwKICAgICAgICAgICAg'
        'cnVuX3JldHJvZGljdGlvbl9kZW1vKCksCiAgICAgICAgICAgIHJ1bl90cmFuc2Zvcm1fZGVtbygp'
        'LAogICAgICAgICAgICBydW5fcmVwcmVzZW50YXRpb25fZGVtbygpLAogICAgICAgICAgICBydW5f'
        'Zm9yZ2VfZGVtbygpLAogICAgICAgIF0KICAgICAgICByZXR1cm4gWwogICAgICAgICAgICBFdmFs'
        'dWF0aW9uUmVzdWx0KAogICAgICAgICAgICAgICAgbmFtZT1uYW1lLAogICAgICAgICAgICAgICAg'
        'cGFzc2VkPWNhc3QoYm9vbCwgcGF5bG9hZFsicGFzc2VkIl0pLAogICAgICAgICAgICAgICAgZGV0'
        'YWlsPWNhc3Qoc3RyLCBwYXlsb2FkWyJkZXRhaWwiXSksCiAgICAgICAgICAgICkKICAgICAgICAg'
        'ICAgZm9yIGRlbW8gaW4gcmVzdWx0cwogICAgICAgICAgICBmb3IgbmFtZSwgcGF5bG9hZCBpbiBk'
        'ZW1vLml0ZW1zKCkKICAgICAgICBdCg=='
    ),
    'olympus/evaluation/olympus_system.py': (
        '...'
        'Y3RpdmVfcmlzayIpLAogICAgKQo='
    ),
    'olympus/forge/__init__.py': (
        'IiIiT2x5bXB1cyBGb3JnZSBiZWhhdmlvciBjb21waWxlciBhbmQgcnVudGltZS4iIiIK'
    ),
    'olympus/forge/compiler.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBweWRhbnRpYyBpbXBvcnQg'
        'RmllbGQKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCmZyb20g'
        'b2x5bXB1cy5mb3JnZS5sYW5ndWFnZSBpbXBvcnQgQmVoYXZpb3JOb2RlLCBCZWhhdmlvclNwZWMs'
        'IE5vZGVLaW5kCgoKY2xhc3MgQ29tcGlsZXJJc3N1ZShTdHJpY3RNb2RlbCk6CiAgICBsZXZlbDog'
        'c3RyCiAgICBkZXRhaWw6IHN0cgoKCmNsYXNzIE5hdHVyYWxMYW5ndWFnZUNvbXBpbGF0aW9uKFN0'
        'cmljdE1vZGVsKToKICAgIHNwZWM6IEJlaGF2aW9yU3BlYwogICAgaXNzdWVzOiBsaXN0W0NvbXBp'
        'bGVySXNzdWVdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCgoKY2xhc3MgTmF0dXJhbExh'
        'bmd1YWdlQmVoYXZpb3JDb21waWxlcjoKICAgIGRlZiBjb21waWxlKAogICAgICAgIHNlbGYsIGRl'
        'c2NyaXB0aW9uOiBzdHIsIG5hbWU6IHN0ciA9ICJjb21waWxlZC1iZWhhdmlvciIKICAgICkgLT4g'
        'TmF0dXJhbExhbmd1YWdlQ29tcGlsYXRpb246CiAgICAgICAgbG93ZXJlZCA9IGRlc2NyaXB0aW9u'
        'Lmxvd2VyKCkKICAgICAgICBub2RlczogbGlzdFtCZWhhdmlvck5vZGVdID0gWwogICAgICAgICAg'
        'ICBCZWhhdmlvck5vZGUoCiAgICAgICAgICAgICAgICBpZD0iaW50ZXJwcmV0Iiwga2luZD1Ob2Rl'
        'S2luZC5JTlRFUlBSRVQsIGRlc2NyaXB0aW9uPSJHZW5lcmF0ZSBpbnRlcnByZXRhdGlvbnMiCiAg'
        'ICAgICAgICAgICksCiAgICAgICAgXQogICAgICAgIGlzc3VlczogbGlzdFtDb21waWxlcklzc3Vl'
        'XSA9IFtdCiAgICAgICAgcHJldmlvdXNfbm9kZSA9ICJpbnRlcnByZXQiCgogICAgICAgIGlmICJj'
        'b25maWRlbmNlIiBpbiBsb3dlcmVkIG9yICJzY29yZSIgaW4gbG93ZXJlZCBvciAidmVyaWZ5IiBp'
        'biBsb3dlcmVkOgogICAgICAgICAgICBub2Rlcy5hcHBlbmQoCiAgICAgICAgICAgICAgICBCZWhh'
        'dmlvck5vZGUoCiAgICAgICAgICAgICAgICAgICAgaWQ9InZlcmlmeSIsCiAgICAgICAgICAgICAg'
        'ICAgICAga2luZD1Ob2RlS2luZC5WRVJJRlksCiAgICAgICAgICAgICAgICAgICAgZGVzY3JpcHRp'
        'b249IlZlcmlmeSBicmFuY2ggc3VwcG9ydCIsCiAgICAgICAgICAgICAgICAgICAgZGVwZW5kc19v'
        'bj1bcHJldmlvdXNfbm9kZV0sCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICkKICAgICAg'
        'ICAgICAgcHJldmlvdXNfbm9kZSA9ICJ2ZXJpZnkiCiAgICAgICAgaWYgInRvb2wiIGluIGxvd2Vy'
        'ZWQgb3IgInRlc3QiIGluIGxvd2VyZWQ6CiAgICAgICAgICAgIG5vZGVzLmFwcGVuZCgKICAgICAg'
        'ICAgICAgICAgIEJlaGF2aW9yTm9kZSgKICAgICAgICAgICAgICAgICAgICBpZD0idG9vbCIsCiAg'
        'ICAgICAgICAgICAgICAgICAga2luZD1Ob2RlS2luZC5UT09MLAogICAgICAgICAgICAgICAgICAg'
        'IGRlc2NyaXB0aW9uPSJFeGVjdXRlIGEgZGV0ZXJtaW5pc3RpYyB0b29sIGNoZWNrIiwKICAgICAg'
        'ICAgICAgICAgICAgICBkZXBlbmRzX29uPVtwcmV2aW91c19ub2RlXSwKICAgICAgICAgICAgICAg'
        'ICkKICAgICAgICAgICAgKQogICAgICAgICAgICBwcmV2aW91c19ub2RlID0gInRvb2wiCiAgICAg'
        'ICAgaWYgIm1lbW9yeSIgaW4gbG93ZXJlZDoKICAgICAgICAgICAgbm9kZXMuYXBwZW5kKAogICAg'
        'ICAgICAgICAgICAgQmVoYXZpb3JOb2RlKAogICAgICAgICAgICAgICAgICAgIGlkPSJtZW1vcnki'
        'LAogICAgICAgICAgICAgICAgICAgIGtpbmQ9Tm9kZUtpbmQuTUVNT1JZX1dSSVRFLAogICAgICAg'
        'ICAgICAgICAgICAgIGRlc2NyaXB0aW9uPSJQZXJzaXN0IHVzZWZ1bCBmaW5kaW5ncyIsCiAgICAg'
        'ICAgICAgICAgICAgICAgZGVwZW5kc19vbj1bcHJldmlvdXNfbm9kZV0sCiAgICAgICAgICAgICAg'
        'ICApCiAgICAgICAgICAgICkKICAgICAgICAgICAgcHJldmlvdXNfbm9kZSA9ICJtZW1vcnkiCgog'
        'ICAgICAgIG5vZGVzLmFwcGVuZCgKICAgICAgICAgICAgQmVoYXZpb3JOb2RlKAogICAgICAgICAg'
        'ICAgICAgaWQ9Im1lcmdlIiwKICAgICAgICAgICAgICAgIGtpbmQ9Tm9kZUtpbmQuTUVSR0UsCiAg'
        'ICAgICAgICAgICAgICBkZXNjcmlwdGlvbj0iTWVyZ2Ugc3Vydml2aW5nIGJyYW5jaGVzIiwKICAg'
        'ICAgICAgICAgICAgIGRlcGVuZHNfb249W3ByZXZpb3VzX25vZGVdLAogICAgICAgICAgICApCiAg'
        'ICAgICAgKQogICAgICAgIG5vZGVzLmFwcGVuZCgKICAgICAgICAgICAgQmVoYXZpb3JOb2RlKAog'
        'ICAgICAgICAgICAgICAgaWQ9Im91dHB1dCIsCiAgICAgICAgICAgICAgICBraW5kPU5vZGVLaW5k'
        'Lk9VVFBVVCwKICAgICAgICAgICAgICAgIGRlc2NyaXB0aW9uPSJSZXR1cm4gZmluYWwgcmVzdWx0'
        'IiwKICAgICAgICAgICAgICAgIGRlcGVuZHNfb249WyJtZXJnZSJdLAogICAgICAgICAgICApCiAg'
        'ICAgICAgKQoKICAgICAgICBpZiAiY3JlYXRpdmUiIGluIGxvd2VyZWQgYW5kICJsb3ctY29uZmlk'
        'ZW5jZSIgbm90IGluIGxvd2VyZWQ6CiAgICAgICAgICAgIGlzc3Vlcy5hcHBlbmQoCiAgICAgICAg'
        'ICAgICAgICBDb21waWxlcklzc3VlKAogICAgICAgICAgICAgICAgICAgIGxldmVsPSJ3YXJuaW5n'
        'IiwKICAgICAgICAgICAgICAgICAgICBkZXRhaWw9KAogICAgICAgICAgICAgICAgICAgICAgICAi'
        'Q3JlYXRpdmUgZXhwbG9yYXRpb24gcmVxdWVzdGVkIHdpdGhvdXQgZXhwbGljaXQgbG93LWNvbmZp'
        'ZGVuY2UgYWxsb3dhbmNlLiIKICAgICAgICAgICAgICAgICAgICApLAogICAgICAgICAgICAgICAg'
        'KQogICAgICAgICAgICApCgogICAgICAgIHNwZWMgPSBCZWhhdmlvclNwZWMoCiAgICAgICAgICAg'
        'IG5hbWU9bmFtZSwKICAgICAgICAgICAgZGVzY3JpcHRpb249ZGVzY3JpcHRpb24sCiAgICAgICAg'
        'ICAgIG9iamVjdGl2ZT1kZXNjcmlwdGlvbiwKICAgICAgICAgICAgbm9kZXM9bm9kZXMsCiAgICAg'
        'ICAgICAgIG91dHB1dHM9WyJvdXRwdXQiXSwKICAgICAgICApCiAgICAgICAgcmV0dXJuIE5hdHVy'
        'YWxMYW5ndWFnZUNvbXBpbGF0aW9uKHNwZWM9c3BlYywgaXNzdWVzPWlzc3VlcykK'
    ),
    'olympus/forge/experiments.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzCmZyb20gc3RhdGlzdGljcyBpbXBvcnQgbWVhbgoKZnJvbSBvbHltcHVzLmZv'
        'cmdlLmNvbXBpbGVyIGltcG9ydCBOYXR1cmFsTGFuZ3VhZ2VCZWhhdmlvckNvbXBpbGVyCmZyb20g'
        'b2x5bXB1cy5mb3JnZS5ydW50aW1lIGltcG9ydCBGb3JnZVJ1bnRpbWUKCgpAZGF0YWNsYXNzKHNs'
        'b3RzPVRydWUpCmNsYXNzIEV4cGVyaW1lbnRTdW1tYXJ5OgogICAgdHJpYWxfY291bnQ6IGludAog'
        'ICAgbWVhbl9jb25maWRlbmNlOiBmbG9hdAogICAgbWVhbl9zdXJ2aXZvcnM6IGZsb2F0CgoKY2xh'
        'c3MgRXhwZXJpbWVudENvbnRyb2xsZXI6CiAgICBkZWYgX19pbml0X18oc2VsZikgLT4gTm9uZToK'
        'ICAgICAgICBzZWxmLmNvbXBpbGVyID0gTmF0dXJhbExhbmd1YWdlQmVoYXZpb3JDb21waWxlcigp'
        'CiAgICAgICAgc2VsZi5ydW50aW1lID0gRm9yZ2VSdW50aW1lKCkKCiAgICBkZWYgcnVuX3RyaWFs'
        'cyhzZWxmLCBiZWhhdmlvcl90ZXh0OiBzdHIsIHByb21wdDogc3RyLCB0cmlhbHM6IGludCA9IDMp'
        'IC0+IEV4cGVyaW1lbnRTdW1tYXJ5OgogICAgICAgIGlmIHRyaWFscyA8IDE6CiAgICAgICAgICAg'
        'IHJhaXNlIFZhbHVlRXJyb3IoInRyaWFscyBtdXN0IGJlIHBvc2l0aXZlIikKICAgICAgICBjb21w'
        'aWxhdGlvbiA9IHNlbGYuY29tcGlsZXIuY29tcGlsZShiZWhhdmlvcl90ZXh0KQogICAgICAgIGNv'
        'bmZpZGVuY2VzOiBsaXN0W2Zsb2F0XSA9IFtdCiAgICAgICAgc3Vydml2b3JzOiBsaXN0W2ludF0g'
        'PSBbXQogICAgICAgIGZvciBfIGluIHJhbmdlKHRyaWFscyk6CiAgICAgICAgICAgIHJlc3VsdCA9'
        'IHNlbGYucnVudGltZS5leGVjdXRlKGNvbXBpbGF0aW9uLnNwZWMsIHByb21wdCkKICAgICAgICAg'
        'ICAgbWVyZ2VkID0gcmVzdWx0Lm91dHB1dHNbIm1lcmdlZCJdCiAgICAgICAgICAgIGNvbmZpZGVu'
        'Y2VzLmFwcGVuZChmbG9hdChtZXJnZWRbImNvbmZpZGVuY2UiXSkpCiAgICAgICAgICAgIHN1cnZp'
        'dm9ycy5hcHBlbmQoaW50KG1lcmdlZFsic3Vydml2b3JzIl0pKQogICAgICAgIHJldHVybiBFeHBl'
        'cmltZW50U3VtbWFyeSgKICAgICAgICAgICAgdHJpYWxfY291bnQ9dHJpYWxzLAogICAgICAgICAg'
        'ICBtZWFuX2NvbmZpZGVuY2U9cm91bmQobWVhbihjb25maWRlbmNlcyksIDQpLAogICAgICAgICAg'
        'ICBtZWFuX3N1cnZpdm9ycz1yb3VuZChtZWFuKHN1cnZpdm9ycyksIDQpLAogICAgICAgICkK'
    ),
    'olympus/forge/language.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KZnJvbSBlbnVt'
        'IGltcG9ydCBTdHJFbnVtCmZyb20gZ3JhcGhsaWIgaW1wb3J0IFRvcG9sb2dpY2FsU29ydGVyCmZy'
        'b20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9ydCB5YW1sCmZyb20gcHlkYW50aWMgaW1wb3J0IEZp'
        'ZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbAoKCmNsYXNz'
        'IE5vZGVLaW5kKFN0ckVudW0pOgogICAgSU5URVJQUkVUID0gImludGVycHJldCIKICAgIFZFUklG'
        'WSA9ICJ2ZXJpZnkiCiAgICBNRVJHRSA9ICJtZXJnZSIKICAgIE1FTU9SWV9XUklURSA9ICJtZW1v'
        'cnlfd3JpdGUiCiAgICBUT09MID0gInRvb2wiCiAgICBPVVRQVVQgPSAib3V0cHV0IgoKCmNsYXNz'
        'IEJlaGF2aW9yTm9kZShTdHJpY3RNb2RlbCk6CiAgICBpZDogc3RyCiAgICBraW5kOiBOb2RlS2lu'
        'ZAogICAgZGVzY3JpcHRpb246IHN0cgogICAgY29uZmlnOiBkaWN0W3N0ciwgQW55XSA9IEZpZWxk'
        'KGRlZmF1bHRfZmFjdG9yeT1kaWN0KQogICAgZGVwZW5kc19vbjogbGlzdFtzdHJdID0gRmllbGQo'
        'ZGVmYXVsdF9mYWN0b3J5PWxpc3QpCgoKY2xhc3MgQmVoYXZpb3JTcGVjKFN0cmljdE1vZGVsKToK'
        'ICAgIHZlcnNpb246IHN0ciA9ICIxLjAiCiAgICBuYW1lOiBzdHIKICAgIGRlc2NyaXB0aW9uOiBz'
        'dHIKICAgIG9iamVjdGl2ZTogc3RyCiAgICBub2RlczogbGlzdFtCZWhhdmlvck5vZGVdCiAgICBv'
        'dXRwdXRzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCiAgICBkZWYg'
        'dG9fbWVybWFpZChzZWxmKSAtPiBzdHI6CiAgICAgICAgbGluZXMgPSBbImdyYXBoIFREIl0KICAg'
        'ICAgICBmb3Igbm9kZSBpbiBzZWxmLm5vZGVzOgogICAgICAgICAgICBsaW5lcy5hcHBlbmQoZicg'
        'IHtub2RlLmlkfVsie25vZGUuaWR9OiB7bm9kZS5raW5kLnZhbHVlfSJdJykKICAgICAgICAgICAg'
        'Zm9yIGRlcGVuZGVuY3kgaW4gbm9kZS5kZXBlbmRzX29uOgogICAgICAgICAgICAgICAgbGluZXMu'
        'YXBwZW5kKGYiICB7ZGVwZW5kZW5jeX0gLS0+IHtub2RlLmlkfSIpCiAgICAgICAgcmV0dXJuICJc'
        'biIuam9pbihsaW5lcykKCiAgICBkZWYgdG9wb2xvZ2ljYWxfb3JkZXIoc2VsZikgLT4gbGlzdFtz'
        'dHJdOgogICAgICAgIGdyYXBoID0ge25vZGUuaWQ6IHNldChub2RlLmRlcGVuZHNfb24pIGZvciBu'
        'b2RlIGluIHNlbGYubm9kZXN9CiAgICAgICAgcmV0dXJuIGxpc3QoVG9wb2xvZ2ljYWxTb3J0ZXIo'
        'Z3JhcGgpLnN0YXRpY19vcmRlcigpKQoKCmNsYXNzIEJlaGF2aW9yTGFuZ3VhZ2U6CiAgICBkZWYg'
        'cGFyc2Uoc2VsZiwgcGF5bG9hZDogc3RyKSAtPiBCZWhhdmlvclNwZWM6CiAgICAgICAgY29udGVu'
        'dCA9ICgKICAgICAgICAgICAgeWFtbC5zYWZlX2xvYWQocGF5bG9hZCkKICAgICAgICAgICAgaWYg'
        'cGF5bG9hZC5zdHJpcCgpLnN0YXJ0c3dpdGgoKCJ2ZXJzaW9uOiIsICJuYW1lOiIpKQogICAgICAg'
        'ICAgICBlbHNlIGpzb24ubG9hZHMocGF5bG9hZCkKICAgICAgICApCiAgICAgICAgcmV0dXJuIEJl'
        'aGF2aW9yU3BlYy5tb2RlbF92YWxpZGF0ZShjb250ZW50KQoKICAgIGRlZiBmb3JtYXQoc2VsZiwg'
        'c3BlYzogQmVoYXZpb3JTcGVjKSAtPiBzdHI6CiAgICAgICAgcmV0dXJuIHlhbWwuc2FmZV9kdW1w'
        'KHNwZWMubW9kZWxfZHVtcChtb2RlPSJqc29uIiksIHNvcnRfa2V5cz1GYWxzZSkKCiAgICBkZWYg'
        'c2NoZW1hKHNlbGYpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgICAgIHJldHVybiBCZWhhdmlvclNw'
        'ZWMubW9kZWxfanNvbl9zY2hlbWEoKQoKICAgIGRlZiBtaWdyYXRlKHNlbGYsIHBheWxvYWQ6IGRp'
        'Y3Rbc3RyLCBBbnldKSAtPiBCZWhhdmlvclNwZWM6CiAgICAgICAgcGF5bG9hZC5zZXRkZWZhdWx0'
        'KCJ2ZXJzaW9uIiwgIjEuMCIpCiAgICAgICAgcGF5bG9hZC5zZXRkZWZhdWx0KCJvdXRwdXRzIiwg'
        'WyJmaW5hbF9vdXRwdXQiXSkKICAgICAgICByZXR1cm4gQmVoYXZpb3JTcGVjLm1vZGVsX3ZhbGlk'
        'YXRlKHBheWxvYWQpCgogICAgZGVmIHNlbWFudGljX2NoZWNrKHNlbGYsIHNwZWM6IEJlaGF2aW9y'
        'U3BlYykgLT4gbGlzdFtzdHJdOgogICAgICAgIG5vZGVfaWRzID0ge25vZGUuaWQgZm9yIG5vZGUg'
        'aW4gc3BlYy5ub2Rlc30KICAgICAgICBlcnJvcnMgPSBbCiAgICAgICAgICAgIGYiTWlzc2luZyBk'
        'ZXBlbmRlbmN5OiB7ZGVwZW5kZW5jeX0iCiAgICAgICAgICAgIGZvciBub2RlIGluIHNwZWMubm9k'
        'ZXMKICAgICAgICAgICAgZm9yIGRlcGVuZGVuY3kgaW4gbm9kZS5kZXBlbmRzX29uCiAgICAgICAg'
        'ICAgIGlmIGRlcGVuZGVuY3kgbm90IGluIG5vZGVfaWRzCiAgICAgICAgXQogICAgICAgIHRyeToK'
        'ICAgICAgICAgICAgc3BlYy50b3BvbG9naWNhbF9vcmRlcigpCiAgICAgICAgZXhjZXB0IEV4Y2Vw'
        'dGlvbiBhcyBlcnJvcjogICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQo'
        'c3RyKGVycm9yKSkKICAgICAgICByZXR1cm4gZXJyb3JzCg=='
    ),
    'olympus/forge/runtime.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmZyb20gb2x5bXB1cy5jb3JlLmlu'
        'dGVycHJldGF0aW9uIGltcG9ydCBJbnRlcnByZXRpdmVTdXBlcnBvc2l0aW9uTmV0d29yawpmcm9t'
        'IG9seW1wdXMuY29yZS5vdXRjb21lIGltcG9ydCBPdXRjb21lUXVhbGl0eVByZWRpY3Rvcgpmcm9t'
        'IG9seW1wdXMuY29yZS52ZXJpZmljYXRpb24gaW1wb3J0IENhbGlicmF0aW9uVmVyaWZpZXIsIENv'
        'bnRyYWRpY3Rpb25WZXJpZmllcgpmcm9tIG9seW1wdXMuZm9yZ2UubGFuZ3VhZ2UgaW1wb3J0IEJl'
        'aGF2aW9yU3BlYywgTm9kZUtpbmQKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIFJ1bnRp'
        'bWVFdmVudDoKICAgIG5vZGVfaWQ6IHN0cgogICAga2luZDogc3RyCiAgICBwYXlsb2FkOiBkaWN0'
        'W3N0ciwgQW55XQoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgUnVudGltZVJlc3VsdDoK'
        'ICAgIG91dHB1dHM6IGRpY3Rbc3RyLCBBbnldCiAgICBldmVudHM6IGxpc3RbUnVudGltZUV2ZW50'
        'XQoKCmNsYXNzIEZvcmdlUnVudGltZToKICAgIGRlZiBfX2luaXRfXyhzZWxmKSAtPiBOb25lOgog'
        'ICAgICAgIHNlbGYuaW50ZXJwcmV0ZXIgPSBJbnRlcnByZXRpdmVTdXBlcnBvc2l0aW9uTmV0d29y'
        'aygpCiAgICAgICAgc2VsZi5vdXRjb21lX3ByZWRpY3RvciA9IE91dGNvbWVRdWFsaXR5UHJlZGlj'
        'dG9yKCkKICAgICAgICBzZWxmLmNvbnRyYWRpY3Rpb25fdmVyaWZpZXIgPSBDb250cmFkaWN0aW9u'
        'VmVyaWZpZXIoKQogICAgICAgIHNlbGYuY2FsaWJyYXRpb25fdmVyaWZpZXIgPSBDYWxpYnJhdGlv'
        'blZlcmlmaWVyKCkKCiAgICBkZWYgZXhlY3V0ZShzZWxmLCBzcGVjOiBCZWhhdmlvclNwZWMsIHBy'
        'b21wdDogc3RyKSAtPiBSdW50aW1lUmVzdWx0OgogICAgICAgIG91dHB1dHM6IGRpY3Rbc3RyLCBB'
        'bnldID0geyJwcm9tcHQiOiBwcm9tcHR9CiAgICAgICAgZXZlbnRzOiBsaXN0W1J1bnRpbWVFdmVu'
        'dF0gPSBbXQogICAgICAgIGludGVycHJldGF0aW9ucyA9IHNlbGYuaW50ZXJwcmV0ZXIuYW5hbHl6'
        'ZShwcm9tcHQpCgogICAgICAgIGZvciBub2RlX2lkIGluIHNwZWMudG9wb2xvZ2ljYWxfb3JkZXIo'
        'KToKICAgICAgICAgICAgbm9kZSA9IG5leHQoaXRlbSBmb3IgaXRlbSBpbiBzcGVjLm5vZGVzIGlm'
        'IGl0ZW0uaWQgPT0gbm9kZV9pZCkKICAgICAgICAgICAgaWYgbm9kZS5raW5kID09IE5vZGVLaW5k'
        'LklOVEVSUFJFVDoKICAgICAgICAgICAgICAgIG91dHB1dHNbImludGVycHJldGF0aW9ucyJdID0g'
        'WwogICAgICAgICAgICAgICAgICAgIGJyYW5jaC5tb2RlbF9kdW1wKG1vZGU9Impzb24iKSBmb3Ig'
        'YnJhbmNoIGluIGludGVycHJldGF0aW9ucwogICAgICAgICAgICAgICAgXQogICAgICAgICAgICAg'
        'ICAgZXZlbnRzLmFwcGVuZCgKICAgICAgICAgICAgICAgICAgICBSdW50aW1lRXZlbnQoCiAgICAg'
        'ICAgICAgICAgICAgICAgICAgIG5vZGVfaWQ9bm9kZS5pZCwKICAgICAgICAgICAgICAgICAgICAg'
        'ICAga2luZD1ub2RlLmtpbmQudmFsdWUsCiAgICAgICAgICAgICAgICAgICAgICAgIHBheWxvYWQ9'
        'eyJjb3VudCI6IGxlbihpbnRlcnByZXRhdGlvbnMpfSwKICAgICAgICAgICAgICAgICAgICApCiAg'
        'ICAgICAgICAgICAgICApCiAgICAgICAgICAgIGVsaWYgbm9kZS5raW5kID09IE5vZGVLaW5kLlZF'
        'UklGWToKICAgICAgICAgICAgICAgIHRvcF9zdGF0ZW1lbnQgPSBpbnRlcnByZXRhdGlvbnNbMF0u'
        'c3RhdGVtZW50CiAgICAgICAgICAgICAgICBjb250cmFkaWN0aW9uID0gc2VsZi5jb250cmFkaWN0'
        'aW9uX3ZlcmlmaWVyLnZlcmlmeShbdG9wX3N0YXRlbWVudF0pCiAgICAgICAgICAgICAgICBjYWxp'
        'YnJhdGlvbiA9IHNlbGYuY2FsaWJyYXRpb25fdmVyaWZpZXIudmVyaWZ5KAogICAgICAgICAgICAg'
        'ICAgICAgIGludGVycHJldGF0aW9uc1swXS5jb25maWRlbmNlLCBjb250cmFkaWN0aW9uLnBhc3Nl'
        'ZAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgb3V0cHV0c1sidmVyaWZpY2F0aW9u'
        'Il0gPSBbY29udHJhZGljdGlvbi5tb2RlbF9kdW1wKCksIGNhbGlicmF0aW9uLm1vZGVsX2R1bXAo'
        'KV0KICAgICAgICAgICAgICAgIGV2ZW50cy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgUnVu'
        'dGltZUV2ZW50KAogICAgICAgICAgICAgICAgICAgICAgICBub2RlX2lkPW5vZGUuaWQsCiAgICAg'
        'ICAgICAgICAgICAgICAgICAgIGtpbmQ9bm9kZS5raW5kLnZhbHVlLAogICAgICAgICAgICAgICAg'
        'ICAgICAgICBwYXlsb2FkPXsicGFzc2VkIjogY29udHJhZGljdGlvbi5wYXNzZWQgYW5kIGNhbGli'
        'cmF0aW9uLnBhc3NlZH0sCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgKQog'
        'ICAgICAgICAgICBlbGlmIG5vZGUua2luZCA9PSBOb2RlS2luZC5UT09MOgogICAgICAgICAgICAg'
        'ICAgcXVhbGl0eSA9IHNlbGYub3V0Y29tZV9wcmVkaWN0b3IucHJlZGljdChwcm9tcHQsIGV2aWRl'
        'bmNlX2NvdW50PTEsIHRvb2xfY291bnQ9MSkKICAgICAgICAgICAgICAgIG91dHB1dHNbInRvb2xf'
        'cmVzdWx0Il0gPSBxdWFsaXR5Lm1vZGVsX2R1bXAoKQogICAgICAgICAgICAgICAgZXZlbnRzLmFw'
        'cGVuZCgKICAgICAgICAgICAgICAgICAgICBSdW50aW1lRXZlbnQoCiAgICAgICAgICAgICAgICAg'
        'ICAgICAgIG5vZGVfaWQ9bm9kZS5pZCwga2luZD1ub2RlLmtpbmQudmFsdWUsIHBheWxvYWQ9b3V0'
        'cHV0c1sidG9vbF9yZXN1bHQiXQogICAgICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAg'
        'ICkKICAgICAgICAgICAgZWxpZiBub2RlLmtpbmQgPT0gTm9kZUtpbmQuTUVNT1JZX1dSSVRFOgog'
        'ICAgICAgICAgICAgICAgb3V0cHV0c1sibWVtb3J5X3dyaXRlIl0gPSB7CiAgICAgICAgICAgICAg'
        'ICAgICAgInN0YXR1cyI6ICJjYXB0dXJlZCIsCiAgICAgICAgICAgICAgICAgICAgImtleXMiOiBb'
        'ImludGVycHJldGF0aW9ucyIsICJ2ZXJpZmljYXRpb24iXSwKICAgICAgICAgICAgICAgIH0KICAg'
        'ICAgICAgICAgICAgIGV2ZW50cy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgUnVudGltZUV2'
        'ZW50KAogICAgICAgICAgICAgICAgICAgICAgICBub2RlX2lkPW5vZGUuaWQsIGtpbmQ9bm9kZS5r'
        'aW5kLnZhbHVlLCBwYXlsb2FkPW91dHB1dHNbIm1lbW9yeV93cml0ZSJdCiAgICAgICAgICAgICAg'
        'ICAgICAgKQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbGlmIG5vZGUua2luZCA9PSBO'
        'b2RlS2luZC5NRVJHRToKICAgICAgICAgICAgICAgIGJlc3QgPSBpbnRlcnByZXRhdGlvbnNbMF0K'
        'ICAgICAgICAgICAgICAgIG91dHB1dHNbIm1lcmdlZCJdID0gewogICAgICAgICAgICAgICAgICAg'
        'ICJzdGF0ZW1lbnQiOiBiZXN0LnN0YXRlbWVudCwKICAgICAgICAgICAgICAgICAgICAiY29uZmlk'
        'ZW5jZSI6IGJlc3QuY29uZmlkZW5jZSwKICAgICAgICAgICAgICAgICAgICAic3Vydml2b3JzIjog'
        'bGVuKGludGVycHJldGF0aW9ucyksCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICBl'
        'dmVudHMuYXBwZW5kKAogICAgICAgICAgICAgICAgICAgIFJ1bnRpbWVFdmVudChub2RlX2lkPW5v'
        'ZGUuaWQsIGtpbmQ9bm9kZS5raW5kLnZhbHVlLCBwYXlsb2FkPW91dHB1dHNbIm1lcmdlZCJdKQog'
        'ICAgICAgICAgICAgICAgKQogICAgICAgICAgICBlbGlmIG5vZGUua2luZCA9PSBOb2RlS2luZC5P'
        'VVRQVVQ6CiAgICAgICAgICAgICAgICBvdXRwdXRzWyJvdXRwdXQiXSA9IG91dHB1dHMuZ2V0KCJt'
        'ZXJnZWQiLCB7fSkKICAgICAgICAgICAgICAgIGV2ZW50cy5hcHBlbmQoCiAgICAgICAgICAgICAg'
        'ICAgICAgUnVudGltZUV2ZW50KG5vZGVfaWQ9bm9kZS5pZCwga2luZD1ub2RlLmtpbmQudmFsdWUs'
        'IHBheWxvYWQ9b3V0cHV0c1sib3V0cHV0Il0pCiAgICAgICAgICAgICAgICApCgogICAgICAgIHJl'
        'dHVybiBSdW50aW1lUmVzdWx0KG91dHB1dHM9b3V0cHV0cywgZXZlbnRzPWV2ZW50cykK'
    ),
    'olympus/foundry/__init__.py': (
        'IiIiRHVyYWJsZSBtb2RlbC1kZXZlbG9wbWVudCBpbmZyYXN0cnVjdHVyZSBmb3IgT2x5bXB1cy4i'
        'IiIKCmZyb20gb2x5bXB1cy5mb3VuZHJ5LnNlcnZpY2UgaW1wb3J0IEZvdW5kcnlTZXJ2aWNlCgpf'
        'X2FsbF9fID0gWyJGb3VuZHJ5U2VydmljZSJdCg=='
    ),
    'olympus/foundry/bigram.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG1h'
        'dGgKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcmFuZG9tIGltcG9ydCBS'
        'YW5kb20KZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgU2VsZgoKZnJvbSBweWRhbnRpYyBpbXBvcnQg'
        'RmllbGQKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCgoKY2xh'
        'c3MgQmlncmFtQ2hlY2twb2ludChTdHJpY3RNb2RlbCk6CiAgICBzY2hlbWFfdmVyc2lvbjogaW50'
        'ID0gMQogICAgbW9kZWxfdHlwZTogc3RyID0gImNoYXJhY3Rlci1iaWdyYW0iCiAgICBhbHBoYWJl'
        'dDogbGlzdFtzdHJdID0gRmllbGQobWluX2xlbmd0aD0yKQogICAgY291bnRzOiBsaXN0W2xpc3Rb'
        'aW50XV0KICAgIHNtb290aGluZzogZmxvYXQgPSBGaWVsZChndD0wLjAsIGxlPTEwLjApCiAgICBz'
        'ZWVkOiBpbnQgPSBGaWVsZChnZT0wLCBsZT0yKiozMiAtIDEpCiAgICB0cmFpbmluZ19jaGFyYWN0'
        'ZXJzOiBpbnQgPSBGaWVsZChndD0xKQogICAgdHJhaW5pbmdfcHJvdmVuYW5jZTogZGljdFtzdHIs'
        'IEFueV0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKCiAgICBkZWYgY2Fub25pY2FsX2J5'
        'dGVzKHNlbGYpIC0+IGJ5dGVzOgogICAgICAgIHJldHVybiBqc29uLmR1bXBzKAogICAgICAgICAg'
        'ICBzZWxmLm1vZGVsX2R1bXAobW9kZT0ianNvbiIpLAogICAgICAgICAgICBlbnN1cmVfYXNjaWk9'
        'RmFsc2UsCiAgICAgICAgICAgIHNvcnRfa2V5cz1UcnVlLAogICAgICAgICAgICBzZXBhcmF0b3Jz'
        'PSgiLCIsICI6IiksCiAgICAgICAgKS5lbmNvZGUoInV0Zi04IikKCiAgICBAY2xhc3NtZXRob2QK'
        'ICAgIGRlZiBmcm9tX2J5dGVzKGNscywgcGF5bG9hZDogYnl0ZXMpIC0+IFNlbGY6CiAgICAgICAg'
        'cmV0dXJuIGNscy5tb2RlbF92YWxpZGF0ZV9qc29uKHBheWxvYWQpCgoKQGRhdGFjbGFzcyhmcm96'
        'ZW49VHJ1ZSwgc2xvdHM9VHJ1ZSkKY2xhc3MgTGFuZ3VhZ2VNb2RlbE1ldHJpY3M6CiAgICBuZWdh'
        'dGl2ZV9sb2dfbGlrZWxpaG9vZDogZmxvYXQKICAgIHBlcnBsZXhpdHk6IGZsb2F0CiAgICB0b3Ax'
        'X2FjY3VyYWN5OiBmbG9hdAogICAgZXZhbHVhdGVkX3RyYW5zaXRpb25zOiBpbnQKCiAgICBkZWYg'
        'YXNfZGljdChzZWxmKSAtPiBkaWN0W3N0ciwgZmxvYXRdOgogICAgICAgIHJldHVybiB7CiAgICAg'
        'ICAgICAgICJuZWdhdGl2ZV9sb2dfbGlrZWxpaG9vZCI6IHNlbGYubmVnYXRpdmVfbG9nX2xpa2Vs'
        'aWhvb2QsCiAgICAgICAgICAgICJwZXJwbGV4aXR5Ijogc2VsZi5wZXJwbGV4aXR5LAogICAgICAg'
        'ICAgICAidG9wMV9hY2N1cmFjeSI6IHNlbGYudG9wMV9hY2N1cmFjeSwKICAgICAgICAgICAgImV2'
        'YWx1YXRlZF90cmFuc2l0aW9ucyI6IGZsb2F0KHNlbGYuZXZhbHVhdGVkX3RyYW5zaXRpb25zKSwK'
        'ICAgICAgICB9CgoKY2xhc3MgQ2hhcmFjdGVyQmlncmFtTW9kZWw6CiAgICAiIiJBIHNtYWxsIHJl'
        'YWwgbGFuZ3VhZ2UgbW9kZWwgdXNlZCB0byB2ZXJpZnkgdGhlIEZvdW5kcnkgbGlmZWN5Y2xlIGNo'
        'ZWFwbHkuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNoZWNrcG9pbnQ6IEJpZ3JhbUNoZWNr'
        'cG9pbnQpIC0+IE5vbmU6CiAgICAgICAgc2l6ZSA9IGxlbihjaGVja3BvaW50LmFscGhhYmV0KQog'
        'ICAgICAgIGlmIGxlbihzZXQoY2hlY2twb2ludC5hbHBoYWJldCkpICE9IHNpemUgb3IgYW55KAog'
        'ICAgICAgICAgICBsZW4oY2hhcmFjdGVyKSAhPSAxIGZvciBjaGFyYWN0ZXIgaW4gY2hlY2twb2lu'
        'dC5hbHBoYWJldAogICAgICAgICk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNr'
        'cG9pbnQgYWxwaGFiZXQgbXVzdCBjb250YWluIHVuaXF1ZSBzaW5nbGUgY2hhcmFjdGVycyIpCiAg'
        'ICAgICAgaWYgbGVuKGNoZWNrcG9pbnQuY291bnRzKSAhPSBzaXplIG9yIGFueShsZW4ocm93KSAh'
        'PSBzaXplIGZvciByb3cgaW4gY2hlY2twb2ludC5jb3VudHMpOgogICAgICAgICAgICByYWlzZSBW'
        'YWx1ZUVycm9yKCJjaGVja3BvaW50IGNvdW50IG1hdHJpeCBkb2VzIG5vdCBtYXRjaCBhbHBoYWJl'
        'dCIpCiAgICAgICAgaWYgYW55KHZhbHVlIDwgMCBmb3Igcm93IGluIGNoZWNrcG9pbnQuY291bnRz'
        'IGZvciB2YWx1ZSBpbiByb3cpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3Bv'
        'aW50IGNvdW50cyBjYW5ub3QgYmUgbmVnYXRpdmUiKQogICAgICAgIHNlbGYuY2hlY2twb2ludCA9'
        'IGNoZWNrcG9pbnQKICAgICAgICBzZWxmLl9pbmRleCA9IHtjaGFyYWN0ZXI6IGluZGV4IGZvciBp'
        'bmRleCwgY2hhcmFjdGVyIGluIGVudW1lcmF0ZShjaGVja3BvaW50LmFscGhhYmV0KX0KCiAgICBA'
        'Y2xhc3NtZXRob2QKICAgIGRlZiB0cmFpbihjbHMsIHRleHQ6IHN0ciwgKiwgc21vb3RoaW5nOiBm'
        'bG9hdCA9IDAuMjUsIHNlZWQ6IGludCA9IDcpIC0+IFNlbGY6CiAgICAgICAgaWYgbGVuKHRleHQp'
        'IDwgMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidHJhaW5pbmcgdGV4dCBtdXN0IGNv'
        'bnRhaW4gYXQgbGVhc3QgdHdvIGNoYXJhY3RlcnMiKQogICAgICAgIGlmIG5vdCBtYXRoLmlzZmlu'
        'aXRlKHNtb290aGluZykgb3Igc21vb3RoaW5nIDw9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVl'
        'RXJyb3IoInNtb290aGluZyBtdXN0IGJlIGEgZmluaXRlIHBvc2l0aXZlIHZhbHVlIikKICAgICAg'
        'ICBhbHBoYWJldCA9IHNvcnRlZChzZXQodGV4dCkpCiAgICAgICAgaWYgbGVuKGFscGhhYmV0KSA8'
        'IDI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRyYWluaW5nIHRleHQgbXVzdCBjb250'
        'YWluIGF0IGxlYXN0IHR3byBkaXN0aW5jdCBjaGFyYWN0ZXJzIikKICAgICAgICBpbmRleCA9IHtj'
        'aGFyYWN0ZXI6IHBvc2l0aW9uIGZvciBwb3NpdGlvbiwgY2hhcmFjdGVyIGluIGVudW1lcmF0ZShh'
        'bHBoYWJldCl9CiAgICAgICAgY291bnRzID0gW1swIGZvciBfIGluIGFscGhhYmV0XSBmb3IgXyBp'
        'biBhbHBoYWJldF0KICAgICAgICBmb3IgY3VycmVudCwgZm9sbG93aW5nIGluIHppcCh0ZXh0LCB0'
        'ZXh0WzE6XSwgc3RyaWN0PUZhbHNlKToKICAgICAgICAgICAgY291bnRzW2luZGV4W2N1cnJlbnRd'
        'XVtpbmRleFtmb2xsb3dpbmddXSArPSAxCiAgICAgICAgcmV0dXJuIGNscygKICAgICAgICAgICAg'
        'QmlncmFtQ2hlY2twb2ludCgKICAgICAgICAgICAgICAgIGFscGhhYmV0PWFscGhhYmV0LAogICAg'
        'ICAgICAgICAgICAgY291bnRzPWNvdW50cywKICAgICAgICAgICAgICAgIHNtb290aGluZz1zbW9v'
        'dGhpbmcsCiAgICAgICAgICAgICAgICBzZWVkPXNlZWQsCiAgICAgICAgICAgICAgICB0cmFpbmlu'
        'Z19jaGFyYWN0ZXJzPWxlbih0ZXh0KSwKICAgICAgICAgICAgKQogICAgICAgICkKCiAgICBkZWYg'
        'X3Byb2JhYmlsaXRpZXMoc2VsZiwgY2hhcmFjdGVyOiBzdHIpIC0+IGxpc3RbZmxvYXRdOgogICAg'
        'ICAgIHJvd19pbmRleCA9IHNlbGYuX2luZGV4LmdldChjaGFyYWN0ZXIpCiAgICAgICAgc2l6ZSA9'
        'IGxlbihzZWxmLmNoZWNrcG9pbnQuYWxwaGFiZXQpCiAgICAgICAgaWYgcm93X2luZGV4IGlzIE5v'
        'bmU6CiAgICAgICAgICAgIHJldHVybiBbMS4wIC8gc2l6ZV0gKiBzaXplCiAgICAgICAgcm93ID0g'
        'c2VsZi5jaGVja3BvaW50LmNvdW50c1tyb3dfaW5kZXhdCiAgICAgICAgZGVub21pbmF0b3IgPSBz'
        'dW0ocm93KSArIHNlbGYuY2hlY2twb2ludC5zbW9vdGhpbmcgKiBzaXplCiAgICAgICAgcmV0dXJu'
        'IFsodmFsdWUgKyBzZWxmLmNoZWNrcG9pbnQuc21vb3RoaW5nKSAvIGRlbm9taW5hdG9yIGZvciB2'
        'YWx1ZSBpbiByb3ddCgogICAgZGVmIGV2YWx1YXRlKHNlbGYsIHRleHQ6IHN0cikgLT4gTGFuZ3Vh'
        'Z2VNb2RlbE1ldHJpY3M6CiAgICAgICAgdHJhbnNpdGlvbnMgPSBsaXN0KHppcCh0ZXh0LCB0ZXh0'
        'WzE6XSwgc3RyaWN0PUZhbHNlKSkKICAgICAgICBpZiBub3QgdHJhbnNpdGlvbnM6CiAgICAgICAg'
        'ICAgIHJhaXNlIFZhbHVlRXJyb3IoImV2YWx1YXRpb24gdGV4dCBtdXN0IGNvbnRhaW4gYXQgbGVh'
        'c3QgdHdvIGNoYXJhY3RlcnMiKQogICAgICAgIG5lZ2F0aXZlX2xvZ19saWtlbGlob29kID0gMC4w'
        'CiAgICAgICAgY29ycmVjdCA9IDAKICAgICAgICBmb3IgY3VycmVudCwgZXhwZWN0ZWQgaW4gdHJh'
        'bnNpdGlvbnM6CiAgICAgICAgICAgIHByb2JhYmlsaXRpZXMgPSBzZWxmLl9wcm9iYWJpbGl0aWVz'
        'KGN1cnJlbnQpCiAgICAgICAgICAgIGV4cGVjdGVkX2luZGV4ID0gc2VsZi5faW5kZXguZ2V0KGV4'
        'cGVjdGVkKQogICAgICAgICAgICBwcm9iYWJpbGl0eSA9ICgKICAgICAgICAgICAgICAgIHByb2Jh'
        'YmlsaXRpZXNbZXhwZWN0ZWRfaW5kZXhdCiAgICAgICAgICAgICAgICBpZiBleHBlY3RlZF9pbmRl'
        'eCBpcyBub3QgTm9uZQogICAgICAgICAgICAgICAgZWxzZSAxLjAgLyBsZW4oc2VsZi5jaGVja3Bv'
        'aW50LmFscGhhYmV0KQogICAgICAgICAgICApCiAgICAgICAgICAgIG5lZ2F0aXZlX2xvZ19saWtl'
        'bGlob29kIC09IG1hdGgubG9nKG1heChwcm9iYWJpbGl0eSwgMWUtMTIpKQogICAgICAgICAgICBw'
        'cmVkaWN0ZWQgPSBtYXgocmFuZ2UobGVuKHByb2JhYmlsaXRpZXMpKSwga2V5PXByb2JhYmlsaXRp'
        'ZXMuX19nZXRpdGVtX18pCiAgICAgICAgICAgIGNvcnJlY3QgKz0gaW50KGV4cGVjdGVkX2luZGV4'
        'ID09IHByZWRpY3RlZCkKICAgICAgICBhdmVyYWdlX25sbCA9IG5lZ2F0aXZlX2xvZ19saWtlbGlo'
        'b29kIC8gbGVuKHRyYW5zaXRpb25zKQogICAgICAgIHJldHVybiBMYW5ndWFnZU1vZGVsTWV0cmlj'
        'cygKICAgICAgICAgICAgbmVnYXRpdmVfbG9nX2xpa2VsaWhvb2Q9YXZlcmFnZV9ubGwsCiAgICAg'
        'ICAgICAgIHBlcnBsZXhpdHk9bWF0aC5leHAoYXZlcmFnZV9ubGwpLAogICAgICAgICAgICB0b3Ax'
        'X2FjY3VyYWN5PWNvcnJlY3QgLyBsZW4odHJhbnNpdGlvbnMpLAogICAgICAgICAgICBldmFsdWF0'
        'ZWRfdHJhbnNpdGlvbnM9bGVuKHRyYW5zaXRpb25zKSwKICAgICAgICApCgogICAgZGVmIGdlbmVy'
        'YXRlKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJvbXB0OiBzdHIsCiAgICAgICAgKiwKICAgICAg'
        'ICBtYXhfY2hhcmFjdGVyczogaW50ID0gMTYwLAogICAgICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9'
        'IDAuNywKICAgICAgICBzZWVkOiBpbnQgfCBOb25lID0gTm9uZSwKICAgICkgLT4gc3RyOgogICAg'
        'ICAgIGlmIG1heF9jaGFyYWN0ZXJzIDw9IDAgb3IgbWF4X2NoYXJhY3RlcnMgPiA0XzA5NjoKICAg'
        'ICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibWF4X2NoYXJhY3RlcnMgbXVzdCBiZSBiZXR3ZWVu'
        'IDEgYW5kIDQwOTYiKQogICAgICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKHRlbXBlcmF0dXJlKSBv'
        'ciB0ZW1wZXJhdHVyZSA8IDAgb3IgdGVtcGVyYXR1cmUgPiAyOgogICAgICAgICAgICByYWlzZSBW'
        'YWx1ZUVycm9yKCJ0ZW1wZXJhdHVyZSBtdXN0IGJlIGJldHdlZW4gMCBhbmQgMiIpCiAgICAgICAg'
        'Y3VycmVudCA9IHByb21wdFstMV0gaWYgcHJvbXB0IGVsc2UgIlxuIgogICAgICAgIGlmIGN1cnJl'
        'bnQgbm90IGluIHNlbGYuX2luZGV4OgogICAgICAgICAgICBjdXJyZW50ID0gIlxuIiBpZiAiXG4i'
        'IGluIHNlbGYuX2luZGV4IGVsc2Ugc2VsZi5jaGVja3BvaW50LmFscGhhYmV0WzBdCiAgICAgICAg'
        'cmFuZG9tID0gUmFuZG9tKHNlbGYuY2hlY2twb2ludC5zZWVkIGlmIHNlZWQgaXMgTm9uZSBlbHNl'
        'IHNlZWQpCiAgICAgICAgZ2VuZXJhdGVkOiBsaXN0W3N0cl0gPSBbXQogICAgICAgIGZvciBfIGlu'
        'IHJhbmdlKG1heF9jaGFyYWN0ZXJzKToKICAgICAgICAgICAgcHJvYmFiaWxpdGllcyA9IHNlbGYu'
        'X3Byb2JhYmlsaXRpZXMoY3VycmVudCkKICAgICAgICAgICAgaWYgdGVtcGVyYXR1cmUgPT0gMDoK'
        'ICAgICAgICAgICAgICAgIG5leHRfaW5kZXggPSBtYXgocmFuZ2UobGVuKHByb2JhYmlsaXRpZXMp'
        'KSwga2V5PXByb2JhYmlsaXRpZXMuX19nZXRpdGVtX18pCiAgICAgICAgICAgIGVsc2U6CiAgICAg'
        'ICAgICAgICAgICAjIENvbW1vbiBwb3NpdGl2ZSByZXNjYWxpbmcgbGVhdmVzIHRoZSBjYXRlZ29y'
        'aWNhbCBsYXcgdW5jaGFuZ2VkLgogICAgICAgICAgICAgICAgIyBEaXZpZGUgYmVmb3JlIHRha2lu'
        'ZyBsb2dzIHRvIHJldGFpbiBkaXN0aW5jdCBuZWFyLW1heGltdW0gZmxvYXRzLgogICAgICAgICAg'
        'ICAgICAgIyBBdCBsZWFzdCBvbmUgd2VpZ2h0IGlzIDEgZXZlbiB3aGVuIGV2ZXJ5IHAqKigxL1Qp'
        'IHdvdWxkIHVuZGVyZmxvdy4KICAgICAgICAgICAgICAgIG1heGltdW1fcHJvYmFiaWxpdHkgPSBt'
        'YXgocHJvYmFiaWxpdGllcykKICAgICAgICAgICAgICAgIHdlaWdodHMgPSBbCiAgICAgICAgICAg'
        'ICAgICAgICAgMC4wCiAgICAgICAgICAgICAgICAgICAgaWYgcHJvYmFiaWxpdHkgPT0gMC4wCiAg'
        'ICAgICAgICAgICAgICAgICAgZWxzZSBtYXRoLmV4cChtYXRoLmxvZyhwcm9iYWJpbGl0eSAvIG1h'
        'eGltdW1fcHJvYmFiaWxpdHkpIC8gdGVtcGVyYXR1cmUpCiAgICAgICAgICAgICAgICAgICAgZm9y'
        'IHByb2JhYmlsaXR5IGluIHByb2JhYmlsaXRpZXMKICAgICAgICAgICAgICAgIF0KICAgICAgICAg'
        'ICAgICAgIG5leHRfaW5kZXggPSByYW5kb20uY2hvaWNlcyhyYW5nZShsZW4od2VpZ2h0cykpLCB3'
        'ZWlnaHRzPXdlaWdodHMsIGs9MSlbMF0KICAgICAgICAgICAgY3VycmVudCA9IHNlbGYuY2hlY2tw'
        'b2ludC5hbHBoYWJldFtuZXh0X2luZGV4XQogICAgICAgICAgICBnZW5lcmF0ZWQuYXBwZW5kKGN1'
        'cnJlbnQpCiAgICAgICAgcmV0dXJuICIiLmpvaW4oZ2VuZXJhdGVkKQoKCmRlZiB1bmlmb3JtX2Jh'
        'c2VsaW5lX21ldHJpY3ModGV4dDogc3RyLCBhbHBoYWJldF9zaXplOiBpbnQpIC0+IExhbmd1YWdl'
        'TW9kZWxNZXRyaWNzOgogICAgaWYgbGVuKHRleHQpIDwgMjoKICAgICAgICByYWlzZSBWYWx1ZUVy'
        'cm9yKCJldmFsdWF0aW9uIHRleHQgbXVzdCBjb250YWluIGF0IGxlYXN0IHR3byBjaGFyYWN0ZXJz'
        'IikKICAgIGlmIGFscGhhYmV0X3NpemUgPCAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImFs'
        'cGhhYmV0X3NpemUgbXVzdCBiZSBhdCBsZWFzdCB0d28iKQogICAgdHJhbnNpdGlvbnMgPSBsZW4o'
        'dGV4dCkgLSAxCiAgICBubGwgPSBtYXRoLmxvZyhhbHBoYWJldF9zaXplKQogICAgcmV0dXJuIExh'
        'bmd1YWdlTW9kZWxNZXRyaWNzKAogICAgICAgIG5lZ2F0aXZlX2xvZ19saWtlbGlob29kPW5sbCwK'
        'ICAgICAgICBwZXJwbGV4aXR5PWZsb2F0KGFscGhhYmV0X3NpemUpLAogICAgICAgIHRvcDFfYWNj'
        'dXJhY3k9MS4wIC8gYWxwaGFiZXRfc2l6ZSwKICAgICAgICBldmFsdWF0ZWRfdHJhbnNpdGlvbnM9'
        'dHJhbnNpdGlvbnMsCiAgICApCg=='
    ),
    'olympus/foundry/data_pipeline.py': (
        '...'
    ),
    'olympus/foundry/eval_suite.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0'
        'IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmltcG9ydCBzdGF0aXN0aWNzCmltcG9ydCB0aW1l'
        'CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGltcG9ydCBOYW1lZFRlbXBv'
        'cmFyeUZpbGUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKaW1wb3J0IHRvcmNoCmZyb20gcHlkYW50'
        'aWMgaW1wb3J0IEZpZWxkCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RN'
        'b2RlbApmcm9tIG9seW1wdXMuZm91bmRyeS5kYXRhX3BpcGVsaW5lIGltcG9ydCBJbnN0cnVjdGlv'
        'bkV4YW1wbGUsIGxvYWRfdmVyaWZpZWRfZGF0YXNldApmcm9tIG9seW1wdXMuZm91bmRyeS5yZXNv'
        'dXJjZXMgaW1wb3J0IG1lbW9yeV9zbmFwc2hvdApmcm9tIG9seW1wdXMuZm91bmRyeS5zZnQgaW1w'
        'b3J0ICgKICAgIExvc3NOb3JtYWxpemF0aW9uLAogICAgVGlueUNhdXNhbExNLAogICAgVGlueU1v'
        'ZGVsQ29uZmlnLAogICAgX2NoZWNrcG9pbnRfdHJhaW5pbmdfY29uZmlnLAogICAgX2VuY29kZV9l'
        'eGFtcGxlcywKICAgIF9tb2RlbF9mcm9tX2NoZWNrcG9pbnQsCiAgICBldmFsdWF0ZV9sb3NzLAog'
        'ICAgZ2VuZXJhdGVfdGV4dCwKKQoKCmRlZiBfc2hhMjU2KHBhdGg6IFBhdGgpIC0+IHN0cjoKICAg'
        'IHJldHVybiBoYXNobGliLnNoYTI1NihwYXRoLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKCgpk'
        'ZWYgX2F0b21pY19qc29uKHBhdGg6IFBhdGgsIHBheWxvYWQ6IG9iamVjdCkgLT4gTm9uZToKICAg'
        'IHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHNlcmlh'
        'bGl6ZWQgPSAoanNvbi5kdW1wcyhwYXlsb2FkLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsg'
        'IlxuIikuZW5jb2RlKCkKICAgIHdpdGggTmFtZWRUZW1wb3JhcnlGaWxlKGRpcj1wYXRoLnBhcmVu'
        'dCwgcHJlZml4PWYiLntwYXRoLm5hbWV9LiIsIGRlbGV0ZT1GYWxzZSkgYXMgaGFuZGxlOgogICAg'
        'ICAgIGhhbmRsZS53cml0ZShzZXJpYWxpemVkKQogICAgICAgIGhhbmRsZS5mbHVzaCgpCiAgICAg'
        'ICAgb3MuZnN5bmMoaGFuZGxlLmZpbGVubygpKQogICAgICAgIHRlbXBvcmFyeSA9IFBhdGgoaGFu'
        'ZGxlLm5hbWUpCiAgICB0ZW1wb3JhcnkucmVwbGFjZShwYXRoKQoKCmNsYXNzIENhdGVnb3J5RXZh'
        'bHVhdGlvbihTdHJpY3RNb2RlbCk6CiAgICBjYXRlZ29yeTogc3RyCiAgICByZWNvcmRzOiBpbnQg'
        'PSBGaWVsZChndD0wKQogICAgYmFzZWxpbmVfbG9zczogZmxvYXQgPSBGaWVsZChnZT0wLjApCiAg'
        'ICBjYW5kaWRhdGVfbG9zczogZmxvYXQgPSBGaWVsZChnZT0wLjApCiAgICBsb3NzX2NoYW5nZV9m'
        'cmFjdGlvbjogZmxvYXQKICAgIHJlZ3Jlc3NlZDogYm9vbAoKCmNsYXNzIFdvcmtmbG93U2NvcmUo'
        'U3RyaWN0TW9kZWwpOgogICAgbmFtZTogc3RyCiAgICBjYXRlZ29yaWVzOiBsaXN0W3N0cl0KICAg'
        'IGV4YW1wbGVzOiBpbnQgPSBGaWVsZChndD0wKQogICAgZXhhY3RfbWF0Y2hfcmF0ZTogZmxvYXQg'
        'PSBGaWVsZChnZT0wLjAsIGxlPTEuMCkKICAgIG5vbmVtcHR5X3JhdGU6IGZsb2F0ID0gRmllbGQo'
        'Z2U9MC4wLCBsZT0xLjApCgoKY2xhc3MgSGVsZE91dEV2YWx1YXRpb24oU3RyaWN0TW9kZWwpOgog'
        'ICAgc2NoZW1hX3ZlcnNpb246IGludCA9IDEKICAgIHN1aXRlOiBzdHIgPSAib2x5bXB1cy1oZWxk'
        'LW91dC1jYXBhYmlsaXRpZXMtdjEiCiAgICBjaGVja3BvaW50X3BhdGg6IHN0cgogICAgY2hlY2tw'
        'b2ludF9zaGEyNTY6IHN0cgogICAgZGF0YXNldF9tYW5pZmVzdF9zaGEyNTY6IHN0cgogICAgYmFz'
        'ZWxpbmVfaWRlbnRpdHk6IHN0cgogICAgdGVzdF9yZWNvcmRzOiBpbnQgPSBGaWVsZChndD0wKQog'
        'ICAgb3ZlcmFsbF9iYXNlbGluZV9sb3NzOiBmbG9hdAogICAgb3ZlcmFsbF9jYW5kaWRhdGVfbG9z'
        'czogZmxvYXQKICAgIG92ZXJhbGxfYmFzZWxpbmVfcGVycGxleGl0eTogZmxvYXQKICAgIG92ZXJh'
        'bGxfY2FuZGlkYXRlX3BlcnBsZXhpdHk6IGZsb2F0CiAgICBjYXRlZ29yeV9yZXN1bHRzOiBsaXN0'
        'W0NhdGVnb3J5RXZhbHVhdGlvbl0KICAgIHJlZ3Jlc3Npb25fY291bnQ6IGludCA9IEZpZWxkKGdl'
        'PTApCiAgICBleGFjdF9tYXRjaF9yYXRlOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQog'
        'ICAgZm9ybWF0X2NvbXBsaWFuY2VfcmF0ZTogZmxvYXQgPSBGaWVsZChnZT0wLjAsIGxlPTEuMCkK'
        'ICAgIHdvcmtmbG93X3Njb3JlczogbGlzdFtXb3JrZmxvd1Njb3JlXQogICAgbWVkaWFuX2dlbmVy'
        'YXRpb25fbGF0ZW5jeV9tczogZmxvYXQgPSBGaWVsZChnZT0wLjApCiAgICBnZW5lcmF0ZWRfYnl0'
        'ZXNfcGVyX3NlY29uZDogZmxvYXQgPSBGaWVsZChnZT0wLjApCiAgICBwZWFrX3Jzc19ieXRlczog'
        'aW50ID0gRmllbGQoZ2U9MCkKICAgIHBhc3NlZF9zbW9rZV9xdWFsaXR5X2dhdGU6IGJvb2wKICAg'
        'IGRlY2lzaW9uOiBzdHIKICAgIGxvc3Nfbm9ybWFsaXphdGlvbjogTG9zc05vcm1hbGl6YXRpb24g'
        'PSAibGVnYWN5X2JhdGNoX21lYW5fdjEiCgoKZGVmIF9sb2FkX3Rlc3RfZXhhbXBsZXMobWFuaWZl'
        'c3RfcGF0aDogUGF0aCkgLT4gdHVwbGVbc3RyLCBsaXN0W0luc3RydWN0aW9uRXhhbXBsZV1dOgog'
        'ICAgbWFuaWZlc3QsIGV4YW1wbGVzX2J5X3NwbGl0ID0gbG9hZF92ZXJpZmllZF9kYXRhc2V0KG1h'
        'bmlmZXN0X3BhdGgpCiAgICBpZiBtYW5pZmVzdC5tYW5pZmVzdF9zaGEyNTYgaXMgTm9uZToKICAg'
        'ICAgICByYWlzZSBWYWx1ZUVycm9yKCJkYXRhc2V0IG1hbmlmZXN0IGhhcyBubyBoYXNoIikKICAg'
        'IHJldHVybiBtYW5pZmVzdC5tYW5pZmVzdF9zaGEyNTYsIGV4YW1wbGVzX2J5X3NwbGl0WyJ0ZXN0'
        'Il0KCgpkZWYgX2Jhc2VsaW5lX21vZGVsKAogICAgY2hlY2twb2ludDogZGljdFtzdHIsIEFueV0s'
        'IGJhc2VfY2hlY2twb2ludDogUGF0aCB8IE5vbmUKKSAtPiB0dXBsZVtUaW55Q2F1c2FsTE0sIHN0'
        'cl06CiAgICBpZiBiYXNlX2NoZWNrcG9pbnQgaXMgbm90IE5vbmU6CiAgICAgICAgbW9kZWwsIF8g'
        'PSBfbW9kZWxfZnJvbV9jaGVja3BvaW50KGJhc2VfY2hlY2twb2ludCwgdG9yY2guZGV2aWNlKCJj'
        'cHUiKSkKICAgICAgICByZXR1cm4gbW9kZWwsIGYiY2hlY2twb2ludDp7X3NoYTI1NihiYXNlX2No'
        'ZWNrcG9pbnQpfSIKICAgIHRyYWluaW5nID0gX2NoZWNrcG9pbnRfdHJhaW5pbmdfY29uZmlnKGNo'
        'ZWNrcG9pbnQpCiAgICB0b3JjaC5tYW51YWxfc2VlZCh0cmFpbmluZy5zZWVkKQogICAgcmV0dXJu'
        'IFRpbnlDYXVzYWxMTShUaW55TW9kZWxDb25maWcubW9kZWxfdmFsaWRhdGUoY2hlY2twb2ludFsi'
        'bW9kZWxfY29uZmlnIl0pKSwgKAogICAgICAgIGYiZGV0ZXJtaW5pc3RpYy11bnRyYWluZWQtc2Vl'
        'ZC17dHJhaW5pbmcuc2VlZH0iCiAgICApCgoKZGVmIF9ub3JtYWxpemUodGV4dDogc3RyKSAtPiBz'
        'dHI6CiAgICByZXR1cm4gIiAiLmpvaW4odGV4dC5jYXNlZm9sZCgpLnN0cmlwKCkuc3BsaXQoKSkK'
        'CgpkZWYgX2Zvcm1hdF9jb21wbGlhbnQoZXhwZWN0ZWQ6IHN0ciwgZ2VuZXJhdGVkOiBzdHIpIC0+'
        'IGJvb2w6CiAgICBpZiBleHBlY3RlZC5sc3RyaXAoKS5zdGFydHN3aXRoKCJ7Iik6CiAgICAgICAg'
        'dHJ5OgogICAgICAgICAgICBwYXJzZWQgPSBqc29uLmxvYWRzKGdlbmVyYXRlZCkKICAgICAgICBl'
        'eGNlcHQganNvbi5KU09ORGVjb2RlRXJyb3I6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAg'
        'ICAgIHJldHVybiBpc2luc3RhbmNlKHBhcnNlZCwgZGljdCkKICAgIHJldHVybiBib29sKGdlbmVy'
        'YXRlZC5zdHJpcCgpKSBhbmQgIlx1ZmZmZCIgbm90IGluIGdlbmVyYXRlZAoKCmRlZiBfd29ya2Zs'
        'b3dfc2NvcmVzKAogICAgZXhhbXBsZXM6IGxpc3RbSW5zdHJ1Y3Rpb25FeGFtcGxlXSwgZ2VuZXJh'
        'dGVkOiBkaWN0W3N0ciwgc3RyXQopIC0+IGxpc3RbV29ya2Zsb3dTY29yZV06CiAgICBkZWZpbml0'
        'aW9ucyA9IHsKICAgICAgICAidG9vbF93b3JrZmxvdyI6IFsidG9vbF91c2UiLCAiYWdlbnRfYmVo'
        'YXZpb3IiXSwKICAgICAgICAicmVzdWx0X2NvbXBhcmlzb24iOiBbInJlYXNvbmluZyIsICJzZWxm'
        'X2NvcnJlY3Rpb24iLCAicmVzZWFyY2giXSwKICAgICAgICAibXVsdGlfc3RlcF9wbGFubmluZyI6'
        'IFsicGxhbm5pbmciLCAic2FmZXR5Il0sCiAgICB9CiAgICByZXN1bHRzOiBsaXN0W1dvcmtmbG93'
        'U2NvcmVdID0gW10KICAgIGZvciBuYW1lLCBjYXRlZ29yaWVzIGluIGRlZmluaXRpb25zLml0ZW1z'
        'KCk6CiAgICAgICAgc2VsZWN0ZWQgPSBbZXhhbXBsZSBmb3IgZXhhbXBsZSBpbiBleGFtcGxlcyBp'
        'ZiBleGFtcGxlLmNhdGVnb3J5IGluIGNhdGVnb3JpZXNdCiAgICAgICAgZXhhY3QgPSBzdW0oCiAg'
        'ICAgICAgICAgIF9ub3JtYWxpemUoZ2VuZXJhdGVkW2l0ZW0uaWRdKSA9PSBfbm9ybWFsaXplKGl0'
        'ZW0ucmVzcG9uc2UpIGZvciBpdGVtIGluIHNlbGVjdGVkCiAgICAgICAgKQogICAgICAgIG5vbmVt'
        'cHR5ID0gc3VtKGJvb2woZ2VuZXJhdGVkW2l0ZW0uaWRdLnN0cmlwKCkpIGZvciBpdGVtIGluIHNl'
        'bGVjdGVkKQogICAgICAgIHJlc3VsdHMuYXBwZW5kKAogICAgICAgICAgICBXb3JrZmxvd1Njb3Jl'
        'KAogICAgICAgICAgICAgICAgbmFtZT1uYW1lLAogICAgICAgICAgICAgICAgY2F0ZWdvcmllcz1j'
        'YXRlZ29yaWVzLAogICAgICAgICAgICAgICAgZXhhbXBsZXM9bGVuKHNlbGVjdGVkKSwKICAgICAg'
        'ICAgICAgICAgIGV4YWN0X21hdGNoX3JhdGU9ZXhhY3QgLyBsZW4oc2VsZWN0ZWQpLAogICAgICAg'
        'ICAgICAgICAgbm9uZW1wdHlfcmF0ZT1ub25lbXB0eSAvIGxlbihzZWxlY3RlZCksCiAgICAgICAg'
        'ICAgICkKICAgICAgICApCiAgICByZXR1cm4gcmVzdWx0cwoKCmRlZiBldmFsdWF0ZV9jaGVja3Bv'
        'aW50KAogICAgY2hlY2twb2ludF9wYXRoOiBQYXRoLAogICAgbWFuaWZlc3RfcGF0aDogUGF0aCwK'
        'ICAgIG91dHB1dF9wYXRoOiBQYXRoLAogICAgKiwKICAgIGJhc2VfY2hlY2twb2ludDogUGF0aCB8'
        'IE5vbmUgPSBOb25lLAogICAgbWF4X2dlbmVyYXRpb25fdG9rZW5zOiBpbnQgPSA0OCwKKSAtPiBI'
        'ZWxkT3V0RXZhbHVhdGlvbjoKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3B1IikKICAgIGNh'
        'bmRpZGF0ZSwgY2hlY2twb2ludCA9IF9tb2RlbF9mcm9tX2NoZWNrcG9pbnQoY2hlY2twb2ludF9w'
        'YXRoLCBkZXZpY2UpCiAgICBiYXNlbGluZSwgYmFzZWxpbmVfaWRlbnRpdHkgPSBfYmFzZWxpbmVf'
        'bW9kZWwoY2hlY2twb2ludCwgYmFzZV9jaGVja3BvaW50KQogICAgbWFuaWZlc3Rfc2hhLCBleGFt'
        'cGxlcyA9IF9sb2FkX3Rlc3RfZXhhbXBsZXMobWFuaWZlc3RfcGF0aCkKICAgIGlmIGNoZWNrcG9p'
        'bnRbImRhdGFzZXRfbWFuaWZlc3Rfc2hhMjU2Il0gIT0gbWFuaWZlc3Rfc2hhOgogICAgICAgIHJh'
        'aXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgYW5kIGV2YWx1YXRpb24gZGF0YXNldCBoYXNoZXMg'
        'ZG8gbm90IG1hdGNoIikKICAgIHRyYWluaW5nID0gX2NoZWNrcG9pbnRfdHJhaW5pbmdfY29uZmln'
        'KGNoZWNrcG9pbnQpCiAgICBldmFsdWF0aW9uX2NvbmZpZyA9IHRyYWluaW5nLm1vZGVsX2NvcHko'
        'dXBkYXRlPXsicGFja19zZXF1ZW5jZXMiOiBGYWxzZX0pCiAgICBhbGxfcm93cyA9IF9lbmNvZGVf'
        'ZXhhbXBsZXMoZXhhbXBsZXMsIGV2YWx1YXRpb25fY29uZmlnKQogICAgYmFzZWxpbmVfbG9zcyA9'
        'IGV2YWx1YXRlX2xvc3MoCiAgICAgICAgYmFzZWxpbmUsIGFsbF9yb3dzLCBkZXZpY2U9ZGV2aWNl'
        'LCBub3JtYWxpemF0aW9uPXRyYWluaW5nLmxvc3Nfbm9ybWFsaXphdGlvbgogICAgKQogICAgY2Fu'
        'ZGlkYXRlX2xvc3MgPSBldmFsdWF0ZV9sb3NzKAogICAgICAgIGNhbmRpZGF0ZSwgYWxsX3Jvd3Ms'
        'IGRldmljZT1kZXZpY2UsIG5vcm1hbGl6YXRpb249dHJhaW5pbmcubG9zc19ub3JtYWxpemF0aW9u'
        'CiAgICApCgogICAgY2F0ZWdvcnlfcmVzdWx0czogbGlzdFtDYXRlZ29yeUV2YWx1YXRpb25dID0g'
        'W10KICAgIGZvciBjYXRlZ29yeSBpbiBzb3J0ZWQoe2V4YW1wbGUuY2F0ZWdvcnkgZm9yIGV4YW1w'
        'bGUgaW4gZXhhbXBsZXN9KToKICAgICAgICBzZWxlY3RlZCA9IFtleGFtcGxlIGZvciBleGFtcGxl'
        'IGluIGV4YW1wbGVzIGlmIGV4YW1wbGUuY2F0ZWdvcnkgPT0gY2F0ZWdvcnldCiAgICAgICAgcm93'
        'cyA9IF9lbmNvZGVfZXhhbXBsZXMoc2VsZWN0ZWQsIGV2YWx1YXRpb25fY29uZmlnKQogICAgICAg'
        'IGJlZm9yZSA9IGV2YWx1YXRlX2xvc3MoCiAgICAgICAgICAgIGJhc2VsaW5lLCByb3dzLCBkZXZp'
        'Y2U9ZGV2aWNlLCBub3JtYWxpemF0aW9uPXRyYWluaW5nLmxvc3Nfbm9ybWFsaXphdGlvbgogICAg'
        'ICAgICkKICAgICAgICBhZnRlciA9IGV2YWx1YXRlX2xvc3MoCiAgICAgICAgICAgIGNhbmRpZGF0'
        'ZSwgcm93cywgZGV2aWNlPWRldmljZSwgbm9ybWFsaXphdGlvbj10cmFpbmluZy5sb3NzX25vcm1h'
        'bGl6YXRpb24KICAgICAgICApCiAgICAgICAgY2hhbmdlID0gKGFmdGVyIC0gYmVmb3JlKSAvIG1h'
        'eChiZWZvcmUsIDFlLTEyKQogICAgICAgIGNhdGVnb3J5X3Jlc3VsdHMuYXBwZW5kKAogICAgICAg'
        'ICAgICBDYXRlZ29yeUV2YWx1YXRpb24oCiAgICAgICAgICAgICAgICBjYXRlZ29yeT1jYXRlZ29y'
        'eSwKICAgICAgICAgICAgICAgIHJlY29yZHM9bGVuKHNlbGVjdGVkKSwKICAgICAgICAgICAgICAg'
        'IGJhc2VsaW5lX2xvc3M9YmVmb3JlLAogICAgICAgICAgICAgICAgY2FuZGlkYXRlX2xvc3M9YWZ0'
        'ZXIsCiAgICAgICAgICAgICAgICBsb3NzX2NoYW5nZV9mcmFjdGlvbj1jaGFuZ2UsCiAgICAgICAg'
        'ICAgICAgICByZWdyZXNzZWQ9Y2hhbmdlID4gMC4wNSwKICAgICAgICAgICAgKQogICAgICAgICkK'
        'CiAgICBnZW5lcmF0ZWQ6IGRpY3Rbc3RyLCBzdHJdID0ge30KICAgIGxhdGVuY2llczogbGlzdFtm'
        'bG9hdF0gPSBbXQogICAgZ2VuZXJhdGVkX2J5dGVzID0gMAogICAgZm9yIGV4YW1wbGUgaW4gZXhh'
        'bXBsZXM6CiAgICAgICAgc3RhcnQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgYW5zd2Vy'
        'ID0gZ2VuZXJhdGVfdGV4dCgKICAgICAgICAgICAgY2FuZGlkYXRlLAogICAgICAgICAgICBleGFt'
        'cGxlLnByb21wdCwKICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICAgICAgbWF4X25l'
        'd190b2tlbnM9bWF4X2dlbmVyYXRpb25fdG9rZW5zLAogICAgICAgICkKICAgICAgICBsYXRlbmNp'
        'ZXMuYXBwZW5kKHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydCkKICAgICAgICBnZW5lcmF0ZWRf'
        'Ynl0ZXMgKz0gbGVuKGFuc3dlci5lbmNvZGUoKSkKICAgICAgICBnZW5lcmF0ZWRbZXhhbXBsZS5p'
        'ZF0gPSBhbnN3ZXIKICAgIGV4YWN0ID0gc3VtKF9ub3JtYWxpemUoZ2VuZXJhdGVkW2l0ZW0uaWRd'
        'KSA9PSBfbm9ybWFsaXplKGl0ZW0ucmVzcG9uc2UpIGZvciBpdGVtIGluIGV4YW1wbGVzKQogICAg'
        'Y29tcGxpYW50ID0gc3VtKF9mb3JtYXRfY29tcGxpYW50KGl0ZW0ucmVzcG9uc2UsIGdlbmVyYXRl'
        'ZFtpdGVtLmlkXSkgZm9yIGl0ZW0gaW4gZXhhbXBsZXMpCiAgICByZWdyZXNzaW9ucyA9IHN1bShy'
        'ZXN1bHQucmVncmVzc2VkIGZvciByZXN1bHQgaW4gY2F0ZWdvcnlfcmVzdWx0cykKICAgIGV4YWN0'
        'X3JhdGUgPSBleGFjdCAvIGxlbihleGFtcGxlcykKICAgIGZvcm1hdF9yYXRlID0gY29tcGxpYW50'
        'IC8gbGVuKGV4YW1wbGVzKQogICAgcGFzc2VkID0gY2FuZGlkYXRlX2xvc3MgPCBiYXNlbGluZV9s'
        'b3NzIGFuZCByZWdyZXNzaW9ucyA9PSAwIGFuZCBmb3JtYXRfcmF0ZSA+PSAwLjgKICAgIGRlY2lz'
        'aW9uID0gKAogICAgICAgICJQYXNzZWQgdGhlIGluZnJhc3RydWN0dXJlIHNtb2tlIGdhdGU7IHRo'
        'aXMgZG9lcyBub3Qgc2F0aXNmeSBIZXJtZXMgcHJvbW90aW9uIGdhdGVzLiIKICAgICAgICBpZiBw'
        'YXNzZWQKICAgICAgICBlbHNlICJGYWlsZWQgdGhlIGluZnJhc3RydWN0dXJlIHNtb2tlIGdhdGUg'
        'YW5kIGlzIGluZWxpZ2libGUgZm9yIG1vZGVsIHByb21vdGlvbi4iCiAgICApCiAgICByZXBvcnQg'
        'PSBIZWxkT3V0RXZhbHVhdGlvbigKICAgICAgICBsb3NzX25vcm1hbGl6YXRpb249dHJhaW5pbmcu'
        'bG9zc19ub3JtYWxpemF0aW9uLAogICAgICAgIGNoZWNrcG9pbnRfcGF0aD1zdHIoY2hlY2twb2lu'
        'dF9wYXRoLnJlc29sdmUoKSksCiAgICAgICAgY2hlY2twb2ludF9zaGEyNTY9X3NoYTI1NihjaGVj'
        'a3BvaW50X3BhdGgpLAogICAgICAgIGRhdGFzZXRfbWFuaWZlc3Rfc2hhMjU2PW1hbmlmZXN0X3No'
        'YSwKICAgICAgICBiYXNlbGluZV9pZGVudGl0eT1iYXNlbGluZV9pZGVudGl0eSwKICAgICAgICB0'
        'ZXN0X3JlY29yZHM9bGVuKGV4YW1wbGVzKSwKICAgICAgICBvdmVyYWxsX2Jhc2VsaW5lX2xvc3M9'
        'YmFzZWxpbmVfbG9zcywKICAgICAgICBvdmVyYWxsX2NhbmRpZGF0ZV9sb3NzPWNhbmRpZGF0ZV9s'
        'b3NzLAogICAgICAgIG92ZXJhbGxfYmFzZWxpbmVfcGVycGxleGl0eT1tYXRoLmV4cChtaW4oYmFz'
        'ZWxpbmVfbG9zcywgMjAuMCkpLAogICAgICAgIG92ZXJhbGxfY2FuZGlkYXRlX3BlcnBsZXhpdHk9'
        'bWF0aC5leHAobWluKGNhbmRpZGF0ZV9sb3NzLCAyMC4wKSksCiAgICAgICAgY2F0ZWdvcnlfcmVz'
        'dWx0cz1jYXRlZ29yeV9yZXN1bHRzLAogICAgICAgIHJlZ3Jlc3Npb25fY291bnQ9cmVncmVzc2lv'
        'bnMsCiAgICAgICAgZXhhY3RfbWF0Y2hfcmF0ZT1leGFjdF9yYXRlLAogICAgICAgIGZvcm1hdF9j'
        'b21wbGlhbmNlX3JhdGU9Zm9ybWF0X3JhdGUsCiAgICAgICAgd29ya2Zsb3dfc2NvcmVzPV93b3Jr'
        'Zmxvd19zY29yZXMoZXhhbXBsZXMsIGdlbmVyYXRlZCksCiAgICAgICAgbWVkaWFuX2dlbmVyYXRp'
        'b25fbGF0ZW5jeV9tcz1zdGF0aXN0aWNzLm1lZGlhbihsYXRlbmNpZXMpICogMV8wMDAsCiAgICAg'
        'ICAgZ2VuZXJhdGVkX2J5dGVzX3Blcl9zZWNvbmQ9KAogICAgICAgICAgICBnZW5lcmF0ZWRfYnl0'
        'ZXMgLyBzdW0obGF0ZW5jaWVzKSBpZiBzdW0obGF0ZW5jaWVzKSBlbHNlIDAuMAogICAgICAgICks'
        'CiAgICAgICAgcGVha19yc3NfYnl0ZXM9bWVtb3J5X3NuYXBzaG90KCkucHJvY2Vzc19wZWFrX3Jz'
        'c19ieXRlcywKICAgICAgICBwYXNzZWRfc21va2VfcXVhbGl0eV9nYXRlPXBhc3NlZCwKICAgICAg'
        'ICBkZWNpc2lvbj1kZWNpc2lvbiwKICAgICkKICAgIF9hdG9taWNfanNvbihvdXRwdXRfcGF0aCwg'
        'cmVwb3J0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpKQogICAgcmV0dXJuIHJlcG9ydAo='
    ),
    'olympus/foundry/load_probe.py': (
        'IiIiQm91bmRlZCBBU0dJIGxvYWQgcHJvYmUgZm9yIGVuZ2luZWVyaW5nIHF1YWxpZmljYXRpb24u'
        'CgpUaGUgcHJvYmUgaXMgaW50ZW50aW9uYWxseSBsYWJlbGxlZCBhcyBhIHNtb2tlIHRlc3QuICBJ'
        'dHMgb3V0cHV0IGNhbm5vdCBzYXRpc2Z5CnRoZSAyNC1ob3VyIHByb2R1Y3Rpb24gZ2F0ZSBpbiBg'
        'YHJlYWRpbmVzcy5weWBgLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMK'
        'CmltcG9ydCBhc3luY2lvCmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmltcG9ydCBvcwppbXBvcnQg'
        'c3RhdGlzdGljcwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0ZW1w'
        'ZmlsZSBpbXBvcnQgTmFtZWRUZW1wb3JhcnlGaWxlCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmlt'
        'cG9ydCBodHRweApmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZAoKZnJvbSBvbHltcHVzLmNvcmUu'
        'c2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKCgpjbGFzcyBMb2FkU21va2VSZXBvcnQoU3RyaWN0'
        'TW9kZWwpOgogICAgc2NoZW1hX3ZlcnNpb246IGludCA9IDEKICAgIGVuZHBvaW50OiBzdHIKICAg'
        'IHJlcXVlc3RzOiBpbnQgPSBGaWVsZChndD0wKQogICAgY29uY3VycmVuY3k6IGludCA9IEZpZWxk'
        'KGd0PTApCiAgICBzdWNjZXNzZnVsX3JlcXVlc3RzOiBpbnQgPSBGaWVsZChnZT0wKQogICAgZmFp'
        'bGVkX3JlcXVlc3RzOiBpbnQgPSBGaWVsZChnZT0wKQogICAgZXJyb3JfcmF0ZTogZmxvYXQgPSBG'
        'aWVsZChnZT0wLCBsZT0xKQogICAgZWxhcHNlZF9zZWNvbmRzOiBmbG9hdCA9IEZpZWxkKGd0PTAp'
        'CiAgICB0aHJvdWdocHV0X3JlcXVlc3RzX3Blcl9zZWNvbmQ6IGZsb2F0ID0gRmllbGQoZ2U9MCkK'
        'ICAgIHA1MF9sYXRlbmN5X21zOiBmbG9hdCA9IEZpZWxkKGdlPTApCiAgICBwOTVfbGF0ZW5jeV9t'
        'czogZmxvYXQgPSBGaWVsZChnZT0wKQogICAgcDk5X2xhdGVuY3lfbXM6IGZsb2F0ID0gRmllbGQo'
        'Z2U9MCkKICAgIHN0YXR1c19jb3VudHM6IGRpY3Rbc3RyLCBpbnRdCiAgICBxdWFsaWZpY2F0aW9u'
        'OiBzdHIgPSAiQk9VTkRFRF9FTkdJTkVFUklOR19TTU9LRV9OT1RfUFJPRFVDVElPTl9FVklERU5D'
        'RSIKCgpkZWYgX3BlcmNlbnRpbGUodmFsdWVzOiBsaXN0W2Zsb2F0XSwgcXVhbnRpbGU6IGZsb2F0'
        'KSAtPiBmbG9hdDoKICAgIGlmIG5vdCB2YWx1ZXM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi'
        'cGVyY2VudGlsZSByZXF1aXJlcyBhdCBsZWFzdCBvbmUgb2JzZXJ2YXRpb24iKQogICAgaWYgbm90'
        'IDAgPD0gcXVhbnRpbGUgPD0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJxdWFudGlsZSBt'
        'dXN0IGJlIGluIFswLCAxXSIpCiAgICBvcmRlcmVkID0gc29ydGVkKHZhbHVlcykKICAgIGluZGV4'
        'ID0gbWluKGxlbihvcmRlcmVkKSAtIDEsIG1heCgwLCBtYXRoLmNlaWwobGVuKG9yZGVyZWQpICog'
        'cXVhbnRpbGUpIC0gMSkpCiAgICByZXR1cm4gb3JkZXJlZFtpbmRleF0KCgpkZWYgX2F0b21pY19q'
        'c29uKHBhdGg6IFBhdGgsIHBheWxvYWQ6IG9iamVjdCkgLT4gTm9uZToKICAgIHBhdGgucGFyZW50'
        'Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHNlcmlhbGl6ZWQgPSAoanNv'
        'bi5kdW1wcyhwYXlsb2FkLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxuIikuZW5jb2Rl'
        'KCkKICAgIHdpdGggTmFtZWRUZW1wb3JhcnlGaWxlKGRpcj1wYXRoLnBhcmVudCwgcHJlZml4PWYi'
        'LntwYXRoLm5hbWV9LiIsIGRlbGV0ZT1GYWxzZSkgYXMgaGFuZGxlOgogICAgICAgIGhhbmRsZS53'
        'cml0ZShzZXJpYWxpemVkKQogICAgICAgIGhhbmRsZS5mbHVzaCgpCiAgICAgICAgb3MuZnN5bmMo'
        'aGFuZGxlLmZpbGVubygpKQogICAgICAgIHRlbXBvcmFyeSA9IFBhdGgoaGFuZGxlLm5hbWUpCiAg'
        'ICB0ZW1wb3JhcnkucmVwbGFjZShwYXRoKQoKCmFzeW5jIGRlZiBwcm9iZV9hc2dpX2FwcCgKICAg'
        'IGFwcDogQW55LAogICAgKiwKICAgIGVuZHBvaW50OiBzdHIgPSAiL2hlYWx0aCIsCiAgICByZXF1'
        'ZXN0czogaW50ID0gMjAwLAogICAgY29uY3VycmVuY3k6IGludCA9IDE2LAopIC0+IExvYWRTbW9r'
        'ZVJlcG9ydDoKICAgIGlmIHJlcXVlc3RzIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJy'
        'ZXF1ZXN0cyBtdXN0IGJlIHBvc2l0aXZlIikKICAgIGlmIGNvbmN1cnJlbmN5IDwgMSBvciBjb25j'
        'dXJyZW5jeSA+IHJlcXVlc3RzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNvbmN1cnJlbmN5'
        'IG11c3QgYmUgaW4gWzEsIHJlcXVlc3RzXSIpCiAgICBzZW1hcGhvcmUgPSBhc3luY2lvLlNlbWFw'
        'aG9yZShjb25jdXJyZW5jeSkKICAgIGxhdGVuY2llczogbGlzdFtmbG9hdF0gPSBbXQogICAgc3Rh'
        'dHVzZXM6IGRpY3Rbc3RyLCBpbnRdID0ge30KCiAgICBhc3luYyB3aXRoIGh0dHB4LkFzeW5jQ2xp'
        'ZW50KAogICAgICAgIHRyYW5zcG9ydD1odHRweC5BU0dJVHJhbnNwb3J0KGFwcD1hcHApLCBiYXNl'
        'X3VybD0iaHR0cDovL29seW1wdXMudGVzdCIKICAgICkgYXMgY2xpZW50OgoKICAgICAgICBhc3lu'
        'YyBkZWYgb25lKCkgLT4gTm9uZToKICAgICAgICAgICAgYXN5bmMgd2l0aCBzZW1hcGhvcmU6CiAg'
        'ICAgICAgICAgICAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgICAgICAg'
        'ICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHJlc3BvbnNlID0gYXdhaXQgY2xpZW50LmdldChl'
        'bmRwb2ludCkKICAgICAgICAgICAgICAgICAgICBrZXkgPSBzdHIocmVzcG9uc2Uuc3RhdHVzX2Nv'
        'ZGUpCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGVycm9yOgogICAgICAgICAg'
        'ICAgICAgICAgIGtleSA9IGYiZXJyb3I6e3R5cGUoZXJyb3IpLl9fbmFtZV9ffSIKICAgICAgICAg'
        'ICAgICAgIGxhdGVuY2llcy5hcHBlbmQoKHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkKSAq'
        'IDFfMDAwKQogICAgICAgICAgICAgICAgc3RhdHVzZXNba2V5XSA9IHN0YXR1c2VzLmdldChrZXks'
        'IDApICsgMQoKICAgICAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGF3'
        'YWl0IGFzeW5jaW8uZ2F0aGVyKCoob25lKCkgZm9yIF8gaW4gcmFuZ2UocmVxdWVzdHMpKSkKICAg'
        'ICAgICBlbGFwc2VkID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQKCiAgICBzdWNjZXNz'
        'ZnVsID0gc3VtKGNvdW50IGZvciBzdGF0dXMsIGNvdW50IGluIHN0YXR1c2VzLml0ZW1zKCkgaWYg'
        'c3RhdHVzLnN0YXJ0c3dpdGgoIjIiKSkKICAgIGZhaWxlZCA9IHJlcXVlc3RzIC0gc3VjY2Vzc2Z1'
        'bAogICAgcmV0dXJuIExvYWRTbW9rZVJlcG9ydCgKICAgICAgICBlbmRwb2ludD1lbmRwb2ludCwK'
        'ICAgICAgICByZXF1ZXN0cz1yZXF1ZXN0cywKICAgICAgICBjb25jdXJyZW5jeT1jb25jdXJyZW5j'
        'eSwKICAgICAgICBzdWNjZXNzZnVsX3JlcXVlc3RzPXN1Y2Nlc3NmdWwsCiAgICAgICAgZmFpbGVk'
        'X3JlcXVlc3RzPWZhaWxlZCwKICAgICAgICBlcnJvcl9yYXRlPWZhaWxlZCAvIHJlcXVlc3RzLAog'
        'ICAgICAgIGVsYXBzZWRfc2Vjb25kcz1lbGFwc2VkLAogICAgICAgIHRocm91Z2hwdXRfcmVxdWVz'
        'dHNfcGVyX3NlY29uZD1yZXF1ZXN0cyAvIGVsYXBzZWQsCiAgICAgICAgcDUwX2xhdGVuY3lfbXM9'
        'c3RhdGlzdGljcy5tZWRpYW4obGF0ZW5jaWVzKSwKICAgICAgICBwOTVfbGF0ZW5jeV9tcz1fcGVy'
        'Y2VudGlsZShsYXRlbmNpZXMsIDAuOTUpLAogICAgICAgIHA5OV9sYXRlbmN5X21zPV9wZXJjZW50'
        'aWxlKGxhdGVuY2llcywgMC45OSksCiAgICAgICAgc3RhdHVzX2NvdW50cz1kaWN0KHNvcnRlZChz'
        'dGF0dXNlcy5pdGVtcygpKSksCiAgICApCgoKZGVmIHdyaXRlX2xvYWRfc21va2VfcmVwb3J0KHJl'
        'cG9ydDogTG9hZFNtb2tlUmVwb3J0LCBvdXRwdXRfcGF0aDogUGF0aCkgLT4gTm9uZToKICAgIF9h'
        'dG9taWNfanNvbihvdXRwdXRfcGF0aCwgcmVwb3J0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpKQo='
    ),
    'olympus/foundry/ollama.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgaHR0cHgK'
        'CmZyb20gb2x5bXB1cy5mb3VuZHJ5LnNjaGVtYXMgaW1wb3J0IEdlbmVyYXRpb25SZXN1bHQKCgpA'
        'ZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIE9sbGFtYUNsaWVudDoKICAgIGJhc2VfdXJsOiBz'
        'dHIgPSAiaHR0cDovLzEyNy4wLjAuMToxMTQzNCIKICAgIHRpbWVvdXRfc2Vjb25kczogZmxvYXQg'
        'PSAxMjAuMAogICAgdHJhbnNwb3J0OiBodHRweC5CYXNlVHJhbnNwb3J0IHwgTm9uZSA9IGZpZWxk'
        'KGRlZmF1bHQ9Tm9uZSwgcmVwcj1GYWxzZSkKCiAgICBkZWYgX19wb3N0X2luaXRfXyhzZWxmKSAt'
        'PiBOb25lOgogICAgICAgIHNlbGYuYmFzZV91cmwgPSBzZWxmLmJhc2VfdXJsLnJzdHJpcCgiLyIp'
        'CiAgICAgICAgaWYgbm90IHNlbGYuYmFzZV91cmwuc3RhcnRzd2l0aCgoImh0dHA6Ly8iLCAiaHR0'
        'cHM6Ly8iKSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk9sbGFtYSBiYXNlX3VybCBt'
        'dXN0IGJlIGFuIEhUVFAgb3IgSFRUUFMgVVJMIikKICAgICAgICBpZiBzZWxmLnRpbWVvdXRfc2Vj'
        'b25kcyA8PSAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0aW1lb3V0X3NlY29uZHMg'
        'bXVzdCBiZSBwb3NpdGl2ZSIpCgogICAgZGVmIF9yZXF1ZXN0KHNlbGYsIG1ldGhvZDogc3RyLCBw'
        'YXRoOiBzdHIsICoqa3dhcmdzOiBBbnkpIC0+IGRpY3Rbc3RyLCBBbnldOgogICAgICAgIHdpdGgg'
        'aHR0cHguQ2xpZW50KAogICAgICAgICAgICBiYXNlX3VybD1zZWxmLmJhc2VfdXJsLAogICAgICAg'
        'ICAgICB0aW1lb3V0PXNlbGYudGltZW91dF9zZWNvbmRzLAogICAgICAgICAgICB0cmFuc3BvcnQ9'
        'c2VsZi50cmFuc3BvcnQsCiAgICAgICAgKSBhcyBjbGllbnQ6CiAgICAgICAgICAgIHJlc3BvbnNl'
        'ID0gY2xpZW50LnJlcXVlc3QobWV0aG9kLCBwYXRoLCAqKmt3YXJncykKICAgICAgICAgICAgcmVz'
        'cG9uc2UucmFpc2VfZm9yX3N0YXR1cygpCiAgICAgICAgICAgIHBheWxvYWQgPSByZXNwb25zZS5q'
        'c29uKCkKICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShwYXlsb2FkLCBkaWN0KToKICAgICAgICAg'
        'ICAgcmFpc2UgVmFsdWVFcnJvcigiT2xsYW1hIHJldHVybmVkIGEgbm9uLW9iamVjdCByZXNwb25z'
        'ZSIpCiAgICAgICAgcmV0dXJuIHBheWxvYWQKCiAgICBkZWYgbGlzdF9tb2RlbHMoc2VsZikgLT4g'
        'bGlzdFtzdHJdOgogICAgICAgIHBheWxvYWQgPSBzZWxmLl9yZXF1ZXN0KCJHRVQiLCAiL2FwaS90'
        'YWdzIikKICAgICAgICBtb2RlbHMgPSBwYXlsb2FkLmdldCgibW9kZWxzIikKICAgICAgICBpZiBu'
        'b3QgaXNpbnN0YW5jZShtb2RlbHMsIGxpc3QpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9y'
        'KCJPbGxhbWEgbW9kZWwgbGlzdCBpcyBtYWxmb3JtZWQiKQogICAgICAgIG5hbWVzOiBsaXN0W3N0'
        'cl0gPSBbXQogICAgICAgIGZvciBtb2RlbCBpbiBtb2RlbHM6CiAgICAgICAgICAgIGlmIG5vdCBp'
        'c2luc3RhbmNlKG1vZGVsLCBkaWN0KSBvciBub3QgaXNpbnN0YW5jZShtb2RlbC5nZXQoIm5hbWUi'
        'KSwgc3RyKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk9sbGFtYSBtb2RlbCBl'
        'bnRyeSBpcyBtYWxmb3JtZWQiKQogICAgICAgICAgICBuYW1lcy5hcHBlbmQobW9kZWxbIm5hbWUi'
        'XSkKICAgICAgICByZXR1cm4gbmFtZXMKCiAgICBkZWYgaGVhbHRoKHNlbGYpIC0+IGRpY3Rbc3Ry'
        'LCBvYmplY3RdOgogICAgICAgIG1vZGVscyA9IHNlbGYubGlzdF9tb2RlbHMoKQogICAgICAgIHJl'
        'dHVybiB7InN0YXR1cyI6ICJvayIsICJwcm92aWRlciI6ICJvbGxhbWEiLCAibW9kZWxzIjogbW9k'
        'ZWxzfQoKICAgIGRlZiBnZW5lcmF0ZSgKICAgICAgICBzZWxmLAogICAgICAgICosCiAgICAgICAg'
        'bW9kZWw6IHN0ciwKICAgICAgICBwcm9tcHQ6IHN0ciwKICAgICAgICBzeXN0ZW06IHN0ciB8IE5v'
        'bmUgPSBOb25lLAogICAgICAgIG1heF90b2tlbnM6IGludCA9IDI1NiwKICAgICAgICBjb250ZXh0'
        'X3Rva2VuczogaW50ID0gMl8wNDgsCiAgICAgICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC4wLAog'
        'ICAgKSAtPiBHZW5lcmF0aW9uUmVzdWx0OgogICAgICAgIGlmIG5vdCBtb2RlbC5zdHJpcCgpOgog'
        'ICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtb2RlbCBpcyByZXF1aXJlZCIpCiAgICAgICAg'
        'aWYgbm90IHByb21wdC5zdHJpcCgpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcm9t'
        'cHQgaXMgcmVxdWlyZWQiKQogICAgICAgIGlmIG5vdCAxIDw9IG1heF90b2tlbnMgPD0gNF8wOTY6'
        'CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1heF90b2tlbnMgbXVzdCBiZSBiZXR3ZWVu'
        'IDEgYW5kIDQwOTYiKQogICAgICAgIGlmIG5vdCAyNTYgPD0gY29udGV4dF90b2tlbnMgPD0gMTMx'
        'XzA3MjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY29udGV4dF90b2tlbnMgbXVzdCBi'
        'ZSBiZXR3ZWVuIDI1NiBhbmQgMTMxMDcyIikKICAgICAgICBpZiBub3QgMCA8PSB0ZW1wZXJhdHVy'
        'ZSA8PSAyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0ZW1wZXJhdHVyZSBtdXN0IGJl'
        'IGJldHdlZW4gMCBhbmQgMiIpCiAgICAgICAgbWVzc2FnZXMgPSBbXQogICAgICAgIGlmIHN5c3Rl'
        'bToKICAgICAgICAgICAgbWVzc2FnZXMuYXBwZW5kKHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVu'
        'dCI6IHN5c3RlbX0pCiAgICAgICAgbWVzc2FnZXMuYXBwZW5kKHsicm9sZSI6ICJ1c2VyIiwgImNv'
        'bnRlbnQiOiBwcm9tcHR9KQogICAgICAgIHBheWxvYWQgPSBzZWxmLl9yZXF1ZXN0KAogICAgICAg'
        'ICAgICAiUE9TVCIsCiAgICAgICAgICAgICIvYXBpL2NoYXQiLAogICAgICAgICAgICBqc29uPXsK'
        'ICAgICAgICAgICAgICAgICJtb2RlbCI6IG1vZGVsLAogICAgICAgICAgICAgICAgIm1lc3NhZ2Vz'
        'IjogbWVzc2FnZXMsCiAgICAgICAgICAgICAgICAic3RyZWFtIjogRmFsc2UsCiAgICAgICAgICAg'
        'ICAgICAidGhpbmsiOiBGYWxzZSwKICAgICAgICAgICAgICAgICJvcHRpb25zIjogewogICAgICAg'
        'ICAgICAgICAgICAgICJudW1fcHJlZGljdCI6IG1heF90b2tlbnMsCiAgICAgICAgICAgICAgICAg'
        'ICAgIm51bV9jdHgiOiBjb250ZXh0X3Rva2VucywKICAgICAgICAgICAgICAgICAgICAidGVtcGVy'
        'YXR1cmUiOiB0ZW1wZXJhdHVyZSwKICAgICAgICAgICAgICAgIH0sCiAgICAgICAgICAgIH0sCiAg'
        'ICAgICAgKQogICAgICAgIG1lc3NhZ2UgPSBwYXlsb2FkLmdldCgibWVzc2FnZSIpCiAgICAgICAg'
        'aWYgbm90IGlzaW5zdGFuY2UobWVzc2FnZSwgZGljdCkgb3Igbm90IGlzaW5zdGFuY2UobWVzc2Fn'
        'ZS5nZXQoImNvbnRlbnQiKSwgc3RyKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiT2xs'
        'YW1hIGNoYXQgcmVzcG9uc2UgaXMgbWFsZm9ybWVkIikKICAgICAgICBjb250ZW50ID0gbWVzc2Fn'
        'ZVsiY29udGVudCJdCiAgICAgICAgcmV0dXJuIEdlbmVyYXRpb25SZXN1bHQoCiAgICAgICAgICAg'
        'IG1vZGVsPW1vZGVsLAogICAgICAgICAgICBjb250ZW50PWNvbnRlbnQsCiAgICAgICAgICAgIGZp'
        'bmlzaF9yZWFzb249InN0b3AiIGlmIHBheWxvYWQuZ2V0KCJkb25lIikgaXMgVHJ1ZSBlbHNlICJs'
        'ZW5ndGgiLAogICAgICAgICAgICBwcm9tcHRfY2hhcmFjdGVycz1sZW4ocHJvbXB0KSwKICAgICAg'
        'ICAgICAgZ2VuZXJhdGVkX2NoYXJhY3RlcnM9bGVuKGNvbnRlbnQpLAogICAgICAgICAgICBldmlk'
        'ZW5jZT17CiAgICAgICAgICAgICAgICAicHJvdmlkZXIiOiAib2xsYW1hIiwKICAgICAgICAgICAg'
        'ICAgICJkb25lX3JlYXNvbiI6IHBheWxvYWQuZ2V0KCJkb25lX3JlYXNvbiIpLAogICAgICAgICAg'
        'ICAgICAgInRvdGFsX2R1cmF0aW9uX25zIjogcGF5bG9hZC5nZXQoInRvdGFsX2R1cmF0aW9uIiks'
        'CiAgICAgICAgICAgICAgICAiZXZhbF9jb3VudCI6IHBheWxvYWQuZ2V0KCJldmFsX2NvdW50Iiks'
        'CiAgICAgICAgICAgICAgICAibG9hZF9kdXJhdGlvbl9ucyI6IHBheWxvYWQuZ2V0KCJsb2FkX2R1'
        'cmF0aW9uIiksCiAgICAgICAgICAgIH0sCiAgICAgICAgKQo='
    ),
    'olympus/foundry/promotion.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0'
        'IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGlt'
        'cG9ydCBOYW1lZFRlbXBvcmFyeUZpbGUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJvbSBvbHlt'
        'cHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQgU3RyaWN0TW9kZWwKZnJvbSBvbHltcHVzLmZvdW5kcnku'
        'ZGF0YV9waXBlbGluZSBpbXBvcnQgRGF0YXNldE1hbmlmZXN0VjIsIHZlcmlmeV9kYXRhc2V0X21h'
        'bmlmZXN0CmZyb20gb2x5bXB1cy5mb3VuZHJ5LmV2YWxfc3VpdGUgaW1wb3J0IEhlbGRPdXRFdmFs'
        'dWF0aW9uCmZyb20gb2x5bXB1cy5mb3VuZHJ5LnF1YW50aXphdGlvbiBpbXBvcnQgUXVhbnRpemF0'
        'aW9uUmVwb3J0CgoKZGVmIF9zaGEyNTYocGF0aDogUGF0aCkgLT4gc3RyOgogICAgcmV0dXJuIGhh'
        'c2hsaWIuc2hhMjU2KHBhdGgucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKQoKCmRlZiBfYXRvbWlj'
        'X2pzb24ocGF0aDogUGF0aCwgcGF5bG9hZDogb2JqZWN0KSAtPiBOb25lOgogICAgcGF0aC5wYXJl'
        'bnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgc2VyaWFsaXplZCA9IChq'
        'c29uLmR1bXBzKHBheWxvYWQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iKS5lbmNv'
        'ZGUoKQogICAgd2l0aCBOYW1lZFRlbXBvcmFyeUZpbGUoZGlyPXBhdGgucGFyZW50LCBwcmVmaXg9'
        'ZiIue3BhdGgubmFtZX0uIiwgZGVsZXRlPUZhbHNlKSBhcyBoYW5kbGU6CiAgICAgICAgaGFuZGxl'
        'LndyaXRlKHNlcmlhbGl6ZWQpCiAgICAgICAgaGFuZGxlLmZsdXNoKCkKICAgICAgICBvcy5mc3lu'
        'YyhoYW5kbGUuZmlsZW5vKCkpCiAgICAgICAgdGVtcG9yYXJ5ID0gUGF0aChoYW5kbGUubmFtZSkK'
        'ICAgIHRlbXBvcmFyeS5yZXBsYWNlKHBhdGgpCgoKY2xhc3MgR2F0ZVJlc3VsdChTdHJpY3RNb2Rl'
        'bCk6CiAgICBnYXRlOiBzdHIKICAgIHBhc3NlZDogYm9vbAogICAgb2JzZXJ2ZWQ6IHN0cgogICAg'
        'cmVxdWlyZW1lbnQ6IHN0cgoKCmNsYXNzIFByb21vdGlvblJlcG9ydChTdHJpY3RNb2RlbCk6CiAg'
        'ICBzY2hlbWFfdmVyc2lvbjogaW50ID0gMQogICAgcmVxdWVzdGVkX21vZGVsX2lkOiBzdHIKICAg'
        'IGNoZWNrcG9pbnRfc2hhMjU2OiBzdHIKICAgIGdhdGVzOiBsaXN0W0dhdGVSZXN1bHRdCiAgICBw'
        'YXNzZWQ6IGJvb2wKICAgIHN0YXR1czogc3RyCiAgICByZWxlYXNlX21hbmlmZXN0X3BhdGg6IHN0'
        'ciB8IE5vbmUKICAgIGJsb2NrZXJzOiBsaXN0W3N0cl0KCgpkZWYgX2dhdGUobmFtZTogc3RyLCBw'
        'YXNzZWQ6IGJvb2wsIG9ic2VydmVkOiBvYmplY3QsIHJlcXVpcmVtZW50OiBzdHIpIC0+IEdhdGVS'
        'ZXN1bHQ6CiAgICByZXR1cm4gR2F0ZVJlc3VsdCgKICAgICAgICBnYXRlPW5hbWUsCiAgICAgICAg'
        'cGFzc2VkPXBhc3NlZCwKICAgICAgICBvYnNlcnZlZD1zdHIob2JzZXJ2ZWQpLAogICAgICAgIHJl'
        'cXVpcmVtZW50PXJlcXVpcmVtZW50LAogICAgKQoKCmRlZiBfd29ya2Zsb3dfZmxvb3IoZXZhbHVh'
        'dGlvbjogSGVsZE91dEV2YWx1YXRpb24pIC0+IGZsb2F0OgogICAgcmV0dXJuIG1pbihzY29yZS5l'
        'eGFjdF9tYXRjaF9yYXRlIGZvciBzY29yZSBpbiBldmFsdWF0aW9uLndvcmtmbG93X3Njb3JlcykK'
        'CgpkZWYgZXZhbHVhdGVfcHJvbW90aW9uKAogICAgKiwKICAgIHJlcXVlc3RlZF9tb2RlbF9pZDog'
        'c3RyLAogICAgY2hlY2twb2ludF9wYXRoOiBQYXRoLAogICAgZGF0YXNldF9tYW5pZmVzdF9wYXRo'
        'OiBQYXRoLAogICAgZXZhbHVhdGlvbl9wYXRoOiBQYXRoLAogICAgcXVhbnRpemF0aW9uX3JlcG9y'
        'dF9wYXRoOiBQYXRoLAogICAgbW9kZWxfY2FyZF9wYXRoOiBQYXRoLAogICAgb3V0cHV0X3BhdGg6'
        'IFBhdGgsCiAgICBhcHByb3ZlZF9iYXNlX2xpY2Vuc2U6IHN0ciwKICAgIHNlcnZpbmdfdmVyaWZp'
        'Y2F0aW9uOiBkaWN0W3N0ciwgQW55XSB8IE5vbmUgPSBOb25lLAopIC0+IFByb21vdGlvblJlcG9y'
        'dDoKICAgIGNoZWNrcG9pbnRfc2hhID0gX3NoYTI1NihjaGVja3BvaW50X3BhdGgpCiAgICBtYW5p'
        'ZmVzdDogRGF0YXNldE1hbmlmZXN0VjIgPSB2ZXJpZnlfZGF0YXNldF9tYW5pZmVzdChkYXRhc2V0'
        'X21hbmlmZXN0X3BhdGgpCiAgICBldmFsdWF0aW9uID0gSGVsZE91dEV2YWx1YXRpb24ubW9kZWxf'
        'dmFsaWRhdGVfanNvbihldmFsdWF0aW9uX3BhdGgucmVhZF9ieXRlcygpKQogICAgcXVhbnRpemF0'
        'aW9uID0gUXVhbnRpemF0aW9uUmVwb3J0Lm1vZGVsX3ZhbGlkYXRlX2pzb24ocXVhbnRpemF0aW9u'
        'X3JlcG9ydF9wYXRoLnJlYWRfYnl0ZXMoKSkKICAgIHRyYWluID0gbmV4dChzcGxpdCBmb3Igc3Bs'
        'aXQgaW4gbWFuaWZlc3Quc3BsaXRzIGlmIHNwbGl0Lm5hbWUgPT0gInRyYWluIikKICAgIHRlc3Qg'
        'PSBuZXh0KHNwbGl0IGZvciBzcGxpdCBpbiBtYW5pZmVzdC5zcGxpdHMgaWYgc3BsaXQubmFtZSA9'
        'PSAidGVzdCIpCiAgICBtaW5pbXVtX2NhdGVnb3J5X3JlY29yZHMgPSBtaW4odGVzdC5jYXRlZ29y'
        'aWVzLnZhbHVlcygpKQogICAgbW9kZWxfY2FyZCA9IG1vZGVsX2NhcmRfcGF0aC5yZWFkX3RleHQo'
        'ZW5jb2Rpbmc9InV0Zi04IikgaWYgbW9kZWxfY2FyZF9wYXRoLmlzX2ZpbGUoKSBlbHNlICIiCiAg'
        'ICBzZXJ2aW5nID0gc2VydmluZ192ZXJpZmljYXRpb24gb3Ige30KICAgIGFsbG93ZWRfbGljZW5z'
        'ZSA9IGFwcHJvdmVkX2Jhc2VfbGljZW5zZSBpbiB7IkFwYWNoZS0yLjAiLCAiTUlUIiwgIkNDLUJZ'
        'LTQuMCJ9CiAgICBnYXRlcyA9IFsKICAgICAgICBfZ2F0ZSgKICAgICAgICAgICAgImNoZWNrcG9p'
        'bnRfaWRlbnRpdHkiLAogICAgICAgICAgICBldmFsdWF0aW9uLmNoZWNrcG9pbnRfc2hhMjU2ID09'
        'IGNoZWNrcG9pbnRfc2hhCiAgICAgICAgICAgIGFuZCBxdWFudGl6YXRpb24uc291cmNlX2NoZWNr'
        'cG9pbnRfc2hhMjU2ID09IGNoZWNrcG9pbnRfc2hhLAogICAgICAgICAgICBjaGVja3BvaW50X3No'
        'YSwKICAgICAgICAgICAgIkV2YWx1YXRpb24gYW5kIHF1YW50aXphdGlvbiBldmlkZW5jZSBtdXN0'
        'IHJlZmVyZW5jZSB0aGUgZXhhY3QgY2hlY2twb2ludCBoYXNoLiIsCiAgICAgICAgKSwKICAgICAg'
        'ICBfZ2F0ZSgKICAgICAgICAgICAgImRhdGFzZXRfaWRlbnRpdHkiLAogICAgICAgICAgICBtYW5p'
        'ZmVzdC5tYW5pZmVzdF9zaGEyNTYgPT0gZXZhbHVhdGlvbi5kYXRhc2V0X21hbmlmZXN0X3NoYTI1'
        'NiwKICAgICAgICAgICAgbWFuaWZlc3QubWFuaWZlc3Rfc2hhMjU2LAogICAgICAgICAgICAiRXZh'
        'bHVhdGlvbiBtdXN0IHJlZmVyZW5jZSB0aGUgZXhhY3QgaW1tdXRhYmxlIGRhdGFzZXQgbWFuaWZl'
        'c3QuIiwKICAgICAgICApLAogICAgICAgIF9nYXRlKAogICAgICAgICAgICAiYmFzZV9saWNlbnNl'
        'IiwKICAgICAgICAgICAgYWxsb3dlZF9saWNlbnNlLAogICAgICAgICAgICBhcHByb3ZlZF9iYXNl'
        'X2xpY2Vuc2UsCiAgICAgICAgICAgICJCYXNlLW1vZGVsIHJpZ2h0cyBtdXN0IGJlIGV4cGxpY2l0'
        'bHkgYXBwcm92ZWQgZm9yIHRyYWluaW5nIGFuZCBkaXN0cmlidXRpb24uIiwKICAgICAgICApLAog'
        'ICAgICAgIF9nYXRlKAogICAgICAgICAgICAidHJhaW5pbmdfc2NhbGUiLAogICAgICAgICAgICB0'
        'cmFpbi5yZWNvcmRzID49IDEwXzAwMCwKICAgICAgICAgICAgdHJhaW4ucmVjb3JkcywKICAgICAg'
        'ICAgICAgIkF0IGxlYXN0IDEwLDAwMCByZXZpZXdlZCBTRlQgdHJhaW5pbmcgcmVjb3JkcyBhcmUg'
        'cmVxdWlyZWQgZm9yIEhlcm1lcyBBbHBoYS4iLAogICAgICAgICksCiAgICAgICAgX2dhdGUoCiAg'
        'ICAgICAgICAgICJoZWxkX291dF9zY2FsZSIsCiAgICAgICAgICAgIHRlc3QucmVjb3JkcyA+PSAx'
        'XzIwMCBhbmQgbWluaW11bV9jYXRlZ29yeV9yZWNvcmRzID49IDEwMCwKICAgICAgICAgICAgZiJ0'
        'b3RhbD17dGVzdC5yZWNvcmRzfSwgbWluaW11bV9wZXJfY2F0ZWdvcnk9e21pbmltdW1fY2F0ZWdv'
        'cnlfcmVjb3Jkc30iLAogICAgICAgICAgICAiQXQgbGVhc3QgMSwyMDAgaGVsZC1vdXQgcmVjb3Jk'
        'cyBhbmQgMTAwIHBlciBjYXRlZ29yeSBhcmUgcmVxdWlyZWQuIiwKICAgICAgICApLAogICAgICAg'
        'IF9nYXRlKAogICAgICAgICAgICAibG9zc19hbmRfcmVncmVzc2lvbiIsCiAgICAgICAgICAgIGV2'
        'YWx1YXRpb24ub3ZlcmFsbF9jYW5kaWRhdGVfbG9zcyA8IGV2YWx1YXRpb24ub3ZlcmFsbF9iYXNl'
        'bGluZV9sb3NzCiAgICAgICAgICAgIGFuZCBldmFsdWF0aW9uLnJlZ3Jlc3Npb25fY291bnQgPT0g'
        'MCwKICAgICAgICAgICAgKAogICAgICAgICAgICAgICAgZiJiYXNlbGluZT17ZXZhbHVhdGlvbi5v'
        'dmVyYWxsX2Jhc2VsaW5lX2xvc3M6LjZmfSwgIgogICAgICAgICAgICAgICAgZiJjYW5kaWRhdGU9'
        'e2V2YWx1YXRpb24ub3ZlcmFsbF9jYW5kaWRhdGVfbG9zczouNmZ9LCAiCiAgICAgICAgICAgICAg'
        'ICBmInJlZ3Jlc3Npb25zPXtldmFsdWF0aW9uLnJlZ3Jlc3Npb25fY291bnR9IgogICAgICAgICAg'
        'ICApLAogICAgICAgICAgICAiQ2FuZGlkYXRlIGxvc3MgbXVzdCBpbXByb3ZlIGFuZCBubyBjYXBh'
        'YmlsaXR5IGNhdGVnb3J5IG1heSByZWdyZXNzIG92ZXIgNSUuIiwKICAgICAgICApLAogICAgICAg'
        'IF9nYXRlKAogICAgICAgICAgICAidGFza19xdWFsaXR5IiwKICAgICAgICAgICAgZXZhbHVhdGlv'
        'bi5leGFjdF9tYXRjaF9yYXRlID49IDAuODAKICAgICAgICAgICAgYW5kIGV2YWx1YXRpb24uZm9y'
        'bWF0X2NvbXBsaWFuY2VfcmF0ZSA+PSAwLjk1CiAgICAgICAgICAgIGFuZCBfd29ya2Zsb3dfZmxv'
        'b3IoZXZhbHVhdGlvbikgPj0gMC43NSwKICAgICAgICAgICAgKAogICAgICAgICAgICAgICAgZiJl'
        'eGFjdD17ZXZhbHVhdGlvbi5leGFjdF9tYXRjaF9yYXRlOi4zZn0sICIKICAgICAgICAgICAgICAg'
        'IGYiZm9ybWF0PXtldmFsdWF0aW9uLmZvcm1hdF9jb21wbGlhbmNlX3JhdGU6LjNmfSwgIgogICAg'
        'ICAgICAgICAgICAgZiJ3b3JrZmxvd19mbG9vcj17X3dvcmtmbG93X2Zsb29yKGV2YWx1YXRpb24p'
        'Oi4zZn0iCiAgICAgICAgICAgICksCiAgICAgICAgICAgICJFeGFjdCBtYXRjaCA+PTgwJSwgZm9y'
        'bWF0IGNvbXBsaWFuY2UgPj05NSUsIHdvcmtmbG93IGZsb29yID49NzUlLiIsCiAgICAgICAgKSwK'
        'ICAgICAgICBfZ2F0ZSgKICAgICAgICAgICAgInF1YW50aXphdGlvbl9xdWFsaXR5IiwKICAgICAg'
        'ICAgICAgcXVhbnRpemF0aW9uLnBhc3NlZF9xdWFsaXR5X2dhdGUKICAgICAgICAgICAgYW5kIHF1'
        'YW50aXphdGlvbi50b29sX2V4YWN0X21hdGNoX3JhdGUgPj0gMC43NSwKICAgICAgICAgICAgKAog'
        'ICAgICAgICAgICAgICAgZiJsb3NzX2NoYW5nZT17cXVhbnRpemF0aW9uLmxvc3NfY2hhbmdlX2Zy'
        'YWN0aW9uOi40Zn0sICIKICAgICAgICAgICAgICAgIGYidG9vbF9leGFjdD17cXVhbnRpemF0aW9u'
        'LnRvb2xfZXhhY3RfbWF0Y2hfcmF0ZTouM2Z9IgogICAgICAgICAgICApLAogICAgICAgICAgICAi'
        'UXVhbnRpemVkIGxvc3MgY2hhbmdlIDw9MiUsIGVuZm9yY2VkIGNvbnRleHQsIGFuZCB0b29sIGV4'
        'YWN0IG1hdGNoID49NzUlLiIsCiAgICAgICAgKSwKICAgICAgICBfZ2F0ZSgKICAgICAgICAgICAg'
        'InNlcnZpbmdfcmVwcm9kdWNpYmlsaXR5IiwKICAgICAgICAgICAgc2VydmluZy5nZXQoInBhc3Nl'
        'ZCIpIGlzIFRydWUKICAgICAgICAgICAgYW5kIHNlcnZpbmcuZ2V0KCJjaGVja3BvaW50X3NoYTI1'
        'NiIpID09IGNoZWNrcG9pbnRfc2hhLAogICAgICAgICAgICBzZXJ2aW5nIG9yICJtaXNzaW5nIiwK'
        'ICAgICAgICAgICAgIkZyZXNoLXByb2Nlc3MgQVBJLCBDTEksIGFuZCB3ZWIgc2VydmluZyBtdXN0'
        'IHBhc3MgYWdhaW5zdCB0aGUgZXhhY3QgY2hlY2twb2ludC4iLAogICAgICAgICksCiAgICAgICAg'
        'X2dhdGUoCiAgICAgICAgICAgICJtb2RlbF9jYXJkIiwKICAgICAgICAgICAgYm9vbChtb2RlbF9j'
        'YXJkKQogICAgICAgICAgICBhbmQgY2hlY2twb2ludF9zaGEgaW4gbW9kZWxfY2FyZAogICAgICAg'
        'ICAgICBhbmQgIkxpbWl0YXRpb25zIiBpbiBtb2RlbF9jYXJkCiAgICAgICAgICAgIGFuZCAiTGlj'
        'ZW5zZSIgaW4gbW9kZWxfY2FyZCwKICAgICAgICAgICAgc3RyKG1vZGVsX2NhcmRfcGF0aCksCiAg'
        'ICAgICAgICAgICJQdWJsaXNoZWQgY2FyZCBtdXN0IG5hbWUgdGhlIGV4YWN0IGhhc2gsIGxpY2Vu'
        'c2UsIGludGVuZGVkIHVzZSwgYW5kIGxpbWl0YXRpb25zLiIsCiAgICAgICAgKSwKICAgIF0KICAg'
        'IHBhc3NlZCA9IGFsbChnYXRlLnBhc3NlZCBmb3IgZ2F0ZSBpbiBnYXRlcykKICAgIHJlbGVhc2Vf'
        'bWFuaWZlc3RfcGF0aDogc3RyIHwgTm9uZSA9IE5vbmUKICAgIGlmIHBhc3NlZDoKICAgICAgICBy'
        'ZWxlYXNlX21hbmlmZXN0ID0gb3V0cHV0X3BhdGgud2l0aF9uYW1lKCJyZWxlYXNlLW1hbmlmZXN0'
        'Lmpzb24iKQogICAgICAgIF9hdG9taWNfanNvbigKICAgICAgICAgICAgcmVsZWFzZV9tYW5pZmVz'
        'dCwKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgInNjaGVtYV92ZXJzaW9uIjogMSwKICAg'
        'ICAgICAgICAgICAgICJtb2RlbF9pZCI6IHJlcXVlc3RlZF9tb2RlbF9pZCwKICAgICAgICAgICAg'
        'ICAgICJjaGVja3BvaW50X3NoYTI1NiI6IGNoZWNrcG9pbnRfc2hhLAogICAgICAgICAgICAgICAg'
        'ImRhdGFzZXRfbWFuaWZlc3Rfc2hhMjU2IjogbWFuaWZlc3QubWFuaWZlc3Rfc2hhMjU2LAogICAg'
        'ICAgICAgICAgICAgImV2YWx1YXRpb25fc2hhMjU2IjogX3NoYTI1NihldmFsdWF0aW9uX3BhdGgp'
        'LAogICAgICAgICAgICAgICAgInF1YW50aXphdGlvbl9yZXBvcnRfc2hhMjU2IjogX3NoYTI1Nihx'
        'dWFudGl6YXRpb25fcmVwb3J0X3BhdGgpLAogICAgICAgICAgICAgICAgIm1vZGVsX2NhcmRfc2hh'
        'MjU2IjogX3NoYTI1Nihtb2RlbF9jYXJkX3BhdGgpLAogICAgICAgICAgICAgICAgImJhc2VfbGlj'
        'ZW5zZSI6IGFwcHJvdmVkX2Jhc2VfbGljZW5zZSwKICAgICAgICAgICAgfSwKICAgICAgICApCiAg'
        'ICAgICAgcmVsZWFzZV9tYW5pZmVzdF9wYXRoID0gc3RyKHJlbGVhc2VfbWFuaWZlc3QucmVzb2x2'
        'ZSgpKQogICAgYmxvY2tlcnMgPSBbZ2F0ZS5nYXRlIGZvciBnYXRlIGluIGdhdGVzIGlmIG5vdCBn'
        'YXRlLnBhc3NlZF0KICAgIHJlcG9ydCA9IFByb21vdGlvblJlcG9ydCgKICAgICAgICByZXF1ZXN0'
        'ZWRfbW9kZWxfaWQ9cmVxdWVzdGVkX21vZGVsX2lkLAogICAgICAgIGNoZWNrcG9pbnRfc2hhMjU2'
        'PWNoZWNrcG9pbnRfc2hhLAogICAgICAgIGdhdGVzPWdhdGVzLAogICAgICAgIHBhc3NlZD1wYXNz'
        'ZWQsCiAgICAgICAgc3RhdHVzPSJQUk9NT1RFRCIgaWYgcGFzc2VkIGVsc2UgIk5PVF9QUk9NT1RF'
        'RCIsCiAgICAgICAgcmVsZWFzZV9tYW5pZmVzdF9wYXRoPXJlbGVhc2VfbWFuaWZlc3RfcGF0aCwK'
        'ICAgICAgICBibG9ja2Vycz1ibG9ja2VycywKICAgICkKICAgIF9hdG9taWNfanNvbihvdXRwdXRf'
        'cGF0aCwgcmVwb3J0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpKQogICAgcmV0dXJuIHJlcG9ydAo='
    ),
    'olympus/foundry/quantization.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0'
        'IGpzb24KaW1wb3J0IG9zCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9t'
        'IHRlbXBmaWxlIGltcG9ydCBOYW1lZFRlbXBvcmFyeUZpbGUKZnJvbSB0eXBpbmcgaW1wb3J0IEFu'
        'eSwgTGl0ZXJhbCwgY2FzdAoKaW1wb3J0IHRvcmNoCmZyb20gcHlkYW50aWMgaW1wb3J0IEZpZWxk'
        'Cgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RNb2RlbApmcm9tIG9seW1w'
        'dXMuZm91bmRyeS5kYXRhX3BpcGVsaW5lIGltcG9ydCBsb2FkX3ZlcmlmaWVkX2RhdGFzZXQKZnJv'
        'bSBvbHltcHVzLmZvdW5kcnkucmVzb3VyY2VzIGltcG9ydCBtZW1vcnlfc25hcHNob3QKZnJvbSBv'
        'bHltcHVzLmZvdW5kcnkuc2Z0IGltcG9ydCAoCiAgICBMb3NzTm9ybWFsaXphdGlvbiwKICAgIFRp'
        'bnlDYXVzYWxMTSwKICAgIFRpbnlNb2RlbENvbmZpZywKICAgIF9jaGVja3BvaW50X3RyYWluaW5n'
        'X2NvbmZpZywKICAgIF9lbmNvZGVfZXhhbXBsZXMsCiAgICBfbW9kZWxfZnJvbV9jaGVja3BvaW50'
        'LAogICAgX3BhY2tfbmliYmxlcywKICAgIF91bnBhY2tfbmliYmxlcywKICAgIGV2YWx1YXRlX2xv'
        'c3MsCiAgICBnZW5lcmF0ZV90ZXh0LAopCgpRdWFudGl6YXRpb25CaXRzID0gTGl0ZXJhbFs0LCA4'
        'XQoKCmRlZiBfc2hhMjU2KHBhdGg6IFBhdGgpIC0+IHN0cjoKICAgIHJldHVybiBoYXNobGliLnNo'
        'YTI1NihwYXRoLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgX2F0b21pY190b3JjaF9z'
        'YXZlKHBhdGg6IFBhdGgsIHBheWxvYWQ6IG9iamVjdCkgLT4gTm9uZToKICAgIHBhdGgucGFyZW50'
        'Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggTmFtZWRUZW1wb3Jh'
        'cnlGaWxlKGRpcj1wYXRoLnBhcmVudCwgcHJlZml4PWYiLntwYXRoLm5hbWV9LiIsIGRlbGV0ZT1G'
        'YWxzZSkgYXMgaGFuZGxlOgogICAgICAgIHRlbXBvcmFyeSA9IFBhdGgoaGFuZGxlLm5hbWUpCiAg'
        'ICB0cnk6CiAgICAgICAgdG9yY2guc2F2ZShwYXlsb2FkLCB0ZW1wb3JhcnkpCiAgICAgICAgd2l0'
        'aCB0ZW1wb3Jhcnkub3BlbigicmIiKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIG9zLmZzeW5jKGhh'
        'bmRsZS5maWxlbm8oKSkKICAgICAgICB0ZW1wb3JhcnkucmVwbGFjZShwYXRoKQogICAgZmluYWxs'
        'eToKICAgICAgICB0ZW1wb3JhcnkudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkKCgpkZWYgX2F0b21p'
        'Y19qc29uKHBhdGg6IFBhdGgsIHBheWxvYWQ6IG9iamVjdCkgLT4gTm9uZToKICAgIHNlcmlhbGl6'
        'ZWQgPSAoanNvbi5kdW1wcyhwYXlsb2FkLCBpbmRlbnQ9Miwgc29ydF9rZXlzPVRydWUpICsgIlxu'
        'IikuZW5jb2RlKCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9'
        'VHJ1ZSkKICAgIHdpdGggTmFtZWRUZW1wb3JhcnlGaWxlKGRpcj1wYXRoLnBhcmVudCwgcHJlZml4'
        'PWYiLntwYXRoLm5hbWV9LiIsIGRlbGV0ZT1GYWxzZSkgYXMgaGFuZGxlOgogICAgICAgIGhhbmRs'
        'ZS53cml0ZShzZXJpYWxpemVkKQogICAgICAgIGhhbmRsZS5mbHVzaCgpCiAgICAgICAgb3MuZnN5'
        'bmMoaGFuZGxlLmZpbGVubygpKQogICAgICAgIHRlbXBvcmFyeSA9IFBhdGgoaGFuZGxlLm5hbWUp'
        'CiAgICB0ZW1wb3JhcnkucmVwbGFjZShwYXRoKQoKCmNsYXNzIFF1YW50aXplZFRlbnNvcihTdHJp'
        'Y3RNb2RlbCk6CiAgICBuYW1lOiBzdHIKICAgIHNoYXBlOiBsaXN0W2ludF0KICAgIGZsb2F0aW5n'
        'OiBib29sCiAgICBheGlzOiBpbnQgfCBOb25lCgoKY2xhc3MgUXVhbnRpemF0aW9uUmVwb3J0KFN0'
        'cmljdE1vZGVsKToKICAgIHNjaGVtYV92ZXJzaW9uOiBpbnQgPSAxCiAgICBmb3JtYXQ6IHN0cgog'
        'ICAgYml0czogUXVhbnRpemF0aW9uQml0cwogICAgc291cmNlX2NoZWNrcG9pbnRfc2hhMjU2OiBz'
        'dHIKICAgIGFydGlmYWN0X3BhdGg6IHN0cgogICAgYXJ0aWZhY3Rfc2hhMjU2OiBzdHIKICAgIGZs'
        'b2F0X3dlaWdodHNfYnl0ZXM6IGludCA9IEZpZWxkKGd0PTApCiAgICBxdWFudGl6ZWRfYnl0ZXM6'
        'IGludCA9IEZpZWxkKGd0PTApCiAgICBzaXplX3JlZHVjdGlvbl9mcmFjdGlvbjogZmxvYXQKICAg'
        'IHNvdXJjZV9sb3NzOiBmbG9hdAogICAgcXVhbnRpemVkX2xvc3M6IGZsb2F0CiAgICBsb3NzX2No'
        'YW5nZV9mcmFjdGlvbjogZmxvYXQKICAgIHNvdXJjZV9sYXRlbmN5X21zOiBmbG9hdAogICAgcXVh'
        'bnRpemVkX2xhdGVuY3lfbXM6IGZsb2F0CiAgICBzdGFydHVwX2xhdGVuY3lfbXM6IGZsb2F0CiAg'
        'ICBtYXhfY29udGV4dF90b2tlbnM6IGludAogICAgY29udGV4dF9saW1pdF9lbmZvcmNlZDogYm9v'
        'bAogICAgdG9vbF9leGFjdF9tYXRjaF9yYXRlOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4w'
        'KQogICAgcGVha19yc3NfYnl0ZXM6IGludCA9IEZpZWxkKGdlPTApCiAgICBwYXNzZWRfcXVhbGl0'
        'eV9nYXRlOiBib29sCiAgICBsb3NzX25vcm1hbGl6YXRpb246IExvc3NOb3JtYWxpemF0aW9uID0g'
        'ImxlZ2FjeV9iYXRjaF9tZWFuX3YxIgoKCmRlZiBfcXVhbnRpemVfdGVuc29yKHRlbnNvcjogdG9y'
        'Y2guVGVuc29yLCBiaXRzOiBRdWFudGl6YXRpb25CaXRzKSAtPiBkaWN0W3N0ciwgQW55XToKICAg'
        'IHZhbHVlID0gdGVuc29yLmRldGFjaCgpLmNwdSgpCiAgICBpZiBub3QgdmFsdWUuaXNfZmxvYXRp'
        'bmdfcG9pbnQoKToKICAgICAgICByZXR1cm4geyJmbG9hdGluZyI6IEZhbHNlLCAic2hhcGUiOiBs'
        'aXN0KHZhbHVlLnNoYXBlKSwgInZhbHVlIjogdmFsdWV9CiAgICBmbG9hdGluZyA9IHZhbHVlLmZs'
        'b2F0KCkKICAgIGF4aXMgPSAwIGlmIGZsb2F0aW5nLm5kaW0gPj0gMiBlbHNlIE5vbmUKICAgIGxp'
        'bWl0ID0gMTI3IGlmIGJpdHMgPT0gOCBlbHNlIDcKICAgIGlmIGF4aXMgPT0gMDoKICAgICAgICBz'
        'Y2FsZSA9IGZsb2F0aW5nLmFicygpLnJlc2hhcGUoZmxvYXRpbmcuc2hhcGVbMF0sIC0xKS5hbWF4'
        'KGRpbT0xKS5jbGFtcF9taW4oMWUtMTIpIC8gbGltaXQKICAgICAgICBzaGFwZWRfc2NhbGUgPSBz'
        'Y2FsZS5yZXNoYXBlKGZsb2F0aW5nLnNoYXBlWzBdLCAqKFsxXSAqIChmbG9hdGluZy5uZGltIC0g'
        'MSkpKQogICAgZWxzZToKICAgICAgICBzY2FsZSA9IGZsb2F0aW5nLmFicygpLmFtYXgoKS5yZXNo'
        'YXBlKDEpLmNsYW1wX21pbigxZS0xMikgLyBsaW1pdAogICAgICAgIHNoYXBlZF9zY2FsZSA9IHNj'
        'YWxlCiAgICBzaWduZWQgPSB0b3JjaC5yb3VuZChmbG9hdGluZyAvIHNoYXBlZF9zY2FsZSkuY2xh'
        'bXAoLWxpbWl0LCBsaW1pdCkudG8odG9yY2guaW50OCkKICAgIGlmIGJpdHMgPT0gODoKICAgICAg'
        'ICBxdWFudGl6ZWQgPSBzaWduZWQKICAgICAgICBjb3VudCA9IHNpZ25lZC5udW1lbCgpCiAgICBl'
        'bHNlOgogICAgICAgIHF1YW50aXplZCA9IF9wYWNrX25pYmJsZXMoc2lnbmVkICsgOCkKICAgICAg'
        'ICBjb3VudCA9IHNpZ25lZC5udW1lbCgpCiAgICByZXR1cm4gewogICAgICAgICJmbG9hdGluZyI6'
        'IFRydWUsCiAgICAgICAgInNoYXBlIjogbGlzdCh2YWx1ZS5zaGFwZSksCiAgICAgICAgImF4aXMi'
        'OiBheGlzLAogICAgICAgICJzY2FsZSI6IHNjYWxlLAogICAgICAgICJxdWFudGl6ZWQiOiBxdWFu'
        'dGl6ZWQsCiAgICAgICAgImNvdW50IjogY291bnQsCiAgICB9CgoKZGVmIF9kZXF1YW50aXplX3Rl'
        'bnNvcihwYXlsb2FkOiBkaWN0W3N0ciwgQW55XSwgYml0czogUXVhbnRpemF0aW9uQml0cykgLT4g'
        'dG9yY2guVGVuc29yOgogICAgaWYgbm90IHBheWxvYWRbImZsb2F0aW5nIl06CiAgICAgICAgcmV0'
        'dXJuIGNhc3QodG9yY2guVGVuc29yLCBwYXlsb2FkWyJ2YWx1ZSJdKQogICAgc2hhcGUgPSBjYXN0'
        'KGxpc3RbaW50XSwgcGF5bG9hZFsic2hhcGUiXSkKICAgIGlmIGJpdHMgPT0gODoKICAgICAgICBz'
        'aWduZWQgPSBjYXN0KHRvcmNoLlRlbnNvciwgcGF5bG9hZFsicXVhbnRpemVkIl0pLnRvKHRvcmNo'
        'LmludDgpCiAgICBlbHNlOgogICAgICAgIHBhY2tlZCA9IGNhc3QodG9yY2guVGVuc29yLCBwYXls'
        'b2FkWyJxdWFudGl6ZWQiXSkKICAgICAgICBzaWduZWQgPSBfdW5wYWNrX25pYmJsZXMocGFja2Vk'
        'LCBpbnQocGF5bG9hZFsiY291bnQiXSkpLnRvKHRvcmNoLmludDgpIC0gOAogICAgc2lnbmVkID0g'
        'c2lnbmVkLnJlc2hhcGUoc2hhcGUpLmZsb2F0KCkKICAgIHNjYWxlID0gY2FzdCh0b3JjaC5UZW5z'
        'b3IsIHBheWxvYWRbInNjYWxlIl0pLmZsb2F0KCkKICAgIGlmIHBheWxvYWRbImF4aXMiXSA9PSAw'
        'OgogICAgICAgIHNjYWxlID0gc2NhbGUucmVzaGFwZShzaGFwZVswXSwgKihbMV0gKiAobGVuKHNo'
        'YXBlKSAtIDEpKSkKICAgIHJldHVybiBzaWduZWQgKiBzY2FsZQoKCmRlZiBsb2FkX3F1YW50aXpl'
        'ZF9tb2RlbChwYXRoOiBQYXRoKSAtPiBUaW55Q2F1c2FsTE06CiAgICBwYXlsb2FkID0gY2FzdChk'
        'aWN0W3N0ciwgQW55XSwgdG9yY2gubG9hZChwYXRoLCBtYXBfbG9jYXRpb249ImNwdSIsIHdlaWdo'
        'dHNfb25seT1UcnVlKSkKICAgIGJpdHMgPSBjYXN0KFF1YW50aXphdGlvbkJpdHMsIGludChwYXls'
        'b2FkWyJiaXRzIl0pKQogICAgbW9kZWwgPSBUaW55Q2F1c2FsTE0oVGlueU1vZGVsQ29uZmlnLm1v'
        'ZGVsX3ZhbGlkYXRlKHBheWxvYWRbIm1vZGVsX2NvbmZpZyJdKSkKICAgIHN0YXRlID0gewogICAg'
        'ICAgIG5hbWU6IF9kZXF1YW50aXplX3RlbnNvcih0ZW5zb3JfcGF5bG9hZCwgYml0cykKICAgICAg'
        'ICBmb3IgbmFtZSwgdGVuc29yX3BheWxvYWQgaW4gY2FzdChkaWN0W3N0ciwgZGljdFtzdHIsIEFu'
        'eV1dLCBwYXlsb2FkWyJ0ZW5zb3JzIl0pLml0ZW1zKCkKICAgIH0KICAgIG1vZGVsLmxvYWRfc3Rh'
        'dGVfZGljdChzdGF0ZSkKICAgIG1vZGVsLmV2YWwoKQogICAgcmV0dXJuIG1vZGVsCgoKZGVmIF9s'
        'YXRlbmN5KG1vZGVsOiBUaW55Q2F1c2FsTE0sIHByb21wdDogc3RyKSAtPiBmbG9hdDoKICAgIHN0'
        'YXJ0ID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZ2VuZXJhdGVfdGV4dChtb2RlbCwgcHJvbXB0'
        'LCBkZXZpY2U9dG9yY2guZGV2aWNlKCJjcHUiKSwgbWF4X25ld190b2tlbnM9MTYpCiAgICByZXR1'
        'cm4gKHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydCkgKiAxXzAwMAoKCmRlZiBxdWFudGl6ZV9j'
        'aGVja3BvaW50KAogICAgY2hlY2twb2ludF9wYXRoOiBQYXRoLAogICAgbWFuaWZlc3RfcGF0aDog'
        'UGF0aCwKICAgIG91dHB1dF9yb290OiBQYXRoLAogICAgKiwKICAgIGJpdHM6IFF1YW50aXphdGlv'
        'bkJpdHMsCikgLT4gUXVhbnRpemF0aW9uUmVwb3J0OgogICAgc291cmNlLCBjaGVja3BvaW50ID0g'
        'X21vZGVsX2Zyb21fY2hlY2twb2ludChjaGVja3BvaW50X3BhdGgsIHRvcmNoLmRldmljZSgiY3B1'
        'IikpCiAgICB0cmFpbmluZyA9IF9jaGVja3BvaW50X3RyYWluaW5nX2NvbmZpZyhjaGVja3BvaW50'
        'KQogICAgaWYgdHJhaW5pbmcubW9kZSAhPSAiZnVsbCI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJv'
        'cigicG9zdC10cmFpbmluZyBxdWFudGl6YXRpb24gY3VycmVudGx5IHJlcXVpcmVzIGEgbWVyZ2Vk'
        'IGZ1bGwgY2hlY2twb2ludCIpCiAgICBtYW5pZmVzdCwgZXhhbXBsZXNfYnlfc3BsaXQgPSBsb2Fk'
        'X3ZlcmlmaWVkX2RhdGFzZXQobWFuaWZlc3RfcGF0aCkKICAgIGlmIGNoZWNrcG9pbnQuZ2V0KCJk'
        'YXRhc2V0X21hbmlmZXN0X3NoYTI1NiIpICE9IG1hbmlmZXN0Lm1hbmlmZXN0X3NoYTI1NjoKICAg'
        'ICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IGFuZCBxdWFudGl6YXRpb24gZGF0YXNl'
        'dCBoYXNoZXMgZG8gbm90IG1hdGNoIikKICAgIGV4YW1wbGVzID0gZXhhbXBsZXNfYnlfc3BsaXRb'
        'InRlc3QiXQogICAgZGVsIGV4YW1wbGVzX2J5X3NwbGl0CiAgICBvdXRwdXRfcm9vdC5ta2Rpcihw'
        'YXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBmbG9hdF9wYXRoID0gb3V0cHV0X3Jvb3Qg'
        'LyAiZmxvYXQzMi1tb2RlbC5wdCIKICAgIF9hdG9taWNfdG9yY2hfc2F2ZShmbG9hdF9wYXRoLCBz'
        'b3VyY2Uuc3RhdGVfZGljdCgpKQogICAgYXJ0aWZhY3QgPSBvdXRwdXRfcm9vdCAvIGYibW9kZWwt'
        'aW50e2JpdHN9LnB0IgogICAgdGVuc29yX3BheWxvYWRzID0gewogICAgICAgIG5hbWU6IF9xdWFu'
        'dGl6ZV90ZW5zb3IodGVuc29yLCBiaXRzKSBmb3IgbmFtZSwgdGVuc29yIGluIHNvdXJjZS5zdGF0'
        'ZV9kaWN0KCkuaXRlbXMoKQogICAgfQogICAgX2F0b21pY190b3JjaF9zYXZlKAogICAgICAgIGFy'
        'dGlmYWN0LAogICAgICAgIHsKICAgICAgICAgICAgInNjaGVtYV92ZXJzaW9uIjogMSwKICAgICAg'
        'ICAgICAgImZvcm1hdCI6IGYib2x5bXB1cy1zeW1tZXRyaWMtaW50e2JpdHN9LXYxIiwKICAgICAg'
        'ICAgICAgImJpdHMiOiBiaXRzLAogICAgICAgICAgICAibW9kZWxfY29uZmlnIjogdHJhaW5pbmcu'
        'bW9kZWwubW9kZWxfZHVtcChtb2RlPSJqc29uIiksCiAgICAgICAgICAgICJzb3VyY2VfY2hlY2tw'
        'b2ludF9zaGEyNTYiOiBfc2hhMjU2KGNoZWNrcG9pbnRfcGF0aCksCiAgICAgICAgICAgICJ0ZW5z'
        'b3JzIjogdGVuc29yX3BheWxvYWRzLAogICAgICAgIH0sCiAgICApCiAgICBzdGFydCA9IHRpbWUu'
        'cGVyZl9jb3VudGVyKCkKICAgIHF1YW50aXplZCA9IGxvYWRfcXVhbnRpemVkX21vZGVsKGFydGlm'
        'YWN0KQogICAgc3RhcnR1cF9tcyA9ICh0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnQpICogMV8w'
        'MDAKCiAgICBldmFsX2NvbmZpZyA9IHRyYWluaW5nLm1vZGVsX2NvcHkodXBkYXRlPXsicGFja19z'
        'ZXF1ZW5jZXMiOiBGYWxzZX0pCiAgICByb3dzID0gX2VuY29kZV9leGFtcGxlcyhleGFtcGxlcywg'
        'ZXZhbF9jb25maWcpCiAgICBzb3VyY2VfbG9zcyA9IGV2YWx1YXRlX2xvc3MoCiAgICAgICAgc291'
        'cmNlLCByb3dzLCBkZXZpY2U9dG9yY2guZGV2aWNlKCJjcHUiKSwgbm9ybWFsaXphdGlvbj10cmFp'
        'bmluZy5sb3NzX25vcm1hbGl6YXRpb24KICAgICkKICAgIHF1YW50aXplZF9sb3NzID0gZXZhbHVh'
        'dGVfbG9zcygKICAgICAgICBxdWFudGl6ZWQsIHJvd3MsIGRldmljZT10b3JjaC5kZXZpY2UoImNw'
        'dSIpLCBub3JtYWxpemF0aW9uPXRyYWluaW5nLmxvc3Nfbm9ybWFsaXphdGlvbgogICAgKQogICAg'
        'cHJvbXB0ID0gIkxpc3QgdGhlIHNhZmUgc3RlcHMgYmVmb3JlIHN0YXJ0aW5nIGEgbWVtb3J5LWhl'
        'YXZ5IGxvY2FsIG1vZGVsIHdvcmtsb2FkLiIKICAgIHNvdXJjZV9sYXRlbmN5ID0gX2xhdGVuY3ko'
        'c291cmNlLCBwcm9tcHQpCiAgICBxdWFudGl6ZWRfbGF0ZW5jeSA9IF9sYXRlbmN5KHF1YW50aXpl'
        'ZCwgcHJvbXB0KQogICAgdG9vbF9leGFtcGxlcyA9IFtpdGVtIGZvciBpdGVtIGluIGV4YW1wbGVz'
        'IGlmIGl0ZW0uY2F0ZWdvcnkgaW4geyJ0b29sX3VzZSIsICJhZ2VudF9iZWhhdmlvciJ9XQogICAg'
        'dG9vbF9leGFjdCA9IHN1bSgKICAgICAgICBnZW5lcmF0ZV90ZXh0KHF1YW50aXplZCwgaXRlbS5w'
        'cm9tcHQsIGRldmljZT10b3JjaC5kZXZpY2UoImNwdSIpLCBtYXhfbmV3X3Rva2Vucz00OCkKICAg'
        'ICAgICAuc3RyaXAoKQogICAgICAgIC5jYXNlZm9sZCgpCiAgICAgICAgPT0gaXRlbS5yZXNwb25z'
        'ZS5zdHJpcCgpLmNhc2Vmb2xkKCkKICAgICAgICBmb3IgaXRlbSBpbiB0b29sX2V4YW1wbGVzCiAg'
        'ICApIC8gbGVuKHRvb2xfZXhhbXBsZXMpCiAgICBjb250ZXh0X2xpbWl0X2VuZm9yY2VkID0gRmFs'
        'c2UKICAgIHRyeToKICAgICAgICBxdWFudGl6ZWQodG9yY2guemVyb3MoKDEsIHRyYWluaW5nLm1v'
        'ZGVsLm1heF9zZXF1ZW5jZV90b2tlbnMgKyAxKSwgZHR5cGU9dG9yY2gubG9uZykpCiAgICBleGNl'
        'cHQgVmFsdWVFcnJvcjoKICAgICAgICBjb250ZXh0X2xpbWl0X2VuZm9yY2VkID0gVHJ1ZQogICAg'
        'ZmxvYXRfc2l6ZSA9IGZsb2F0X3BhdGguc3RhdCgpLnN0X3NpemUKICAgIHF1YW50aXplZF9zaXpl'
        'ID0gYXJ0aWZhY3Quc3RhdCgpLnN0X3NpemUKICAgIGxvc3NfY2hhbmdlID0gKHF1YW50aXplZF9s'
        'b3NzIC0gc291cmNlX2xvc3MpIC8gbWF4KHNvdXJjZV9sb3NzLCAxZS0xMikKICAgIHJlcG9ydCA9'
        'IFF1YW50aXphdGlvblJlcG9ydCgKICAgICAgICBsb3NzX25vcm1hbGl6YXRpb249dHJhaW5pbmcu'
        'bG9zc19ub3JtYWxpemF0aW9uLAogICAgICAgIGZvcm1hdD1mIm9seW1wdXMtc3ltbWV0cmljLWlu'
        'dHtiaXRzfS12MSIsCiAgICAgICAgYml0cz1iaXRzLAogICAgICAgIHNvdXJjZV9jaGVja3BvaW50'
        'X3NoYTI1Nj1fc2hhMjU2KGNoZWNrcG9pbnRfcGF0aCksCiAgICAgICAgYXJ0aWZhY3RfcGF0aD1z'
        'dHIoYXJ0aWZhY3QucmVzb2x2ZSgpKSwKICAgICAgICBhcnRpZmFjdF9zaGEyNTY9X3NoYTI1Nihh'
        'cnRpZmFjdCksCiAgICAgICAgZmxvYXRfd2VpZ2h0c19ieXRlcz1mbG9hdF9zaXplLAogICAgICAg'
        'IHF1YW50aXplZF9ieXRlcz1xdWFudGl6ZWRfc2l6ZSwKICAgICAgICBzaXplX3JlZHVjdGlvbl9m'
        'cmFjdGlvbj0xLjAgLSBxdWFudGl6ZWRfc2l6ZSAvIGZsb2F0X3NpemUsCiAgICAgICAgc291cmNl'
        'X2xvc3M9c291cmNlX2xvc3MsCiAgICAgICAgcXVhbnRpemVkX2xvc3M9cXVhbnRpemVkX2xvc3Ms'
        'CiAgICAgICAgbG9zc19jaGFuZ2VfZnJhY3Rpb249bG9zc19jaGFuZ2UsCiAgICAgICAgc291cmNl'
        'X2xhdGVuY3lfbXM9c291cmNlX2xhdGVuY3ksCiAgICAgICAgcXVhbnRpemVkX2xhdGVuY3lfbXM9'
        'cXVhbnRpemVkX2xhdGVuY3ksCiAgICAgICAgc3RhcnR1cF9sYXRlbmN5X21zPXN0YXJ0dXBfbXMs'
        'CiAgICAgICAgbWF4X2NvbnRleHRfdG9rZW5zPXRyYWluaW5nLm1vZGVsLm1heF9zZXF1ZW5jZV90'
        'b2tlbnMsCiAgICAgICAgY29udGV4dF9saW1pdF9lbmZvcmNlZD1jb250ZXh0X2xpbWl0X2VuZm9y'
        'Y2VkLAogICAgICAgIHRvb2xfZXhhY3RfbWF0Y2hfcmF0ZT10b29sX2V4YWN0LAogICAgICAgIHBl'
        'YWtfcnNzX2J5dGVzPW1lbW9yeV9zbmFwc2hvdCgpLnByb2Nlc3NfcGVha19yc3NfYnl0ZXMsCiAg'
        'ICAgICAgcGFzc2VkX3F1YWxpdHlfZ2F0ZT0oYWJzKGxvc3NfY2hhbmdlKSA8PSAwLjAyIGFuZCBj'
        'b250ZXh0X2xpbWl0X2VuZm9yY2VkKSwKICAgICkKICAgIHJlcG9ydF9wYXRoID0gb3V0cHV0X3Jv'
        'b3QgLyBmImludHtiaXRzfS1yZXBvcnQuanNvbiIKICAgIF9hdG9taWNfanNvbihyZXBvcnRfcGF0'
        'aCwgcmVwb3J0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpKQogICAgcmV0dXJuIHJlcG9ydAo='
    ),
    'olympus/foundry/readiness.py': (
        '...'
        'ICAgKQogICAgKQo='
    ),
    'olympus/foundry/resources.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGZjbnRsCmltcG9ydCBv'
        'cwppbXBvcnQgcGxhdGZvcm0KaW1wb3J0IHJlCmltcG9ydCByZXNvdXJjZQppbXBvcnQgc3VicHJv'
        'Y2Vzcwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9y'
        'dCBQYXRoCmZyb20gdHlwaW5nIGltcG9ydCBJTywgU2VsZgoKCkBkYXRhY2xhc3MoZnJvemVuPVRy'
        'dWUsIHNsb3RzPVRydWUpCmNsYXNzIE1lbW9yeVNuYXBzaG90OgogICAgdG90YWxfYnl0ZXM6IGlu'
        'dAogICAgYXZhaWxhYmxlX2J5dGVzOiBpbnQKICAgIHN3YXBfdG90YWxfYnl0ZXM6IGludAogICAg'
        'c3dhcF91c2VkX2J5dGVzOiBpbnQKICAgIHByb2Nlc3NfcGVha19yc3NfYnl0ZXM6IGludAoKICAg'
        'IEBwcm9wZXJ0eQogICAgZGVmIHN3YXBfZnJhY3Rpb24oc2VsZikgLT4gZmxvYXQ6CiAgICAgICAg'
        'cmV0dXJuIDAuMCBpZiBzZWxmLnN3YXBfdG90YWxfYnl0ZXMgPT0gMCBlbHNlIHNlbGYuc3dhcF91'
        'c2VkX2J5dGVzIC8gc2VsZi5zd2FwX3RvdGFsX2J5dGVzCgogICAgZGVmIGFzX2RpY3Qoc2VsZikg'
        'LT4gZGljdFtzdHIsIGludCB8IGZsb2F0XToKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAi'
        'dG90YWxfYnl0ZXMiOiBzZWxmLnRvdGFsX2J5dGVzLAogICAgICAgICAgICAiYXZhaWxhYmxlX2J5'
        'dGVzIjogc2VsZi5hdmFpbGFibGVfYnl0ZXMsCiAgICAgICAgICAgICJzd2FwX3RvdGFsX2J5dGVz'
        'Ijogc2VsZi5zd2FwX3RvdGFsX2J5dGVzLAogICAgICAgICAgICAic3dhcF91c2VkX2J5dGVzIjog'
        'c2VsZi5zd2FwX3VzZWRfYnl0ZXMsCiAgICAgICAgICAgICJzd2FwX2ZyYWN0aW9uIjogc2VsZi5z'
        'd2FwX2ZyYWN0aW9uLAogICAgICAgICAgICAicHJvY2Vzc19wZWFrX3Jzc19ieXRlcyI6IHNlbGYu'
        'cHJvY2Vzc19wZWFrX3Jzc19ieXRlcywKICAgICAgICB9CgoKZGVmIF9ydW4oY29tbWFuZDogbGlz'
        'dFtzdHJdKSAtPiBzdHI6CiAgICB0cnk6CiAgICAgICAgcmVzdWx0ID0gc3VicHJvY2Vzcy5ydW4o'
        'Y29tbWFuZCwgY2hlY2s9VHJ1ZSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1l'
        'b3V0PTUpCiAgICBleGNlcHQgKE9TRXJyb3IsIHN1YnByb2Nlc3MuU3VicHJvY2Vzc0Vycm9yKSBh'
        'cyBlcnJvcjoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJyZXNvdXJjZSBpbnNwZWN0aW9u'
        'IGNvbW1hbmQgZmFpbGVkOiB7Y29tbWFuZFswXX0iKSBmcm9tIGVycm9yCiAgICByZXR1cm4gcmVz'
        'dWx0LnN0ZG91dAoKCmRlZiBfcGVha19yc3NfYnl0ZXMoKSAtPiBpbnQ6CiAgICB2YWx1ZSA9IHJl'
        'c291cmNlLmdldHJ1c2FnZShyZXNvdXJjZS5SVVNBR0VfU0VMRikucnVfbWF4cnNzCiAgICByZXR1'
        'cm4gaW50KHZhbHVlIGlmIHBsYXRmb3JtLnN5c3RlbSgpID09ICJEYXJ3aW4iIGVsc2UgdmFsdWUg'
        'KiAxMDI0KQoKCmRlZiBfbWFjb3NfbWVtb3J5KCkgLT4gTWVtb3J5U25hcHNob3Q6CiAgICB0b3Rh'
        'bCA9IGludChfcnVuKFsic3lzY3RsIiwgIi1uIiwgImh3Lm1lbXNpemUiXSkuc3RyaXAoKSkKICAg'
        'IHZtX291dHB1dCA9IF9ydW4oWyJ2bV9zdGF0Il0pCiAgICBwYWdlX3NpemVfbWF0Y2ggPSByZS5z'
        'ZWFyY2gociJwYWdlIHNpemUgb2YgKFxkKykgYnl0ZXMiLCB2bV9vdXRwdXQpCiAgICBpZiBwYWdl'
        'X3NpemVfbWF0Y2ggaXMgTm9uZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImNhbm5vdCBw'
        'YXJzZSBtYWNPUyBWTSBwYWdlIHNpemUiKQogICAgcGFnZV9zaXplID0gaW50KHBhZ2Vfc2l6ZV9t'
        'YXRjaC5ncm91cCgxKSkKICAgIHBhZ2VzOiBkaWN0W3N0ciwgaW50XSA9IHt9CiAgICBmb3IgbGlu'
        'ZSBpbiB2bV9vdXRwdXQuc3BsaXRsaW5lcygpOgogICAgICAgIG1hdGNoID0gcmUubWF0Y2gociIo'
        'W146XSspOlxzKyhcZCspXC4iLCBsaW5lKQogICAgICAgIGlmIG1hdGNoOgogICAgICAgICAgICBw'
        'YWdlc1ttYXRjaC5ncm91cCgxKV0gPSBpbnQobWF0Y2guZ3JvdXAoMikpCiAgICBhdmFpbGFibGVf'
        'cGFnZXMgPSBzdW0oCiAgICAgICAgcGFnZXMuZ2V0KG5hbWUsIDApCiAgICAgICAgZm9yIG5hbWUg'
        'aW4gKCJQYWdlcyBmcmVlIiwgIlBhZ2VzIGluYWN0aXZlIiwgIlBhZ2VzIHNwZWN1bGF0aXZlIiwg'
        'IlBhZ2VzIHB1cmdlYWJsZSIpCiAgICApCiAgICBzd2FwX291dHB1dCA9IF9ydW4oWyJzeXNjdGwi'
        'LCAidm0uc3dhcHVzYWdlIl0pCiAgICBzd2FwX21hdGNoID0gcmUuc2VhcmNoKHIidG90YWwgPSAo'
        'WzAtOS5dKylNXHMrdXNlZCA9IChbMC05Ll0rKU0iLCBzd2FwX291dHB1dCkKICAgIGlmIHN3YXBf'
        'bWF0Y2ggaXMgTm9uZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoImNhbm5vdCBwYXJzZSBt'
        'YWNPUyBzd2FwIHVzYWdlIikKICAgIHJldHVybiBNZW1vcnlTbmFwc2hvdCgKICAgICAgICB0b3Rh'
        'bF9ieXRlcz10b3RhbCwKICAgICAgICBhdmFpbGFibGVfYnl0ZXM9YXZhaWxhYmxlX3BhZ2VzICog'
        'cGFnZV9zaXplLAogICAgICAgIHN3YXBfdG90YWxfYnl0ZXM9aW50KGZsb2F0KHN3YXBfbWF0Y2gu'
        'Z3JvdXAoMSkpICogMTAyNCAqIDEwMjQpLAogICAgICAgIHN3YXBfdXNlZF9ieXRlcz1pbnQoZmxv'
        'YXQoc3dhcF9tYXRjaC5ncm91cCgyKSkgKiAxMDI0ICogMTAyNCksCiAgICAgICAgcHJvY2Vzc19w'
        'ZWFrX3Jzc19ieXRlcz1fcGVha19yc3NfYnl0ZXMoKSwKICAgICkKCgpkZWYgX2xpbnV4X21lbW9y'
        'eSgpIC0+IE1lbW9yeVNuYXBzaG90OgogICAgdmFsdWVzOiBkaWN0W3N0ciwgaW50XSA9IHt9CiAg'
        'ICBmb3IgbGluZSBpbiBQYXRoKCIvcHJvYy9tZW1pbmZvIikucmVhZF90ZXh0KGVuY29kaW5nPSJ1'
        'dGYtOCIpLnNwbGl0bGluZXMoKToKICAgICAgICBrZXksIHJhdyA9IGxpbmUuc3BsaXQoIjoiLCAx'
        'KQogICAgICAgIHZhbHVlc1trZXldID0gaW50KHJhdy5zdHJpcCgpLnNwbGl0KClbMF0pICogMTAy'
        'NAogICAgcmV0dXJuIE1lbW9yeVNuYXBzaG90KAogICAgICAgIHRvdGFsX2J5dGVzPXZhbHVlc1si'
        'TWVtVG90YWwiXSwKICAgICAgICBhdmFpbGFibGVfYnl0ZXM9dmFsdWVzWyJNZW1BdmFpbGFibGUi'
        'XSwKICAgICAgICBzd2FwX3RvdGFsX2J5dGVzPXZhbHVlc1siU3dhcFRvdGFsIl0sCiAgICAgICAg'
        'c3dhcF91c2VkX2J5dGVzPXZhbHVlc1siU3dhcFRvdGFsIl0gLSB2YWx1ZXNbIlN3YXBGcmVlIl0s'
        'CiAgICAgICAgcHJvY2Vzc19wZWFrX3Jzc19ieXRlcz1fcGVha19yc3NfYnl0ZXMoKSwKICAgICkK'
        'CgpkZWYgbWVtb3J5X3NuYXBzaG90KCkgLT4gTWVtb3J5U25hcHNob3Q6CiAgICBzeXN0ZW0gPSBw'
        'bGF0Zm9ybS5zeXN0ZW0oKQogICAgaWYgc3lzdGVtID09ICJEYXJ3aW4iOgogICAgICAgIHJldHVy'
        'biBfbWFjb3NfbWVtb3J5KCkKICAgIGlmIHN5c3RlbSA9PSAiTGludXgiOgogICAgICAgIHJldHVy'
        'biBfbGludXhfbWVtb3J5KCkKICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmInJlc291cmNlIGdvdmVy'
        'bm9yIGRvZXMgbm90IHN1cHBvcnQge3N5c3RlbX0iKQoKCmNsYXNzIFJlc291cmNlR292ZXJub3I6'
        'CiAgICAiIiJFeGNsdXNpdmUgbG9jYWwgd29ya2xvYWQgbG9jayBwbHVzIHByZWZsaWdodCBtZW1v'
        'cnkgYW5kIHN3YXAgZW5mb3JjZW1lbnQuIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNl'
        'bGYsCiAgICAgICAgbG9ja19wYXRoOiBQYXRoLAogICAgICAgICosCiAgICAgICAgbWluX2F2YWls'
        'YWJsZV9ieXRlczogaW50ID0gMiAqIDEwMjQqKjMsCiAgICAgICAgbWF4X3N3YXBfZnJhY3Rpb246'
        'IGZsb2F0ID0gMC45MCwKICAgICkgLT4gTm9uZToKICAgICAgICBpZiBtaW5fYXZhaWxhYmxlX2J5'
        'dGVzIDwgMjU2ICogMTAyNCoqMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibWluX2F2'
        'YWlsYWJsZV9ieXRlcyBtdXN0IGJlIGF0IGxlYXN0IDI1NiBNaUIiKQogICAgICAgIGlmIG5vdCAw'
        'LjAgPCBtYXhfc3dhcF9mcmFjdGlvbiA8PSAxLjA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJy'
        'b3IoIm1heF9zd2FwX2ZyYWN0aW9uIG11c3QgYmUgaW4gKDAsIDFdIikKICAgICAgICBzZWxmLmxv'
        'Y2tfcGF0aCA9IGxvY2tfcGF0aC5yZXNvbHZlKCkKICAgICAgICBzZWxmLm1pbl9hdmFpbGFibGVf'
        'Ynl0ZXMgPSBtaW5fYXZhaWxhYmxlX2J5dGVzCiAgICAgICAgc2VsZi5tYXhfc3dhcF9mcmFjdGlv'
        'biA9IG1heF9zd2FwX2ZyYWN0aW9uCiAgICAgICAgc2VsZi5faGFuZGxlOiBJT1tzdHJdIHwgTm9u'
        'ZSA9IE5vbmUKICAgICAgICBzZWxmLnByZWZsaWdodDogTWVtb3J5U25hcHNob3QgfCBOb25lID0g'
        'Tm9uZQoKICAgIGRlZiBhY3F1aXJlKHNlbGYpIC0+IE1lbW9yeVNuYXBzaG90OgogICAgICAgIHNl'
        'bGYubG9ja19wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAg'
        'ICAgICAgaGFuZGxlID0gc2VsZi5sb2NrX3BhdGgub3BlbigiYSsiLCBlbmNvZGluZz0idXRmLTgi'
        'KQogICAgICAgIHRyeToKICAgICAgICAgICAgZmNudGwuZmxvY2soaGFuZGxlLmZpbGVubygpLCBm'
        'Y250bC5MT0NLX0VYIHwgZmNudGwuTE9DS19OQikKICAgICAgICBleGNlcHQgQmxvY2tpbmdJT0Vy'
        'cm9yIGFzIGVycm9yOgogICAgICAgICAgICBoYW5kbGUuY2xvc2UoKQogICAgICAgICAgICByYWlz'
        'ZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAgICAgICAiYW5vdGhlciBtb2RlbC1sb2FkaW5nIG9y'
        'IHRyYWluaW5nIHdvcmtsb2FkIGhvbGRzIHRoZSBsb2NrIgogICAgICAgICAgICApIGZyb20gZXJy'
        'b3IKCiAgICAgICAgdHJ5OgogICAgICAgICAgICBzbmFwc2hvdCA9IG1lbW9yeV9zbmFwc2hvdCgp'
        'CiAgICAgICAgICAgIGlmIHNuYXBzaG90LmF2YWlsYWJsZV9ieXRlcyA8IHNlbGYubWluX2F2YWls'
        'YWJsZV9ieXRlczoKICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAg'
        'ICAgICAgICAgICBmImF2YWlsYWJsZSBtZW1vcnkge3NuYXBzaG90LmF2YWlsYWJsZV9ieXRlc30g'
        'aXMgYmVsb3cgdGhlICIKICAgICAgICAgICAgICAgICAgICBmIntzZWxmLm1pbl9hdmFpbGFibGVf'
        'Ynl0ZXN9LWJ5dGUgc2FmZXR5IGZsb29yIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBp'
        'ZiBzbmFwc2hvdC5zd2FwX2ZyYWN0aW9uID4gc2VsZi5tYXhfc3dhcF9mcmFjdGlvbjoKICAgICAg'
        'ICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigKICAgICAgICAgICAgICAgICAgICBmInN3YXAg'
        'dXRpbGl6YXRpb24ge3NuYXBzaG90LnN3YXBfZnJhY3Rpb246LjElfSBleGNlZWRzIHRoZSAiCiAg'
        'ICAgICAgICAgICAgICAgICAgZiJ7c2VsZi5tYXhfc3dhcF9mcmFjdGlvbjouMSV9IHNhZmV0eSBj'
        'ZWlsaW5nIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBoYW5kbGUuc2VlaygwKQogICAg'
        'ICAgICAgICBoYW5kbGUudHJ1bmNhdGUoKQogICAgICAgICAgICBoYW5kbGUud3JpdGUoZiJwaWQ9'
        'e29zLmdldHBpZCgpfVxuIikKICAgICAgICAgICAgaGFuZGxlLmZsdXNoKCkKICAgICAgICAgICAg'
        'b3MuZnN5bmMoaGFuZGxlLmZpbGVubygpKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAg'
        'ICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGZjbnRsLmZsb2NrKGhhbmRsZS5maWxlbm8oKSwg'
        'ZmNudGwuTE9DS19VTikKICAgICAgICAgICAgZmluYWxseToKICAgICAgICAgICAgICAgIGhhbmRs'
        'ZS5jbG9zZSgpCiAgICAgICAgICAgIHJhaXNlCgogICAgICAgIHNlbGYuX2hhbmRsZSA9IGhhbmRs'
        'ZQogICAgICAgIHNlbGYucHJlZmxpZ2h0ID0gc25hcHNob3QKICAgICAgICByZXR1cm4gc25hcHNo'
        'b3QKCiAgICBkZWYgcmVsZWFzZShzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYuX2hhbmRs'
        'ZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgZmNudGwuZmxvY2soc2VsZi5faGFuZGxlLmZpbGVu'
        'bygpLCBmY250bC5MT0NLX1VOKQogICAgICAgICAgICBzZWxmLl9oYW5kbGUuY2xvc2UoKQogICAg'
        'ICAgICAgICBzZWxmLl9oYW5kbGUgPSBOb25lCgogICAgZGVmIF9fZW50ZXJfXyhzZWxmKSAtPiBT'
        'ZWxmOgogICAgICAgIHNlbGYuYWNxdWlyZSgpCiAgICAgICAgcmV0dXJuIHNlbGYKCiAgICBkZWYg'
        'X19leGl0X18oc2VsZiwgKl86IG9iamVjdCkgLT4gTm9uZToKICAgICAgICBzZWxmLnJlbGVhc2Uo'
        'KQo='
    ),
    'olympus/foundry/schemas.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBlbnVtIGltcG9ydCBTdHJF'
        'bnVtCmZyb20gdHlwaW5nIGltcG9ydCBBbnksIExpdGVyYWwKCmZyb20gcHlkYW50aWMgaW1wb3J0'
        'IEZpZWxkLCBtb2RlbF92YWxpZGF0b3IKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0'
        'IFN0cmljdE1vZGVsCgoKY2xhc3MgQXJ0aWZhY3RTdGF0dXMoU3RyRW51bSk6CiAgICBERVNJR05F'
        'RCA9ICJERVNJR05FRCIKICAgIFJVTk5JTkdfRVhQRVJJTUVOVCA9ICJSVU5OSU5HX0VYUEVSSU1F'
        'TlQiCiAgICBDSEVDS1BPSU5URUQgPSAiQ0hFQ0tQT0lOVEVEIgogICAgRVZBTFVBVEVEID0gIkVW'
        'QUxVQVRFRCIKICAgIFZFUklGSUVEID0gIlZFUklGSUVEIgogICAgTkVHQVRJVkVfUkVTVUxUID0g'
        'Ik5FR0FUSVZFX1JFU1VMVCIKICAgIElOQ09OQ0xVU0lWRSA9ICJJTkNPTkNMVVNJVkUiCiAgICBG'
        'QUlMRUQgPSAiRkFJTEVEIgoKCmNsYXNzIERhdGFzZXRSZWNvcmQoU3RyaWN0TW9kZWwpOgogICAg'
        'ZGF0YXNldF9pZDogc3RyID0gRmllbGQocGF0dGVybj1yIl5bYS16MC05XVthLXowLTkuXy1dezEs'
        'NjN9JCIpCiAgICB2ZXJzaW9uOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9'
        'NjQpCiAgICBzaGEyNTY6IHN0ciA9IEZpZWxkKHBhdHRlcm49ciJeWzAtOWEtZl17NjR9JCIpCiAg'
        'ICBtYXRlcmlhbGl6ZWRfcGF0aDogc3RyID0gRmllbGQobWluX2xlbmd0aD0xKQogICAgc291cmNl'
        'OiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9Ml8wMDApCiAgICBvd25lcjog'
        'c3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTIwMCkKICAgIGxpY2Vuc2U6IHN0'
        'ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSwgbWF4X2xlbmd0aD0yMDApCiAgICBwcm92ZW5hbmNlOiBz'
        'dHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9NF8wMDApCiAgICBwcml2YWN5X2Ns'
        'YXNzaWZpY2F0aW9uOiBMaXRlcmFsWyJwdWJsaWMiLCAiaW50ZXJuYWwiLCAicHJpdmF0ZSIsICJy'
        'ZXN0cmljdGVkIl0KICAgIHN5bnRoZXRpYzogYm9vbAogICAgZ2VuZXJhdG9yOiBzdHIgfCBOb25l'
        'ID0gRmllbGQoZGVmYXVsdD1Ob25lLCBtYXhfbGVuZ3RoPTUwMCkKICAgIGJ5dGVfY291bnQ6IGlu'
        'dCA9IEZpZWxkKGd0PTApCiAgICBjaGFyYWN0ZXJfY291bnQ6IGludCA9IEZpZWxkKGd0PTApCiAg'
        'ICBjcmVhdGVkX2F0OiBzdHIKCiAgICBAbW9kZWxfdmFsaWRhdG9yKG1vZGU9ImFmdGVyIikKICAg'
        'IGRlZiByZXF1aXJlX3N5bnRoZXRpY19nZW5lcmF0b3Ioc2VsZikgLT4gRGF0YXNldFJlY29yZDoK'
        'ICAgICAgICBpZiBzZWxmLnN5bnRoZXRpYyBhbmQgbm90IHNlbGYuZ2VuZXJhdG9yOgogICAgICAg'
        'ICAgICByYWlzZSBWYWx1ZUVycm9yKCJzeW50aGV0aWMgZGF0YXNldHMgbXVzdCBpZGVudGlmeSB0'
        'aGVpciBnZW5lcmF0b3IiKQogICAgICAgIHJldHVybiBzZWxmCgoKY2xhc3MgRXhwZXJpbWVudFJl'
        'Y29yZChTdHJpY3RNb2RlbCk6CiAgICBleHBlcmltZW50X2lkOiBzdHIgPSBGaWVsZChwYXR0ZXJu'
        'PXIiXltBLVpdW0EtWjAtOV8tXXsyLDYzfSQiKQogICAgaHlwb3RoZXNpczogc3RyID0gRmllbGQo'
        'bWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTRfMDAwKQogICAgbW9kZWxfZmFtaWx5OiBzdHIgPSBG'
        'aWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9MTAwKQogICAgZGF0YXNldF9pZDogc3RyCiAg'
        'ICBkYXRhc2V0X3ZlcnNpb246IHN0cgogICAgZGF0YXNldF9zaGEyNTY6IHN0ciA9IEZpZWxkKHBh'
        'dHRlcm49ciJeWzAtOWEtZl17NjR9JCIpCiAgICBzZWVkOiBpbnQgPSBGaWVsZChnZT0wLCBsZT0y'
        'KiozMiAtIDEpCiAgICBjb25maWc6IGRpY3Rbc3RyLCBBbnldCiAgICBjb2RlX2NvbW1pdDogc3Ry'
        'ID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTY0KQogICAgaGFyZHdhcmU6IGRpY3Rb'
        'c3RyLCBBbnldCiAgICBzdGF0dXM6IEFydGlmYWN0U3RhdHVzCiAgICBmYWlsdXJlX3JlYXNvbjog'
        'c3RyIHwgTm9uZSA9IE5vbmUKICAgIGNyZWF0ZWRfYXQ6IHN0cgogICAgdXBkYXRlZF9hdDogc3Ry'
        'CgoKY2xhc3MgQ2hlY2twb2ludFJlY29yZChTdHJpY3RNb2RlbCk6CiAgICBjaGVja3BvaW50X2lk'
        'OiBzdHIgPSBGaWVsZChwYXR0ZXJuPXIiXmNrcHRfWzAtOWEtZl17MTZ9JCIpCiAgICBleHBlcmlt'
        'ZW50X2lkOiBzdHIKICAgIHNoYTI1Njogc3RyID0gRmllbGQocGF0dGVybj1yIl5bMC05YS1mXXs2'
        'NH0kIikKICAgIHBhdGg6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSkKICAgIGZvcm1hdDogc3Ry'
        'ID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTEwMCkKICAgIGJ5dGVfY291bnQ6IGlu'
        'dCA9IEZpZWxkKGd0PTApCiAgICBtZXRyaWNzOiBkaWN0W3N0ciwgZmxvYXRdCiAgICBjcmVhdGVk'
        'X2F0OiBzdHIKCgpjbGFzcyBFdmFsdWF0aW9uUmVjb3JkKFN0cmljdE1vZGVsKToKICAgIGV2YWx1'
        'YXRpb25faWQ6IHN0ciA9IEZpZWxkKHBhdHRlcm49ciJeZXZhbF9bMC05YS1mXXsxNn0kIikKICAg'
        'IGNoZWNrcG9pbnRfaWQ6IHN0cgogICAgc3VpdGU6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSwg'
        'bWF4X2xlbmd0aD0yMDApCiAgICBkYXRhc2V0X3NoYTI1Njogc3RyID0gRmllbGQocGF0dGVybj1y'
        'Il5bMC05YS1mXXs2NH0kIikKICAgIGJhc2VsaW5lX21ldHJpY3M6IGRpY3Rbc3RyLCBmbG9hdF0K'
        'ICAgIGNhbmRpZGF0ZV9tZXRyaWNzOiBkaWN0W3N0ciwgZmxvYXRdCiAgICBwYXNzZWQ6IGJvb2wK'
        'ICAgIGRlY2lzaW9uOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9Ml8wMDAp'
        'CiAgICBjcmVhdGVkX2F0OiBzdHIKCgpjbGFzcyBNb2RlbFJlY29yZChTdHJpY3RNb2RlbCk6CiAg'
        'ICBtb2RlbF9pZDogc3RyID0gRmllbGQocGF0dGVybj1yIl5bYS16MC05XVthLXowLTkuXy1dezEs'
        'MTI3fSQiKQogICAgZmFtaWx5OiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9'
        'MTAwKQogICAgdmVyc2lvbjogc3RyID0gRmllbGQobWluX2xlbmd0aD0xLCBtYXhfbGVuZ3RoPTY0'
        'KQogICAgY2hlY2twb2ludF9pZDogc3RyCiAgICBzdGF0dXM6IEFydGlmYWN0U3RhdHVzCiAgICBy'
        'dW50aW1lOiBzdHIgPSBGaWVsZChtaW5fbGVuZ3RoPTEsIG1heF9sZW5ndGg9MTAwKQogICAgY2Fw'
        'YWJpbGl0aWVzOiBsaXN0W3N0cl0KICAgIGludGVuZGVkX3VzZTogc3RyID0gRmllbGQobWluX2xl'
        'bmd0aD0xLCBtYXhfbGVuZ3RoPTJfMDAwKQogICAgbGltaXRhdGlvbnM6IGxpc3Rbc3RyXQogICAg'
        'Y3JlYXRlZF9hdDogc3RyCgoKY2xhc3MgRXZpZGVuY2VSZWNvcmQoU3RyaWN0TW9kZWwpOgogICAg'
        'c2VxdWVuY2U6IGludCA9IEZpZWxkKGd0PTApCiAgICB0aW1lc3RhbXA6IHN0cgogICAgZW50aXR5'
        'X3R5cGU6IHN0cgogICAgZW50aXR5X2lkOiBzdHIKICAgIGFjdGlvbjogc3RyCiAgICBwYXlsb2Fk'
        'OiBkaWN0W3N0ciwgQW55XQoKCmNsYXNzIEZvdW5kcnlQaXBlbGluZVJlc3VsdChTdHJpY3RNb2Rl'
        'bCk6CiAgICBkYXRhc2V0OiBEYXRhc2V0UmVjb3JkCiAgICBleHBlcmltZW50OiBFeHBlcmltZW50'
        'UmVjb3JkCiAgICBjaGVja3BvaW50OiBDaGVja3BvaW50UmVjb3JkCiAgICBldmFsdWF0aW9uOiBF'
        'dmFsdWF0aW9uUmVjb3JkCiAgICBtb2RlbDogTW9kZWxSZWNvcmQKICAgIGV4cG9ydF9wYXRoOiBz'
        'dHIKCgpjbGFzcyBHZW5lcmF0aW9uUmVzdWx0KFN0cmljdE1vZGVsKToKICAgIG1vZGVsOiBzdHIK'
        'ICAgIGNvbnRlbnQ6IHN0cgogICAgZmluaXNoX3JlYXNvbjogTGl0ZXJhbFsic3RvcCIsICJsZW5n'
        'dGgiXQogICAgcHJvbXB0X2NoYXJhY3RlcnM6IGludCA9IEZpZWxkKGdlPTApCiAgICBnZW5lcmF0'
        'ZWRfY2hhcmFjdGVyczogaW50ID0gRmllbGQoZ2U9MCkKICAgIGV2aWRlbmNlOiBkaWN0W3N0ciwg'
        'QW55XQo='
    ),
    'olympus/foundry/service.py': (
        '...'
    ),
    'olympus/foundry/sft.py': (
        '...'
        'ICAgICAgICAgKQo='
    ),
    'olympus/foundry/store.py': (
        '...'
        'Tm9uZToKICAgICAgICBzZWxmLmNsb3NlKCkK'
    ),
    'olympus/labos/__init__.py': (
        'IiIiTGFiT1MgcG9ydGZvbGlvIG9yY2hlc3RyYXRpb24uIiIiCgpmcm9tIG9seW1wdXMubGFib3Mu'
        'ZGlzY292ZXJ5IGltcG9ydCBkaXNjb3Zlcl9wcm9qZWN0cwpmcm9tIG9seW1wdXMubGFib3MubWFu'
        'aWZlc3QgaW1wb3J0IFByb2plY3RNYW5pZmVzdApmcm9tIG9seW1wdXMubGFib3MucG9ydGZvbGlv'
        'IGltcG9ydCBQb3J0Zm9saW9TZXJ2aWNlCmZyb20gb2x5bXB1cy5sYWJvcy5yZXBvcnRzIGltcG9y'
        'dCBQb3J0Zm9saW9SZXBvcnRlcgpmcm9tIG9seW1wdXMubGFib3MucnVubmVyIGltcG9ydCBQb3J0'
        'Zm9saW9SdW5uZXIKZnJvbSBvbHltcHVzLmxhYm9zLnZhbGlkYXRpb24gaW1wb3J0IHZhbGlkYXRl'
        'X3Byb2plY3QKCl9fYWxsX18gPSBbCiAgICAiUG9ydGZvbGlvU2VydmljZSIsCiAgICAiUG9ydGZv'
        'bGlvUmVwb3J0ZXIiLAogICAgIlBvcnRmb2xpb1J1bm5lciIsCiAgICAiUHJvamVjdE1hbmlmZXN0'
        'IiwKICAgICJkaXNjb3Zlcl9wcm9qZWN0cyIsCiAgICAidmFsaWRhdGVfcHJvamVjdCIsCl0K'
    ),
    'olympus/labos/discovery.py': (
        '...'
        'ZXR1cm4gbmFtZQogICAgcmV0dXJuIE5vbmUK'
    ),
    'olympus/labos/manifest.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBlbnVtIGltcG9ydCBTdHJF'
        'bnVtCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55Cgpmcm9t'
        'IHB5ZGFudGljIGltcG9ydCBGaWVsZAoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBvcnQg'
        'U3RyaWN0TW9kZWwKCgpjbGFzcyBNYW5pZmVzdFNvdXJjZShTdHJFbnVtKToKICAgIERFQ0xBUkVE'
        'ID0gImRlY2xhcmVkIgogICAgSU5GRVJSRUQgPSAiaW5mZXJyZWQiCgoKY2xhc3MgUHJvamVjdExh'
        'bmd1YWdlKFN0ckVudW0pOgogICAgUFlUSE9OID0gInB5dGhvbiIKICAgIFRZUEVTQ1JJUFQgPSAi'
        'dHlwZXNjcmlwdCIKICAgIE1JWEVEID0gIm1peGVkIgogICAgVU5LTk9XTiA9ICJ1bmtub3duIgoK'
        'CmNsYXNzIFByb2plY3RTdGF0dXMoU3RyRW51bSk6CiAgICBDT01QTEVURSA9ICJjb21wbGV0ZSIK'
        'ICAgIFNDSUVOVElGSUNBTExZX1JVTk5BQkxFID0gInNjaWVudGlmaWNhbGx5X3J1bm5hYmxlIgog'
        'ICAgU01PS0VfVEVTVEVEID0gInNtb2tlX3Rlc3RlZCIKICAgIEZVTExfQkVOQ0hNQVJLX1BFTkRJ'
        'TkdfQ09NUFVURSA9ICJmdWxsX2JlbmNobWFya19wZW5kaW5nX2NvbXB1dGUiCiAgICBCTE9DS0VE'
        'X0JZX0VYVEVSTkFMX0RBVEFTRVQgPSAiYmxvY2tlZF9ieV9leHRlcm5hbF9kYXRhc2V0IgogICAg'
        'QkxPQ0tFRF9CWV9DUkVERU5USUFMID0gImJsb2NrZWRfYnlfY3JlZGVudGlhbCIKICAgIEJMT0NL'
        'RURfQllfUEVSTUlTU0lPTiA9ICJibG9ja2VkX2J5X3Blcm1pc3Npb24iCiAgICBESVNDT1ZFUkVE'
        'X09OTFkgPSAiZGlzY292ZXJlZF9vbmx5IgoKCmNsYXNzIFByb2plY3RDbGFzc2lmaWNhdGlvbihT'
        'dHJFbnVtKToKICAgIEZMQUdTSElQX0NPTVBBTlkgPSAiZmxhZ3NoaXBfY29tcGFueSIKICAgIEhJ'
        'R0hfUE9URU5USUFMX01WUCA9ICJoaWdoX3BvdGVudGlhbF9tdnAiCiAgICBVU0VGVUxfRkVBVFVS'
        'RSA9ICJ1c2VmdWxfZmVhdHVyZV9mb3JfYW5vdGhlcl9jb21wYW55IgogICAgUkVTRUFSQ0hfUFJP'
        'SkVDVCA9ICJyZXNlYXJjaF9wcm9qZWN0IgogICAgUkVRVUlSRVNfVkFMSURBVElPTiA9ICJyZXF1'
        'aXJlc192YWxpZGF0aW9uIgogICAgV0VBS19ESUZGRVJFTlRJQVRJT04gPSAid2Vha19kaWZmZXJl'
        'bnRpYXRpb24iCiAgICBNRVJHRV9XSVRIX0FOT1RIRVJfSURFQSA9ICJtZXJnZV93aXRoX2Fub3Ro'
        'ZXJfaWRlYSIKICAgIEFSQ0hJVkUgPSAiYXJjaGl2ZSIKCgpjbGFzcyBSZXNvdXJjZVByb2ZpbGUo'
        'U3RyRW51bSk6CiAgICBTTU9LRSA9ICJzbW9rZSIKICAgIERFVkVMT1BNRU5UID0gImRldmVsb3Bt'
        'ZW50IgogICAgQkVOQ0hNQVJLID0gImJlbmNobWFyayIKICAgIEZVTEwgPSAiZnVsbCIKCgpjbGFz'
        'cyBEYXRhc2V0RGVwZW5kZW5jeShTdHJpY3RNb2RlbCk6CiAgICBuYW1lOiBzdHIKICAgIHJlcXVp'
        'cmVkOiBib29sID0gVHJ1ZQogICAgbm90ZXM6IHN0ciA9ICIiCiAgICBibG9ja2VkX2J5X3VzZXJf'
        'YWN0aW9uOiBib29sID0gRmFsc2UKCgpjbGFzcyBFbnRyeVBvaW50KFN0cmljdE1vZGVsKToKICAg'
        'IG5hbWU6IHN0cgogICAgY29tbWFuZDogc3RyCiAgICBwcm9maWxlOiBSZXNvdXJjZVByb2ZpbGUK'
        'ICAgIGV4cGVjdGVkX291dHB1dHM6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1s'
        'aXN0KQogICAgZW52OiBkaWN0W3N0ciwgc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0'
        'KQogICAgdGltZW91dF9zZWNvbmRzOiBpbnQgPSAxMjAKCgpjbGFzcyBSZXNvdXJjZVJlcXVpcmVt'
        'ZW50cyhTdHJpY3RNb2RlbCk6CiAgICBwcm9maWxlOiBSZXNvdXJjZVByb2ZpbGUgPSBSZXNvdXJj'
        'ZVByb2ZpbGUuU01PS0UKICAgIGNwdV9vbmx5OiBib29sID0gVHJ1ZQogICAgZ3B1X29wdGlvbmFs'
        'OiBib29sID0gRmFsc2UKICAgIG1lbW9yeV9nYjogaW50ID0gMgogICAgbm90ZXM6IHN0ciA9ICIi'
        'CgoKY2xhc3MgUHJvamVjdE1hbmlmZXN0KFN0cmljdE1vZGVsKToKICAgIHByb2plY3RfaWQ6IHN0'
        'cgogICAgdGl0bGU6IHN0cgogICAgZGVzY3JpcHRpb246IHN0cgogICAgaHlwb3RoZXNpczogc3Ry'
        'CiAgICByb290X3BhdGg6IHN0cgogICAgZG9tYWluOiBzdHIKICAgIGxhbmd1YWdlOiBQcm9qZWN0'
        'TGFuZ3VhZ2UgPSBQcm9qZWN0TGFuZ3VhZ2UuVU5LTk9XTgogICAgbWFuaWZlc3Rfc291cmNlOiBN'
        'YW5pZmVzdFNvdXJjZSA9IE1hbmlmZXN0U291cmNlLklORkVSUkVECiAgICBlbnRyeV9wb2ludHM6'
        'IGxpc3RbRW50cnlQb2ludF0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGRhdGFz'
        'ZXRzOiBsaXN0W0RhdGFzZXREZXBlbmRlbmN5XSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0'
        'KQogICAgZXhwZWN0ZWRfb3V0cHV0czogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5'
        'PWxpc3QpCiAgICB2YWxpZGF0aW9uX2NvbW1hbmRzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0'
        'X2ZhY3Rvcnk9bGlzdCkKICAgIHRhZ3M6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9y'
        'eT1saXN0KQogICAgY2xhc3NpZmljYXRpb246IFByb2plY3RDbGFzc2lmaWNhdGlvbiA9IFByb2pl'
        'Y3RDbGFzc2lmaWNhdGlvbi5SRVNFQVJDSF9QUk9KRUNUCiAgICB0YXJnZXRfdXNlcjogc3RyID0g'
        'IlJlc2VhcmNoIGVuZ2luZWVyIG9yIHRlY2huaWNhbCBmb3VuZGVyIGV2YWx1YXRpbmcgdGhlIHBy'
        'b2plY3QuIgogICAgY3VycmVudF9hbHRlcm5hdGl2ZTogc3RyID0gKAogICAgICAgICJNYW51YWwg'
        'bm90ZWJvb2tzLCBvbmUtb2ZmIHNjcmlwdHMsIG9yIHVuZ292ZXJuZWQgcHJvdG90eXBlIHJlcG9z'
        'aXRvcmllcy4iCiAgICApCiAgICBwcm9wb3NlZF9zb2x1dGlvbjogc3RyID0gKAogICAgICAgICJB'
        'IHJlcHJvZHVjaWJsZSBsb2NhbCBpbXBsZW1lbnRhdGlvbiB3aXRoIG1hbmlmZXN0cywgdGVzdHMs'
        'IHJlcG9ydHMsICIKICAgICAgICAiYW5kIGV4cGxpY2l0IHJlbGVhc2UgZ2F0ZXMuIgogICAgKQog'
        'ICAgZGlmZmVyZW50aWF0aW9uOiBzdHIgPSAoCiAgICAgICAgIkV2aWRlbmNlLWZpcnN0IGV4ZWN1'
        'dGlvbiB3aXRoIG1hY2hpbmUtcmVhZGFibGUgc3RhdHVzIGFuZCBibG9ja2VkLWFjdGlvbiB0cmFj'
        'a2luZy4iCiAgICApCiAgICBtYXJrZXQ6IHN0ciA9ICgKICAgICAgICAiU3BlY2lhbGl6ZWQgcmVz'
        'ZWFyY2gsIGRldmVsb3BlciwgYW5kIGZpbnRlY2gvTUwgaW5mcmFzdHJ1Y3R1cmUgbWFya2V0cyAi'
        'CiAgICAgICAgImRlcGVuZGluZyBvbiB0aGUgcHJvamVjdCBkb21haW4uIgogICAgKQogICAgYnVz'
        'aW5lc3NfbW9kZWw6IHN0ciA9ICgKICAgICAgICAiUmVxdWlyZXMgcHJvamVjdC1zcGVjaWZpYyB2'
        'YWxpZGF0aW9uIGJlZm9yZSBwcmljaW5nOyBsaWtlbHkgc3RhcnRzIGFzICIKICAgICAgICAiY29u'
        'c3VsdGluZywgU2FhUywgb3IgaW5mcmFzdHJ1Y3R1cmUgbGljZW5zaW5nLiIKICAgICkKICAgIGRp'
        'c3RyaWJ1dGlvbl9zdHJhdGVneTogc3RyID0gKAogICAgICAgICJUZWNobmljYWwgY29udGVudCwg'
        'Zm91bmRlci1sZWQgZGVzaWduIHBhcnRuZXJzaGlwcywgb3BlbiBkZW1vcywgYW5kIHRhcmdldGVk'
        'IHBpbG90cy4iCiAgICApCiAgICB0ZWNobmljYWxfcmVxdWlyZW1lbnRzOiBsaXN0W3N0cl0gPSBG'
        'aWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGRhdGFfcmVxdWlyZW1lbnRzOiBsaXN0W3N0'
        'cl0gPSBGaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIHJlZ3VsYXRvcnlfY29uc2lkZXJh'
        'dGlvbnM6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgc2VjdXJp'
        'dHlfcmlza3M6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgY2Fw'
        'aXRhbF9pbnRlbnNpdHk6IHN0ciA9ICJsb3ctdG8tbWVkaXVtIGJlZm9yZSBmdWxsIGJlbmNobWFy'
        'ayBzY2FsZS11cCIKICAgIHRpbWVfdG9fbXZwOiBzdHIgPSAiMS00IHdlZWtzIGZvciBhIG5hcnJv'
        'dyBzbW9rZS10ZXN0ZWQgTVZQIHdoZW4gZGVwZW5kZW5jaWVzIGFyZSBsb2NhbCIKICAgIGV2aWRl'
        'bmNlX2F2YWlsYWJsZTogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAg'
        'ICBmYXRhbF9hc3N1bXB0aW9uczogbGlzdFtzdHJdID0gRmllbGQoZGVmYXVsdF9mYWN0b3J5PWxp'
        'c3QpCiAgICB2YWxpZGF0aW9uX2V4cGVyaW1lbnRzOiBsaXN0W3N0cl0gPSBGaWVsZChkZWZhdWx0'
        'X2ZhY3Rvcnk9bGlzdCkKICAgIGdvX25vX2dvX2NyaXRlcmlhOiBsaXN0W3N0cl0gPSBGaWVsZChk'
        'ZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGJlbmNobWFya19wbGFuOiBsaXN0W3N0cl0gPSBGaWVs'
        'ZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGFibGF0aW9uX3BsYW46IGxpc3Rbc3RyXSA9IEZp'
        'ZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQogICAgcmVsZWFzZV9nYXRlczogbGlzdFtzdHJdID0g'
        'RmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpCiAgICBzY2llbnRpZmljX2NhdmVhdDogc3RyID0g'
        'KAogICAgICAgICJTbW9rZSB0ZXN0cyBhbmQgc3ludGhldGljIGV4cGVyaW1lbnRzIGFyZSBlbmdp'
        'bmVlcmluZyBldmlkZW5jZSwgbm90IHByb29mIG9mICIKICAgICAgICAic2NpZW50aWZpYyBzdXBl'
        'cmlvcml0eSBvciByZWd1bGF0ZWQgcHJvZHVjdGlvbiByZWFkaW5lc3MuIgogICAgKQogICAgcmVz'
        'b3VyY2VfcmVxdWlyZW1lbnRzOiBSZXNvdXJjZVJlcXVpcmVtZW50cyA9IEZpZWxkKGRlZmF1bHRf'
        'ZmFjdG9yeT1SZXNvdXJjZVJlcXVpcmVtZW50cykKICAgIG1ldGFkYXRhOiBkaWN0W3N0ciwgQW55'
        'XSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHBh'
        'dGgoc2VsZikgLT4gUGF0aDoKICAgICAgICByZXR1cm4gUGF0aChzZWxmLnJvb3RfcGF0aCkKCiAg'
        'ICBkZWYgc21va2VfZW50cnlfcG9pbnRzKHNlbGYpIC0+IGxpc3RbRW50cnlQb2ludF06CiAgICAg'
        'ICAgcmV0dXJuIFtpdGVtIGZvciBpdGVtIGluIHNlbGYuZW50cnlfcG9pbnRzIGlmIGl0ZW0ucHJv'
        'ZmlsZSA9PSBSZXNvdXJjZVByb2ZpbGUuU01PS0VdCgoKY2xhc3MgVmFsaWRhdGlvbklzc3VlKFN0'
        'cmljdE1vZGVsKToKICAgIHNldmVyaXR5OiBzdHIKICAgIG1lc3NhZ2U6IHN0cgoKCmNsYXNzIFZh'
        'bGlkYXRpb25SZXN1bHQoU3RyaWN0TW9kZWwpOgogICAgcHJvamVjdF9pZDogc3RyCiAgICB2YWxp'
        'ZDogYm9vbAogICAgaXNzdWVzOiBsaXN0W1ZhbGlkYXRpb25Jc3N1ZV0gPSBGaWVsZChkZWZhdWx0'
        'X2ZhY3Rvcnk9bGlzdCkKCgpjbGFzcyBQcm9qZWN0UmVjb3JkKFN0cmljdE1vZGVsKToKICAgIG1h'
        'bmlmZXN0OiBQcm9qZWN0TWFuaWZlc3QKICAgIHZhbGlkYXRpb246IFZhbGlkYXRpb25SZXN1bHQK'
        'ICAgIHN0YXR1czogUHJvamVjdFN0YXR1cyA9IFByb2plY3RTdGF0dXMuRElTQ09WRVJFRF9PTkxZ'
        'CiAgICBzdW1tYXJ5OiBzdHIgPSAiIgogICAgYmxvY2tlZF9yZWFzb246IHN0ciA9ICIiCg=='
    ),
    'olympus/labos/portfolio.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQ'
        'YXRoCgpmcm9tIG9seW1wdXMubGFib3MuZGlzY292ZXJ5IGltcG9ydCBkaXNjb3Zlcl9wcm9qZWN0'
        'cwpmcm9tIG9seW1wdXMubGFib3MubWFuaWZlc3QgaW1wb3J0ICgKICAgIFByb2plY3RNYW5pZmVz'
        'dCwKICAgIFByb2plY3RSZWNvcmQsCiAgICBQcm9qZWN0U3RhdHVzLAogICAgUmVzb3VyY2VQcm9m'
        'aWxlLAopCmZyb20gb2x5bXB1cy5sYWJvcy5yZXBvcnRzIGltcG9ydCBQb3J0Zm9saW9SZXBvcnRl'
        'cgpmcm9tIG9seW1wdXMubGFib3MucnVubmVyIGltcG9ydCBQb3J0Zm9saW9SdW5uZXIKZnJvbSBv'
        'bHltcHVzLmxhYm9zLnZhbGlkYXRpb24gaW1wb3J0IHZhbGlkYXRlX3Byb2plY3QKCgpjbGFzcyBQ'
        'b3J0Zm9saW9TZXJ2aWNlOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHdvcmtzcGFjZV9yb290OiBQ'
        'YXRoLCBhcnRpZmFjdF9yb290OiBQYXRoKSAtPiBOb25lOgogICAgICAgIHNlbGYud29ya3NwYWNl'
        'X3Jvb3QgPSB3b3Jrc3BhY2Vfcm9vdAogICAgICAgIHNlbGYuYXJ0aWZhY3Rfcm9vdCA9IGFydGlm'
        'YWN0X3Jvb3QKICAgICAgICBzZWxmLnJ1bm5lciA9IFBvcnRmb2xpb1J1bm5lcihhcnRpZmFjdF9y'
        'b290KQogICAgICAgIHNlbGYucmVwb3J0ZXIgPSBQb3J0Zm9saW9SZXBvcnRlcihhcnRpZmFjdF9y'
        'b290LCB3b3Jrc3BhY2Vfcm9vdCkKCiAgICBkZWYgY2xvc2Uoc2VsZikgLT4gTm9uZToKICAgICAg'
        'ICBzZWxmLnJlcG9ydGVyLmNsb3NlKCkKICAgICAgICBzZWxmLnJ1bm5lci5jbG9zZSgpCgogICAg'
        'ZGVmIF9fZW50ZXJfXyhzZWxmKSAtPiBQb3J0Zm9saW9TZXJ2aWNlOgogICAgICAgIHJldHVybiBz'
        'ZWxmCgogICAgZGVmIF9fZXhpdF9fKHNlbGYsICphcmdzOiBvYmplY3QpIC0+IE5vbmU6CiAgICAg'
        'ICAgc2VsZi5jbG9zZSgpCgogICAgZGVmIGRpc2NvdmVyKHNlbGYpIC0+IGxpc3RbUHJvamVjdFJl'
        'Y29yZF06CiAgICAgICAgbWFuaWZlc3RzID0gZGlzY292ZXJfcHJvamVjdHMoc2VsZi53b3Jrc3Bh'
        'Y2Vfcm9vdCkKICAgICAgICByZWNvcmRzID0gW3NlbGYuX3JlY29yZF9mcm9tX21hbmlmZXN0KG1h'
        'bmlmZXN0KSBmb3IgbWFuaWZlc3QgaW4gbWFuaWZlc3RzXQogICAgICAgIHJldHVybiBzZWxmLl9h'
        'dHRhY2hfcnVuX2hpc3RvcnkocmVjb3JkcykKCiAgICBkZWYgdmFsaWRhdGUoc2VsZikgLT4gbGlz'
        'dFtQcm9qZWN0UmVjb3JkXToKICAgICAgICByZXR1cm4gc2VsZi5kaXNjb3ZlcigpCgogICAgZGVm'
        'IHJ1bl9zbW9rZSgKICAgICAgICBzZWxmLAogICAgICAgIHByb2plY3RfaWRzOiBzZXRbc3RyXSB8'
        'IE5vbmUgPSBOb25lLAogICAgICAgIHByb2ZpbGU6IFJlc291cmNlUHJvZmlsZSA9IFJlc291cmNl'
        'UHJvZmlsZS5TTU9LRSwKICAgICkgLT4gbGlzdFtQcm9qZWN0UmVjb3JkXToKICAgICAgICByZWNv'
        'cmRzID0gc2VsZi5kaXNjb3ZlcigpCiAgICAgICAgZm9yIHJlY29yZCBpbiByZWNvcmRzOgogICAg'
        'ICAgICAgICBpZiBwcm9qZWN0X2lkcyBpcyBub3QgTm9uZSBhbmQgcmVjb3JkLm1hbmlmZXN0LnBy'
        'b2plY3RfaWQgbm90IGluIHByb2plY3RfaWRzOgogICAgICAgICAgICAgICAgY29udGludWUKICAg'
        'ICAgICAgICAgcmVzdWx0cyA9IHNlbGYucnVubmVyLnJ1bl9tYW5pZmVzdChyZWNvcmQubWFuaWZl'
        'c3QsIHByb2ZpbGU9cHJvZmlsZSkKICAgICAgICAgICAgaWYgcmVzdWx0czoKICAgICAgICAgICAg'
        'ICAgIHJlY29yZC5zdGF0dXMgPSByZXN1bHRzWy0xXS5zdGF0dXMKICAgICAgICAgICAgICAgIGlm'
        'ICgKICAgICAgICAgICAgICAgICAgICByZWNvcmQuc3RhdHVzID09IFByb2plY3RTdGF0dXMuU01P'
        'S0VfVEVTVEVECiAgICAgICAgICAgICAgICAgICAgYW5kIGFueShkYXRhc2V0LmJsb2NrZWRfYnlf'
        'dXNlcl9hY3Rpb24gZm9yIGRhdGFzZXQgaW4gcmVjb3JkLm1hbmlmZXN0LmRhdGFzZXRzKQogICAg'
        'ICAgICAgICAgICAgKToKICAgICAgICAgICAgICAgICAgICByZWNvcmQuc3RhdHVzID0gUHJvamVj'
        'dFN0YXR1cy5GVUxMX0JFTkNITUFSS19QRU5ESU5HX0NPTVBVVEUKICAgICAgICAgICAgICAgIHJl'
        'Y29yZC5zdW1tYXJ5ID0gKAogICAgICAgICAgICAgICAgICAgIGYiRXhlY3V0ZWQge2xlbihyZXN1'
        'bHRzKX0ge3Byb2ZpbGUudmFsdWV9IGNvbW1hbmQocyk7ICIKICAgICAgICAgICAgICAgICAgICBm'
        'Imxhc3QgY2xhc3NpZmljYXRpb249e3Jlc3VsdHNbLTFdLmNsYXNzaWZpY2F0aW9ufS4iCiAgICAg'
        'ICAgICAgICAgICApCiAgICAgICAgICAgICAgICBpZiByZXN1bHRzWy0xXS5jbGFzc2lmaWNhdGlv'
        'biAhPSAib2siOgogICAgICAgICAgICAgICAgICAgIHJlY29yZC5ibG9ja2VkX3JlYXNvbiA9IF9k'
        'ZXNjcmliZV9ydW5fZmFpbHVyZSgKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0c1stMV0u'
        'Y2xhc3NpZmljYXRpb24sCiAgICAgICAgICAgICAgICAgICAgICAgIHJlc3VsdHNbLTFdLnN0ZG91'
        'dCwKICAgICAgICAgICAgICAgICAgICAgICAgcmVzdWx0c1stMV0uc3RkZXJyLAogICAgICAgICAg'
        'ICAgICAgICAgICkKICAgICAgICAgICAgICAgIGVsaWYgYW55KAogICAgICAgICAgICAgICAgICAg'
        'IGRhdGFzZXQuYmxvY2tlZF9ieV91c2VyX2FjdGlvbiBmb3IgZGF0YXNldCBpbiByZWNvcmQubWFu'
        'aWZlc3QuZGF0YXNldHMKICAgICAgICAgICAgICAgICk6CiAgICAgICAgICAgICAgICAgICAgcmVj'
        'b3JkLmJsb2NrZWRfcmVhc29uID0gKAogICAgICAgICAgICAgICAgICAgICAgICAiT25lIG9yIG1v'
        'cmUgZGF0YXNldHMgcmVxdWlyZSBtYW51YWwgZG93bmxvYWQgb3IgYWNjZXB0YW5jZS4iCiAgICAg'
        'ICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAg'
        'ICByZWNvcmQuYmxvY2tlZF9yZWFzb24gPSAiIgogICAgICAgICAgICBlbHNlOgogICAgICAgICAg'
        'ICAgICAgcmVjb3JkLnN1bW1hcnkgPSAiTm8gbWF0Y2hpbmcgZW50cnkgcG9pbnRzIGZvciB0aGUg'
        'cmVxdWVzdGVkIHByb2ZpbGUuIgogICAgICAgIHJldHVybiByZWNvcmRzCgogICAgZGVmIHdyaXRl'
        'X3JlcG9ydHMoCiAgICAgICAgc2VsZiwKICAgICAgICByZWNvcmRzOiBsaXN0W1Byb2plY3RSZWNv'
        'cmRdLAogICAgICAgIG91dHB1dF9yb290OiBQYXRoLAogICAgKSAtPiBOb25lOgogICAgICAgIG91'
        'dHB1dF9yb290Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBzZWxm'
        'LnJlcG9ydGVyLndyaXRlX3N0YXR1c19qc29uKHJlY29yZHMsIG91dHB1dF9yb290IC8gInBvcnRm'
        'b2xpb19zdGF0dXMuanNvbiIpCiAgICAgICAgc2VsZi5yZXBvcnRlci53cml0ZV9jb21wbGV0aW9u'
        'X3JlcG9ydCgKICAgICAgICAgICAgcmVjb3JkcywKICAgICAgICAgICAgb3V0cHV0X3Jvb3QgLyAi'
        'UE9SVEZPTElPX0NPTVBMRVRJT05fUkVQT1JULm1kIiwKICAgICAgICApCiAgICAgICAgc2VsZi5y'
        'ZXBvcnRlci53cml0ZV9ydW5zX2pzb24ob3V0cHV0X3Jvb3QgLyAicG9ydGZvbGlvX3J1bnMuanNv'
        'biIpCiAgICAgICAgc2VsZi5yZXBvcnRlci53cml0ZV9zY2llbnRpZmljX3ZhbGlkYXRpb25fcmVw'
        'b3J0KAogICAgICAgICAgICByZWNvcmRzLAogICAgICAgICAgICBvdXRwdXRfcm9vdCAvICJTQ0lF'
        'TlRJRklDX1ZBTElEQVRJT05fUkVQT1JULm1kIiwKICAgICAgICApCiAgICAgICAgc2VsZi5yZXBv'
        'cnRlci53cml0ZV9yZXByb2R1Y2liaWxpdHlfcmVwb3J0KAogICAgICAgICAgICByZWNvcmRzLAog'
        'ICAgICAgICAgICBvdXRwdXRfcm9vdCAvICJSRVBST0RVQ0lCSUxJVFkubWQiLAogICAgICAgICkK'
        'ICAgICAgICBzZWxmLnJlcG9ydGVyLndyaXRlX3JlbWFpbmluZ19hY3Rpb25zKAogICAgICAgICAg'
        'ICByZWNvcmRzLAogICAgICAgICAgICBvdXRwdXRfcm9vdCAvICJSRU1BSU5JTkdfRVhURVJOQUxf'
        'QUNUSU9OUy5tZCIsCiAgICAgICAgKQoKICAgIGRlZiBfcmVjb3JkX2Zyb21fbWFuaWZlc3Qoc2Vs'
        'ZiwgbWFuaWZlc3Q6IFByb2plY3RNYW5pZmVzdCkgLT4gUHJvamVjdFJlY29yZDoKICAgICAgICB2'
        'YWxpZGF0aW9uID0gdmFsaWRhdGVfcHJvamVjdChtYW5pZmVzdCkKICAgICAgICBzdGF0dXMgPSBQ'
        'cm9qZWN0U3RhdHVzLkRJU0NPVkVSRURfT05MWQogICAgICAgIGJsb2NrZWRfcmVhc29uID0gIiIK'
        'ICAgICAgICBpZiBtYW5pZmVzdC5tYW5pZmVzdF9zb3VyY2UudmFsdWUgPT0gImRlY2xhcmVkIiBh'
        'bmQgdmFsaWRhdGlvbi52YWxpZDoKICAgICAgICAgICAgc3RhdHVzID0gUHJvamVjdFN0YXR1cy5T'
        'Q0lFTlRJRklDQUxMWV9SVU5OQUJMRQogICAgICAgIGlmIGFueShkYXRhc2V0LmJsb2NrZWRfYnlf'
        'dXNlcl9hY3Rpb24gZm9yIGRhdGFzZXQgaW4gbWFuaWZlc3QuZGF0YXNldHMpOgogICAgICAgICAg'
        'ICBibG9ja2VkX3JlYXNvbiA9ICJPbmUgb3IgbW9yZSBkYXRhc2V0cyByZXF1aXJlIG1hbnVhbCBk'
        'b3dubG9hZCBvciBhY2NlcHRhbmNlLiIKICAgICAgICAgICAgaWYgc3RhdHVzID09IFByb2plY3RT'
        'dGF0dXMuRElTQ09WRVJFRF9PTkxZOgogICAgICAgICAgICAgICAgc3RhdHVzID0gUHJvamVjdFN0'
        'YXR1cy5CTE9DS0VEX0JZX0VYVEVSTkFMX0RBVEFTRVQKICAgICAgICBpZiAoCiAgICAgICAgICAg'
        'IGFueShpc3N1ZS5zZXZlcml0eSA9PSAid2FybmluZyIgZm9yIGlzc3VlIGluIHZhbGlkYXRpb24u'
        'aXNzdWVzKQogICAgICAgICAgICBhbmQgc3RhdHVzID09IFByb2plY3RTdGF0dXMuQ09NUExFVEUK'
        'ICAgICAgICApOgogICAgICAgICAgICBzdGF0dXMgPSBQcm9qZWN0U3RhdHVzLlNDSUVOVElGSUNB'
        'TExZX1JVTk5BQkxFCiAgICAgICAgc3VtbWFyeSA9IG1hbmlmZXN0LmRlc2NyaXB0aW9uCiAgICAg'
        'ICAgcmV0dXJuIFByb2plY3RSZWNvcmQoCiAgICAgICAgICAgIG1hbmlmZXN0PW1hbmlmZXN0LAog'
        'ICAgICAgICAgICB2YWxpZGF0aW9uPXZhbGlkYXRpb24sCiAgICAgICAgICAgIHN0YXR1cz1zdGF0'
        'dXMsCiAgICAgICAgICAgIHN1bW1hcnk9c3VtbWFyeSwKICAgICAgICAgICAgYmxvY2tlZF9yZWFz'
        'b249YmxvY2tlZF9yZWFzb24sCiAgICAgICAgKQoKICAgIGRlZiBfYXR0YWNoX3J1bl9oaXN0b3J5'
        'KHNlbGYsIHJlY29yZHM6IGxpc3RbUHJvamVjdFJlY29yZF0pIC0+IGxpc3RbUHJvamVjdFJlY29y'
        'ZF06CiAgICAgICAgbGF0ZXN0X2J5X3Byb2plY3Q6IGRpY3Rbc3RyLCBkaWN0W3N0ciwgb2JqZWN0'
        'XV0gPSB7fQogICAgICAgIGZvciBoaXN0b3J5X3JvdyBpbiBzZWxmLnJ1bm5lci5ydW5fZGIuZGV0'
        'YWlsZWRfcm93cygpOgogICAgICAgICAgICBsYXRlc3RfYnlfcHJvamVjdFtzdHIoaGlzdG9yeV9y'
        'b3dbInByb2plY3RfaWQiXSldID0gaGlzdG9yeV9yb3cKICAgICAgICBmb3IgcmVjb3JkIGluIHJl'
        'Y29yZHM6CiAgICAgICAgICAgIGxhdGVzdF9yb3cgPSBsYXRlc3RfYnlfcHJvamVjdC5nZXQocmVj'
        'b3JkLm1hbmlmZXN0LnByb2plY3RfaWQpCiAgICAgICAgICAgIGlmIGxhdGVzdF9yb3cgaXMgTm9u'
        'ZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHN0YXR1c19uYW1lID0gc3Ry'
        'KGxhdGVzdF9yb3dbInN0YXR1cyJdKQogICAgICAgICAgICByZWNvcmQuc3RhdHVzID0gUHJvamVj'
        'dFN0YXR1cyhzdGF0dXNfbmFtZSkKICAgICAgICAgICAgaWYgKAogICAgICAgICAgICAgICAgcmVj'
        'b3JkLnN0YXR1cyA9PSBQcm9qZWN0U3RhdHVzLlNNT0tFX1RFU1RFRAogICAgICAgICAgICAgICAg'
        'YW5kIGFueShkYXRhc2V0LmJsb2NrZWRfYnlfdXNlcl9hY3Rpb24gZm9yIGRhdGFzZXQgaW4gcmVj'
        'b3JkLm1hbmlmZXN0LmRhdGFzZXRzKQogICAgICAgICAgICApOgogICAgICAgICAgICAgICAgcmVj'
        'b3JkLnN0YXR1cyA9IFByb2plY3RTdGF0dXMuRlVMTF9CRU5DSE1BUktfUEVORElOR19DT01QVVRF'
        'CiAgICAgICAgICAgIHJlY29yZC5zdW1tYXJ5ID0gKAogICAgICAgICAgICAgICAgZiJMYXN0IHJ1'
        'biBge2xhdGVzdF9yb3dbJ2NvbW1hbmQnXX1gIGNvbXBsZXRlZCB3aXRoIHN0YXR1cyAiCiAgICAg'
        'ICAgICAgICAgICBmImB7cmVjb3JkLnN0YXR1cy52YWx1ZX1gLiIKICAgICAgICAgICAgKQogICAg'
        'ICAgICAgICBjbGFzc2lmaWNhdGlvbiA9IHN0cihsYXRlc3Rfcm93WyJjbGFzc2lmaWNhdGlvbiJd'
        'KQogICAgICAgICAgICBpZiBjbGFzc2lmaWNhdGlvbiAhPSAib2siOgogICAgICAgICAgICAgICAg'
        'cmVjb3JkLmJsb2NrZWRfcmVhc29uID0gX2Rlc2NyaWJlX3J1bl9mYWlsdXJlKAogICAgICAgICAg'
        'ICAgICAgICAgIGNsYXNzaWZpY2F0aW9uLAogICAgICAgICAgICAgICAgICAgIHN0cihsYXRlc3Rf'
        'cm93LmdldCgic3Rkb3V0IiwgIiIpKSwKICAgICAgICAgICAgICAgICAgICBzdHIobGF0ZXN0X3Jv'
        'dy5nZXQoInN0ZGVyciIsICIiKSksCiAgICAgICAgICAgICAgICApCiAgICAgICAgcmV0dXJuIHJl'
        'Y29yZHMKCgpkZWYgX2Rlc2NyaWJlX3J1bl9mYWlsdXJlKGNsYXNzaWZpY2F0aW9uOiBzdHIsIHN0'
        'ZG91dDogc3RyLCBzdGRlcnI6IHN0cikgLT4gc3RyOgogICAgaWYgY2xhc3NpZmljYXRpb24gPT0g'
        'Im9rIjoKICAgICAgICByZXR1cm4gIiIKCiAgICB0ZXh0ID0gIlxuIi5qb2luKHBhcnQgZm9yIHBh'
        'cnQgaW4gKHN0ZG91dCwgc3RkZXJyKSBpZiBwYXJ0KQogICAgbGluZXMgPSBbbGluZS5zdHJpcCgp'
        'IGZvciBsaW5lIGluIHRleHQuc3BsaXRsaW5lcygpIGlmIGxpbmUuc3RyaXAoKV0KCiAgICBpZiBj'
        'bGFzc2lmaWNhdGlvbiA9PSAiZXh0ZXJuYWxfZGF0YXNldCI6CiAgICAgICAgcmV0dXJuICJPbmUg'
        'b3IgbW9yZSByZXF1aXJlZCBkYXRhc2V0cyBhcmUgbWlzc2luZyBvciByZXF1aXJlIG1hbnVhbCBh'
        'Y2Nlc3MuIgogICAgaWYgY2xhc3NpZmljYXRpb24gPT0gImNyZWRlbnRpYWwiOgogICAgICAgIHJl'
        'dHVybiAiQSByZXF1aXJlZCBjcmVkZW50aWFsIG9yIEFQSSBrZXkgaXMgbWlzc2luZy4iCiAgICBp'
        'ZiBjbGFzc2lmaWNhdGlvbiA9PSAiZGVwZW5kZW5jeSI6CiAgICAgICAgcmV0dXJuICJBIHJlcXVp'
        'cmVkIG1vZHVsZSBvciBjb21tYW5kIGlzIG1pc3NpbmcgZnJvbSB0aGUgcnVudGltZSBlbnZpcm9u'
        'bWVudC4iCgogICAgcGVybWlzc2lvbl9saW5lID0gbmV4dCgKICAgICAgICAoCiAgICAgICAgICAg'
        'IGxpbmUKICAgICAgICAgICAgZm9yIGxpbmUgaW4gbGluZXMKICAgICAgICAgICAgaWYgIlBlcm1p'
        'c3Npb25FcnJvcjoiIGluIGxpbmUKICAgICAgICAgICAgb3IgIk9wZXJhdGlvbiBub3QgcGVybWl0'
        'dGVkIiBpbiBsaW5lCiAgICAgICAgICAgIG9yICJQZXJtaXNzaW9uIGRlbmllZCIgaW4gbGluZQog'
        'ICAgICAgICksCiAgICAgICAgIiIsCiAgICApCiAgICBpZiBjbGFzc2lmaWNhdGlvbiA9PSAicGVy'
        'bWlzc2lvbiIgYW5kIHBlcm1pc3Npb25fbGluZToKICAgICAgICByZXR1cm4gcGVybWlzc2lvbl9s'
        'aW5lCgogICAgZmFpbGVkX2xpbmUgPSBuZXh0KAogICAgICAgICgKICAgICAgICAgICAgbGluZS5y'
        'ZW1vdmVwcmVmaXgoIkZBSUxFRCAiKS5zdHJpcCgpCiAgICAgICAgICAgIGZvciBsaW5lIGluIHJl'
        'dmVyc2VkKGxpbmVzKQogICAgICAgICAgICBpZiBsaW5lLnN0YXJ0c3dpdGgoIkZBSUxFRCAiKQog'
        'ICAgICAgICksCiAgICAgICAgIiIsCiAgICApCiAgICBlcnJvcl9saW5lID0gbmV4dCgKICAgICAg'
        'ICAoCiAgICAgICAgICAgIGxpbmUucmVtb3ZlcHJlZml4KCJFIikuc3RyaXAoKQogICAgICAgICAg'
        'ICBmb3IgbGluZSBpbiByZXZlcnNlZChsaW5lcykKICAgICAgICAgICAgaWYgIkFzc2VydGlvbkVy'
        'cm9yOiIgaW4gbGluZQogICAgICAgICAgICBvciAiUGVybWlzc2lvbkVycm9yOiIgaW4gbGluZQog'
        'ICAgICAgICAgICBvciAiTW9kdWxlTm90Rm91bmRFcnJvcjoiIGluIGxpbmUKICAgICAgICAgICAg'
        'b3IgIkltcG9ydEVycm9yOiIgaW4gbGluZQogICAgICAgICAgICBvciAiRmlsZU5vdEZvdW5kRXJy'
        'b3I6IiBpbiBsaW5lCiAgICAgICAgKSwKICAgICAgICAiIiwKICAgICkKICAgIGFzc2VydF9saW5l'
        'ID0gbmV4dCgKICAgICAgICAoCiAgICAgICAgICAgIGxpbmUubHN0cmlwKCI+ICIpLnN0cmlwKCkK'
        'ICAgICAgICAgICAgZm9yIGxpbmUgaW4gcmV2ZXJzZWQobGluZXMpCiAgICAgICAgICAgIGlmIGxp'
        'bmUubHN0cmlwKCI+ICIpLnN0cmlwKCkuc3RhcnRzd2l0aCgiYXNzZXJ0ICIpCiAgICAgICAgKSwK'
        'ICAgICAgICAiIiwKICAgICkKCiAgICBpZiBmYWlsZWRfbGluZSBhbmQgYXNzZXJ0X2xpbmU6CiAg'
        'ICAgICAgcmV0dXJuIGYie2ZhaWxlZF9saW5lfToge2Fzc2VydF9saW5lfSIKICAgIGlmIGZhaWxl'
        'ZF9saW5lIGFuZCBlcnJvcl9saW5lOgogICAgICAgIHJldHVybiBmIntmYWlsZWRfbGluZX06IHtl'
        'cnJvcl9saW5lfSIKICAgIGlmIGZhaWxlZF9saW5lOgogICAgICAgIHJldHVybiBmYWlsZWRfbGlu'
        'ZQogICAgaWYgcGVybWlzc2lvbl9saW5lOgogICAgICAgIHJldHVybiBwZXJtaXNzaW9uX2xpbmUK'
        'ICAgIGlmIGVycm9yX2xpbmU6CiAgICAgICAgcmV0dXJuIGVycm9yX2xpbmUKICAgIHJldHVybiBj'
        'bGFzc2lmaWNhdGlvbgo='
    ),
    'olympus/labos/reports.py': (
        '...'
    ),
    'olympus/labos/runner.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IG9z'
        'CmltcG9ydCBzaGxleAppbXBvcnQgc3FsaXRlMwppbXBvcnQgc3VicHJvY2Vzcwpmcm9tIGRhdGFj'
        'bGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBkYXRldGltZSBpbXBvcnQgVVRDLCBkYXRldGlt'
        'ZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gb2x5bXB1cy5sYWJvcy5tYW5pZmVzdCBp'
        'bXBvcnQgUHJvamVjdE1hbmlmZXN0LCBQcm9qZWN0U3RhdHVzLCBSZXNvdXJjZVByb2ZpbGUKCgpA'
        'ZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIFJ1blJlc3VsdDoKICAgIHByb2plY3RfaWQ6IHN0'
        'cgogICAgY29tbWFuZDogc3RyCiAgICByZXR1cm5fY29kZTogaW50CiAgICBzdGRvdXQ6IHN0cgog'
        'ICAgc3RkZXJyOiBzdHIKICAgIHN0YXR1czogUHJvamVjdFN0YXR1cwogICAgc3RhcnRlZF9hdDog'
        'c3RyCiAgICBmaW5pc2hlZF9hdDogc3RyCiAgICBwcm9maWxlOiBzdHIKICAgIGN3ZDogc3RyCiAg'
        'ICBjbGFzc2lmaWNhdGlvbjogc3RyCgoKY2xhc3MgUnVuRGF0YWJhc2U6CiAgICBkZWYgX19pbml0'
        'X18oc2VsZiwgcGF0aDogUGF0aCkgLT4gTm9uZToKICAgICAgICBzZWxmLnBhdGggPSBwYXRoCiAg'
        'ICAgICAgc2VsZi5wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUp'
        'CiAgICAgICAgc2VsZi5jb25uZWN0aW9uID0gc3FsaXRlMy5jb25uZWN0KHBhdGgpCiAgICAgICAg'
        'c2VsZi5jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICIiIgogICAgICAgICAgICBDUkVB'
        'VEUgVEFCTEUgSUYgTk9UIEVYSVNUUyBydW5zICgKICAgICAgICAgICAgICAgIGlkIElOVEVHRVIg'
        'UFJJTUFSWSBLRVkgQVVUT0lOQ1JFTUVOVCwKICAgICAgICAgICAgICAgIHByb2plY3RfaWQgVEVY'
        'VCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIGNvbW1hbmQgVEVYVCBOT1QgTlVMTCwKICAgICAg'
        'ICAgICAgICAgIHByb2ZpbGUgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIGN3ZCBURVhU'
        'IE5PVCBOVUxMLAogICAgICAgICAgICAgICAgcmV0dXJuX2NvZGUgSU5URUdFUiBOT1QgTlVMTCwK'
        'ICAgICAgICAgICAgICAgIHN0YXR1cyBURVhUIE5PVCBOVUxMLAogICAgICAgICAgICAgICAgY2xh'
        'c3NpZmljYXRpb24gVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIHN0YXJ0ZWRfYXQgVEVY'
        'VCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIGZpbmlzaGVkX2F0IFRFWFQgTk9UIE5VTEwsCiAg'
        'ICAgICAgICAgICAgICBzdGRvdXQgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIHN0ZGVy'
        'ciBURVhUIE5PVCBOVUxMCiAgICAgICAgICAgICkKICAgICAgICAgICAgIiIiCiAgICAgICAgKQog'
        'ICAgICAgIHNlbGYuY29ubmVjdGlvbi5jb21taXQoKQoKICAgIGRlZiBjbG9zZShzZWxmKSAtPiBO'
        'b25lOgogICAgICAgIHNlbGYuY29ubmVjdGlvbi5jbG9zZSgpCgogICAgZGVmIF9fZW50ZXJfXyhz'
        'ZWxmKSAtPiBSdW5EYXRhYmFzZToKICAgICAgICByZXR1cm4gc2VsZgoKICAgIGRlZiBfX2V4aXRf'
        'XyhzZWxmLCAqYXJnczogb2JqZWN0KSAtPiBOb25lOgogICAgICAgIHNlbGYuY2xvc2UoKQoKICAg'
        'IGRlZiBfX2RlbF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBzZWxm'
        'LmNsb3NlKCkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOiAgIyBub3FhOiBCTEUwMDEKICAgICAg'
        'ICAgICAgcGFzcwoKICAgIGRlZiBpbnNlcnQoc2VsZiwgcmVzdWx0OiBSdW5SZXN1bHQpIC0+IE5v'
        'bmU6CiAgICAgICAgc2VsZi5jb25uZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICIiIgogICAg'
        'ICAgICAgICBJTlNFUlQgSU5UTyBydW5zICgKICAgICAgICAgICAgICAgIHByb2plY3RfaWQsIGNv'
        'bW1hbmQsIHByb2ZpbGUsIGN3ZCwgcmV0dXJuX2NvZGUsIHN0YXR1cywKICAgICAgICAgICAgICAg'
        'IGNsYXNzaWZpY2F0aW9uLCBzdGFydGVkX2F0LCBmaW5pc2hlZF9hdCwgc3Rkb3V0LCBzdGRlcnIK'
        'ICAgICAgICAgICAgKSBWQUxVRVMgKD8sID8sID8sID8sID8sID8sID8sID8sID8sID8sID8pCiAg'
        'ICAgICAgICAgICIiIiwKICAgICAgICAgICAgKAogICAgICAgICAgICAgICAgcmVzdWx0LnByb2pl'
        'Y3RfaWQsCiAgICAgICAgICAgICAgICByZXN1bHQuY29tbWFuZCwKICAgICAgICAgICAgICAgIHJl'
        'c3VsdC5wcm9maWxlLAogICAgICAgICAgICAgICAgcmVzdWx0LmN3ZCwKICAgICAgICAgICAgICAg'
        'IHJlc3VsdC5yZXR1cm5fY29kZSwKICAgICAgICAgICAgICAgIHJlc3VsdC5zdGF0dXMudmFsdWUs'
        'CiAgICAgICAgICAgICAgICByZXN1bHQuY2xhc3NpZmljYXRpb24sCiAgICAgICAgICAgICAgICBy'
        'ZXN1bHQuc3RhcnRlZF9hdCwKICAgICAgICAgICAgICAgIHJlc3VsdC5maW5pc2hlZF9hdCwKICAg'
        'ICAgICAgICAgICAgIHJlc3VsdC5zdGRvdXQsCiAgICAgICAgICAgICAgICByZXN1bHQuc3RkZXJy'
        'LAogICAgICAgICAgICApLAogICAgICAgICkKICAgICAgICBzZWxmLmNvbm5lY3Rpb24uY29tbWl0'
        'KCkKCiAgICBkZWYgZmFpbGVkX3J1bnMoc2VsZikgLT4gbGlzdFt0dXBsZVtzdHIsIHN0cl1dOgog'
        'ICAgICAgIHJvd3MgPSBzZWxmLmNvbm5lY3Rpb24uZXhlY3V0ZSgKICAgICAgICAgICAgIlNFTEVD'
        'VCBwcm9qZWN0X2lkLCBjb21tYW5kIEZST00gcnVucyBXSEVSRSByZXR1cm5fY29kZSAhPSAwIE9S'
        'REVSIEJZIGlkIERFU0MiCiAgICAgICAgKS5mZXRjaGFsbCgpCiAgICAgICAgcmV0dXJuIFsoc3Ry'
        'KHJvd1swXSksIHN0cihyb3dbMV0pKSBmb3Igcm93IGluIHJvd3NdCgogICAgZGVmIHJvd3Moc2Vs'
        'ZikgLT4gbGlzdFtkaWN0W3N0ciwgb2JqZWN0XV06CiAgICAgICAgY3Vyc29yID0gc2VsZi5jb25u'
        'ZWN0aW9uLmV4ZWN1dGUoCiAgICAgICAgICAgICIiIgogICAgICAgICAgICBTRUxFQ1QgcHJvamVj'
        'dF9pZCwgY29tbWFuZCwgcHJvZmlsZSwgY3dkLCByZXR1cm5fY29kZSwgc3RhdHVzLAogICAgICAg'
        'ICAgICAgICAgICAgY2xhc3NpZmljYXRpb24sIHN0YXJ0ZWRfYXQsIGZpbmlzaGVkX2F0CiAgICAg'
        'ICAgICAgIEZST00gcnVucyBPUkRFUiBCWSBpZAogICAgICAgICAgICAiIiIKICAgICAgICApCiAg'
        'ICAgICAgY29sdW1ucyA9IFtpdGVtWzBdIGZvciBpdGVtIGluIGN1cnNvci5kZXNjcmlwdGlvbl0K'
        'ICAgICAgICByZXR1cm4gW2RpY3QoemlwKGNvbHVtbnMsIHJvdywgc3RyaWN0PVRydWUpKSBmb3Ig'
        'cm93IGluIGN1cnNvci5mZXRjaGFsbCgpXQoKICAgIGRlZiBkZXRhaWxlZF9yb3dzKHNlbGYpIC0+'
        'IGxpc3RbZGljdFtzdHIsIG9iamVjdF1dOgogICAgICAgIGN1cnNvciA9IHNlbGYuY29ubmVjdGlv'
        'bi5leGVjdXRlKAogICAgICAgICAgICAiIiIKICAgICAgICAgICAgU0VMRUNUIHByb2plY3RfaWQs'
        'IGNvbW1hbmQsIHByb2ZpbGUsIGN3ZCwgcmV0dXJuX2NvZGUsIHN0YXR1cywKICAgICAgICAgICAg'
        'ICAgICAgIGNsYXNzaWZpY2F0aW9uLCBzdGFydGVkX2F0LCBmaW5pc2hlZF9hdCwgc3Rkb3V0LCBz'
        'dGRlcnIKICAgICAgICAgICAgRlJPTSBydW5zIE9SREVSIEJZIGlkCiAgICAgICAgICAgICIiIgog'
        'ICAgICAgICkKICAgICAgICBjb2x1bW5zID0gW2l0ZW1bMF0gZm9yIGl0ZW0gaW4gY3Vyc29yLmRl'
        'c2NyaXB0aW9uXQogICAgICAgIHJldHVybiBbZGljdCh6aXAoY29sdW1ucywgcm93LCBzdHJpY3Q9'
        'VHJ1ZSkpIGZvciByb3cgaW4gY3Vyc29yLmZldGNoYWxsKCldCgoKY2xhc3MgRXZlbnRMb2dnZXI6'
        'CiAgICBkZWYgX19pbml0X18oc2VsZiwgcGF0aDogUGF0aCkgLT4gTm9uZToKICAgICAgICBzZWxm'
        'LnBhdGggPSBwYXRoCiAgICAgICAgc2VsZi5wYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUs'
        'IGV4aXN0X29rPVRydWUpCgogICAgZGVmIGxvZyhzZWxmLCBwYXlsb2FkOiBkaWN0W3N0ciwgb2Jq'
        'ZWN0XSkgLT4gTm9uZToKICAgICAgICB3aXRoIHNlbGYucGF0aC5vcGVuKCJhIiwgZW5jb2Rpbmc9'
        'InV0Zi04IikgYXMgaGFuZGxlOgogICAgICAgICAgICBoYW5kbGUud3JpdGUoanNvbi5kdW1wcyhw'
        'YXlsb2FkLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iKQoKCmNsYXNzIFBvcnRmb2xpb1J1bm5lcjoK'
        'ICAgIGRlZiBfX2luaXRfXyhzZWxmLCBhcnRpZmFjdF9yb290OiBQYXRoKSAtPiBOb25lOgogICAg'
        'ICAgIHNlbGYuYXJ0aWZhY3Rfcm9vdCA9IGFydGlmYWN0X3Jvb3QKICAgICAgICBzZWxmLnJ1bl9k'
        'YiA9IFJ1bkRhdGFiYXNlKGFydGlmYWN0X3Jvb3QgLyAibGFib3NfcnVucy5zcWxpdGUzIikKICAg'
        'ICAgICBzZWxmLmV2ZW50X2xvZ2dlciA9IEV2ZW50TG9nZ2VyKGFydGlmYWN0X3Jvb3QgLyAibGFi'
        'b3NfZXZlbnRzLmpzb25sIikKCiAgICBkZWYgY2xvc2Uoc2VsZikgLT4gTm9uZToKICAgICAgICBz'
        'ZWxmLnJ1bl9kYi5jbG9zZSgpCgogICAgZGVmIHJ1bl9tYW5pZmVzdCgKICAgICAgICBzZWxmLAog'
        'ICAgICAgIG1hbmlmZXN0OiBQcm9qZWN0TWFuaWZlc3QsCiAgICAgICAgcHJvZmlsZTogUmVzb3Vy'
        'Y2VQcm9maWxlID0gUmVzb3VyY2VQcm9maWxlLlNNT0tFLAogICAgICAgIHRpbWVvdXRfc2Vjb25k'
        'czogaW50ID0gMTIwLAogICAgKSAtPiBsaXN0W1J1blJlc3VsdF06CiAgICAgICAgY29tbWFuZHMg'
        'PSBbZW50cnkgZm9yIGVudHJ5IGluIG1hbmlmZXN0LmVudHJ5X3BvaW50cyBpZiBlbnRyeS5wcm9m'
        'aWxlID09IHByb2ZpbGVdCiAgICAgICAgaWYgbm90IGNvbW1hbmRzIGFuZCBwcm9maWxlID09IFJl'
        'c291cmNlUHJvZmlsZS5TTU9LRToKICAgICAgICAgICAgcmV0dXJuIFtdCiAgICAgICAgcmVzdWx0'
        'czogbGlzdFtSdW5SZXN1bHRdID0gW10KICAgICAgICBmb3IgZW50cnkgaW4gY29tbWFuZHM6CiAg'
        'ICAgICAgICAgIHJlc3VsdCA9IHNlbGYucnVuX2NvbW1hbmQoCiAgICAgICAgICAgICAgICBwcm9q'
        'ZWN0X2lkPW1hbmlmZXN0LnByb2plY3RfaWQsCiAgICAgICAgICAgICAgICBjb21tYW5kPWVudHJ5'
        'LmNvbW1hbmQsCiAgICAgICAgICAgICAgICBjd2Q9bWFuaWZlc3QucGF0aCwKICAgICAgICAgICAg'
        'ICAgIHByb2ZpbGU9cHJvZmlsZSwKICAgICAgICAgICAgICAgIHRpbWVvdXRfc2Vjb25kcz1lbnRy'
        'eS50aW1lb3V0X3NlY29uZHMgb3IgdGltZW91dF9zZWNvbmRzLAogICAgICAgICAgICAgICAgZXh0'
        'cmFfZW52PWVudHJ5LmVudiwKICAgICAgICAgICAgKQogICAgICAgICAgICByZXN1bHRzLmFwcGVu'
        'ZChyZXN1bHQpCiAgICAgICAgcmV0dXJuIHJlc3VsdHMKCiAgICBkZWYgcnVuX2NvbW1hbmQoCiAg'
        'ICAgICAgc2VsZiwKICAgICAgICBwcm9qZWN0X2lkOiBzdHIsCiAgICAgICAgY29tbWFuZDogc3Ry'
        'LAogICAgICAgIGN3ZDogUGF0aCwKICAgICAgICBwcm9maWxlOiBSZXNvdXJjZVByb2ZpbGUsCiAg'
        'ICAgICAgdGltZW91dF9zZWNvbmRzOiBmbG9hdCA9IDEyMCwKICAgICAgICBleHRyYV9lbnY6IGRp'
        'Y3Rbc3RyLCBzdHJdIHwgTm9uZSA9IE5vbmUsCiAgICApIC0+IFJ1blJlc3VsdDoKICAgICAgICBz'
        'dGFydGVkX2F0ID0gZGF0ZXRpbWUubm93KFVUQykuaXNvZm9ybWF0KCkKICAgICAgICBlbnYgPSBv'
        'cy5lbnZpcm9uLmNvcHkoKQogICAgICAgIGVudlsiUFlUSE9ORE9OVFdSSVRFQllURUNPREUiXSA9'
        'ICIxIgogICAgICAgIGVudlsiUFlURVNUX0FERE9QVFMiXSA9ICItcCBubzpjYWNoZXByb3ZpZGVy'
        'IgogICAgICAgIGlmIGV4dHJhX2VudjoKICAgICAgICAgICAgZW52LnVwZGF0ZShleHRyYV9lbnYp'
        'CiAgICAgICAgdHJ5OgogICAgICAgICAgICBjb21wbGV0ZWQgPSBzdWJwcm9jZXNzLnJ1bigKICAg'
        'ICAgICAgICAgICAgIHNobGV4LnNwbGl0KGNvbW1hbmQpLAogICAgICAgICAgICAgICAgY3dkPWN3'
        'ZCwKICAgICAgICAgICAgICAgIGVudj1lbnYsCiAgICAgICAgICAgICAgICBjYXB0dXJlX291dHB1'
        'dD1UcnVlLAogICAgICAgICAgICAgICAgdGV4dD1UcnVlLAogICAgICAgICAgICAgICAgdGltZW91'
        'dD10aW1lb3V0X3NlY29uZHMsCiAgICAgICAgICAgICAgICBjaGVjaz1GYWxzZSwKICAgICAgICAg'
        'ICAgKQogICAgICAgICAgICByZXR1cm5fY29kZSA9IGNvbXBsZXRlZC5yZXR1cm5jb2RlCiAgICAg'
        'ICAgICAgIHN0ZG91dCA9IGNvbXBsZXRlZC5zdGRvdXQKICAgICAgICAgICAgc3RkZXJyID0gY29t'
        'cGxldGVkLnN0ZGVycgogICAgICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkIGFz'
        'IGV4YzoKICAgICAgICAgICAgcmV0dXJuX2NvZGUgPSAxMjQKICAgICAgICAgICAgc3Rkb3V0ID0g'
        'X2RlY29kZV90aW1lb3V0X3N0cmVhbShleGMuc3Rkb3V0KQogICAgICAgICAgICBzdGRlcnIgPSBf'
        'ZGVjb2RlX3RpbWVvdXRfc3RyZWFtKGV4Yy5zdGRlcnIpCiAgICAgICAgICAgIHN0ZGVyciA9IChz'
        'dGRlcnIgKyAiXG4iIGlmIHN0ZGVyciBlbHNlICIiKSArICgKICAgICAgICAgICAgICAgIGYiQ29t'
        'bWFuZCB0aW1lZCBvdXQgYWZ0ZXIge3RpbWVvdXRfc2Vjb25kc30gc2Vjb25kcy4iCiAgICAgICAg'
        'ICAgICkKICAgICAgICBmaW5pc2hlZF9hdCA9IGRhdGV0aW1lLm5vdyhVVEMpLmlzb2Zvcm1hdCgp'
        'CiAgICAgICAgY2xhc3NpZmljYXRpb24gPSBjbGFzc2lmeV9ydW4ocmV0dXJuX2NvZGUsIHN0ZGVy'
        'cikKICAgICAgICBzdGF0dXMgPSBzdGF0dXNfZnJvbV9ydW4ocmV0dXJuX2NvZGUsIGNsYXNzaWZp'
        'Y2F0aW9uLCBwcm9maWxlKQogICAgICAgIHJlc3VsdCA9IFJ1blJlc3VsdCgKICAgICAgICAgICAg'
        'cHJvamVjdF9pZD1wcm9qZWN0X2lkLAogICAgICAgICAgICBjb21tYW5kPWNvbW1hbmQsCiAgICAg'
        'ICAgICAgIHJldHVybl9jb2RlPXJldHVybl9jb2RlLAogICAgICAgICAgICBzdGRvdXQ9c3Rkb3V0'
        'Wy0yMF8wMDA6XSwKICAgICAgICAgICAgc3RkZXJyPXN0ZGVyclstMjBfMDAwOl0sCiAgICAgICAg'
        'ICAgIHN0YXR1cz1zdGF0dXMsCiAgICAgICAgICAgIHN0YXJ0ZWRfYXQ9c3RhcnRlZF9hdCwKICAg'
        'ICAgICAgICAgZmluaXNoZWRfYXQ9ZmluaXNoZWRfYXQsCiAgICAgICAgICAgIHByb2ZpbGU9cHJv'
        'ZmlsZS52YWx1ZSwKICAgICAgICAgICAgY3dkPXN0cihjd2QpLAogICAgICAgICAgICBjbGFzc2lm'
        'aWNhdGlvbj1jbGFzc2lmaWNhdGlvbiwKICAgICAgICApCiAgICAgICAgc2VsZi5ydW5fZGIuaW5z'
        'ZXJ0KHJlc3VsdCkKICAgICAgICBzZWxmLmV2ZW50X2xvZ2dlci5sb2coCiAgICAgICAgICAgIHsK'
        'ICAgICAgICAgICAgICAgICJwcm9qZWN0X2lkIjogcHJvamVjdF9pZCwKICAgICAgICAgICAgICAg'
        'ICJjb21tYW5kIjogY29tbWFuZCwKICAgICAgICAgICAgICAgICJwcm9maWxlIjogcHJvZmlsZS52'
        'YWx1ZSwKICAgICAgICAgICAgICAgICJjd2QiOiBzdHIoY3dkKSwKICAgICAgICAgICAgICAgICJy'
        'ZXR1cm5fY29kZSI6IHJldHVybl9jb2RlLAogICAgICAgICAgICAgICAgImNsYXNzaWZpY2F0aW9u'
        'IjogY2xhc3NpZmljYXRpb24sCiAgICAgICAgICAgICAgICAic3RhdHVzIjogc3RhdHVzLnZhbHVl'
        'LAogICAgICAgICAgICAgICAgInN0YXJ0ZWRfYXQiOiBzdGFydGVkX2F0LAogICAgICAgICAgICAg'
        'ICAgImZpbmlzaGVkX2F0IjogZmluaXNoZWRfYXQsCiAgICAgICAgICAgICAgICAiZW52X292ZXJy'
        'aWRlcyI6IGV4dHJhX2VudiBvciB7fSwKICAgICAgICAgICAgfQogICAgICAgICkKICAgICAgICBy'
        'ZXR1cm4gcmVzdWx0CgoKZGVmIGNsYXNzaWZ5X3J1bihyZXR1cm5fY29kZTogaW50LCBzdGRlcnI6'
        'IHN0cikgLT4gc3RyOgogICAgbG93ZXJlZCA9IHN0ZGVyci5sb3dlcigpCiAgICBpZiByZXR1cm5f'
        'Y29kZSA9PSAwOgogICAgICAgIHJldHVybiAib2siCiAgICBpZiByZXR1cm5fY29kZSA9PSAxMjQg'
        'b3IgInRpbWVkIG91dCIgaW4gbG93ZXJlZCBvciAidGltZW91dCIgaW4gbG93ZXJlZDoKICAgICAg'
        'ICByZXR1cm4gInRpbWVvdXQiCiAgICBpZiAicGVybWlzc2lvbiBkZW5pZWQiIGluIGxvd2VyZWQg'
        'b3IgIm9wZXJhdGlvbiBub3QgcGVybWl0dGVkIiBpbiBsb3dlcmVkOgogICAgICAgIHJldHVybiAi'
        'cGVybWlzc2lvbiIKICAgIGlmICJubyBtb2R1bGUgbmFtZWQiIGluIGxvd2VyZWQgb3IgImNvbW1h'
        'bmQgbm90IGZvdW5kIiBpbiBsb3dlcmVkOgogICAgICAgIHJldHVybiAiZGVwZW5kZW5jeSIKICAg'
        'IGlmICJub3QgZm91bmQiIGluIGxvd2VyZWQgYW5kICJkYXRhc2V0IiBpbiBsb3dlcmVkOgogICAg'
        'ICAgIHJldHVybiAiZXh0ZXJuYWxfZGF0YXNldCIKICAgIGlmICJjcmVkZW50aWFsIiBpbiBsb3dl'
        'cmVkIG9yICJhcGkga2V5IiBpbiBsb3dlcmVkOgogICAgICAgIHJldHVybiAiY3JlZGVudGlhbCIK'
        'ICAgIHJldHVybiAiZmFpbGVkX2NvbW1hbmQiCgoKZGVmIHN0YXR1c19mcm9tX3J1bigKICAgIHJl'
        'dHVybl9jb2RlOiBpbnQsCiAgICBjbGFzc2lmaWNhdGlvbjogc3RyLAogICAgcHJvZmlsZTogUmVz'
        'b3VyY2VQcm9maWxlLAopIC0+IFByb2plY3RTdGF0dXM6CiAgICBpZiByZXR1cm5fY29kZSA9PSAw'
        'IGFuZCBwcm9maWxlID09IFJlc291cmNlUHJvZmlsZS5TTU9LRToKICAgICAgICByZXR1cm4gUHJv'
        'amVjdFN0YXR1cy5TTU9LRV9URVNURUQKICAgIGlmIHJldHVybl9jb2RlID09IDA6CiAgICAgICAg'
        'cmV0dXJuIFByb2plY3RTdGF0dXMuU0NJRU5USUZJQ0FMTFlfUlVOTkFCTEUKICAgIGlmIGNsYXNz'
        'aWZpY2F0aW9uID09ICJleHRlcm5hbF9kYXRhc2V0IjoKICAgICAgICByZXR1cm4gUHJvamVjdFN0'
        'YXR1cy5CTE9DS0VEX0JZX0VYVEVSTkFMX0RBVEFTRVQKICAgIGlmIGNsYXNzaWZpY2F0aW9uID09'
        'ICJjcmVkZW50aWFsIjoKICAgICAgICByZXR1cm4gUHJvamVjdFN0YXR1cy5CTE9DS0VEX0JZX0NS'
        'RURFTlRJQUwKICAgIGlmIGNsYXNzaWZpY2F0aW9uID09ICJwZXJtaXNzaW9uIjoKICAgICAgICBy'
        'ZXR1cm4gUHJvamVjdFN0YXR1cy5CTE9DS0VEX0JZX1BFUk1JU1NJT04KICAgIGlmIGNsYXNzaWZp'
        'Y2F0aW9uID09ICJ0aW1lb3V0IjoKICAgICAgICByZXR1cm4gUHJvamVjdFN0YXR1cy5GVUxMX0JF'
        'TkNITUFSS19QRU5ESU5HX0NPTVBVVEUKICAgIHJldHVybiBQcm9qZWN0U3RhdHVzLkRJU0NPVkVS'
        'RURfT05MWQoKCmRlZiBfZGVjb2RlX3RpbWVvdXRfc3RyZWFtKHZhbHVlOiBzdHIgfCBieXRlcyB8'
        'IE5vbmUpIC0+IHN0cjoKICAgIGlmIHZhbHVlIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuICIiCiAg'
        'ICBpZiBpc2luc3RhbmNlKHZhbHVlLCBieXRlcyk6CiAgICAgICAgcmV0dXJuIHZhbHVlLmRlY29k'
        'ZShlcnJvcnM9InJlcGxhY2UiKQogICAgcmV0dXJuIHZhbHVlCg=='
    ),
    'olympus/labos/scheduler.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzCgpmcm9tIG9seW1wdXMubGFib3MubWFuaWZlc3QgaW1wb3J0IFByb2plY3RN'
        'YW5pZmVzdCwgUmVzb3VyY2VQcm9maWxlCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFzcyBT'
        'Y2hlZHVsZWRUYXNrOgogICAgcHJvamVjdF9pZDogc3RyCiAgICBjb21tYW5kOiBzdHIKICAgIHBy'
        'aW9yaXR5OiBpbnQKICAgIHByb2ZpbGU6IFJlc291cmNlUHJvZmlsZQoKCmNsYXNzIFJlc291cmNl'
        'QXdhcmVTY2hlZHVsZXI6CiAgICBkZWYgc2NoZWR1bGUoCiAgICAgICAgc2VsZiwKICAgICAgICBt'
        'YW5pZmVzdHM6IGxpc3RbUHJvamVjdE1hbmlmZXN0XSwKICAgICAgICBwcm9maWxlOiBSZXNvdXJj'
        'ZVByb2ZpbGUsCiAgICApIC0+IGxpc3RbU2NoZWR1bGVkVGFza106CiAgICAgICAgdGFza3M6IGxp'
        'c3RbU2NoZWR1bGVkVGFza10gPSBbXQogICAgICAgIGZvciBtYW5pZmVzdCBpbiBtYW5pZmVzdHM6'
        'CiAgICAgICAgICAgIGZvciBlbnRyeSBpbiBtYW5pZmVzdC5lbnRyeV9wb2ludHM6CiAgICAgICAg'
        'ICAgICAgICBpZiBlbnRyeS5wcm9maWxlICE9IHByb2ZpbGU6CiAgICAgICAgICAgICAgICAgICAg'
        'Y29udGludWUKICAgICAgICAgICAgICAgIHByaW9yaXR5ID0gMAogICAgICAgICAgICAgICAgaWYg'
        'bWFuaWZlc3QubWFuaWZlc3Rfc291cmNlLnZhbHVlID09ICJkZWNsYXJlZCI6CiAgICAgICAgICAg'
        'ICAgICAgICAgcHJpb3JpdHkgKz0gMgogICAgICAgICAgICAgICAgaWYgbWFuaWZlc3QucHJvamVj'
        'dF9pZCA9PSAib2x5bXB1cyI6CiAgICAgICAgICAgICAgICAgICAgcHJpb3JpdHkgKz0gMwogICAg'
        'ICAgICAgICAgICAgaWYgbWFuaWZlc3QucmVzb3VyY2VfcmVxdWlyZW1lbnRzLmdwdV9vcHRpb25h'
        'bDoKICAgICAgICAgICAgICAgICAgICBwcmlvcml0eSAtPSAxCiAgICAgICAgICAgICAgICB0YXNr'
        'cy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgU2NoZWR1bGVkVGFzaygKICAgICAgICAgICAg'
        'ICAgICAgICAgICAgcHJvamVjdF9pZD1tYW5pZmVzdC5wcm9qZWN0X2lkLAogICAgICAgICAgICAg'
        'ICAgICAgICAgICBjb21tYW5kPWVudHJ5LmNvbW1hbmQsCiAgICAgICAgICAgICAgICAgICAgICAg'
        'IHByaW9yaXR5PXByaW9yaXR5LAogICAgICAgICAgICAgICAgICAgICAgICBwcm9maWxlPXByb2Zp'
        'bGUsCiAgICAgICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgKQogICAgICAgIHJldHVy'
        'biBzb3J0ZWQodGFza3MsIGtleT1sYW1iZGEgaXRlbTogaXRlbS5wcmlvcml0eSwgcmV2ZXJzZT1U'
        'cnVlKQo='
    ),
    'olympus/labos/validation.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBwYXRobGliIGltcG9ydCBQ'
        'YXRoCgpmcm9tIG9seW1wdXMubGFib3MubWFuaWZlc3QgaW1wb3J0IFByb2plY3RNYW5pZmVzdCwg'
        'VmFsaWRhdGlvbklzc3VlLCBWYWxpZGF0aW9uUmVzdWx0CgoKZGVmIHZhbGlkYXRlX3Byb2plY3Qo'
        'bWFuaWZlc3Q6IFByb2plY3RNYW5pZmVzdCkgLT4gVmFsaWRhdGlvblJlc3VsdDoKICAgIGlzc3Vl'
        'czogbGlzdFtWYWxpZGF0aW9uSXNzdWVdID0gW10KICAgIHByb2plY3Rfcm9vdCA9IFBhdGgobWFu'
        'aWZlc3Qucm9vdF9wYXRoKQogICAgaWYgbm90IHByb2plY3Rfcm9vdC5leGlzdHMoKToKICAgICAg'
        'ICBpc3N1ZXMuYXBwZW5kKFZhbGlkYXRpb25Jc3N1ZShzZXZlcml0eT0iZXJyb3IiLCBtZXNzYWdl'
        'PSJQcm9qZWN0IHJvb3QgZG9lcyBub3QgZXhpc3QuIikpCiAgICBpZiBub3QgX2hhc19yZWFkbWUo'
        'cHJvamVjdF9yb290KToKICAgICAgICBpc3N1ZXMuYXBwZW5kKFZhbGlkYXRpb25Jc3N1ZShzZXZl'
        'cml0eT0id2FybmluZyIsIG1lc3NhZ2U9IlJFQURNRSBpcyBtaXNzaW5nLiIpKQogICAgaWYgbm90'
        'IG1hbmlmZXN0LmVudHJ5X3BvaW50czoKICAgICAgICBpc3N1ZXMuYXBwZW5kKAogICAgICAgICAg'
        'ICBWYWxpZGF0aW9uSXNzdWUoCiAgICAgICAgICAgICAgICBzZXZlcml0eT0id2FybmluZyIsCiAg'
        'ICAgICAgICAgICAgICBtZXNzYWdlPSJObyBydW5uYWJsZSBlbnRyeSBwb2ludHMgZGV0ZWN0ZWQu'
        'IiwKICAgICAgICAgICAgKQogICAgICAgICkKICAgIGlmIG5vdCBtYW5pZmVzdC52YWxpZGF0aW9u'
        'X2NvbW1hbmRzOgogICAgICAgIGlzc3Vlcy5hcHBlbmQoCiAgICAgICAgICAgIFZhbGlkYXRpb25J'
        'c3N1ZSgKICAgICAgICAgICAgICAgIHNldmVyaXR5PSJ3YXJuaW5nIiwKICAgICAgICAgICAgICAg'
        'IG1lc3NhZ2U9Ik5vIHZhbGlkYXRpb24gY29tbWFuZHMgZGV0ZWN0ZWQuIiwKICAgICAgICAgICAg'
        'KQogICAgICAgICkKICAgIGZvciBkYXRhc2V0IGluIG1hbmlmZXN0LmRhdGFzZXRzOgogICAgICAg'
        'IGlmIGRhdGFzZXQuYmxvY2tlZF9ieV91c2VyX2FjdGlvbjoKICAgICAgICAgICAgaXNzdWVzLmFw'
        'cGVuZCgKICAgICAgICAgICAgICAgIFZhbGlkYXRpb25Jc3N1ZSgKICAgICAgICAgICAgICAgICAg'
        'ICBzZXZlcml0eT0iaW5mbyIsCiAgICAgICAgICAgICAgICAgICAgbWVzc2FnZT1mIkRhdGFzZXQg'
        'J3tkYXRhc2V0Lm5hbWV9JyBtYXkgcmVxdWlyZSBtYW51YWwgZG93bmxvYWQgb3IgYWNjZXB0YW5j'
        'ZS4iLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICApCiAgICByZXR1cm4gVmFsaWRhdGlv'
        'blJlc3VsdCgKICAgICAgICBwcm9qZWN0X2lkPW1hbmlmZXN0LnByb2plY3RfaWQsCiAgICAgICAg'
        'dmFsaWQ9bm90IGFueShpc3N1ZS5zZXZlcml0eSA9PSAiZXJyb3IiIGZvciBpc3N1ZSBpbiBpc3N1'
        'ZXMpLAogICAgICAgIGlzc3Vlcz1pc3N1ZXMsCiAgICApCgoKZGVmIF9oYXNfcmVhZG1lKHByb2pl'
        'Y3Rfcm9vdDogUGF0aCkgLT4gYm9vbDoKICAgIHJldHVybiBhbnkoKHByb2plY3Rfcm9vdCAvIG5h'
        'bWUpLmV4aXN0cygpIGZvciBuYW1lIGluICgiUkVBRE1FLm1kIiwgIlJFQURNRS5NRCIsICJyZWFk'
        'bWUubWQiKSkK'
    ),
    'olympus/memory/__init__.py': (
        'IiIiTWVtb3J5IHN5c3RlbS4iIiIK'
    ),
    'olympus/memory/store.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHNxbGl0ZTMKZnJvbSBw'
        'YXRobGliIGltcG9ydCBQYXRoCmZyb20gdGhyZWFkaW5nIGltcG9ydCBSTG9jawoKZnJvbSBweWRh'
        'bnRpYyBpbXBvcnQgRmllbGQKCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmlj'
        'dE1vZGVsCmZyb20gb2x5bXB1cy5jb3JlLnNxbGl0ZV9qb3VybmFsIGltcG9ydCBhcHBseV9kdXJh'
        'YmxlX2pvdXJuYWxfbW9kZQoKCmNsYXNzIE1lbW9yeVJlY29yZChTdHJpY3RNb2RlbCk6CiAgICBr'
        'aW5kOiBzdHIKICAgIGtleTogc3RyCiAgICB2YWx1ZTogc3RyCiAgICBzYWxpZW5jZTogZmxvYXQg'
        'PSAwLjUKICAgIHRhZ3M6IGxpc3Rbc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KQoK'
        'CmNsYXNzIE1lbW9yeVN0b3JlOgogICAgIiIiU2VyaWFsaXplIGluZGl2aWR1YWwgcHVibGljIG9w'
        'ZXJhdGlvbnMgb24gdGhlIHNoYXJlZCBTUUxpdGUgY29ubmVjdGlvbi4KCiAgICBEaXJlY3QgYWNj'
        'ZXNzIHRvIGBgY29ubmVjdGlvbmBgIG11c3QgYmUgZXh0ZXJuYWxseSBzeW5jaHJvbml6ZWQuIFRo'
        'aXMgbG9jawogICAgZG9lcyBub3QgbWFrZSBhIHNlcXVlbmNlIG9mIG1ldGhvZCBjYWxscyBvbmUg'
        'dHJhbnNhY3Rpb24gb3IgY29vcmRpbmF0ZQogICAgc2VwYXJhdGUgc3RvcmUgaW5zdGFuY2VzL3By'
        'b2Nlc3NlczsgU1FMaXRlIGhhbmRsZXMgdGhlaXIgZGF0YWJhc2UgbG9ja2luZy4KICAgICIiIgoK'
        'ICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwYXRoOiBQYXRoKSAtPiBOb25lOgogICAgICAgIHNlbGYu'
        'X2xvY2sgPSBSTG9jaygpCiAgICAgICAgc2VsZi5wYXRoID0gcGF0aAogICAgICAgIHBhdGgucGFy'
        'ZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBzZWxmLmNvbm5l'
        'Y3Rpb24gPSBzcWxpdGUzLmNvbm5lY3QocGF0aCwgdGltZW91dD0zMC4wLCBjaGVja19zYW1lX3Ro'
        'cmVhZD1GYWxzZSkKICAgICAgICBhcHBseV9kdXJhYmxlX2pvdXJuYWxfbW9kZShzZWxmLmNvbm5l'
        'Y3Rpb24pCiAgICAgICAgc2VsZi5jb25uZWN0aW9uLmV4ZWN1dGUoIlBSQUdNQSBzeW5jaHJvbm91'
        'cyA9IEZVTEwiKQogICAgICAgIHNlbGYuY29ubmVjdGlvbi5leGVjdXRlKAogICAgICAgICAgICAi'
        'IiIKICAgICAgICAgICAgQ1JFQVRFIFRBQkxFIElGIE5PVCBFWElTVFMgbWVtb3J5ICgKICAgICAg'
        'ICAgICAgICAgIGtpbmQgVEVYVCBOT1QgTlVMTCwKICAgICAgICAgICAgICAgIGtleSBURVhUIFBS'
        'SU1BUlkgS0VZLAogICAgICAgICAgICAgICAgdmFsdWUgVEVYVCBOT1QgTlVMTCwKICAgICAgICAg'
        'ICAgICAgIHNhbGllbmNlIFJFQUwgTk9UIE5VTEwsCiAgICAgICAgICAgICAgICB0YWdzIFRFWFQg'
        'Tk9UIE5VTEwKICAgICAgICAgICAgKQogICAgICAgICAgICAiIiIKICAgICAgICApCiAgICAgICAg'
        'c2VsZi5jb25uZWN0aW9uLmNvbW1pdCgpCgogICAgZGVmIGNsb3NlKHNlbGYpIC0+IE5vbmU6CiAg'
        'ICAgICAgd2l0aCBzZWxmLl9sb2NrOgogICAgICAgICAgICBzZWxmLmNvbm5lY3Rpb24uY2xvc2Uo'
        'KQoKICAgIGRlZiBfX2VudGVyX18oc2VsZikgLT4gTWVtb3J5U3RvcmU6CiAgICAgICAgcmV0dXJu'
        'IHNlbGYKCiAgICBkZWYgX19leGl0X18oc2VsZiwgKmFyZ3M6IG9iamVjdCkgLT4gTm9uZToKICAg'
        'ICAgICBzZWxmLmNsb3NlKCkKCiAgICBkZWYgcHV0KHNlbGYsIHJlY29yZDogTWVtb3J5UmVjb3Jk'
        'KSAtPiBOb25lOgogICAgICAgICMgSG9sZCB0aGUgbG9jayB0aHJvdWdoIGNvbW1pdC9yb2xsYmFj'
        'aywgbm90IGp1c3Qgc3RhdGVtZW50IGV4ZWN1dGlvbi4KICAgICAgICB3aXRoIHNlbGYuX2xvY2ss'
        'IHNlbGYuY29ubmVjdGlvbjoKICAgICAgICAgICAgc2VsZi5jb25uZWN0aW9uLmV4ZWN1dGUoCiAg'
        'ICAgICAgICAgICAgICAiUkVQTEFDRSBJTlRPIG1lbW9yeSAoa2luZCwga2V5LCB2YWx1ZSwgc2Fs'
        'aWVuY2UsIHRhZ3MpIFZBTFVFUyAoPywgPywgPywgPywgPykiLAogICAgICAgICAgICAgICAgKHJl'
        'Y29yZC5raW5kLCByZWNvcmQua2V5LCByZWNvcmQudmFsdWUsIHJlY29yZC5zYWxpZW5jZSwgIiwi'
        'LmpvaW4ocmVjb3JkLnRhZ3MpKSwKICAgICAgICAgICAgKQoKICAgIGRlZiBnZXQoc2VsZiwga2V5'
        'OiBzdHIpIC0+IE1lbW9yeVJlY29yZCB8IE5vbmU6CiAgICAgICAgd2l0aCBzZWxmLl9sb2NrOgog'
        'ICAgICAgICAgICByb3cgPSBzZWxmLmNvbm5lY3Rpb24uZXhlY3V0ZSgKICAgICAgICAgICAgICAg'
        'ICJTRUxFQ1Qga2luZCwga2V5LCB2YWx1ZSwgc2FsaWVuY2UsIHRhZ3MgRlJPTSBtZW1vcnkgV0hF'
        'UkUga2V5ID0gPyIsCiAgICAgICAgICAgICAgICAoa2V5LCksCiAgICAgICAgICAgICkuZmV0Y2hv'
        'bmUoKQogICAgICAgIGlmIHJvdyBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAg'
        'ICAgIHJldHVybiBNZW1vcnlSZWNvcmQoCiAgICAgICAgICAgIGtpbmQ9cm93WzBdLAogICAgICAg'
        'ICAgICBrZXk9cm93WzFdLAogICAgICAgICAgICB2YWx1ZT1yb3dbMl0sCiAgICAgICAgICAgIHNh'
        'bGllbmNlPXJvd1szXSwKICAgICAgICAgICAgdGFncz1baXRlbSBmb3IgaXRlbSBpbiByb3dbNF0u'
        'c3BsaXQoIiwiKSBpZiBpdGVtXSwKICAgICAgICApCgogICAgZGVmIHF1ZXJ5KHNlbGYsIGtpbmQ6'
        'IHN0cikgLT4gbGlzdFtNZW1vcnlSZWNvcmRdOgogICAgICAgIHdpdGggc2VsZi5fbG9jazoKICAg'
        'ICAgICAgICAgcm93cyA9IHNlbGYuY29ubmVjdGlvbi5leGVjdXRlKAogICAgICAgICAgICAgICAg'
        'KAogICAgICAgICAgICAgICAgICAgICJTRUxFQ1Qga2luZCwga2V5LCB2YWx1ZSwgc2FsaWVuY2Us'
        'IHRhZ3MgRlJPTSBtZW1vcnkgIgogICAgICAgICAgICAgICAgICAgICJXSEVSRSBraW5kID0gPyBP'
        'UkRFUiBCWSBzYWxpZW5jZSBERVNDIgogICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgICAg'
        'IChraW5kLCksCiAgICAgICAgICAgICkuZmV0Y2hhbGwoKQogICAgICAgIHJldHVybiBbCiAgICAg'
        'ICAgICAgIE1lbW9yeVJlY29yZCgKICAgICAgICAgICAgICAgIGtpbmQ9cm93WzBdLAogICAgICAg'
        'ICAgICAgICAga2V5PXJvd1sxXSwKICAgICAgICAgICAgICAgIHZhbHVlPXJvd1syXSwKICAgICAg'
        'ICAgICAgICAgIHNhbGllbmNlPXJvd1szXSwKICAgICAgICAgICAgICAgIHRhZ3M9W2l0ZW0gZm9y'
        'IGl0ZW0gaW4gcm93WzRdLnNwbGl0KCIsIikgaWYgaXRlbV0sCiAgICAgICAgICAgICkKICAgICAg'
        'ICAgICAgZm9yIHJvdyBpbiByb3dzCiAgICAgICAgXQo='
    ),
    'olympus/models/__init__.py': (
        'IiIiTGF6aWx5IGV4cG9ydGVkIG1vZGVsIGltcGxlbWVudGF0aW9uczsgbWV0YWRhdGEgaW1wb3J0'
        'cyBuZXZlciBsb2FkIFRvcmNoLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9u'
        'cwoKZnJvbSBpbXBvcnRsaWIgaW1wb3J0IGltcG9ydF9tb2R1bGUKZnJvbSB0eXBpbmcgaW1wb3J0'
        'IFRZUEVfQ0hFQ0tJTkcsIEFueQoKaWYgVFlQRV9DSEVDS0lORzoKICAgIGZyb20gb2x5bXB1cy5t'
        'b2RlbHMuY2FwYWNpdHkgaW1wb3J0IENhcGFjaXR5RXN0aW1hdGUsIGVzdGltYXRlX2NhcGFjaXR5'
        'CiAgICBmcm9tIG9seW1wdXMubW9kZWxzLmVkaXRjbGV2ciBpbXBvcnQgKAogICAgICAgIEVESVRf'
        'RkFDVE9SUywKICAgICAgICBGVVNJT05fTU9ERVMsCiAgICAgICAgRWRpdENMRVZST3V0cHV0LAog'
        'ICAgICAgIE9seW1wdXNFZGl0Q0xFVlJNb2RlbCwKICAgICAgICBlZGl0Y2xldnJfbG9zcywKICAg'
        'ICkKICAgIGZyb20gb2x5bXB1cy5tb2RlbHMuZmxhZ3NoaXAgaW1wb3J0ICgKICAgICAgICBBaW9u'
        'RmxhZ3NoaXBNb2RlbCwKICAgICAgICBBdGxhc0ZsYWdzaGlwTW9kZWwsCiAgICAgICAgSGVybWVz'
        'RmxhZ3NoaXBNb2RlbCwKICAgICAgICBLcm9ub3NGbGFnc2hpcE1vZGVsLAogICAgICAgIE9seW1w'
        'dXNGbGFnc2hpcFN5c3RlbSwKICAgICAgICBQZXJzZXVzRmxhZ3NoaXBNb2RlbCwKICAgICAgICBQ'
        'cm9tZXRoZXVzRmxhZ3NoaXBNb2RlbCwKICAgICAgICBkZWZhdWx0X2ZsYWdzaGlwX2NvbmZpZywK'
        'ICAgICkKICAgIGZyb20gb2x5bXB1cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0gaW1wb3J0ICgKICAg'
        'ICAgICBGQU1JTFlfVkVDVE9SX05BTUVTLAogICAgICAgIE9seW1wdXNSZWZlcmVuY2VTeXN0ZW0s'
        'CiAgICAgICAgT2x5bXB1c1N5c3RlbUNvbmZpZywKICAgICAgICBPbHltcHVzU3lzdGVtTG9zcywK'
        'ICAgICAgICBPbHltcHVzU3lzdGVtT3V0cHV0LAogICAgICAgIGJ1aWxkX29seW1wdXNfc3lzdGVt'
        'LAogICAgICAgIG9seW1wdXNfc3lzdGVtX2xvc3MsCiAgICApCiAgICBmcm9tIG9seW1wdXMubW9k'
        'ZWxzLnBhcmFtZXRlcl9ncm93dGggaW1wb3J0ICgKICAgICAgICBHcm93dGhSZWNlaXB0LAogICAg'
        'ICAgIEpFUEFQYXJhbWV0ZXJHcm93dGgsCiAgICAgICAgd2lkZW5fZmVlZGZvcndhcmQsCiAgICAp'
        'CiAgICBmcm9tIG9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyBpbXBvcnQgKAogICAg'
        'ICAgIEFpb25SZWZlcmVuY2VDb25maWcsCiAgICAgICAgQWlvblJlZmVyZW5jZU1vZGVsLAogICAg'
        'ICAgIEF0bGFzUmVmZXJlbmNlQ29uZmlnLAogICAgICAgIEF0bGFzUmVmZXJlbmNlTW9kZWwsCiAg'
        'ICAgICAgSGVybWVzUmVmZXJlbmNlQ29uZmlnLAogICAgICAgIEhlcm1lc1JlZmVyZW5jZU1vZGVs'
        'LAogICAgICAgIEtyb25vc1JlZmVyZW5jZUNvbmZpZywKICAgICAgICBLcm9ub3NSZWZlcmVuY2VN'
        'b2RlbCwKICAgICAgICBQZXJzZXVzUmVmZXJlbmNlQ29uZmlnLAogICAgICAgIFBlcnNldXNSZWZl'
        'cmVuY2VNb2RlbCwKICAgICAgICBQcm9tZXRoZXVzUmVmZXJlbmNlQ29uZmlnLAogICAgICAgIFBy'
        'b21ldGhldXNSZWZlcmVuY2VNb2RlbCwKICAgICAgICBSZWZlcmVuY2VUcmFuc2Zvcm1lckNvbmZp'
        'ZywKICAgICAgICBidWlsZF9yZWZlcmVuY2VfbW9kZWwsCiAgICAgICAgZGVmYXVsdF9yZWZlcmVu'
        'Y2VfY29uZmlncywKICAgICkKICAgIGZyb20gb2x5bXB1cy5tb2RlbHMucmVsaWFiaWxpdHlfZnVz'
        'aW9uIGltcG9ydCAoCiAgICAgICAgUmVsaWFiaWxpdHlBd2FyZU9seW1wdXNTeXN0ZW0sCiAgICAg'
        'ICAgUmVsaWFiaWxpdHlGdXNpb25Db25maWcsCiAgICAgICAgUmVsaWFiaWxpdHlGdXNpb25Mb3Nz'
        'LAogICAgICAgIFJlbGlhYmlsaXR5RnVzaW9uT3V0cHV0LAogICAgICAgIFJlbGlhYmlsaXR5TG9z'
        'c0NvbmZpZywKICAgICAgICByZWxpYWJpbGl0eV9mdXNpb25fbG9zcywKICAgICkKICAgIGZyb20g'
        'b2x5bXB1cy5tb2RlbHMucnVudGltZSBpbXBvcnQgKAogICAgICAgIE9seW1wdXNDaGVja3BvaW50'
        'UmVjb3JkLAogICAgICAgIE9seW1wdXNJbmZlcmVuY2VSZXN1bHQsCiAgICAgICAgT2x5bXB1c1J1'
        'bnRpbWVDb25maWcsCiAgICAgICAgT2x5bXB1c1N5c3RlbVJ1bnRpbWUsCiAgICAgICAgT2x5bXB1'
        'c1RyYWluU3RlcFJlc3VsdCwKICAgICAgICBsb2FkX29seW1wdXNfY2hlY2twb2ludCwKICAgICAg'
        'ICBzYXZlX29seW1wdXNfY2hlY2twb2ludCwKICAgICAgICB0cmFpbl9vbHltcHVzX3N0ZXAsCiAg'
        'ICApCiAgICBmcm9tIG9seW1wdXMubW9kZWxzLnNldmVuX21vZGVsX2NvbnRyYWN0IGltcG9ydCAo'
        'CiAgICAgICAgU0VWRU5fTU9ERUxfSURTLAogICAgICAgIFNldmVuTW9kZWxFbmdpbmVlcmluZ1N0'
        'YXR1cywKICAgICAgICBTZXZlbk1vZGVsSW52ZW50b3J5U3RhdHVzLAogICAgICAgIGF1ZGl0X3Nl'
        'dmVuX21vZGVsX2NvbnRyYWN0cywKICAgICAgICBidWlsZF9zZXZlbl9tb2RlbF9yZWZlcmVuY2Us'
        'CiAgICAgICAgZGVmYXVsdF9zZXZlbl9tb2RlbF9jb25maWdzLAogICAgICAgIGluc3BlY3Rfc2V2'
        'ZW5fbW9kZWxfcmVmZXJlbmNlcywKICAgICkKCl9FWFBPUlRfTU9EVUxFUyA9IHsKICAgICJFRElU'
        'X0ZBQ1RPUlMiOiAib2x5bXB1cy5tb2RlbHMuZWRpdGNsZXZyIiwKICAgICJGVVNJT05fTU9ERVMi'
        'OiAib2x5bXB1cy5tb2RlbHMuZWRpdGNsZXZyIiwKICAgICJFZGl0Q0xFVlJPdXRwdXQiOiAib2x5'
        'bXB1cy5tb2RlbHMuZWRpdGNsZXZyIiwKICAgICJPbHltcHVzRWRpdENMRVZSTW9kZWwiOiAib2x5'
        'bXB1cy5tb2RlbHMuZWRpdGNsZXZyIiwKICAgICJlZGl0Y2xldnJfbG9zcyI6ICJvbHltcHVzLm1v'
        'ZGVscy5lZGl0Y2xldnIiLAogICAgIkZBTUlMWV9WRUNUT1JfTkFNRVMiOiAib2x5bXB1cy5tb2Rl'
        'bHMub2x5bXB1c19zeXN0ZW0iLAogICAgIk9seW1wdXNSZWZlcmVuY2VTeXN0ZW0iOiAib2x5bXB1'
        'cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0iLAogICAgIk9seW1wdXNTeXN0ZW1Db25maWciOiAib2x5'
        'bXB1cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0iLAogICAgIk9seW1wdXNTeXN0ZW1Mb3NzIjogIm9s'
        'eW1wdXMubW9kZWxzLm9seW1wdXNfc3lzdGVtIiwKICAgICJPbHltcHVzU3lzdGVtT3V0cHV0Ijog'
        'Im9seW1wdXMubW9kZWxzLm9seW1wdXNfc3lzdGVtIiwKICAgICJvbHltcHVzX3N5c3RlbV9sb3Nz'
        'IjogIm9seW1wdXMubW9kZWxzLm9seW1wdXNfc3lzdGVtIiwKICAgICJBaW9uUmVmZXJlbmNlQ29u'
        'ZmlnIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyIsCiAgICAiQWlvblJlZmVy'
        'ZW5jZU1vZGVsIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyIsCiAgICAiQXRs'
        'YXNSZWZlcmVuY2VDb25maWciOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwK'
        'ICAgICJBdGxhc1JlZmVyZW5jZU1vZGVsIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1p'
        'bGllcyIsCiAgICAiSGVybWVzUmVmZXJlbmNlQ29uZmlnIjogIm9seW1wdXMubW9kZWxzLnJlZmVy'
        'ZW5jZV9mYW1pbGllcyIsCiAgICAiSGVybWVzUmVmZXJlbmNlTW9kZWwiOiAib2x5bXB1cy5tb2Rl'
        'bHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJLcm9ub3NSZWZlcmVuY2VDb25maWciOiAib2x5'
        'bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJLcm9ub3NSZWZlcmVuY2VNb2Rl'
        'bCI6ICJvbHltcHVzLm1vZGVscy5yZWZlcmVuY2VfZmFtaWxpZXMiLAogICAgIlBlcnNldXNSZWZl'
        'cmVuY2VDb25maWciOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJQ'
        'ZXJzZXVzUmVmZXJlbmNlTW9kZWwiOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVz'
        'IiwKICAgICJQcm9tZXRoZXVzUmVmZXJlbmNlQ29uZmlnIjogIm9seW1wdXMubW9kZWxzLnJlZmVy'
        'ZW5jZV9mYW1pbGllcyIsCiAgICAiUHJvbWV0aGV1c1JlZmVyZW5jZU1vZGVsIjogIm9seW1wdXMu'
        'bW9kZWxzLnJlZmVyZW5jZV9mYW1pbGllcyIsCiAgICAiUmVmZXJlbmNlVHJhbnNmb3JtZXJDb25m'
        'aWciOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJidWlsZF9yZWZl'
        'cmVuY2VfbW9kZWwiOiAib2x5bXB1cy5tb2RlbHMucmVmZXJlbmNlX2ZhbWlsaWVzIiwKICAgICJk'
        'ZWZhdWx0X3JlZmVyZW5jZV9jb25maWdzIjogIm9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1p'
        'bGllcyIsCiAgICAiUmVsaWFiaWxpdHlBd2FyZU9seW1wdXNTeXN0ZW0iOiAib2x5bXB1cy5tb2Rl'
        'bHMucmVsaWFiaWxpdHlfZnVzaW9uIiwKICAgICJSZWxpYWJpbGl0eUZ1c2lvbkNvbmZpZyI6ICJv'
        'bHltcHVzLm1vZGVscy5yZWxpYWJpbGl0eV9mdXNpb24iLAogICAgIlJlbGlhYmlsaXR5RnVzaW9u'
        'TG9zcyI6ICJvbHltcHVzLm1vZGVscy5yZWxpYWJpbGl0eV9mdXNpb24iLAogICAgIlJlbGlhYmls'
        'aXR5RnVzaW9uT3V0cHV0IjogIm9seW1wdXMubW9kZWxzLnJlbGlhYmlsaXR5X2Z1c2lvbiIsCiAg'
        'ICAiUmVsaWFiaWxpdHlMb3NzQ29uZmlnIjogIm9seW1wdXMubW9kZWxzLnJlbGlhYmlsaXR5X2Z1'
        'c2lvbiIsCiAgICAicmVsaWFiaWxpdHlfZnVzaW9uX2xvc3MiOiAib2x5bXB1cy5tb2RlbHMucmVs'
        'aWFiaWxpdHlfZnVzaW9uIiwKICAgICJPbHltcHVzQ2hlY2twb2ludFJlY29yZCI6ICJvbHltcHVz'
        'Lm1vZGVscy5ydW50aW1lIiwKICAgICJPbHltcHVzSW5mZXJlbmNlUmVzdWx0IjogIm9seW1wdXMu'
        'bW9kZWxzLnJ1bnRpbWUiLAogICAgIk9seW1wdXNSdW50aW1lQ29uZmlnIjogIm9seW1wdXMubW9k'
        'ZWxzLnJ1bnRpbWUiLAogICAgIk9seW1wdXNTeXN0ZW1SdW50aW1lIjogIm9seW1wdXMubW9kZWxz'
        'LnJ1bnRpbWUiLAogICAgIk9seW1wdXNUcmFpblN0ZXBSZXN1bHQiOiAib2x5bXB1cy5tb2RlbHMu'
        'cnVudGltZSIsCiAgICAibG9hZF9vbHltcHVzX2NoZWNrcG9pbnQiOiAib2x5bXB1cy5tb2RlbHMu'
        'cnVudGltZSIsCiAgICAic2F2ZV9vbHltcHVzX2NoZWNrcG9pbnQiOiAib2x5bXB1cy5tb2RlbHMu'
        'cnVudGltZSIsCiAgICAidHJhaW5fb2x5bXB1c19zdGVwIjogIm9seW1wdXMubW9kZWxzLnJ1bnRp'
        'bWUiLAogICAgIlNFVkVOX01PREVMX0lEUyI6ICJvbHltcHVzLm1vZGVscy5zZXZlbl9tb2RlbF9j'
        'b250cmFjdCIsCiAgICAiU2V2ZW5Nb2RlbEVuZ2luZWVyaW5nU3RhdHVzIjogIm9seW1wdXMubW9k'
        'ZWxzLnNldmVuX21vZGVsX2NvbnRyYWN0IiwKICAgICJTZXZlbk1vZGVsSW52ZW50b3J5U3RhdHVz'
        'IjogIm9seW1wdXMubW9kZWxzLnNldmVuX21vZGVsX2NvbnRyYWN0IiwKICAgICJhdWRpdF9zZXZl'
        'bl9tb2RlbF9jb250cmFjdHMiOiAib2x5bXB1cy5tb2RlbHMuc2V2ZW5fbW9kZWxfY29udHJhY3Qi'
        'LAogICAgImJ1aWxkX3NldmVuX21vZGVsX3JlZmVyZW5jZSI6ICJvbHltcHVzLm1vZGVscy5zZXZl'
        'bl9tb2RlbF9jb250cmFjdCIsCiAgICAiZGVmYXVsdF9zZXZlbl9tb2RlbF9jb25maWdzIjogIm9s'
        'eW1wdXMubW9kZWxzLnNldmVuX21vZGVsX2NvbnRyYWN0IiwKICAgICJpbnNwZWN0X3NldmVuX21v'
        'ZGVsX3JlZmVyZW5jZXMiOiAib2x5bXB1cy5tb2RlbHMuc2V2ZW5fbW9kZWxfY29udHJhY3QiLAog'
        'ICAgIkNhcGFjaXR5RXN0aW1hdGUiOiAib2x5bXB1cy5tb2RlbHMuY2FwYWNpdHkiLAogICAgImVz'
        'dGltYXRlX2NhcGFjaXR5IjogIm9seW1wdXMubW9kZWxzLmNhcGFjaXR5IiwKICAgICJBaW9uRmxh'
        'Z3NoaXBNb2RlbCI6ICJvbHltcHVzLm1vZGVscy5mbGFnc2hpcCIsCiAgICAiQXRsYXNGbGFnc2hp'
        'cE1vZGVsIjogIm9seW1wdXMubW9kZWxzLmZsYWdzaGlwIiwKICAgICJIZXJtZXNGbGFnc2hpcE1v'
        'ZGVsIjogIm9seW1wdXMubW9kZWxzLmZsYWdzaGlwIiwKICAgICJLcm9ub3NGbGFnc2hpcE1vZGVs'
        'IjogIm9seW1wdXMubW9kZWxzLmZsYWdzaGlwIiwKICAgICJPbHltcHVzRmxhZ3NoaXBTeXN0ZW0i'
        'OiAib2x5bXB1cy5tb2RlbHMuZmxhZ3NoaXAiLAogICAgIlBlcnNldXNGbGFnc2hpcE1vZGVsIjog'
        'Im9seW1wdXMubW9kZWxzLmZsYWdzaGlwIiwKICAgICJQcm9tZXRoZXVzRmxhZ3NoaXBNb2RlbCI6'
        'ICJvbHltcHVzLm1vZGVscy5mbGFnc2hpcCIsCiAgICAiZGVmYXVsdF9mbGFnc2hpcF9jb25maWci'
        'OiAib2x5bXB1cy5tb2RlbHMuZmxhZ3NoaXAiLAogICAgIkpFUEFQYXJhbWV0ZXJHcm93dGgiOiAi'
        'b2x5bXB1cy5tb2RlbHMucGFyYW1ldGVyX2dyb3d0aCIsCiAgICAiR3Jvd3RoUmVjZWlwdCI6ICJv'
        'bHltcHVzLm1vZGVscy5wYXJhbWV0ZXJfZ3Jvd3RoIiwKICAgICJ3aWRlbl9mZWVkZm9yd2FyZCI6'
        'ICJvbHltcHVzLm1vZGVscy5wYXJhbWV0ZXJfZ3Jvd3RoIiwKICAgICJidWlsZF9vbHltcHVzX3N5'
        'c3RlbSI6ICJvbHltcHVzLm1vZGVscy5vbHltcHVzX3N5c3RlbSIsCn0KCl9fYWxsX18gPSBbCiAg'
        'ICAiRURJVF9GQUNUT1JTIiwKICAgICJGVVNJT05fTU9ERVMiLAogICAgIkVkaXRDTEVWUk91dHB1'
        'dCIsCiAgICAiRkFNSUxZX1ZFQ1RPUl9OQU1FUyIsCiAgICAiT2x5bXB1c1JlZmVyZW5jZVN5c3Rl'
        'bSIsCiAgICAiT2x5bXB1c1N5c3RlbUNvbmZpZyIsCiAgICAiT2x5bXB1c1N5c3RlbUxvc3MiLAog'
        'ICAgIk9seW1wdXNTeXN0ZW1PdXRwdXQiLAogICAgIk9seW1wdXNDaGVja3BvaW50UmVjb3JkIiwK'
        'ICAgICJPbHltcHVzSW5mZXJlbmNlUmVzdWx0IiwKICAgICJPbHltcHVzRWRpdENMRVZSTW9kZWwi'
        'LAogICAgIk9seW1wdXNSdW50aW1lQ29uZmlnIiwKICAgICJPbHltcHVzU3lzdGVtUnVudGltZSIs'
        'CiAgICAiT2x5bXB1c1RyYWluU3RlcFJlc3VsdCIsCiAgICAiUmVsaWFiaWxpdHlBd2FyZU9seW1w'
        'dXNTeXN0ZW0iLAogICAgIlJlbGlhYmlsaXR5RnVzaW9uQ29uZmlnIiwKICAgICJSZWxpYWJpbGl0'
        'eUZ1c2lvbkxvc3MiLAogICAgIlJlbGlhYmlsaXR5RnVzaW9uT3V0cHV0IiwKICAgICJSZWxpYWJp'
        'bGl0eUxvc3NDb25maWciLAogICAgIlNFVkVOX01PREVMX0lEUyIsCiAgICAiU2V2ZW5Nb2RlbEVu'
        'Z2luZWVyaW5nU3RhdHVzIiwKICAgICJTZXZlbk1vZGVsSW52ZW50b3J5U3RhdHVzIiwKICAgICJB'
        'aW9uUmVmZXJlbmNlQ29uZmlnIiwKICAgICJBaW9uUmVmZXJlbmNlTW9kZWwiLAogICAgIkF0bGFz'
        'UmVmZXJlbmNlQ29uZmlnIiwKICAgICJBdGxhc1JlZmVyZW5jZU1vZGVsIiwKICAgICJIZXJtZXNS'
        'ZWZlcmVuY2VDb25maWciLAogICAgIkhlcm1lc1JlZmVyZW5jZU1vZGVsIiwKICAgICJLcm9ub3NS'
        'ZWZlcmVuY2VDb25maWciLAogICAgIktyb25vc1JlZmVyZW5jZU1vZGVsIiwKICAgICJQZXJzZXVz'
        'UmVmZXJlbmNlQ29uZmlnIiwKICAgICJQZXJzZXVzUmVmZXJlbmNlTW9kZWwiLAogICAgIlByb21l'
        'dGhldXNSZWZlcmVuY2VDb25maWciLAogICAgIlByb21ldGhldXNSZWZlcmVuY2VNb2RlbCIsCiAg'
        'ICAiUmVmZXJlbmNlVHJhbnNmb3JtZXJDb25maWciLAogICAgImJ1aWxkX3JlZmVyZW5jZV9tb2Rl'
        'bCIsCiAgICAiYnVpbGRfc2V2ZW5fbW9kZWxfcmVmZXJlbmNlIiwKICAgICJhdWRpdF9zZXZlbl9t'
        'b2RlbF9jb250cmFjdHMiLAogICAgImRlZmF1bHRfcmVmZXJlbmNlX2NvbmZpZ3MiLAogICAgImRl'
        'ZmF1bHRfc2V2ZW5fbW9kZWxfY29uZmlncyIsCiAgICAiaW5zcGVjdF9zZXZlbl9tb2RlbF9yZWZl'
        'cmVuY2VzIiwKICAgICJlZGl0Y2xldnJfbG9zcyIsCiAgICAibG9hZF9vbHltcHVzX2NoZWNrcG9p'
        'bnQiLAogICAgIm9seW1wdXNfc3lzdGVtX2xvc3MiLAogICAgInJlbGlhYmlsaXR5X2Z1c2lvbl9s'
        'b3NzIiwKICAgICJzYXZlX29seW1wdXNfY2hlY2twb2ludCIsCiAgICAidHJhaW5fb2x5bXB1c19z'
        'dGVwIiwKICAgICJDYXBhY2l0eUVzdGltYXRlIiwKICAgICJlc3RpbWF0ZV9jYXBhY2l0eSIsCiAg'
        'ICAiQWlvbkZsYWdzaGlwTW9kZWwiLAogICAgIkF0bGFzRmxhZ3NoaXBNb2RlbCIsCiAgICAiSGVy'
        'bWVzRmxhZ3NoaXBNb2RlbCIsCiAgICAiS3Jvbm9zRmxhZ3NoaXBNb2RlbCIsCiAgICAiT2x5bXB1'
        'c0ZsYWdzaGlwU3lzdGVtIiwKICAgICJQZXJzZXVzRmxhZ3NoaXBNb2RlbCIsCiAgICAiUHJvbWV0'
        'aGV1c0ZsYWdzaGlwTW9kZWwiLAogICAgImRlZmF1bHRfZmxhZ3NoaXBfY29uZmlnIiwKICAgICJK'
        'RVBBUGFyYW1ldGVyR3Jvd3RoIiwKICAgICJHcm93dGhSZWNlaXB0IiwKICAgICJ3aWRlbl9mZWVk'
        'Zm9yd2FyZCIsCiAgICAiYnVpbGRfb2x5bXB1c19zeXN0ZW0iLApdCgoKZGVmIF9fZ2V0YXR0cl9f'
        'KG5hbWU6IHN0cikgLT4gQW55OgogICAgbW9kdWxlID0gX0VYUE9SVF9NT0RVTEVTLmdldChuYW1l'
        'KQogICAgaWYgbW9kdWxlIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgQXR0cmlidXRlRXJyb3IoZiJt'
        'b2R1bGUge19fbmFtZV9fIXJ9IGhhcyBubyBhdHRyaWJ1dGUge25hbWUhcn0iKQogICAgdmFsdWUg'
        'PSBnZXRhdHRyKGltcG9ydF9tb2R1bGUobW9kdWxlKSwgbmFtZSkKICAgIGdsb2JhbHMoKVtuYW1l'
        'XSA9IHZhbHVlCiAgICByZXR1cm4gdmFsdWUKCgpkZWYgX19kaXJfXygpIC0+IGxpc3Rbc3RyXToK'
        'ICAgIHJldHVybiBzb3J0ZWQoc2V0KGdsb2JhbHMoKSkgfCBzZXQoX19hbGxfXykpCg=='
    ),
    'olympus/models/capacity.py': (
        'IiIiQWxsb2NhdGlvbi1mcmVlIHBhcmFtZXRlciBhY2NvdW50aW5nOyBzcGFyc2UgYWN0aXZhdGlv'
        'biBuZXZlciByZW1vdmVzIGJhY2tpbmcgd2VpZ2h0cy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBv'
        'cnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGFzZGljdCwgZGF0YWNsYXNz'
        'CgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgQ2FwYWNpdHlFc3RpbWF0ZToKICAgIHRv'
        'dGFsX3BhcmFtZXRlcnM6IGludAogICAgYWN0aXZlX3BhcmFtZXRlcnM6IGludAogICAgcmVzaWRl'
        'bnRfcGFyYW1ldGVyczogaW50CiAgICB3ZWlnaHRfYml0czogaW50CiAgICBiYWNraW5nX3dlaWdo'
        'dF9ieXRlczogaW50CiAgICByZXNpZGVudF93ZWlnaHRfYnl0ZXM6IGludAogICAgYWN0aXZlX3dl'
        'aWdodF9ieXRlczogaW50CiAgICB0cmFpbmFibGVfc3RhdGVfYnl0ZXM6IGludAogICAgYWN0aXZh'
        'dGlvbl9hbmRfa3ZfcmVzZXJ2ZV9ieXRlczogaW50CiAgICByZXF1aXJlZF9kZXZpY2VfYnl0ZXM6'
        'IGludAogICAgZGV2aWNlX2J1ZGdldF9ieXRlczogaW50CiAgICBmaXRzX2RlY2xhcmVkX2Rldmlj'
        'ZV9idWRnZXQ6IGJvb2wKICAgIGJhY2tpbmdfc3RvcmFnZV9hdmFpbGFibGU6IGJvb2wKICAgIHJ1'
        'bm5hYmxlOiBib29sCgogICAgZGVmIGFzX2RpY3Qoc2VsZikgLT4gZGljdFtzdHIsIGludCB8IGJv'
        'b2xdOgogICAgICAgIHJldHVybiBhc2RpY3Qoc2VsZikKCgpkZWYgZXN0aW1hdGVfY2FwYWNpdHko'
        'CiAgICAqLAogICAgdG90YWxfcGFyYW1ldGVyczogaW50LAogICAgYWN0aXZlX3BhcmFtZXRlcnM6'
        'IGludCwKICAgIHJlc2lkZW50X3BhcmFtZXRlcnM6IGludCwKICAgIGRldmljZV9idWRnZXRfYnl0'
        'ZXM6IGludCwKICAgIGJhY2tpbmdfc3RvcmFnZV9ieXRlczogaW50LAogICAgd2VpZ2h0X2JpdHM6'
        'IGludCA9IDE2LAogICAgdHJhaW5hYmxlX3BhcmFtZXRlcnM6IGludCA9IDAsCiAgICB0cmFpbmFi'
        'bGVfYnl0ZXNfcGVyX3BhcmFtZXRlcjogaW50ID0gMTYsCiAgICBhY3RpdmF0aW9uX2FuZF9rdl9y'
        'ZXNlcnZlX2J5dGVzOiBpbnQgPSAwLAopIC0+IENhcGFjaXR5RXN0aW1hdGU6CiAgICAiIiJDb25z'
        'ZXJ2YXRpdmUgZGVjbGFyZWQgd2VpZ2h0L3N0YXRlIGJ1ZGdldCwgd2l0aG91dCBhbGxvY2F0aW5n'
        'IGFueSB0ZW5zb3IuCgogICAgVHJhaW5pbmcgc3RhdGUgYW5kIHJlc2VydmUgbXVzdCBiZSBzcGVj'
        'aWZpZWQgYnkgdGhlIGNhbGxpbmcgd29ya2xvYWQuIFF1YW50aXplcgogICAgbWV0YWRhdGEsIHRy'
        'YW5zZmVycywgY29sbGVjdGl2ZSBidWZmZXJzIGFuZCBhbGxvY2F0b3Igb3ZlcmhlYWQgbXVzdCBi'
        'ZSBpbmNsdWRlZAogICAgaW4gdGhhdCByZXNlcnZlOyB0aGlzIGNhbGN1bGF0aW9uIGlzIGEgbmVj'
        'ZXNzYXJ5IGFkbWlzc2lvbiB0ZXN0LCBub3QgYSBiZW5jaG1hcmsuCiAgICAiIiIKICAgIHZhbHVl'
        'cyA9ICgKICAgICAgICB0b3RhbF9wYXJhbWV0ZXJzLAogICAgICAgIGFjdGl2ZV9wYXJhbWV0ZXJz'
        'LAogICAgICAgIHJlc2lkZW50X3BhcmFtZXRlcnMsCiAgICAgICAgZGV2aWNlX2J1ZGdldF9ieXRl'
        'cywKICAgICAgICBiYWNraW5nX3N0b3JhZ2VfYnl0ZXMsCiAgICAgICAgdHJhaW5hYmxlX3BhcmFt'
        'ZXRlcnMsCiAgICAgICAgdHJhaW5hYmxlX2J5dGVzX3Blcl9wYXJhbWV0ZXIsCiAgICAgICAgYWN0'
        'aXZhdGlvbl9hbmRfa3ZfcmVzZXJ2ZV9ieXRlcywKICAgICkKICAgIGlmIGFueSh0eXBlKHZhbHVl'
        'KSBpcyBub3QgaW50IG9yIHZhbHVlIDwgMCBmb3IgdmFsdWUgaW4gdmFsdWVzKToKICAgICAgICBy'
        'YWlzZSBWYWx1ZUVycm9yKCJjYXBhY2l0eSBjb3VudHMgYW5kIGJ1ZGdldHMgbXVzdCBiZSBub25u'
        'ZWdhdGl2ZSBpbnRlZ2VycyIpCiAgICBpZiBub3QgMCA8IGFjdGl2ZV9wYXJhbWV0ZXJzIDw9IHJl'
        'c2lkZW50X3BhcmFtZXRlcnMgPD0gdG90YWxfcGFyYW1ldGVyczoKICAgICAgICByYWlzZSBWYWx1'
        'ZUVycm9yKCJyZXF1aXJlIDAgPCBhY3RpdmUgPD0gcmVzaWRlbnQgPD0gdG90YWwgcGFyYW1ldGVy'
        'cyIpCiAgICBpZiB3ZWlnaHRfYml0cyBub3QgaW4gezQsIDgsIDE2LCAzMn0gb3IgdHlwZSh3ZWln'
        'aHRfYml0cykgaXMgbm90IGludDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ3ZWlnaHQgYml0'
        'cyBtdXN0IGJlIDQsIDgsIDE2LCBvciAzMiIpCiAgICBpZiB0cmFpbmFibGVfcGFyYW1ldGVycyA+'
        'IHJlc2lkZW50X3BhcmFtZXRlcnMgb3IgdHJhaW5hYmxlX2J5dGVzX3Blcl9wYXJhbWV0ZXIgPCAx'
        'OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRyYWluYWJsZSBwYXJhbWV0ZXJzIG11c3QgZml0'
        'IHJlc2lkZW5jeSB3aXRoIHBvc2l0aXZlIHN0YXRlIGFjY291bnRpbmciKQogICAgYmFja2luZyA9'
        'ICh0b3RhbF9wYXJhbWV0ZXJzICogd2VpZ2h0X2JpdHMgKyA3KSAvLyA4CiAgICByZXNpZGVudCA9'
        'IChyZXNpZGVudF9wYXJhbWV0ZXJzICogd2VpZ2h0X2JpdHMgKyA3KSAvLyA4CiAgICBhY3RpdmUg'
        'PSAoYWN0aXZlX3BhcmFtZXRlcnMgKiB3ZWlnaHRfYml0cyArIDcpIC8vIDgKICAgIHRyYWluaW5n'
        'ID0gdHJhaW5hYmxlX3BhcmFtZXRlcnMgKiB0cmFpbmFibGVfYnl0ZXNfcGVyX3BhcmFtZXRlcgog'
        'ICAgcmVxdWlyZWQgPSByZXNpZGVudCArIHRyYWluaW5nICsgYWN0aXZhdGlvbl9hbmRfa3ZfcmVz'
        'ZXJ2ZV9ieXRlcwogICAgZml0cyA9IHJlcXVpcmVkIDw9IGRldmljZV9idWRnZXRfYnl0ZXMKICAg'
        'IHN0b3JlZCA9IGJhY2tpbmcgPD0gYmFja2luZ19zdG9yYWdlX2J5dGVzCiAgICByZXR1cm4gQ2Fw'
        'YWNpdHlFc3RpbWF0ZSgKICAgICAgICB0b3RhbF9wYXJhbWV0ZXJzLAogICAgICAgIGFjdGl2ZV9w'
        'YXJhbWV0ZXJzLAogICAgICAgIHJlc2lkZW50X3BhcmFtZXRlcnMsCiAgICAgICAgd2VpZ2h0X2Jp'
        'dHMsCiAgICAgICAgYmFja2luZywKICAgICAgICByZXNpZGVudCwKICAgICAgICBhY3RpdmUsCiAg'
        'ICAgICAgdHJhaW5pbmcsCiAgICAgICAgYWN0aXZhdGlvbl9hbmRfa3ZfcmVzZXJ2ZV9ieXRlcywK'
        'ICAgICAgICByZXF1aXJlZCwKICAgICAgICBkZXZpY2VfYnVkZ2V0X2J5dGVzLAogICAgICAgIGZp'
        'dHMsCiAgICAgICAgc3RvcmVkLAogICAgICAgIGZpdHMgYW5kIHN0b3JlZCwKICAgICkK'
    ),
    'olympus/models/conservative_routing_v6.py': (
        'IiIiRGlzam9pbnQtc3RhY2tlZCByb3V0aW5nIHdpdGggZXhwbGljaXQgcmVmZXJlbmNlIGludGVy'
        'dmVudGlvbiAodjYpLgoKVGhpcyBpcyBhIGJvdW5kZWQgcmVmZXJlbmNlIG1vZGVsLCBub3QgYSBz'
        'YWZldHkgb3IgaW1wcm92ZW1lbnQgZ3VhcmFudGVlLgpHcm91cCBpZGVudGlmaWVycyBhcmUgdXNl'
        'ZCBvbmx5IGJ5IHRoZSB0cmFpbmluZyBvYmplY3RpdmUsIG5ldmVyIGluZmVyZW5jZS4KIiIiCgpm'
        'cm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgY29weQoKaW1wb3J0IHRv'
        'cmNoCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvciwgbm4KCgpkZWYgdmFsaWRhdGVfcHJvYmFiaWxp'
        'dGllcyhwOiBUZW5zb3IpIC0+IE5vbmU6CiAgICBpZiBwLm5kaW0gPCAyIG9yIHAuc2hhcGVbMF0g'
        'PT0gMCBvciBwLnNoYXBlWy0xXSA8IDI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicHJvYmFi'
        'aWxpdGllcyByZXF1aXJlIG5vbmVtcHR5IGV4YW1wbGVzIGFuZCA+PTIgY2xhc3NlcyIpCiAgICBp'
        'ZiBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShwKS5hbGwoKSkgb3IgYm9vbCgocCA8IDApLmFueSgp'
        'KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcm9iYWJpbGl0aWVzIG11c3QgYmUgZmluaXRl'
        'IGFuZCBub25uZWdhdGl2ZSIpCiAgICBpZiBub3QgdG9yY2guYWxsY2xvc2UocC5zdW0oLTEpLCB0'
        'b3JjaC5vbmVzX2xpa2UocFsuLi4sIDBdKSwgYXRvbD0xZS01LCBydG9sPTApOgogICAgICAgIHJh'
        'aXNlIFZhbHVlRXJyb3IoInByb2JhYmlsaXRpZXMgbXVzdCBzdW0gdG8gb25lIikKCgpkZWYgYmxl'
        'bmQocmVmZXJlbmNlOiBUZW5zb3IsIGFsdGVybmF0aXZlOiBUZW5zb3IsIGFscGhhOiBUZW5zb3Ip'
        'IC0+IFRlbnNvcjoKICAgIHZhbGlkYXRlX3Byb2JhYmlsaXRpZXMocmVmZXJlbmNlKQogICAgdmFs'
        'aWRhdGVfcHJvYmFiaWxpdGllcyhhbHRlcm5hdGl2ZSkKICAgIGlmIHJlZmVyZW5jZS5zaGFwZSAh'
        'PSBhbHRlcm5hdGl2ZS5zaGFwZSBvciBhbHBoYS5zaGFwZSAhPSByZWZlcmVuY2Uuc2hhcGVbOi0x'
        'XSArICgxLCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVmZXJlbmNlL2FsdGVybmF0aXZl'
        'L2FscGhhIHNoYXBlcyBkaXNhZ3JlZSIpCiAgICBpZiBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShh'
        'bHBoYSkuYWxsKCkpIG9yIGJvb2woKChhbHBoYSA8IDApIHwgKGFscGhhID4gMSkpLmFueSgpKToK'
        'ICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJhbHBoYSBtdXN0IGJlIGZpbml0ZSBpbiBbMCwxXSIp'
        'CiAgICByZXR1cm4gKDEgLSBhbHBoYSkgKiByZWZlcmVuY2UgKyBhbHBoYSAqIGFsdGVybmF0aXZl'
        'CgoKZGVmIGZlYXR1cmVzKHo6IFRlbnNvciwgcHJvYmFiaWxpdGllczogVGVuc29yKSAtPiBUZW5z'
        'b3I6CiAgICBpZiB6Lm5kaW0gIT0gMyBvciB6LnNoYXBlWzE6XSAhPSAoNSwgMTYpIG9yIHByb2Jh'
        'YmlsaXRpZXMuc2hhcGUgIT0gKGxlbih6KSwgNSwgNCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJv'
        'cigiZXhwZWN0ZWQgW04sNSwxNl0gc3RhdGVzIGFuZCBbTiw1LDRdIHByb2JhYmlsaXRpZXMiKQog'
        'ICAgdmFsaWRhdGVfcHJvYmFiaWxpdGllcyhwcm9iYWJpbGl0aWVzKQogICAgaWYgbm90IGJvb2wo'
        'dG9yY2guaXNmaW5pdGUoeikuYWxsKCkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm5vbmZp'
        'bml0ZSBleHBlcnQgc3RhdGVzIikKICAgIGVudHJvcHkgPSAtKHByb2JhYmlsaXRpZXMgKiBwcm9i'
        'YWJpbGl0aWVzLmNsYW1wX21pbigxZS04KS5sb2coKSkuc3VtKC0xKQogICAgcmV0dXJuIHRvcmNo'
        'LmNhdCgoei5mbGF0dGVuKDEpLCBwcm9iYWJpbGl0aWVzLmZsYXR0ZW4oMSksIGVudHJvcHkpLCAt'
        'MSkuZGV0YWNoKCkKCgpkZWYgcm91dGluZ19uZXR3b3JrKCkgLT4gbm4uU2VxdWVudGlhbDoKICAg'
        'IHJldHVybiBubi5TZXF1ZW50aWFsKG5uLkxheWVyTm9ybSgxMDUpLCBubi5MaW5lYXIoMTA1LCAz'
        'MiksIG5uLkdFTFUoKSwgbm4uTGluZWFyKDMyLCA1KSkKCgpjbGFzcyBSZWZlcmVuY2VSb3V0ZXMo'
        'bm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIHN1cGVy'
        'KCkuX19pbml0X18oKQogICAgICAgIHNlbGYuc2VtYW50aWMgPSByb3V0aW5nX25ldHdvcmsoKQog'
        'ICAgICAgIHNlbGYuZml4ZWQgPSBubi5QYXJhbWV0ZXIodG9yY2guemVyb3MoNSkpCgogICAgZGVm'
        'IGZvcndhcmQoc2VsZiwgeDogVGVuc29yLCBleHBlcnRzOiBUZW5zb3IpIC0+IGRpY3Rbc3RyLCBU'
        'ZW5zb3JdOgogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICoqe2YiZXhwZXJ0X3tpfSI6IGV4'
        'cGVydHNbOiwgaV0gZm9yIGkgaW4gcmFuZ2UoNSl9LAogICAgICAgICAgICAidW5pZm9ybSI6IGV4'
        'cGVydHMubWVhbigxKSwKICAgICAgICAgICAgImZpeGVkIjogKHNlbGYuZml4ZWQuc29mdG1heCgt'
        'MSlbTm9uZSwgOiwgTm9uZV0gKiBleHBlcnRzKS5zdW0oMSksCiAgICAgICAgICAgICJzZW1hbnRp'
        'YyI6IChzZWxmLnNlbWFudGljKHgpLnNvZnRtYXgoLTEpWy4uLiwgTm9uZV0gKiBleHBlcnRzKS5z'
        'dW0oMSksCiAgICAgICAgfQoKCmNsYXNzIENvcnJlY3Rpb24obm4uTW9kdWxlKToKICAgICIiIlNo'
        'YXJlZCBhbHRlcm5hdGl2ZSBpbml0aWFsaXphdGlvbjsgb25seSB0aGUgZGVjbGFyZWQgY29ycmVj'
        'dGlvbiBkaWZmZXJzLgoKICAgIERpcmVjdCB2YXJpYW50cyBoYXZlIG5vIGludGVydmVudGlvbiBw'
        'YXJhbWV0ZXJzLiBDb25zdGFudCB2YXJpYW50cyBoYXZlIG9uZTsKICAgIGFkYXB0aXZlIHZhcmlh'
        'bnRzIGhhdmUgMTA2LiBDb3VudHMgYXJlIHJlcG9ydGVkIHJhdGhlciB0aGFuIHBhZGRlZCB3aXRo'
        'IGRlYWQKICAgIHBhcmFtZXRlcnMuIEEgY29uc3RhbnQgYmxlbmQgaXMgYSB0cmFpbmVkIGlucHV0'
        'LWluZGVwZW5kZW50IGFscGhhIGNvbnRyb2wuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2Vs'
        'ZiwgaW5pdGlhbDogbm4uU2VxdWVudGlhbCwga2luZDogc3RyLCBhbHBoYV9pbml0aWFsOiBmbG9h'
        'dCA9IDAuMSkgLT4gTm9uZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiBr'
        'aW5kIG5vdCBpbiB7ImRpcmVjdCIsICJhbmNob3JlZCIsICJjb25zdGFudCJ9IG9yIG5vdCAwIDwg'
        'YWxwaGFfaW5pdGlhbCA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImludmFsaWQg'
        'Y29ycmVjdGlvbiBraW5kIG9yIGFscGhhIGluaXRpYWxpemF0aW9uIikKICAgICAgICBzZWxmLmtp'
        'bmQgPSBraW5kCiAgICAgICAgc2VsZi5hbHRlcm5hdGl2ZSA9IGNvcHkuZGVlcGNvcHkoaW5pdGlh'
        'bCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgICAgICBpZiBraW5kID09ICJhbmNob3JlZCI6CiAg'
        'ICAgICAgICAgIHNlbGYuaW50ZXJ2ZW50aW9uOiBubi5Nb2R1bGUgPSBubi5MaW5lYXIoMTA1LCAx'
        'KQogICAgICAgICAgICBubi5pbml0Lnplcm9zXyhzZWxmLmludGVydmVudGlvbi53ZWlnaHQpCiAg'
        'ICAgICAgICAgIG5uLmluaXQuY29uc3RhbnRfKAogICAgICAgICAgICAgICAgc2VsZi5pbnRlcnZl'
        'bnRpb24uYmlhcywgZmxvYXQodG9yY2gubG9naXQodG9yY2gudGVuc29yKGFscGhhX2luaXRpYWwp'
        'KSkKICAgICAgICAgICAgKQogICAgICAgIGVsaWYga2luZCA9PSAiY29uc3RhbnQiOgogICAgICAg'
        'ICAgICBzZWxmLmludGVydmVudGlvbiA9IG5uLkxpbmVhcigxLCAxLCBiaWFzPUZhbHNlKQogICAg'
        'ICAgICAgICBubi5pbml0LmNvbnN0YW50XygKICAgICAgICAgICAgICAgIHNlbGYuaW50ZXJ2ZW50'
        'aW9uLndlaWdodCwgZmxvYXQodG9yY2gubG9naXQodG9yY2gudGVuc29yKGFscGhhX2luaXRpYWwp'
        'KSkKICAgICAgICAgICAgKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuaW50ZXJ2ZW50'
        'aW9uID0gbm4uSWRlbnRpdHkoKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IFRlbnNvciwgZXhw'
        'ZXJ0czogVGVuc29yLCByZWZlcmVuY2U6IFRlbnNvcikgLT4gdHVwbGVbVGVuc29yLCBUZW5zb3Jd'
        'OgogICAgICAgIGFsdGVybmF0aXZlID0gKHNlbGYuYWx0ZXJuYXRpdmUoeCkuc29mdG1heCgtMSlb'
        'Li4uLCBOb25lXSAqIGV4cGVydHMpLnN1bSgxKQogICAgICAgIGlmIHNlbGYua2luZCA9PSAiZGly'
        'ZWN0IjoKICAgICAgICAgICAgYWxwaGEgPSB0b3JjaC5vbmVzX2xpa2UocmVmZXJlbmNlWzosIDox'
        'XSkKICAgICAgICBlbGlmIHNlbGYua2luZCA9PSAiY29uc3RhbnQiOgogICAgICAgICAgICBhbHBo'
        'YSA9IHNlbGYuaW50ZXJ2ZW50aW9uKHRvcmNoLm9uZXNfbGlrZShyZWZlcmVuY2VbOiwgOjFdKSku'
        'c2lnbW9pZCgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgYWxwaGEgPSBzZWxmLmludGVydmVu'
        'dGlvbih4KS5zaWdtb2lkKCkKICAgICAgICByZXR1cm4gYmxlbmQocmVmZXJlbmNlLCBhbHRlcm5h'
        'dGl2ZSwgYWxwaGEpLCBhbHBoYQoKCmRlZiBjb3JyZWN0aW9uX2xvc3MoCiAgICBwcmVkaWN0aW9u'
        'czogVGVuc29yLAogICAgbGFiZWxzOiBUZW5zb3IsCiAgICBncm91cHM6IFRlbnNvciwKICAgIGFs'
        'cGhhOiBUZW5zb3IsCiAgICAqLAogICAgcm9idXN0OiBib29sLAogICAgcGVuYWx0eTogZmxvYXQs'
        'CiAgICBncm91cF9jb3VudDogaW50ID0gMywKKSAtPiBUZW5zb3I6CiAgICB2YWxpZGF0ZV9wcm9i'
        'YWJpbGl0aWVzKHByZWRpY3Rpb25zKQogICAgaWYgbGFiZWxzLnNoYXBlICE9IChsZW4ocHJlZGlj'
        'dGlvbnMpLCkgb3IgZ3JvdXBzLnNoYXBlICE9IGxhYmVscy5zaGFwZToKICAgICAgICByYWlzZSBW'
        'YWx1ZUVycm9yKCJsYWJlbHMvZ3JvdXBzIG11c3QgaGF2ZSBvbmUgZW50cnkgcGVyIHByZWRpY3Rp'
        'b24iKQogICAgaWYgbGFiZWxzLmR0eXBlICE9IHRvcmNoLmxvbmcgb3IgZ3JvdXBzLmR0eXBlICE9'
        'IHRvcmNoLmxvbmc6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW50ZWdlciBsYWJlbHMvZ3Jv'
        'dXBzIHJlcXVpcmVkIikKICAgIGlmIGJvb2woKChsYWJlbHMgPCAwKSB8IChsYWJlbHMgPj0gcHJl'
        'ZGljdGlvbnMuc2hhcGVbLTFdKSkuYW55KCkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNs'
        'YXNzIG91dCBvZiByYW5nZSIpCiAgICBpZiBncm91cF9jb3VudCA8IDEgb3IgYm9vbCgoKGdyb3Vw'
        'cyA8IDApIHwgKGdyb3VwcyA+PSBncm91cF9jb3VudCkpLmFueSgpKToKICAgICAgICByYWlzZSBW'
        'YWx1ZUVycm9yKCJncm91cCBvdXQgb2YgcmFuZ2UiKQogICAgaWYgcGVuYWx0eSA8IDAgb3Igbm90'
        'IF9faW1wb3J0X18oIm1hdGgiKS5pc2Zpbml0ZShwZW5hbHR5KToKICAgICAgICByYWlzZSBWYWx1'
        'ZUVycm9yKCJpbnZhbGlkIGFuY2hvciBwZW5hbHR5IikKICAgIGlmIGFscGhhLnNoYXBlICE9IChs'
        'ZW4ocHJlZGljdGlvbnMpLCAxKSBvciBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShhbHBoYSkuYWxs'
        'KCkpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImludmFsaWQgYWxwaGEiKQogICAgaWYgYm9v'
        'bCgoKGFscGhhIDwgMCkgfCAoYWxwaGEgPiAxKSkuYW55KCkpOgogICAgICAgIHJhaXNlIFZhbHVl'
        'RXJyb3IoImFscGhhIG91dCBvZiByYW5nZSIpCiAgICBsb3NzZXMgPSAtcHJlZGljdGlvbnMuZ2F0'
        'aGVyKDEsIGxhYmVsc1s6LCBOb25lXSkuY2xhbXBfbWluKDFlLTgpLmxvZygpLnNxdWVlemUoMSkK'
        'ICAgIHBlcl9ncm91cCA9IFtdCiAgICBmb3IgZ3JvdXAgaW4gcmFuZ2UoZ3JvdXBfY291bnQpOgog'
        'ICAgICAgIG1hc2sgPSBncm91cHMgPT0gZ3JvdXAKICAgICAgICBpZiBub3QgYm9vbChtYXNrLmFu'
        'eSgpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZXZlcnkgZGVjbGFyZWQgZ3JvdXAg'
        'bXVzdCBiZSBwcmVzZW50IikKICAgICAgICBwZXJfZ3JvdXAuYXBwZW5kKGxvc3Nlc1ttYXNrXS5t'
        'ZWFuKCkpCiAgICB2YWx1ZXMgPSB0b3JjaC5zdGFjayhwZXJfZ3JvdXApCiAgICBvYmplY3RpdmUg'
        'PSB2YWx1ZXMubWF4KCkgaWYgcm9idXN0IGVsc2UgdmFsdWVzLm1lYW4oKQogICAgcmV0dXJuIG9i'
        'amVjdGl2ZSArIHBlbmFsdHkgKiBhbHBoYS5tZWFuKCkKCgpkZWYgaW50ZXJ2ZW50aW9uX2RpYWdu'
        'b3N0aWNzKHA6IFRlbnNvciwgcmVmZXJlbmNlOiBUZW5zb3IsIHk6IFRlbnNvcikgLT4gZGljdFtz'
        'dHIsIGludCB8IGZsb2F0XToKICAgIGdvb2QsIHJlZl9nb29kID0gcC5hcmdtYXgoLTEpLmVxKHkp'
        'LCByZWZlcmVuY2UuYXJnbWF4KC0xKS5lcSh5KQogICAgcmVzY3VlcyA9IGludCgoZ29vZCAmIH5y'
        'ZWZfZ29vZCkuc3VtKCkpCiAgICBoYXJtcyA9IGludCgofmdvb2QgJiByZWZfZ29vZCkuc3VtKCkp'
        'CiAgICByZXR1cm4gewogICAgICAgICJleGFtcGxlcyI6IGxlbih5KSwKICAgICAgICAicmVzY3Vl'
        'cyI6IHJlc2N1ZXMsCiAgICAgICAgImhhcm1zIjogaGFybXMsCiAgICAgICAgImFjY3VyYWN5X2Rl'
        'bHRhIjogKHJlc2N1ZXMgLSBoYXJtcykgLyBsZW4oeSksCiAgICAgICAgInByZWRpY3Rpb25fY2hh'
        'bmdlcyI6IGludChwLmFyZ21heCgtMSkubmUocmVmZXJlbmNlLmFyZ21heCgtMSkpLnN1bSgpKSwK'
        'ICAgIH0K'
    ),
    'olympus/models/editclevr.py': (
        'IiIiUGFpcmVkLWltYWdlIE9seW1wdXMgbW9kZWwgZm9yIHRoZSBFZGl0Q0xFVlIgaW50ZXJ2ZW50'
        'aW9uIGJlbmNobWFyay4KClRoaXMgbW9kZWwgaXMgaW50ZW50aW9uYWxseSBuYXJyb3c6IGl0IHBy'
        'ZWRpY3RzIHdoaWNoIG9iamVjdCBmYWN0b3IgY2hhbmdlZCBpbgphIGJlZm9yZS9hZnRlciBpbWFn'
        'ZSBwYWlyLiBJdCBkb2VzIG5vdCBmYWJyaWNhdGUgbGFuZ3VhZ2UsIHRlbXBvcmFsLCByaXNrLCBv'
        'cgpnb3Zlcm5hbmNlIGlucHV0cyB0aGF0IEVkaXRDTEVWUiBkb2VzIG5vdCBjb250YWluLgoiIiIK'
        'CmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1w'
        'b3J0IGRhdGFjbGFzcwpmcm9tIHR5cGluZyBpbXBvcnQgY2FzdAoKaW1wb3J0IHRvcmNoCmZyb20g'
        'dG9yY2ggaW1wb3J0IFRlbnNvciwgbm4KCkVESVRfRkFDVE9SUyA9ICgibm9uZSIsICJjb2xvciIs'
        'ICJtYXRlcmlhbCIsICJzaXplIiwgInNoYXBlIikKRlVTSU9OX01PREVTID0gKCJsZWFybmVkIiwg'
        'InVuaWZvcm0iLCAicGFpcl9vbmx5IiwgImRlbHRhX29ubHkiKQoKCmNsYXNzIF9JbWFnZUVuY29k'
        'ZXIobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCB3aWR0aDogaW50KSAtPiBOb25l'
        'OgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYubmV0d29yayA9IG5uLlNl'
        'cXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYyZCgzLCB3aWR0aCwgNSwgc3RyaWRlPTIsIHBh'
        'ZGRpbmc9MiksCiAgICAgICAgICAgIG5uLkdyb3VwTm9ybSg0LCB3aWR0aCksCiAgICAgICAgICAg'
        'IG5uLkdFTFUoKSwKICAgICAgICAgICAgbm4uQ29udjJkKHdpZHRoLCB3aWR0aCAqIDIsIDMsIHN0'
        'cmlkZT0yLCBwYWRkaW5nPTEpLAogICAgICAgICAgICBubi5Hcm91cE5vcm0oOCwgd2lkdGggKiAy'
        'KSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5Db252MmQod2lkdGggKiAy'
        'LCB3aWR0aCAqIDQsIDMsIHN0cmlkZT0yLCBwYWRkaW5nPTEpLAogICAgICAgICAgICBubi5Hcm91'
        'cE5vcm0oOCwgd2lkdGggKiA0KSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBu'
        'bi5Db252MmQod2lkdGggKiA0LCB3aWR0aCAqIDQsIDMsIHN0cmlkZT0yLCBwYWRkaW5nPTEpLAog'
        'ICAgICAgICAgICBubi5Hcm91cE5vcm0oOCwgd2lkdGggKiA0KSwKICAgICAgICAgICAgbm4uR0VM'
        'VSgpLAogICAgICAgICAgICBubi5BZGFwdGl2ZUF2Z1Bvb2wyZCgxKSwKICAgICAgICAgICAgbm4u'
        'RmxhdHRlbigpLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBpbWFnZTogVGVuc29y'
        'KSAtPiBUZW5zb3I6CiAgICAgICAgcmV0dXJuIGNhc3QoVGVuc29yLCBzZWxmLm5ldHdvcmsoaW1h'
        'Z2UpKQoKCkBkYXRhY2xhc3Moc2xvdHM9VHJ1ZSkKY2xhc3MgRWRpdENMRVZST3V0cHV0OgogICAg'
        'cHJvYmFiaWxpdGllczogZGljdFtzdHIsIFRlbnNvcl0KICAgIGdhdGVzOiBUZW5zb3IKICAgIGV4'
        'cGVydF9sb2dpdHM6IFRlbnNvcgogICAgcGFpcl9lbWJlZGRpbmc6IFRlbnNvcgoKCmNsYXNzIE9s'
        'eW1wdXNFZGl0Q0xFVlJNb2RlbChubi5Nb2R1bGUpOgogICAgIiIiU2hhcmVkIGVuY29kZXJzIHBs'
        'dXMgZXhwbGljaXQgZXhwZXJ0cyBhbmQgYW4gb2JzZXJ2YWJsZSByb3V0ZXIuCgogICAgVGhlIGRp'
        'cmVjdCBwYWlyIGV4cGVydCBpcyBhIHN0cm9uZyBjb250cm9sLiBUaGUgbGVhcm5lZCByb3V0ZXIg'
        'bXVzdCBpbXByb3ZlCiAgICBvbiBpdCB1bmRlciBoZWxkLW91dCBoYXJkIGFuZCBDb0dlblQgc2hp'
        'ZnRzOyBiZWF0aW5nIGEgZGVsaWJlcmF0ZWx5IHdlYWsKICAgIHVuaWZvcm0gYXZlcmFnZSBpcyBp'
        'bnN1ZmZpY2llbnQgZm9yIHRoZSByZWdpc3RlcmVkIHByaW1hcnkuCiAgICAiIiIKCiAgICBkZWYg'
        'X19pbml0X18oc2VsZiwgKiwgd2lkdGg6IGludCA9IDMyLCBlbWJlZGRpbmdfZGltOiBpbnQgPSAx'
        'MjgpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgd2lkdGgg'
        'PCA4IG9yIHdpZHRoICUgNDoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigid2lkdGggbXVz'
        'dCBiZSBhdCBsZWFzdCA4IGFuZCBkaXZpc2libGUgYnkgNCIpCiAgICAgICAgaWYgZW1iZWRkaW5n'
        'X2RpbSA8IDMyOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJlbWJlZGRpbmdfZGltIG11'
        'c3QgYmUgYXQgbGVhc3QgMzIiKQogICAgICAgIGVuY29kZWQgPSB3aWR0aCAqIDQKICAgICAgICBz'
        'ZWxmLmltYWdlX2VuY29kZXIgPSBfSW1hZ2VFbmNvZGVyKHdpZHRoKQogICAgICAgIHNlbGYuZGVs'
        'dGFfZW5jb2RlciA9IF9JbWFnZUVuY29kZXIod2lkdGgpCiAgICAgICAgc2VsZi5wcm9qZWN0ID0g'
        'bm4uU2VxdWVudGlhbChubi5MYXllck5vcm0oZW5jb2RlZCksIG5uLkxpbmVhcihlbmNvZGVkLCBl'
        'bWJlZGRpbmdfZGltKSkKICAgICAgICBzZWxmLmV4cGVydF9oZWFkcyA9IG5uLk1vZHVsZUxpc3Qo'
        'CiAgICAgICAgICAgIG5uLkxpbmVhcihlbWJlZGRpbmdfZGltLCBsZW4oRURJVF9GQUNUT1JTKSkg'
        'Zm9yIF8gaW4gcmFuZ2UoNCkKICAgICAgICApCiAgICAgICAgcGFpcl93aWR0aCA9IGVtYmVkZGlu'
        'Z19kaW0gKiA0CiAgICAgICAgc2VsZi5wYWlyX25ldHdvcmsgPSBubi5TZXF1ZW50aWFsKAogICAg'
        'ICAgICAgICBubi5MYXllck5vcm0ocGFpcl93aWR0aCksCiAgICAgICAgICAgIG5uLkxpbmVhcihw'
        'YWlyX3dpZHRoLCBlbWJlZGRpbmdfZGltICogMiksCiAgICAgICAgICAgIG5uLkdFTFUoKSwKICAg'
        'ICAgICAgICAgbm4uRHJvcG91dCgwLjEpLAogICAgICAgICAgICBubi5MaW5lYXIoZW1iZWRkaW5n'
        'X2RpbSAqIDIsIGVtYmVkZGluZ19kaW0pLAogICAgICAgICkKICAgICAgICByb3V0ZXJfd2lkdGgg'
        'PSBlbWJlZGRpbmdfZGltICogNCArIGxlbihFRElUX0ZBQ1RPUlMpICogNCArIDQKICAgICAgICBz'
        'ZWxmLnJvdXRlciA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxheWVyTm9ybShyb3V0'
        'ZXJfd2lkdGgpLAogICAgICAgICAgICBubi5MaW5lYXIocm91dGVyX3dpZHRoLCBlbWJlZGRpbmdf'
        'ZGltKSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIoZW1iZWRk'
        'aW5nX2RpbSwgNCksCiAgICAgICAgKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGJlZm9yZTogVGVu'
        'c29yLCBhZnRlcjogVGVuc29yKSAtPiBFZGl0Q0xFVlJPdXRwdXQ6CiAgICAgICAgaWYgYmVmb3Jl'
        'LnNoYXBlICE9IGFmdGVyLnNoYXBlIG9yIGJlZm9yZS5uZGltICE9IDQgb3IgYmVmb3JlLnNoYXBl'
        'WzFdICE9IDM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJlZm9yZSBhbmQgYWZ0ZXIg'
        'bXVzdCBzaGFyZSBzaGFwZSBbYmF0Y2gsIDMsIGhlaWdodCwgd2lkdGhdIikKICAgICAgICBpZiBi'
        'ZWZvcmUuc2hhcGVbLTJdIDwgMzIgb3IgYmVmb3JlLnNoYXBlWy0xXSA8IDMyOgogICAgICAgICAg'
        'ICByYWlzZSBWYWx1ZUVycm9yKCJpbWFnZXMgbXVzdCBiZSBhdCBsZWFzdCAzMiBieSAzMiIpCiAg'
        'ICAgICAgaWYgbm90IGJvb2wodG9yY2guaXNmaW5pdGUoYmVmb3JlKS5hbGwoKSBhbmQgdG9yY2gu'
        'aXNmaW5pdGUoYWZ0ZXIpLmFsbCgpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiaW1h'
        'Z2VzIGNvbnRhaW4gbm9uLWZpbml0ZSB2YWx1ZXMiKQoKICAgICAgICBiZWZvcmVfeiA9IHNlbGYu'
        'cHJvamVjdChzZWxmLmltYWdlX2VuY29kZXIoYmVmb3JlKSkKICAgICAgICBhZnRlcl96ID0gc2Vs'
        'Zi5wcm9qZWN0KHNlbGYuaW1hZ2VfZW5jb2RlcihhZnRlcikpCiAgICAgICAgIyBJbnB1dHMgYXJl'
        'IG5vcm1hbGl6ZWQgdG8gWy0xLCAxXS4gS2VlcCBib3RoIGRlcml2ZWQgdmlld3MgaW4gYQogICAg'
        'ICAgICMgYm91bmRlZCByYW5nZSBzbyB0aGVpciBlbmNvZGVyIHJlY2VpdmVzIHRoZSBzYW1lIHNj'
        'YWxlIGFzIFJHQi4KICAgICAgICBzaWduZWRfeiA9IHNlbGYucHJvamVjdChzZWxmLmRlbHRhX2Vu'
        'Y29kZXIoKGFmdGVyIC0gYmVmb3JlKSAvIDIuMCkpCiAgICAgICAgYWJzb2x1dGVfeiA9IHNlbGYu'
        'cHJvamVjdChzZWxmLmRlbHRhX2VuY29kZXIoKGFmdGVyIC0gYmVmb3JlKS5hYnMoKSAvIDIuMCkp'
        'CiAgICAgICAgcGFpcl96ID0gc2VsZi5wYWlyX25ldHdvcmsodG9yY2guY2F0KChiZWZvcmVfeiwg'
        'YWZ0ZXJfeiwgc2lnbmVkX3osIGFic29sdXRlX3opLCBkaW09LTEpKQogICAgICAgIGVtYmVkZGlu'
        'Z3MgPSB0b3JjaC5zdGFjaygoYmVmb3JlX3osIGFmdGVyX3osIGFic29sdXRlX3osIHBhaXJfeiks'
        'IGRpbT0xKQogICAgICAgIGxvZ2l0cyA9IHRvcmNoLnN0YWNrKAogICAgICAgICAgICBbaGVhZChl'
        'bWJlZGRpbmdzWzosIGluZGV4XSkgZm9yIGluZGV4LCBoZWFkIGluIGVudW1lcmF0ZShzZWxmLmV4'
        'cGVydF9oZWFkcyldLCBkaW09MQogICAgICAgICkKICAgICAgICBleHBlcnRfcHJvYmFiaWxpdGll'
        'cyA9IGxvZ2l0cy5zb2Z0bWF4KGRpbT0tMSkKICAgICAgICBlbnRyb3B5ID0gLSgKICAgICAgICAg'
        'ICAgZXhwZXJ0X3Byb2JhYmlsaXRpZXMgKiBleHBlcnRfcHJvYmFiaWxpdGllcy5jbGFtcF9taW4o'
        'MWUtOCkubG9nKCkKICAgICAgICApLnN1bShkaW09LTEpCiAgICAgICAgZmVhdHVyZXMgPSB0b3Jj'
        'aC5jYXQoCiAgICAgICAgICAgIChlbWJlZGRpbmdzLmZsYXR0ZW4oMSksIGV4cGVydF9wcm9iYWJp'
        'bGl0aWVzLmZsYXR0ZW4oMSksIGVudHJvcHkpLCBkaW09LTEKICAgICAgICApCiAgICAgICAgZ2F0'
        'ZXMgPSBzZWxmLnJvdXRlcihmZWF0dXJlcykuc29mdG1heChkaW09LTEpCiAgICAgICAgcHJvYmFi'
        'aWxpdGllcyA9IHsKICAgICAgICAgICAgImxlYXJuZWQiOiAoZ2F0ZXNbLi4uLCBOb25lXSAqIGV4'
        'cGVydF9wcm9iYWJpbGl0aWVzKS5zdW0oZGltPTEpLAogICAgICAgICAgICAidW5pZm9ybSI6IGV4'
        'cGVydF9wcm9iYWJpbGl0aWVzLm1lYW4oZGltPTEpLAogICAgICAgICAgICAicGFpcl9vbmx5Ijog'
        'ZXhwZXJ0X3Byb2JhYmlsaXRpZXNbOiwgM10sCiAgICAgICAgICAgICJkZWx0YV9vbmx5IjogZXhw'
        'ZXJ0X3Byb2JhYmlsaXRpZXNbOiwgMl0sCiAgICAgICAgfQogICAgICAgIHJldHVybiBFZGl0Q0xF'
        'VlJPdXRwdXQoCiAgICAgICAgICAgIHByb2JhYmlsaXRpZXM9cHJvYmFiaWxpdGllcywKICAgICAg'
        'ICAgICAgZ2F0ZXM9Z2F0ZXMsCiAgICAgICAgICAgIGV4cGVydF9sb2dpdHM9bG9naXRzLAogICAg'
        'ICAgICAgICBwYWlyX2VtYmVkZGluZz1wYWlyX3osCiAgICAgICAgKQoKCmRlZiBlZGl0Y2xldnJf'
        'bG9zcyhvdXRwdXQ6IEVkaXRDTEVWUk91dHB1dCwgbGFiZWxzOiBUZW5zb3IpIC0+IFRlbnNvcjoK'
        'ICAgIGlmIGxhYmVscy5uZGltICE9IDEgb3IgbGFiZWxzLnNoYXBlWzBdICE9IG91dHB1dC5leHBl'
        'cnRfbG9naXRzLnNoYXBlWzBdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxhYmVscyBtdXN0'
        'IGhhdmUgc2hhcGUgW2JhdGNoXSIpCiAgICBpZiBsYWJlbHMuZHR5cGUgIT0gdG9yY2gubG9uZzoK'
        'ICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJsYWJlbHMgbXVzdCB1c2UgdG9yY2gubG9uZyIpCiAg'
        'ICBpZiBsYWJlbHMubnVtZWwoKSA9PSAwIG9yIGxhYmVscy5taW4oKSA8IDAgb3IgbGFiZWxzLm1h'
        'eCgpID49IGxlbihFRElUX0ZBQ1RPUlMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImxhYmVs'
        'cyBmYWxsIG91dHNpZGUgRWRpdENMRVZSIGNsYXNzZXMiKQogICAgIyBFdmVyeSBleHBlcnQgaXMg'
        'aW5kZXBlbmRlbnRseSB1c2VmdWw7IHRoZSByb3V0ZXIgY2Fubm90IG1hbnVmYWN0dXJlIGdhaW5z'
        'CiAgICAjIGJ5IHN0YXJ2aW5nIGEgY29udHJvbCBoZWFkIG9mIHN1cGVydmlzaW9uLgogICAgZXhw'
        'ZXJ0ID0gbm4uZnVuY3Rpb25hbC5jcm9zc19lbnRyb3B5KAogICAgICAgIG91dHB1dC5leHBlcnRf'
        'bG9naXRzLmZsYXR0ZW4oMCwgMSksIGxhYmVscy5yZXBlYXRfaW50ZXJsZWF2ZSg0KQogICAgKQog'
        'ICAgbGVhcm5lZCA9IG5uLmZ1bmN0aW9uYWwubmxsX2xvc3Mob3V0cHV0LnByb2JhYmlsaXRpZXNb'
        'ImxlYXJuZWQiXS5jbGFtcF9taW4oMWUtOCkubG9nKCksIGxhYmVscykKICAgIGJhbGFuY2UgPSAo'
        'b3V0cHV0LmdhdGVzLm1lYW4oZGltPTApIC0gMC4yNSkuc3F1YXJlKCkubWVhbigpCiAgICByZXR1'
        'cm4gZXhwZXJ0ICsgbGVhcm5lZCArIDAuMDEgKiBiYWxhbmNlCg=='
    ),
    'olympus/models/encoder_quantization.py': (
        'IiIiRGlmZmVyZW50aWFibGUgZW5jb2Rlci10by1yZXNpZHVhbC12ZWN0b3ItcXVhbnRpemF0aW9u'
        'IHBhdGh3YXkuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHR5'
        'cGluZyBpbXBvcnQgTmFtZWRUdXBsZSwgY2FzdAoKaW1wb3J0IHRvcmNoCmZyb20gcHlkYW50aWMg'
        'aW1wb3J0IEZpZWxkCmZyb20gdG9yY2ggaW1wb3J0IFRlbnNvciwgbm4KCmZyb20gb2x5bXB1cy5j'
        'b3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVsCgoKY2xhc3MgRW5jb2RlclF1YW50aXphdGlv'
        'bkNvbmZpZyhTdHJpY3RNb2RlbCk6CiAgICBpbnB1dF9kaW06IGludCA9IEZpZWxkKGdlPTEpCiAg'
        'ICBsYXRlbnRfZGltOiBpbnQgPSBGaWVsZChnZT0yKQogICAgaGlkZGVuX2RpbTogaW50ID0gRmll'
        'bGQoZGVmYXVsdD0yNTYsIGdlPTIpCiAgICBjb2RlYm9va19zaXplOiBpbnQgPSBGaWVsZChkZWZh'
        'dWx0PTI1NiwgZ2U9MiwgbGU9NjVfNTM2KQogICAgcmVzaWR1YWxfY29kZWJvb2tzOiBpbnQgPSBG'
        'aWVsZChkZWZhdWx0PTIsIGdlPTEsIGxlPTE2KQogICAgY29tbWl0bWVudF93ZWlnaHQ6IGZsb2F0'
        'ID0gRmllbGQoZGVmYXVsdD0wLjI1LCBnZT0wLjAsIGxlPTEwLjApCgoKY2xhc3MgUXVhbnRpemF0'
        'aW9uT3V0cHV0KE5hbWVkVHVwbGUpOgogICAgcXVhbnRpemVkOiBUZW5zb3IKICAgIGluZGljZXM6'
        'IFRlbnNvcgogICAgbG9zczogVGVuc29yCiAgICBjb21taXRtZW50X2xvc3M6IFRlbnNvcgogICAg'
        'Y29kZWJvb2tfbG9zczogVGVuc29yCiAgICBxdWFudGl6YXRpb25fZXJyb3I6IFRlbnNvcgogICAg'
        'dXRpbGl6YXRpb246IFRlbnNvcgoKCmNsYXNzIFJlc2lkdWFsVmVjdG9yUXVhbnRpemVyKG5uLk1v'
        'ZHVsZSk6CiAgICAiIiJSZXNpZHVhbCBWUSB3aXRoIG1hc2tlZCB0b2tlbnMgYW5kIGEgc3RyYWln'
        'aHQtdGhyb3VnaCBlbmNvZGVyIGdyYWRpZW50LiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBj'
        'b25maWc6IEVuY29kZXJRdWFudGl6YXRpb25Db25maWcpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIo'
        'KS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5jb25maWcgPSBjb25maWcKICAgICAgICBzZWxmLmNv'
        'ZGVib29rcyA9IG5uLlBhcmFtZXRlcigKICAgICAgICAgICAgdG9yY2guZW1wdHkoY29uZmlnLnJl'
        'c2lkdWFsX2NvZGVib29rcywgY29uZmlnLmNvZGVib29rX3NpemUsIGNvbmZpZy5sYXRlbnRfZGlt'
        'KQogICAgICAgICkKICAgICAgICBubi5pbml0LnVuaWZvcm1fKAogICAgICAgICAgICBzZWxmLmNv'
        'ZGVib29rcywKICAgICAgICAgICAgLTEuMCAvIGNvbmZpZy5jb2RlYm9va19zaXplLAogICAgICAg'
        'ICAgICAxLjAgLyBjb25maWcuY29kZWJvb2tfc2l6ZSwKICAgICAgICApCgogICAgZGVmIGZvcndh'
        'cmQoc2VsZiwgbGF0ZW50OiBUZW5zb3IsIG1hc2s6IFRlbnNvciB8IE5vbmUgPSBOb25lKSAtPiBR'
        'dWFudGl6YXRpb25PdXRwdXQ6CiAgICAgICAgaWYgbGF0ZW50Lm5kaW0gPCAyIG9yIGxhdGVudC5z'
        'aGFwZVstMV0gIT0gc2VsZi5jb25maWcubGF0ZW50X2RpbToKICAgICAgICAgICAgcmFpc2UgVmFs'
        'dWVFcnJvcihmImV4cGVjdGVkIFsuLi4sIHtzZWxmLmNvbmZpZy5sYXRlbnRfZGltfV0gbGF0ZW50'
        'IHRlbnNvciIpCiAgICAgICAgdG9rZW5fc2hhcGUgPSBsYXRlbnQuc2hhcGVbOi0xXQogICAgICAg'
        'IGZsYXQgPSBsYXRlbnQucmVzaGFwZSgtMSwgc2VsZi5jb25maWcubGF0ZW50X2RpbSkKICAgICAg'
        'ICBpZiBtYXNrIGlzIE5vbmU6CiAgICAgICAgICAgIGFjdGl2ZSA9IHRvcmNoLm9uZXMoZmxhdC5z'
        'aGFwZVswXSwgZHR5cGU9dG9yY2guYm9vbCwgZGV2aWNlPWZsYXQuZGV2aWNlKQogICAgICAgIGVs'
        'c2U6CiAgICAgICAgICAgIGlmIHR1cGxlKG1hc2suc2hhcGUpICE9IHR1cGxlKHRva2VuX3NoYXBl'
        'KToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1hc2sgc2hhcGUgbXVzdCBlcXVh'
        'bCB0aGUgbGF0ZW50IHRva2VuIGRpbWVuc2lvbnMiKQogICAgICAgICAgICBhY3RpdmUgPSBtYXNr'
        'LnRvKGRldmljZT1mbGF0LmRldmljZSwgZHR5cGU9dG9yY2guYm9vbCkucmVzaGFwZSgtMSkKICAg'
        'ICAgICBpZiBub3QgYm9vbChhY3RpdmUuYW55KCkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVy'
        'cm9yKCJhdCBsZWFzdCBvbmUgbGF0ZW50IHRva2VuIG11c3QgYmUgYWN0aXZlIikKCiAgICAgICAg'
        'cmVzaWR1YWwgPSBmbGF0CiAgICAgICAgcXVhbnRpemVkX3N1bSA9IHRvcmNoLnplcm9zX2xpa2Uo'
        'ZmxhdCkKICAgICAgICBpbmRpY2VzOiBsaXN0W1RlbnNvcl0gPSBbXQogICAgICAgIGNvbW1pdG1l'
        'bnRfdGVybXM6IGxpc3RbVGVuc29yXSA9IFtdCiAgICAgICAgY29kZWJvb2tfdGVybXM6IGxpc3Rb'
        'VGVuc29yXSA9IFtdCiAgICAgICAgdXRpbGl6YXRpb25fdGVybXM6IGxpc3RbVGVuc29yXSA9IFtd'
        'CgogICAgICAgIGZvciBjb2RlYm9vayBpbiBzZWxmLmNvZGVib29rczoKICAgICAgICAgICAgYWN0'
        'aXZlX3Jlc2lkdWFsID0gcmVzaWR1YWxbYWN0aXZlXQogICAgICAgICAgICBkaXN0YW5jZXMgPSAo'
        'CiAgICAgICAgICAgICAgICBhY3RpdmVfcmVzaWR1YWwuc3F1YXJlKCkuc3VtKGRpbT0xLCBrZWVw'
        'ZGltPVRydWUpCiAgICAgICAgICAgICAgICAtIDIuMCAqIGFjdGl2ZV9yZXNpZHVhbCBAIGNvZGVi'
        'b29rLnRyYW5zcG9zZSgwLCAxKQogICAgICAgICAgICAgICAgKyBjb2RlYm9vay5zcXVhcmUoKS5z'
        'dW0oZGltPTEpLnVuc3F1ZWV6ZSgwKQogICAgICAgICAgICApCiAgICAgICAgICAgIGFjdGl2ZV9p'
        'bmRpY2VzID0gZGlzdGFuY2VzLmFyZ21pbihkaW09MSkKICAgICAgICAgICAgc2VsZWN0ZWRfYWN0'
        'aXZlID0gY29kZWJvb2tbYWN0aXZlX2luZGljZXNdCiAgICAgICAgICAgIHNlbGVjdGVkID0gdG9y'
        'Y2guemVyb3NfbGlrZShmbGF0KQogICAgICAgICAgICBzZWxlY3RlZFthY3RpdmVdID0gc2VsZWN0'
        'ZWRfYWN0aXZlCiAgICAgICAgICAgIGZ1bGxfaW5kaWNlcyA9IHRvcmNoLmZ1bGwoKGZsYXQuc2hh'
        'cGVbMF0sKSwgLTEsIGR0eXBlPXRvcmNoLmxvbmcsIGRldmljZT1mbGF0LmRldmljZSkKICAgICAg'
        'ICAgICAgZnVsbF9pbmRpY2VzW2FjdGl2ZV0gPSBhY3RpdmVfaW5kaWNlcwogICAgICAgICAgICBp'
        'bmRpY2VzLmFwcGVuZChmdWxsX2luZGljZXMpCiAgICAgICAgICAgIGNvbW1pdG1lbnRfdGVybXMu'
        'YXBwZW5kKAogICAgICAgICAgICAgICAgbm4uZnVuY3Rpb25hbC5tc2VfbG9zcyhhY3RpdmVfcmVz'
        'aWR1YWwsIHNlbGVjdGVkX2FjdGl2ZS5kZXRhY2goKSkKICAgICAgICAgICAgKQogICAgICAgICAg'
        'ICBjb2RlYm9va190ZXJtcy5hcHBlbmQobm4uZnVuY3Rpb25hbC5tc2VfbG9zcyhzZWxlY3RlZF9h'
        'Y3RpdmUsIGFjdGl2ZV9yZXNpZHVhbC5kZXRhY2goKSkpCiAgICAgICAgICAgIHV0aWxpemF0aW9u'
        'X3Rlcm1zLmFwcGVuZCgKICAgICAgICAgICAgICAgIGFjdGl2ZV9pbmRpY2VzLnVuaXF1ZSgpLm51'
        'bWVsKCkKICAgICAgICAgICAgICAgIC8gdG9yY2gudGVuc29yKAogICAgICAgICAgICAgICAgICAg'
        'IHNlbGYuY29uZmlnLmNvZGVib29rX3NpemUsCiAgICAgICAgICAgICAgICAgICAgZHR5cGU9Zmxh'
        'dC5kdHlwZSwKICAgICAgICAgICAgICAgICAgICBkZXZpY2U9ZmxhdC5kZXZpY2UsCiAgICAgICAg'
        'ICAgICAgICApCiAgICAgICAgICAgICkKICAgICAgICAgICAgcXVhbnRpemVkX3N1bSA9IHF1YW50'
        'aXplZF9zdW0gKyBzZWxlY3RlZAogICAgICAgICAgICByZXNpZHVhbCA9IHJlc2lkdWFsIC0gc2Vs'
        'ZWN0ZWQuZGV0YWNoKCkKCiAgICAgICAgY29tbWl0bWVudCA9IHRvcmNoLnN0YWNrKGNvbW1pdG1l'
        'bnRfdGVybXMpLm1lYW4oKQogICAgICAgIGNvZGVib29rX2xvc3MgPSB0b3JjaC5zdGFjayhjb2Rl'
        'Ym9va190ZXJtcykubWVhbigpCiAgICAgICAgbG9zcyA9IGNvZGVib29rX2xvc3MgKyBzZWxmLmNv'
        'bmZpZy5jb21taXRtZW50X3dlaWdodCAqIGNvbW1pdG1lbnQKICAgICAgICBzdHJhaWdodF90aHJv'
        'dWdoID0gZmxhdCArIChxdWFudGl6ZWRfc3VtIC0gZmxhdCkuZGV0YWNoKCkKICAgICAgICBzdHJh'
        'aWdodF90aHJvdWdoID0gdG9yY2gud2hlcmUoYWN0aXZlLnVuc3F1ZWV6ZSgxKSwgc3RyYWlnaHRf'
        'dGhyb3VnaCwgZmxhdCkKICAgICAgICBlcnJvciA9IG5uLmZ1bmN0aW9uYWwubXNlX2xvc3MocXVh'
        'bnRpemVkX3N1bVthY3RpdmVdLCBmbGF0W2FjdGl2ZV0pCiAgICAgICAgcmV0dXJuIFF1YW50aXph'
        'dGlvbk91dHB1dCgKICAgICAgICAgICAgcXVhbnRpemVkPXN0cmFpZ2h0X3Rocm91Z2gucmVzaGFw'
        'ZSgqdG9rZW5fc2hhcGUsIHNlbGYuY29uZmlnLmxhdGVudF9kaW0pLAogICAgICAgICAgICBpbmRp'
        'Y2VzPXRvcmNoLnN0YWNrKGluZGljZXMsIGRpbT0tMSkucmVzaGFwZSgKICAgICAgICAgICAgICAg'
        'ICp0b2tlbl9zaGFwZSwgc2VsZi5jb25maWcucmVzaWR1YWxfY29kZWJvb2tzCiAgICAgICAgICAg'
        'ICksCiAgICAgICAgICAgIGxvc3M9bG9zcywKICAgICAgICAgICAgY29tbWl0bWVudF9sb3NzPWNv'
        'bW1pdG1lbnQsCiAgICAgICAgICAgIGNvZGVib29rX2xvc3M9Y29kZWJvb2tfbG9zcywKICAgICAg'
        'ICAgICAgcXVhbnRpemF0aW9uX2Vycm9yPWVycm9yLAogICAgICAgICAgICB1dGlsaXphdGlvbj10'
        'b3JjaC5zdGFjayh1dGlsaXphdGlvbl90ZXJtcykubWVhbigpLAogICAgICAgICkKCgpjbGFzcyBR'
        'dWFudGl6ZWRFbmNvZGVyKG5uLk1vZHVsZSk6CiAgICAiIiJUcmFpbmFibGUgZW5jb2RlciB3aG9z'
        'ZSBjb250aW51b3VzIHN0YXRlcyBwYXNzIHRocm91Z2ggcmVzaWR1YWwgVlEuIiIiCgogICAgZGVm'
        'IF9faW5pdF9fKHNlbGYsIGNvbmZpZzogRW5jb2RlclF1YW50aXphdGlvbkNvbmZpZykgLT4gTm9u'
        'ZToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZp'
        'ZwogICAgICAgIHNlbGYuZW5jb2RlciA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkxp'
        'bmVhcihjb25maWcuaW5wdXRfZGltLCBjb25maWcuaGlkZGVuX2RpbSksCiAgICAgICAgICAgIG5u'
        'LkdFTFUoKSwKICAgICAgICAgICAgbm4uTGluZWFyKGNvbmZpZy5oaWRkZW5fZGltLCBjb25maWcu'
        'bGF0ZW50X2RpbSksCiAgICAgICAgICAgIG5uLkxheWVyTm9ybShjb25maWcubGF0ZW50X2RpbSks'
        'CiAgICAgICAgKQogICAgICAgIHNlbGYucXVhbnRpemVyID0gUmVzaWR1YWxWZWN0b3JRdWFudGl6'
        'ZXIoY29uZmlnKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIGlucHV0czogVGVuc29yLCBtYXNrOiBU'
        'ZW5zb3IgfCBOb25lID0gTm9uZSkgLT4gUXVhbnRpemF0aW9uT3V0cHV0OgogICAgICAgIGlmIGlu'
        'cHV0cy5uZGltIDwgMiBvciBpbnB1dHMuc2hhcGVbLTFdICE9IHNlbGYuY29uZmlnLmlucHV0X2Rp'
        'bToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImV4cGVjdGVkIFsuLi4sIHtzZWxmLmNv'
        'bmZpZy5pbnB1dF9kaW19XSBlbmNvZGVyIGlucHV0IikKICAgICAgICByZXR1cm4gY2FzdChRdWFu'
        'dGl6YXRpb25PdXRwdXQsIHNlbGYucXVhbnRpemVyKHNlbGYuZW5jb2RlcihpbnB1dHMpLCBtYXNr'
        'PW1hc2spKQoKCmNsYXNzIFF1YW50aXplZEF1dG9lbmNvZGVyT3V0cHV0KE5hbWVkVHVwbGUpOgog'
        'ICAgcmVjb25zdHJ1Y3Rpb246IFRlbnNvcgogICAgcXVhbnRpemF0aW9uOiBRdWFudGl6YXRpb25P'
        'dXRwdXQKCgpjbGFzcyBRdWFudGl6ZWRBdXRvZW5jb2Rlcihubi5Nb2R1bGUpOgogICAgIiIiUmF0'
        'ZS1kaXN0b3J0aW9uIHRyYWluaW5nIHdyYXBwZXIgYXJvdW5kIHRoZSBlbmNvZGVyLXF1YW50aXph'
        'dGlvbiBwYXRod2F5LiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjb25maWc6IEVuY29kZXJR'
        'dWFudGl6YXRpb25Db25maWcpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAg'
        'ICAgICAgc2VsZi5jb25maWcgPSBjb25maWcKICAgICAgICBzZWxmLmVuY29kZXIgPSBRdWFudGl6'
        'ZWRFbmNvZGVyKGNvbmZpZykKICAgICAgICBzZWxmLmRlY29kZXIgPSBubi5TZXF1ZW50aWFsKAog'
        'ICAgICAgICAgICBubi5MaW5lYXIoY29uZmlnLmxhdGVudF9kaW0sIGNvbmZpZy5oaWRkZW5fZGlt'
        'KSwKICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICBubi5MaW5lYXIoY29uZmlnLmhp'
        'ZGRlbl9kaW0sIGNvbmZpZy5pbnB1dF9kaW0pLAogICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChz'
        'ZWxmLCBpbnB1dHM6IFRlbnNvciwgbWFzazogVGVuc29yIHwgTm9uZSA9IE5vbmUpIC0+IFF1YW50'
        'aXplZEF1dG9lbmNvZGVyT3V0cHV0OgogICAgICAgIHF1YW50aXphdGlvbiA9IHNlbGYuZW5jb2Rl'
        'cihpbnB1dHMsIG1hc2s9bWFzaykKICAgICAgICByZWNvbnN0cnVjdGlvbiA9IGNhc3QoVGVuc29y'
        'LCBzZWxmLmRlY29kZXIocXVhbnRpemF0aW9uLnF1YW50aXplZCkpCiAgICAgICAgcmV0dXJuIFF1'
        'YW50aXplZEF1dG9lbmNvZGVyT3V0cHV0KAogICAgICAgICAgICByZWNvbnN0cnVjdGlvbj1yZWNv'
        'bnN0cnVjdGlvbiwKICAgICAgICAgICAgcXVhbnRpemF0aW9uPXF1YW50aXphdGlvbiwKICAgICAg'
        'ICApCg=='
    ),
    'olympus/models/families.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSB0eXBpbmcgaW1wb3J0IFR5'
        'cGVkRGljdAoKZnJvbSBvbHltcHVzLmNvcmUuaW50ZXJwcmV0YXRpb24gaW1wb3J0IEludGVycHJl'
        'dGl2ZVN1cGVycG9zaXRpb25OZXR3b3JrCmZyb20gb2x5bXB1cy5tZW1vcnkuc3RvcmUgaW1wb3J0'
        'IE1lbW9yeVJlY29yZCwgTWVtb3J5U3RvcmUKCgpjbGFzcyBIZXJtZXNSZXNwb25zZShUeXBlZERp'
        'Y3QpOgogICAgcmVzcG9uc2U6IHN0cgogICAgY29uZmlkZW5jZTogZmxvYXQKICAgIGNhdGVnb3J5'
        'OiBzdHIKCgpjbGFzcyBIZXJtZXNOYW5vOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1lbW9yeTog'
        'TWVtb3J5U3RvcmUpIC0+IE5vbmU6CiAgICAgICAgc2VsZi5tZW1vcnkgPSBtZW1vcnkKICAgICAg'
        'ICBzZWxmLmludGVycHJldGVyID0gSW50ZXJwcmV0aXZlU3VwZXJwb3NpdGlvbk5ldHdvcmsoKQoK'
        'ICAgIGRlZiByZXNwb25kKHNlbGYsIHByb21wdDogc3RyKSAtPiBIZXJtZXNSZXNwb25zZToKICAg'
        'ICAgICBicmFuY2hlcyA9IHNlbGYuaW50ZXJwcmV0ZXIuYW5hbHl6ZShwcm9tcHQpCiAgICAgICAg'
        'YmVzdCA9IGJyYW5jaGVzWzBdCiAgICAgICAgc2VsZi5tZW1vcnkucHV0KAogICAgICAgICAgICBN'
        'ZW1vcnlSZWNvcmQoCiAgICAgICAgICAgICAgICBraW5kPSJlcGlzb2RpYyIsCiAgICAgICAgICAg'
        'ICAgICBrZXk9ZiJwcm9tcHQ6e2FicyhoYXNoKHByb21wdCkpfSIsCiAgICAgICAgICAgICAgICB2'
        'YWx1ZT1iZXN0LnN0YXRlbWVudCwKICAgICAgICAgICAgICAgIHNhbGllbmNlPWJlc3QuY29uZmlk'
        'ZW5jZSwKICAgICAgICAgICAgICAgIHRhZ3M9WyJoZXJtZXMiLCBiZXN0LmNhdGVnb3J5LnZhbHVl'
        'XSwKICAgICAgICAgICAgKQogICAgICAgICkKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAi'
        'cmVzcG9uc2UiOiBiZXN0LnN0YXRlbWVudCwKICAgICAgICAgICAgImNvbmZpZGVuY2UiOiBiZXN0'
        'LmNvbmZpZGVuY2UsCiAgICAgICAgICAgICJjYXRlZ29yeSI6IGJlc3QuY2F0ZWdvcnkudmFsdWUs'
        'CiAgICAgICAgfQo='
    ),
    'olympus/models/flagship.py': (
        '...'
    ),
    'olympus/models/olympus_system.py': (
        '...'
    ),
    'olympus/models/oracle_fusion_v4.py': (
        'IiIiVjQ6IGZyb3plbiBPbHltcHVzIGV4cGVydCBiYW5rcyB3aXRoIGNvbW1vbiwgY3Jvc3MtZml0'
        'dGVkIHJvdXRpbmcgY29udHJvbHMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRp'
        'b25zCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yLCBubgoKZnJvbSBvbHlt'
        'cHVzLm1vZGVscy5vbHltcHVzX3N5c3RlbSBpbXBvcnQgT2x5bXB1c1JlZmVyZW5jZVN5c3RlbSwg'
        'T2x5bXB1c1N5c3RlbUNvbmZpZwpmcm9tIG9seW1wdXMubW9kZWxzLnJlZmVyZW5jZV9mYW1pbGll'
        'cyBpbXBvcnQgKAogICAgQWlvblJlZmVyZW5jZUNvbmZpZywKICAgIEF0bGFzUmVmZXJlbmNlQ29u'
        'ZmlnLAogICAgSGVybWVzUmVmZXJlbmNlQ29uZmlnLAogICAgS3Jvbm9zUmVmZXJlbmNlQ29uZmln'
        'LAogICAgUGVyc2V1c1JlZmVyZW5jZUNvbmZpZywKICAgIFByb21ldGhldXNSZWZlcmVuY2VDb25m'
        'aWcsCiAgICBSZWZlcmVuY2VUcmFuc2Zvcm1lckNvbmZpZywKKQoKTU9ERVMgPSAoCiAgICAidW5p'
        'Zm9ybSIsCiAgICAiZml4ZWQiLAogICAgInNlbWFudGljIiwKICAgICJ1dGlsaXR5IiwKICAgICJj'
        'b3JydXB0aW9uIiwKICAgICJvcmFjbGVfY29ycnVwdGlvbiIsCiAgICAib3JhY2xlX3Rhc2siLAog'
        'ICAgImJlc3Rfc2luZ2xlIiwKICAgICJzZW1hbnRpY190cmFpbl9tZWFuIiwKKQpJTlBVVFMgPSAo'
        'InRva2VucyIsICJpbWFnZSIsICJzdGF0ZXMiLCAiYWN0aW9ucyIsICJhbGxvd2VkX3RyYW5zaXRp'
        'b25zIiwgInJpc2tfc2NvcmUiKQoKCmRlZiBzeXN0ZW1fY29uZmlnKCkgLT4gT2x5bXB1c1N5c3Rl'
        'bUNvbmZpZzoKICAgIGJhY2tib25lID0gUmVmZXJlbmNlVHJhbnNmb3JtZXJDb25maWcoCiAgICAg'
        'ICAgdm9jYWJfc2l6ZT0zMiwKICAgICAgICBtb2RlbF9kaW09MTYsCiAgICAgICAgbGF5ZXJzPTEs'
        'CiAgICAgICAgaGVhZHM9NCwKICAgICAgICBmZWVkZm9yd2FyZF9kaW09MzIsCiAgICAgICAgbWF4'
        'X3NlcXVlbmNlX2xlbmd0aD04LAogICAgKQogICAgcmV0dXJuIE9seW1wdXNTeXN0ZW1Db25maWco'
        'CiAgICAgICAgaGVybWVzPUhlcm1lc1JlZmVyZW5jZUNvbmZpZyhiYWNrYm9uZT1iYWNrYm9uZSwg'
        'dG9vbF9jb3VudD00KSwKICAgICAgICBwcm9tZXRoZXVzPVByb21ldGhldXNSZWZlcmVuY2VDb25m'
        'aWcoCiAgICAgICAgICAgIGJhY2tib25lPWJhY2tib25lLCBkb21haW5zPTMsIGFkYXB0ZXJfZGlt'
        'PTgsIGV2aWRlbmNlX2NsYXNzZXM9NAogICAgICAgICksCiAgICAgICAgcGVyc2V1cz1QZXJzZXVz'
        'UmVmZXJlbmNlQ29uZmlnKAogICAgICAgICAgICBiYWNrYm9uZT1iYWNrYm9uZSwgZXhwZXJ0cz00'
        'LCBhY3RpdmVfZXhwZXJ0cz0yLCBleHBlcnRfZGltPTMyCiAgICAgICAgKSwKICAgICAgICBhdGxh'
        'cz1BdGxhc1JlZmVyZW5jZUNvbmZpZygKICAgICAgICAgICAgYmFja2JvbmU9YmFja2JvbmUsIGlt'
        'YWdlX2NoYW5uZWxzPTMsIHBhdGNoX3NpemU9MiwgcmV0cmlldmFsX2RpbT0xMgogICAgICAgICks'
        'CiAgICAgICAga3Jvbm9zPUtyb25vc1JlZmVyZW5jZUNvbmZpZygKICAgICAgICAgICAgc3RhdGVf'
        'ZGltPTUsIGFjdGlvbl9kaW09MywgaGlkZGVuX2RpbT0xMiwgZGlzY3JldGVfYWN0aW9ucz00LCBw'
        'bGFubmluZ19ob3Jpem9uPTQKICAgICAgICApLAogICAgICAgIGFpb249QWlvblJlZmVyZW5jZUNv'
        'bmZpZyhzdGF0ZV9kaW09OCwgaGlkZGVuX2RpbT0xMiwgcmVzZWFyY2hfc3RhZ2VzPTQsIHN0b3Bf'
        'c3RhZ2U9MyksCiAgICAgICAgZnVzaW9uX2RpbT0xNiwKICAgICAgICBkZWNpc2lvbl9jbGFzc2Vz'
        'PTQsCiAgICAgICAgZnVzaW9uX3N0cmF0ZWd5PSJ1bmlmb3JtIiwKICAgICkKCgpjbGFzcyBFeHBl'
        'cnRCYW5rKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZikgLT4gTm9uZToKICAgICAg'
        'ICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnN5c3RlbSA9IE9seW1wdXNSZWZlcmVu'
        'Y2VTeXN0ZW0oc3lzdGVtX2NvbmZpZygpKQogICAgICAgIHNlbGYuaGVhZHMgPSBubi5Nb2R1bGVM'
        'aXN0KG5uLkxpbmVhcigxNiwgNCkgZm9yIF8gaW4gcmFuZ2UoNSkpCgogICAgZGVmIGZvcndhcmQo'
        'c2VsZiwgYmF0Y2g6IGRpY3Rbc3RyLCBUZW5zb3JdKSAtPiB0dXBsZVtUZW5zb3IsIFRlbnNvciwg'
        'VGVuc29yXToKICAgICAgICBvdXRwdXQgPSBzZWxmLnN5c3RlbSgqKntrZXk6IGJhdGNoW2tleV0g'
        'Zm9yIGtleSBpbiBJTlBVVFN9LCBkb21haW5faWRzPWJhdGNoWyJkb21haW5faWRzIl0pCiAgICAg'
        'ICAgeiA9IG91dHB1dC5jb21wb25lbnRfc3RhdGVzCiAgICAgICAgbG9naXRzID0gdG9yY2guc3Rh'
        'Y2soW2hlYWQoels6LCBpXSkgZm9yIGksIGhlYWQgaW4gZW51bWVyYXRlKHNlbGYuaGVhZHMpXSwg'
        'MSkKICAgICAgICByZXR1cm4geiwgbG9naXRzLCBvdXRwdXQua3Jvbm9zLm5leHRfc3RhdGUKCgpk'
        'ZWYgZXhwZXJ0X2xvc3MobG9naXRzOiBUZW5zb3IsIHRhcmdldDogVGVuc29yLCB3ZWlnaHQ6IGZs'
        'b2F0KSAtPiBUZW5zb3I6CiAgICAjIEFsbCBleHBlcnRzIGhhdmUgZXhwbGljaXQgdGFzayBzdXBl'
        'cnZpc2lvbjsgbm8gaGVhZCBpcyByZXdhcmRlZCBmb3IgYSBsYWJlbAogICAgIyBpdCBvbmx5IHNh'
        'dyB0aHJvdWdoIHRhcmdldC1kZXJpdmVkIG1ldGFkYXRhLiBDRiBpcyB0aGUgT05MWSBwYWlyZWQg'
        'Y2hhbmdlLgogICAgY2UgPSBubi5mdW5jdGlvbmFsLmNyb3NzX2VudHJvcHkobG9naXRzLmZsYXR0'
        'ZW4oMCwgMSksIHRhcmdldC5yZXBlYXRfaW50ZXJsZWF2ZSg1KSkKICAgIHAgPSBsb2dpdHMuc29m'
        'dG1heCgtMSkKICAgIGZ1bGwgPSBwLm1lYW4oMSkuZGV0YWNoKCkKICAgIGxlYXZlID0gKHAuc3Vt'
        'KDEsIGtlZXBkaW09VHJ1ZSkgLSBwKSAvIDQKICAgIGNvbnNpc3RlbmN5ID0gKAogICAgICAgIG5u'
        'LmZ1bmN0aW9uYWwua2xfZGl2KAogICAgICAgICAgICBsZWF2ZS5jbGFtcF9taW4oMWUtOCkubG9n'
        'KCksIGZ1bGxbOiwgTm9uZV0uZXhwYW5kX2FzKGxlYXZlKSwgcmVkdWN0aW9uPSJub25lIgogICAg'
        'ICAgICkKICAgICAgICAuc3VtKC0xKQogICAgICAgIC5tZWFuKCkKICAgICkKICAgIHJldHVybiBj'
        'ZSArIHdlaWdodCAqIGNvbnNpc3RlbmN5CgoKZGVmIHJvdXRlcl9mZWF0dXJlcyh6OiBUZW5zb3Is'
        'IGxvZ2l0czogVGVuc29yKSAtPiBUZW5zb3I6CiAgICBwID0gbG9naXRzLnNvZnRtYXgoLTEpCiAg'
        'ICBlbnRyb3B5ID0gLShwICogcC5jbGFtcF9taW4oMWUtOCkubG9nKCkpLnN1bSgtMSkKICAgIHJl'
        'dHVybiB0b3JjaC5jYXQoW3ouZmxhdHRlbigxKSwgcC5mbGF0dGVuKDEpLCBlbnRyb3B5XSwgMSku'
        'ZGV0YWNoKCkKCgpjbGFzcyBSb3V0ZXJzKG5uLk1vZHVsZSk6CiAgICAiIiJFdmVyeSBjb250cm9s'
        'IHNoYXJlcyB0aGlzIGV4YWN0IHRyYWluZWQgZ3JhcGggYW5kIGZyb3plbiBleHBlcnQgcHJlZGlj'
        'dGlvbnMuCgogICAgSGVhZHMgZG8gbm90IHNoYXJlIHRyYWluYWJsZSBoaWRkZW4gbGF5ZXJzLCBh'
        'dm9pZGluZyBhdXhpbGlhcnktbG9zcyB0cmFuc2ZlcgogICAgYmV0d2VlbiBzZW1hbnRpYywgdXRp'
        'bGl0eSwgYW5kIGNvcnJ1cHRpb24gY29udHJvbHMuIEFsbCBhcmUgYWx3YXlzIGV4ZWN1dGVkLgog'
        'ICAgIiIiCgogICAgdHJhaW5pbmdfbWVhbl9nYXRlczogVGVuc29yCgogICAgZGVmIF9faW5pdF9f'
        'KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5m'
        'aXhlZCA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcyg1KSkKICAgICAgICBzZWxmLnJlZ2lzdGVy'
        'X2J1ZmZlcigidHJhaW5pbmdfbWVhbl9nYXRlcyIsIHRvcmNoLm9uZXMoNSkgLyA1KQogICAgICAg'
        'IHNlbGYuaGVhZHMgPSBubi5Nb2R1bGVEaWN0KAogICAgICAgICAgICB7CiAgICAgICAgICAgICAg'
        'ICBuYW1lOiBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICAgICAgICAgIG5uLkxheWVyTm9ybSgx'
        'MDUpLCBubi5MaW5lYXIoMTA1LCAzMiksIG5uLkdFTFUoKSwgbm4uTGluZWFyKDMyLCA1KQogICAg'
        'ICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgZm9yIG5hbWUgaW4gKCJzZW1hbnRpYyIsICJ1'
        'dGlsaXR5IiwgImNvcnJ1cHRpb24iKQogICAgICAgICAgICB9CiAgICAgICAgKQoKICAgIGRlZiBm'
        'b3J3YXJkKHNlbGYsIGZlYXR1cmVzOiBUZW5zb3IpIC0+IGRpY3Rbc3RyLCBUZW5zb3JdOgogICAg'
        'ICAgIHJldHVybiB7CiAgICAgICAgICAgICoqe25hbWU6IGhlYWQoZmVhdHVyZXMpIGZvciBuYW1l'
        'LCBoZWFkIGluIHNlbGYuaGVhZHMuaXRlbXMoKX0sCiAgICAgICAgICAgICJmaXhlZCI6IHNlbGYu'
        'Zml4ZWQuZXhwYW5kKGZlYXR1cmVzLnNoYXBlWzBdLCAtMSksCiAgICAgICAgICAgICJzZW1hbnRp'
        'Y190cmFpbl9tZWFuIjogc2VsZi50cmFpbmluZ19tZWFuX2dhdGVzLmV4cGFuZChmZWF0dXJlcy5z'
        'aGFwZVswXSwgLTEpLAogICAgICAgIH0KCgpkZWYgbm9ybWFsaXplX3F1YWxpdHkocXVhbGl0eTog'
        'VGVuc29yKSAtPiBUZW5zb3I6CiAgICBtYXNzID0gcXVhbGl0eS5zdW0oLTEsIGtlZXBkaW09VHJ1'
        'ZSkKICAgIHJldHVybiB0b3JjaC53aGVyZShtYXNzID4gMCwgcXVhbGl0eSAvIG1hc3MuY2xhbXBf'
        'bWluKDFlLTgpLCB0b3JjaC5vbmVzX2xpa2UocXVhbGl0eSkgLyA1KQoKCmRlZiBwcm9iYWJpbGl0'
        'aWVzKAogICAgbG9naXRzOiBUZW5zb3IsCiAgICBzY29yZXM6IGRpY3Rbc3RyLCBUZW5zb3JdLAog'
        'ICAgbW9kZTogc3RyLAogICAgKiwKICAgIHF1YWxpdHk6IFRlbnNvciB8IE5vbmUgPSBOb25lLAog'
        'ICAgbGFiZWxzOiBUZW5zb3IgfCBOb25lID0gTm9uZSwKICAgIGJlc3Rfc2luZ2xlOiBpbnQgPSAw'
        'LAopIC0+IHR1cGxlW1RlbnNvciwgVGVuc29yXToKICAgIHAgPSBsb2dpdHMuc29mdG1heCgtMSkK'
        'ICAgIGlmIG1vZGUgPT0gInVuaWZvcm0iOgogICAgICAgIGdhdGVzID0gdG9yY2gub25lc19saWtl'
        'KHBbLi4uLCAwXSkgLyA1CiAgICBlbGlmIG1vZGUgaW4geyJmaXhlZCIsICJzZW1hbnRpYyJ9Ogog'
        'ICAgICAgIGdhdGVzID0gc2NvcmVzW21vZGVdLnNvZnRtYXgoLTEpCiAgICBlbGlmIG1vZGUgPT0g'
        'InNlbWFudGljX3RyYWluX21lYW4iOgogICAgICAgIGdhdGVzID0gc2NvcmVzW21vZGVdCiAgICBl'
        'bGlmIG1vZGUgaW4geyJ1dGlsaXR5IiwgImNvcnJ1cHRpb24ifToKICAgICAgICBnYXRlcyA9IG5v'
        'cm1hbGl6ZV9xdWFsaXR5KHNjb3Jlc1ttb2RlXS5zaWdtb2lkKCkpCiAgICBlbGlmIG1vZGUgPT0g'
        'Im9yYWNsZV9jb3JydXB0aW9uIjoKICAgICAgICBpZiBxdWFsaXR5IGlzIE5vbmU6CiAgICAgICAg'
        'ICAgIHJhaXNlIFZhbHVlRXJyb3IoInByaXZpbGVnZWQgY29ycnVwdGlvbiBvcmFjbGUgcmVxdWly'
        'ZXMgcXVhbGl0eSBtZXRhZGF0YSIpCiAgICAgICAgZ2F0ZXMgPSBub3JtYWxpemVfcXVhbGl0eShx'
        'dWFsaXR5KQogICAgZWxpZiBtb2RlID09ICJvcmFjbGVfdGFzayI6CiAgICAgICAgaWYgbGFiZWxz'
        'IGlzIE5vbmU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInByaXZpbGVnZWQgdGFzayBv'
        'cmFjbGUgcmVxdWlyZXMgdHJ1ZSBsYWJlbHMiKQogICAgICAgIHV0aWxpdHkgPSBwLmdhdGhlcigy'
        'LCBsYWJlbHNbOiwgTm9uZSwgTm9uZV0uZXhwYW5kKC0xLCA1LCAxKSkuc3F1ZWV6ZSgtMSkKICAg'
        'ICAgICBnYXRlcyA9IG5uLmZ1bmN0aW9uYWwub25lX2hvdCh1dGlsaXR5LmFyZ21heCgtMSksIDUp'
        'LnRvKHApCiAgICBlbGlmIG1vZGUgPT0gImJlc3Rfc2luZ2xlIjoKICAgICAgICBnYXRlcyA9IHRv'
        'cmNoLnplcm9zX2xpa2UocFsuLi4sIDBdKQogICAgICAgIGdhdGVzWzosIGJlc3Rfc2luZ2xlXSA9'
        'IDEKICAgIGVsc2U6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihtb2RlKQogICAgcmV0dXJuIChn'
        'YXRlc1suLi4sIE5vbmVdICogcCkuc3VtKDEpLCBnYXRlcwo='
    ),
    'olympus/models/parameter_growth.py': (
        'IiIiRnVuY3Rpb24tcHJlc2VydmluZyBGRk4gZ3Jvd3RoIGFuZCBtYXNrZWQgbGF0ZW50IGxlYXJu'
        'aW5nIGZvciBhZGRlZCBjYXBhY2l0eS4KCldpZGVuaW5nIGZvbGxvd3MgTmV0Mk5ldCdzIHJlcGxp'
        'Y2F0aW9uL2NvdW50IGNvbXBlbnNhdGlvbi4gSkVQQSBzdXBwbGllcyBhIHRyYWluaW5nCm9iamVj'
        'dGl2ZSBmb3IgYSBuZXcgcmVzaWR1YWwgYWRhcHRlcjsgbmVpdGhlciBvcGVyYXRpb24gY3JlYXRl'
        'cyBwcmV0cmFpbmVkIGtub3dsZWRnZS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90'
        'YXRpb25zCgppbXBvcnQgY29weQppbXBvcnQgbWF0aApmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBk'
        'YXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IGNhc3QKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNo'
        'IGltcG9ydCBUZW5zb3IsIG5uCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgR3Jvd3Ro'
        'UmVjZWlwdDoKICAgIG9yaWdpbmFsX3BhcmFtZXRlcnM6IGludAogICAgZXhwYW5kZWRfcGFyYW1l'
        'dGVyczogaW50CiAgICBvcmlnaW5hbF93aWR0aDogaW50CiAgICBleHBhbmRlZF93aWR0aDogaW50'
        'CiAgICBvcHRpbWl6ZXJfcmVidWlsZF9yZXF1aXJlZDogYm9vbCA9IFRydWUKICAgIHRyYWluZWRf'
        'dGFyZ2V0X3NjYWxlOiBib29sID0gRmFsc2UKCgpkZWYgd2lkZW5fZmVlZGZvcndhcmQoCiAgICBi'
        'bG9jazogbm4uU2VxdWVudGlhbCwKICAgIHdpZHRoOiBpbnQsCiAgICAqLAogICAgbWF4X3BhcmFt'
        'ZXRlcl9ieXRlczogaW50LAopIC0+IHR1cGxlW25uLlNlcXVlbnRpYWwsIEdyb3d0aFJlY2VpcHRd'
        'OgogICAgIiIiQ29weSBhIExpbmVhci9hY3RpdmF0aW9uWy96ZXJvLWRyb3BvdXRdL0xpbmVhciBi'
        'bG9jayBpbnRvIGEgd2lkZXIgZXF1aXZhbGVudC4KCiAgICBUaGUgb2xkIG1vZHVsZSBhbmQgb3B0'
        'aW1pemVyIGFyZSBuZXZlciBtdXRhdGVkLiBUaGUgY2FsbGVyIG11c3QgYnVpbGQgYSBuZXcKICAg'
        'IG9wdGltaXplcjsgcHJlc2VydmluZyBvdXRwdXRzIGF0IGluaXRpYWxpemF0aW9uIGRvZXMgbm90'
        'IHByZXNlcnZlIHRyYWluaW5nIGR5bmFtaWNzLgogICAgIiIiCiAgICBsYXllcnMgPSBsaXN0KGJs'
        'b2NrLmNoaWxkcmVuKCkpCiAgICBpZiAoCiAgICAgICAgbGVuKGxheWVycykgbm90IGluIHszLCA0'
        'fQogICAgICAgIG9yIG5vdCBpc2luc3RhbmNlKGxheWVyc1swXSwgbm4uTGluZWFyKQogICAgICAg'
        'IG9yIG5vdCBpc2luc3RhbmNlKGxheWVyc1stMV0sIG5uLkxpbmVhcikKICAgICAgICBvciBub3Qg'
        'aXNpbnN0YW5jZShsYXllcnNbMV0sIChubi5HRUxVLCBubi5TaUxVLCBubi5SZUxVKSkKICAgICAg'
        'ICBvciAobGVuKGxheWVycykgPT0gNCBhbmQgKG5vdCBpc2luc3RhbmNlKGxheWVyc1syXSwgbm4u'
        'RHJvcG91dCkgb3IgbGF5ZXJzWzJdLnAgIT0gMCkpCiAgICApOgogICAgICAgIHJhaXNlIFZhbHVl'
        'RXJyb3IoImdyb3d0aCBzdXBwb3J0cyBMaW5lYXIvYWN0aXZhdGlvblsvemVyby1kcm9wb3V0XS9M'
        'aW5lYXIgb25seSIpCiAgICBmaXJzdCwgbGFzdCA9IGxheWVyc1swXSwgbGF5ZXJzWy0xXQogICAg'
        'YXNzZXJ0IGlzaW5zdGFuY2UoZmlyc3QsIG5uLkxpbmVhcikgYW5kIGlzaW5zdGFuY2UobGFzdCwg'
        'bm4uTGluZWFyKQogICAgb2xkID0gZmlyc3Qub3V0X2ZlYXR1cmVzCiAgICBpZiB0eXBlKHdpZHRo'
        'KSBpcyBub3QgaW50IG9yIHdpZHRoIDw9IG9sZCBvciBsYXN0LmluX2ZlYXR1cmVzICE9IG9sZDoK'
        'ICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJncm93dGggd2lkdGggbXVzdCBleGNlZWQgdGhlIGFs'
        'aWduZWQgb3JpZ2luYWwgaGlkZGVuIHdpZHRoIikKICAgIHBhcmFtZXRlcnMgPSBsaXN0KGJsb2Nr'
        'LnBhcmFtZXRlcnMoKSkKICAgIGlmIGFueSgKICAgICAgICBwLmlzX21ldGEgb3Igbm90IHAuaXNf'
        'ZmxvYXRpbmdfcG9pbnQoKSBvciBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShwKS5hbGwoKSkKICAg'
        'ICAgICBmb3IgcCBpbiBwYXJhbWV0ZXJzCiAgICApOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3Io'
        'Imdyb3d0aCByZXF1aXJlcyBtYXRlcmlhbGl6ZWQgZmluaXRlIGZsb2F0aW5nLXBvaW50IHBhcmFt'
        'ZXRlcnMiKQogICAgaWYgYW55KHAuZHR5cGUgIT0gZmlyc3Qud2VpZ2h0LmR0eXBlIG9yIHAuZGV2'
        'aWNlICE9IGZpcnN0LndlaWdodC5kZXZpY2UgZm9yIHAgaW4gcGFyYW1ldGVycyk6CiAgICAgICAg'
        'cmFpc2UgVmFsdWVFcnJvcigiZ3Jvd3RoIHBhcmFtZXRlcnMgbXVzdCBzaGFyZSBkdHlwZSBhbmQg'
        'ZGV2aWNlIikKICAgIGNvdW50ID0gd2lkdGggKiAoZmlyc3QuaW5fZmVhdHVyZXMgKyBsYXN0Lm91'
        'dF9mZWF0dXJlcyArIGludChmaXJzdC5iaWFzIGlzIG5vdCBOb25lKSkKICAgIGNvdW50ICs9IGxh'
        'c3Qub3V0X2ZlYXR1cmVzIGlmIGxhc3QuYmlhcyBpcyBub3QgTm9uZSBlbHNlIDAKICAgIG9sZF9j'
        'b3VudCA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gcGFyYW1ldGVycykKICAgIGlmICgKICAgICAg'
        'ICB0eXBlKG1heF9wYXJhbWV0ZXJfYnl0ZXMpIGlzIG5vdCBpbnQKICAgICAgICBvciAoY291bnQg'
        'KyBvbGRfY291bnQpICogZmlyc3Qud2VpZ2h0LmVsZW1lbnRfc2l6ZSgpID4gbWF4X3BhcmFtZXRl'
        'cl9ieXRlcwogICAgKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJncm93dGggZXhjZWVkcyB0'
        'aGUgY29tYmluZWQgb2xkL25ldyBwYXJhbWV0ZXIgYnl0ZSBidWRnZXQiKQogICAgZGV2aWNlLCBk'
        'dHlwZSA9IGZpcnN0LndlaWdodC5kZXZpY2UsIGZpcnN0LndlaWdodC5kdHlwZQogICAgIyBBbGxv'
        'Y2F0ZSB1bmluaXRpYWxpemVkIHRlbnNvcnMgd2l0aG91dCBjb25zdW1pbmcgYW55IGFwcGxpY2F0'
        'aW9uIFJORyBzdGF0ZS4KICAgIGxlZnQgPSBubi5MaW5lYXIoZmlyc3QuaW5fZmVhdHVyZXMsIHdp'
        'ZHRoLCBiaWFzPWZpcnN0LmJpYXMgaXMgbm90IE5vbmUsIGRldmljZT0ibWV0YSIpCiAgICByaWdo'
        'dCA9IG5uLkxpbmVhcih3aWR0aCwgbGFzdC5vdXRfZmVhdHVyZXMsIGJpYXM9bGFzdC5iaWFzIGlz'
        'IG5vdCBOb25lLCBkZXZpY2U9Im1ldGEiKQogICAgbWFwcGluZyA9IHRvcmNoLmFyYW5nZSh3aWR0'
        'aCwgZGV2aWNlPWRldmljZSkgJSBvbGQKICAgIGNvdW50cyA9IHRvcmNoLmJpbmNvdW50KG1hcHBp'
        'bmcsIG1pbmxlbmd0aD1vbGQpLnRvKGR0eXBlKQogICAgbGVmdC53ZWlnaHQgPSBubi5QYXJhbWV0'
        'ZXIoZmlyc3Qud2VpZ2h0LmRldGFjaCgpW21hcHBpbmddLmNsb25lKCksIGZpcnN0LndlaWdodC5y'
        'ZXF1aXJlc19ncmFkKQogICAgcmlnaHQud2VpZ2h0ID0gbm4uUGFyYW1ldGVyKAogICAgICAgIChs'
        'YXN0LndlaWdodC5kZXRhY2goKVs6LCBtYXBwaW5nXSAvIGNvdW50c1ttYXBwaW5nXSkuY2xvbmUo'
        'KSwgbGFzdC53ZWlnaHQucmVxdWlyZXNfZ3JhZAogICAgKQogICAgaWYgZmlyc3QuYmlhcyBpcyBu'
        'b3QgTm9uZToKICAgICAgICBsZWZ0LmJpYXMgPSBubi5QYXJhbWV0ZXIoZmlyc3QuYmlhcy5kZXRh'
        'Y2goKVttYXBwaW5nXS5jbG9uZSgpLCBmaXJzdC5iaWFzLnJlcXVpcmVzX2dyYWQpCiAgICBpZiBs'
        'YXN0LmJpYXMgaXMgbm90IE5vbmU6CiAgICAgICAgcmlnaHQuYmlhcyA9IG5uLlBhcmFtZXRlcihs'
        'YXN0LmJpYXMuZGV0YWNoKCkuY2xvbmUoKSwgbGFzdC5iaWFzLnJlcXVpcmVzX2dyYWQpCiAgICBy'
        'ZXN1bHQgPSBubi5TZXF1ZW50aWFsKGxlZnQsICooY29weS5kZWVwY29weShsYXllcikgZm9yIGxh'
        'eWVyIGluIGxheWVyc1sxOi0xXSksIHJpZ2h0KQogICAgcmVzdWx0LnRyYWluKGJsb2NrLnRyYWlu'
        'aW5nKQogICAgcmV0dXJuIHJlc3VsdCwgR3Jvd3RoUmVjZWlwdChvbGRfY291bnQsIGNvdW50LCBv'
        'bGQsIHdpZHRoKQoKCmNsYXNzIEpFUEFQYXJhbWV0ZXJHcm93dGgobm4uTW9kdWxlKToKICAgICIi'
        'IklkZW50aXR5LWluaXRpYWxpemVkIHJlc2lkdWFsIGNhcGFjaXR5IHRyYWluZWQgd2l0aCBtYXNr'
        'ZWQgbGF0ZW50IHByZWRpY3Rpb24uIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHdpZHRoOiBp'
        'bnQsIGFkZGVkX3dpZHRoOiBpbnQsICosIG1vbWVudHVtOiBmbG9hdCA9IDAuOTkpIC0+IE5vbmU6'
        'CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgdHlwZSh3aWR0aCkgaXMgbm90'
        'IGludCBvciB0eXBlKGFkZGVkX3dpZHRoKSBpcyBub3QgaW50IG9yIG1pbih3aWR0aCwgYWRkZWRf'
        'd2lkdGgpIDwgMjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigid2lkdGhzIG11c3QgYmUg'
        'aW50ZWdlcnMgPj0gMiIpCiAgICAgICAgaWYgbWF4KHdpZHRoLCBhZGRlZF93aWR0aCkgPiA4MTky'
        'IG9yIG5vdCBtYXRoLmlzZmluaXRlKG1vbWVudHVtKSBvciBub3QgMCA8PSBtb21lbnR1bSA8IDE6'
        'CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImdyb3d0aCBkaW1lbnNpb25zIG9yIEVNQSBt'
        'b21lbnR1bSBleGNlZWQgdGhlIGJvdW5kZWQgY29udHJhY3QiKQogICAgICAgIHNlbGYud2lkdGgs'
        'IHNlbGYubW9tZW50dW0gPSB3aWR0aCwgbW9tZW50dW0KICAgICAgICBzZWxmLmNvbnRleHRfZW5j'
        'b2RlciA9IG5uLkxpbmVhcih3aWR0aCwgYWRkZWRfd2lkdGgpCiAgICAgICAgc2VsZi50YXJnZXRf'
        'ZW5jb2RlciA9IGNvcHkuZGVlcGNvcHkoc2VsZi5jb250ZXh0X2VuY29kZXIpLnJlcXVpcmVzX2dy'
        'YWRfKEZhbHNlKQogICAgICAgIHNlbGYucHJlZGljdG9yID0gbm4uU2VxdWVudGlhbChubi5MaW5l'
        'YXIoYWRkZWRfd2lkdGgsIGFkZGVkX3dpZHRoKSwgbm4uR0VMVSgpKQogICAgICAgIHNlbGYucmVz'
        'aWR1YWxfaGVhZCA9IG5uLkxpbmVhcihhZGRlZF93aWR0aCwgd2lkdGgpCiAgICAgICAgbm4uaW5p'
        'dC56ZXJvc18oc2VsZi5yZXNpZHVhbF9oZWFkLndlaWdodCkKICAgICAgICBubi5pbml0Lnplcm9z'
        'XyhzZWxmLnJlc2lkdWFsX2hlYWQuYmlhcykKCiAgICBkZWYgX3ZhbGlkYXRlKHNlbGYsIGhpZGRl'
        'bjogVGVuc29yKSAtPiBOb25lOgogICAgICAgIHJlZmVyZW5jZSA9IHNlbGYuY29udGV4dF9lbmNv'
        'ZGVyLndlaWdodAogICAgICAgIGlmICgKICAgICAgICAgICAgaGlkZGVuLm5kaW0gIT0gMwogICAg'
        'ICAgICAgICBvciBoaWRkZW4uc2hhcGVbLTFdICE9IHNlbGYud2lkdGgKICAgICAgICAgICAgb3Ig'
        'bWluKGhpZGRlbi5zaGFwZVs6Ml0pIDwgMQogICAgICAgICAgICBvciBoaWRkZW4uZHR5cGUgIT0g'
        'cmVmZXJlbmNlLmR0eXBlCiAgICAgICAgICAgIG9yIGhpZGRlbi5kZXZpY2UgIT0gcmVmZXJlbmNl'
        'LmRldmljZQogICAgICAgICAgICBvciBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShoaWRkZW4pLmFs'
        'bCgpKQogICAgICAgICk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAg'
        'ICAgICAiaGlkZGVuIHN0YXRlcyBtdXN0IGJlIGZpbml0ZSBbYmF0Y2gsIHRva2Vucywgd2lkdGhd'
        'IG1hdGNoaW5nIHBhcmFtZXRlcnMiCiAgICAgICAgICAgICkKCiAgICBkZWYgZm9yd2FyZChzZWxm'
        'LCBoaWRkZW46IFRlbnNvcikgLT4gVGVuc29yOgogICAgICAgIHNlbGYuX3ZhbGlkYXRlKGhpZGRl'
        'bikKICAgICAgICByZXR1cm4gY2FzdCgKICAgICAgICAgICAgVGVuc29yLCBoaWRkZW4gKyBzZWxm'
        'LnJlc2lkdWFsX2hlYWQoc2VsZi5wcmVkaWN0b3Ioc2VsZi5jb250ZXh0X2VuY29kZXIoaGlkZGVu'
        'KSkpCiAgICAgICAgKQoKICAgIGRlZiBtYXNrZWRfb2JqZWN0aXZlKHNlbGYsIGhpZGRlbjogVGVu'
        'c29yLCB0YXJnZXRfbWFzazogVGVuc29yKSAtPiBUZW5zb3I6CiAgICAgICAgIiIiRXhjbHVkZSB0'
        'YXJnZXQgcG9zaXRpb25zIGZyb20gdGhpcyBhZGFwdGVyJ3MgbGF0ZW50IHByZWRpY3Rpb24gY29u'
        'dGV4dC4KCiAgICAgICAgVGhpcyBjYW5ub3QgdW5kbyBpbmZvcm1hdGlvbiBhbHJlYWR5IG1peGVk'
        'IGJ5IGFuIHVwc3RyZWFtIGVuY29kZXIuIEZvcgogICAgICAgIHJhdy1pbnB1dCBtYXNraW5nIHN0'
        'dWRpZXMgdGhlIGNhbGxlciBtdXN0IGVuY29kZSBjb250ZXh0IGZyb20gc2VwYXJhdGVseQogICAg'
        'ICAgIG1hc2tlZCBpbnB1dHM7IHRoaXMgYm91bmRlZCBvYmplY3RpdmUgYWxvbmUgZG9lcyBub3Qg'
        'Y2VydGlmeSB0aGF0IGJvdW5kYXJ5LgogICAgICAgICIiIgogICAgICAgIHNlbGYuX3ZhbGlkYXRl'
        'KGhpZGRlbikKICAgICAgICBpZiAoCiAgICAgICAgICAgIHRhcmdldF9tYXNrLnNoYXBlICE9IGhp'
        'ZGRlbi5zaGFwZVs6Ml0KICAgICAgICAgICAgb3IgdGFyZ2V0X21hc2suZHR5cGUgIT0gdG9yY2gu'
        'Ym9vbAogICAgICAgICAgICBvciB0YXJnZXRfbWFzay5kZXZpY2UgIT0gaGlkZGVuLmRldmljZQog'
        'ICAgICAgICAgICBvciBub3QgYm9vbCh0YXJnZXRfbWFzay5hbnkoLTEpLmFsbCgpKQogICAgICAg'
        'ICAgICBvciBub3QgYm9vbCgofnRhcmdldF9tYXNrKS5hbnkoLTEpLmFsbCgpKQogICAgICAgICk6'
        'CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImVhY2ggc2VxdWVuY2UgbmVlZHMgQm9vbGVh'
        'biB2aXNpYmxlIGFuZCB0YXJnZXQgcG9zaXRpb25zIikKICAgICAgICB2aXNpYmxlID0gfnRhcmdl'
        'dF9tYXNrCiAgICAgICAgY29udGV4dCA9IHNlbGYuY29udGV4dF9lbmNvZGVyKGhpZGRlbi5tYXNr'
        'ZWRfZmlsbCh0YXJnZXRfbWFza1suLi4sIE5vbmVdLCAwKSkKICAgICAgICBwb29sZWQgPSAoY29u'
        'dGV4dCAqIHZpc2libGVbLi4uLCBOb25lXSkuc3VtKDEpIC8gdmlzaWJsZS5zdW0oMSwga2VlcGRp'
        'bT1UcnVlKQogICAgICAgIHByZWRpY3RlZCA9IHNlbGYucHJlZGljdG9yKHBvb2xlZClbOiwgTm9u'
        'ZV0uZXhwYW5kKC0xLCBoaWRkZW4uc2hhcGVbMV0sIC0xKQogICAgICAgIHdpdGggdG9yY2gubm9f'
        'Z3JhZCgpOgogICAgICAgICAgICB0YXJnZXQgPSBzZWxmLnRhcmdldF9lbmNvZGVyKGhpZGRlbi5k'
        'ZXRhY2goKSkKICAgICAgICByZXR1cm4gY2FzdChUZW5zb3IsIChwcmVkaWN0ZWRbdGFyZ2V0X21h'
        'c2tdIC0gdGFyZ2V0W3RhcmdldF9tYXNrXSkuc3F1YXJlKCkubWVhbigpKQoKICAgIEB0b3JjaC5u'
        'b19ncmFkKCkKICAgIGRlZiB1cGRhdGVfdGFyZ2V0KHNlbGYpIC0+IE5vbmU6CiAgICAgICAgb25s'
        'aW5lID0gdHVwbGUoc2VsZi5jb250ZXh0X2VuY29kZXIucGFyYW1ldGVycygpKQogICAgICAgIHRh'
        'cmdldCA9IHR1cGxlKHNlbGYudGFyZ2V0X2VuY29kZXIucGFyYW1ldGVycygpKQogICAgICAgIGlm'
        'IGFueShub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZShwKS5hbGwoKSkgZm9yIHAgaW4gKCpvbmxpbmUs'
        'ICp0YXJnZXQpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRU1BIHVwZGF0ZSByZXF1'
        'aXJlcyBmaW5pdGUgb25saW5lIGFuZCB0YXJnZXQgcGFyYW1ldGVycyIpCiAgICAgICAgZm9yIHRl'
        'YWNoZXIsIHN0dWRlbnQgaW4gemlwKHRhcmdldCwgb25saW5lLCBzdHJpY3Q9VHJ1ZSk6CiAgICAg'
        'ICAgICAgIHRlYWNoZXIubGVycF8oc3R1ZGVudCwgMSAtIHNlbGYubW9tZW50dW0pCg=='
    ),
    'olympus/models/reference_families.py': (
        '...'
        'MSksIHRpbWUpLCBkaW09LTEpCg=='
    ),
    'olympus/models/reliability_fusion.py': (
        'IiIiUmVsaWFiaWxpdHktc3VwZXJ2aXNlZCBzdWNjZXNzb3IgdG8gdGhlIGZyb3plbiBPbHltcHVz'
        'IHYyIGZ1c2lvbiBtZWNoYW5pc20uCgpUaGUgbW9kdWxlIGlzIGFkZGl0aXZlOiBpdCBsZWF2ZXMg'
        'dGhlIHYyIHJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbiBpbnRhY3QgYW5kCnJldXNlcyBpdHMgZmFt'
        'aWx5IGVuY29kZXJzLiAgRXZlcnkgcm91dGluZyBjb250cm9sIGluc3RhbnRpYXRlcyBhbmQgZXhl'
        'Y3V0ZXMKdGhlIHNhbWUgcmVsaWFiaWxpdHkgaGVhZHMgc28gcGFyYW1ldGVyIGNvdW50IGFuZCBm'
        'b3J3YXJkIGNvbXB1dGUgcmVtYWluCm1hdGNoZWQuICBSZWxpYWJpbGl0eSBlc3RpbWF0ZXMgYXJl'
        'IHByZWRpY3Rpb25zIGZyb20gbW9kZWwgc3RhdGU7IHByb3RlY3RlZAp0YXJnZXRzIGFyZSBuZXZl'
        'ciBpbnB1dHMgdG8gdGhlIGZvcndhcmQgcGFzcy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0'
        'IGFubm90YXRpb25zCgppbXBvcnQgbWF0aApmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xh'
        'c3MsIHJlcGxhY2UKZnJvbSB0eXBpbmcgaW1wb3J0IExpdGVyYWwKCmltcG9ydCB0b3JjaApmcm9t'
        'IHB5ZGFudGljIGltcG9ydCBGaWVsZCwgbW9kZWxfdmFsaWRhdG9yCmZyb20gdG9yY2ggaW1wb3J0'
        'IFRlbnNvciwgbm4KCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0IFN0cmljdE1vZGVs'
        'CmZyb20gb2x5bXB1cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0gaW1wb3J0ICgKICAgIE9seW1wdXNS'
        'ZWZlcmVuY2VTeXN0ZW0sCiAgICBPbHltcHVzU3lzdGVtQ29uZmlnLAogICAgT2x5bXB1c1N5c3Rl'
        'bUxvc3MsCiAgICBPbHltcHVzU3lzdGVtT3V0cHV0LAogICAgb2x5bXB1c19zeXN0ZW1fbG9zcywK'
        'KQoKUmVsaWFiaWxpdHlSb3V0aW5nTW9kZSA9IExpdGVyYWxbCiAgICAicmVsaWFiaWxpdHlfc2hy'
        'aW5rYWdlIiwKICAgICJyZWxpYWJpbGl0eV9vbmx5IiwKICAgICJzZW1hbnRpYyIsCiAgICAidW5p'
        'Zm9ybSIsCl0KCgpjbGFzcyBSZWxpYWJpbGl0eUZ1c2lvbkNvbmZpZyhTdHJpY3RNb2RlbCk6CiAg'
        'ICAiIiJGcm96ZW4sIGJvdW5kZWQgY29uZmlndXJhdGlvbiBmb3IgdGhlIHYzIHN1Y2Nlc3NvciBt'
        'ZWNoYW5pc20uIiIiCgogICAgc3lzdGVtOiBPbHltcHVzU3lzdGVtQ29uZmlnCiAgICByb3V0aW5n'
        'X21vZGU6IFJlbGlhYmlsaXR5Um91dGluZ01vZGUgPSAicmVsaWFiaWxpdHlfc2hyaW5rYWdlIgog'
        'ICAgcmVsaWFiaWxpdHlfbG9naXRfc3RyZW5ndGg6IGZsb2F0ID0gRmllbGQoZGVmYXVsdD0xLjAs'
        'IGdlPTAuMCwgbGU9MTYuMCkKICAgIHNocmlua2FnZV90b19hZGFwdGl2ZTogZmxvYXQgPSBGaWVs'
        'ZChkZWZhdWx0PTAuNzUsIGdlPTAuMCwgbGU9MS4wKQogICAgcHJvYmFiaWxpdHlfZmxvb3I6IGZs'
        'b2F0ID0gRmllbGQoZGVmYXVsdD0xZS02LCBndD0wLjAsIGxlPTFlLTIpCgogICAgQG1vZGVsX3Zh'
        'bGlkYXRvcihtb2RlPSJhZnRlciIpCiAgICBkZWYgdmFsaWRfbWVjaGFuaXNtKHNlbGYpIC0+IFJl'
        'bGlhYmlsaXR5RnVzaW9uQ29uZmlnOgogICAgICAgIGlmIHNlbGYucm91dGluZ19tb2RlIGluIHsi'
        'cmVsaWFiaWxpdHlfc2hyaW5rYWdlIiwgInJlbGlhYmlsaXR5X29ubHkifToKICAgICAgICAgICAg'
        'aWYgc2VsZi5yZWxpYWJpbGl0eV9sb2dpdF9zdHJlbmd0aCA8PSAwLjA6CiAgICAgICAgICAgICAg'
        'ICByYWlzZSBWYWx1ZUVycm9yKCJyZWxpYWJpbGl0eS1hd2FyZSByb3V0aW5nIHJlcXVpcmVzIHBv'
        'c2l0aXZlIGxvZ2l0IHN0cmVuZ3RoIikKICAgICAgICAgICAgaWYgc2VsZi5zaHJpbmthZ2VfdG9f'
        'YWRhcHRpdmUgPD0gMC4wOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVsaWFi'
        'aWxpdHktYXdhcmUgcm91dGluZyByZXF1aXJlcyBwb3NpdGl2ZSBhZGFwdGl2ZSBtYXNzIikKICAg'
        'ICAgICByZXR1cm4gc2VsZgoKCmNsYXNzIFJlbGlhYmlsaXR5TG9zc0NvbmZpZyhTdHJpY3RNb2Rl'
        'bCk6CiAgICAiIiJKb2ludCB2MyBvYmplY3RpdmUgd2l0aCBleHBsaWNpdCByZWxpYWJpbGl0eSBj'
        'YWxpYnJhdGlvbi4iIiIKCiAgICB0cmFuc2l0aW9uX3dlaWdodDogZmxvYXQgPSBGaWVsZChkZWZh'
        'dWx0PTAuNSwgZ2U9MC4wKQogICAgdGVtcG9yYWxfd2VpZ2h0OiBmbG9hdCA9IEZpZWxkKGRlZmF1'
        'bHQ9MC4yNSwgZ2U9MC4wKQogICAgdW5jZXJ0YWludHlfd2VpZ2h0OiBmbG9hdCA9IEZpZWxkKGRl'
        'ZmF1bHQ9MC4xLCBnZT0wLjApCiAgICByaXNrX3dlaWdodDogZmxvYXQgPSBGaWVsZChkZWZhdWx0'
        'PTAuMSwgZ2U9MC4wKQogICAgZ2F0ZV9iYWxhbmNlX3dlaWdodDogZmxvYXQgPSBGaWVsZChkZWZh'
        'dWx0PTAuMDEsIGdlPTAuMCkKICAgIGNvdW50ZXJmYWN0dWFsX3dlaWdodDogZmxvYXQgPSBGaWVs'
        'ZChkZWZhdWx0PTAuMDUsIGdlPTAuMCkKICAgIHJlbGlhYmlsaXR5X3dlaWdodDogZmxvYXQgPSBG'
        'aWVsZChkZWZhdWx0PTAuMjUsIGdlPTAuMCkKCiAgICBAbW9kZWxfdmFsaWRhdG9yKG1vZGU9ImFm'
        'dGVyIikKICAgIGRlZiBmaW5pdGVfd2VpZ2h0cyhzZWxmKSAtPiBSZWxpYWJpbGl0eUxvc3NDb25m'
        'aWc6CiAgICAgICAgaWYgbm90IGFsbChtYXRoLmlzZmluaXRlKHZhbHVlKSBmb3IgdmFsdWUgaW4g'
        'c2VsZi5tb2RlbF9kdW1wKCkudmFsdWVzKCkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9y'
        'KCJsb3NzIHdlaWdodHMgbXVzdCBiZSBmaW5pdGUiKQogICAgICAgIHJldHVybiBzZWxmCgogICAg'
        'ZGVmIGJhc2Vfd2VpZ2h0cyhzZWxmKSAtPiBkaWN0W3N0ciwgZmxvYXRdOgogICAgICAgIHZhbHVl'
        'cyA9IHNlbGYubW9kZWxfZHVtcCgpCiAgICAgICAgdmFsdWVzLnBvcCgicmVsaWFiaWxpdHlfd2Vp'
        'Z2h0IikKICAgICAgICByZXR1cm4gdmFsdWVzCgoKQGRhdGFjbGFzcyhzbG90cz1UcnVlKQpjbGFz'
        'cyBSZWxpYWJpbGl0eUZ1c2lvbk91dHB1dDoKICAgIHN5c3RlbTogT2x5bXB1c1N5c3RlbU91dHB1'
        'dAogICAgc2VtYW50aWNfZ2F0ZXM6IFRlbnNvcgogICAgY29tcG9uZW50X3JlbGlhYmlsaXR5X2xv'
        'Z2l0czogVGVuc29yCiAgICBjb21wb25lbnRfcmVsaWFiaWxpdHk6IFRlbnNvcgoKCkBkYXRhY2xh'
        'c3Moc2xvdHM9VHJ1ZSkKY2xhc3MgUmVsaWFiaWxpdHlGdXNpb25Mb3NzOgogICAgdG90YWw6IFRl'
        'bnNvcgogICAgc3lzdGVtOiBPbHltcHVzU3lzdGVtTG9zcwogICAgcmVsaWFiaWxpdHlfY2FsaWJy'
        'YXRpb246IFRlbnNvcgoKCmNsYXNzIFJlbGlhYmlsaXR5QXdhcmVPbHltcHVzU3lzdGVtKG5uLk1v'
        'ZHVsZSk6CiAgICAiIiJSb3V0ZSBmYW1pbHkgc3RhdGVzIHVzaW5nIGNhbGlicmF0ZWQgcmVsaWFi'
        'aWxpdHkgd2l0aCB1bmlmb3JtIHNocmlua2FnZS4iIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwg'
        'Y29uZmlnOiBSZWxpYWJpbGl0eUZ1c2lvbkNvbmZpZykgLT4gTm9uZToKICAgICAgICBzdXBlcigp'
        'Ll9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZpZwogICAgICAgIHNlbGYuc3lz'
        'dGVtID0gT2x5bXB1c1JlZmVyZW5jZVN5c3RlbShjb25maWcuc3lzdGVtKQogICAgICAgIHNlbGYu'
        'cmVsaWFiaWxpdHlfaGVhZHMgPSBubi5Nb2R1bGVEaWN0KAogICAgICAgICAgICB7CiAgICAgICAg'
        'ICAgICAgICBuYW1lOiBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICAgICAgICAgIG5uLkxheWVy'
        'Tm9ybShjb25maWcuc3lzdGVtLmZ1c2lvbl9kaW0pLAogICAgICAgICAgICAgICAgICAgIG5uLkxp'
        'bmVhcihjb25maWcuc3lzdGVtLmZ1c2lvbl9kaW0sIGNvbmZpZy5zeXN0ZW0uZnVzaW9uX2RpbSks'
        'CiAgICAgICAgICAgICAgICAgICAgbm4uR0VMVSgpLAogICAgICAgICAgICAgICAgICAgIG5uLkxp'
        'bmVhcihjb25maWcuc3lzdGVtLmZ1c2lvbl9kaW0sIDEpLAogICAgICAgICAgICAgICAgKQogICAg'
        'ICAgICAgICAgICAgZm9yIG5hbWUgaW4gc2VsZi5zeXN0ZW0uY29tcG9uZW50X25hbWVzCiAgICAg'
        'ICAgICAgIH0KICAgICAgICApCgogICAgQHByb3BlcnR5CiAgICBkZWYgY29tcG9uZW50X25hbWVz'
        'KHNlbGYpIC0+IHR1cGxlW3N0ciwgLi4uXToKICAgICAgICByZXR1cm4gc2VsZi5zeXN0ZW0uY29t'
        'cG9uZW50X25hbWVzCgogICAgZGVmIGZvcndhcmQoCiAgICAgICAgc2VsZiwKICAgICAgICB0b2tl'
        'bnM6IFRlbnNvciwKICAgICAgICBpbWFnZTogVGVuc29yLAogICAgICAgIHN0YXRlczogVGVuc29y'
        'LAogICAgICAgIGFjdGlvbnM6IFRlbnNvciwKICAgICAgICBhbGxvd2VkX3RyYW5zaXRpb25zOiBU'
        'ZW5zb3IsCiAgICAgICAgcmlza19zY29yZTogVGVuc29yLAogICAgICAgICosCiAgICAgICAgYXR0'
        'ZW50aW9uX21hc2s6IFRlbnNvciB8IE5vbmUgPSBOb25lLAogICAgICAgIGRvbWFpbl9pZHM6IFRl'
        'bnNvciB8IE5vbmUgPSBOb25lLAogICAgICAgIHRlbXBvcmFsX21hc2s6IFRlbnNvciB8IE5vbmUg'
        'PSBOb25lLAogICAgICAgIHRpbWVzdGFtcDogVGVuc29yIHwgTm9uZSA9IE5vbmUsCiAgICAgICAg'
        'ZXh0ZW5zaW9uX2ZlYXR1cmVzOiBkaWN0W3N0ciwgVGVuc29yXSB8IE5vbmUgPSBOb25lLAogICAg'
        'KSAtPiBSZWxpYWJpbGl0eUZ1c2lvbk91dHB1dDoKICAgICAgICBiYXNlID0gc2VsZi5zeXN0ZW0o'
        'CiAgICAgICAgICAgIHRva2VucywKICAgICAgICAgICAgaW1hZ2UsCiAgICAgICAgICAgIHN0YXRl'
        'cywKICAgICAgICAgICAgYWN0aW9ucywKICAgICAgICAgICAgYWxsb3dlZF90cmFuc2l0aW9ucywK'
        'ICAgICAgICAgICAgcmlza19zY29yZSwKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s9YXR0ZW50'
        'aW9uX21hc2ssCiAgICAgICAgICAgIGRvbWFpbl9pZHM9ZG9tYWluX2lkcywKICAgICAgICAgICAg'
        'dGVtcG9yYWxfbWFzaz10ZW1wb3JhbF9tYXNrLAogICAgICAgICAgICB0aW1lc3RhbXA9dGltZXN0'
        'YW1wLAogICAgICAgICAgICBleHRlbnNpb25fZmVhdHVyZXM9ZXh0ZW5zaW9uX2ZlYXR1cmVzLAog'
        'ICAgICAgICkKICAgICAgICB2ZWN0b3JzID0gYmFzZS5jb21wb25lbnRfc3RhdGVzCiAgICAgICAg'
        'cmVsaWFiaWxpdHlfbG9naXRzID0gdG9yY2guY2F0KAogICAgICAgICAgICBbCiAgICAgICAgICAg'
        'ICAgICBzZWxmLnJlbGlhYmlsaXR5X2hlYWRzW25hbWVdKHZlY3RvcnNbOiwgaW5kZXhdKQogICAg'
        'ICAgICAgICAgICAgZm9yIGluZGV4LCBuYW1lIGluIGVudW1lcmF0ZShzZWxmLmNvbXBvbmVudF9u'
        'YW1lcykKICAgICAgICAgICAgXSwKICAgICAgICAgICAgZGltPS0xLAogICAgICAgICkKICAgICAg'
        'ICByZWxpYWJpbGl0eSA9IHJlbGlhYmlsaXR5X2xvZ2l0cy5zaWdtb2lkKCkKICAgICAgICBnYXRl'
        'cyA9IHNlbGYuX3JvdXRlKGJhc2UuZmFtaWx5X2dhdGVzLCByZWxpYWJpbGl0eSwgcmVsaWFiaWxp'
        'dHlfbG9naXRzKQogICAgICAgIHN5c3RlbV9vdXRwdXQgPSBzZWxmLl9yZWNvbXB1dGVfc3lzdGVt'
        'X291dHB1dCgKICAgICAgICAgICAgYmFzZSwKICAgICAgICAgICAgZ2F0ZXMsCiAgICAgICAgICAg'
        'IGFsbG93ZWRfdHJhbnNpdGlvbnMsCiAgICAgICAgICAgIHJpc2tfc2NvcmUsCiAgICAgICAgKQog'
        'ICAgICAgIHJldHVybiBSZWxpYWJpbGl0eUZ1c2lvbk91dHB1dCgKICAgICAgICAgICAgc3lzdGVt'
        'PXN5c3RlbV9vdXRwdXQsCiAgICAgICAgICAgIHNlbWFudGljX2dhdGVzPWJhc2UuZmFtaWx5X2dh'
        'dGVzLAogICAgICAgICAgICBjb21wb25lbnRfcmVsaWFiaWxpdHlfbG9naXRzPXJlbGlhYmlsaXR5'
        'X2xvZ2l0cywKICAgICAgICAgICAgY29tcG9uZW50X3JlbGlhYmlsaXR5PXJlbGlhYmlsaXR5LAog'
        'ICAgICAgICkKCiAgICBkZWYgX3JvdXRlKAogICAgICAgIHNlbGYsCiAgICAgICAgc2VtYW50aWNf'
        'Z2F0ZXM6IFRlbnNvciwKICAgICAgICByZWxpYWJpbGl0eTogVGVuc29yLAogICAgICAgIHJlbGlh'
        'YmlsaXR5X2xvZ2l0czogVGVuc29yLAogICAgKSAtPiBUZW5zb3I6CiAgICAgICAgdW5pZm9ybSA9'
        'IHRvcmNoLmZ1bGxfbGlrZShzZW1hbnRpY19nYXRlcywgMS4wIC8gc2VtYW50aWNfZ2F0ZXMuc2hh'
        'cGVbLTFdKQogICAgICAgIG1vZGUgPSBzZWxmLmNvbmZpZy5yb3V0aW5nX21vZGUKICAgICAgICBp'
        'ZiBtb2RlID09ICJ1bmlmb3JtIjoKICAgICAgICAgICAgIyBQcmVzZXJ2ZSB0aGUgZnVsbCBtYXRj'
        'aGVkIGdyYXBoIHdoaWxlIGRpc2FibGluZyBib3RoIGxlYXJuZWQgc2lnbmFscy4KICAgICAgICAg'
        'ICAgcmV0dXJuIHVuaWZvcm0gKyAwLjAgKiAoc2VtYW50aWNfZ2F0ZXMgKyByZWxpYWJpbGl0eV9s'
        'b2dpdHMpCiAgICAgICAgaWYgbW9kZSA9PSAic2VtYW50aWMiOgogICAgICAgICAgICByZXR1cm4g'
        'c2VtYW50aWNfZ2F0ZXMgKyAwLjAgKiByZWxpYWJpbGl0eV9sb2dpdHMKCiAgICAgICAgZmxvb3Ig'
        'PSBzZWxmLmNvbmZpZy5wcm9iYWJpbGl0eV9mbG9vcgogICAgICAgIHJlbGlhYmlsaXR5X3Rlcm0g'
        'PSAoCiAgICAgICAgICAgIHNlbGYuY29uZmlnLnJlbGlhYmlsaXR5X2xvZ2l0X3N0cmVuZ3RoICog'
        'cmVsaWFiaWxpdHkuY2xhbXBfbWluKGZsb29yKS5sb2coKQogICAgICAgICkKICAgICAgICBpZiBt'
        'b2RlID09ICJyZWxpYWJpbGl0eV9vbmx5IjoKICAgICAgICAgICAgYWRhcHRpdmUgPSByZWxpYWJp'
        'bGl0eV90ZXJtLnNvZnRtYXgoZGltPS0xKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbWFu'
        'dGljX3Rlcm0gPSBzZW1hbnRpY19nYXRlcy5jbGFtcF9taW4oZmxvb3IpLmxvZygpCiAgICAgICAg'
        'ICAgIGFkYXB0aXZlID0gKHNlbWFudGljX3Rlcm0gKyByZWxpYWJpbGl0eV90ZXJtKS5zb2Z0bWF4'
        'KGRpbT0tMSkKICAgICAgICBtYXNzID0gc2VsZi5jb25maWcuc2hyaW5rYWdlX3RvX2FkYXB0aXZl'
        'CiAgICAgICAgcmV0dXJuICgxLjAgLSBtYXNzKSAqIHVuaWZvcm0gKyBtYXNzICogYWRhcHRpdmUK'
        'CiAgICBkZWYgX3JlY29tcHV0ZV9zeXN0ZW1fb3V0cHV0KAogICAgICAgIHNlbGYsCiAgICAgICAg'
        'YmFzZTogT2x5bXB1c1N5c3RlbU91dHB1dCwKICAgICAgICBnYXRlczogVGVuc29yLAogICAgICAg'
        'IGFsbG93ZWRfdHJhbnNpdGlvbnM6IFRlbnNvciwKICAgICAgICByaXNrX3Njb3JlOiBUZW5zb3Is'
        'CiAgICApIC0+IE9seW1wdXNTeXN0ZW1PdXRwdXQ6CiAgICAgICAgdmVjdG9ycyA9IGJhc2UuY29t'
        'cG9uZW50X3N0YXRlcwogICAgICAgIHdlaWdodGVkID0gdmVjdG9ycyAqIGdhdGVzLnVuc3F1ZWV6'
        'ZSgtMSkKICAgICAgICBwcmVfbm9ybWFsaXplZCA9IHdlaWdodGVkLnN1bShkaW09MSkKICAgICAg'
        'ICBmdXNlZCA9IHNlbGYuc3lzdGVtLmZ1c2lvbl9ub3JtKHByZV9ub3JtYWxpemVkKQoKICAgICAg'
        'ICBub3JtYWxpemVkX3ZlY3RvcnMgPSBubi5mdW5jdGlvbmFsLm5vcm1hbGl6ZSh2ZWN0b3JzLCBk'
        'aW09LTEpCiAgICAgICAgbm9ybWFsaXplZF9jZW50cm9pZCA9IG5uLmZ1bmN0aW9uYWwubm9ybWFs'
        'aXplKAogICAgICAgICAgICAobm9ybWFsaXplZF92ZWN0b3JzICogZ2F0ZXMudW5zcXVlZXplKC0x'
        'KSkuc3VtKGRpbT0xKSwgZGltPS0xCiAgICAgICAgKQogICAgICAgIGNvc2luZV90b19jZW50cm9p'
        'ZCA9IChub3JtYWxpemVkX3ZlY3RvcnMgKiBub3JtYWxpemVkX2NlbnRyb2lkLnVuc3F1ZWV6ZSgx'
        'KSkuc3VtKGRpbT0tMSkKICAgICAgICBkaXNhZ3JlZW1lbnQgPSAoKDEuMCAtIGNvc2luZV90b19j'
        'ZW50cm9pZCkuY2xhbXAoMC4wLCAyLjApICogZ2F0ZXMpLnN1bShkaW09LTEpIC8gMi4wCiAgICAg'
        'ICAgcmlza19kaXNhZ3JlZW1lbnQgPSAoCiAgICAgICAgICAgIGRpc2FncmVlbWVudCBpZiBzZWxm'
        'LmNvbmZpZy5zeXN0ZW0uZGlzYWdyZWVtZW50X3Jpc2sgZWxzZSBkaXNhZ3JlZW1lbnQgKiAwLjAK'
        'ICAgICAgICApCiAgICAgICAgcmlza19mZWF0dXJlcyA9IHRvcmNoLmNhdCgoZnVzZWQsIHJpc2tf'
        'ZGlzYWdyZWVtZW50LnVuc3F1ZWV6ZSgtMSkpLCBkaW09LTEpCiAgICAgICAgaW50cmluc2ljX3Jp'
        'c2sgPSBzZWxmLnN5c3RlbS5pbnRyaW5zaWNfcmlza19oZWFkKHJpc2tfZmVhdHVyZXMpLnNxdWVl'
        'emUoLTEpLnNpZ21vaWQoKQoKICAgICAgICBiYXRjaCA9IGZ1c2VkLnNoYXBlWzBdCiAgICAgICAg'
        'aG9zdF9yaXNrID0gcmlza19zY29yZS50byhkZXZpY2U9ZnVzZWQuZGV2aWNlLCBkdHlwZT1mdXNl'
        'ZC5kdHlwZSkKICAgICAgICBpZiBob3N0X3Jpc2suc2hhcGUgIT0gKGJhdGNoLCkgb3Igbm90IGJv'
        'b2wodG9yY2guaXNmaW5pdGUoaG9zdF9yaXNrKS5hbGwoKSk6CiAgICAgICAgICAgIHJhaXNlIFZh'
        'bHVlRXJyb3IoInJpc2sgc2NvcmUgbXVzdCBjb250YWluIG9uZSBmaW5pdGUgdmFsdWUgcGVyIGJh'
        'dGNoIGl0ZW0iKQogICAgICAgIGlmIGJvb2woKChob3N0X3Jpc2sgPCAwLjApIHwgKGhvc3Rfcmlz'
        'ayA+IDEuMCkpLmFueSgpKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmlzayBzY29y'
        'ZXMgbXVzdCBmYWxsIGluIFswLCAxXSIpCiAgICAgICAgZWZmZWN0aXZlX3Jpc2sgPSAxLjAgLSAo'
        'MS4wIC0gaG9zdF9yaXNrKSAqICgxLjAgLSBpbnRyaW5zaWNfcmlzaykKICAgICAgICBhaW9uID0g'
        'c2VsZi5zeXN0ZW0uYWlvbihzZWxmLnN5c3RlbS5haW9uX3N0YXRlKGZ1c2VkKSwgYWxsb3dlZF90'
        'cmFuc2l0aW9ucywgZWZmZWN0aXZlX3Jpc2spCiAgICAgICAgcmF3X2RlY2lzaW9uID0gc2VsZi5z'
        'eXN0ZW0uZGVjaXNpb25faGVhZChmdXNlZCkKCiAgICAgICAgbGVhdmVfb25lX291dDogbGlzdFtU'
        'ZW5zb3JdID0gW10KICAgICAgICBmb3IgaW5kZXggaW4gcmFuZ2UodmVjdG9ycy5zaGFwZVsxXSk6'
        'CiAgICAgICAgICAgIHJlbWFpbmluZ19tYXNzID0gKDEuMCAtIGdhdGVzWzosIGluZGV4XSkuY2xh'
        'bXBfbWluKDFlLTYpLnVuc3F1ZWV6ZSgtMSkKICAgICAgICAgICAgY291bnRlcmZhY3R1YWwgPSBz'
        'ZWxmLnN5c3RlbS5mdXNpb25fbm9ybSgKICAgICAgICAgICAgICAgIChwcmVfbm9ybWFsaXplZCAt'
        'IHdlaWdodGVkWzosIGluZGV4XSkgLyByZW1haW5pbmdfbWFzcwogICAgICAgICAgICApCiAgICAg'
        'ICAgICAgIGxlYXZlX29uZV9vdXQuYXBwZW5kKHNlbGYuc3lzdGVtLmRlY2lzaW9uX2hlYWQoY291'
        'bnRlcmZhY3R1YWwpKQogICAgICAgIGNvdW50ZXJmYWN0dWFsX2xvZ2l0cyA9IHRvcmNoLnN0YWNr'
        'KGxlYXZlX29uZV9vdXQsIGRpbT0xKQogICAgICAgIGRlY2lzaW9uX2F2YWlsYWJsZSA9IH5haW9u'
        'LmZvcmNlZF9zdG9wCiAgICAgICAgZGVjaXNpb24gPSB0b3JjaC53aGVyZSgKICAgICAgICAgICAg'
        'ZGVjaXNpb25fYXZhaWxhYmxlLnVuc3F1ZWV6ZSgtMSksIHJhd19kZWNpc2lvbiwgdG9yY2guemVy'
        'b3NfbGlrZShyYXdfZGVjaXNpb24pCiAgICAgICAgKQogICAgICAgIHVuY2VydGFpbnR5ID0gbm4u'
        'ZnVuY3Rpb25hbC5zb2Z0cGx1cyhzZWxmLnN5c3RlbS51bmNlcnRhaW50eV9oZWFkKGZ1c2VkKS5z'
        'cXVlZXplKC0xKSkgKyAxZS02CiAgICAgICAgcmV0dXJuIHJlcGxhY2UoCiAgICAgICAgICAgIGJh'
        'c2UsCiAgICAgICAgICAgIGFpb249YWlvbiwKICAgICAgICAgICAgZmFtaWx5X2dhdGVzPWdhdGVz'
        'LAogICAgICAgICAgICBmdXNlZF9zdGF0ZT1mdXNlZCwKICAgICAgICAgICAgY291bnRlcmZhY3R1'
        'YWxfZGVjaXNpb25fbG9naXRzPWNvdW50ZXJmYWN0dWFsX2xvZ2l0cywKICAgICAgICAgICAgcmF3'
        'X2RlY2lzaW9uX2xvZ2l0cz1yYXdfZGVjaXNpb24sCiAgICAgICAgICAgIGRlY2lzaW9uX2xvZ2l0'
        'cz1kZWNpc2lvbiwKICAgICAgICAgICAgZGVjaXNpb25fYXZhaWxhYmxlPWRlY2lzaW9uX2F2YWls'
        'YWJsZSwKICAgICAgICAgICAgdW5jZXJ0YWludHk9dW5jZXJ0YWludHksCiAgICAgICAgICAgIGZh'
        'bWlseV9kaXNhZ3JlZW1lbnQ9ZGlzYWdyZWVtZW50LAogICAgICAgICAgICBpbnRyaW5zaWNfcmlz'
        'az1pbnRyaW5zaWNfcmlzaywKICAgICAgICAgICAgZWZmZWN0aXZlX3Jpc2s9ZWZmZWN0aXZlX3Jp'
        'c2ssCiAgICAgICAgKQoKCmRlZiByZWxpYWJpbGl0eV9mdXNpb25fbG9zcygKICAgIG91dHB1dDog'
        'UmVsaWFiaWxpdHlGdXNpb25PdXRwdXQsCiAgICAqLAogICAgZGVjaXNpb25fdGFyZ2V0czogVGVu'
        'c29yLAogICAgdHJhbnNpdGlvbl90YXJnZXRzOiBUZW5zb3IsCiAgICBuZXh0X3N0YXRlX3Rhcmdl'
        'dHM6IFRlbnNvciwKICAgIGNvbXBvbmVudF9yZWxpYWJpbGl0eV90YXJnZXRzOiBUZW5zb3IsCiAg'
        'ICBpbnRyaW5zaWNfcmlza190YXJnZXRzOiBUZW5zb3IgfCBOb25lID0gTm9uZSwKICAgIHRlbXBv'
        'cmFsX21hc2s6IFRlbnNvciB8IE5vbmUgPSBOb25lLAogICAgY29uZmlnOiBSZWxpYWJpbGl0eUxv'
        'c3NDb25maWcgfCBOb25lID0gTm9uZSwKKSAtPiBSZWxpYWJpbGl0eUZ1c2lvbkxvc3M6CiAgICAi'
        'IiJPcHRpbWl6ZSB0YXNrIGJlaGF2aW9yIGFuZCBpbmRlcGVuZGVudGx5IG9ic2VydmFibGUgcmVs'
        'aWFiaWxpdHkgdGFyZ2V0cy4iIiIKCiAgICB3ZWlnaHRzID0gY29uZmlnIG9yIFJlbGlhYmlsaXR5'
        'TG9zc0NvbmZpZygpCiAgICB0YXJnZXRzID0gY29tcG9uZW50X3JlbGlhYmlsaXR5X3RhcmdldHMu'
        'dG8oCiAgICAgICAgZGV2aWNlPW91dHB1dC5jb21wb25lbnRfcmVsaWFiaWxpdHkuZGV2aWNlLAog'
        'ICAgICAgIGR0eXBlPW91dHB1dC5jb21wb25lbnRfcmVsaWFiaWxpdHkuZHR5cGUsCiAgICApCiAg'
        'ICBpZiB0YXJnZXRzLnNoYXBlICE9IG91dHB1dC5jb21wb25lbnRfcmVsaWFiaWxpdHkuc2hhcGU6'
        'CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVsaWFiaWxpdHkgdGFyZ2V0cyBtdXN0IG1hdGNo'
        'IFtiYXRjaCwgY29tcG9uZW50XSIpCiAgICBpZiBub3QgYm9vbCh0b3JjaC5pc2Zpbml0ZSh0YXJn'
        'ZXRzKS5hbGwoKSkgb3IgYm9vbCgoKHRhcmdldHMgPCAwLjApIHwgKHRhcmdldHMgPiAxLjApKS5h'
        'bnkoKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigicmVsaWFiaWxpdHkgdGFyZ2V0cyBtdXN0'
        'IGJlIGZpbml0ZSB2YWx1ZXMgaW4gWzAsIDFdIikKICAgIHN5c3RlbV9sb3NzID0gb2x5bXB1c19z'
        'eXN0ZW1fbG9zcygKICAgICAgICBvdXRwdXQuc3lzdGVtLAogICAgICAgIGRlY2lzaW9uX3Rhcmdl'
        'dHM9ZGVjaXNpb25fdGFyZ2V0cywKICAgICAgICB0cmFuc2l0aW9uX3RhcmdldHM9dHJhbnNpdGlv'
        'bl90YXJnZXRzLAogICAgICAgIG5leHRfc3RhdGVfdGFyZ2V0cz1uZXh0X3N0YXRlX3RhcmdldHMs'
        'CiAgICAgICAgaW50cmluc2ljX3Jpc2tfdGFyZ2V0cz1pbnRyaW5zaWNfcmlza190YXJnZXRzLAog'
        'ICAgICAgIHRlbXBvcmFsX21hc2s9dGVtcG9yYWxfbWFzaywKICAgICAgICAqKndlaWdodHMuYmFz'
        'ZV93ZWlnaHRzKCksCiAgICApCiAgICByZWxpYWJpbGl0eV9jYWxpYnJhdGlvbiA9IG5uLmZ1bmN0'
        'aW9uYWwuYmluYXJ5X2Nyb3NzX2VudHJvcHlfd2l0aF9sb2dpdHMoCiAgICAgICAgb3V0cHV0LmNv'
        'bXBvbmVudF9yZWxpYWJpbGl0eV9sb2dpdHMsCiAgICAgICAgdGFyZ2V0cywKICAgICkKICAgIHRv'
        'dGFsID0gc3lzdGVtX2xvc3MudG90YWwgKyB3ZWlnaHRzLnJlbGlhYmlsaXR5X3dlaWdodCAqIHJl'
        'bGlhYmlsaXR5X2NhbGlicmF0aW9uCiAgICByZXR1cm4gUmVsaWFiaWxpdHlGdXNpb25Mb3NzKAog'
        'ICAgICAgIHRvdGFsPXRvdGFsLAogICAgICAgIHN5c3RlbT1zeXN0ZW1fbG9zcywKICAgICAgICBy'
        'ZWxpYWJpbGl0eV9jYWxpYnJhdGlvbj1yZWxpYWJpbGl0eV9jYWxpYnJhdGlvbiwKICAgICkK'
    ),
    'olympus/models/robust_fusion_v5.py': (
        'IiIiTWF0Y2hlZCByb3V0aW5nLWZlYXR1cmUgYW5kIHRlbXBvcmFsLXNraXAgYWJsYXRpb25zIG9u'
        'IGZyb3plbiB2NCBiYW5rcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMK'
        'CmltcG9ydCBjb3B5CgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgVGVuc29yLCBubgoK'
        'ZnJvbSBvbHltcHVzLm1vZGVscy5vcmFjbGVfZnVzaW9uX3Y0IGltcG9ydCBub3JtYWxpemVfcXVh'
        'bGl0eSwgcm91dGVyX2ZlYXR1cmVzCgpST1VURVMgPSAoCiAgICAidW5pZm9ybSIsCiAgICAiZml4'
        'ZWQiLAogICAgInNlbWFudGljIiwKICAgICJwcmVkaWN0aW9uX29ubHkiLAogICAgInV0aWxpdHki'
        'LAogICAgImNvcnJ1cHRpb24iLAogICAgImJsZW5kIiwKICAgICJvcmFjbGVfdGFzayIsCikKVEVN'
        'UE9SQUwgPSAoInY0IiwgImRpcmVjdCIsICJyZXNpZHVhbCIsICJwZXJzaXN0ZW5jZSIpCgoKY2xh'
        'c3MgSGVhZHMobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAg'
        'ICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZml4ZWQgPSBubi5QYXJhbWV0ZXIo'
        'dG9yY2guemVyb3MoNSkpCiAgICAgICAgbmV0ID0gbm4uU2VxdWVudGlhbChubi5MYXllck5vcm0o'
        'MTA1KSwgbm4uTGluZWFyKDEwNSwgMzIpLCBubi5HRUxVKCksIG5uLkxpbmVhcigzMiwgNSkpCiAg'
        'ICAgICAgc2VsZi5yb3V0aW5nID0gbm4uTW9kdWxlRGljdCgKICAgICAgICAgICAgewogICAgICAg'
        'ICAgICAgICAgbmFtZTogY29weS5kZWVwY29weShuZXQpCiAgICAgICAgICAgICAgICBmb3IgbmFt'
        'ZSBpbiAoInNlbWFudGljIiwgInByZWRpY3Rpb25fb25seSIsICJ1dGlsaXR5IiwgImNvcnJ1cHRp'
        'b24iKQogICAgICAgICAgICB9CiAgICAgICAgKQogICAgICAgICMgSWRlbnRpY2FsIGluaXRpYWxp'
        'emF0aW9uL3BhcmFtZXRlcnMvdXBkYXRlcyBmb3IgdGhlIHRlbXBvcmFsIHBhaXIuCiAgICAgICAg'
        'dGVtcG9yYWwgPSBubi5TZXF1ZW50aWFsKG5uLkxpbmVhcig4OCwgNjQpLCBubi5HRUxVKCksIG5u'
        'LkxpbmVhcig2NCwgNSkpCiAgICAgICAgYXNzZXJ0IGlzaW5zdGFuY2UodGVtcG9yYWxbLTFdLCBu'
        'bi5MaW5lYXIpCiAgICAgICAgbm4uaW5pdC56ZXJvc18odGVtcG9yYWxbLTFdLndlaWdodCkKICAg'
        'ICAgICBubi5pbml0Lnplcm9zXyh0ZW1wb3JhbFstMV0uYmlhcykKICAgICAgICBzZWxmLnRlbXBv'
        'cmFsID0gbm4uTW9kdWxlRGljdCgKICAgICAgICAgICAge25hbWU6IGNvcHkuZGVlcGNvcHkodGVt'
        'cG9yYWwpIGZvciBuYW1lIGluICgiZGlyZWN0IiwgInJlc2lkdWFsIil9CiAgICAgICAgKQoKICAg'
        'IGRlZiBmb3J3YXJkKAogICAgICAgIHNlbGYsIHo6IFRlbnNvciwgbG9naXRzOiBUZW5zb3IsIHN0'
        'YXRlczogVGVuc29yLCBhY3Rpb25zOiBUZW5zb3IKICAgICkgLT4gdHVwbGVbZGljdFtzdHIsIFRl'
        'bnNvcl0sIGRpY3Rbc3RyLCBUZW5zb3JdXToKICAgICAgICBmZWF0dXJlcyA9IHJvdXRlcl9mZWF0'
        'dXJlcyh6LCBsb2dpdHMpCiAgICAgICAgIyBTYW1lIDEwNS1kaW1lbnNpb25hbCBuZXR3b3JrOyB6'
        'ZXJvaW5nIE9OTFkgdGhlIGxhdGVudCBmZWF0dXJlcyBpc29sYXRlcwogICAgICAgICMgcmVwcmVz'
        'ZW50YXRpb24gZGVwZW5kZW5jZSB3aXRob3V0IGNoYW5naW5nIHBhcmFtZXRlciBjb3VudHMuCiAg'
        'ICAgICAgcHJlZGljdGlvbl9mZWF0dXJlcyA9IHRvcmNoLmNhdCgodG9yY2guemVyb3NfbGlrZShm'
        'ZWF0dXJlc1s6LCA6ODBdKSwgZmVhdHVyZXNbOiwgODA6XSksIC0xKQogICAgICAgIHNjb3JlcyA9'
        'IHsKICAgICAgICAgICAgbmFtZTogbmV0KHByZWRpY3Rpb25fZmVhdHVyZXMgaWYgbmFtZSA9PSAi'
        'cHJlZGljdGlvbl9vbmx5IiBlbHNlIGZlYXR1cmVzKQogICAgICAgICAgICBmb3IgbmFtZSwgbmV0'
        'IGluIHNlbGYucm91dGluZy5pdGVtcygpCiAgICAgICAgfQogICAgICAgIHNjb3Jlc1siZml4ZWQi'
        'XSA9IHNlbGYuZml4ZWQuZXhwYW5kKGxlbih6KSwgLTEpCiAgICAgICAgY29udGV4dCA9IHouZmxh'
        'dHRlbigxKS5kZXRhY2goKVs6LCBOb25lXS5leHBhbmQoLTEsIHN0YXRlcy5zaGFwZVsxXSwgLTEp'
        'CiAgICAgICAgcmF3ID0gdG9yY2guY2F0KChzdGF0ZXMsIGFjdGlvbnMsIGNvbnRleHQpLCAtMSkK'
        'ICAgICAgICB0ZW1wb3JhbCA9IHsKICAgICAgICAgICAgbmFtZTogbmV0KHJhdykgKyAoc3RhdGVz'
        'IGlmIG5hbWUgPT0gInJlc2lkdWFsIiBlbHNlIDApCiAgICAgICAgICAgIGZvciBuYW1lLCBuZXQg'
        'aW4gc2VsZi50ZW1wb3JhbC5pdGVtcygpCiAgICAgICAgfQogICAgICAgIHJldHVybiBzY29yZXMs'
        'IHRlbXBvcmFsCgoKZGVmIHJvdXRlZCgKICAgIGxvZ2l0czogVGVuc29yLCBzY29yZXM6IGRpY3Rb'
        'c3RyLCBUZW5zb3JdLCBsYWJlbHM6IFRlbnNvciB8IE5vbmUgPSBOb25lCikgLT4gZGljdFtzdHIs'
        'IFRlbnNvcl06CiAgICBwID0gbG9naXRzLnNvZnRtYXgoLTEpCiAgICBnYXRlcyA9IHtuYW1lOiBz'
        'Y29yZXNbbmFtZV0uc29mdG1heCgtMSkgZm9yIG5hbWUgaW4gKCJmaXhlZCIsICJzZW1hbnRpYyIs'
        'ICJwcmVkaWN0aW9uX29ubHkiKX0KICAgIGdhdGVzLnVwZGF0ZSgKICAgICAgICB7bmFtZTogbm9y'
        'bWFsaXplX3F1YWxpdHkoc2NvcmVzW25hbWVdLnNpZ21vaWQoKSkgZm9yIG5hbWUgaW4gKCJ1dGls'
        'aXR5IiwgImNvcnJ1cHRpb24iKX0KICAgICkKICAgIGdhdGVzWyJ1bmlmb3JtIl0gPSB0b3JjaC5v'
        'bmVzX2xpa2UocFsuLi4sIDBdKSAvIDUKICAgIGdhdGVzWyJibGVuZCJdID0gKGdhdGVzWyJwcmVk'
        'aWN0aW9uX29ubHkiXSArIGdhdGVzWyJ1dGlsaXR5Il0pIC8gMgogICAgdmFsdWVzID0ge25hbWU6'
        'IChnYXRlWy4uLiwgTm9uZV0gKiBwKS5zdW0oMSkgZm9yIG5hbWUsIGdhdGUgaW4gZ2F0ZXMuaXRl'
        'bXMoKX0KICAgIGlmIGxhYmVscyBpcyBub3QgTm9uZToKICAgICAgICBiZXN0ID0gcC5nYXRoZXIo'
        'MiwgbGFiZWxzWzosIE5vbmUsIE5vbmVdLmV4cGFuZCgtMSwgNSwgMSkpLnNxdWVlemUoLTEpLmFy'
        'Z21heCgtMSkKICAgICAgICB2YWx1ZXNbIm9yYWNsZV90YXNrIl0gPSBwW3RvcmNoLmFyYW5nZShs'
        'ZW4ocCkpLCBiZXN0XQogICAgcmV0dXJuIHZhbHVlcwo='
    ),
    'olympus/models/runtime.py': (
        'IiIiUmVzb3VyY2UtZ292ZXJuZWQgcnVudGltZSBhbmQgaW1tdXRhYmxlIGNoZWNrcG9pbnRzIGZv'
        'ciB0aGUgT2x5bXB1cyBzeXN0ZW0uIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRp'
        'b25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW8KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmltcG9y'
        'dCB0aW1lCmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBJdGVyYXRvciwgTWFwcGluZwpmcm9t'
        'IGNvbnRleHRsaWIgaW1wb3J0IGNvbnRleHRtYW5hZ2VyLCBudWxsY29udGV4dApmcm9tIGRhdGFj'
        'bGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdGVt'
        'cGZpbGUgaW1wb3J0IE5hbWVkVGVtcG9yYXJ5RmlsZQpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBM'
        'aXRlcmFsLCBjYXN0CgppbXBvcnQgdG9yY2gKZnJvbSBweWRhbnRpYyBpbXBvcnQgRmllbGQKZnJv'
        'bSB0b3JjaCBpbXBvcnQgVGVuc29yLCBubgoKZnJvbSBvbHltcHVzLmNvcmUuc2NoZW1hcyBpbXBv'
        'cnQgU3RyaWN0TW9kZWwKZnJvbSBvbHltcHVzLmZvdW5kcnkucmVzb3VyY2VzIGltcG9ydCBSZXNv'
        'dXJjZUdvdmVybm9yCmZyb20gb2x5bXB1cy5tb2RlbHMub2x5bXB1c19zeXN0ZW0gaW1wb3J0ICgK'
        'ICAgIE9seW1wdXNSZWZlcmVuY2VTeXN0ZW0sCiAgICBPbHltcHVzU3lzdGVtQ29uZmlnLAogICAg'
        'T2x5bXB1c1N5c3RlbUxvc3MsCiAgICBPbHltcHVzU3lzdGVtTG9zc0NvbmZpZywKICAgIE9seW1w'
        'dXNTeXN0ZW1PdXRwdXQsCiAgICBidWlsZF9vbHltcHVzX3N5c3RlbSwKICAgIG9seW1wdXNfc3lz'
        'dGVtX2xvc3MsCikKClJ1bnRpbWVEZXZpY2UgPSBMaXRlcmFsWyJhdXRvIiwgImNwdSIsICJjdWRh'
        'IiwgIm1wcyJdClJ1bnRpbWVJbnB1dCA9IFRlbnNvciB8IGRpY3Rbc3RyLCBUZW5zb3JdCgoKY2xh'
        'c3MgT2x5bXB1c1J1bnRpbWVDb25maWcoU3RyaWN0TW9kZWwpOgogICAgIiIiRXhlY3V0aW9uIGNv'
        'bnRyb2xzIHRoYXQgYXJlIHNlcGFyYXRlIGZyb20gbGVhcm5lZCBtb2RlbCBjb25maWd1cmF0aW9u'
        'LiIiIgoKICAgIGRldmljZTogUnVudGltZURldmljZSA9ICJhdXRvIgogICAgbWl4ZWRfcHJlY2lz'
        'aW9uOiBib29sID0gRmFsc2UKICAgIGNvbXBpbGVfbW9kZWw6IGJvb2wgPSBGYWxzZQogICAgZGV0'
        'ZXJtaW5pc3RpYzogYm9vbCA9IFRydWUKICAgIHJlc291cmNlX3ByZWZsaWdodDogYm9vbCA9IFRy'
        'dWUKICAgIHdvcmtsb2FkX2xvY2s6IFBhdGggPSBQYXRoKCJhcnRpZmFjdHMvbG9ja3Mvb2x5bXB1'
        'cy1zeXN0ZW0ubG9jayIpCiAgICBtaW5fYXZhaWxhYmxlX2J5dGVzOiBpbnQgPSBGaWVsZChkZWZh'
        'dWx0PTIgKiAxMDI0KiozLCBnZT0yNTYgKiAxMDI0KioyKQogICAgbWF4X3N3YXBfZnJhY3Rpb246'
        'IGZsb2F0ID0gRmllbGQoZGVmYXVsdD0wLjkwLCBndD0wLjAsIGxlPTEuMCkKCgpjbGFzcyBPbHlt'
        'cHVzQ2hlY2twb2ludFJlY29yZChTdHJpY3RNb2RlbCk6CiAgICBzY2hlbWFfdmVyc2lvbjogTGl0'
        'ZXJhbFsxXSA9IDEKICAgIHBhdGg6IHN0cgogICAgc2hhMjU2OiBzdHIgPSBGaWVsZChwYXR0ZXJu'
        'PXIiXlswLTlhLWZdezY0fSQiKQogICAgcGFyYW1ldGVyczogaW50ID0gRmllbGQoZ3Q9MCkKICAg'
        'IHN5c3RlbV9jb25maWc6IE9seW1wdXNTeXN0ZW1Db25maWcKICAgIG1ldGFkYXRhOiBkaWN0W3N0'
        'ciwgc3RyXSA9IEZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKCkBkYXRhY2xhc3Moc2xvdHM9'
        'VHJ1ZSkKY2xhc3MgT2x5bXB1c0luZmVyZW5jZVJlc3VsdDoKICAgIG91dHB1dDogT2x5bXB1c1N5'
        'c3RlbU91dHB1dAogICAgZWxhcHNlZF9zZWNvbmRzOiBmbG9hdAogICAgZGV2aWNlOiBzdHIKICAg'
        'IG1peGVkX3ByZWNpc2lvbl9lZmZlY3RpdmU6IGJvb2wKCgpAZGF0YWNsYXNzKHNsb3RzPVRydWUp'
        'CmNsYXNzIE9seW1wdXNUcmFpblN0ZXBSZXN1bHQ6CiAgICBvdXRwdXQ6IE9seW1wdXNTeXN0ZW1P'
        'dXRwdXQKICAgIGxvc3M6IE9seW1wdXNTeXN0ZW1Mb3NzCiAgICBncmFkaWVudF9ub3JtOiBmbG9h'
        'dAoKCmRlZiBfc2hhMjU2KHBheWxvYWQ6IGJ5dGVzKSAtPiBzdHI6CiAgICByZXR1cm4gaGFzaGxp'
        'Yi5zaGEyNTYocGF5bG9hZCkuaGV4ZGlnZXN0KCkKCgpkZWYgcmVzb2x2ZV9vbHltcHVzX2Rldmlj'
        'ZShyZXF1ZXN0ZWQ6IFJ1bnRpbWVEZXZpY2UpIC0+IHRvcmNoLmRldmljZToKICAgIGlmIHJlcXVl'
        'c3RlZCA9PSAiYXV0byI6CiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAg'
        'ICAgICAgICAgcmV0dXJuIHRvcmNoLmRldmljZSgiY3VkYSIpCiAgICAgICAgaWYgdG9yY2guYmFj'
        'a2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByZXR1cm4gdG9yY2guZGV2aWNl'
        'KCJtcHMiKQogICAgICAgIHJldHVybiB0b3JjaC5kZXZpY2UoImNwdSIpCiAgICBpZiByZXF1ZXN0'
        'ZWQgPT0gImN1ZGEiIGFuZCBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICBy'
        'YWlzZSBSdW50aW1lRXJyb3IoIkNVREEgd2FzIHJlcXVlc3RlZCBidXQgaXMgdW5hdmFpbGFibGUi'
        'KQogICAgaWYgcmVxdWVzdGVkID09ICJtcHMiIGFuZCBub3QgdG9yY2guYmFja2VuZHMubXBzLmlz'
        'X2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigiTVBTIHdhcyByZXF1ZXN0'
        'ZWQgYnV0IGlzIHVuYXZhaWxhYmxlIikKICAgIHJldHVybiB0b3JjaC5kZXZpY2UocmVxdWVzdGVk'
        'KQoKCmRlZiBfbW92ZV9pbnB1dHMoCiAgICBpbnB1dHM6IE1hcHBpbmdbc3RyLCBSdW50aW1lSW5w'
        'dXRdLCBkZXZpY2U6IHRvcmNoLmRldmljZQopIC0+IGRpY3Rbc3RyLCBSdW50aW1lSW5wdXRdOgog'
        'ICAgbW92ZWQ6IGRpY3Rbc3RyLCBSdW50aW1lSW5wdXRdID0ge30KICAgIGZvciBuYW1lLCB2YWx1'
        'ZSBpbiBpbnB1dHMuaXRlbXMoKToKICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBUZW5zb3Ip'
        'OgogICAgICAgICAgICBtb3ZlZFtuYW1lXSA9IHZhbHVlLnRvKGRldmljZT1kZXZpY2UpCiAgICAg'
        'ICAgZWxzZToKICAgICAgICAgICAgbW92ZWRbbmFtZV0gPSB7a2V5OiB0ZW5zb3IudG8oZGV2aWNl'
        'PWRldmljZSkgZm9yIGtleSwgdGVuc29yIGluIHZhbHVlLml0ZW1zKCl9CiAgICByZXR1cm4gbW92'
        'ZWQKCgpjbGFzcyBPbHltcHVzU3lzdGVtUnVudGltZToKICAgICIiIkJvdW5kZWQgaW5mZXJlbmNl'
        'IHJ1bnRpbWUgd2l0aCBleHBsaWNpdCByZXNvdXJjZSBhbmQgcHJlY2lzaW9uIGNvbnRyb2xzLiIi'
        'IgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIG1vZGVsOiBPbHltcHVz'
        'UmVmZXJlbmNlU3lzdGVtLAogICAgICAgIGNvbmZpZzogT2x5bXB1c1J1bnRpbWVDb25maWcgfCBO'
        'b25lID0gTm9uZSwKICAgICkgLT4gTm9uZToKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZpZyBv'
        'ciBPbHltcHVzUnVudGltZUNvbmZpZygpCiAgICAgICAgc2VsZi5kZXZpY2UgPSByZXNvbHZlX29s'
        'eW1wdXNfZGV2aWNlKHNlbGYuY29uZmlnLmRldmljZSkKICAgICAgICB3aXRoIHNlbGYuX3Jlc291'
        'cmNlX2NvbnRleHQoKToKICAgICAgICAgICAgc2VsZi5tb2RlbCA9IG1vZGVsLnRvKHNlbGYuZGV2'
        'aWNlKQogICAgICAgICAgICBzZWxmLl9jYWxsYWJsZTogbm4uTW9kdWxlID0gc2VsZi5tb2RlbAog'
        'ICAgICAgICAgICBpZiBzZWxmLmNvbmZpZy5jb21waWxlX21vZGVsOgogICAgICAgICAgICAgICAg'
        'c2VsZi5fY2FsbGFibGUgPSBjYXN0KG5uLk1vZHVsZSwgdG9yY2guY29tcGlsZShzZWxmLm1vZGVs'
        'LCBkeW5hbWljPUZhbHNlKSkKCiAgICBkZWYgX3Jlc291cmNlX2NvbnRleHQoc2VsZikgLT4gUmVz'
        'b3VyY2VHb3Zlcm5vciB8IG51bGxjb250ZXh0W05vbmVdOgogICAgICAgIGlmIG5vdCBzZWxmLmNv'
        'bmZpZy5yZXNvdXJjZV9wcmVmbGlnaHQ6CiAgICAgICAgICAgIHJldHVybiBudWxsY29udGV4dCgp'
        'CiAgICAgICAgcmV0dXJuIFJlc291cmNlR292ZXJub3IoCiAgICAgICAgICAgIHNlbGYuY29uZmln'
        'Lndvcmtsb2FkX2xvY2ssCiAgICAgICAgICAgIG1pbl9hdmFpbGFibGVfYnl0ZXM9c2VsZi5jb25m'
        'aWcubWluX2F2YWlsYWJsZV9ieXRlcywKICAgICAgICAgICAgbWF4X3N3YXBfZnJhY3Rpb249c2Vs'
        'Zi5jb25maWcubWF4X3N3YXBfZnJhY3Rpb24sCiAgICAgICAgKQoKICAgIEBwcm9wZXJ0eQogICAg'
        'ZGVmIG1peGVkX3ByZWNpc2lvbl9lZmZlY3RpdmUoc2VsZikgLT4gYm9vbDoKICAgICAgICByZXR1'
        'cm4gc2VsZi5jb25maWcubWl4ZWRfcHJlY2lzaW9uIGFuZCBzZWxmLmRldmljZS50eXBlIGluIHsi'
        'Y3VkYSIsICJtcHMifQoKICAgIGRlZiBpbmZlcihzZWxmLCBpbnB1dHM6IE1hcHBpbmdbc3RyLCBS'
        'dW50aW1lSW5wdXRdKSAtPiBPbHltcHVzSW5mZXJlbmNlUmVzdWx0OgogICAgICAgIHByZWNpc2lv'
        'biA9ICgKICAgICAgICAgICAgdG9yY2guYXV0b2Nhc3QoZGV2aWNlX3R5cGU9c2VsZi5kZXZpY2Uu'
        'dHlwZSwgZHR5cGU9dG9yY2guZmxvYXQxNikKICAgICAgICAgICAgaWYgc2VsZi5taXhlZF9wcmVj'
        'aXNpb25fZWZmZWN0aXZlCiAgICAgICAgICAgIGVsc2UgbnVsbGNvbnRleHQoKQogICAgICAgICkK'
        'ICAgICAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHdpdGggKAogICAg'
        'ICAgICAgICBzZWxmLl9yZXNvdXJjZV9jb250ZXh0KCksCiAgICAgICAgICAgIF9kZXRlcm1pbmlz'
        'dGljX2FsZ29yaXRobXMoc2VsZi5jb25maWcuZGV0ZXJtaW5pc3RpYyksCiAgICAgICAgICAgIF9l'
        'dmFsdWF0aW9uX21vZGUoc2VsZi5tb2RlbCksCiAgICAgICAgICAgIHByZWNpc2lvbiwKICAgICAg'
        'ICApOgogICAgICAgICAgICBtb3ZlZCA9IF9tb3ZlX2lucHV0cyhpbnB1dHMsIHNlbGYuZGV2aWNl'
        'KQogICAgICAgICAgICBvdXRwdXQgPSBjYXN0KE9seW1wdXNTeXN0ZW1PdXRwdXQsIHNlbGYuX2Nh'
        'bGxhYmxlKCoqbW92ZWQpKQogICAgICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0g'
        'c3RhcnRlZAogICAgICAgIGlmIG5vdCBib29sKHRvcmNoLmlzZmluaXRlKG91dHB1dC5yYXdfZGVj'
        'aXNpb25fbG9naXRzKS5hbGwoKSk6CiAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJv'
        'cigiT2x5bXB1cyBpbmZlcmVuY2UgcHJvZHVjZWQgbm9uLWZpbml0ZSBkZWNpc2lvbiBsb2dpdHMi'
        'KQogICAgICAgIHJldHVybiBPbHltcHVzSW5mZXJlbmNlUmVzdWx0KAogICAgICAgICAgICBvdXRw'
        'dXQ9b3V0cHV0LAogICAgICAgICAgICBlbGFwc2VkX3NlY29uZHM9ZWxhcHNlZCwKICAgICAgICAg'
        'ICAgZGV2aWNlPXN0cihzZWxmLmRldmljZSksCiAgICAgICAgICAgIG1peGVkX3ByZWNpc2lvbl9l'
        'ZmZlY3RpdmU9c2VsZi5taXhlZF9wcmVjaXNpb25fZWZmZWN0aXZlLAogICAgICAgICkKCgpAY29u'
        'dGV4dG1hbmFnZXIKZGVmIF9ldmFsdWF0aW9uX21vZGUobW9kZWw6IG5uLk1vZHVsZSkgLT4gSXRl'
        'cmF0b3JbTm9uZV06CiAgICAiIiJSZXN0b3JlIGV2ZXJ5IG1vZHVsZSdzIG1vZGUsIGluY2x1ZGlu'
        'ZyBkZWxpYmVyYXRlbHkgZnJvemVuIHN1Ym1vZHVsZXMuIiIiCgogICAgcHJldmlvdXNfbW9kZXMg'
        'PSB0dXBsZSgobW9kdWxlLCBtb2R1bGUudHJhaW5pbmcpIGZvciBtb2R1bGUgaW4gbW9kZWwubW9k'
        'dWxlcygpKQogICAgbW9kZWwuZXZhbCgpCiAgICB0cnk6CiAgICAgICAgeWllbGQKICAgIGZpbmFs'
        'bHk6CiAgICAgICAgZm9yIG1vZHVsZSwgdHJhaW5pbmcgaW4gcHJldmlvdXNfbW9kZXM6CiAgICAg'
        'ICAgICAgIG1vZHVsZS50cmFpbmluZyA9IHRyYWluaW5nCgoKQGNvbnRleHRtYW5hZ2VyCmRlZiBf'
        'ZGV0ZXJtaW5pc3RpY19hbGdvcml0aG1zKGVuYWJsZWQ6IGJvb2wpIC0+IEl0ZXJhdG9yW05vbmVd'
        'OgogICAgcHJldmlvdXNfZW5hYmxlZCA9IHRvcmNoLmFyZV9kZXRlcm1pbmlzdGljX2FsZ29yaXRo'
        'bXNfZW5hYmxlZCgpCiAgICBwcmV2aW91c193YXJuX29ubHkgPSB0b3JjaC5pc19kZXRlcm1pbmlz'
        'dGljX2FsZ29yaXRobXNfd2Fybl9vbmx5X2VuYWJsZWQoKQogICAgaWYgZW5hYmxlZDoKICAgICAg'
        'ICB0b3JjaC51c2VfZGV0ZXJtaW5pc3RpY19hbGdvcml0aG1zKFRydWUsIHdhcm5fb25seT1UcnVl'
        'KQogICAgdHJ5OgogICAgICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAgICAgICAg'
        'ICAgeWllbGQKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgZW5hYmxlZDoKICAgICAgICAgICAgdG9y'
        'Y2gudXNlX2RldGVybWluaXN0aWNfYWxnb3JpdGhtcygKICAgICAgICAgICAgICAgIHByZXZpb3Vz'
        'X2VuYWJsZWQsCiAgICAgICAgICAgICAgICB3YXJuX29ubHk9cHJldmlvdXNfd2Fybl9vbmx5LAog'
        'ICAgICAgICAgICApCgoKZGVmIHRyYWluX29seW1wdXNfc3RlcCgKICAgIG1vZGVsOiBPbHltcHVz'
        'UmVmZXJlbmNlU3lzdGVtLAogICAgb3B0aW1pemVyOiB0b3JjaC5vcHRpbS5PcHRpbWl6ZXIsCiAg'
        'ICBpbnB1dHM6IE1hcHBpbmdbc3RyLCBSdW50aW1lSW5wdXRdLAogICAgdGFyZ2V0czogTWFwcGlu'
        'Z1tzdHIsIFRlbnNvcl0sCiAgICAqLAogICAgZ3JhZGllbnRfY2xpcF9ub3JtOiBmbG9hdCA9IDEu'
        'MCwKICAgIGxvc3NfY29uZmlnOiBPbHltcHVzU3lzdGVtTG9zc0NvbmZpZyB8IE5vbmUgPSBOb25l'
        'LAopIC0+IE9seW1wdXNUcmFpblN0ZXBSZXN1bHQ6CiAgICAiIiJFeGVjdXRlIG9uZSBmaW5pdGUs'
        'IGNsaXBwZWQgam9pbnQtc3lzdGVtIG9wdGltaXphdGlvbiBzdGVwLiIiIgoKICAgIGlmIG5vdCBt'
        'YXRoLmlzZmluaXRlKGdyYWRpZW50X2NsaXBfbm9ybSkgb3IgZ3JhZGllbnRfY2xpcF9ub3JtIDw9'
        'IDAuMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJncmFkaWVudCBjbGlwIG5vcm0gbXVzdCBi'
        'ZSBmaW5pdGUgYW5kIHBvc2l0aXZlIikKICAgIG1vZGVsLnRyYWluKCkKICAgIG9wdGltaXplci56'
        'ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgIG91dHB1dCA9IGNhc3QoT2x5bXB1c1N5c3Rl'
        'bU91dHB1dCwgbW9kZWwoKipkaWN0KGlucHV0cykpKQogICAgcmVxdWlyZWRfdGFyZ2V0cyA9IHsK'
        'ICAgICAgICAiZGVjaXNpb25fdGFyZ2V0cyIsCiAgICAgICAgInRyYW5zaXRpb25fdGFyZ2V0cyIs'
        'CiAgICAgICAgIm5leHRfc3RhdGVfdGFyZ2V0cyIsCiAgICB9CiAgICBvcHRpb25hbF90YXJnZXRz'
        'ID0geyJpbnRyaW5zaWNfcmlza190YXJnZXRzIn0KICAgIG1pc3NpbmdfdGFyZ2V0cyA9IHNvcnRl'
        'ZChyZXF1aXJlZF90YXJnZXRzIC0gc2V0KHRhcmdldHMpKQogICAgdW5leHBlY3RlZF90YXJnZXRz'
        'ID0gc29ydGVkKHNldCh0YXJnZXRzKSAtIHJlcXVpcmVkX3RhcmdldHMgLSBvcHRpb25hbF90YXJn'
        'ZXRzKQogICAgaWYgbWlzc2luZ190YXJnZXRzIG9yIHVuZXhwZWN0ZWRfdGFyZ2V0czoKICAgICAg'
        'ICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAidHJhaW5pbmcgdGFyZ2V0cyBkbyBub3Qg'
        'bWF0Y2ggdGhlIE9seW1wdXMgbG9zcyBjb250cmFjdDsgIgogICAgICAgICAgICBmIm1pc3Npbmc9'
        'e21pc3NpbmdfdGFyZ2V0c30sIHVuZXhwZWN0ZWQ9e3VuZXhwZWN0ZWRfdGFyZ2V0c30iCiAgICAg'
        'ICAgKQogICAgdGVtcG9yYWxfbWFzayA9IGlucHV0cy5nZXQoInRlbXBvcmFsX21hc2siKQogICAg'
        'aWYgdGVtcG9yYWxfbWFzayBpcyBub3QgTm9uZSBhbmQgbm90IGlzaW5zdGFuY2UodGVtcG9yYWxf'
        'bWFzaywgVGVuc29yKToKICAgICAgICByYWlzZSBUeXBlRXJyb3IoInRlbXBvcmFsIG1hc2sgbXVz'
        'dCBiZSBhIHRlbnNvciIpCiAgICB3ZWlnaHRzID0gKGxvc3NfY29uZmlnIG9yIE9seW1wdXNTeXN0'
        'ZW1Mb3NzQ29uZmlnKCkpLm1vZGVsX2R1bXAoKQogICAgbG9zcyA9IG9seW1wdXNfc3lzdGVtX2xv'
        'c3MoCiAgICAgICAgb3V0cHV0LAogICAgICAgIGRlY2lzaW9uX3RhcmdldHM9dGFyZ2V0c1siZGVj'
        'aXNpb25fdGFyZ2V0cyJdLAogICAgICAgIHRyYW5zaXRpb25fdGFyZ2V0cz10YXJnZXRzWyJ0cmFu'
        'c2l0aW9uX3RhcmdldHMiXSwKICAgICAgICBuZXh0X3N0YXRlX3RhcmdldHM9dGFyZ2V0c1sibmV4'
        'dF9zdGF0ZV90YXJnZXRzIl0sCiAgICAgICAgaW50cmluc2ljX3Jpc2tfdGFyZ2V0cz10YXJnZXRz'
        'LmdldCgiaW50cmluc2ljX3Jpc2tfdGFyZ2V0cyIpLAogICAgICAgIHRlbXBvcmFsX21hc2s9dGVt'
        'cG9yYWxfbWFzaywKICAgICAgICAqKndlaWdodHMsCiAgICApCiAgICBpZiBub3QgYm9vbCh0b3Jj'
        'aC5pc2Zpbml0ZShsb3NzLnRvdGFsKSk6CiAgICAgICAgcmFpc2UgRmxvYXRpbmdQb2ludEVycm9y'
        'KCJPbHltcHVzIHRyYWluaW5nIGxvc3MgaXMgbm9uLWZpbml0ZSIpCiAgICBsb3NzLnRvdGFsLmJh'
        'Y2t3YXJkKCkgICMgdHlwZTogaWdub3JlW25vLXVudHlwZWQtY2FsbF0KICAgIGdyYWRpZW50ID0g'
        'bm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgZ3JhZGllbnRfY2xp'
        'cF9ub3JtKQogICAgaWYgbm90IGJvb2wodG9yY2guaXNmaW5pdGUoZ3JhZGllbnQpKToKICAgICAg'
        'ICBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgcmFpc2UgRmxv'
        'YXRpbmdQb2ludEVycm9yKCJPbHltcHVzIGdyYWRpZW50IG5vcm0gaXMgbm9uLWZpbml0ZSIpCiAg'
        'ICBvcHRpbWl6ZXIuc3RlcCgpCiAgICByZXR1cm4gT2x5bXB1c1RyYWluU3RlcFJlc3VsdCgKICAg'
        'ICAgICBvdXRwdXQ9b3V0cHV0LAogICAgICAgIGxvc3M9bG9zcywKICAgICAgICBncmFkaWVudF9u'
        'b3JtPWZsb2F0KGdyYWRpZW50LmRldGFjaCgpLml0ZW0oKSksCiAgICApCgoKZGVmIHNhdmVfb2x5'
        'bXB1c19jaGVja3BvaW50KAogICAgbW9kZWw6IE9seW1wdXNSZWZlcmVuY2VTeXN0ZW0sCiAgICBw'
        'YXRoOiBQYXRoLAogICAgKiwKICAgIG1ldGFkYXRhOiBNYXBwaW5nW3N0ciwgc3RyXSB8IE5vbmUg'
        'PSBOb25lLAopIC0+IE9seW1wdXNDaGVja3BvaW50UmVjb3JkOgogICAgIiIiV3JpdGUgYSBuZXcg'
        'aW1tdXRhYmxlIGNoZWNrcG9pbnQgYW5kIHJldHVybiBpdHMgY29udGVudCBpZGVudGl0eS4iIiIK'
        'CiAgICBpZiBwYXRoLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcigicmVm'
        'dXNpbmcgdG8gb3ZlcndyaXRlIGFuIGV4aXN0aW5nIE9seW1wdXMgY2hlY2twb2ludCIpCiAgICBw'
        'YXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwYXlsb2Fk'
        'ID0gewogICAgICAgICJzY2hlbWFfdmVyc2lvbiI6IDEsCiAgICAgICAgIm1vZGVsX2NvbmZpZyI6'
        'IG1vZGVsLmNvbmZpZy5tb2RlbF9kdW1wKG1vZGU9Impzb24iKSwKICAgICAgICAibW9kZWxfc3Rh'
        'dGUiOiBtb2RlbC5zdGF0ZV9kaWN0KCksCiAgICAgICAgIm1ldGFkYXRhIjogZGljdChtZXRhZGF0'
        'YSBvciB7fSksCiAgICB9CiAgICB3aXRoIE5hbWVkVGVtcG9yYXJ5RmlsZShkaXI9cGF0aC5wYXJl'
        'bnQsIHByZWZpeD1mIi57cGF0aC5uYW1lfS4iLCBkZWxldGU9RmFsc2UpIGFzIGhhbmRsZToKICAg'
        'ICAgICB0ZW1wb3JhcnkgPSBQYXRoKGhhbmRsZS5uYW1lKQogICAgdHJ5OgogICAgICAgIHRvcmNo'
        'LnNhdmUocGF5bG9hZCwgdGVtcG9yYXJ5KQogICAgICAgIHdpdGggdGVtcG9yYXJ5Lm9wZW4oInJi'
        'IikgYXMgaGFuZGxlOgogICAgICAgICAgICBvcy5mc3luYyhoYW5kbGUuZmlsZW5vKCkpCiAgICAg'
        'ICAgIyBIYXJkLWxpbmsgcHVibGljYXRpb24gaXMgYXRvbWljIGFuZCByZWZ1c2VzIGFuIGV4aXN0'
        'aW5nIGRlc3RpbmF0aW9uLgogICAgICAgICMgQW4gZXhpc3RzKCkvcmVwbGFjZSgpIHBhaXIgY2Fu'
        'IG92ZXJ3cml0ZSBhbm90aGVyIHdyaXRlcidzIGNoZWNrcG9pbnQuCiAgICAgICAgdHJ5OgogICAg'
        'ICAgICAgICBvcy5saW5rKHRlbXBvcmFyeSwgcGF0aCkKICAgICAgICBleGNlcHQgRmlsZUV4aXN0'
        'c0Vycm9yIGFzIGVycm9yOgogICAgICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoInJlZnVz'
        'aW5nIHRvIG92ZXJ3cml0ZSBhbiBleGlzdGluZyBPbHltcHVzIGNoZWNrcG9pbnQiKSBmcm9tIGVy'
        'cm9yCiAgICBmaW5hbGx5OgogICAgICAgIHRlbXBvcmFyeS51bmxpbmsobWlzc2luZ19vaz1UcnVl'
        'KQogICAgY29udGVudCA9IHBhdGgucmVhZF9ieXRlcygpCiAgICByZXR1cm4gT2x5bXB1c0NoZWNr'
        'cG9pbnRSZWNvcmQoCiAgICAgICAgcGF0aD1zdHIocGF0aC5yZXNvbHZlKCkpLAogICAgICAgIHNo'
        'YTI1Nj1fc2hhMjU2KGNvbnRlbnQpLAogICAgICAgIHBhcmFtZXRlcnM9c3VtKHBhcmFtZXRlci5u'
        'dW1lbCgpIGZvciBwYXJhbWV0ZXIgaW4gbW9kZWwucGFyYW1ldGVycygpKSwKICAgICAgICBzeXN0'
        'ZW1fY29uZmlnPW1vZGVsLmNvbmZpZywKICAgICAgICBtZXRhZGF0YT1kaWN0KG1ldGFkYXRhIG9y'
        'IHt9KSwKICAgICkKCgpkZWYgbG9hZF9vbHltcHVzX2NoZWNrcG9pbnQoCiAgICBwYXRoOiBQYXRo'
        'LAogICAgKiwKICAgIGV4cGVjdGVkX3NoYTI1Njogc3RyLAopIC0+IHR1cGxlW09seW1wdXNSZWZl'
        'cmVuY2VTeXN0ZW0sIE9seW1wdXNDaGVja3BvaW50UmVjb3JkXToKICAgICIiIkxvYWQgYSBkaWdl'
        'c3QtcGlubmVkIGNoZWNrcG9pbnQgd2l0aG91dCBhbGxvd2luZyBhcmJpdHJhcnkgcGlja2xlIG9i'
        'amVjdHMuIiIiCgogICAgaWYgcGF0aC5pc19zeW1saW5rKCkgb3Igbm90IHBhdGguaXNfZmlsZSgp'
        'OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgbXVzdCBiZSBhIHJlZ3VsYXIg'
        'bm9uLXN5bWxpbmsgZmlsZSIpCiAgICBjb250ZW50ID0gcGF0aC5yZWFkX2J5dGVzKCkKICAgIG9i'
        'c2VydmVkID0gX3NoYTI1Nihjb250ZW50KQogICAgaWYgb2JzZXJ2ZWQgIT0gZXhwZWN0ZWRfc2hh'
        'MjU2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgZGlnZXN0IGRvZXMgbm90'
        'IG1hdGNoIHRoZSBleHBlY3RlZCBTSEEtMjU2IikKICAgIHBheWxvYWQ6IEFueSA9IHRvcmNoLmxv'
        'YWQoaW8uQnl0ZXNJTyhjb250ZW50KSwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9'
        'VHJ1ZSkKICAgIGlmIG5vdCBpc2luc3RhbmNlKHBheWxvYWQsIGRpY3QpIG9yIHBheWxvYWQuZ2V0'
        'KCJzY2hlbWFfdmVyc2lvbiIpICE9IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2tw'
        'b2ludCBoYXMgYW4gdW5zdXBwb3J0ZWQgc2NoZW1hIikKICAgIHJhd19jb25maWcgPSBwYXlsb2Fk'
        'LmdldCgibW9kZWxfY29uZmlnIikKICAgIHJhd19zdGF0ZSA9IHBheWxvYWQuZ2V0KCJtb2RlbF9z'
        'dGF0ZSIpCiAgICByYXdfbWV0YWRhdGEgPSBwYXlsb2FkLmdldCgibWV0YWRhdGEiLCB7fSkKICAg'
        'IGlmIG5vdCBpc2luc3RhbmNlKHJhd19jb25maWcsIGRpY3QpIG9yIG5vdCBpc2luc3RhbmNlKHJh'
        'd19zdGF0ZSwgZGljdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBpcyBt'
        'aXNzaW5nIGl0cyBtb2RlbCBjb25maWd1cmF0aW9uIG9yIHN0YXRlIikKICAgIGlmIG5vdCBpc2lu'
        'c3RhbmNlKHJhd19tZXRhZGF0YSwgZGljdCkgb3Igbm90IGFsbCgKICAgICAgICBpc2luc3RhbmNl'
        'KGtleSwgc3RyKSBhbmQgaXNpbnN0YW5jZSh2YWx1ZSwgc3RyKSBmb3Iga2V5LCB2YWx1ZSBpbiBy'
        'YXdfbWV0YWRhdGEuaXRlbXMoKQogICAgKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVj'
        'a3BvaW50IG1ldGFkYXRhIG11c3QgbWFwIHN0cmluZ3MgdG8gc3RyaW5ncyIpCiAgICBtb2RlbF9j'
        'b25maWcgPSBPbHltcHVzU3lzdGVtQ29uZmlnLm1vZGVsX3ZhbGlkYXRlKHJhd19jb25maWcpCiAg'
        'ICBtb2RlbCA9IGJ1aWxkX29seW1wdXNfc3lzdGVtKG1vZGVsX2NvbmZpZykKICAgIG1vZGVsLmxv'
        'YWRfc3RhdGVfZGljdChjYXN0KGRpY3Rbc3RyLCBUZW5zb3JdLCByYXdfc3RhdGUpLCBzdHJpY3Q9'
        'VHJ1ZSkKICAgIG1vZGVsLmV2YWwoKQogICAgcmVjb3JkID0gT2x5bXB1c0NoZWNrcG9pbnRSZWNv'
        'cmQoCiAgICAgICAgcGF0aD1zdHIocGF0aC5yZXNvbHZlKCkpLAogICAgICAgIHNoYTI1Nj1vYnNl'
        'cnZlZCwKICAgICAgICBwYXJhbWV0ZXJzPXN1bShwYXJhbWV0ZXIubnVtZWwoKSBmb3IgcGFyYW1l'
        'dGVyIGluIG1vZGVsLnBhcmFtZXRlcnMoKSksCiAgICAgICAgc3lzdGVtX2NvbmZpZz1tb2RlbF9j'
        'b25maWcsCiAgICAgICAgbWV0YWRhdGE9Y2FzdChkaWN0W3N0ciwgc3RyXSwgcmF3X21ldGFkYXRh'
        'KSwKICAgICkKICAgIHJldHVybiBtb2RlbCwgcmVjb3JkCg=='
    ),
    'olympus/models/seven_model_contract.py': (
        '...'
    ),
    'olympus/models/spatiotemporal_lazy.py': (
        '...'
        'c3RvcmFnZXMudmFsdWVzKCkpCg=='
    ),
    'olympus/models/specifications.py': (
        'IiIiQ2Fub25pY2FsLCBldmlkZW5jZS1ib3VuZCBzcGVjaWZpY2F0aW9ucyBmb3IgdGhlIEJVMUxE'
        'IG1vZGVsIHByb2dyYW1tZS4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMK'
        'CmZyb20gdHlwaW5nIGltcG9ydCBMaXRlcmFsCgpmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZCwg'
        'bW9kZWxfdmFsaWRhdG9yCgpmcm9tIG9seW1wdXMuY29yZS5zY2hlbWFzIGltcG9ydCBTdHJpY3RN'
        'b2RlbAoKTW9kZWxGYW1pbHlJZCA9IExpdGVyYWxbImhlcm1lcyIsICJwcm9tZXRoZXVzIiwgInBl'
        'cnNldXMiLCAiYXRsYXMiLCAia3Jvbm9zIiwgImFpb24iXQoKCmNsYXNzIE1vZGVsVGFyZ2V0U3Bl'
        'Y2lmaWNhdGlvbihTdHJpY3RNb2RlbCk6CiAgICAiIiJBbiBpbnRlbmRlZCBtb2RlbCB0YXJnZXQs'
        'IG5ldmVyIGV2aWRlbmNlIHRoYXQgdGhlIHRhcmdldCBhbHJlYWR5IGV4aXN0cy4iIiIKCiAgICBm'
        'YW1pbHlfaWQ6IE1vZGVsRmFtaWx5SWQKICAgIGRpc3BsYXlfbmFtZTogc3RyCiAgICB0YXJnZXRf'
        'cGFyYW1ldGVyczogaW50IHwgTm9uZSA9IEZpZWxkKGRlZmF1bHQ9Tm9uZSwgZ3Q9MCkKICAgIHRh'
        'cmdldF9wYXJhbWV0ZXJzX21pbjogaW50IHwgTm9uZSA9IEZpZWxkKGRlZmF1bHQ9Tm9uZSwgZ3Q9'
        'MCkKICAgIHRhcmdldF9wYXJhbWV0ZXJzX21heDogaW50IHwgTm9uZSA9IEZpZWxkKGRlZmF1bHQ9'
        'Tm9uZSwgZ3Q9MCkKICAgIGludGVuZGVkX3JvbGU6IHN0cgogICAgY2FwYWJpbGl0eV9nYXRlczog'
        'bGlzdFtzdHJdID0gRmllbGQobWluX2xlbmd0aD0xKQogICAgc3RhdHVzOiBMaXRlcmFsWyJUQVJH'
        'RVRfU1BFQ0lGSUNBVElPTiJdID0gIlRBUkdFVF9TUEVDSUZJQ0FUSU9OIgogICAgY2FwYWJpbGl0'
        'eV9jbGFpbWVkOiBMaXRlcmFsW0ZhbHNlXSA9IEZhbHNlCgogICAgQG1vZGVsX3ZhbGlkYXRvciht'
        'b2RlPSJhZnRlciIpCiAgICBkZWYgdmFsaWRfdGFyZ2V0X3JhbmdlKHNlbGYpIC0+IE1vZGVsVGFy'
        'Z2V0U3BlY2lmaWNhdGlvbjoKICAgICAgICBsb3csIGhpZ2ggPSBzZWxmLnRhcmdldF9wYXJhbWV0'
        'ZXJzX21pbiwgc2VsZi50YXJnZXRfcGFyYW1ldGVyc19tYXgKICAgICAgICBpZiAobG93IGlzIE5v'
        'bmUpICE9IChoaWdoIGlzIE5vbmUpIG9yIChsb3cgaXMgbm90IE5vbmUgYW5kIGhpZ2ggaXMgbm90'
        'IE5vbmUgYW5kIGxvdyA+IGhpZ2gpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0YXJn'
        'ZXQgcmFuZ2VzIHJlcXVpcmUgb3JkZXJlZCBtaW5pbXVtIGFuZCBtYXhpbXVtIikKICAgICAgICBp'
        'ZiBsb3cgaXMgbm90IE5vbmUgYW5kIGhpZ2ggaXMgbm90IE5vbmUgYW5kIHNlbGYudGFyZ2V0X3Bh'
        'cmFtZXRlcnMgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGlmIG5vdCBsb3cgPD0gc2VsZi50YXJn'
        'ZXRfcGFyYW1ldGVycyA8PSBoaWdoOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi'
        'cG9pbnQgdGFyZ2V0IG11c3QgbGllIGluc2lkZSB0aGUgdGFyZ2V0IHJhbmdlIikKICAgICAgICBy'
        'ZXR1cm4gc2VsZgoKCkJVMUxEX01PREVMX1NQRUNTOiB0dXBsZVtNb2RlbFRhcmdldFNwZWNpZmlj'
        'YXRpb24sIC4uLl0gPSAoCiAgICBNb2RlbFRhcmdldFNwZWNpZmljYXRpb24oCiAgICAgICAgZmFt'
        'aWx5X2lkPSJoZXJtZXMiLAogICAgICAgIGRpc3BsYXlfbmFtZT0iSGVybWVzIiwKICAgICAgICB0'
        'YXJnZXRfcGFyYW1ldGVycz0xMl8wMDBfMDAwXzAwMCwKICAgICAgICBpbnRlbmRlZF9yb2xlPSJQ'
        'cml2YXRlLCBsb2NhbC1maXJzdCBhc3Npc3RhbnQgd2l0aCBncm91bmRlZCB0b29sIHVzZSBhbmQg'
        'cmVjb3ZlcnkuIiwKICAgICAgICBjYXBhYmlsaXR5X2dhdGVzPVsKICAgICAgICAgICAgImhlbGQt'
        'b3V0IGV4YWN0IHRhc2sgY29tcGxldGlvbiIsCiAgICAgICAgICAgICJ0eXBlZCB0b29sIGFyZ3Vt'
        'ZW50cyBhbmQgZGVuaWVkLWNhbGwgaGFuZGxpbmciLAogICAgICAgICAgICAiZ3JvdW5kZWQgY2l0'
        'YXRpb24gYW5kIGNhbGlicmF0ZWQgYWJzdGVudGlvbiIsCiAgICAgICAgICAgICJib3VuZGVkIGxv'
        'Y2FsIG1lbW9yeSwgbGF0ZW5jeSwgYW5kIHByaXZhY3kiLAogICAgICAgIF0sCiAgICApLAogICAg'
        'TW9kZWxUYXJnZXRTcGVjaWZpY2F0aW9uKAogICAgICAgIGZhbWlseV9pZD0icHJvbWV0aGV1cyIs'
        'CiAgICAgICAgZGlzcGxheV9uYW1lPSJQcm9tZXRoZXVzIiwKICAgICAgICB0YXJnZXRfcGFyYW1l'
        'dGVyc19taW49MTJfMDAwXzAwMF8wMDAsCiAgICAgICAgdGFyZ2V0X3BhcmFtZXRlcnNfbWF4PTIw'
        'XzAwMF8wMDBfMDAwLAogICAgICAgIGludGVuZGVkX3JvbGU9KAogICAgICAgICAgICAiRG9tYWlu'
        'IHNwZWNpYWxpc3QgYWRhcHRlZCBmb3IgcmVzZWFyY2gsIHNvZnR3YXJlIGVuZ2luZWVyaW5nLCBh'
        'bmQgZGVzaWduIHdpdGhvdXQgIgogICAgICAgICAgICAiZXJhc2luZyBnZW5lcmFsIHNhZmV0eSBv'
        'ciBjYWxpYnJhdGlvbi4iCiAgICAgICAgKSwKICAgICAgICBjYXBhYmlsaXR5X2dhdGVzPVsKICAg'
        'ICAgICAgICAgImRvbWFpbi1kaXNqb2ludCBzcGVjaWFsaXN0IGV2YWx1YXRpb25zIiwKICAgICAg'
        'ICAgICAgImJhc2UtdmVyc3VzLWFkYXB0ZXIgcmVncmVzc2lvbiBjb250cm9scyIsCiAgICAgICAg'
        'ICAgICJzb3VyY2UtZ3JvdW5kZWQgcmVzZWFyY2ggYW5kIGV4ZWN1dGFibGUgdmVyaWZpY2F0aW9u'
        'IiwKICAgICAgICAgICAgImFkYXB0ZXIgaXNvbGF0aW9uLCByb2xsYmFjaywgYW5kIHByb3ZlbmFu'
        'Y2UiLAogICAgICAgIF0sCiAgICApLAogICAgTW9kZWxUYXJnZXRTcGVjaWZpY2F0aW9uKAogICAg'
        'ICAgIGZhbWlseV9pZD0icGVyc2V1cyIsCiAgICAgICAgZGlzcGxheV9uYW1lPSJQZXJzZXVzIiwK'
        'ICAgICAgICB0YXJnZXRfcGFyYW1ldGVycz0xXzAyNF8wMDBfMDAwXzAwMCwKICAgICAgICBpbnRl'
        'bmRlZF9yb2xlPSgKICAgICAgICAgICAgIlNwYXJzZSBnZW5lcmFsaXN0IGRlc2lnbmVkIGZvciBn'
        'bG9iYWxseSBhY2Nlc3NpYmxlIGluZmVyZW5jZSB3aXRoIHJlbGlhYmxlIHJvdXRpbmcuIgogICAg'
        'ICAgICksCiAgICAgICAgY2FwYWJpbGl0eV9nYXRlcz1bCiAgICAgICAgICAgICJhY3RpdmUtcGFy'
        'YW1ldGVyIGFuZCBtZW1vcnkgYnVkZ2V0IGNvbXBsaWFuY2UiLAogICAgICAgICAgICAibXVsdGls'
        'aW5ndWFsIGFuZCBsb3ctcmVzb3VyY2UgZXZhbHVhdGlvbnMiLAogICAgICAgICAgICAidG9vbC11'
        'c2UgcmVsaWFiaWxpdHkgYW5kIHNpZGUtZWZmZWN0IGNvbnRhaW5tZW50IiwKICAgICAgICAgICAg'
        'InF1YWxpdHkgcGFyaXR5IGFnYWluc3QgZGVuc2UgYW5kIHNtYWxsZXIgc3BhcnNlIGJhc2VsaW5l'
        'cyIsCiAgICAgICAgXSwKICAgICksCiAgICBNb2RlbFRhcmdldFNwZWNpZmljYXRpb24oCiAgICAg'
        'ICAgZmFtaWx5X2lkPSJhdGxhcyIsCiAgICAgICAgZGlzcGxheV9uYW1lPSJBdGxhcyIsCiAgICAg'
        'ICAgdGFyZ2V0X3BhcmFtZXRlcnM9Tm9uZSwKICAgICAgICBpbnRlbmRlZF9yb2xlPSgKICAgICAg'
        'ICAgICAgIkRlZXAgbXVsdGltb2RhbCBpbnRlbGxpZ2VuY2Ugd2l0aCBsb25nLWNvbnRleHQgZ3Jv'
        'dW5kaW5nIGFuZCBleHBsaWNpdCB1bmNlcnRhaW50eS4iCiAgICAgICAgKSwKICAgICAgICBjYXBh'
        'YmlsaXR5X2dhdGVzPVsKICAgICAgICAgICAgImxvbmctY29udGV4dCByZXRyaWV2YWwgYW5kIGNp'
        'dGF0aW9uIGNvcnJlY3RuZXNzIiwKICAgICAgICAgICAgInNwYXRpb3RlbXBvcmFsIHdvcmxkLXN0'
        'YXRlIGNvbnNpc3RlbmN5IiwKICAgICAgICAgICAgIm5vdmVsIGhlbGQtb3V0IHJlYXNvbmluZyB3'
        'aXRoIHN0cm9uZyBleHRlcm5hbCBiYXNlbGluZXMiLAogICAgICAgICAgICAiYmVoYXZpb3JhbCBj'
        'YXBhYmlsaXR5IHJlcG9ydGluZyB3aXRob3V0IHNlbnRpZW5jZSBjbGFpbXMiLAogICAgICAgIF0s'
        'CiAgICApLAogICAgTW9kZWxUYXJnZXRTcGVjaWZpY2F0aW9uKAogICAgICAgIGZhbWlseV9pZD0i'
        'a3Jvbm9zIiwKICAgICAgICBkaXNwbGF5X25hbWU9Iktyb25vcyIsCiAgICAgICAgdGFyZ2V0X3Bh'
        'cmFtZXRlcnM9OF8wMDBfMDAwXzAwMF8wMDAsCiAgICAgICAgaW50ZW5kZWRfcm9sZT0oCiAgICAg'
        'ICAgICAgICJMb25nLWhvcml6b24gZ2VuZXJhbCBpbnRlbGxpZ2VuY2UgcmVzZWFyY2ggc3lzdGVt'
        'IHdpdGggcGxhbm5pbmcsIGFkYXB0YXRpb24sIGFuZCAiCiAgICAgICAgICAgICJmYWlsLWNsb3Nl'
        'ZCBjb250cm9sLiIKICAgICAgICApLAogICAgICAgIGNhcGFiaWxpdHlfZ2F0ZXM9WwogICAgICAg'
        'ICAgICAibG9uZy1ob3Jpem9uIHBsYW4gZXhlY3V0aW9uIGFuZCByZWNvdmVyeSIsCiAgICAgICAg'
        'ICAgICJjb250aW51YWwtbGVhcm5pbmcgcmV0ZW50aW9uIGFuZCByb2xsYmFjayIsCiAgICAgICAg'
        'ICAgICJpbmRlcGVuZGVudCBzYWZldHkgYW5kIGF1dG9ub215IGV2YWx1YXRpb24iLAogICAgICAg'
        'ICAgICAiZGlzdHJpYnV0ZWQtdHJhaW5pbmcsIHNlcnZpbmcsIGFuZCBjb3N0IGV2aWRlbmNlIiwK'
        'ICAgICAgICBdLAogICAgKSwKICAgIE1vZGVsVGFyZ2V0U3BlY2lmaWNhdGlvbigKICAgICAgICBm'
        'YW1pbHlfaWQ9ImFpb24iLAogICAgICAgIGRpc3BsYXlfbmFtZT0iQWlvbiIsCiAgICAgICAgdGFy'
        'Z2V0X3BhcmFtZXRlcnM9Tm9uZSwKICAgICAgICBpbnRlbmRlZF9yb2xlPSgKICAgICAgICAgICAg'
        'IkdvdmVybmVkIHJlc2VhcmNoLXByb3RvY29sIG9yY2hlc3RyYXRpb24gd2l0aCBkZXRlcm1pbmlz'
        'dGljIGF1dGhvcml0eSBib3VuZGFyaWVzICIKICAgICAgICAgICAgImFuZCBhbiBvcHRpb25hbCBs'
        'ZWFybmVkIHRyYW5zaXRpb24gcm91dGVyLiIKICAgICAgICApLAogICAgICAgIGNhcGFiaWxpdHlf'
        'Z2F0ZXM9WwogICAgICAgICAgICAiZnJvemVuLXByb3RvY29sIGFuZCBldmlkZW5jZS1saW5lYWdl'
        'IGludGVncml0eSIsCiAgICAgICAgICAgICJodW1hbiBhdXRob3JpdHkgZm9yIGV4ZWN1dGlvbiBh'
        'bmQgcHJvbW90aW9uIHRyYW5zaXRpb25zIiwKICAgICAgICAgICAgImJ1ZGdldCwga2lsbC1ydWxl'
        'LCBhbmQgaWxsZWdhbC10cmFuc2l0aW9uIGVuZm9yY2VtZW50IiwKICAgICAgICAgICAgImFkdmFu'
        'dGFnZSBvdmVyIHRoZSBkZXRlcm1pbmlzdGljIGNvbnRyb2xsZXIgd2l0aG91dCBmYWxzZSBwcm9t'
        'b3Rpb24iLAogICAgICAgIF0sCiAgICApLAopCgoKZGVmIG1vZGVsX3NwZWMoZmFtaWx5X2lkOiBN'
        'b2RlbEZhbWlseUlkKSAtPiBNb2RlbFRhcmdldFNwZWNpZmljYXRpb246CiAgICAiIiJSZXR1cm4g'
        'b25lIGltbXV0YWJsZSB0YXJnZXQgc3BlY2lmaWNhdGlvbiBieSBmYW1pbHkgaWRlbnRpZmllci4i'
        'IiIKCiAgICByZXR1cm4gbmV4dChzcGVjIGZvciBzcGVjIGluIEJVMUxEX01PREVMX1NQRUNTIGlm'
        'IHNwZWMuZmFtaWx5X2lkID09IGZhbWlseV9pZCkK'
    ),
    'olympus/research_state.py': (
        '...'
        'KQo='
    ),
    'olympus/sdk.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgppbXBvcnQgaHR0cHgK'
        'CgpAZGF0YWNsYXNzKHNsb3RzPVRydWUpCmNsYXNzIE9seW1wdXNTREs6CiAgICBiYXNlX3VybDog'
        'c3RyCiAgICB0aW1lb3V0X3NlY29uZHM6IGZsb2F0ID0gMTAuMAogICAgdHJhbnNwb3J0OiBodHRw'
        'eC5CYXNlVHJhbnNwb3J0IHwgTm9uZSA9IGZpZWxkKGRlZmF1bHQ9Tm9uZSwgcmVwcj1GYWxzZSkK'
        'CiAgICBkZWYgX19wb3N0X2luaXRfXyhzZWxmKSAtPiBOb25lOgogICAgICAgIHNlbGYuYmFzZV91'
        'cmwgPSBzZWxmLmJhc2VfdXJsLnJzdHJpcCgiLyIpCiAgICAgICAgaWYgbm90IHNlbGYuYmFzZV91'
        'cmwuc3RhcnRzd2l0aCgoImh0dHA6Ly8iLCAiaHR0cHM6Ly8iKSk6CiAgICAgICAgICAgIHJhaXNl'
        'IFZhbHVlRXJyb3IoImJhc2VfdXJsIG11c3QgYmUgYW4gSFRUUCBvciBIVFRQUyBVUkwiKQogICAg'
        'ICAgIGlmIHNlbGYudGltZW91dF9zZWNvbmRzIDw9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVl'
        'RXJyb3IoInRpbWVvdXRfc2Vjb25kcyBtdXN0IGJlIHBvc2l0aXZlIikKCiAgICBkZWYgX3JlcXVl'
        'c3Qoc2VsZiwgbWV0aG9kOiBzdHIsIHBhdGg6IHN0ciwgKiprd2FyZ3M6IEFueSkgLT4gZGljdFtz'
        'dHIsIEFueV06CiAgICAgICAgd2l0aCBodHRweC5DbGllbnQoCiAgICAgICAgICAgIGJhc2VfdXJs'
        'PXNlbGYuYmFzZV91cmwsCiAgICAgICAgICAgIHRpbWVvdXQ9c2VsZi50aW1lb3V0X3NlY29uZHMs'
        'CiAgICAgICAgICAgIHRyYW5zcG9ydD1zZWxmLnRyYW5zcG9ydCwKICAgICAgICApIGFzIGNsaWVu'
        'dDoKICAgICAgICAgICAgcmVzcG9uc2UgPSBjbGllbnQucmVxdWVzdChtZXRob2QsIHBhdGgsICoq'
        'a3dhcmdzKQogICAgICAgICAgICByZXNwb25zZS5yYWlzZV9mb3Jfc3RhdHVzKCkKICAgICAgICAg'
        'ICAgcGF5bG9hZCA9IHJlc3BvbnNlLmpzb24oKQogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHBh'
        'eWxvYWQsIGRpY3QpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPbHltcHVzIEFQSSBy'
        'ZXR1cm5lZCBhIG5vbi1vYmplY3QgSlNPTiByZXNwb25zZSIpCiAgICAgICAgcmV0dXJuIHBheWxv'
        'YWQKCiAgICBkZWYgaGVhbHRoKHNlbGYpIC0+IGRpY3Rbc3RyLCBzdHJdOgogICAgICAgIHBheWxv'
        'YWQgPSBzZWxmLl9yZXF1ZXN0KCJHRVQiLCAiL2hlYWx0aCIpCiAgICAgICAgc3RyaW5nX2ZpZWxk'
        'cyA9IGFsbCgKICAgICAgICAgICAgaXNpbnN0YW5jZShrZXksIHN0cikgYW5kIGlzaW5zdGFuY2Uo'
        'dmFsdWUsIHN0cikgZm9yIGtleSwgdmFsdWUgaW4gcGF5bG9hZC5pdGVtcygpCiAgICAgICAgKQog'
        'ICAgICAgIGlmIG5vdCBzdHJpbmdfZmllbGRzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9y'
        'KCJPbHltcHVzIGhlYWx0aCByZXNwb25zZSBtdXN0IGNvbnRhaW4gb25seSBzdHJpbmcgZmllbGRz'
        'IikKICAgICAgICByZXR1cm4gcGF5bG9hZAoKICAgIGRlZiBjb21waWxlX2JlaGF2aW9yKHNlbGYs'
        'IGJlaGF2aW9yOiBzdHIpIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgICAgIHJldHVybiBzZWxm'
        'Ll9yZXF1ZXN0KAogICAgICAgICAgICAiUE9TVCIsCiAgICAgICAgICAgICIvZm9yZ2UvY29tcGls'
        'ZSIsCiAgICAgICAgICAgIGpzb249eyJiZWhhdmlvciI6IGJlaGF2aW9yfSwKICAgICAgICApCgog'
        'ICAgZGVmIGZvdW5kcnlfc3RhdHVzKHNlbGYpIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgICAg'
        'IHJldHVybiBzZWxmLl9yZXF1ZXN0KCJHRVQiLCAiL2ZvdW5kcnkvc3RhdHVzIikKCiAgICBkZWYg'
        'bGlzdF9tb2RlbHMoc2VsZikgLT4gbGlzdFtkaWN0W3N0ciwgb2JqZWN0XV06CiAgICAgICAgcGF5'
        'bG9hZCA9IHNlbGYuX3JlcXVlc3QoIkdFVCIsICIvdjEvbW9kZWxzIikKICAgICAgICBtb2RlbHMg'
        'PSBwYXlsb2FkLmdldCgiZGF0YSIpCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UobW9kZWxzLCBs'
        'aXN0KSBvciBhbnkobm90IGlzaW5zdGFuY2UobW9kZWwsIGRpY3QpIGZvciBtb2RlbCBpbiBtb2Rl'
        'bHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPbHltcHVzIG1vZGVsIHJlZ2lzdHJ5'
        'IHJlc3BvbnNlIGlzIG1hbGZvcm1lZCIpCiAgICAgICAgcmV0dXJuIG1vZGVscwoKICAgIGRlZiBn'
        'ZW5lcmF0ZSgKICAgICAgICBzZWxmLAogICAgICAgICosCiAgICAgICAgbW9kZWw6IHN0ciwKICAg'
        'ICAgICBwcm9tcHQ6IHN0ciwKICAgICAgICBtYXhfdG9rZW5zOiBpbnQgPSAxNjAsCiAgICAgICAg'
        'dGVtcGVyYXR1cmU6IGZsb2F0ID0gMC43LAogICAgICAgIHNlZWQ6IGludCB8IE5vbmUgPSBOb25l'
        'LAogICAgKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgICAgICBpZiBub3QgbW9kZWwuc3RyaXAo'
        'KToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigibW9kZWwgaXMgcmVxdWlyZWQiKQogICAg'
        'ICAgIGlmIG5vdCBwcm9tcHQuc3RyaXAoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi'
        'cHJvbXB0IGlzIHJlcXVpcmVkIikKICAgICAgICByZXR1cm4gc2VsZi5fcmVxdWVzdCgKICAgICAg'
        'ICAgICAgIlBPU1QiLAogICAgICAgICAgICAiL3YxL2NoYXQvY29tcGxldGlvbnMiLAogICAgICAg'
        'ICAgICBqc29uPXsKICAgICAgICAgICAgICAgICJtb2RlbCI6IG1vZGVsLAogICAgICAgICAgICAg'
        'ICAgIm1lc3NhZ2VzIjogW3sicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBwcm9tcHR9XSwKICAg'
        'ICAgICAgICAgICAgICJtYXhfdG9rZW5zIjogbWF4X3Rva2VucywKICAgICAgICAgICAgICAgICJ0'
        'ZW1wZXJhdHVyZSI6IHRlbXBlcmF0dXJlLAogICAgICAgICAgICAgICAgInNlZWQiOiBzZWVkLAog'
        'ICAgICAgICAgICAgICAgInN0cmVhbSI6IEZhbHNlLAogICAgICAgICAgICB9LAogICAgICAgICkK'
    ),
    'olympus/training/__init__.py': (
        'IiIiVHJhaW5pbmcgZnJhbWV3b3JrLiIiIgo='
    ),
    'olympus/training/campaign.py': (
        '...'
    ),
    'olympus/training/checkpoints.py': (
        'IiIiVmVyc2lvbmVkIG9wdGltaXplci1ib3VuZGFyeSBjaGVja3BvaW50czsgbG9hZCBvbmx5IHBp'
        'bm5lZCB3ZWlnaHRzLW9ubHkgcGF5bG9hZHMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFu'
        'bm90YXRpb25zCgppbXBvcnQgaW8KaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmlt'
        'cG9ydCByYW5kb20KaW1wb3J0IHV1aWQKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmZyb20gdGVt'
        'cGZpbGUgaW1wb3J0IE5hbWVkVGVtcG9yYXJ5RmlsZQpmcm9tIHR5cGluZyBpbXBvcnQgQW55Cgpp'
        'bXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgpmcm9tIG9seW1wdXMudHJhaW5pbmcucHJv'
        'dmVuYW5jZSBpbXBvcnQgYXRvbWljX2pzb24sIGRpZ2VzdF9qc29uLCBmaWxlX3NoYTI1Niwgc3lu'
        'Y19kaXJlY3RvcnkKCkNIRUNLUE9JTlRfU0NIRU1BID0gMgoKCmRlZiBfdmFsaWRhdGVfY2hlY2tw'
        'b2ludF9jb250cmFjdCgKICAgIHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldLCBwb2ludGVyOiBkaWN0'
        'W3N0ciwgQW55XSwgYmluZGluZzogZGljdFtzdHIsIEFueV0KKSAtPiBOb25lOgogICAgIiIiVmFs'
        'aWRhdGUgdGhlIHJlc3VtZSBib3VuZGFyeSB1c2luZyBwcmltaXRpdmUgbWV0YWRhdGEgb25seSwg'
        'YmVmb3JlIG11dGF0aW9uLiIiIgoKICAgIGZvciBsYWJlbCwgcmVjb3JkIGluICgoInBheWxvYWQi'
        'LCBwYXlsb2FkKSwgKCJwb2ludGVyIiwgcG9pbnRlcikpOgogICAgICAgIGlmICgKICAgICAgICAg'
        'ICAgdHlwZShyZWNvcmQuZ2V0KCJzY2hlbWFfdmVyc2lvbiIpKSBpcyBub3QgaW50CiAgICAgICAg'
        'ICAgIG9yIHJlY29yZFsic2NoZW1hX3ZlcnNpb24iXSAhPSBDSEVDS1BPSU5UX1NDSEVNQQogICAg'
        'ICAgICk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bnN1cHBvcnRlZCBjaGVja3Bv'
        'aW50IHtsYWJlbH0gc2NoZW1hIikKICAgICAgICBpZiByZWNvcmQuZ2V0KCJiaW5kaW5nIikgIT0g'
        'YmluZGluZzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImNoZWNrcG9pbnQge2xhYmVs'
        'fSBpZGVudGl0eSBtaXNtYXRjaCIpCiAgICBjb25maWcgPSBwYXlsb2FkLmdldCgidHJhaW5pbmdf'
        'Y29uZmlnIikKICAgIGlmIG5vdCBpc2luc3RhbmNlKGNvbmZpZywgZGljdCkgb3Igbm90IGlzaW5z'
        'dGFuY2UoY29uZmlnLmdldCgibW9kZWwiKSwgZGljdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJv'
        'cigiY2hlY2twb2ludCBtdXN0IGNvbnRhaW4gaXRzIHRyYWluaW5nIGFuZCBtb2RlbCBjb25maWd1'
        'cmF0aW9uIikKICAgIHJhd19jb25maWcgPSBkaWN0KGNvbmZpZykKICAgIHJhd19jb25maWcucG9w'
        'KCJ3b3JrbG9hZF9sb2NrIiwgTm9uZSkKICAgIHRyeToKICAgICAgICBjb25maWdfc2hhMjU2ID0g'
        'ZGlnZXN0X2pzb24ocmF3X2NvbmZpZykKICAgIGV4Y2VwdCAoVHlwZUVycm9yLCBWYWx1ZUVycm9y'
        'KSBhcyBlcnJvcjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IGNvbmZpZ3Vy'
        'YXRpb24gbXVzdCBjb250YWluIGZpbml0ZSBKU09OIHZhbHVlcyIpIGZyb20gZXJyb3IKICAgIGlm'
        'IGNvbmZpZ19zaGEyNTYgIT0gYmluZGluZy5nZXQoInRyYWluaW5nX2NvbmZpZ19zaGEyNTYiKToK'
        'ICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IHJhdyBjb25maWd1cmF0aW9uIGRp'
        'Z2VzdCBtaXNtYXRjaCIpCiAgICB0b3RhbF9zdGVwcyA9IGNvbmZpZy5nZXQoInN0ZXBzIikKICAg'
        'IGlmIHR5cGUodG90YWxfc3RlcHMpIGlzIG5vdCBpbnQgb3IgdG90YWxfc3RlcHMgPCAxOgogICAg'
        'ICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgdG90YWwgc3RlcCBidWRnZXQgbXVzdCBi'
        'ZSBhIHBvc2l0aXZlIGludGVnZXIiKQogICAgY29tcGxldGVkID0gcGF5bG9hZC5nZXQoImNvbXBs'
        'ZXRlZF9zdGVwcyIpCiAgICBpZiB0eXBlKGNvbXBsZXRlZCkgaXMgbm90IGludCBvciBub3QgMCA8'
        'PSBjb21wbGV0ZWQgPD0gdG90YWxfc3RlcHM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hl'
        'Y2twb2ludCBjb21wbGV0ZWQgc3RlcHMgZmFsbCBvdXRzaWRlIHRoZSBkZWNsYXJlZCB0cmFpbmlu'
        'ZyBidWRnZXQiKQogICAgaWYgdHlwZShwb2ludGVyLmdldCgiY29tcGxldGVkX3N0ZXBzIikpIGlz'
        'IG5vdCBpbnQgb3IgcG9pbnRlclsiY29tcGxldGVkX3N0ZXBzIl0gIT0gY29tcGxldGVkOgogICAg'
        'ICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgcG9pbnRlciBjb3VudGVyIG1pc21hdGNo'
        'IikKICAgIGNvdW50ZXJzID0gcGF5bG9hZC5nZXQoImNvdW50ZXJzIikKICAgIGlmIG5vdCBpc2lu'
        'c3RhbmNlKGNvdW50ZXJzLCBkaWN0KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3Bv'
        'aW50IGNvdW50ZXJzIG11c3QgYmUgYSBtYXBwaW5nIikKICAgIGlmIHR5cGUoY291bnRlcnMuZ2V0'
        'KCJjb21wbGV0ZWRfc3RlcHMiKSkgaXMgbm90IGludCBvciBjb3VudGVyc1siY29tcGxldGVkX3N0'
        'ZXBzIl0gIT0gY29tcGxldGVkOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQg'
        'bmVzdGVkIGNvbXBsZXRlZC1zdGVwIGNvdW50ZXIgbWlzbWF0Y2giKQogICAgYmVzdF9zdGVwID0g'
        'Y291bnRlcnMuZ2V0KCJiZXN0X3N0ZXAiKQogICAgaWYgdHlwZShiZXN0X3N0ZXApIGlzIG5vdCBp'
        'bnQgb3Igbm90IDAgPD0gYmVzdF9zdGVwIDw9IGNvbXBsZXRlZDoKICAgICAgICByYWlzZSBWYWx1'
        'ZUVycm9yKCJjaGVja3BvaW50IGJlc3Qgc3RlcCBtdXN0IGlkZW50aWZ5IGEgY29tcGxldGVkIG9w'
        'dGltaXplciBib3VuZGFyeSIpCiAgICBmb3IgbmFtZSBpbiAoCiAgICAgICAgImJlc3RfdmFsaWRh'
        'dGlvbl9sb3NzIiwKICAgICAgICAibWF4aW11bV9ncmFkaWVudF9ub3JtIiwKICAgICAgICAiZmlu'
        'YWxfbGVhcm5pbmdfcmF0ZSIsCiAgICAgICAgImVsYXBzZWRfc2Vjb25kcyIsCiAgICApOgogICAg'
        'ICAgIHZhbHVlID0gY291bnRlcnMuZ2V0KG5hbWUpCiAgICAgICAgaWYgKAogICAgICAgICAgICBp'
        'c2luc3RhbmNlKHZhbHVlLCBib29sKQogICAgICAgICAgICBvciBub3QgaXNpbnN0YW5jZSh2YWx1'
        'ZSwgKGludCwgZmxvYXQpKQogICAgICAgICAgICBvciBub3QgbWF0aC5pc2Zpbml0ZSh2YWx1ZSkK'
        'ICAgICAgICApOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiY2hlY2twb2ludCB7bmFt'
        'ZX0gbXVzdCBiZSBmaW5pdGUiKQogICAgICAgIGlmIG5hbWUgIT0gImJlc3RfdmFsaWRhdGlvbl9s'
        'b3NzIiBhbmQgdmFsdWUgPCAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiY2hlY2tw'
        'b2ludCB7bmFtZX0gbXVzdCBiZSBub24tbmVnYXRpdmUiKQogICAgZm9yIG5hbWUgaW4gKCJmaXJz'
        'dF9sb3NzIiwgImZpbmFsX2xvc3MiKToKICAgICAgICB2YWx1ZSA9IGNvdW50ZXJzLmdldChuYW1l'
        'KQogICAgICAgIGlmIGNvbXBsZXRlZCA9PSAwOgogICAgICAgICAgICBpZiBuYW1lIG5vdCBpbiBj'
        'b3VudGVycyBvciB2YWx1ZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVl'
        'RXJyb3IoZiJjaGVja3BvaW50IHtuYW1lfSBtdXN0IGJlIG51bGwgYmVmb3JlIHRoZSBmaXJzdCBv'
        'cHRpbWl6ZXIgc3RlcCIpCiAgICAgICAgZWxpZiAoCiAgICAgICAgICAgIGlzaW5zdGFuY2UodmFs'
        'dWUsIGJvb2wpCiAgICAgICAgICAgIG9yIG5vdCBpc2luc3RhbmNlKHZhbHVlLCAoaW50LCBmbG9h'
        'dCkpCiAgICAgICAgICAgIG9yIG5vdCBtYXRoLmlzZmluaXRlKHZhbHVlKQogICAgICAgICk6CiAg'
        'ICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJjaGVja3BvaW50IHtuYW1lfSBtdXN0IGJlIGZp'
        'bml0ZSBhZnRlciBvcHRpbWl6ZXIgZXhlY3V0aW9uIikKICAgIHNjaGVkdWxlciA9IHBheWxvYWQu'
        'Z2V0KCJzY2hlZHVsZXIiKQogICAgaWYgKAogICAgICAgIG5vdCBpc2luc3RhbmNlKHNjaGVkdWxl'
        'ciwgZGljdCkKICAgICAgICBvciBzY2hlZHVsZXIuZ2V0KCJraW5kIikgIT0gImZ1bmN0aW9uYWxf'
        'd2FybXVwX2Nvc2luZSIKICAgICAgICBvciB0eXBlKHNjaGVkdWxlci5nZXQoInRvdGFsX3N0ZXBz'
        'IikpIGlzIG5vdCBpbnQKICAgICAgICBvciBzY2hlZHVsZXJbInRvdGFsX3N0ZXBzIl0gIT0gdG90'
        'YWxfc3RlcHMKICAgICk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBzY2hl'
        'ZHVsZXIgZG9lcyBub3QgbWF0Y2ggdGhlIGZpeGVkIHRyYWluaW5nIGJ1ZGdldCIpCiAgICBpZiAo'
        'CiAgICAgICAgcGF5bG9hZC5nZXQoInByZWNpc2lvbiIpICE9ICJmcDMyIgogICAgICAgIG9yIGNv'
        'bmZpZy5nZXQoInByZWNpc2lvbiIpICE9ICJmcDMyIgogICAgICAgIG9yICJzY2FsZXIiIG5vdCBp'
        'biBwYXlsb2FkCiAgICAgICAgb3IgcGF5bG9hZFsic2NhbGVyIl0gaXMgbm90IE5vbmUKICAgICk6'
        'CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBzdXBwb3J0cyBvbmx5IHRoZSBk'
        'ZWNsYXJlZCBmcDMyIHBhdGggd2l0aG91dCBhbiBBTVAgc2NhbGVyIikKICAgIG1vZGVsX3N0YXRl'
        'ID0gcGF5bG9hZC5nZXQoIm1vZGVsX3N0YXRlIikKICAgIGlmICgKICAgICAgICBub3QgaXNpbnN0'
        'YW5jZShtb2RlbF9zdGF0ZSwgZGljdCkKICAgICAgICBvciBub3QgbW9kZWxfc3RhdGUKICAgICAg'
        'ICBvciBub3QgYWxsKGlzaW5zdGFuY2UobmFtZSwgc3RyKSBhbmQgbmFtZSBmb3IgbmFtZSBpbiBt'
        'b2RlbF9zdGF0ZSkKICAgICk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBt'
        'b2RlbCBzdGF0ZSBtdXN0IGJlIGEgbm9uLWVtcHR5IG5hbWVkIG1hcHBpbmciKQogICAgb3B0aW1p'
        'emVyID0gcGF5bG9hZC5nZXQoIm9wdGltaXplcl9zdGF0ZSIpCiAgICBpZiAoCiAgICAgICAgbm90'
        'IGlzaW5zdGFuY2Uob3B0aW1pemVyLCBkaWN0KQogICAgICAgIG9yIG5vdCBpc2luc3RhbmNlKG9w'
        'dGltaXplci5nZXQoInN0YXRlIiksIGRpY3QpCiAgICAgICAgb3Igbm90IGlzaW5zdGFuY2Uob3B0'
        'aW1pemVyLmdldCgicGFyYW1fZ3JvdXBzIiksIGxpc3QpCiAgICAgICAgb3Igbm90IG9wdGltaXpl'
        'clsicGFyYW1fZ3JvdXBzIl0KICAgICAgICBvciBub3QgYWxsKAogICAgICAgICAgICBpc2luc3Rh'
        'bmNlKGdyb3VwLCBkaWN0KQogICAgICAgICAgICBhbmQgaXNpbnN0YW5jZShncm91cC5nZXQoInBh'
        'cmFtcyIpLCBsaXN0KQogICAgICAgICAgICBhbmQgYWxsKHR5cGUoaW5kZXgpIGlzIGludCBhbmQg'
        'aW5kZXggPj0gMCBmb3IgaW5kZXggaW4gZ3JvdXBbInBhcmFtcyJdKQogICAgICAgICAgICBmb3Ig'
        'Z3JvdXAgaW4gb3B0aW1pemVyWyJwYXJhbV9ncm91cHMiXQogICAgICAgICkKICAgICk6CiAgICAg'
        'ICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBvcHRpbWl6ZXIgc3RhdGUgbXVzdCBjb250'
        'YWluIHN0YXRlIGFuZCBwYXJhbWV0ZXIgZ3JvdXBzIikKICAgIHNvdXJjZSA9IHBheWxvYWQuZ2V0'
        'KCJzb3VyY2VfaWRlbnRpdHkiKQogICAgaWYgbm90IGlzaW5zdGFuY2Uoc291cmNlLCBkaWN0KSBv'
        'ciBzb3VyY2UuZ2V0KCJzaGEyNTYiKSAhPSBiaW5kaW5nLmdldCgic291cmNlX3NoYTI1NiIpOgog'
        'ICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgc291cmNlIGlkZW50aXR5IGRvZXMg'
        'bm90IG1hdGNoIGl0cyBiaW5kaW5nIikKICAgIGlmIG5vdCBpc2luc3RhbmNlKHBheWxvYWQuZ2V0'
        'KCJpbml0aWFsX2V2YWx1YXRpb24iKSwgZGljdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi'
        'Y2hlY2twb2ludCBpbml0aWFsIGV2YWx1YXRpb24gbXVzdCBiZSBhIG1hcHBpbmciKQogICAgcm5n'
        'ID0gcGF5bG9hZC5nZXQoInJuZyIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShybmcsIGRpY3QpIG9y'
        'IG5vdCB7CiAgICAgICAgInB5dGhvbiIsCiAgICAgICAgIm51bXB5IiwKICAgICAgICAidG9yY2hf'
        'Y3B1IiwKICAgICAgICAidG9yY2hfY3VkYSIsCiAgICAgICAgInNhbXBsZXIiLAogICAgfS5pc3N1'
        'YnNldChybmcpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgbXVzdCBjb250'
        'YWluIGV2ZXJ5IHN1cHBvcnRlZCBSTkcgc3RhdGUiKQoKCmRlZiBfdmFsaWRhdGVfY2hlY2twb2lu'
        'dF90ZW5zb3JzKHBheWxvYWQ6IGRpY3Rbc3RyLCBBbnldKSAtPiBOb25lOgogICAgaWYgbm90IGFs'
        'bChpc2luc3RhbmNlKHZhbHVlLCB0b3JjaC5UZW5zb3IpIGZvciB2YWx1ZSBpbiBwYXlsb2FkWyJt'
        'b2RlbF9zdGF0ZSJdLnZhbHVlcygpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3Bv'
        'aW50IG1vZGVsIHN0YXRlIG11c3QgY29udGFpbiBvbmx5IHRlbnNvcnMiKQogICAgcm5nID0gcGF5'
        'bG9hZFsicm5nIl0KICAgIGlmIG5vdCBhbGwoaXNpbnN0YW5jZShybmdbbmFtZV0sIHRvcmNoLlRl'
        'bnNvcikgZm9yIG5hbWUgaW4gKCJ0b3JjaF9jcHUiLCAic2FtcGxlciIpKToKICAgICAgICByYWlz'
        'ZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IENQVSBhbmQgc2FtcGxlciBSTkcgc3RhdGVzIG11c3Qg'
        'YmUgdGVuc29ycyIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShybmdbInRvcmNoX2N1ZGEiXSwgbGlz'
        'dCkgb3Igbm90IGFsbCgKICAgICAgICBpc2luc3RhbmNlKHZhbHVlLCB0b3JjaC5UZW5zb3IpIGZv'
        'ciB2YWx1ZSBpbiBybmdbInRvcmNoX2N1ZGEiXQogICAgKToKICAgICAgICByYWlzZSBWYWx1ZUVy'
        'cm9yKCJjaGVja3BvaW50IENVREEgUk5HIHN0YXRlIG11c3QgYmUgYSBsaXN0IG9mIHRlbnNvcnMi'
        'KQoKCmRlZiBjYXB0dXJlX3JuZyhnZW5lcmF0b3I6IHRvcmNoLkdlbmVyYXRvciwgKiwgY3VkYTog'
        'Ym9vbCkgLT4gZGljdFtzdHIsIEFueV06CiAgICBhbGdvcml0aG0sIGtleXMsIHBvc2l0aW9uLCBo'
        'YXNfZ2F1c3MsIGNhY2hlZF9nYXVzcyA9IG5wLnJhbmRvbS5nZXRfc3RhdGUoKQogICAgcmV0dXJu'
        'IHsKICAgICAgICAicHl0aG9uIjogcmFuZG9tLmdldHN0YXRlKCksCiAgICAgICAgIm51bXB5Ijog'
        'W2FsZ29yaXRobSwga2V5cy50b2xpc3QoKSwgcG9zaXRpb24sIGhhc19nYXVzcywgY2FjaGVkX2dh'
        'dXNzXSwKICAgICAgICAidG9yY2hfY3B1IjogdG9yY2guZ2V0X3JuZ19zdGF0ZSgpLAogICAgICAg'
        'ICJ0b3JjaF9jdWRhIjogdG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlX2FsbCgpIGlmIGN1ZGEgZWxz'
        'ZSBbXSwKICAgICAgICAic2FtcGxlciI6IGdlbmVyYXRvci5nZXRfc3RhdGUoKSwKICAgIH0KCgpk'
        'ZWYgcmVzdG9yZV9ybmcoc3RhdGU6IGRpY3Rbc3RyLCBBbnldLCBnZW5lcmF0b3I6IHRvcmNoLkdl'
        'bmVyYXRvciwgKiwgY3VkYTogYm9vbCkgLT4gTm9uZToKICAgIHJhbmRvbS5zZXRzdGF0ZShzdGF0'
        'ZVsicHl0aG9uIl0pCiAgICBhbGdvcml0aG0sIGtleXMsIHBvc2l0aW9uLCBoYXNfZ2F1c3MsIGNh'
        'Y2hlZF9nYXVzcyA9IHN0YXRlWyJudW1weSJdCiAgICBucC5yYW5kb20uc2V0X3N0YXRlKAogICAg'
        'ICAgIChhbGdvcml0aG0sIG5wLmFycmF5KGtleXMsIGR0eXBlPW5wLnVpbnQzMiksIHBvc2l0aW9u'
        'LCBoYXNfZ2F1c3MsIGNhY2hlZF9nYXVzcykKICAgICkKICAgIHRvcmNoLnNldF9ybmdfc3RhdGUo'
        'c3RhdGVbInRvcmNoX2NwdSJdKQogICAgZ2VuZXJhdG9yLnNldF9zdGF0ZShzdGF0ZVsic2FtcGxl'
        'ciJdKQogICAgaWYgY3VkYToKICAgICAgICBpZiBsZW4oc3RhdGVbInRvcmNoX2N1ZGEiXSkgIT0g'
        'dG9yY2guY3VkYS5kZXZpY2VfY291bnQoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi'
        'cmVzdW1lIENVREEgZGV2aWNlIGNvdW50IGRpZmZlcnMgZnJvbSB0aGUgc2F2ZWQgUk5HIHN0YXRl'
        'IikKICAgICAgICB0b3JjaC5jdWRhLnNldF9ybmdfc3RhdGVfYWxsKHN0YXRlWyJ0b3JjaF9jdWRh'
        'Il0pCgoKZGVmIHNhdmVfdHJhaW5pbmdfY2hlY2twb2ludCgKICAgIGRpcmVjdG9yeTogUGF0aCwg'
        'cGF5bG9hZDogZGljdFtzdHIsIEFueV0sICosIGJlc3Q6IGJvb2wKKSAtPiBkaWN0W3N0ciwgQW55'
        'XToKICAgICIiIlB1Ymxpc2ggaW1tdXRhYmxlIGNvbnRlbnRzLCB0aGVuIGF0b21pY2FsbHkgbW92'
        'ZSBsYXRlc3QvYmVzdCBtZXRhZGF0YSBwb2ludGVycy4KCiAgICBBIGtpbGxlZCB3cml0ZSBsZWF2'
        'ZXMgdGhlIHByaW9yIHBvaW50ZXIgaW50YWN0LiBVbnJlZmVyZW5jZWQgcGFydGlhbCBmaWxlcyBu'
        'ZXZlcgogICAgY291bnQgYXMgY2hlY2twb2ludHMuIE9sZCBzbmFwc2hvdHMgYXJlIHJldGFpbmVk'
        'IGZvciBpbnRlcnJ1cHRpb24gcHJvdmVuYW5jZS4KICAgICIiIgogICAgdmFsaWRhdGVkX3BheWxv'
        'YWQgPSB7KipwYXlsb2FkLCAic2NoZW1hX3ZlcnNpb24iOiBDSEVDS1BPSU5UX1NDSEVNQX0KICAg'
        'IGNvbnRyYWN0X3BvaW50ZXIgPSB7CiAgICAgICAgInNjaGVtYV92ZXJzaW9uIjogQ0hFQ0tQT0lO'
        'VF9TQ0hFTUEsCiAgICAgICAgImNvbXBsZXRlZF9zdGVwcyI6IHBheWxvYWQuZ2V0KCJjb21wbGV0'
        'ZWRfc3RlcHMiKSwKICAgICAgICAiYmluZGluZyI6IHBheWxvYWQuZ2V0KCJiaW5kaW5nIiksCiAg'
        'ICB9CiAgICBiaW5kaW5nID0gcGF5bG9hZC5nZXQoImJpbmRpbmciKQogICAgaWYgbm90IGlzaW5z'
        'dGFuY2UoYmluZGluZywgZGljdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2lu'
        'dCBiaW5kaW5nIG11c3QgYmUgYSBtYXBwaW5nIikKICAgIF92YWxpZGF0ZV9jaGVja3BvaW50X2Nv'
        'bnRyYWN0KHZhbGlkYXRlZF9wYXlsb2FkLCBjb250cmFjdF9wb2ludGVyLCBiaW5kaW5nKQogICAg'
        'X3ZhbGlkYXRlX2NoZWNrcG9pbnRfdGVuc29ycyh2YWxpZGF0ZWRfcGF5bG9hZCkKICAgIGRpcmVj'
        'dG9yeS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBmaWxlbmFtZSA9IGYi'
        'c3RlcC17cGF5bG9hZFsnY29tcGxldGVkX3N0ZXBzJ106MDlkfS17dXVpZC51dWlkNCgpLmhleH0u'
        'cHQiCiAgICB0YXJnZXQgPSBkaXJlY3RvcnkgLyBmaWxlbmFtZQogICAgdGVtcG9yYXJ5OiBQYXRo'
        'IHwgTm9uZSA9IE5vbmUKICAgIHRyeToKICAgICAgICB3aXRoIE5hbWVkVGVtcG9yYXJ5RmlsZShk'
        'aXI9ZGlyZWN0b3J5LCBwcmVmaXg9Ii5jaGVja3BvaW50LSIsIGRlbGV0ZT1GYWxzZSkgYXMgaGFu'
        'ZGxlOgogICAgICAgICAgICB0ZW1wb3JhcnkgPSBQYXRoKGhhbmRsZS5uYW1lKQogICAgICAgICAg'
        'ICB0b3JjaC5zYXZlKHZhbGlkYXRlZF9wYXlsb2FkLCBoYW5kbGUpCiAgICAgICAgICAgIGhhbmRs'
        'ZS5mbHVzaCgpCiAgICAgICAgICAgIG9zLmZzeW5jKGhhbmRsZS5maWxlbm8oKSkKICAgICAgICBv'
        'cy5saW5rKHRlbXBvcmFyeSwgdGFyZ2V0KQogICAgICAgIHN5bmNfZGlyZWN0b3J5KGRpcmVjdG9y'
        'eSkKICAgIGZpbmFsbHk6CiAgICAgICAgaWYgdGVtcG9yYXJ5IGlzIG5vdCBOb25lOgogICAgICAg'
        'ICAgICB0ZW1wb3JhcnkudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkKICAgIHBvaW50ZXIgPSB7CiAg'
        'ICAgICAgInNjaGVtYV92ZXJzaW9uIjogQ0hFQ0tQT0lOVF9TQ0hFTUEsCiAgICAgICAgInBhdGgi'
        'OiBmaWxlbmFtZSwKICAgICAgICAic2hhMjU2IjogZmlsZV9zaGEyNTYodGFyZ2V0KSwKICAgICAg'
        'ICAiY29tcGxldGVkX3N0ZXBzIjogcGF5bG9hZFsiY29tcGxldGVkX3N0ZXBzIl0sCiAgICAgICAg'
        'ImJpbmRpbmciOiBwYXlsb2FkWyJiaW5kaW5nIl0sCiAgICB9CiAgICAjIEJlc3QgZmlyc3Q6IGxh'
        'dGVzdCBhbHdheXMgcmVjb3JkcyBhIHJlY292ZXJhYmxlIGJlc3QgY2hlY2twb2ludC4KICAgIGlm'
        'IGJlc3Q6CiAgICAgICAgYXRvbWljX2pzb24oZGlyZWN0b3J5IC8gImJlc3QuanNvbiIsIHBvaW50'
        'ZXIpCiAgICBhdG9taWNfanNvbihkaXJlY3RvcnkgLyAibGF0ZXN0Lmpzb24iLCBwb2ludGVyKQog'
        'ICAgcmV0dXJuIHBvaW50ZXIKCgpkZWYgbG9hZF90cmFpbmluZ19jaGVja3BvaW50KAogICAgZGly'
        'ZWN0b3J5OiBQYXRoLCAqLCBraW5kOiBzdHIsIGJpbmRpbmc6IGRpY3Rbc3RyLCBBbnldCikgLT4g'
        'ZGljdFtzdHIsIEFueV06CiAgICBpZiBraW5kIG5vdCBpbiB7ImJlc3QiLCAibGF0ZXN0In06CiAg'
        'ICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBtdXN0IGJlIGJlc3Qgb3IgbGF0ZXN0'
        'IikKICAgIHBvaW50ZXJfcGF0aCA9IGRpcmVjdG9yeSAvIGYie2tpbmR9Lmpzb24iCiAgICBpZiBw'
        'b2ludGVyX3BhdGguaXNfc3ltbGluaygpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNr'
        'cG9pbnQgcG9pbnRlciBtdXN0IG5vdCBiZSBhIHN5bWxpbmsiKQogICAgcG9pbnRlciA9IGpzb24u'
        'bG9hZHMocG9pbnRlcl9wYXRoLnJlYWRfdGV4dCgpKQogICAgaWYgbm90IGlzaW5zdGFuY2UocG9p'
        'bnRlciwgZGljdCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2hlY2twb2ludCBwb2ludGVy'
        'IG11c3QgYmUgYSBtYXBwaW5nIikKICAgIGZpbGVuYW1lID0gcG9pbnRlci5nZXQoInBhdGgiKQog'
        'ICAgaWYgbm90IGlzaW5zdGFuY2UoZmlsZW5hbWUsIHN0cikgb3IgUGF0aChmaWxlbmFtZSkubmFt'
        'ZSAhPSBmaWxlbmFtZToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IHBvaW50'
        'ZXIgcGF0aCBtdXN0IGJlIGEgY29udGFpbmVkIGZpbGVuYW1lIikKICAgIHBhdGggPSBkaXJlY3Rv'
        'cnkgLyBmaWxlbmFtZQogICAgaWYgcGF0aC5pc19zeW1saW5rKCkgb3Igbm90IHBhdGguaXNfZmls'
        'ZSgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoZWNrcG9pbnQgbXVzdCBiZSBhIHJlZ3Vs'
        'YXIgbm9uLXN5bWxpbmsgZmlsZSIpCiAgICBjb250ZW50ID0gcGF0aC5yZWFkX2J5dGVzKCkKICAg'
        'IGltcG9ydCBoYXNobGliCgogICAgaWYgaGFzaGxpYi5zaGEyNTYoY29udGVudCkuaGV4ZGlnZXN0'
        'KCkgIT0gcG9pbnRlci5nZXQoInNoYTI1NiIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInRy'
        'YWluaW5nIGNoZWNrcG9pbnQgZGlnZXN0IG1pc21hdGNoIikKICAgIGlmIHBvaW50ZXIuZ2V0KCJi'
        'aW5kaW5nIikgIT0gYmluZGluZyBvciBwb2ludGVyLmdldCgic2NoZW1hX3ZlcnNpb24iKSAhPSBD'
        'SEVDS1BPSU5UX1NDSEVNQToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJjaGVja3BvaW50IHNv'
        'dXJjZSwgY29uZmlndXJhdGlvbiwgZGF0YSBvciBlbnZpcm9ubWVudCBpZGVudGl0eSBtaXNtYXRj'
        'aCIpCiAgICAjIE5vIGN1c3RvbSBwaWNrbGUgZ2xvYmFscyBvciB1bnNhZmUgZmFsbGJhY2suIFRy'
        'ZWF0IHN1cHBsaWVkIGZpbGVzIGFzIHVudHJ1c3RlZC4KICAgIHBheWxvYWQgPSB0b3JjaC5sb2Fk'
        'KGlvLkJ5dGVzSU8oY29udGVudCksIHdlaWdodHNfb25seT1UcnVlLCBtYXBfbG9jYXRpb249ImNw'
        'dSIpCiAgICBpZiBub3QgaXNpbnN0YW5jZShwYXlsb2FkLCBkaWN0KSBvciBwYXlsb2FkLmdldCgi'
        'c2NoZW1hX3ZlcnNpb24iKSAhPSBDSEVDS1BPSU5UX1NDSEVNQToKICAgICAgICByYWlzZSBWYWx1'
        'ZUVycm9yKCJ1bnN1cHBvcnRlZCB0cmFpbmluZyBjaGVja3BvaW50IHNjaGVtYSIpCiAgICBfdmFs'
        'aWRhdGVfY2hlY2twb2ludF9jb250cmFjdChwYXlsb2FkLCBwb2ludGVyLCBiaW5kaW5nKQogICAg'
        'X3ZhbGlkYXRlX2NoZWNrcG9pbnRfdGVuc29ycyhwYXlsb2FkKQogICAgcmV0dXJuIHBheWxvYWQK'
    ),
    'olympus/training/encoder_quantization.py': (
        'IiIiSGFzaC1ib3VuZCByYXRlLWRpc3RvcnRpb24gdHJhaW5pbmcgZm9yIHRoZSBlbmNvZGVyLXF1'
        'YW50aXphdGlvbiBwYXRod2F5LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9u'
        'cwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGlvCmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmltcG9y'
        'dCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgsIFB1cmVQb3NpeFBhdGgKZnJvbSB0ZW1wZmls'
        'ZSBpbXBvcnQgTmFtZWRUZW1wb3JhcnlGaWxlCmZyb20gdHlwaW5nIGltcG9ydCBBbnkKCmltcG9y'
        'dCB0b3JjaApmcm9tIHB5ZGFudGljIGltcG9ydCBGaWVsZCwgbW9kZWxfdmFsaWRhdG9yCmZyb20g'
        'dG9yY2ggaW1wb3J0IFRlbnNvciwgbm4KCmZyb20gb2x5bXB1cy5jb3JlLnNjaGVtYXMgaW1wb3J0'
        'IFN0cmljdE1vZGVsCmZyb20gb2x5bXB1cy5tb2RlbHMuZW5jb2Rlcl9xdWFudGl6YXRpb24gaW1w'
        'b3J0ICgKICAgIEVuY29kZXJRdWFudGl6YXRpb25Db25maWcsCiAgICBRdWFudGl6ZWRBdXRvZW5j'
        'b2RlciwKKQoKCmNsYXNzIEZlYXR1cmVUZW5zb3JBcnRpZmFjdChTdHJpY3RNb2RlbCk6CiAgICBw'
        'YXRoOiBzdHIKICAgIHNoYTI1Njogc3RyID0gRmllbGQocGF0dGVybj1yIl5bMC05YS1mXXs2NH0k'
        'IikKICAgIHNoYXBlOiBsaXN0W2ludF0gPSBGaWVsZChtaW5fbGVuZ3RoPTIsIG1heF9sZW5ndGg9'
        'MykKCiAgICBAbW9kZWxfdmFsaWRhdG9yKG1vZGU9ImFmdGVyIikKICAgIGRlZiB2YWxpZF9hcnRp'
        'ZmFjdChzZWxmKSAtPiBGZWF0dXJlVGVuc29yQXJ0aWZhY3Q6CiAgICAgICAgcmVsYXRpdmUgPSBQ'
        'dXJlUG9zaXhQYXRoKHNlbGYucGF0aCkKICAgICAgICBpZiByZWxhdGl2ZS5pc19hYnNvbHV0ZSgp'
        'IG9yICIuLiIgaW4gcmVsYXRpdmUucGFydHMgb3Igc2VsZi5wYXRoIGluIHsiIiwgIi4ifToKICAg'
        'ICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZmVhdHVyZSBwYXRoIG11c3QgYmUgYSBjb250YWlu'
        'ZWQgcmVsYXRpdmUgcGF0aCIpCiAgICAgICAgaWYgYW55KGRpbWVuc2lvbiA8IDEgZm9yIGRpbWVu'
        'c2lvbiBpbiBzZWxmLnNoYXBlKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZmVhdHVy'
        'ZSB0ZW5zb3IgZGltZW5zaW9ucyBtdXN0IGJlIHBvc2l0aXZlIikKICAgICAgICByZXR1cm4gc2Vs'
        'ZgoKCmNsYXNzIEVuY29kZXJEYXRhc2V0TWFuaWZlc3QoU3RyaWN0TW9kZWwpOgogICAgc2NoZW1h'
        'X3ZlcnNpb246IGludCA9IDEKICAgIGxpY2Vuc2U6IHN0ciA9IEZpZWxkKG1pbl9sZW5ndGg9MSkK'
        'ICAgIHRyYWluX3NvdXJjZV9pZHM6IGxpc3Rbc3RyXSA9IEZpZWxkKG1pbl9sZW5ndGg9MSkKICAg'
        'IGV2YWx1YXRpb25fc291cmNlX2lkczogbGlzdFtzdHJdID0gRmllbGQobWluX2xlbmd0aD0xKQog'
        'ICAgdHJhaW5fZmVhdHVyZXM6IEZlYXR1cmVUZW5zb3JBcnRpZmFjdAogICAgZXZhbHVhdGlvbl9m'
        'ZWF0dXJlczogRmVhdHVyZVRlbnNvckFydGlmYWN0CgogICAgQG1vZGVsX3ZhbGlkYXRvcihtb2Rl'
        'PSJhZnRlciIpCiAgICBkZWYgZGlzam9pbnRfc291cmNlcyhzZWxmKSAtPiBFbmNvZGVyRGF0YXNl'
        'dE1hbmlmZXN0OgogICAgICAgIGlmIHNldChzZWxmLnRyYWluX3NvdXJjZV9pZHMpICYgc2V0KHNl'
        'bGYuZXZhbHVhdGlvbl9zb3VyY2VfaWRzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigi'
        'dHJhaW5pbmcgYW5kIGV2YWx1YXRpb24gc291cmNlIGlkZW50aXRpZXMgbXVzdCBiZSBkaXNqb2lu'
        'dCIpCiAgICAgICAgaWYgbGVuKHNldChzZWxmLnRyYWluX3NvdXJjZV9pZHMpKSAhPSBsZW4oc2Vs'
        'Zi50cmFpbl9zb3VyY2VfaWRzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigidHJhaW5p'
        'bmcgc291cmNlIGlkZW50aXRpZXMgbXVzdCBiZSB1bmlxdWUiKQogICAgICAgIGlmIGxlbihzZXQo'
        'c2VsZi5ldmFsdWF0aW9uX3NvdXJjZV9pZHMpKSAhPSBsZW4oc2VsZi5ldmFsdWF0aW9uX3NvdXJj'
        'ZV9pZHMpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJldmFsdWF0aW9uIHNvdXJjZSBp'
        'ZGVudGl0aWVzIG11c3QgYmUgdW5pcXVlIikKICAgICAgICByZXR1cm4gc2VsZgoKCmNsYXNzIEVu'
        'Y29kZXJRdWFudGl6YXRpb25UcmFpbmluZ0NvbmZpZyhTdHJpY3RNb2RlbCk6CiAgICBzZWVkOiBp'
        'bnQgPSBGaWVsZChkZWZhdWx0PTIwMjYxMDExLCBnZT0wLCBsZT0yKiozMiAtIDEpCiAgICBzdGVw'
        'czogaW50ID0gRmllbGQoZGVmYXVsdD0xXzAwMCwgZ2U9MSwgbGU9MTBfMDAwXzAwMCkKICAgIGJh'
        'dGNoX3NpemU6IGludCA9IEZpZWxkKGRlZmF1bHQ9OCwgZ2U9MSwgbGU9NF8wOTYpCiAgICBsZWFy'
        'bmluZ19yYXRlOiBmbG9hdCA9IEZpZWxkKGRlZmF1bHQ9MWUtMywgZ3Q9MC4wLCBsZT0wLjEpCiAg'
        'ICBncmFkaWVudF9jbGlwX25vcm06IGZsb2F0ID0gRmllbGQoZGVmYXVsdD0xLjAsIGd0PTAuMCwg'
        'bGU9MTAwLjApCiAgICBlbmNvZGVyOiBFbmNvZGVyUXVhbnRpemF0aW9uQ29uZmlnCgoKY2xhc3Mg'
        'RW5jb2RlclF1YW50aXphdGlvbkV2YWx1YXRpb24oU3RyaWN0TW9kZWwpOgogICAgZXhhbXBsZXM6'
        'IGludCA9IEZpZWxkKGd0PTApCiAgICByZWNvbnN0cnVjdGlvbl9tc2U6IGZsb2F0ID0gRmllbGQo'
        'Z2U9MC4wKQogICAgcXVhbnRpemF0aW9uX2Vycm9yOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCkKICAg'
        'IGNvZGVib29rX3V0aWxpemF0aW9uOiBmbG9hdCA9IEZpZWxkKGdlPTAuMCwgbGU9MS4wKQoKCmNs'
        'YXNzIEVuY29kZXJRdWFudGl6YXRpb25UcmFpbmluZ1JlcG9ydChTdHJpY3RNb2RlbCk6CiAgICBz'
        'Y2hlbWFfdmVyc2lvbjogaW50ID0gMQogICAgc3RhdHVzOiBzdHIKICAgIGRhdGFzZXRfbWFuaWZl'
        'c3Rfc2hhMjU2OiBzdHIgPSBGaWVsZChwYXR0ZXJuPXIiXlswLTlhLWZdezY0fSQiKQogICAgY29t'
        'cGxldGVkX3N0ZXBzOiBpbnQgPSBGaWVsZChnZT0xKQogICAgZmluYWxfdHJhaW5pbmdfbG9zczog'
        'ZmxvYXQKICAgIGNoZWNrcG9pbnRfcGF0aDogc3RyCiAgICBjaGVja3BvaW50X3NoYTI1Njogc3Ry'
        'ID0gRmllbGQocGF0dGVybj1yIl5bMC05YS1mXXs2NH0kIikKICAgIGV2YWx1YXRpb246IEVuY29k'
        'ZXJRdWFudGl6YXRpb25FdmFsdWF0aW9uCiAgICBwcm9tb3Rpb25fYXV0aG9yaXplZDogYm9vbCA9'
        'IEZhbHNlCiAgICBjbGFpbV9ib3VuZGFyeTogc3RyCgoKZGVmIF9zaGEyNTYocGF5bG9hZDogYnl0'
        'ZXMpIC0+IHN0cjoKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihwYXlsb2FkKS5oZXhkaWdlc3Qo'
        'KQoKCmRlZiBfYXRvbWljX2pzb24ocGF0aDogUGF0aCwgcGF5bG9hZDogb2JqZWN0KSAtPiBOb25l'
        'OgogICAgc2VyaWFsaXplZCA9IChqc29uLmR1bXBzKHBheWxvYWQsIGluZGVudD0yLCBzb3J0X2tl'
        'eXM9VHJ1ZSkgKyAiXG4iKS5lbmNvZGUoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1U'
        'cnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBOYW1lZFRlbXBvcmFyeUZpbGUoZGlyPXBhdGgu'
        'cGFyZW50LCBwcmVmaXg9ZiIue3BhdGgubmFtZX0uIiwgZGVsZXRlPUZhbHNlKSBhcyBoYW5kbGU6'
        'CiAgICAgICAgaGFuZGxlLndyaXRlKHNlcmlhbGl6ZWQpCiAgICAgICAgaGFuZGxlLmZsdXNoKCkK'
        'ICAgICAgICBvcy5mc3luYyhoYW5kbGUuZmlsZW5vKCkpCiAgICAgICAgdGVtcG9yYXJ5ID0gUGF0'
        'aChoYW5kbGUubmFtZSkKICAgIHRlbXBvcmFyeS5yZXBsYWNlKHBhdGgpCgoKZGVmIF9hdG9taWNf'
        'dG9yY2hfc2F2ZShwYXRoOiBQYXRoLCBwYXlsb2FkOiBvYmplY3QpIC0+IE5vbmU6CiAgICBwYXRo'
        'LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB3aXRoIE5hbWVk'
        'VGVtcG9yYXJ5RmlsZShkaXI9cGF0aC5wYXJlbnQsIHByZWZpeD1mIi57cGF0aC5uYW1lfS4iLCBk'
        'ZWxldGU9RmFsc2UpIGFzIGhhbmRsZToKICAgICAgICB0ZW1wb3JhcnkgPSBQYXRoKGhhbmRsZS5u'
        'YW1lKQogICAgdHJ5OgogICAgICAgIHRvcmNoLnNhdmUocGF5bG9hZCwgdGVtcG9yYXJ5KQogICAg'
        'ICAgIHdpdGggdGVtcG9yYXJ5Lm9wZW4oInJiIikgYXMgaGFuZGxlOgogICAgICAgICAgICBvcy5m'
        'c3luYyhoYW5kbGUuZmlsZW5vKCkpCiAgICAgICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKICAg'
        'IGZpbmFsbHk6CiAgICAgICAgdGVtcG9yYXJ5LnVubGluayhtaXNzaW5nX29rPVRydWUpCgoKZGVm'
        'IGZlYXR1cmVfdGVuc29yX2FydGlmYWN0KHBhdGg6IFBhdGgsICosIHJvb3Q6IFBhdGgpIC0+IEZl'
        'YXR1cmVUZW5zb3JBcnRpZmFjdDoKICAgIHJlc29sdmVkX3Jvb3QgPSByb290LnJlc29sdmUoKQog'
        'ICAgcmVzb2x2ZWQgPSBwYXRoLnJlc29sdmUoKQogICAgaWYgcGF0aC5pc19zeW1saW5rKCkgb3Ig'
        'bm90IHJlc29sdmVkLmlzX3JlbGF0aXZlX3RvKHJlc29sdmVkX3Jvb3QpOgogICAgICAgIHJhaXNl'
        'IFZhbHVlRXJyb3IoImZlYXR1cmUgdGVuc29yIG11c3QgYmUgYSBjb250YWluZWQgcmVndWxhciBm'
        'aWxlIikKICAgIGNvbnRlbnQgPSByZXNvbHZlZC5yZWFkX2J5dGVzKCkKICAgIGxvYWRlZDogQW55'
        'ID0gdG9yY2gubG9hZChpby5CeXRlc0lPKGNvbnRlbnQpLCBtYXBfbG9jYXRpb249ImNwdSIsIHdl'
        'aWdodHNfb25seT1UcnVlKQogICAgaWYgbm90IGlzaW5zdGFuY2UobG9hZGVkLCBUZW5zb3IpIG9y'
        'IGxvYWRlZC5uZGltIG5vdCBpbiB7MiwgM306CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZmVh'
        'dHVyZSBhcnRpZmFjdCBtdXN0IGNvbnRhaW4gb25lIHJhbmstMiBvciByYW5rLTMgdGVuc29yIikK'
        'ICAgIGlmIG5vdCBsb2FkZWQuaXNfZmxvYXRpbmdfcG9pbnQoKToKICAgICAgICByYWlzZSBWYWx1'
        'ZUVycm9yKCJmZWF0dXJlIGFydGlmYWN0IG11c3QgY29udGFpbiBmbG9hdGluZy1wb2ludCB2YWx1'
        'ZXMiKQogICAgcmV0dXJuIEZlYXR1cmVUZW5zb3JBcnRpZmFjdCgKICAgICAgICBwYXRoPXJlc29s'
        'dmVkLnJlbGF0aXZlX3RvKHJlc29sdmVkX3Jvb3QpLmFzX3Bvc2l4KCksCiAgICAgICAgc2hhMjU2'
        'PV9zaGEyNTYoY29udGVudCksCiAgICAgICAgc2hhcGU9bGlzdChsb2FkZWQuc2hhcGUpLAogICAg'
        'KQoKCmRlZiB3cml0ZV9lbmNvZGVyX2RhdGFzZXRfbWFuaWZlc3QoCiAgICBvdXRwdXQ6IFBhdGgs'
        'CiAgICAqLAogICAgbGljZW5zZV9uYW1lOiBzdHIsCiAgICB0cmFpbl9zb3VyY2VfaWRzOiBsaXN0'
        'W3N0cl0sCiAgICBldmFsdWF0aW9uX3NvdXJjZV9pZHM6IGxpc3Rbc3RyXSwKICAgIHRyYWluX2Zl'
        'YXR1cmVzOiBQYXRoLAogICAgZXZhbHVhdGlvbl9mZWF0dXJlczogUGF0aCwKKSAtPiBFbmNvZGVy'
        'RGF0YXNldE1hbmlmZXN0OgogICAgcm9vdCA9IG91dHB1dC5yZXNvbHZlKCkucGFyZW50CiAgICBt'
        'YW5pZmVzdCA9IEVuY29kZXJEYXRhc2V0TWFuaWZlc3QoCiAgICAgICAgbGljZW5zZT1saWNlbnNl'
        'X25hbWUsCiAgICAgICAgdHJhaW5fc291cmNlX2lkcz10cmFpbl9zb3VyY2VfaWRzLAogICAgICAg'
        'IGV2YWx1YXRpb25fc291cmNlX2lkcz1ldmFsdWF0aW9uX3NvdXJjZV9pZHMsCiAgICAgICAgdHJh'
        'aW5fZmVhdHVyZXM9ZmVhdHVyZV90ZW5zb3JfYXJ0aWZhY3QodHJhaW5fZmVhdHVyZXMsIHJvb3Q9'
        'cm9vdCksCiAgICAgICAgZXZhbHVhdGlvbl9mZWF0dXJlcz1mZWF0dXJlX3RlbnNvcl9hcnRpZmFj'
        'dChldmFsdWF0aW9uX2ZlYXR1cmVzLCByb290PXJvb3QpLAogICAgKQogICAgX2F0b21pY19qc29u'
        'KG91dHB1dCwgbWFuaWZlc3QubW9kZWxfZHVtcChtb2RlPSJqc29uIikpCiAgICByZXR1cm4gbWFu'
        'aWZlc3QKCgpkZWYgX2xvYWRfZmVhdHVyZXMocm9vdDogUGF0aCwgYXJ0aWZhY3Q6IEZlYXR1cmVU'
        'ZW5zb3JBcnRpZmFjdCkgLT4gVGVuc29yOgogICAgcGF0aCA9IHJvb3QgLyBhcnRpZmFjdC5wYXRo'
        'CiAgICBpZiBwYXRoLmlzX3N5bWxpbmsoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJmZWF0'
        'dXJlIHRlbnNvciBtdXN0IG5vdCBiZSBhIHN5bWxpbmsiKQogICAgcmVzb2x2ZWQgPSBwYXRoLnJl'
        'c29sdmUoKQogICAgaWYgbm90IHJlc29sdmVkLmlzX3JlbGF0aXZlX3RvKHJvb3QpOgogICAgICAg'
        'IHJhaXNlIFZhbHVlRXJyb3IoImZlYXR1cmUgdGVuc29yIGVzY2FwZXMgdGhlIG1hbmlmZXN0IHJv'
        'b3QiKQogICAgY29udGVudCA9IHJlc29sdmVkLnJlYWRfYnl0ZXMoKQogICAgaWYgX3NoYTI1Nihj'
        'b250ZW50KSAhPSBhcnRpZmFjdC5zaGEyNTY6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImZl'
        'YXR1cmUgdGVuc29yIGRpZ2VzdCBjaGFuZ2VkOiB7YXJ0aWZhY3QucGF0aH0iKQogICAgbG9hZGVk'
        'OiBBbnkgPSB0b3JjaC5sb2FkKGlvLkJ5dGVzSU8oY29udGVudCksIG1hcF9sb2NhdGlvbj0iY3B1'
        'Iiwgd2VpZ2h0c19vbmx5PVRydWUpCiAgICBpZiBub3QgaXNpbnN0YW5jZShsb2FkZWQsIFRlbnNv'
        'cikgb3IgbGlzdChsb2FkZWQuc2hhcGUpICE9IGFydGlmYWN0LnNoYXBlOgogICAgICAgIHJhaXNl'
        'IFZhbHVlRXJyb3IoZiJmZWF0dXJlIHRlbnNvciBzaGFwZSBjaGFuZ2VkOiB7YXJ0aWZhY3QucGF0'
        'aH0iKQogICAgZmVhdHVyZXMgPSBsb2FkZWQuZmxvYXQoKQogICAgaWYgbm90IGJvb2wodG9yY2gu'
        'aXNmaW5pdGUoZmVhdHVyZXMpLmFsbCgpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZmVh'
        'dHVyZSB0ZW5zb3IgY29udGFpbnMgbm9uLWZpbml0ZSB2YWx1ZXM6IHthcnRpZmFjdC5wYXRofSIp'
        'CiAgICByZXR1cm4gZmVhdHVyZXMKCgpkZWYgX2V2YWx1YXRlKG1vZGVsOiBRdWFudGl6ZWRBdXRv'
        'ZW5jb2RlciwgZmVhdHVyZXM6IFRlbnNvcikgLT4gRW5jb2RlclF1YW50aXphdGlvbkV2YWx1YXRp'
        'b246CiAgICBtb2RlbC5ldmFsKCkKICAgIHdpdGggdG9yY2guaW5mZXJlbmNlX21vZGUoKToKICAg'
        'ICAgICBvdXRwdXQgPSBtb2RlbChmZWF0dXJlcykKICAgICAgICByZXR1cm4gRW5jb2RlclF1YW50'
        'aXphdGlvbkV2YWx1YXRpb24oCiAgICAgICAgICAgIGV4YW1wbGVzPWZlYXR1cmVzLnNoYXBlWzBd'
        'LAogICAgICAgICAgICByZWNvbnN0cnVjdGlvbl9tc2U9ZmxvYXQoCiAgICAgICAgICAgICAgICBu'
        'bi5mdW5jdGlvbmFsLm1zZV9sb3NzKG91dHB1dC5yZWNvbnN0cnVjdGlvbiwgZmVhdHVyZXMpLml0'
        'ZW0oKQogICAgICAgICAgICApLAogICAgICAgICAgICBxdWFudGl6YXRpb25fZXJyb3I9ZmxvYXQo'
        'b3V0cHV0LnF1YW50aXphdGlvbi5xdWFudGl6YXRpb25fZXJyb3IuaXRlbSgpKSwKICAgICAgICAg'
        'ICAgY29kZWJvb2tfdXRpbGl6YXRpb249ZmxvYXQob3V0cHV0LnF1YW50aXphdGlvbi51dGlsaXph'
        'dGlvbi5pdGVtKCkpLAogICAgICAgICkKCgpkZWYgdHJhaW5fZW5jb2Rlcl9xdWFudGl6ZXIoCiAg'
        'ICBtYW5pZmVzdF9wYXRoOiBQYXRoLAogICAgb3V0cHV0X2RpcmVjdG9yeTogUGF0aCwKICAgICos'
        'CiAgICBjb25maWc6IEVuY29kZXJRdWFudGl6YXRpb25UcmFpbmluZ0NvbmZpZywKKSAtPiBFbmNv'
        'ZGVyUXVhbnRpemF0aW9uVHJhaW5pbmdSZXBvcnQ6CiAgICAiIiJUcmFpbiBhbmQgZXZhbHVhdGUg'
        'YSByZXNpZHVhbC1WUSBlbmNvZGVyIHdpdGhvdXQgYXV0aG9yaXppbmcgcHJvbW90aW9uLiIiIgoK'
        'ICAgIG1hbmlmZXN0X2J5dGVzID0gbWFuaWZlc3RfcGF0aC5yZWFkX2J5dGVzKCkKICAgIG1hbmlm'
        'ZXN0ID0gRW5jb2RlckRhdGFzZXRNYW5pZmVzdC5tb2RlbF92YWxpZGF0ZV9qc29uKG1hbmlmZXN0'
        'X2J5dGVzKQogICAgcm9vdCA9IG1hbmlmZXN0X3BhdGgucmVzb2x2ZSgpLnBhcmVudAogICAgdHJh'
        'aW5fZmVhdHVyZXMgPSBfbG9hZF9mZWF0dXJlcyhyb290LCBtYW5pZmVzdC50cmFpbl9mZWF0dXJl'
        'cykKICAgIGV2YWx1YXRpb25fZmVhdHVyZXMgPSBfbG9hZF9mZWF0dXJlcyhyb290LCBtYW5pZmVz'
        'dC5ldmFsdWF0aW9uX2ZlYXR1cmVzKQogICAgaWYgdHJhaW5fZmVhdHVyZXMuc2hhcGVbLTFdICE9'
        'IGNvbmZpZy5lbmNvZGVyLmlucHV0X2RpbToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ0cmFp'
        'bmluZyBmZWF0dXJlIHdpZHRoIGRvZXMgbm90IG1hdGNoIGVuY29kZXIgaW5wdXQiKQogICAgaWYg'
        'ZXZhbHVhdGlvbl9mZWF0dXJlcy5zaGFwZVstMV0gIT0gY29uZmlnLmVuY29kZXIuaW5wdXRfZGlt'
        'OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImV2YWx1YXRpb24gZmVhdHVyZSB3aWR0aCBkb2Vz'
        'IG5vdCBtYXRjaCBlbmNvZGVyIGlucHV0IikKICAgIGlmIHRyYWluX2ZlYXR1cmVzLnNoYXBlWzBd'
        'ICE9IGxlbihtYW5pZmVzdC50cmFpbl9zb3VyY2VfaWRzKToKICAgICAgICByYWlzZSBWYWx1ZUVy'
        'cm9yKCJ0cmFpbmluZyBmZWF0dXJlIHJvd3MgbXVzdCBtYXRjaCB0cmFpbmluZyBzb3VyY2UgaWRl'
        'bnRpdGllcyIpCiAgICBpZiBldmFsdWF0aW9uX2ZlYXR1cmVzLnNoYXBlWzBdICE9IGxlbihtYW5p'
        'ZmVzdC5ldmFsdWF0aW9uX3NvdXJjZV9pZHMpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImV2'
        'YWx1YXRpb24gZmVhdHVyZSByb3dzIG11c3QgbWF0Y2ggZXZhbHVhdGlvbiBzb3VyY2UgaWRlbnRp'
        'dGllcyIpCiAgICBpZiBjb25maWcuYmF0Y2hfc2l6ZSA+IHRyYWluX2ZlYXR1cmVzLnNoYXBlWzBd'
        'OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJhdGNoIHNpemUgbXVzdCBub3QgZXhjZWVkIHRy'
        'YWluaW5nIGV4YW1wbGVzIikKICAgIHJlcG9ydF9wYXRoID0gb3V0cHV0X2RpcmVjdG9yeSAvICJy'
        'ZXBvcnQuanNvbiIKICAgIGNoZWNrcG9pbnRfcGF0aCA9IG91dHB1dF9kaXJlY3RvcnkgLyAiY2hl'
        'Y2twb2ludC5wdCIKICAgIGlmIHJlcG9ydF9wYXRoLmV4aXN0cygpIG9yIGNoZWNrcG9pbnRfcGF0'
        'aC5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlRXhpc3RzRXJyb3IoInJlZnVzaW5nIHRvIG92'
        'ZXJ3cml0ZSBhbiBleGlzdGluZyBlbmNvZGVyIHJ1biIpCgogICAgdG9yY2gubWFudWFsX3NlZWQo'
        'Y29uZmlnLnNlZWQpCiAgICBnZW5lcmF0b3IgPSB0b3JjaC5HZW5lcmF0b3IoZGV2aWNlPSJjcHUi'
        'KS5tYW51YWxfc2VlZChjb25maWcuc2VlZCkKICAgIG1vZGVsID0gUXVhbnRpemVkQXV0b2VuY29k'
        'ZXIoY29uZmlnLmVuY29kZXIpCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtVyhtb2Rl'
        'bC5wYXJhbWV0ZXJzKCksIGxyPWNvbmZpZy5sZWFybmluZ19yYXRlKQogICAgZmluYWxfbG9zcyA9'
        'IG1hdGgubmFuCiAgICBtb2RlbC50cmFpbigpCiAgICBmb3IgX3N0ZXAgaW4gcmFuZ2UoY29uZmln'
        'LnN0ZXBzKToKICAgICAgICBpbmRpY2VzID0gdG9yY2gucmFuZGludCh0cmFpbl9mZWF0dXJlcy5z'
        'aGFwZVswXSwgKGNvbmZpZy5iYXRjaF9zaXplLCksIGdlbmVyYXRvcj1nZW5lcmF0b3IpCiAgICAg'
        'ICAgYmF0Y2ggPSB0cmFpbl9mZWF0dXJlc1tpbmRpY2VzXQogICAgICAgIG9wdGltaXplci56ZXJv'
        'X2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICBvdXRwdXQgPSBtb2RlbChiYXRjaCkKICAg'
        'ICAgICByZWNvbnN0cnVjdGlvbiA9IG5uLmZ1bmN0aW9uYWwubXNlX2xvc3Mob3V0cHV0LnJlY29u'
        'c3RydWN0aW9uLCBiYXRjaCkKICAgICAgICBsb3NzID0gcmVjb25zdHJ1Y3Rpb24gKyBvdXRwdXQu'
        'cXVhbnRpemF0aW9uLmxvc3MKICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUobG9zcyk6CiAg'
        'ICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcigibm9uLWZpbml0ZSBlbmNvZGVyLXF1'
        'YW50aXphdGlvbiBsb3NzIikKICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICBncmFkaWVu'
        'dF9ub3JtID0gbm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgY29u'
        'ZmlnLmdyYWRpZW50X2NsaXBfbm9ybSkKICAgICAgICBpZiBub3QgdG9yY2guaXNmaW5pdGUoZ3Jh'
        'ZGllbnRfbm9ybSk6CiAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcigibm9uLWZp'
        'bml0ZSBlbmNvZGVyLXF1YW50aXphdGlvbiBncmFkaWVudCIpCiAgICAgICAgb3B0aW1pemVyLnN0'
        'ZXAoKQogICAgICAgIGZpbmFsX2xvc3MgPSBmbG9hdChsb3NzLmRldGFjaCgpLml0ZW0oKSkKCiAg'
        'ICBldmFsdWF0aW9uID0gX2V2YWx1YXRlKG1vZGVsLCBldmFsdWF0aW9uX2ZlYXR1cmVzKQogICAg'
        'Y2hlY2twb2ludCA9IHsKICAgICAgICAic2NoZW1hX3ZlcnNpb24iOiAxLAogICAgICAgICJkYXRh'
        'c2V0X21hbmlmZXN0X3NoYTI1NiI6IF9zaGEyNTYobWFuaWZlc3RfYnl0ZXMpLAogICAgICAgICJj'
        'b25maWciOiBjb25maWcubW9kZWxfZHVtcChtb2RlPSJqc29uIiksCiAgICAgICAgIm1vZGVsX3N0'
        'YXRlIjogbW9kZWwuc3RhdGVfZGljdCgpLAogICAgICAgICJvcHRpbWl6ZXJfc3RhdGUiOiBvcHRp'
        'bWl6ZXIuc3RhdGVfZGljdCgpLAogICAgICAgICJjb21wbGV0ZWRfc3RlcHMiOiBjb25maWcuc3Rl'
        'cHMsCiAgICAgICAgImZpbmFsX3RyYWluaW5nX2xvc3MiOiBmaW5hbF9sb3NzLAogICAgICAgICJl'
        'dmFsdWF0aW9uIjogZXZhbHVhdGlvbi5tb2RlbF9kdW1wKG1vZGU9Impzb24iKSwKICAgIH0KICAg'
        'IF9hdG9taWNfdG9yY2hfc2F2ZShjaGVja3BvaW50X3BhdGgsIGNoZWNrcG9pbnQpCiAgICByZXBv'
        'cnQgPSBFbmNvZGVyUXVhbnRpemF0aW9uVHJhaW5pbmdSZXBvcnQoCiAgICAgICAgc3RhdHVzPSJD'
        'T01QTEVURURfREVWRUxPUE1FTlRfTk9UX1BST01PVEVEIiwKICAgICAgICBkYXRhc2V0X21hbmlm'
        'ZXN0X3NoYTI1Nj1fc2hhMjU2KG1hbmlmZXN0X2J5dGVzKSwKICAgICAgICBjb21wbGV0ZWRfc3Rl'
        'cHM9Y29uZmlnLnN0ZXBzLAogICAgICAgIGZpbmFsX3RyYWluaW5nX2xvc3M9ZmluYWxfbG9zcywK'
        'ICAgICAgICBjaGVja3BvaW50X3BhdGg9c3RyKGNoZWNrcG9pbnRfcGF0aC5yZXNvbHZlKCkpLAog'
        'ICAgICAgIGNoZWNrcG9pbnRfc2hhMjU2PV9zaGEyNTYoY2hlY2twb2ludF9wYXRoLnJlYWRfYnl0'
        'ZXMoKSksCiAgICAgICAgZXZhbHVhdGlvbj1ldmFsdWF0aW9uLAogICAgICAgIHByb21vdGlvbl9h'
        'dXRob3JpemVkPUZhbHNlLAogICAgICAgIGNsYWltX2JvdW5kYXJ5PSgKICAgICAgICAgICAgIlJh'
        'dGUtZGlzdG9ydGlvbiBleGVjdXRpb24gaXMgbm90IGV2aWRlbmNlIG9mIGRvd25zdHJlYW0gdGFz'
        'ayBxdWFsaXR5LCBhIHRyYWluZWQgIgogICAgICAgICAgICAiQlUxTEQgZmFtaWx5LCBvciB0YXJn'
        'ZXQtc2NhbGUgcXVhbnRpemVkIGluZmVyZW5jZS4iCiAgICAgICAgKSwKICAgICkKICAgIF9hdG9t'
        'aWNfanNvbihyZXBvcnRfcGF0aCwgcmVwb3J0Lm1vZGVsX2R1bXAobW9kZT0ianNvbiIpKQogICAg'
        'cmV0dXJuIHJlcG9ydAo='
    ),
    'olympus/training/olympus_system.py': (
        '...'
    ),
    'olympus/training/provenance.py': (
        'IiIiU21hbGwsIFRvcmNoLWZyZWUgaWRlbnRpdHkgYW5kIGR1cmFibGUtd3JpdGUgaGVscGVycyBm'
        'b3IgbG9jYWwgY2FtcGFpZ25zLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9u'
        'cwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzdWJwcm9jZXNz'
        'CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRlbXBmaWxlIGltcG9ydCBOYW1lZFRlbXBv'
        'cmFyeUZpbGUKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKCmRlZiBkaWdlc3RfanNvbih2YWx1ZTog'
        'b2JqZWN0KSAtPiBzdHI6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYoCiAgICAgICAganNvbi5k'
        'dW1wcyh2YWx1ZSwgc29ydF9rZXlzPVRydWUsIHNlcGFyYXRvcnM9KCIsIiwgIjoiKSwgYWxsb3df'
        'bmFuPUZhbHNlKS5lbmNvZGUoKQogICAgKS5oZXhkaWdlc3QoKQoKCmRlZiBmaWxlX3NoYTI1Nihw'
        'YXRoOiBQYXRoKSAtPiBzdHI6CiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NigpCiAgICB3aXRo'
        'IHBhdGgub3BlbigicmIiKSBhcyBoYW5kbGU6CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFt'
        'YmRhOiBoYW5kbGUucmVhZCgxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgIGRpZ2VzdC51'
        'cGRhdGUoYmxvY2spCiAgICByZXR1cm4gZGlnZXN0LmhleGRpZ2VzdCgpCgoKZGVmIGF0b21pY19q'
        'c29uKHBhdGg6IFBhdGgsIHZhbHVlOiBvYmplY3QpIC0+IE5vbmU6CiAgICAiIiJSZXBsYWNlIG1l'
        'dGFkYXRhIG9ubHkgYWZ0ZXIgaXRzIGNvbXBsZXRlIGNvbnRlbnRzIGhhdmUgcmVhY2hlZCB0aGUg'
        'ZmlsZXN5c3RlbS4iIiIKICAgIHBheWxvYWQgPSAoanNvbi5kdW1wcyh2YWx1ZSwgaW5kZW50PTIs'
        'IHNvcnRfa2V5cz1UcnVlLCBhbGxvd19uYW49RmFsc2UpICsgIlxuIikuZW5jb2RlKCkKICAgIHBh'
        'dGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRlbXBvcmFy'
        'eTogUGF0aCB8IE5vbmUgPSBOb25lCiAgICB0cnk6CiAgICAgICAgd2l0aCBOYW1lZFRlbXBvcmFy'
        'eUZpbGUoZGlyPXBhdGgucGFyZW50LCBwcmVmaXg9ZiIue3BhdGgubmFtZX0uIiwgZGVsZXRlPUZh'
        'bHNlKSBhcyBoYW5kbGU6CiAgICAgICAgICAgIHRlbXBvcmFyeSA9IFBhdGgoaGFuZGxlLm5hbWUp'
        'CiAgICAgICAgICAgIGhhbmRsZS53cml0ZShwYXlsb2FkKQogICAgICAgICAgICBoYW5kbGUuZmx1'
        'c2goKQogICAgICAgICAgICBvcy5mc3luYyhoYW5kbGUuZmlsZW5vKCkpCiAgICAgICAgdGVtcG9y'
        'YXJ5LnJlcGxhY2UocGF0aCkKICAgICAgICBzeW5jX2RpcmVjdG9yeShwYXRoLnBhcmVudCkKICAg'
        'IGZpbmFsbHk6CiAgICAgICAgaWYgdGVtcG9yYXJ5IGlzIG5vdCBOb25lOgogICAgICAgICAgICB0'
        'ZW1wb3JhcnkudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkKCgpkZWYgc3luY19kaXJlY3RvcnkocGF0'
        'aDogUGF0aCkgLT4gTm9uZToKICAgIGRlc2NyaXB0b3IgPSBvcy5vcGVuKHBhdGgsIG9zLk9fUkRP'
        'TkxZKQogICAgdHJ5OgogICAgICAgIG9zLmZzeW5jKGRlc2NyaXB0b3IpCiAgICBmaW5hbGx5Ogog'
        'ICAgICAgIG9zLmNsb3NlKGRlc2NyaXB0b3IpCgoKZGVmIHNvdXJjZV9pZGVudGl0eSgpIC0+IGRp'
        'Y3Rbc3RyLCBBbnldOgogICAgIiIiSGFzaCBhY3R1YWwgc291cmNlIGJ5dGVzLCBpbmNsdWRpbmcg'
        'dW5jb21taXR0ZWQvdW50cmFja2VkIGltcGxlbWVudGF0aW9uIGZpbGVzLgoKICAgIEdpdCBtZXRh'
        'ZGF0YSBpcyBpbmZvcm1hdGlvbmFsOyB0aGUgc291cmNlIGRpZ2VzdCBpcyBzdGFibGUgYWNyb3Nz'
        'IG91dHB1dCB3cml0ZXMsCiAgICByZXBvc2l0b3J5IHJlbG9jYXRpb24gYW5kIGNvbW1pdHMgdGhh'
        'dCBkbyBub3QgY2hhbmdlIGltcGxlbWVudGF0aW9uIGJ5dGVzLgogICAgTm8gZW52aXJvbm1lbnQg'
        'dmFyaWFibGVzLCByZW1vdGUgVVJMcyBvciBhcmJpdHJhcnkgd29ya3NwYWNlIGZpbGVzIGFyZSBy'
        'ZWNvcmRlZC4KICAgICIiIgogICAgcm9vdCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJl'
        'bnRzWzJdCiAgICBmaWxlcyA9IHNvcnRlZChbKnJvb3QuZ2xvYigib2x5bXB1cy8qKi8qLnB5Iiks'
        'ICpyb290Lmdsb2IoInNjcmlwdHMvKi5weSIpXSkKICAgIGZpbGVzLmV4dGVuZChwYXRoIGZvciBw'
        'YXRoIGluIFtyb290IC8gInB5cHJvamVjdC50b21sIl0gaWYgcGF0aC5pc19maWxlKCkpCiAgICBo'
        'YXNoZXMgPSB7c3RyKHBhdGgucmVsYXRpdmVfdG8ocm9vdCkpOiBmaWxlX3NoYTI1NihwYXRoKSBm'
        'b3IgcGF0aCBpbiBmaWxlc30KCiAgICBkZWYgZ2l0KCphcmdzOiBzdHIpIC0+IHN0ciB8IE5vbmU6'
        'CiAgICAgICAgdHJ5OgogICAgICAgICAgICByZXR1cm4gc3VicHJvY2Vzcy5jaGVja19vdXRwdXQo'
        'CiAgICAgICAgICAgICAgICBbImdpdCIsICphcmdzXSwgY3dkPXJvb3QsIHN0ZGVycj1zdWJwcm9j'
        'ZXNzLkRFVk5VTEwsIHRleHQ9VHJ1ZSwgdGltZW91dD01CiAgICAgICAgICAgICkuc3RyaXAoKQog'
        'ICAgICAgIGV4Y2VwdCAoT1NFcnJvciwgc3VicHJvY2Vzcy5TdWJwcm9jZXNzRXJyb3IpOiAgIyBm'
        'bXQ6IHNraXAKICAgICAgICAgICAgcmV0dXJuIE5vbmUKCiAgICBzdGF0dXMgPSBnaXQoInN0YXR1'
        'cyIsICItLXBvcmNlbGFpbiIsICItLXVudHJhY2tlZC1maWxlcz1ub3JtYWwiKQogICAgcmV0dXJu'
        'IHsKICAgICAgICAic2hhMjU2IjogZGlnZXN0X2pzb24oaGFzaGVzKSwKICAgICAgICAiZmlsZXMi'
        'OiBoYXNoZXMsCiAgICAgICAgImhlYWQiOiBnaXQoInJldi1wYXJzZSIsICJIRUFEIiksCiAgICAg'
        'ICAgImRpcnR5IjogTm9uZSBpZiBzdGF0dXMgaXMgTm9uZSBlbHNlIGJvb2woc3RhdHVzKSwKICAg'
        'IH0K'
    ),
    'olympus/training/synthetic.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCgoKZGVmIGFt'
        'YmlndW91c19pbnRlcnByZXRhdGlvbl9kYXRhc2V0KCkgLT4gbGlzdFt0dXBsZVtzdHIsIHN0cl1d'
        'OgogICAgcmV0dXJuIFsKICAgICAgICAoIlRoZSBjbG91ZCB3YXMgYSBkcmFnb24gYWJvdmUgdGhl'
        'IGNpdHkuIiwgImFuYWxvZ3kiKSwKICAgICAgICAoIlRoZSBsb2cgbGluZSBtZWFucyB0aGUgc2Vy'
        'dmljZSByZXN0YXJ0ZWQuIiwgImxpdGVyYWwiKSwKICAgICAgICAoIlRoZSBtYXJrZXQgYnJlYXRo'
        'ZWQgYmVmb3JlIHRoZSBjcmFzaC4iLCAic3ltYm9saWMiKSwKICAgIF0KCgpkZWYgdHJhbnNmb3Jt'
        'X3NpZ25hbHMoKSAtPiB0b3JjaC5UZW5zb3I6CiAgICB0aW1lID0gdG9yY2gubGluc3BhY2UoMC4w'
        'LCA2LjI4LCBzdGVwcz02NCkKICAgIHN0YWNrZWQgPSB0b3JjaC5zdGFjaygKICAgICAgICBbCiAg'
        'ICAgICAgICAgIHRvcmNoLnNpbih0aW1lKSwKICAgICAgICAgICAgdG9yY2guY29zKHRpbWUpLAog'
        'ICAgICAgICAgICB0b3JjaC5zaW4odGltZSAqIDIuMCkgKiAwLjUsCiAgICAgICAgXSwKICAgICAg'
        'ICBkaW09MSwKICAgICkKICAgIHJldHVybiBzdGFja2VkCg=='
    ),
    'olympus/training/trainer.py': (
        'ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBv'
        'cnQgZGF0YWNsYXNzCgpmcm9tIG9seW1wdXMuY29yZS5wcmVkaWN0aW9uIGltcG9ydCBKRVBBVHJh'
        'aW5pbmdSZXN1bHQsIHRyYWluX2plcGEKZnJvbSBvbHltcHVzLmNvcmUudHJhbnNmb3JtIGltcG9y'
        'dCBUcmFuc2Zvcm1NZXRyaWNzLCB0cmFpbl9sZWFybmVkX3RyYW5zZm9ybQpmcm9tIG9seW1wdXMu'
        'dHJhaW5pbmcuc3ludGhldGljIGltcG9ydCB0cmFuc2Zvcm1fc2lnbmFscwoKCkBkYXRhY2xhc3Mo'
        'c2xvdHM9VHJ1ZSkKY2xhc3MgVHJhaW5pbmdSZXBvcnQ6CiAgICBzZWVkOiBpbnQKICAgIGplcGE6'
        'IEpFUEFUcmFpbmluZ1Jlc3VsdAogICAgdHJhbnNmb3JtOiBUcmFuc2Zvcm1NZXRyaWNzCgoKY2xh'
        'c3MgT2x5bXB1c1RyYWluZXI6CiAgICBkZWYgcnVuKHNlbGYsIHNlZWQ6IGludCA9IDcpIC0+IFRy'
        'YWluaW5nUmVwb3J0OgogICAgICAgIGplcGEgPSB0cmFpbl9qZXBhKHNlZWQ9c2VlZCkKICAgICAg'
        'ICB0cmFuc2Zvcm0gPSB0cmFpbl9sZWFybmVkX3RyYW5zZm9ybSh0cmFuc2Zvcm1fc2lnbmFscygp'
        'LCBzZWVkPXNlZWQpCiAgICAgICAgcmV0dXJuIFRyYWluaW5nUmVwb3J0KHNlZWQ9c2VlZCwgamVw'
        'YT1qZXBhLCB0cmFuc2Zvcm09dHJhbnNmb3JtKQo='
    ),
    'olympus/upscaler/__init__.py': (
        'IiIiSkVQQS1iYXNlZCBzdXBlci1yZXNvbHV0aW9uIHJlc2VhcmNoIGNvbXBvbmVudHMuIiIiCgpm'
        'cm9tIG9seW1wdXMudXBzY2FsZXIuamVwYSBpbXBvcnQgKAogICAgSkVQQVVwc2NhbGVyLAogICAg'
        'SkVQQVVwc2NhbGVyQ29uZmlnLAogICAgSkVQQVVwc2NhbGVyTG9zcywKICAgIEpFUEFVcHNjYWxl'
        'ck91dHB1dCwKICAgIGplcGFfdXBzY2FsZXJfbG9zcywKICAgIHVwc2NhbGVfc2VsZl9lbnNlbWJs'
        'ZSwKICAgIHVwc2NhbGVfdGlsZWQsCikKZnJvbSBvbHltcHVzLnVwc2NhbGVyLnRyYWluaW5nIGlt'
        'cG9ydCBsb2FkX2plcGFfdXBzY2FsZXJfY2hlY2twb2ludAoKX19hbGxfXyA9IFsKICAgICJKRVBB'
        'VXBzY2FsZXIiLAogICAgIkpFUEFVcHNjYWxlckNvbmZpZyIsCiAgICAiSkVQQVVwc2NhbGVyTG9z'
        'cyIsCiAgICAiSkVQQVVwc2NhbGVyT3V0cHV0IiwKICAgICJqZXBhX3Vwc2NhbGVyX2xvc3MiLAog'
        'ICAgImxvYWRfamVwYV91cHNjYWxlcl9jaGVja3BvaW50IiwKICAgICJ1cHNjYWxlX3NlbGZfZW5z'
        'ZW1ibGUiLAogICAgInVwc2NhbGVfdGlsZWQiLApdCg=='
    ),
    'olympus/upscaler/jepa.py': (
        '...'
    ),
    'olympus/upscaler/training.py': (
        '...'
    ),
    'research/olympus_routing_v6/protocol.json': (
        'ewogICJpZCI6ICJvbHltcHVzLXJvdXRpbmctdjYuMC4yIiwKICAic2NvcGUiOiAiQm91bmRlZCBz'
        'eW50aGV0aWMgY29uc2VydmF0aXZlIHN0YWNraW5nIHN0dWR5OyBubyB0YXJnZXQtc2NhbGUgb3Ig'
        'ZnVsbC1mYW1pbHkgY2FwYWJpbGl0eSBjbGFpbSIsCiAgImh5cG90aGVzaXMiOiAiRXhhbXBsZS1k'
        'ZXBlbmRlbnQgYW5jaG9yZWQgY29ycmVjdGlvbiB3aXRoIHdvcnN0LWdyb3VwIHRyYWluaW5nIGlt'
        'cHJvdmVzIGZyZXNoIGNvbXBvc2l0ZS1zaGlmdCBhY2N1cmFjeSBieSBhdCBsZWFzdCAyIHBwIG92'
        'ZXIgdGhlIGRldmVsb3BtZW50LXNlbGVjdGVkIHN0cm9uZ2VzdCBjb250cm9sIHdpdGhvdXQgZGVj'
        'bGFyZWQgcmVnaW1lIHJlZ3Jlc3Npb25zLiIsCiAgImRhdGFzZXRfc2VlZHMiOiBbCiAgICAyMDI2'
        'NzEwMCwKICAgIDIwMjY3MjAwLAogICAgMjAyNjczMDAsCiAgICAyMDI2NzQwMCwKICAgIDIwMjY3'
        'NTAwCiAgXSwKICAidHJhaW5pbmdfc2VlZHMiOiBbCiAgICAyMDI2NzYwMSwKICAgIDIwMjY3NjAy'
        'CiAgXSwKICAiYmFua19leGFtcGxlcyI6IDEwMjQsCiAgIm1ldGFfZXhhbXBsZXNfcGVyX3JlZ2lt'
        'ZSI6IDI1NiwKICAidGVzdF9leGFtcGxlc19wZXJfcmVnaW1lIjogMTAyNCwKICAiYmFua19zdGVw'
        'cyI6IDI1MCwKICAicmVmZXJlbmNlX3N0ZXBzIjogMzAwLAogICJjb3JyZWN0aW9uX3N0ZXBzIjog'
        'MzAwLAogICJiYXRjaF9wZXJfZ3JvdXAiOiAzMiwKICAibHIiOiAwLjAwMSwKICAid2VpZ2h0X2Rl'
        'Y2F5IjogMC4wMSwKICAiZ3JhZGllbnRfY2xpcCI6IDEuMCwKICAiYW5jaG9yX3BlbmFsdHkiOiAw'
        'LjAxLAogICJhbHBoYV9pbml0aWFsIjogMC4xLAogICJncm91cF9vYmplY3RpdmUiOiAibWF4aW11'
        'bSBvZiB0aHJlZSBlcXVhbGx5IHNpemVkIGdyb3VwIG1lYW4gTkxMcyBwZXIgdXBkYXRlOyBkZXRl'
        'cm1pbmlzdGljIHRvcmNoLm1heCBzdWJncmFkaWVudDsgbm90IGFkYXB0aXZlLXdlaWdodCBncm91'
        'cCBEUk8iLAogICJkZXZlbG9wbWVudF9yZWdpbWVzIjogWwogICAgImNsZWFuIiwKICAgICJjb3Jy'
        'dXB0ZWQiLAogICAgImtub3duX3NoaWZ0IgogIF0sCiAgInByb3RlY3RlZF9yZWdpbWVzIjogWwog'
        'ICAgImNsZWFuIiwKICAgICJjb3JydXB0ZWQiLAogICAgImtub3duX3NoaWZ0IiwKICAgICJjb21w'
        'b3NpdGVfc2hpZnQiCiAgXSwKICAic3BsaXRfb2Zmc2V0cyI6IHsKICAgICJiYW5rIjogMSwKICAg'
        'ICJyZWZlcmVuY2VfZml0IjogMTAsCiAgICAicmVmZXJlbmNlX3NlbGVjdCI6IDIwLAogICAgImNv'
        'cnJlY3Rpb25fZml0IjogMzAsCiAgICAic2VsZWN0aW9uIjogNDAsCiAgICAicHJvdGVjdGVkIjog'
        'NjAKICB9LAogICJ2YXJpYW50cyI6IFsKICAgICJkaXJlY3RfZXJtIiwKICAgICJkaXJlY3Rfcm9i'
        'dXN0IiwKICAgICJhbmNob3JlZF9lcm0iLAogICAgImFuY2hvcmVkX3JvYnVzdCIsCiAgICAiY29u'
        'c3RhbnRfZXJtIiwKICAgICJjb25zdGFudF9yb2J1c3QiLAogICAgImFuY2hvcmVkX3JvYnVzdF9u'
        'b19wZW5hbHR5IgogIF0sCiAgInByaW1hcnlfY2FuZGlkYXRlIjogImFuY2hvcmVkX3JvYnVzdCIs'
        'CiAgImNvbnRyb2xfY2FuZGlkYXRlcyI6IFsKICAgICJ1bmlmb3JtIiwKICAgICJmaXhlZCIsCiAg'
        'ICAic2VtYW50aWMiLAogICAgImRpcmVjdF9lcm0iLAogICAgImRpcmVjdF9yb2J1c3QiLAogICAg'
        'ImNvbnN0YW50X3JvYnVzdCIsCiAgICAiY29uc3RhbnRfZXJtIiwKICAgICJleHBlcnRfMCIsCiAg'
        'ICAiZXhwZXJ0XzEiLAogICAgImV4cGVydF8yIiwKICAgICJleHBlcnRfMyIsCiAgICAiZXhwZXJ0'
        'XzQiCiAgXSwKICAic2VsZWN0aW9uIjogIm1heGltaXplIG1pbmltdW0gcmVnaW1lIGFjY3VyYWN5'
        'LCB0aGVuIG1lYW4gYWNjdXJhY3ksIHRoZW4gZGVjbGFyZWQgbGlzdCBvcmRlciIsCiAgInByaW1h'
        'cnlfcmVnaW1lIjogImNvbXBvc2l0ZV9zaGlmdCIsCiAgIm1pbmltdW1fZWZmZWN0IjogMC4wMiwK'
        'ICAibm9ucmVncmVzc2lvbl9tZWFuIjogLTAuMDEsCiAgIm5vbnJlZ3Jlc3Npb25faW50ZXJ2YWxf'
        'bG93ZXIiOiAtMC4wMiwKICAiYm9vdHN0cmFwX3NhbXBsZXMiOiAxMDAwMCwKICAiYm9vdHN0cmFw'
        'X3NlZWQiOiAyMDI2Nzk5OSwKICAiaW50ZXJ2YWwiOiAicGFpcmVkIGRhdGFzZXQtY2x1c3RlciBw'
        'ZXJjZW50aWxlIDk1JTsgZml2ZSBpbmRlcGVuZGVudCBkYXRhc2V0cywgdHdvIGluaXRpYWxpemF0'
        'aW9ucyBhdmVyYWdlZDsgZGVzY3JpcHRpdmUgcGlsb3Qgb25seSIsCiAgIm1heF9kZXZlbG9wbWVu'
        'dF9yZXZpc2lvbnMiOiAwLAogICJtYXhfY2FtcGFpZ25fc2Vjb25kcyI6IDM2MDAsCiAgIm1heF9x'
        'dWFsaWZpY2F0aW9uX3NlY29uZHMiOiAxMjAsCiAgIm1pbmltdW1fYXZhaWxhYmxlX2J5dGVzIjog'
        'MjE0NzQ4MzY0OCwKICAibWF4aW11bV9zd2FwX2ZyYWN0aW9uIjogMC45LAogICJsb2NhbF9mcmVl'
        'X2J5dGVzX2Zsb29yIjogNTM2ODcwOTEyLAogICJwYWlkX2pvYnMiOiAwLAogICJleHRlcm5hbF9k'
        'b3dubG9hZHMiOiAwLAogICJkZXZpY2UiOiAiQ1BVIiwKICAidGhyZWFkcyI6IDEsCiAgInN0b3Bw'
        'aW5nIjogIk9uZSBmcm96ZW4gY2FtcGFpZ247IHJlc291cmNlL3RpbWUvbm9uZmluaXRlL3Byb3Zl'
        'bmFuY2UgZmFpbHVyZXMgc3RvcCBhbmQgcHJlc2VydmUgcGFydGlhbCBhcnRpZmFjdHMuIE5vIHJl'
        'cGVhdGVkIGNvbmZpcm1hdG9yeSBzZWFyY2ggZm9yIGEgcG9zaXRpdmUgcmVzdWx0LiIsCiAgImNs'
        'YWltX2ZhbHNpZmllciI6ICJQcmltYXJ5IHByYWN0aWNhbCBlZmZlY3Qgb3IgaW50ZXJ2YWwgY3Jp'
        'dGVyaW9uIGZhaWxzLCBvciBhbnkgY2xlYW4vY29ycnVwdGVkL2tub3duLXNoaWZ0IG5vbnJlZ3Jl'
        'c3Npb24gZ2F0ZSBmYWlscy4gSW52YWxpZCBwcm92ZW5hbmNlL251bWVyaWNzIHlpZWxkcyBJTlZB'
        'TElELCBhYnNlbnQgZXhlY3V0aW9uIHlpZWxkcyBOT1RfRVZBTFVBVEVELiIsCiAgIm5vbnJlZ3Jl'
        'c3Npb25fY29tcGFyYXRvcnMiOiBbCiAgICAicmVmZXJlbmNlIiwKICAgICJzZWxlY3RlZF9jb250'
        'cm9sIgogIF0sCiAgImFtZW5kbWVudCI6ICJQcmUtZXhlY3V0aW9uIHNvdXJjZSByZXZpZXcgYWRk'
        'ZWQgZXhwZXJ0LW9ubHkgY29udHJvbHMsIGV4cGxpY2l0IGJhbmsgdHJhaW5pbmcgaHlwZXJwYXJh'
        'bWV0ZXJzIGFuZCBjb3JyZWN0ZWQgaGVsZC1vdXQtc2hpZnQgdGVybWlub2xvZ3kuIE5vIHNjaWVu'
        'dGlmaWMgb3V0Y29tZXMgb2JzZXJ2ZWQ7IGVhcmxpZXIgcHJvdG9jb2wgdmVyc2lvbnMgcmV0YWlu'
        'ZWQuIiwKICAiYmFua19iYXRjaF9zaXplIjogNjQsCiAgImJhbmtfdGVtcG9yYWxfd2VpZ2h0Ijog'
        'MC4yNSwKICAicmVmZXJlbmNlX2NhbmRpZGF0ZXMiOiBbCiAgICAidW5pZm9ybSIsCiAgICAiZml4'
        'ZWQiLAogICAgInNlbWFudGljIiwKICAgICJleHBlcnRfMCIsCiAgICAiZXhwZXJ0XzEiLAogICAg'
        'ImV4cGVydF8yIiwKICAgICJleHBlcnRfMyIsCiAgICAiZXhwZXJ0XzQiCiAgXSwKICAic2hpZnRf'
        'Ym91bmRhcnkiOiAiY29tcG9zaXRlX3NoaWZ0IGlzIGEgcmVzZXJ2ZWQgY29uZmlndXJhdGlvbiBv'
        'ZiBwcmV2aW91c2x5IG9ic2VydmVkIHJlbG9jYXRpb24sIG9jY2x1c2lvbiBhbmQgYWZmaW5lLXN0'
        'YXRlIHBlcnR1cmJhdGlvbiB0eXBlczsgb25lIGRhbWFnZWQgbW9kYWxpdHkgcGVyIGV4YW1wbGUs'
        'IG5vdCBhIG5vdmVsIG1lY2hhbmlzbSBvciBuYXR1cmFsLWRhdGEgT09ELiIKfQo='
    ),
    'scripts/__init__.py': (
        'IiIiUmVwb3NpdG9yeSBtYWludGVuYW5jZSBhbmQgcmVzZWFyY2ggZW50cnkgcG9pbnRzLiIiIgo='
    ),
    'scripts/qualify_flagship_v2.py': (
        'IiIiUnVuIGJvdW5kZWQgYXJ0aWZpY2lhbCBmbGFnc2hpcCBxdWFsaWZpY2F0aW9uIGFmdGVyIGhv'
        'c3QgYWRtaXNzaW9uLgoKVXNhZ2U6IHB5dGhvbiAtbSBzY3JpcHRzLnF1YWxpZnlfZmxhZ3NoaXBf'
        'djIgLS1vdXRwdXQgTkVXX0RJUkVDVE9SWSAtLWRldmljZSBjcHUKVGhlIHNhbWUgY29tbWFuZCB3'
        'b3JrcyBvZmZsaW5lIG9uIEthZ2dsZTsgdXNlIGN1ZGEgb25seSBmb3IgYW4gYWxsb2NhdGVkIEdQ'
        'VS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFy'
        'c2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgcmUKaW1wb3J0'
        'IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBh'
        'dGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJvbSBvbHltcHVzLmZvdW5kcnkucmVzb3VyY2Vz'
        'IGltcG9ydCBSZXNvdXJjZUdvdmVybm9yCmZyb20gb2x5bXB1cy50cmFpbmluZy5wcm92ZW5hbmNl'
        'IGltcG9ydCBhdG9taWNfanNvbiwgZmlsZV9zaGEyNTYsIHNvdXJjZV9pZGVudGl0eQoKCmRlZiBf'
        'bWFjb3Nfdm1fY291bnRlcnMoKSAtPiB0dXBsZVtpbnQsIGludF06CiAgICByZXN1bHQgPSBzdWJw'
        'cm9jZXNzLnJ1bigKICAgICAgICBbInZtX3N0YXQiXSwgY2hlY2s9VHJ1ZSwgY2FwdHVyZV9vdXRw'
        'dXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTUKICAgICkuc3Rkb3V0CiAgICBwYWdlX3NpemUg'
        'PSByZS5zZWFyY2gociJwYWdlIHNpemUgb2YgKFxkKykgYnl0ZXMiLCByZXN1bHQpCiAgICBmcmVl'
        'ID0gcmUuc2VhcmNoKHIiXlBhZ2VzIGZyZWU6XHMrKFxkKylcLiIsIHJlc3VsdCwgcmUuTVVMVElM'
        'SU5FKQogICAgcGFnZW91dHMgPSByZS5zZWFyY2gociJeUGFnZW91dHM6XHMrKFxkKylcLiIsIHJl'
        'c3VsdCwgcmUuTVVMVElMSU5FKQogICAgaWYgcGFnZV9zaXplIGlzIE5vbmUgb3IgZnJlZSBpcyBO'
        'b25lIG9yIHBhZ2VvdXRzIGlzIE5vbmU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJjYW5u'
        'b3QgaW5zcGVjdCBtYWNPUyBmcmVlIHBhZ2VzIGFuZCBwYWdlb3V0cyIpCiAgICByZXR1cm4gaW50'
        'KHBhZ2Vfc2l6ZVsxXSkgKiBpbnQoZnJlZVsxXSksIGludChwYWdlb3V0c1sxXSkKCgpkZWYgbG9j'
        'YWxfdGVuc29yX2FkbWlzc2lvbigpIC0+IGRpY3Rbc3RyLCBvYmplY3RdOgogICAgIiIiUmV0YWlu'
        'IHRoZSBsb2NhbCA1MTIgTWlCL3N0YWJsZS1wYWdlb3V0IGZsb29yIGJlZm9yZSBsb2FkaW5nIFRv'
        'cmNoLgoKICAgIFRoaXMgaXMgYWRkaXRpb25hbCB0byAobm90IGEgcmVwbGFjZW1lbnQgZm9yKSBS'
        'ZXNvdXJjZUdvdmVybm9yJ3MgMiBHaUIKICAgIGF2YWlsYWJsZS1tZW1vcnkgYW5kIDkwIHBlcmNl'
        'bnQgc3dhcCBjZWlsaW5ncy4gTGludXggdXNlcyB0aGF0IGdvdmVybm9yJ3MKICAgIE1lbUF2YWls'
        'YWJsZSBhZG1pc3Npb247IHJhdyBtYWNPUyBmcmVlIHBhZ2VzIGFyZSBub3QgaW50ZXJjaGFuZ2Vh'
        'YmxlIHdpdGggaXQuCiAgICAiIiIKICAgIGlmIHBsYXRmb3JtLnN5c3RlbSgpICE9ICJEYXJ3aW4i'
        'OgogICAgICAgIHJldHVybiB7InBsYXRmb3JtIjogcGxhdGZvcm0uc3lzdGVtKCksICJleHRyYV9t'
        'YWNvc19ndWFyZCI6ICJOT1RfQVBQTElDQUJMRSJ9CiAgICBmaXJzdCA9IF9tYWNvc192bV9jb3Vu'
        'dGVycygpCiAgICB0aW1lLnNsZWVwKDEpCiAgICBzZWNvbmQgPSBfbWFjb3Nfdm1fY291bnRlcnMo'
        'KQogICAgaWYgbWluKGZpcnN0WzBdLCBzZWNvbmRbMF0pIDwgNTEyICogMTAyNCoqMiBvciBzZWNv'
        'bmRbMV0gPiBmaXJzdFsxXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgICAg'
        'IGYibG9jYWwgdGVuc29yIGFkbWlzc2lvbiByZWZ1c2VkOiBmcmVlIGJ5dGVzIHtmaXJzdFswXX0s'
        'IHtzZWNvbmRbMF19OyAiCiAgICAgICAgICAgIGYicGFnZW91dHMge2ZpcnN0WzFdfSwge3NlY29u'
        'ZFsxXX07IHJlcXVpcmUgPj01MTIgTWlCIGFuZCBubyByaXNpbmcgcGFnZW91dHMiCiAgICAgICAg'
        'KQogICAgcmV0dXJuIHsKICAgICAgICAicGxhdGZvcm0iOiAiRGFyd2luIiwKICAgICAgICAiZnJl'
        'ZV9ieXRlcyI6IFtmaXJzdFswXSwgc2Vjb25kWzBdXSwKICAgICAgICAicGFnZW91dHMiOiBsaXN0'
        'KChmaXJzdFsxXSwgc2Vjb25kWzFdKSksCiAgICB9CgoKZGVmIHF1YWxpZnkob3V0cHV0OiBQYXRo'
        'LCAqLCBkZXZpY2U6IHN0ciA9ICJjcHUiLCB3YWxsX3NlY29uZHM6IGludCA9IDEyMCkgLT4gZGlj'
        'dFtzdHIsIEFueV06CiAgICBpZiBkZXZpY2Ugbm90IGluIHsiY3B1IiwgImN1ZGEifSBvciBub3Qg'
        'MTAgPD0gd2FsbF9zZWNvbmRzIDw9IDEyMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJkZXZp'
        'Y2UgbXVzdCBiZSBjcHUvY3VkYSBhbmQgd2FsbCBidWRnZXQgbXVzdCBiZSAxMC4uMTIwIHNlY29u'
        'ZHMiKQogICAgb3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9RmFsc2UpCiAgICBz'
        'dGFydGVkID0gdGltZS5tb25vdG9uaWMoKQogICAgcm9vdCA9IFBhdGgoX19maWxlX18pLnJlc29s'
        'dmUoKS5wYXJlbnRzWzFdCiAgICBwcm90b2NvbCA9IHJvb3QgLyAicmVzZWFyY2gvZmxhZ3NoaXBf'
        'ZW5naW5lZXJpbmdfdjIvUFJPVE9DT0wubWQiCiAgICBzb3VyY2UgPSBzb3VyY2VfaWRlbnRpdHko'
        'KQogICAgcmVjZWlwdDogZGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgInNjaGVtYV92ZXJzaW9u'
        'IjogMSwKICAgICAgICAic3R1ZHlfaWQiOiAiZmxhZ3NoaXAtZW5naW5lZXJpbmctdjIiLAogICAg'
        'ICAgICJzdGF0dXMiOiAiU1RBUlRFRCIsCiAgICAgICAgImRldmljZV9yZXF1ZXN0ZWQiOiBkZXZp'
        'Y2UsCiAgICAgICAgInNvdXJjZSI6IHNvdXJjZSwKICAgICAgICAicHJvdG9jb2xfc2hhMjU2Ijog'
        'ZmlsZV9zaGEyNTYocHJvdG9jb2wpLAogICAgICAgICJyZWdyZXNzaW9uX3NvdXJjZV9zaGEyNTYi'
        'OiBmaWxlX3NoYTI1Nihyb290IC8gInRlc3RzL3Rlc3RfZmxhZ3NoaXBfdjIucHkiKSwKICAgICAg'
        'ICAibWF4X3dhbGxfc2Vjb25kcyI6IHdhbGxfc2Vjb25kcywKICAgICAgICAic2NpZW50aWZpY19l'
        'dmFsdWF0aW9uIjogRmFsc2UsCiAgICAgICAgInByb21vdGlvbl9hdXRob3JpemVkIjogRmFsc2Us'
        'CiAgICAgICAgInRhcmdldF9zY2FsZV90cmFpbmluZyI6IEZhbHNlLAogICAgfQogICAgYXRvbWlj'
        'X2pzb24ob3V0cHV0IC8gInJlY2VpcHQuanNvbiIsIHJlY2VpcHQpCiAgICB0cnk6CiAgICAgICAg'
        'cmVjZWlwdFsibG9jYWxfdGVuc29yX2FkbWlzc2lvbiJdID0gbG9jYWxfdGVuc29yX2FkbWlzc2lv'
        'bigpCiAgICAgICAgd2l0aCBSZXNvdXJjZUdvdmVybm9yKHJvb3QgLyAiYXJ0aWZhY3RzL2xvY2tz'
        'L2ZsYWdzaGlwLXF1YWxpZmljYXRpb24ubG9jayIpIGFzIGdvdmVybm9yOgogICAgICAgICAgICBy'
        'ZWNlaXB0WyJyZXNvdXJjZV9wcmVmbGlnaHQiXSA9ICgKICAgICAgICAgICAgICAgIGdvdmVybm9y'
        'LnByZWZsaWdodC5hc19kaWN0KCkgaWYgZ292ZXJub3IucHJlZmxpZ2h0IGVsc2UgTm9uZQogICAg'
        'ICAgICAgICApCiAgICAgICAgICAgIG9zLmVudmlyb24uc2V0ZGVmYXVsdCgiQ1VCTEFTX1dPUktT'
        'UEFDRV9DT05GSUciLCAiOjQwOTY6OCIpCiAgICAgICAgICAgIG9zLmVudmlyb25bIk9NUF9OVU1f'
        'VEhSRUFEUyJdID0gIjEiCiAgICAgICAgICAgIG9zLmVudmlyb25bIk1LTF9OVU1fVEhSRUFEUyJd'
        'ID0gIjEiCiAgICAgICAgICAgIGltcG9ydCB0b3JjaAoKICAgICAgICAgICAgaWYgZGV2aWNlID09'
        'ICJjdWRhIiBhbmQgbm90IHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgICAg'
        'ICByYWlzZSBSdW50aW1lRXJyb3IoIkNVREEgcmVxdWVzdGVkIGJ1dCB1bmF2YWlsYWJsZTsgQ1BV'
        'IGZhbGxiYWNrIGlzIGZvcmJpZGRlbiIpCiAgICAgICAgICAgIHJlY2VpcHRbInJ1bnRpbWUiXSA9'
        'IHsKICAgICAgICAgICAgICAgICJweXRob24iOiBzeXMudmVyc2lvbiwKICAgICAgICAgICAgICAg'
        'ICJ0b3JjaCI6IHN0cih0b3JjaC5fX3ZlcnNpb25fXyksCiAgICAgICAgICAgICAgICAiY3VkYSI6'
        'IHRvcmNoLnZlcnNpb24uY3VkYSwKICAgICAgICAgICAgICAgICJkZXZpY2UiOiBkZXZpY2UsCiAg'
        'ICAgICAgICAgICAgICAiZGV2aWNlX25hbWUiOiB0b3JjaC5jdWRhLmdldF9kZXZpY2VfbmFtZSgw'
        'KSBpZiBkZXZpY2UgPT0gImN1ZGEiIGVsc2UgIkNQVSIsCiAgICAgICAgICAgIH0KICAgICAgICAg'
        'ICAgdGVzdHMgPSBbc3RyKHJvb3QgLyAidGVzdHMvdGVzdF9mbGFnc2hpcF92Mi5weSIpXQogICAg'
        'ICAgICAgICBpZiBkZXZpY2UgPT0gImNwdSI6CiAgICAgICAgICAgICAgICB0ZXN0cyArPSBbIi1r'
        'IiwgIm5vdCBjdWRhIl0KICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcmVzdWx0ID0g'
        'c3VicHJvY2Vzcy5ydW4oCiAgICAgICAgICAgICAgICAgICAgWwogICAgICAgICAgICAgICAgICAg'
        'ICAgICBzeXMuZXhlY3V0YWJsZSwKICAgICAgICAgICAgICAgICAgICAgICAgIi1tIiwKICAgICAg'
        'ICAgICAgICAgICAgICAgICAgInB5dGVzdCIsCiAgICAgICAgICAgICAgICAgICAgICAgICItcSIs'
        'CiAgICAgICAgICAgICAgICAgICAgICAgICp0ZXN0cywKICAgICAgICAgICAgICAgICAgICAgICAg'
        'Ii0tanVuaXR4bWwiLAogICAgICAgICAgICAgICAgICAgICAgICBzdHIob3V0cHV0LnJlc29sdmUo'
        'KSAvICJyZWdyZXNzaW9ucy54bWwiKSwKICAgICAgICAgICAgICAgICAgICBdLAogICAgICAgICAg'
        'ICAgICAgICAgIGN3ZD1yb290LAogICAgICAgICAgICAgICAgICAgIGVudj1kaWN0KG9zLmVudmly'
        'b24sIFBZVEVTVF9ESVNBQkxFX1BMVUdJTl9BVVRPTE9BRD0iMSIpLAogICAgICAgICAgICAgICAg'
        'ICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsCiAgICAgICAgICAgICAgICAgICAgdGV4dD1UcnVlLAog'
        'ICAgICAgICAgICAgICAgICAgIHRpbWVvdXQ9bWF4KDEsIHdhbGxfc2Vjb25kcyAtICh0aW1lLm1v'
        'bm90b25pYygpIC0gc3RhcnRlZCkpLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBleGNl'
        'cHQgc3VicHJvY2Vzcy5UaW1lb3V0RXhwaXJlZCBhcyBlcnJvcjoKICAgICAgICAgICAgICAgIHN0'
        'ZG91dCA9IGVycm9yLnN0ZG91dCBvciBiIiIKICAgICAgICAgICAgICAgIHN0ZGVyciA9IGVycm9y'
        'LnN0ZGVyciBvciBiIiIKICAgICAgICAgICAgICAgIChvdXRwdXQgLyAicmVncmVzc2lvbnMubG9n'
        'Iikud3JpdGVfYnl0ZXMoCiAgICAgICAgICAgICAgICAgICAgKHN0ZG91dC5lbmNvZGUoKSBpZiBp'
        'c2luc3RhbmNlKHN0ZG91dCwgc3RyKSBlbHNlIHN0ZG91dCkKICAgICAgICAgICAgICAgICAgICAr'
        'IChzdGRlcnIuZW5jb2RlKCkgaWYgaXNpbnN0YW5jZShzdGRlcnIsIHN0cikgZWxzZSBzdGRlcnIp'
        'CiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICByYWlzZQogICAgICAgICAgICAob3V0'
        'cHV0IC8gInJlZ3Jlc3Npb25zLmxvZyIpLndyaXRlX3RleHQocmVzdWx0LnN0ZG91dCArIHJlc3Vs'
        'dC5zdGRlcnIpCiAgICAgICAgICAgIGlmIHJlc3VsdC5yZXR1cm5jb2RlOgogICAgICAgICAgICAg'
        'ICAgcmFpc2UgUnVudGltZUVycm9yKGYiZmxhZ3NoaXAgcmVncmVzc2lvbnMgZmFpbGVkIHdpdGgg'
        'ZXhpdCB7cmVzdWx0LnJldHVybmNvZGV9IikKICAgICAgICAgICAgIyBBcnRpZmljaWFsIG9wdGlt'
        'aXplciBxdWFsaWZpY2F0aW9uLCBubyBwcm90ZWN0ZWQgZGF0YXNldCBvciBiZW5jaG1hcmsuCiAg'
        'ICAgICAgICAgIGZyb20gc2NyaXB0cy5mbGFnc2hpcF9zbW9rZSBpbXBvcnQgdHJhaW5fc21va2UK'
        'CiAgICAgICAgICAgIHJlY2VpcHRbIm9wdGltaXplcl9zbW9rZSJdID0gdHJhaW5fc21va2UoCiAg'
        'ICAgICAgICAgICAgICBvdXRwdXQgLyAic21va2UucHQiLAogICAgICAgICAgICAgICAgZGV2aWNl'
        'PWRldmljZSwKICAgICAgICAgICAgICAgIGRlYWRsaW5lPXN0YXJ0ZWQgKyB3YWxsX3NlY29uZHMs'
        'CiAgICAgICAgICAgICkKICAgICAgICAgICAgaWYgdGltZS5tb25vdG9uaWMoKSAtIHN0YXJ0ZWQg'
        'PiB3YWxsX3NlY29uZHM6CiAgICAgICAgICAgICAgICByYWlzZSBUaW1lb3V0RXJyb3IoInF1YWxp'
        'ZmljYXRpb24gZXhjZWVkZWQgdGhlIGZyb3plbiB3YWxsIGJ1ZGdldCIpCiAgICAgICAgICAgIHJl'
        'Y2VpcHRbInN0YXR1cyJdID0gIkVOR0lORUVSSU5HX1FVQUxJRklFRCIKICAgIGV4Y2VwdCBFeGNl'
        'cHRpb24gYXMgZXJyb3I6CiAgICAgICAgcmVjZWlwdFsic3RhdHVzIl0gPSAiRkFJTEVEX09SX1JF'
        'U09VUkNFX0JMT0NLRUQiCiAgICAgICAgcmVjZWlwdFsiZXJyb3IiXSA9IGYie3R5cGUoZXJyb3Ip'
        'Ll9fbmFtZV9ffToge2Vycm9yfSIKICAgICAgICByYWlzZQogICAgZmluYWxseToKICAgICAgICBy'
        'ZWNlaXB0WyJlbGFwc2VkX3NlY29uZHMiXSA9IHRpbWUubW9ub3RvbmljKCkgLSBzdGFydGVkCiAg'
        'ICAgICAgcmVjZWlwdFsiYXJ0aWZhY3RzIl0gPSB7CiAgICAgICAgICAgIHN0cihwYXRoLnJlbGF0'
        'aXZlX3RvKG91dHB1dCkpOiBmaWxlX3NoYTI1NihwYXRoKQogICAgICAgICAgICBmb3IgcGF0aCBp'
        'biBzb3J0ZWQob3V0cHV0LnJnbG9iKCIqIikpCiAgICAgICAgICAgIGlmIHBhdGguaXNfZmlsZSgp'
        'IGFuZCBwYXRoLm5hbWUgIT0gInJlY2VpcHQuanNvbiIKICAgICAgICB9CiAgICAgICAgYXRvbWlj'
        'X2pzb24ob3V0cHV0IC8gInJlY2VpcHQuanNvbiIsIHJlY2VpcHQpCiAgICByZXR1cm4gcmVjZWlw'
        'dAoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFy'
        'c2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1'
        'dCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0t'
        'ZGV2aWNlIiwgY2hvaWNlcz0oImNwdSIsICJjdWRhIiksIGRlZmF1bHQ9ImNwdSIpCiAgICBhcmdz'
        'ID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoanNvbi5kdW1wcyhxdWFsaWZ5KGFyZ3Mu'
        'b3V0cHV0LCBkZXZpY2U9YXJncy5kZXZpY2UpLCBpbmRlbnQ9MikpCgoKaWYgX19uYW1lX18gPT0g'
        'Il9fbWFpbl9fIjoKICAgIG1haW4oKQo='
    ),
    'scripts/run_conservative_routing_v6.py': (
        '...'
    ),
    'scripts/run_oracle_fusion_v4.py': (
        '...'
    ),
    'tests/__init__.py': (
        'IiIiT2x5bXB1cyByZWdyZXNzaW9uIHRlc3RzLiIiIgo='
    ),
    'tests/test_conservative_routing_v6.py': (
        'IiIiQXJ0aWZpY2lhbCBlcXVhdGlvbiwgZ3JhZGllbnQsIGRpc2pvaW50bmVzcyBhbmQgZnVsbCBs'
        'aWZlY3ljbGUgcmVncmVzc2lvbnMuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRp'
        'b25zCgppbXBvcnQgY29weQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBweXRlc3QK'
        'aW1wb3J0IHRvcmNoCgpmcm9tIG9seW1wdXMubW9kZWxzLmNvbnNlcnZhdGl2ZV9yb3V0aW5nX3Y2'
        'IGltcG9ydCAoCiAgICBDb3JyZWN0aW9uLAogICAgUmVmZXJlbmNlUm91dGVzLAogICAgYmxlbmQs'
        'CiAgICBjb3JyZWN0aW9uX2xvc3MsCiAgICBmZWF0dXJlcywKICAgIGludGVydmVudGlvbl9kaWFn'
        'bm9zdGljcywKKQpmcm9tIHNjcmlwdHMgaW1wb3J0IHJ1bl9jb25zZXJ2YXRpdmVfcm91dGluZ192'
        'NiBhcyBydW5uZXIKZnJvbSBzY3JpcHRzIGltcG9ydCBydW5fb3JhY2xlX2Z1c2lvbl92NCBhcyB2'
        'NAoKCmRlZiBmaXh0dXJlKCkgLT4gdHVwbGVbdG9yY2guVGVuc29yLCB0b3JjaC5UZW5zb3IsIHRv'
        'cmNoLlRlbnNvcl06CiAgICBnZW5lcmF0b3IgPSB0b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2Vl'
        'ZCg5OTEpCiAgICBwID0gdG9yY2gucmFuZG4oMTIsIDUsIDQsIGdlbmVyYXRvcj1nZW5lcmF0b3Ip'
        'LnNvZnRtYXgoLTEpCiAgICB4ID0gZmVhdHVyZXModG9yY2gucmFuZG4oMTIsIDUsIDE2LCBnZW5l'
        'cmF0b3I9Z2VuZXJhdG9yKSwgcCkKICAgIHJldHVybiB4LCBwLCBwLm1lYW4oMSkKCgpkZWYgdGVz'
        'dF9leGFjdF9lbmRwb2ludHNfYW5kX2NvbnZleF9ib3VuZCgpIC0+IE5vbmU6CiAgICBfLCBwLCBy'
        'ZWYgPSBmaXh0dXJlKCkKICAgIGFsdCA9IHBbOiwgMF0KICAgIGFzc2VydCB0b3JjaC5lcXVhbChi'
        'bGVuZChyZWYsIGFsdCwgdG9yY2guemVyb3MoMTIsIDEpKSwgcmVmKQogICAgYXNzZXJ0IHRvcmNo'
        'LmVxdWFsKGJsZW5kKHJlZiwgYWx0LCB0b3JjaC5vbmVzKDEyLCAxKSksIGFsdCkKICAgIGFscGhh'
        'ID0gdG9yY2guZnVsbCgoMTIsIDEpLCAwLjIpCiAgICByZXN1bHQgPSBibGVuZChyZWYsIGFsdCwg'
        'YWxwaGEpCiAgICBhc3NlcnQgdG9yY2guYWxsY2xvc2UocmVzdWx0LnN1bSgtMSksIHRvcmNoLm9u'
        'ZXMoMTIpKQogICAgIyBUb3RhbCB2YXJpYXRpb24gZGlzdGFuY2UgZnJvbSByZWZlcmVuY2UgY2Fu'
        'bm90IGV4Y2VlZCBhbHBoYS4KICAgIGFzc2VydCBib29sKCgocmVzdWx0IC0gcmVmKS5hYnMoKS5z'
        'dW0oLTEpIC8gMiA8PSAwLjIwMDAwMSkuYWxsKCkpCgoKQHB5dGVzdC5tYXJrLnBhcmFtZXRyaXpl'
        'KCJraW5kIiwgWyJkaXJlY3QiLCAiY29uc3RhbnQiLCAiYW5jaG9yZWQiXSkKZGVmIHRlc3RfZ3Jh'
        'ZGllbnRfYW5kX3JlbG9hZChraW5kOiBzdHIpIC0+IE5vbmU6CiAgICB4LCBwLCByZWYgPSBmaXh0'
        'dXJlKCkKICAgIG1vZGVsID0gQ29ycmVjdGlvbihSZWZlcmVuY2VSb3V0ZXMoKS5ldmFsKCkucmVx'
        'dWlyZXNfZ3JhZF8oRmFsc2UpLnNlbWFudGljLCBraW5kKQogICAgcHJlZCwgYWxwaGEgPSBtb2Rl'
        'bCh4LCBwLCByZWYpCiAgICBsb3NzID0gY29ycmVjdGlvbl9sb3NzKAogICAgICAgIHByZWQsIHRv'
        'cmNoLmFyYW5nZSgxMikgJSA0LCB0b3JjaC5hcmFuZ2UoMTIpICUgMywgYWxwaGEsIHJvYnVzdD1U'
        'cnVlLCBwZW5hbHR5PTAuMDEKICAgICkKICAgIHRvcmNoLmF1dG9ncmFkLmJhY2t3YXJkKGxvc3Mp'
        'CiAgICBhc3NlcnQgYWxsKAogICAgICAgIHYuZ3JhZCBpcyBub3QgTm9uZSBhbmQgYm9vbCh0b3Jj'
        'aC5pc2Zpbml0ZSh2LmdyYWQpLmFsbCgpKSBmb3IgdiBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkKICAg'
        'ICkKICAgIG90aGVyID0gY29weS5kZWVwY29weShtb2RlbCkKICAgIG90aGVyLmxvYWRfc3RhdGVf'
        'ZGljdChtb2RlbC5zdGF0ZV9kaWN0KCkpCiAgICBhc3NlcnQgdG9yY2guZXF1YWwob3RoZXIoeCwg'
        'cCwgcmVmKVswXSwgcHJlZCkKCgpkZWYgdGVzdF9jb25zdGFudF9pc19pbnB1dF9pbmRlcGVuZGVu'
        'dF9hbmRfYWRhcHRpdmVfc3RhcnRzX2F0X2RlY2xhcmVkX2FscGhhKCkgLT4gTm9uZToKICAgIHgs'
        'IHAsIHJlZiA9IGZpeHR1cmUoKQogICAgZm9yIGtpbmQgaW4gKCJjb25zdGFudCIsICJhbmNob3Jl'
        'ZCIpOgogICAgICAgIG1vZGVsID0gQ29ycmVjdGlvbihSZWZlcmVuY2VSb3V0ZXMoKS5ldmFsKCku'
        'cmVxdWlyZXNfZ3JhZF8oRmFsc2UpLnNlbWFudGljLCBraW5kKQogICAgICAgIF8sIGFscGhhID0g'
        'bW9kZWwoeCwgcCwgcmVmKQogICAgICAgIGFzc2VydCB0b3JjaC5hbGxjbG9zZShhbHBoYSwgdG9y'
        'Y2guZnVsbF9saWtlKGFscGhhLCAwLjEpKQoKCmRlZiB0ZXN0X2dyb3VwX29iamVjdGl2ZXNfaGFu'
        'ZF9jYWxjdWxhdGlvbigpIC0+IE5vbmU6CiAgICBwcm9iYWJpbGl0aWVzID0gdG9yY2gudGVuc29y'
        'KFtbMC44LCAwLjJdLCBbMC41LCAwLjVdLCBbMC4yNSwgMC43NV1dKQogICAgbGFiZWxzLCBncm91'
        'cHMgPSB0b3JjaC56ZXJvcygzLCBkdHlwZT10b3JjaC5sb25nKSwgdG9yY2guYXJhbmdlKDMpCiAg'
        'ICBhbHBoYSA9IHRvcmNoLnplcm9zKDMsIDEpCiAgICBleHBlY3RlZCA9IC1wcm9iYWJpbGl0aWVz'
        'WzosIDBdLmxvZygpCiAgICBhc3NlcnQgdG9yY2guYWxsY2xvc2UoCiAgICAgICAgY29ycmVjdGlv'
        'bl9sb3NzKHByb2JhYmlsaXRpZXMsIGxhYmVscywgZ3JvdXBzLCBhbHBoYSwgcm9idXN0PUZhbHNl'
        'LCBwZW5hbHR5PTApLAogICAgICAgIGV4cGVjdGVkLm1lYW4oKSwKICAgICkKICAgIGFzc2VydCB0'
        'b3JjaC5hbGxjbG9zZSgKICAgICAgICBjb3JyZWN0aW9uX2xvc3MocHJvYmFiaWxpdGllcywgbGFi'
        'ZWxzLCBncm91cHMsIGFscGhhLCByb2J1c3Q9VHJ1ZSwgcGVuYWx0eT0wKSwKICAgICAgICBleHBl'
        'Y3RlZC5tYXgoKSwKICAgICkKICAgIHdpdGggcHl0ZXN0LnJhaXNlcyhWYWx1ZUVycm9yLCBtYXRj'
        'aD0iZXZlcnkgZGVjbGFyZWQgZ3JvdXAiKToKICAgICAgICBjb3JyZWN0aW9uX2xvc3MoCiAgICAg'
        'ICAgICAgIHByb2JhYmlsaXRpZXMsIGxhYmVscywgdG9yY2guemVyb3NfbGlrZShncm91cHMpLCBh'
        'bHBoYSwgcm9idXN0PVRydWUsIHBlbmFsdHk9MAogICAgICAgICkKCgpAcHl0ZXN0Lm1hcmsucGFy'
        'YW1ldHJpemUoInZhbHVlIiwgWy0wLjEsIDEuMSwgZmxvYXQoIm5hbiIpXSkKZGVmIHRlc3RfaW52'
        'YWxpZF9hbHBoYV9yZWplY3RlZCh2YWx1ZTogZmxvYXQpIC0+IE5vbmU6CiAgICBfLCBwLCByZWYg'
        'PSBmaXh0dXJlKCkKICAgIHdpdGggcHl0ZXN0LnJhaXNlcyhWYWx1ZUVycm9yKToKICAgICAgICBi'
        'bGVuZChyZWYsIHBbOiwgMF0sIHRvcmNoLmZ1bGwoKDEyLCAxKSwgdmFsdWUpKQoKCmRlZiB0ZXN0'
        'X2hhcm1fcmVzY3VlX2lkZW50aXR5KCkgLT4gTm9uZToKICAgIHJlZiA9IHRvcmNoLnRlbnNvcihb'
        'WzAuOSwgMC4xXSwgWzAuMSwgMC45XSwgWzAuMSwgMC45XV0pCiAgICBwcmVkID0gdG9yY2gudGVu'
        'c29yKFtbMC4xLCAwLjldLCBbMC45LCAwLjFdLCBbMC45LCAwLjFdXSkKICAgIHJlc3VsdCA9IGlu'
        'dGVydmVudGlvbl9kaWFnbm9zdGljcyhwcmVkLCByZWYsIHRvcmNoLnplcm9zKDMsIGR0eXBlPXRv'
        'cmNoLmxvbmcpKQogICAgYXNzZXJ0IHJlc3VsdCA9PSB7CiAgICAgICAgImV4YW1wbGVzIjogMywK'
        'ICAgICAgICAicmVzY3VlcyI6IDIsCiAgICAgICAgImhhcm1zIjogMSwKICAgICAgICAiYWNjdXJh'
        'Y3lfZGVsdGEiOiAxIC8gMywKICAgICAgICAicHJlZGljdGlvbl9jaGFuZ2VzIjogMywKICAgIH0K'
        'CgpkZWYgdGVzdF9kaXNqb2ludG5lc3NfcmVqZWN0c19kdXBsaWNhdGVzX2FuZF9vdmVybGFwKCkg'
        'LT4gTm9uZToKICAgIHJ1bm5lci5lbnN1cmVfZGlzam9pbnQoeyJhIjogWzEsIDJdLCAiYiI6IFsz'
        'XX0pCiAgICBmb3IgcGFydHMgaW4gKHsiYSI6IFsxLCAxXX0sIHsiYSI6IFsxLCAyXSwgImIiOiBb'
        'MiwgM119KToKICAgICAgICB3aXRoIHB5dGVzdC5yYWlzZXMoVmFsdWVFcnJvciwgbWF0Y2g9Im92'
        'ZXJsYXAiKToKICAgICAgICAgICAgcnVubmVyLmVuc3VyZV9kaXNqb2ludChwYXJ0cykKCgpkZWYg'
        'dGVzdF9zZWxlY3Rpb25fdGllX29yZGVyX2FuZF93b3JzdF9yZWdpbWUoKSAtPiBOb25lOgogICAg'
        'cmVwb3J0cyA9IHtyOiB7ImEiOiB7ImFjY3VyYWN5IjogMC44fSwgImIiOiB7ImFjY3VyYWN5Ijog'
        'MC44fX0gZm9yIHIgaW4gcnVubmVyLlJFR0lNRVN9CiAgICBhc3NlcnQgcnVubmVyLnNlbGVjdChy'
        'ZXBvcnRzLCBbImIiLCAiYSJdKSA9PSAiYiIKICAgIHJlcG9ydHNbcnVubmVyLlJFR0lNRVNbMF1d'
        'WyJiIl1bImFjY3VyYWN5Il0gPSAwLjcKICAgIGFzc2VydCBydW5uZXIuc2VsZWN0KHJlcG9ydHMs'
        'IFsiYiIsICJhIl0pID09ICJhIgoKCmRlZiB0ZXN0X3Jlc2VydmVkX3NoaWZ0X3ByZXNlcnZlc190'
        'YXJnZXRzX2FuZF9zb3VyY2VfaWRlbnRpdHkoKSAtPiBOb25lOgogICAgZnJvbSBvbHltcHVzLmRh'
        'dGEuY29uc2VydmF0aXZlX3JvdXRpbmdfdjYgaW1wb3J0IGdlbmVyYXRlCgogICAgY2xlYW4gPSBn'
        'ZW5lcmF0ZSgyMDI2ODAwMSwgMTYsICJjbGVhbiIpCiAgICBzaGlmdGVkID0gZ2VuZXJhdGUoMjAy'
        'NjgwMDEsIDE2LCAiY29tcG9zaXRlX3NoaWZ0IikKICAgIGZvciBrZXkgaW4gKCJzb3VyY2VfaWRz'
        'IiwgImRlY2lzaW9uX3RhcmdldHMiLCAibmV4dF9zdGF0ZV90YXJnZXRzIik6CiAgICAgICAgYXNz'
        'ZXJ0IHRvcmNoLmVxdWFsKGNsZWFuW2tleV0sIHNoaWZ0ZWRba2V5XSkKCgpkZWYgdGVzdF9hcnRp'
        'ZmljaWFsX2xpZmVjeWNsZV9hbmRfdGFtcGVyX3JlZnVzYWwoCiAgICB0bXBfcGF0aDogUGF0aCwg'
        'bW9ua2V5cGF0Y2g6IHB5dGVzdC5Nb25rZXlQYXRjaAopIC0+IE5vbmU6CiAgICAiIiJPbmUgdGlu'
        'eSBmaXh0dXJlOyBub3QgYW4gYWRtaXR0ZWQgc2NpZW50aWZpYyBjYW1wYWlnbiBvciByZXN1bHQu'
        'IiIiCiAgICBwcm90b2NvbCA9IGNvcHkuZGVlcGNvcHkocnVubmVyLlBST1RPQ09MKQogICAgcHJv'
        'dG9jb2wudXBkYXRlKAogICAgICAgIGRhdGFzZXRfc2VlZHM9WzIwMjY4OTAwXSwKICAgICAgICB0'
        'cmFpbmluZ19zZWVkcz1bMjAyNjg5NTBdLAogICAgICAgIGJhbmtfZXhhbXBsZXM9MTYsCiAgICAg'
        'ICAgYmFua19zdGVwcz0yLAogICAgICAgIGJhbmtfYmF0Y2hfc2l6ZT00LAogICAgICAgIG1ldGFf'
        'ZXhhbXBsZXNfcGVyX3JlZ2ltZT04LAogICAgICAgIHRlc3RfZXhhbXBsZXNfcGVyX3JlZ2ltZT04'
        'LAogICAgICAgIHJlZmVyZW5jZV9zdGVwcz0yLAogICAgICAgIGNvcnJlY3Rpb25fc3RlcHM9MiwK'
        'ICAgICAgICBiYXRjaF9wZXJfZ3JvdXA9NCwKICAgICAgICBib290c3RyYXBfc2FtcGxlcz0yMCwK'
        'ICAgICkKICAgIG1vbmtleXBhdGNoLnNldGF0dHIocnVubmVyLCAiUFJPVE9DT0wiLCBwcm90b2Nv'
        'bCkKICAgIGJhbmtfcHJvdG9jb2wgPSBkaWN0KHY0LlBST1RPQ09MLCBleHBlcnRfc3RlcHM9Miwg'
        'YmF0Y2hfc2l6ZT00KQogICAgbW9ua2V5cGF0Y2guc2V0YXR0cih2NCwgIlBST1RPQ09MIiwgYmFu'
        'a19wcm90b2NvbCkKICAgICMgVGhlIGhvc3Qgd2FzIGFkbWl0dGVkIGJ5IHRoZSBvdXRlciBxdWFs'
        'aWZpY2F0aW9uIGNvbW1hbmQ7IG5vIHJlc291cmNlCiAgICAjIHRocmVzaG9sZHMgYXJlIGRpc2Fi'
        'bGVkLCBhbmQgdGhlIHNhbWUgZ3VhcmQgaXMgY2FsbGVkIGluc2lkZSB0aGUgZml4dHVyZS4KICAg'
        'IHY0LndyaXRlKAogICAgICAgIHRtcF9wYXRoIC8gIkNPTlRSQUNULmpzb24iLAogICAgICAgIHsi'
        'c291cmNlcyI6IHJ1bm5lci5zb3VyY2VzKCksICJwcm90b2NvbCI6IHByb3RvY29sLCAic2NvcGUi'
        'OiAiYXJ0aWZpY2lhbF9maXh0dXJlIn0sCiAgICApCiAgICBpbXBvcnQgdGltZQoKICAgIGRlYWRs'
        'aW5lID0gdGltZS5tb25vdG9uaWMoKSArIDkwCiAgICBydW5uZXIuZGV2ZWxvcCh0bXBfcGF0aCwg'
        'ZGVhZGxpbmUpCiAgICBydW5uZXIuZnJlZXplKHRtcF9wYXRoKQogICAgcnVubmVyLmNvbmZpcm0o'
        'dG1wX3BhdGgsIGRlYWRsaW5lKQogICAgdjQud3JpdGUodG1wX3BhdGggLyAiU1VNTUFSWS5qc29u'
        'IiwgcnVubmVyLnN1bW1hcml6ZSh0bXBfcGF0aCkpCiAgICBhc3NlcnQgcnVubmVyLnZlcmlmeSh0'
        'bXBfcGF0aClbInJhd19yZXBvcnRzX3JlY29tcHV0ZWQiXSA9PSA0CiAgICByYXdfcGF0aCA9IG5l'
        'eHQoKHRtcF9wYXRoIC8gInByb3RlY3RlZCIpLmdsb2IoIiovY2xlYW4ucHQiKSkKICAgIHJlcG9y'
        'dF9wYXRoID0gcmF3X3BhdGgud2l0aF9zdWZmaXgoIi5qc29uIikKICAgIG9yaWdpbmFsX3Jhdywg'
        'b3JpZ2luYWxfcmVwb3J0ID0gcmF3X3BhdGgucmVhZF9ieXRlcygpLCByZXBvcnRfcGF0aC5yZWFk'
        'X2J5dGVzKCkKICAgIHJhdyA9IHRvcmNoLmxvYWQocmF3X3BhdGgsIHdlaWdodHNfb25seT1UcnVl'
        'KQogICAgcmF3WyJzb3VyY2VfaWRzIl0gPSByYXdbInNvdXJjZV9pZHMiXSArIDEwMDAwMDAwMDAw'
        'MAogICAgdG9yY2guc2F2ZShyYXcsIHJhd19wYXRoKQogICAgcmVwb3J0ID0gdjQucmVhZChyZXBv'
        'cnRfcGF0aCkKICAgIHJlcG9ydFsicHJlZGljdGlvbl9zaGEyNTYiXSA9IHY0LnNoYShyYXdfcGF0'
        'aCkKICAgIGltcG9ydCBqc29uCgogICAgcmVwb3J0X3BhdGgud3JpdGVfdGV4dChqc29uLmR1bXBz'
        'KHJlcG9ydCkpCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoUnVudGltZUVycm9yLCBtYXRjaD0icmVn'
        'ZW5lcmF0ZWQgc291cmNlIElEcyIpOgogICAgICAgIHJ1bm5lci52ZXJpZnkodG1wX3BhdGgpCiAg'
        'ICByYXdfcGF0aC53cml0ZV9ieXRlcyhvcmlnaW5hbF9yYXcpCiAgICByZXBvcnRfcGF0aC53cml0'
        'ZV9ieXRlcyhvcmlnaW5hbF9yZXBvcnQpCiAgICAjIEEgc2VsZi1jb25zaXN0ZW50IGNoYW5nZWQg'
        'cHJlZGljdGlvbi9oYXNoL21ldHJpYyBpcyBzdGlsbCBub3QgY2hlY2twb2ludCBldmlkZW5jZS4K'
        'ICAgIHJhdyA9IHRvcmNoLmxvYWQocmF3X3BhdGgsIHdlaWdodHNfb25seT1UcnVlKQogICAgcmF3'
        'WyJwcm9iYWJpbGl0aWVzIl1bImFuY2hvcmVkX3JvYnVzdCJdID0gcmF3WyJwcm9iYWJpbGl0aWVz'
        'Il1bImFuY2hvcmVkX3JvYnVzdCJdLnJvbGwoMSwgLTEpCiAgICB0b3JjaC5zYXZlKHJhdywgcmF3'
        'X3BhdGgpCiAgICByZXBvcnQgPSB2NC5yZWFkKHJlcG9ydF9wYXRoKQogICAgcmVwb3J0WyJwcmVk'
        'aWN0aW9uX3NoYTI1NiJdID0gdjQuc2hhKHJhd19wYXRoKQogICAgcmVwb3J0WyJtZXRyaWNzIl0g'
        'PSBydW5uZXIuZGVzY3JpYmUocmF3WyJwcm9iYWJpbGl0aWVzIl0sIHJhd1siYWxwaGEiXSwgcmF3'
        'WyJsYWJlbHMiXSkKICAgIHJlcG9ydF9wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQp'
        'KQogICAgd2l0aCBweXRlc3QucmFpc2VzKFJ1bnRpbWVFcnJvciwgbWF0Y2g9ImNoZWNrcG9pbnQg'
        'cHJlZGljdGlvbiByZXBsYXkiKToKICAgICAgICBydW5uZXIudmVyaWZ5KHRtcF9wYXRoKQogICAg'
        'cmF3X3BhdGgud3JpdGVfYnl0ZXMob3JpZ2luYWxfcmF3KQogICAgcmVwb3J0X3BhdGgud3JpdGVf'
        'Ynl0ZXMob3JpZ2luYWxfcmVwb3J0KQogICAgZnJlZXplID0gdG1wX3BhdGggLyAiU0NJRU5USUZJ'
        'Q19GUkVFWkUuanNvbiIKICAgIG9yaWdpbmFsX2ZyZWV6ZSA9IGZyZWV6ZS5yZWFkX2J5dGVzKCkK'
        'ICAgIGFsdGVyZWQgPSB2NC5yZWFkKGZyZWV6ZSkKICAgIGFsdGVyZWRbImZpbGVzIl0ucG9wKG5l'
        'eHQoaXRlcihhbHRlcmVkWyJmaWxlcyJdKSkpCiAgICBmcmVlemUud3JpdGVfdGV4dChqc29uLmR1'
        'bXBzKGFsdGVyZWQpKQogICAgd2l0aCBweXRlc3QucmFpc2VzKFJ1bnRpbWVFcnJvciwgbWF0Y2g9'
        'Im1hbmlmZXN0IG1lbWJlcnNoaXAiKToKICAgICAgICBydW5uZXIudmFsaWRhdGVfZnJlZXplKHRt'
        'cF9wYXRoKQogICAgZnJlZXplLndyaXRlX2J5dGVzKG9yaWdpbmFsX2ZyZWV6ZSkKICAgIGNvbnRy'
        'YWN0ID0gdG1wX3BhdGggLyAiQ09OVFJBQ1QuanNvbiIKICAgIG9yaWdpbmFsX2NvbnRyYWN0ID0g'
        'Y29udHJhY3QucmVhZF9ieXRlcygpCiAgICBjaGFuZ2VkX2NvbnRyYWN0ID0gdjQucmVhZChjb250'
        'cmFjdCkKICAgIGNoYW5nZWRfY29udHJhY3RbInB5dGhvbiJdID0gInRhbXBlcmVkIGVudmlyb25t'
        'ZW50IgogICAgY29udHJhY3Qud3JpdGVfdGV4dChqc29uLmR1bXBzKGNoYW5nZWRfY29udHJhY3Qp'
        'KQogICAgd2l0aCBweXRlc3QucmFpc2VzKFJ1bnRpbWVFcnJvciwgbWF0Y2g9ImNoYW5nZWQgZnJv'
        'emVuIGZpbGUiKToKICAgICAgICBydW5uZXIudmFsaWRhdGVfZnJlZXplKHRtcF9wYXRoKQogICAg'
        'Y29udHJhY3Qud3JpdGVfYnl0ZXMob3JpZ2luYWxfY29udHJhY3QpCiAgICBjaGVja3BvaW50ID0g'
        'bmV4dCgodG1wX3BhdGggLyAiZGV2ZWxvcG1lbnQiKS5nbG9iKCIqL2NoZWNrcG9pbnQucHQiKSkK'
        'ICAgIHdpdGggY2hlY2twb2ludC5vcGVuKCJhYiIpIGFzIHN0cmVhbToKICAgICAgICBzdHJlYW0u'
        'd3JpdGUoYiJ0YW1wZXIiKQogICAgd2l0aCBweXRlc3QucmFpc2VzKFJ1bnRpbWVFcnJvciwgbWF0'
        'Y2g9ImNoZWNrcG9pbnQgaGFzaHxjaGFuZ2VkIGZyb3plbiBmaWxlIik6CiAgICAgICAgcnVubmVy'
        'LnZlcmlmeSh0bXBfcGF0aCkK'
    ),
}
HASHES = {
    'LICENSE': (
        'f253151b912014fbdbd7283a4226dc34dd022f639a18eae552e847ad27d4ee7b'
    ),
    'olympus/__init__.py': (
        '3bc54c19fb529b525983b06e5233d11ef274a8046cc8ef71d67627809d1d958b'
    ),
    'olympus/api.py': (
        '4a1fc967440d75b7e14b4c093b0d404801d03845e5c318954c822e042ec05516'
    ),
    'olympus/cli.py': (
        '77ea945f430885185f671dd5b70f63d22d1e5d2d0b3f99463f0271630f585838'
    ),
    'olympus/core/__init__.py': (
        '990c5157585a49d86fe4e605b667f2985785f7f92b5cc1d3eff69151e6b8a9b6'
    ),
    'olympus/core/budget.py': (
        'd4ea7843b31888d60c76410c3c6e22e279b38faeb942c43e6206e0541e54d01c'
    ),
    'olympus/core/division.py': (
        'a189dc9f498c9abec25138c8b420d4d5a4d91fe518e7eb5d2420a45b7857dcd4'
    ),
    'olympus/core/interpretation.py': (
        '6110433e0965c62c33514f6977e8fcb3c3d6ade86ac943264b68b63751ff2ebe'
    ),
    'olympus/core/mapping.py': (
        'efde41cf5caa58f64ea80e463413a93824cdacd0917ff10b34eb2a2f93b18e39'
    ),
    'olympus/core/modalities.py': (
        'e31662d3f950bf3d87d269d8e61b02bad737d9ec39c52a0b46e9e921d0f2ce40'
    ),
    'olympus/core/observability.py': (
        '5184a3adf97a0e842f35929af47ed40f00aea99ae781d6a2be8452bbfef4508d'
    ),
    'olympus/core/outcome.py': (
        '6b06a54a4bc00b822fa3de8f77495938cb91dfbfb0166e8df4ec6031cbc3f296'
    ),
    'olympus/core/perspective.py': (
        '9067a6540a9b86a1c128932fb266ce10cb50a7ca5a8cf5b5be40c3539062c1a7'
    ),
    'olympus/core/prediction.py': (
        'beef7ad50e25a54006abc3d95d46da259bf30673c956d2b0b50ed0999f611b16'
    ),
    'olympus/core/representation.py': (
        'e369de60759ddfbfb2aa58624621e88a404c5db4c53b2b2127b6fa319ff4e465'
    ),
    'olympus/core/retrodiction.py': (
        '81f4baf7dda3a5e89bcefb2df665019b4b01646d3ea0848e2df4f12d94b57c2c'
    ),
    'olympus/core/schemas.py': (
        'e89c498176fb86b52f1a49b8c9961aa8f5cb431ef6a953abfff09cbef1e597f0'
    ),
    'olympus/core/security.py': (
        '1e33d05c8316d39dec9bd4bb27879cb4ae8bbdad958a600ee8f7de1c98ad959b'
    ),
    'olympus/core/sqlite_journal.py': (
        'd577ca0e3e5e0c1b3807386cc93efb797ae81c9a0f63c43cd7936db68c3845b5'
    ),
    'olympus/core/transform.py': (
        '2f608764a1cfe924c3d1b1b565bc053c17f4bd4faf3de10792ac76a9afdc034d'
    ),
    'olympus/core/verification.py': (
        '7f76edc54a733107798f994de38587d12ceccb61e9e512f1188a6077469f5623'
    ),
    'olympus/core/workspace.py': (
        '623f077705e27254712decda3160598ff895681abcf4af82f67c9e161d6b3f68'
    ),
    'olympus/data/__init__.py': (
        'aed6813343b71dcbb8cedcedf695d690fae94525576445d660da03b7739b3cf4'
    ),
    'olympus/data/conservative_routing_v6.py': (
        '65697cb3cbb2ca640087f0bbd3d6c4e57d676e9352c377b60d8f1ce9fbb502a3'
    ),
    'olympus/data/controlled_fusion.py': (
        'd23804807f19d465a50cb12fa223dc2950c73479973dbe477eff1b3e21bd10ad'
    ),
    'olympus/data/editclevr.py': (
        '9c1d1f50d01385e573eb1891b3633419f41facaed7e2918b9a7863e9075e55f0'
    ),
    'olympus/data/ingestion.py': (
        '6d6599177e01eaf84baf41d2c24e6026765cd189ff32027032865e7a9da072bd'
    ),
    'olympus/data/manifests.py': (
        '91875d590c50d0b3dca7126c6bca235d75cf7fe5ba9489e11498df7e9f2a7999'
    ),
    'olympus/data/olympus_dataset.py': (
        'e612a824f6572490423d6894f57d2b7b9e4bf767dd83c2659fbde87c82ff96b3'
    ),
    'olympus/data/oracle_fusion_v4.py': (
        '93a2221fd675bf660eec86cdabd58085cb1917d08a484d16cc7c8f61951f191f'
    ),
    'olympus/data/registry.py': (
        '23834ae066cb9b38318baf819b7d23c176cb3d6bc00f003c8df0193ae011bb32'
    ),
    'olympus/data/reliability_fusion.py': (
        'acf2d9d0592705e762548c068464d9f6c2d735274d0cd8c3b053f5353261c944'
    ),
    'olympus/data/retrieval.py': (
        '4316227d671a3dc8c0d7b49b4b76c9472658c2e183da030c428e107d4b0a8861'
    ),
    'olympus/data/robust_fusion_v5.py': (
        '6d0d92739543d83378a9da8b658e4f6645fe56947a69290240fede95d280d3e0'
    ),
    'olympus/demos.py': (
        'ffed31e5a5eede5adaa0274f97588ee6135f71acdbb9c17293712b9860957a85'
    ),
    'olympus/evaluation/__init__.py': (
        'f449c9ce782ca03d42fc8d52c372adad800e09326e700d611d382277c48f8eba'
    ),
    'olympus/evaluation/decision_dossiers.py': (
        'a5ee4f8dee52d610e9cdc5d76867f3dc2c290a28f7082b75bb4d3933f120a932'
    ),
    'olympus/evaluation/harness.py': (
        '99d1966c0d7d783a3773c92b61c501e722cc8f25810610cb427f2823754527ec'
    ),
    'olympus/evaluation/olympus_system.py': (
        'bac95fd017308fb8c1c60c68929f6d8bb9dd4c49b4c7a9d0a7674c69fcfc9103'
    ),
    'olympus/forge/__init__.py': (
        '6f11962b71aec233a5417d3dbfe118d5dc9440feeb9af9de79a6147d2fba6b9e'
    ),
    'olympus/forge/compiler.py': (
        'c74bafa8606f28bb3ec24f5d3f0d723f252f02ec52359b2daaa6dd678263c2d9'
    ),
    'olympus/forge/experiments.py': (
        'c5a8d11609420f52aeb7b8145be4ccb2ccb2ad3c133f03345ecf1c7572cf03e5'
    ),
    'olympus/forge/language.py': (
        '5d5a355469e8e6d20a768e0eed74fd8b09b2e7bed97a01ea2d9998de88f41af1'
    ),
    'olympus/forge/runtime.py': (
        '0beeff2d03954bc046ccc9844342539664822a47ce92b52e76754f55b35b0c17'
    ),
    'olympus/foundry/__init__.py': (
        '8478c8acc64a9acfd87e26b3ef436923caa7ca83afd4e6bf81434ab78b8a9431'
    ),
    'olympus/foundry/bigram.py': (
        '371ffe3ccd1f7f1e535ad18fa067924f8af0780448f54ec4e1d7848a0ba96f53'
    ),
    'olympus/foundry/data_pipeline.py': (
        'ecc3a9b40279bc5b8796cfe0d8625a841ed40711aab7fe8b0b67e434acaacd24'
    ),
    'olympus/foundry/eval_suite.py': (
        '4ed8843b36fe89a56334843e42b83b520f428ace8cb350bf3353c3115e89f151'
    ),
    'olympus/foundry/load_probe.py': (
        '2444f85d880e7b894b4b8d0c6ec3c1782d0223b72dc984f08de712ab22cdc8ed'
    ),
    'olympus/foundry/ollama.py': (
        '4371570711b19015c2720691ac2c37535520e5b6b5c29972f5b3f363087827a3'
    ),
    'olympus/foundry/promotion.py': (
        '88885642ba659e34e09f475b5e9c93cfbb60c01b6e8d4b80408e553f180bb620'
    ),
    'olympus/foundry/quantization.py': (
        '5f1a0a41a5d5920210b03c30f1027fb42579a33080188b2a08ba3721452ac0f1'
    ),
    'olympus/foundry/readiness.py': (
        'ddf2b29651cedd87e18205d53bfb388a384fdd5e4ff4f1a8323273ee0434943b'
    ),
    'olympus/foundry/resources.py': (
        '2a09fa333d409dd0f60e10212214a930508e84a1ff71fb3a3f674d4dc77b73ad'
    ),
    'olympus/foundry/schemas.py': (
        'cb6bbc4e7f8362245f75a28f210b50bc7b4b54e36f8b8490b5a7c37b5953f19f'
    ),
    'olympus/foundry/service.py': (
        '869f0e277bed42fb2214e9944012b05652d184f167df9ca8c27007b40207b46e'
    ),
    'olympus/foundry/sft.py': (
        '5429e6e120106f50be085b6d2d820843a47dbc167b5e80c4c2670f9bbe60be87'
    ),
    'olympus/foundry/store.py': (
        '0450caa8686116c0f0b72b66907234e0ef4f60ebb2ebc6a43c1f592f8f648e0d'
    ),
    'olympus/labos/__init__.py': (
        '1785439a11bf594dc7a042eb9005ab0a2586e34ac8cf06d4e36231cee9a18ff7'
    ),
    'olympus/labos/discovery.py': (
        'b700da24bb355946e245a0b8097cf75b8fd06d04be7b99ed8ddf32bdaddc8e58'
    ),
    'olympus/labos/manifest.py': (
        '9da9f6b93b905117af9cddca2a3186c7a77cd5fbf2dee39c53cb9e0819376be7'
    ),
    'olympus/labos/portfolio.py': (
        '768719f78da837fe015ad33d0d551dde3d0df0025452e3752f2aa20874b9b53a'
    ),
    'olympus/labos/reports.py': (
        'ff174d504ead84ac8ecc9d634b9aaf733a783def25908bf6c8586f841fa94d46'
    ),
    'olympus/labos/runner.py': (
        '21fc396eb8bcb952fa24dc1fdcd7fab6207bd6ef2e4f2ef8ee4bcf9b670b42b6'
    ),
    'olympus/labos/scheduler.py': (
        '9a9fb0b1834c09d1d6de21f96e984c6cf8aca188093187265e71546fea8064cf'
    ),
    'olympus/labos/validation.py': (
        '3758f74c38e9148e585e675f7e064909243e384924827420133d4e574bd174a1'
    ),
    'olympus/memory/__init__.py': (
        '9a8ad140f0403fb7f1d2dbd2440db3201e5eb0a5807b67c8f9e74642959886f4'
    ),
    'olympus/memory/store.py': (
        '27823c8aac59e2718e420515504f88b12411ff99833859ea9c4452b087bfe2e1'
    ),
    'olympus/models/__init__.py': (
        '11daeb99f263347b641b68e84973ac84ce01af360294af85691912037855730a'
    ),
    'olympus/models/capacity.py': (
        'ab6939d77cb61ae6eca707b16fef6e560314c675deca691290b2d3908a9b8c0e'
    ),
    'olympus/models/conservative_routing_v6.py': (
        '3b9067f8a518147396d8d3181e107f748990c2968bfa8968ee2977a1a5a26270'
    ),
    'olympus/models/editclevr.py': (
        'ba122fee340e5d29f8525406e2fe5809fb74230df1981b0ce7a6a8e0bfefab4b'
    ),
    'olympus/models/encoder_quantization.py': (
        '047d212d11bed763cc79405adda670efb5fa406c5cb85350551fe526f92073b3'
    ),
    'olympus/models/families.py': (
        '1144b12e02ba5840157c0546bd541db4147edfa30da4f41a71f553a41a5616a2'
    ),
    'olympus/models/flagship.py': (
        'e3f10f99e33133ceb981f2ae6e01bbb762c574f661afb55780a25cb80a34a640'
    ),
    'olympus/models/olympus_system.py': (
        'ca9b40f292f62a599714ae756465ee1e46242ec9fa84eb07754ff8350857469f'
    ),
    'olympus/models/oracle_fusion_v4.py': (
        '1e3bcb8af4814ae303bbd846313f361c72fae8fa8310a11598ce003d2d0fa60a'
    ),
    'olympus/models/parameter_growth.py': (
        'faf7eb277513dfbaab53ebeab63f13415dd7c9d2d95438863467c47d5c11a11b'
    ),
    'olympus/models/reference_families.py': (
        'b0def64a3c2258cd8b19f2e5bf45f148061f6bcc0e232f14306841209ebc69e4'
    ),
    'olympus/models/reliability_fusion.py': (
        'a1c2291fc47f552376791a4880cd9bd12bc039bbe0ed8fd6bfe51fabccd79f78'
    ),
    'olympus/models/robust_fusion_v5.py': (
        '124f0e7aa7b7f53ca837304b22b192875fa2a93dd2e8a19e8e61affebc235c85'
    ),
    'olympus/models/runtime.py': (
        '80dae9d7860901f19666fcd2caed40fbf287c50f3178800cea104560532dbe65'
    ),
    'olympus/models/seven_model_contract.py': (
        'f8dd9a244fe1797d6b5389c39ab9783e063a80275b3ae48d4d4e4f5da2738154'
    ),
    'olympus/models/spatiotemporal_lazy.py': (
        '9c9a86d1cbc6ba1da8b9b354d30a5e94b9b46f206077952ea858c7160c703c0b'
    ),
    'olympus/models/specifications.py': (
        '9156dad138b72ddc69dbf13c3b056de5973606d92c9bbc77d27d944a0b2dd2d7'
    ),
    'olympus/research_state.py': (
        'dc19a257db15da6dccec9a99ce5dcac1507c660c1263136d8e206df84255e07a'
    ),
    'olympus/sdk.py': (
        '1a3811422972240b013291b279418e55ead8a875240f04eaddedec0027807266'
    ),
    'olympus/training/__init__.py': (
        'a6c17a8ec8867f67ec9f30764f9579f390f45189913638714a611a7d22dd05e4'
    ),
    'olympus/training/campaign.py': (
        '9f36427f65ac1480a4edacd7f7952c49cb5a94c9854c8e7dbbfa8159d3e6c830'
    ),
    'olympus/training/checkpoints.py': (
        'd35a8ae76fea0a018efafb0728c7a7725044db3a135ebead9d5303385153ea7e'
    ),
    'olympus/training/encoder_quantization.py': (
        '24e07232438cc59503081c381bbf81023dd51b331b6c50fadd7e73d7bb6f9803'
    ),
    'olympus/training/olympus_system.py': (
        'e32e529862878662617b43ba4533ce81a2a35077a5469fa029ddd97f4e19f10c'
    ),
    'olympus/training/provenance.py': (
        'b048f0c004038f7a31fcfecc03e1f5d35dc1841b96b534e7172e2e36e0f9e3a3'
    ),
    'olympus/training/synthetic.py': (
        '1a7290890ea1a8c7750e333cbb0cd03db3ede25d5cd90729c0792548486e5888'
    ),
    'olympus/training/trainer.py': (
        '2aad19a32437cfb5c1d2672b331623450785c0fa2f90722bf8de92f0ff27ab05'
    ),
    'olympus/upscaler/__init__.py': (
        '6f1636998ce43e356911a1b5cfebef7e3393216782cb260b7ac549d9c08fd8fe'
    ),
    'olympus/upscaler/jepa.py': (
        '50676264f84cb2e7d40eb3642645ffe482f4f25f444b079f2eb53cdfa02b5a5c'
    ),
    'olympus/upscaler/training.py': (
        '2dc6d917cf4df68caa2601aadeda523946c2d00dddc1c81730e9c018a356227e'
    ),
    'research/olympus_routing_v6/protocol.json': (
        '3ac832bade0e5d1321ee5c6d305d81fe43d3806f119c209a817d2759a8d5cd7a'
    ),
    'scripts/__init__.py': (
        'a3f27293bb35acb28573d82c89720dd2acf0b1f512637319d1614ade52742414'
    ),
    'scripts/qualify_flagship_v2.py': (
        '36f0ea4a0ec339c2e6bd331396bdbed0a51235cb7ab3168e2983ddb291efc56f'
    ),
    'scripts/run_conservative_routing_v6.py': (
        'a9d080804819a0f253aab5395869270f438c57d5c3f4476db8592b6b1bc82723'
    ),
    'scripts/run_oracle_fusion_v4.py': (
        '560e742f7494a8e3df01d5872fa74d17ec576684ac30c44485ae7ca788416c58'
    ),
    'tests/__init__.py': (
        '34c86913f2bcd1b314d95de2d2a76e04f609e6fcedaf61c3a3f477d341e49a7b'
    ),
    'tests/test_conservative_routing_v6.py': (
        '51c189b56e362ba7321a3c182a1e358a9c0f52e04d55abc8074fa32d85bfb927'
    ),
}
EXPECTED = 'e2773e00d757af02e92ef0f2bdc5d15b8de629fd94cf56d8636b47ef040035e9'
assert hashlib.sha256(json.dumps(HASHES, sort_keys=True).encode()).hexdigest() == EXPECTED
BASE = Path('/kaggle/working') / ('olympus-v6-' + EXPECTED[:12])
SOURCE, RESULTS = BASE / 'source', BASE / 'results'
SOURCE.mkdir(parents=True, exist_ok=True)
for name, encoded in PAYLOAD.items():
    relative = Path(name)
    if relative.is_absolute() or '..' in relative.parts:
        raise ValueError('unsafe source member')
    value = base64.b64decode(encoded)
    assert hashlib.sha256(value).hexdigest() == HASHES[name]
    destination = SOURCE / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert destination.read_bytes() == value, 'Existing source differs'
    else:
        destination.write_bytes(value)
RESULTS.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ, PYTHONPATH=str(SOURCE), PYTEST_DISABLE_PLUGIN_AUTOLOAD='1',
           OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
def command(action, destination, *extra):
    subprocess.run([sys.executable, '-m', 'scripts.run_conservative_routing_v6',
                    action, '--output', str(destination), *map(str, extra)],
                   cwd=SOURCE, env=ENV, check=True)
def preserve():
    archive = BASE / ('retained-' + str(time.time_ns()) + '.tar.gz')
    with tarfile.open(archive, 'w:gz') as stream:
        stream.add(RESULTS, arcname='results')
    print('RETAINED', archive, hashlib.sha256(archive.read_bytes()).hexdigest())
print('SOURCE VERIFIED', EXPECTED)


In [ ]:
# Keep False: the first campaign has already been admitted on GitHub CPU.
# Enable only for a separately authorized independent reproduction, never a
# favorable replacement result. Missing dependencies fail without downloads.
RUN_INDEPENDENT_REPRODUCTION = False
qualification = RESULTS / 'qualification'
try:
    command('qualify', qualification)
    if RUN_INDEPENDENT_REPRODUCTION:
        command('run', RESULTS / 'independent-reproduction', '--qualification', qualification)
finally:
    preserve()
